In [ ]:
# import os

# def save_dataframes(df_averaged, df_master, basedir):
#     """
#     Save both df_averaged and df_master to the Results directory.

#     Parameters:
#     - df_averaged (pd.DataFrame): The averaged DataFrame.
#     - df_master (pd.DataFrame): The master DataFrame.
#     - basedir (str): Base directory to save the files.
#     """
#     # Define the output directory
#     output_dir = os.path.join(basedir, 'Results')
#     os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

#     # Define the file paths
#     averaged_csv_path = os.path.join(output_dir, 'df_averaged.csv')
#     master_csv_path = os.path.join(output_dir, 'df_master.csv')

#     # Save the DataFrames as CSV
#     df_averaged.to_csv(averaged_csv_path, index=True)
#     print(f'df_averaged saved to {averaged_csv_path}')
    
#     df_master.to_csv(master_csv_path, index=True)
#     print(f'df_master saved to {master_csv_path}')

# # Example usage
# save_dataframes(df_averaged, df_master, basedir)

In [ ]:
### IMPORT MODULES/LIBRARIES ###
import os
import warnings
import numpy as np
import math
import pickle
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.cluster import AgglomerativeClustering, SpectralClustering, KMeans
from sklearn.model_selection import GridSearchCV
from sklearn import linear_model

import scipy.stats as stats
from scipy.stats import f_oneway, tukey_hsd
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.signal import correlate
from scipy.signal import find_peaks

import scipy.io as sio
from scipy import interpolate
from sklearn.metrics import roc_auc_score as auROC

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
np.set_printoptions(suppress=True)
from tqdm import tqdm #progress bar
import pprint #for printing dictionaries


## RUN THIS FOR MACHINE LEARNING ETC

###################################
## This is for TensorFlow (a popular deep learning library)
import tensorflow as tf
import tensortools as tt
# Print TensorFlow version to confirm it's installed correctly
print(f"TensorFlow version: {tf.__version__}")
# Check for GPU availability with TensorFlow
print(f"TensorFlow has access to the following devices:\n{tf.config.list_physical_devices()}")

### This is for PyTorch (another popular deep learning library)
import torch
# Print PyTorch version to confirm it's installed correctly
print(f"Pytorch version: {torch.__version__}")
# Check if GPU is available for PyTorch
print(f"Is the GPU available to Pytorch: {torch.backends.mps.is_available()}")

# Establish key parameters

In [ ]:
### SAVE ALIGNED DATA TO DATA FOLDER? ###
### ONLY SAY yes BELOW IF YOU WANT TO OVERWRITE EXISTING FILES!!! ###
alphalevel = 0.05 ### set to preferred alpha for denoting significant responders
bh_correction = 'no' ### set to 'yes' or 'no' if you want to correct pvalues for multiple comparisons when plotting significant or notsignicant neurons

extractedsignalsonly = 'no' ###keep this as no for now, will update below if using the analyze single session function
mineventstoanalyze = 3 ### set to preferred minimum number of events required for data to be analyzed. Otherwise, FOV won't be analyzed

save_aligned_data = 'yes' ###'yes' or 'no'
save_auc_array = 'yes' ###'yes' or 'no'

delete_overlaps = 'no'  ### if an event occurs within selected number ms (below) of a previous event, that event is deleted
separation_requirement = 0 ### Set to 0 if delete_overlaps = 'no'. IN MILLISECONDS SO 1000 = 1 second

### SET THESE YOURSELF ###
frameaveraging = 4 ###cumulative frames averaged per second online and during post-precessing
framerate = 30 ###raw framerate (unaveraged)
timebetweenframes = 33.333333 ###time between each raw frame (unaveraged)
averagedframerate = framerate/frameaveraging 

### Z-score the data?
z_score_data = 'yes'

### PICK THE FRAMES THAT YOU WANT TO BE PLOTTED AND ANALYZED
pre_window_size = int(3*averagedframerate) #How many frames per trial before origin to be plotted? 7.5*3 is 22.5 frames, 23
window_size =  int(15*averagedframerate) ###for astrocyte analysis and longer trial window.. 15*7.5 is 112.5 frames, 113
post_window_size = window_size - pre_window_size

baselinefirstframe = 0
baselinelastframe = 22
tracefirstframe = pre_window_size
tracelastframe = int(pre_window_size+(3*averagedframerate)) ##from frames 23-45
infusionframe = int(pre_window_size+(3*averagedframerate)+1) ###frame 45 sucrose delivery--1ms after trace period ends
aucfirstframe = pre_window_size ###start auROC analysis at cue onset, frame 23
auclastframe = int(12*averagedframerate) ###for 6 sec post sucrose to be analyzed, frame 68

### INDICATE EVENT OF INTEREST OUT OF POSSIBILITIES BELOW FOR ANALYSIS ###
### Possible events: activelever, inactivelever, activelevertimeout, inactivelevertimeout, activeleverall, inactiveleverall,
### licks, cues, infusions
eventofinterest = 'cues'

### ASSIGN YOUR FOLDER FOR ANALYSIS
# basedir =  r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'

<table>
  <tr>
    <th>Variable</th>
    <th>Frame Number</th>
  </tr>
  <tr>
    <td>baselinefirstframe</td>
    <td>0</td>
  </tr>
  <tr>
    <td>baselinelastframe</td>
    <td>22</td>
  </tr>
  <tr>
    <td>pre_window_size</td>
    <td>22</td>
  </tr>
  <tr>
    <td>tracefirstframe</td>
    <td>22</td>
  </tr>
  <tr>
    <td>aucfirstframe</td>
    <td>22</td>
  </tr>
  <tr>
    <td>tracelastframe</td>
    <td>44</td>
  </tr>
  <tr>
    <td>infusionframe</td>
    <td>45</td>
  </tr>
  <tr>
    <td>auclastframe</td>
    <td>68</td>
  </tr>
  <tr>
    <td>post_window_size</td>
    <td>90</td>
  </tr>
  <tr>
    <td>window_size</td>
    <td>112</td>
  </tr>
</table>


# Create the analyze_single_session function

In [ ]:
import os
import numpy as np
import scipy.io as sio

def analyze_single_session(indir, window_size, pre_window_size, run_omission_test):
    # Get file paths
    tempfiles = next(os.walk(indir))[2]
    npyfiles = [f for f in tempfiles if os.path.splitext(f)[1]=='.npy' and 'extractedsignals_raw' in f]
    matfiles = [f for f in tempfiles if os.path.splitext(f)[1]=='.mat']
    if len(npyfiles) > 1:
        npyfile = [f for f in tempfiles if os.path.splitext(f)[1]=='.npy' and 'extractedsignals_raw' in f and not 'part2' in f and not 'part3' in f and not 'part4' in f]
        npyfile = npyfile[0]
        matfile = [f for f in tempfiles if os.path.splitext(f)[1]=='.mat' and not 'results' in f and not 'part2' in f and not 'part3' in f and not 'part4' in f]
        matfile = matfile[0]
    else:
        npyfile = npyfiles[0]
        matfile = matfiles[0]

    # Load signals and behavior data
    signals = np.squeeze(np.load(os.path.join(indir, npyfile)))
    numrois = signals.shape[0] 

    behaviordata = sio.loadmat(os.path.join(indir, matfile))
    eventlog = np.squeeze(behaviordata['eventlog'])
    licks = np.squeeze(behaviordata['licks'])
    lastframe_timestamp_part1 = np.max(eventlog)
    
    if extractedsignalsonly == 'yes':
        return signals

    # Extract events
    cueplus = eventlog[eventlog[:,0]==5,1]  # CS+
    cueminus = eventlog[eventlog[:,0]==6,1]  # CS-
    laseron = eventlog[eventlog[:,0]==7,1]
    infusions = eventlog[eventlog[:,0]==4,1]
    fxdpumps = np.squeeze(behaviordata['fxdpumps'])

    # Get day/phase from directory path
    day = os.path.basename(os.path.dirname(os.path.dirname(indir)))
    
    # Create reward status dictionary
    reward_status = {}
    
    # Handle reward status based on experimental phase
    if day in ['0_Early', '1_Late', '3_CueOmit']:
        # CS+ is rewarded, CS- is unrewarded
        for t in cueplus:
            reward_status[t] = 'rewarded'
        for t in cueminus:
            reward_status[t] = 'unrewarded'
            
    elif day == '2_SucOmit':
        # Check each CS+ trial for reward
        temp = fxdpumps - 3000  # Expected reward times
        for t in cueplus:
            if t in temp:
                reward_status[t] = 'rewarded'
            else:
                reward_status[t] = 'unrewarded'
        # CS- trials are unrewarded
        for t in cueminus:
            reward_status[t] = 'unrewarded'
            
    elif day in ['4_RevEarly', '5_RevLate']:
        # Reversal: CS+ is unrewarded, CS- is rewarded
        for t in cueplus:
            reward_status[t] = 'unrewarded'
        for t in cueminus:
            reward_status[t] = 'rewarded'
            
    # Save reward status and timestamps
    np.save(os.path.join(indir, f'reward_status_{eventofinterest}.npy'), reward_status)
    np.save(os.path.join(indir, f'trial_timestamps_plus_{eventofinterest}.npy'), cueplus)
    np.save(os.path.join(indir, f'trial_timestamps_minus_{eventofinterest}.npy'), cueminus)

    # Handle cue events if needed
    if 'cues' in eventofinterest:
        newarray_cueplus = np.array([])
        newarray_cueminus = np.array([])
        temp = np.hstack((cueplus, cueminus))
        for i in range(len(temp)):
            if temp[i] in cueplus:
                newarray_cueplus = np.append(newarray_cueplus, temp[i])
        cueplus = newarray_cueplus    
        for i in range(len(temp)):
            if temp[i] in cueminus:
                newarray_cueminus = np.append(newarray_cueminus, temp[i])
        cueminus = newarray_cueminus    
    
    if eventofinterest == 'cues':
        events_plus = cueplus.copy()
        events_minus = cueminus.copy()
    
    # Validate events
    if events_plus.size == 0:
        raise ValueError(f"events_plus is empty for {os.path.basename(indir)}")
    if events_minus.size == 0:
        raise ValueError(f"events_minus is empty for {os.path.basename(indir)}")
    
    # NEW ADDITION: Calculate timestamp-based lick rates
    def calculate_timestamp_lick_rates(event_timestamps, licks):
        baseline_window_ms = 3000  # 3 seconds in ms
        cue_window_ms = 3000       # 3 seconds in ms
        
        # Filter valid timestamps
        valid_timestamps = []
        for t in event_timestamps:
            if not np.isnan(t):
                valid_timestamps.append(t)
        
        if not valid_timestamps:
            return np.array([]), np.array([]), np.array([])
            
        # Calculate lick counts for each trial
        baseline_licks = np.array([np.sum((licks > t - baseline_window_ms) & (licks < t)) for t in valid_timestamps])
        cue_period_licks = np.array([np.sum((licks > t) & (licks < t + cue_window_ms)) for t in valid_timestamps])
        
        # Calculate rates (licks per second)
        baseline_rate = baseline_licks / (baseline_window_ms/1000)
        cue_period_rate = cue_period_licks / (cue_window_ms/1000)
        
        # Calculate delta rates
        delta_rate = cue_period_rate - baseline_rate
        
        return delta_rate, baseline_licks, cue_period_licks
    
    # Calculate timestamp-based lick metrics
    ts_delta_rates_plus, ts_baseline_licks_plus, ts_cue_licks_plus = calculate_timestamp_lick_rates(events_plus, licks)
    ts_delta_rates_minus, ts_baseline_licks_minus, ts_cue_licks_minus = calculate_timestamp_lick_rates(events_minus, licks)
    
    # Save timestamp-based metrics
    np.save(os.path.join(indir, f'ts_delta_rates_plus_{eventofinterest}.npy'), ts_delta_rates_plus)
    np.save(os.path.join(indir, f'ts_delta_rates_minus_{eventofinterest}.npy'), ts_delta_rates_minus)
    np.save(os.path.join(indir, f'ts_baseline_licks_plus_{eventofinterest}.npy'), ts_baseline_licks_plus)
    np.save(os.path.join(indir, f'ts_baseline_licks_minus_{eventofinterest}.npy'), ts_baseline_licks_minus)
    np.save(os.path.join(indir, f'ts_cue_licks_plus_{eventofinterest}.npy'), ts_cue_licks_plus)
    np.save(os.path.join(indir, f'ts_cue_licks_minus_{eventofinterest}.npy'), ts_cue_licks_minus)
    
    # Continue with frame-based analysis
    # Handle frame timestamps
    if animal == 'CTL1' or animal == 'ER-L1' or animal == 'ER-L2' or animal == 'IG-19' or animal == 'IG-28' or animal == 'PGa-T1' or animal == 'XYZ':
        frame_timestamps = np.array(assumed_frame_timestamps)
    else:
        frame_timestamps = fix_any_dropped_frames(eventlog[eventlog[:,0]==9,1])

    frame_timestamps = frame_timestamps[::frameaveraging]

    if signals.shape[1] > frame_timestamps.shape[0]:
        signals = signals[:,:frame_timestamps.shape[0]-1]

    final_frame_timestamp = frame_timestamps[signals.shape[1]]
    seconds_monitored = int(signals.shape[1]/averagedframerate)
    
    if int(max(events_plus)) > int(max(events_minus)):
        seconds_behavior = int(max(events_plus/1000))
    else:
        seconds_behavior = int(max(events_minus/1000)) 

    # Process signals
    signals /= np.nanmean(signals, axis=1)[:, None]
    
    if z_score_data == 'yes':
        for neuron in range(signals.shape[0]):
            mean = np.nanmean(signals[neuron])
            std = np.nanstd(signals[neuron])
            signals[neuron] = (signals[neuron] - mean)/std
    
    signalsT = signals.T
    
    def licks_to_frames(licks, frame_timestamps):
        lick_frames = np.searchsorted(frame_timestamps, licks, side='right') - 1
        return lick_frames

    def calculate_aligneddata_forevent(signalsT2, events2_plus, events2_minus, licks):
        framenumberfor_eventofinterest_plus = np.squeeze(framenumberforevent_plus(events2_plus, frame_timestamps))
        framenumberfor_eventofinterest_minus = np.squeeze(framenumberforevent_minus(events2_minus, frame_timestamps))
        
        numtrials_plus = framenumberfor_eventofinterest_plus.shape[0]
        numtrials_minus = framenumberfor_eventofinterest_minus.shape[0]
        alignedevents_plus = np.NAN*np.zeros([numtrials_plus,window_size,numrois])
        alignedevents_minus = np.NAN*np.zeros([numtrials_minus,window_size,numrois])

        lick_frames = licks_to_frames(licks, frame_timestamps)
        aligned_licks_plus = np.zeros((numtrials_plus, window_size))
        aligned_licks_minus = np.zeros((numtrials_minus, window_size))

        for i in np.flip(range(numtrials_plus)):
            eventindex_plus = framenumberfor_eventofinterest_plus[i]

            if np.isfinite(eventindex_plus) and eventindex_plus > pre_window_size and eventindex_plus < np.shape(signalsT2)[0]-post_window_size:
                eventindex_plus = int(eventindex_plus)
                alignedevents_plus[i, :, :] = signalsT2[eventindex_plus-pre_window_size:eventindex_plus+post_window_size, :]
                trial_licks = ((lick_frames >= eventindex_plus - pre_window_size) & 
                               (lick_frames < eventindex_plus + post_window_size))
                aligned_licks_plus[i] = np.histogram(lick_frames[trial_licks] - (eventindex_plus - pre_window_size), 
                                                     bins=window_size, range=(0, window_size))[0]
            else:
                alignedevents_plus = np.delete(alignedevents_plus, i, axis = 0)
                aligned_licks_plus = np.delete(aligned_licks_plus, i, axis = 0)

        for i in np.flip(range(numtrials_minus)):
            eventindex_minus = framenumberfor_eventofinterest_minus[i]

            if np.isfinite(eventindex_minus) and eventindex_minus > pre_window_size and eventindex_minus < np.shape(signalsT2)[0]-post_window_size:
                eventindex_minus = int(eventindex_minus)
                alignedevents_minus[i, :, :] = signalsT2[eventindex_minus-pre_window_size:eventindex_minus+post_window_size, :]
                trial_licks = ((lick_frames >= eventindex_minus - pre_window_size) & 
                               (lick_frames < eventindex_minus + post_window_size))
                aligned_licks_minus[i] = np.histogram(lick_frames[trial_licks] - (eventindex_minus - pre_window_size), 
                                                      bins=window_size, range=(0, window_size))[0]
            else:
                alignedevents_minus = np.delete(alignedevents_minus, i, axis = 0)
                aligned_licks_minus = np.delete(aligned_licks_minus, i, axis = 0)

        return alignedevents_plus, alignedevents_minus, aligned_licks_plus, aligned_licks_minus

    alignedevents_plus, alignedevents_minus, aligned_licks_plus, aligned_licks_minus = calculate_aligneddata_forevent(signalsT, events_plus, events_minus, licks) 

    # Check for NaNs in signals
    for i in range(signals.shape[0]):
        if np.isnan(np.mean(signals[i,:])):
            print(f"{os.path.basename(indir)} IMAGE J ROI.ZIP CELL NUMBER {i+1} HAS NaNs AND SHOULD BE CHANGED")
    
    alignedevents_plus = np.swapaxes(alignedevents_plus, 0,2)
    alignedevents_minus = np.swapaxes(alignedevents_minus, 0,2)
        
    popevents_plus = np.nanmean(alignedevents_plus, axis=2)
    popevents_minus = np.nanmean(alignedevents_minus, axis=2)

    # Save aligned data if requested
    if save_aligned_data == 'yes':
        np.save(os.path.join(indir, f'popevents_plus_{eventofinterest}.npy'), popevents_plus)
        np.save(os.path.join(indir, f'popevents_minus_{eventofinterest}.npy'), popevents_minus)
        np.save(os.path.join(indir, f'alignedevents_plus_{eventofinterest}.npy'), alignedevents_plus)
        np.save(os.path.join(indir, f'alignedevents_minus_{eventofinterest}.npy'), alignedevents_minus)
        np.save(os.path.join(indir, f'aligned_licks_plus_{eventofinterest}.npy'), aligned_licks_plus)
        np.save(os.path.join(indir, f'aligned_licks_minus_{eventofinterest}.npy'), aligned_licks_minus)

    # Return both frame-based and timestamp-based results
    return (popevents_plus, popevents_minus, alignedevents_plus, alignedevents_minus, 
            aligned_licks_plus, aligned_licks_minus, ts_delta_rates_plus, ts_delta_rates_minus)

# These helper functions remain unchanged
def framenumberforevent_plus(event, frame_timestamps):
    framenumber_plus = np.nan*np.zeros(event.shape)
    for ie, e in enumerate(event):
        if np.isnan(e):
            framenumber_plus[ie] = np.nan
        else:
            temp = np.nonzero(frame_timestamps<=e)[0]
            if temp.shape[0]>0:
                framenumber_plus[ie] = np.nonzero(frame_timestamps<=e)[0][-1]
            else:
                framenumber_plus[ie] = 0
    return framenumber_plus

def framenumberforevent_minus(event, frame_timestamps):
    framenumber_minus = np.nan*np.zeros(event.shape)
    for ie, e in enumerate(event):
        if np.isnan(e):
            framenumber_minus[ie] = np.nan
        else:
            temp = np.nonzero(frame_timestamps<=e)[0]
            if temp.shape[0]>0:
                framenumber_minus[ie] = np.nonzero(frame_timestamps<=e)[0][-1]
            else:
                framenumber_minus[ie] = 0
    return framenumber_minus

# Deal with frame issues, define the auROC and Hoch Correction Functions

In [ ]:

import numpy as np
from scipy import stats
from sklearn.metrics import roc_auc_score

if save_aligned_data == 'yes':
    ### FUNCTIONS FOR FIXING FRAMES FOR EVENT ALIGNMENT. SKIP IF LOADING PAST DATA ###
    def fix_any_dropped_frames(frame_timestamps):
        first_frame = np.array([0])
        last_frame = np.array([int(np.max(frame_timestamps)+(500*timebetweenframes))])
        frame_index_temp = np.concatenate((first_frame,frame_timestamps, last_frame)) ###adds frame to timepoint '0' and an extra 500 frames at end
        frames_missed = [] ###creates empty list for us to add timestamps for missed frames
        for i in range(len(frame_index_temp)-1): ###iterates through each collected frame
                numframes_missed = int(np.round((frame_index_temp[i+1]-frame_index_temp[i])\
                    /timebetweenframes)-1) ### number of missed frames per frame interval
                if numframes_missed > 0: 
                    for j in range(numframes_missed):
                        frame_missed = np.array([frame_index_temp[i] + (int(timebetweenframes * (j+1)))])
                        frames_missed = np.concatenate((frames_missed, frame_missed))
        corrected_frame_index = np.array(sorted(np.concatenate((frame_index_temp, frames_missed))))
        return corrected_frame_index

    def fix_assumed_frames(frames):
        dropped_frames = []
        diff_frames = np.diff(frames)
        inter_frame_interval = 33
        frame_drop_idx = np.where(diff_frames>1.5*inter_frame_interval)[0]
        for idx in frame_drop_idx:
            numframesdropped = int(np.round((frames[idx+1]-frames[idx])/(inter_frame_interval+0.0))-1)
            temp = [frames[idx]+a*inter_frame_interval for a in range(1,numframesdropped+1)]
            dropped_frames.extend(temp)


        corrected_frames = np.sort(np.concatenate((frames, np.array(dropped_frames))))
        return corrected_frames

def calculate_auROC(x, y, offset_to_zero=True):
    U, p = stats.mannwhitneyu(x, y)
    labels = np.concatenate((np.ones(x.shape), np.zeros(y.shape)))
    data = np.concatenate((x, y))
    
    A = auROC(labels, data)

    if offset_to_zero:
        return (2*(A-0.5), p)
    else:
        return (A, p)
    
def Benjamini_Hochberg_correction(vector_of_pvals,
                                  alpha = 0.05):
    # This function implements the BH FDR correction
    
    # Parameters:
    # Vector of p values from the different tests
    # alpha:significance level
    
    # Returns: Corrected p values. All the p values that are above the FDR threshold are set to 1. 
    #          Remaining p values are unchanged.
    
    sortedpvals = np.sort(vector_of_pvals)
    orderofpvals = np.argsort(vector_of_pvals)
    m = sortedpvals[np.isfinite(sortedpvals)].shape[0] #Total number of hypotheses
    for i in range(m):
        if sortedpvals[i] > (i+1)*alpha/m:
            k = i
            break
        elif i == m-1:
            k = m-1
        
    correctedpvals = np.copy(vector_of_pvals)
    correctedpvals[orderofpvals[k:]] = 1
    correctedpvals[np.isnan(vector_of_pvals)] = np.nan
    return correctedpvals

def iterate_dirs(basedir, days):
    for day in sorted(days):
        animals = next(os.walk(os.path.join(basedir, day)))[1]
        for animal in sorted(animals):
            FOVs = next(os.walk(os.path.join(basedir, day, animal)))[1]
            for fov in sorted(FOVs):
                yield basedir, day, animal, fov


# Edited event alignment

In [ ]:
### EVENT ALIGNMENT: SAVES AND/OR LOADS ALIGNED FLUORESCENT DATA (2D: ROIS BY TIME) SURROUNDING EVENT ###
### NOTE: NEW DATA WILL BE SAVED IN YOUR FILE FOLDERS IF save_aligned_data == 'yes'!!!

### SELECTS ALL DAYS WITHIN A DIRECTORY ###
tempdays = next(os.walk(os.path.join(basedir)))[1]
days = []
for t in tempdays:
    if t != 'CellTracking' and t != 'Codes' and t != '.ipynb_checkpoints' and t != 'Other' and t != 'Results':
        days = np.append(days, t)
numdays = len(days)

# Define the folder_config dictionary with reward information
folder_config = {
    '0_Early': {'omission_test': False, 'cs_plus_reward': True, 'cs_minus_reward': False},
    '1_Late': {'omission_test': False, 'cs_plus_reward': True, 'cs_minus_reward': False},
    '2_SucOmit': {'omission_test': True, 'cs_plus_reward': 'variable', 'cs_minus_reward': False},
    '3_CueOmit': {'omission_test': True, 'cs_plus_reward': True, 'cs_minus_reward': False},
    '4_RevEarly': {'omission_test': False, 'cs_plus_reward': False, 'cs_minus_reward': True},
    '5_RevLate': {'omission_test': False, 'cs_plus_reward': False, 'cs_minus_reward': True}
}

### DICTIONARIES FOR ALL DATA FOR ALL DAYS AND FOVS ###
popevents_plus_day = {}  # Dictionary for all CS+ popevents data for all days
popevents_minus_day = {}  # Dictionary for all CS- popevents data for all days
popevents_plus_fov = {}  # Dictionary for all CS+ popevents data for all fovs
popevents_minus_fov = {}  # Dictionary for all CS- popevents data for all fovs
licks_plus_day = {}  # Dictionary for all CS+ lick data for all days
licks_minus_day = {}  # Dictionary for all CS- lick data for all days
licks_plus_fov = {}  # Dictionary for all CS+ lick data for all fovs
licks_minus_fov = {}  # Dictionary for all CS- lick data for all fovs

# New dictionaries for reward status
reward_status_plus_day = {}  # Dictionary for CS+ reward status by day
reward_status_minus_day = {}  # Dictionary for CS- reward status by day
reward_status_plus_fov = {}  # Dictionary for CS+ reward status by FOV
reward_status_minus_fov = {}  # Dictionary for CS- reward status by FOV

# NEW: Dictionaries for timestamp-based lick data
ts_delta_rates_plus_day = {}  # Dictionary for all CS+ timestamp-based delta rates by day
ts_delta_rates_minus_day = {}  # Dictionary for all CS- timestamp-based delta rates by day
ts_delta_rates_plus_fov = {}  # Dictionary for all CS+ timestamp-based delta rates by FOV
ts_delta_rates_minus_fov = {}  # Dictionary for all CS- timestamp-based delta rates by FOV

for day in sorted(days):
    phase_config = folder_config.get(day, {'omission_test': False, 'cs_plus_reward': None, 'cs_minus_reward': None})
    run_omission_test = phase_config['omission_test']
    print(f"Processing {day} - CS+ reward: {phase_config['cs_plus_reward']}, CS- reward: {phase_config['cs_minus_reward']}")
    
    if day not in popevents_plus_fov:
        popevents_plus_day[day] = np.nan*np.ones((1, window_size))
        popevents_plus_fov[day] = {}
        licks_plus_day[day] = np.nan*np.ones((1, window_size))
        licks_plus_fov[day] = {}
        reward_status_plus_day[day] = []
        reward_status_plus_fov[day] = {}
        # NEW: Initialize timestamp-based dictionaries
        ts_delta_rates_plus_day[day] = np.array([])
        ts_delta_rates_plus_fov[day] = {}
        print(day, 'CS+')
        
    if day not in popevents_minus_fov:
        popevents_minus_day[day] = np.nan*np.ones((1, window_size))
        popevents_minus_fov[day] = {}
        licks_minus_day[day] = np.nan*np.ones((1, window_size))
        licks_minus_fov[day] = {}
        reward_status_minus_day[day] = []
        reward_status_minus_fov[day] = {}
        # NEW: Initialize timestamp-based dictionaries
        ts_delta_rates_minus_day[day] = np.array([])
        ts_delta_rates_minus_fov[day] = {}
        print(day, 'CS-')
    
    for animal in next(os.walk(os.path.join(basedir, day)))[1]:
        for fov in next(os.walk(os.path.join(basedir, day, animal)))[1]:
            if save_aligned_data == 'yes':
                # UPDATED: Handle additional return values from analyze_single_session
                temp1, temp2, temp3, temp4, temp5, temp6, temp7, temp8 = analyze_single_session(
                    os.path.join(basedir, day, animal, fov), window_size, pre_window_size, run_omission_test)
                
                print(f"Alignedevents shape for {animal}, {fov}: CS+ {temp3.shape}, CS- {temp4.shape}")
                print(f"Aligned licks shape for {animal}, {fov}: CS+ {temp5.shape}, CS- {temp6.shape}")
                
                # NEW: Print timestamp-based lick data
                if len(temp7) > 0 and len(temp8) > 0:
                    print(f"Timestamp-based delta rates for {animal}, {fov}: CS+ {temp7.shape}, CS- {temp8.shape}")
                    # Initialize animal/FOV entries if needed
                    if animal not in ts_delta_rates_plus_fov[day]:
                        ts_delta_rates_plus_fov[day][animal] = {}
                        ts_delta_rates_minus_fov[day][animal] = {}
                    
                    # Save timestamp-based delta rates
                    ts_delta_rates_plus_fov[day][animal][fov] = temp7
                    ts_delta_rates_minus_fov[day][animal][fov] = temp8
                    
                    # Append to day-level arrays
                    if len(ts_delta_rates_plus_day[day]) == 0:
                        ts_delta_rates_plus_day[day] = temp7
                    else:
                        ts_delta_rates_plus_day[day] = np.concatenate((ts_delta_rates_plus_day[day], temp7))
                    
                    if len(ts_delta_rates_minus_day[day]) == 0:
                        ts_delta_rates_minus_day[day] = temp8
                    else:
                        ts_delta_rates_minus_day[day] = np.concatenate((ts_delta_rates_minus_day[day], temp8))

            tempfiles = next(os.walk(os.path.join(basedir, day, animal, fov)))[2]
            
            # Load reward status if available
            if f'reward_status_{eventofinterest}.npy' in tempfiles:
                reward_status = np.load(os.path.join(basedir, day, animal, fov, f'reward_status_{eventofinterest}.npy'), allow_pickle=True).item()
                if animal not in reward_status_plus_fov[day]:
                    reward_status_plus_fov[day][animal] = {}
                    reward_status_minus_fov[day][animal] = {}
                reward_status_plus_fov[day][animal][fov] = reward_status
                reward_status_minus_fov[day][animal][fov] = reward_status
            
            # NEW: Load timestamp-based delta rates if available
            if f'ts_delta_rates_plus_{eventofinterest}.npy' in tempfiles and not save_aligned_data == 'yes':
                if animal not in ts_delta_rates_plus_fov[day]:
                    ts_delta_rates_plus_fov[day][animal] = {}
                temp7 = np.load(os.path.join(basedir, day, animal, fov, f'ts_delta_rates_plus_{eventofinterest}.npy'))
                if len(temp7) > 0:
                    ts_delta_rates_plus_fov[day][animal][fov] = temp7
                    if len(ts_delta_rates_plus_day[day]) == 0:
                        ts_delta_rates_plus_day[day] = temp7
                    else:
                        ts_delta_rates_plus_day[day] = np.concatenate((ts_delta_rates_plus_day[day], temp7))
            
            if f'ts_delta_rates_minus_{eventofinterest}.npy' in tempfiles and not save_aligned_data == 'yes':
                if animal not in ts_delta_rates_minus_fov[day]:
                    ts_delta_rates_minus_fov[day][animal] = {}
                temp8 = np.load(os.path.join(basedir, day, animal, fov, f'ts_delta_rates_minus_{eventofinterest}.npy'))
                if len(temp8) > 0:
                    ts_delta_rates_minus_fov[day][animal][fov] = temp8
                    if len(ts_delta_rates_minus_day[day]) == 0:
                        ts_delta_rates_minus_day[day] = temp8
                    else:
                        ts_delta_rates_minus_day[day] = np.concatenate((ts_delta_rates_minus_day[day], temp8))
            
            # Load existing frame-based data (unchanged)
            if 'popevents_plus_%s.npy'%(eventofinterest) in tempfiles: 
                if animal not in popevents_plus_fov[day]:
                    popevents_plus_fov[day][animal] = {}
                temp1 = np.load(os.path.join(basedir, day, animal, fov, 'popevents_plus_%s.npy'%(eventofinterest)))  
                if np.isfinite(np.nanmean(temp1)):
                    popevents_plus_day[day] = np.vstack((popevents_plus_day[day], temp1))
                    popevents_plus_fov[day][animal][fov] = temp1
                    baseline_plus = np.mean(popevents_plus_fov[day][animal][fov][:, baselinefirstframe:baselinelastframe], axis=1)
                    popevents_plus_fov[day][animal][fov] = popevents_plus_fov[day][animal][fov] - baseline_plus[:, None]
                    
            if 'popevents_minus_%s.npy'%(eventofinterest) in tempfiles: 
                if animal not in popevents_minus_fov[day]:
                    popevents_minus_fov[day][animal] = {}
                temp2 = np.load(os.path.join(basedir, day, animal, fov, 'popevents_minus_%s.npy'%(eventofinterest)))  
                if np.isfinite(np.nanmean(temp2)):
                    popevents_minus_day[day] = np.vstack((popevents_minus_day[day], temp2))
                    popevents_minus_fov[day][animal][fov] = temp2
                    baseline_minus = np.mean(popevents_minus_fov[day][animal][fov][:, baselinefirstframe:baselinelastframe], axis=1)
                    popevents_minus_fov[day][animal][fov] = popevents_minus_fov[day][animal][fov] - baseline_minus[:, None]
            
            if 'aligned_licks_plus_%s.npy'%(eventofinterest) in tempfiles: 
                if animal not in licks_plus_fov[day]:
                    licks_plus_fov[day][animal] = {}
                temp5 = np.load(os.path.join(basedir, day, animal, fov, 'aligned_licks_plus_%s.npy'%(eventofinterest)))  
                if np.isfinite(np.nanmean(temp5)):
                    licks_plus_day[day] = np.vstack((licks_plus_day[day], temp5))
                    licks_plus_fov[day][animal][fov] = temp5

            if 'aligned_licks_minus_%s.npy'%(eventofinterest) in tempfiles: 
                if animal not in licks_minus_fov[day]:
                    licks_minus_fov[day][animal] = {}
                temp6 = np.load(os.path.join(basedir, day, animal, fov, 'aligned_licks_minus_%s.npy'%(eventofinterest)))  
                if np.isfinite(np.nanmean(temp6)):
                    licks_minus_day[day] = np.vstack((licks_minus_day[day], temp6))
                    licks_minus_fov[day][animal][fov] = temp6
                    
for day in sorted(days):
    # Original processing remains the same
    popevents_plus_day[day] = popevents_plus_day[day][1:, :]  
    popevents_minus_day[day] = popevents_minus_day[day][1:, :]  
    baseline_plus = np.mean(popevents_plus_day[day][:, baselinefirstframe:baselinelastframe], axis=1)
    baseline_minus = np.mean(popevents_minus_day[day][:, baselinefirstframe:baselinelastframe], axis=1)
    popevents_plus_day[day] = popevents_plus_day[day] - baseline_plus[:, None]
    popevents_minus_day[day] = popevents_minus_day[day] - baseline_minus[:, None]
    
    licks_plus_day[day] = licks_plus_day[day][1:, :]  
    licks_minus_day[day] = licks_minus_day[day][1:, :]  
    
    if save_aligned_data == 'yes':
        # Save original data
        np.save(os.path.join(basedir, day, 'popevents_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), popevents_plus_day[day])
        np.save(os.path.join(basedir, day, 'popevents_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), popevents_minus_day[day])
        np.save(os.path.join(basedir, day, 'licks_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), licks_plus_day[day])
        np.save(os.path.join(basedir, day, 'licks_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), licks_minus_day[day])
        
        # Save reward status
        np.save(os.path.join(basedir, day, 'reward_status_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), reward_status_plus_day[day])
        np.save(os.path.join(basedir, day, 'reward_status_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), reward_status_minus_day[day])
        
        # NEW: Save timestamp-based delta rates
        np.save(os.path.join(basedir, day, 'ts_delta_rates_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), 
                ts_delta_rates_plus_day[day])
        np.save(os.path.join(basedir, day, 'ts_delta_rates_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), 
                ts_delta_rates_minus_day[day])
        
### STACK ALL DATA AND SAVE THE RESULTS

# Stack original data
popevents_plus_all = np.vstack([popevents_plus_day[day] for day in sorted(days)])
popevents_minus_all = np.vstack([popevents_minus_day[day] for day in sorted(days)])
popevents_all = np.stack((popevents_plus_all, popevents_minus_all), axis=-1)

licks_plus_all = np.vstack([licks_plus_day[day] for day in sorted(days)])
licks_minus_all = np.vstack([licks_minus_day[day] for day in sorted(days)])

# Stack reward status
reward_status_plus_all = np.concatenate([reward_status_plus_day[day] for day in sorted(days)])
reward_status_minus_all = np.concatenate([reward_status_minus_day[day] for day in sorted(days)])

# NEW: Stack timestamp-based delta rates
ts_delta_rates_plus_all = np.concatenate([ts_delta_rates_plus_day[day] for day in sorted(days)])
ts_delta_rates_minus_all = np.concatenate([ts_delta_rates_minus_day[day] for day in sorted(days)])

print(f"Final popevents_plus_all shape (Cells X Frames): {popevents_plus_all.shape}")
print(f"Final popevents_minus_all shape (Cells X Frames): {popevents_minus_all.shape}")
print(f"Final popevents_all shape (Cells X Frames X CS Type): {popevents_all.shape}")
print(f"Final licks_plus_all shape (Trials X Frames): {licks_plus_all.shape}")
print(f"Final licks_minus_all shape (Trials X Frames): {licks_minus_all.shape}")
# NEW: Print timestamp-based data shapes
print(f"Final ts_delta_rates_plus_all shape (Trials): {ts_delta_rates_plus_all.shape}")
print(f"Final ts_delta_rates_minus_all shape (Trials): {ts_delta_rates_minus_all.shape}")

# Calculate and display high licking percentages for comparison
threshold = 1.5  # Hz
frame_high_plus = ((licks_plus_all[:, pre_window_size:infusionframe].sum(axis=1) / 
                   ((infusionframe - pre_window_size) / averagedframerate)) - 
                  (licks_plus_all[:, baselinefirstframe:baselinelastframe].sum(axis=1) / 
                   ((baselinelastframe - baselinefirstframe) / averagedframerate)) > threshold).sum()
                   
frame_high_minus = ((licks_minus_all[:, pre_window_size:infusionframe].sum(axis=1) / 
                    ((infusionframe - pre_window_size) / averagedframerate)) - 
                   (licks_minus_all[:, baselinefirstframe:baselinelastframe].sum(axis=1) / 
                    ((baselinelastframe - baselinefirstframe) / averagedframerate)) > threshold).sum()

ts_high_plus = (ts_delta_rates_plus_all > threshold).sum()
ts_high_minus = (ts_delta_rates_minus_all > threshold).sum()

print("\nHigh licking trial comparison (frame-based vs timestamp-based):")
print(f"CS+ frame-based: {frame_high_plus}/{len(licks_plus_all)} trials ({frame_high_plus/len(licks_plus_all)*100:.1f}%)")
print(f"CS+ timestamp-based: {ts_high_plus}/{len(ts_delta_rates_plus_all)} trials ({ts_high_plus/len(ts_delta_rates_plus_all)*100:.1f}%)")
print(f"CS- frame-based: {frame_high_minus}/{len(licks_minus_all)} trials ({frame_high_minus/len(licks_minus_all)*100:.1f}%)")
print(f"CS- timestamp-based: {ts_high_minus}/{len(ts_delta_rates_minus_all)} trials ({ts_high_minus/len(ts_delta_rates_minus_all)*100:.1f}%)")

# Saving all files
if save_aligned_data == 'yes':
    # Save original data
    np.save(os.path.join(basedir, 'Results', 'popevents_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), popevents_plus_all)
    np.save(os.path.join(basedir, 'Results', 'popevents_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), popevents_minus_all)
    np.save(os.path.join(basedir, 'Results', 'popevents_%s_%s.npy'%(eventofinterest, separation_requirement)), popevents_all)
    np.save(os.path.join(basedir, 'Results', 'licks_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), licks_plus_all)
    np.save(os.path.join(basedir, 'Results', 'licks_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), licks_minus_all)
    
    # Save reward status data
    np.save(os.path.join(basedir, 'Results', 'reward_status_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), reward_status_plus_all)
    np.save(os.path.join(basedir, 'Results', 'reward_status_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), reward_status_minus_all)
    
    # NEW: Save timestamp-based delta rates
    np.save(os.path.join(basedir, 'Results', 'ts_delta_rates_plus_%s_%s.npy'%(eventofinterest, separation_requirement)), ts_delta_rates_plus_all)
    np.save(os.path.join(basedir, 'Results', 'ts_delta_rates_minus_%s_%s.npy'%(eventofinterest, separation_requirement)), ts_delta_rates_minus_all)
    
    # Save experimental phase configuration
    np.save(os.path.join(basedir, 'Results', 'experiment_phases_%s_%s.npy'%(eventofinterest, separation_requirement)), folder_config)

In [ ]:
### TO FIND SIGNIFICANTLY RESPONDING NEURONS, FIRST CALCULATE OR LOAD EXISTING AUC SCORES AND ASSOCIATED PVALUES ###
### NOTE: NEW DATA WILL BE SAVED IN YOUR FILE FOLDERS IF save_auc_array == 'yes'!!!

### DICTIONARIES FOR ALL DATA FOR ALL DAYS AND FOVS ###
aucpvals_plus_day = {}  # Dictionary for all CS+ aucpvals data for all days
aucpvals_minus_day = {}  # Dictionary for all CS- aucpvals data for all days
aucpvals_plus_fov = {}  # Dictionary for all CS+ aucpvals data for all fovs
aucpvals_minus_fov = {}  # Dictionary for all CS- aucpvals data for all fovs

if save_auc_array == 'yes':
    for basedir, day, animal, fov in iterate_dirs(basedir, sorted(days)):
        tempfiles = next(os.walk(os.path.join(basedir, day, animal, fov)))[2]
        
        if 'alignedevents_plus_%s.npy' % (eventofinterest) in tempfiles: 
            temp3 = np.load(os.path.join(basedir, day, animal, fov, 'alignedevents_plus_%s.npy' % (eventofinterest)))
            if np.isfinite(np.nanmean(temp3)):
                aucpvals_plus_fov_temp = np.nan * np.ones((temp3.shape[0], 2))
                baseline_plus = np.mean(temp3[:, baselinefirstframe:baselinelastframe, :], axis=1)
                alignedevents_plus = temp3 - baseline_plus[:, None, :]
                newbaseline_plus = np.mean(alignedevents_plus[:, baselinefirstframe:baselinelastframe, :], axis=1)
                event_plus = np.mean(alignedevents_plus[:, aucfirstframe:auclastframe, :], axis=1)
                for neuron in range(np.shape(temp3)[0]):
                    auc_plus, pval_plus = calculate_auROC(event_plus[neuron, :], newbaseline_plus[neuron, :])
                    aucpvals_plus_fov_temp[neuron, 0] = auc_plus
                    aucpvals_plus_fov_temp[neuron, 1] = pval_plus
                np.save(os.path.join(basedir, day, animal, fov, 'aucpvals_plus_%s_%s.npy' % (eventofinterest, separation_requirement)), aucpvals_plus_fov_temp)
        
        if 'alignedevents_minus_%s.npy' % (eventofinterest) in tempfiles: 
            temp4 = np.load(os.path.join(basedir, day, animal, fov, 'alignedevents_minus_%s.npy' % (eventofinterest)))
            if np.isfinite(np.nanmean(temp4)):
                aucpvals_minus_fov_temp = np.nan * np.ones((temp4.shape[0], 2))
                baseline_minus = np.mean(temp4[:, baselinefirstframe:baselinelastframe, :], axis=1)
                alignedevents_minus = temp4 - baseline_minus[:, None, :]
                newbaseline_minus = np.mean(alignedevents_minus[:, baselinefirstframe:baselinelastframe, :], axis=1)
                event_minus = np.mean(alignedevents_minus[:, aucfirstframe:auclastframe, :], axis=1)
                for neuron in range(np.shape(temp4)[0]):
                    auc_minus, pval_minus = calculate_auROC(event_minus[neuron, :], newbaseline_minus[neuron, :])
                    aucpvals_minus_fov_temp[neuron, 0] = auc_minus
                    aucpvals_minus_fov_temp[neuron, 1] = pval_minus
                np.save(os.path.join(basedir, day, animal, fov, 'aucpvals_minus_%s_%s.npy' % (eventofinterest, separation_requirement)), aucpvals_minus_fov_temp)

for basedir, day, animal, fov in iterate_dirs(basedir, sorted(days)):
    if day not in aucpvals_plus_fov:
        aucpvals_plus_day[day] = np.nan * np.ones((1, 2))
        aucpvals_plus_fov[day] = {}
        print(day, 'CS+')
    tempfiles = next(os.walk(os.path.join(basedir, day, animal, fov)))[2]
    if 'aucpvals_plus_%s_%s.npy' % (eventofinterest, separation_requirement) in tempfiles: 
        if animal not in aucpvals_plus_fov[day]:
            aucpvals_plus_fov[day][animal] = {}
        aucpvals_plus_fov[day][animal][fov] = np.load(os.path.join(basedir, day, animal, fov, 'aucpvals_plus_%s_%s.npy' % (eventofinterest, separation_requirement)))
        aucpvals_plus_day[day] = np.vstack((aucpvals_plus_day[day], aucpvals_plus_fov[day][animal][fov]))

    if day not in aucpvals_minus_fov:
        aucpvals_minus_day[day] = np.nan * np.ones((1, 2))
        aucpvals_minus_fov[day] = {}
        print(day, 'CS-')
    tempfiles = next(os.walk(os.path.join(basedir, day, animal, fov)))[2]
    if 'aucpvals_minus_%s_%s.npy' % (eventofinterest, separation_requirement) in tempfiles: 
        if animal not in aucpvals_minus_fov[day]:
            aucpvals_minus_fov[day][animal] = {}
        aucpvals_minus_fov[day][animal][fov] = np.load(os.path.join(basedir, day, animal, fov, 'aucpvals_minus_%s_%s.npy' % (eventofinterest, separation_requirement)))
        aucpvals_minus_day[day] = np.vstack((aucpvals_minus_day[day], aucpvals_minus_fov[day][animal][fov]))

for day in sorted(days):
    aucpvals_plus_day[day] = aucpvals_plus_day[day][1:, :]  # Remove the first row of NaNs
    aucpvals_minus_day[day] = aucpvals_minus_day[day][1:, :]
    if save_auc_array == 'yes':
        np.save(os.path.join(basedir, day, 'aucpvals_plus_%s_%s.npy' % (eventofinterest, separation_requirement)), aucpvals_plus_day[day])
        np.save(os.path.join(basedir, day, 'aucpvals_minus_%s_%s.npy' % (eventofinterest, separation_requirement)), aucpvals_minus_day[day])        

aucpvals_plus_all = np.vstack([aucpvals_plus_day[day] for day in sorted(days)])
aucpvals_minus_all = np.vstack([aucpvals_minus_day[day] for day in sorted(days)])
aucpvals_all = np.stack((aucpvals_plus_all, aucpvals_minus_all), axis=-1)

print(f"Final aucpvals_plus_all shape (each cells AUC then pVal): {aucpvals_plus_all.shape}")
print(f"Final aucpvals_minus_all shape (each cells AUC then pVal): {aucpvals_minus_all.shape}")
print(f"Final aucpvals_all shape (each cell with its AUC then pVal - with each CStype): {aucpvals_all.shape}")

np.save(os.path.join(basedir, 'Results', 'aucpvals_plus_%s_%s.npy' % (eventofinterest, separation_requirement)), aucpvals_plus_all)
np.save(os.path.join(basedir, 'Results', 'aucpvals_minus_%s_%s.npy' % (eventofinterest, separation_requirement)), aucpvals_minus_all)
np.save(os.path.join(basedir, 'Results', 'aucpvals_%s_%s.npy' % (eventofinterest, separation_requirement)), aucpvals_all)

# Establish the comprehensive data frame (df_master) with metadata
## add in a column that contains the ROI number from the ImageJ RoiSet.zip and cross reference this to cell_number
## add timestamps for each trial from the matlab files

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.patches import Polygon
import roifile

def get_roi_numbers(basedir, day, animal, fov):
    roi_folder = os.path.join(basedir, day, animal, fov)
    roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
    
    if roi_files:
        roi_file = roi_files[0]
        roi_path = os.path.join(roi_folder, roi_file)
        rois = roifile.ImagejRoi.fromfile(roi_path)
        
        # Create a dictionary mapping cell index to ROI number
        roi_numbers = {i: i+1 for i in range(len(rois))}
        return roi_numbers
    else:
        print(f"No RoiSet.zip found for {day}, {animal}, {fov}")
        return {}

def verify_roi_numbers(basedir, day, animal, fov, df_master):
    roi_numbers = get_roi_numbers(basedir, day, animal, fov)
    df_fov = df_master[(df_master['day'] == day) & 
                       (df_master['animal'] == animal) & 
                       (df_master['fov'] == fov)]
    
    for cell_idx, roi_number in roi_numbers.items():
        df_roi_number = df_fov[df_fov['cell_number'] == cell_idx + 1]['roi_number'].iloc[0]
        assert roi_number == df_roi_number, f"Mismatch for {day}, {animal}, {fov}, ROI {roi_number}"
    
    print(f"Verification passed for {day}, {animal}, {fov}")

def create_efficient_master_dataframe(basedir, days, eventofinterest, separation_requirement, baselinefirstframe, baselinelastframe, averagedframerate):
    master_data = []
    
    cell_id_counter = 0
    for basedir, day, animal, fov in iterate_dirs(basedir, sorted(days)):
        print(f"Processing {day}, {animal}, {fov}")
        
        # Get ROI numbers for this FOV
        roi_numbers = get_roi_numbers(basedir, day, animal, fov)
        
        # Load data
        alignedevents_plus = np.load(os.path.join(basedir, day, animal, fov, f'alignedevents_plus_{eventofinterest}.npy'))
        alignedevents_minus = np.load(os.path.join(basedir, day, animal, fov, f'alignedevents_minus_{eventofinterest}.npy'))
        aligned_licks_plus = np.load(os.path.join(basedir, day, animal, fov, f'aligned_licks_plus_{eventofinterest}.npy'))
        aligned_licks_minus = np.load(os.path.join(basedir, day, animal, fov, f'aligned_licks_minus_{eventofinterest}.npy'))
        aucpvals_plus = np.load(os.path.join(basedir, day, animal, fov, f'aucpvals_plus_{eventofinterest}_{separation_requirement}.npy'))
        aucpvals_minus = np.load(os.path.join(basedir, day, animal, fov, f'aucpvals_minus_{eventofinterest}_{separation_requirement}.npy'))
        
        # Load timestamps and reward status
        trial_timestamps_plus = np.load(os.path.join(basedir, day, animal, fov, f'trial_timestamps_plus_{eventofinterest}.npy'))
        trial_timestamps_minus = np.load(os.path.join(basedir, day, animal, fov, f'trial_timestamps_minus_{eventofinterest}.npy'))
        reward_status = np.load(os.path.join(basedir, day, animal, fov, f'reward_status_{eventofinterest}.npy'), allow_pickle=True).item()


### THIS CODE IS USED TO GENERATE A SESSION WIDE GLOBAL BASELINE NORMALIZATION (THE CURRENT JIM STRAT)
        
        # Uses average baseline across all trials for each cell
        baseline_plus = np.mean(alignedevents_plus[:, baselinefirstframe:baselinelastframe, :], axis=(1, 2))  # Average across time AND trials
        baseline_minus = np.mean(alignedevents_minus[:, baselinefirstframe:baselinelastframe, :], axis=(1, 2))  # Average across time AND trials

        # Now subtract the cell's average baseline from all frames of all trials
        alignedevents_plus_corrected = alignedevents_plus - baseline_plus[:, np.newaxis, np.newaxis]  # Broadcast across time and trials
        alignedevents_minus_corrected = alignedevents_minus - baseline_minus[:, np.newaxis, np.newaxis]  # Broadcast across time and trials



# # ### THIS CODE IS USED TO GENERATE A PER-TRIAL BASELINE CORRECTION 
# # This approach calculates a UNIQUE baseline for each individual trial
# # BEST FOR: Trial-by-trial decoding, within-session condition comparison

#         # MODIFIED CODE FOR TRUE TRIAL-BY-TRIAL That uses each unique Trials BAseline: This works for Trial Decoding
#         baseline_plus = np.mean(alignedevents_plus[:, baselinefirstframe:baselinelastframe, :], axis=1)  # Only average across time
#         baseline_minus = np.mean(alignedevents_minus[:, baselinefirstframe:baselinelastframe, :], axis=1)  # Only average across time

#         alignedevents_plus_corrected = alignedevents_plus - baseline_plus[:, np.newaxis, :]  # Subtract each trial's own baseline
#         alignedevents_minus_corrected = alignedevents_minus - baseline_minus[:, np.newaxis, :]  # Subtract each trial's own baseline

        # Calculate delta lick rates (frame-based)
        delta_licks_plus = np.zeros_like(aligned_licks_plus)
        delta_licks_minus = np.zeros_like(aligned_licks_minus)
        
        for trial in range(aligned_licks_plus.shape[0]):
            baseline_licks_plus = np.mean(aligned_licks_plus[trial, baselinefirstframe:baselinelastframe])
            delta_licks_plus[trial] = aligned_licks_plus[trial] / (1 / averagedframerate) - baseline_licks_plus
        
        for trial in range(aligned_licks_minus.shape[0]):
            baseline_licks_minus = np.mean(aligned_licks_minus[trial, baselinefirstframe:baselinelastframe])
            delta_licks_minus[trial] = aligned_licks_minus[trial] / (1 / averagedframerate) - baseline_licks_minus
        
        # NEW: Load timestamp-based delta rates if available
        ts_delta_rates_plus = np.array([])
        ts_delta_rates_minus = np.array([])
        
        ts_file_plus = os.path.join(basedir, day, animal, fov, f'ts_delta_rates_plus_{eventofinterest}.npy')
        ts_file_minus = os.path.join(basedir, day, animal, fov, f'ts_delta_rates_minus_{eventofinterest}.npy')
        
        if os.path.exists(ts_file_plus):
            ts_delta_rates_plus = np.load(ts_file_plus)
        
        if os.path.exists(ts_file_minus):
            ts_delta_rates_minus = np.load(ts_file_minus)
        
        # Get phase config
        phase_config = folder_config[day]
        
        # Process data
        for cell_idx in range(alignedevents_plus.shape[0]):
            cell_base = {
                'cell_id': cell_id_counter,
                'cell_number': cell_idx + 1,
                'roi_number': roi_numbers.get(cell_idx, np.nan),
                'day': day,
                'animal': animal,
                'fov': fov,
                'auc_plus': aucpvals_plus[cell_idx, 0],
                'pval_plus': aucpvals_plus[cell_idx, 1],
                'auc_minus': aucpvals_minus[cell_idx, 0],
                'pval_minus': aucpvals_minus[cell_idx, 1],
                'experimental_phase': day
            }
            
            # Add CS+ trials
            for trial_idx in range(alignedevents_plus.shape[2]):
                timestamp = trial_timestamps_plus[trial_idx]
                trial_data = cell_base.copy()
                trial_data.update({
                    'trial_id': f"{animal}_{fov}_CSplus_trial{trial_idx}",
                    'cs_type': 'CS+',
                    'timestamp': timestamp,
                    'reward_status': reward_status.get(timestamp, 'unknown'),
                    'expected_reward': phase_config['cs_plus_reward']
                })
                
                # Add neural data and frame-based lick data
                trial_data.update({f'fluor_plus_frame_{frame}': alignedevents_plus_corrected[cell_idx, frame, trial_idx] 
                                  for frame in range(alignedevents_plus.shape[1])})
                trial_data.update({f'delta_licks_frame_{frame}': delta_licks_plus[trial_idx, frame] 
                                  for frame in range(delta_licks_plus.shape[1])})
                
                # NEW: Add timestamp-based delta rates if available
                if len(ts_delta_rates_plus) > trial_idx:
                    trial_data['ts_delta_rate'] = float(ts_delta_rates_plus[trial_idx])
                    # Set high licking flag based on timestamp data
                    trial_data['ts_high_licking'] = 1 if ts_delta_rates_plus[trial_idx] > 1.5 else 0
                
                master_data.append(trial_data)
            
            # Add CS- trials
            for trial_idx in range(alignedevents_minus.shape[2]):
                timestamp = trial_timestamps_minus[trial_idx]
                trial_data = cell_base.copy()
                trial_data.update({
                    'trial_id': f"{animal}_{fov}_CSminus_trial{trial_idx}",
                    'cs_type': 'CS-',
                    'timestamp': timestamp,
                    'reward_status': reward_status.get(timestamp, 'unknown'),
                    'expected_reward': phase_config['cs_minus_reward']
                })
                
                # Add neural data and frame-based lick data
                trial_data.update({f'fluor_minus_frame_{frame}': alignedevents_minus_corrected[cell_idx, frame, trial_idx] 
                                  for frame in range(alignedevents_minus.shape[1])})
                trial_data.update({f'delta_licks_frame_{frame}': delta_licks_minus[trial_idx, frame] 
                                  for frame in range(delta_licks_minus.shape[1])})
                
                # NEW: Add timestamp-based delta rates if available
                if len(ts_delta_rates_minus) > trial_idx:
                    trial_data['ts_delta_rate'] = float(ts_delta_rates_minus[trial_idx])
                    # Set high licking flag based on timestamp data
                    trial_data['ts_high_licking'] = 1 if ts_delta_rates_minus[trial_idx] > 1.5 else 0
                
                master_data.append(trial_data)
            
            cell_id_counter += 1
    
    # Create master DataFrame
    df_master = pd.DataFrame(master_data)
    
    # Add reward expectation met column
    df_master['reward_expectation_met'] = df_master.apply(
        lambda row: row['reward_status'] == 'rewarded' if row['expected_reward'] == True else
                   row['reward_status'] == 'unrewarded' if row['expected_reward'] == False else
                   True,  # For variable reward expectation
        axis=1
    )
    
    # NEW: Calculate frame-based high licking classification
    # Define the event and baseline windows in frames
    event_window = slice(pre_window_size, infusionframe)
    baseline_window = slice(baselinefirstframe, baselinelastframe)
    
    # Get all delta lick frame columns
    delta_lick_cols = [col for col in df_master.columns if col.startswith('delta_licks_frame_')]
    
    # Group by trial and calculate lick rates
    def classify_high_licking(group):
        # Extract lick data for this trial
        lick_data = group[delta_lick_cols].values[0]  # Take first row (all rows in group are same trial)
        
        # Calculate event and baseline rates
        event_rate = np.sum(lick_data[event_window]) / ((infusionframe - pre_window_size) / averagedframerate)
        baseline_rate = np.sum(lick_data[baseline_window]) / ((baselinelastframe - baselinefirstframe) / averagedframerate)
        
        # Calculate delta rate and classify
        delta_rate = event_rate - baseline_rate
        is_high = delta_rate > 1.5
        
        # Set values for all rows in the group
        group['frame_delta_rate'] = delta_rate
        group['high_licking'] = int(is_high)
        
        return group
    
    # Apply the function to each trial group
    df_master = df_master.groupby(['animal', 'fov', 'trial_id']).apply(classify_high_licking)
    
    # Fill NaN values in timestamp columns if they exist
    if 'ts_delta_rate' in df_master.columns and 'ts_high_licking' in df_master.columns:
        # For trials without timestamp data, use frame-based values
        df_master['ts_delta_rate'].fillna(df_master['frame_delta_rate'], inplace=True)
        df_master['ts_high_licking'].fillna(df_master['high_licking'], inplace=True)
        
        # Calculate disagreement
        mismatch_count = (df_master['high_licking'] != df_master['ts_high_licking']).sum()
        total_trials = len(df_master.drop_duplicates(['animal', 'fov', 'trial_id']))
        mismatch_percent = mismatch_count / total_trials * 100
        
        print(f"\nMismatch between frame-based and timestamp-based classifications:")
        print(f"{mismatch_count} trials out of {total_trials} ({mismatch_percent:.1f}%)")
    
    print("Master DataFrame Shape:", df_master.shape)
    print("\nSample of Master DataFrame:")
    display(df_master.head())
    
    return df_master

# Usage
df_master = create_efficient_master_dataframe(basedir, days, eventofinterest, separation_requirement, baselinefirstframe, baselinelastframe, averagedframerate)

# Adding data to df_master
## dump the index that was used to get ms-based lick detection and thresholding
## adding literal trial order (real order of CS+/CS- presentation)
## adding a simple trial counter (a within CS type trial counter)
## adding trial transition type hot encoding (eg., if current trial was CS+, was the last trial CS+ if so set repeated_CS+ to 1)
## adding a series of columns that detail the spatial position of the ROIs
## adding a sex column

In [ ]:
### DUMP THE INDEX NEEDED TO GET THE LICK DETECTION CORRECT BASED OFF OF MS #### 


# Drop the problematic index levels before resetting
df_master = df_master.reset_index(drop=True)

# Drop duplicate columns if they exist
df_master = df_master.loc[:, ~df_master.columns.duplicated()]

# Verify the changes
print("Final DataFrame Structure:")
print(df_master.info())


### Add literal trial_order column
def assign_trial_order(df_master):
    try:
        # Create the trial_order column
        df_master['trial_order'] = (
            df_master.groupby(['cell_id', 'fov'])['timestamp']
            .rank(method='first')
            .astype(int)
        )
        return df_master

    except Exception as e:
        logger.error(f"Error in assigning trial order: {str(e)}")
        raise

# Apply the function
df_master = assign_trial_order(df_master)

### Add within CS type trial column
# Add new trial column
df_master['trial'] = df_master['trial_id'].str.extract(r'trial(\d+)').astype(int)


### Adding in Location of the cells in the ROI
def add_spatial_information_to_df(df_master, basedir):
    """
    Add spatial information columns to df_master using ROI bounding box properties.
    """
    # Create new columns
    df_master['roi_center_x'] = np.nan
    df_master['roi_center_y'] = np.nan
    df_master['roi_left'] = np.nan
    df_master['roi_right'] = np.nan
    df_master['roi_top'] = np.nan
    df_master['roi_bottom'] = np.nan
    df_master['roi_width'] = np.nan
    df_master['roi_height'] = np.nan
    
    # Get unique combinations of day, animal, and FOV
    unique_combinations = df_master[['day', 'animal', 'fov']].drop_duplicates()
    
    for _, row in unique_combinations.iterrows():
        day, animal, fov = row['day'], row['animal'], row['fov']
        
        # Load ROI file
        roi_folder = os.path.join(basedir, day, animal, fov)
        roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
        
        if roi_files:
            roi_path = os.path.join(roi_folder, roi_files[0])
            rois = roifile.ImagejRoi.fromfile(roi_path)
            
            # Update df_master for this day/animal/FOV combination
            mask = ((df_master['day'] == day) & 
                   (df_master['animal'] == animal) & 
                   (df_master['fov'] == fov))
            
            # Map ROI numbers (1-based) to array indices (0-based)
            for cell_number in range(1, len(rois) + 1):
                roi = rois[cell_number-1]
                cell_mask = mask & (df_master['cell_number'] == cell_number)
                
                # Calculate center from bounding box
                center_x = (roi.left + roi.right) / 2
                center_y = (roi.top + roi.bottom) / 2
                width = roi.right - roi.left
                height = roi.bottom - roi.top
                
                # Update all spatial information
                df_master.loc[cell_mask, 'roi_center_x'] = center_x
                df_master.loc[cell_mask, 'roi_center_y'] = center_y
                df_master.loc[cell_mask, 'roi_left'] = roi.left
                df_master.loc[cell_mask, 'roi_right'] = roi.right
                df_master.loc[cell_mask, 'roi_top'] = roi.top
                df_master.loc[cell_mask, 'roi_bottom'] = roi.bottom
                df_master.loc[cell_mask, 'roi_width'] = width
                df_master.loc[cell_mask, 'roi_height'] = height
    
    # Verify the results
    print("\nSpatial information added to df_master:")
    print(f"Number of rows with spatial data: {df_master['roi_center_x'].notna().sum()}")
    print("\nSample of updated df_master:")
    cols_to_show = ['day', 'animal', 'fov', 'cell_number', 
                    'roi_center_x', 'roi_center_y', 
                    'roi_width', 'roi_height']
    print(df_master[cols_to_show].head())
    
    return df_master

# Usage
df_master = add_spatial_information_to_df(df_master, basedir)


### Adding In Biological Sex As A Column in Our Data Frame
def add_sex_column(df):
    # Define the mapping for sex based on your provided names
    sex_mapping = {
        'Astro1': 'female',
        'Astro2': 'female',
        'Astro3': 'male',
        'Astro4': 'female',
        'Astro5': 'male',
        'Astro15': 'male',
        'Astro17': 'male',
        'Astro20': 'male',
        'Astro33': 'male',
        'Astro34': 'male',
        'Astro35': 'female',
        'Astro37': 'female',
        'Astro38': 'female',
        'Astro39': 'male',
        'Astro40': 'male',
        'Gi2': 'male',
        'Gi5': 'female',
        'Gi7': 'female',
        'Gi8': 'male',
        'mCh1': 'male',
    }

    # Apply the mapping to create the new column
    df['sex'] = df['animal'].map(sex_mapping)
    
    # Check the updated DataFrame
    print("\nUpdated DataFrame with sex Column:")
    display(df.head())
    
    return df
df_master = add_sex_column(df_master)


# *SEPARATE THE REWARDED AND UNREWARDED TRIALS AND RECALCULATE AUROC*

In [ ]:
def create_omission_dataframe(df_master):
    """
    Create df_master_omission_jp with additional columns for sucrose omission analysis.
    
    Parameters:
    -----------
    df_master : pandas DataFrame
        Original master dataframe
        
    Returns:
    --------
    df_master_omission_jp : pandas DataFrame
        New dataframe with CS+ Reward Status and separate AUC columns for rewarded/unrewarded
    """
    # Create a copy of the original dataframe
    df_master_omission_jp = df_master.copy()
    
    # Initialize new columns
    df_master_omission_jp['CS+ Reward Status'] = np.nan
    df_master_omission_jp['auROC_SucOmit_rewarded'] = np.nan
    df_master_omission_jp['pval_SucOmit_rewarded'] = np.nan
    df_master_omission_jp['auROC_SucOmit_unrewarded'] = np.nan
    df_master_omission_jp['pval_SucOmit_unrewarded'] = np.nan
    
    # Populate CS+ Reward Status column
    # For 2_SucOmit day, CS+ trials
    sucrose_omit_mask = (df_master_omission_jp['day'] == '2_SucOmit') & (df_master_omission_jp['cs_type'] == 'CS+')
    
    # Set binary values based on reward_status
    rewarded_mask = sucrose_omit_mask & (df_master_omission_jp['reward_status'] == 'rewarded')
    unrewarded_mask = sucrose_omit_mask & (df_master_omission_jp['reward_status'] == 'unrewarded')
    
    df_master_omission_jp.loc[rewarded_mask, 'CS+ Reward Status'] = 1
    df_master_omission_jp.loc[unrewarded_mask, 'CS+ Reward Status'] = 0
    
    print(f"Created df_master_omission_jp with shape: {df_master_omission_jp.shape}")
    print(f"\n2_SucOmit CS+ trial breakdown:")
    sucrose_omit_cs_plus = df_master_omission_jp[sucrose_omit_mask]
    print(f"  Total CS+ trials: {len(sucrose_omit_cs_plus.drop_duplicates('trial_id'))}")
    print(f"  Rewarded (1): {(df_master_omission_jp['CS+ Reward Status'] == 1).sum()} trial-cell pairs")
    print(f"  Unrewarded (0): {(df_master_omission_jp['CS+ Reward Status'] == 0).sum()} trial-cell pairs")
    
    return df_master_omission_jp


def calculate_sucrose_omission_auc(df_master_omission_jp, baselinefirstframe, baselinelastframe, 
                                   aucfirstframe, auclastframe):
    """
    Calculate AUC scores for rewarded and unrewarded CS+ trials separately during 2_SucOmit day.
    Each cell gets two AUC values: one for rewarded trials, one for unrewarded trials.
    
    Parameters:
    -----------
    df_master_omission_jp : pandas DataFrame
        Omission dataframe with CS+ Reward Status column
    baselinefirstframe : int
        First frame of baseline period
    baselinelastframe : int
        Last frame of baseline period
    aucfirstframe : int
        First frame for AUC calculation
    auclastframe : int
        Last frame for AUC calculation
        
    Returns:
    --------
    df_master_omission_jp : pandas DataFrame
        Updated dataframe with calculated auROC and pval values for both conditions
    """
    print("Calculating AUC scores for sucrose omission trials...")
    
    # Filter for 2_SucOmit day and CS+ trials only
    sucrose_omit_data = df_master_omission_jp[
        (df_master_omission_jp['day'] == '2_SucOmit') & 
        (df_master_omission_jp['cs_type'] == 'CS+')
    ]
    
    # Get unique cell IDs for this day
    cell_ids = sucrose_omit_data['cell_id'].unique()
    
    print(f"Processing {len(cell_ids)} unique cells from 2_SucOmit day")
    
    cells_processed = 0
    cells_with_rewarded = 0
    cells_with_unrewarded = 0
    cells_with_both = 0
    
    for cell_id in cell_ids:
        cells_processed += 1
        if cells_processed % 50 == 0 or cells_processed == 1 or cells_processed == len(cell_ids):
            print(f"Processing cell {cells_processed}/{len(cell_ids)}")
        
        # Get trials for this cell
        cell_data = sucrose_omit_data[sucrose_omit_data['cell_id'] == cell_id]
        
        # Separate rewarded and unrewarded trials
        rewarded_trials = cell_data[cell_data['CS+ Reward Status'] == 1]
        unrewarded_trials = cell_data[cell_data['CS+ Reward Status'] == 0]
        
        # Extract fluorescence columns (CS+ uses fluor_plus_frame_X columns)
        fluor_columns = [col for col in cell_data.columns if col.startswith('fluor_plus_frame_')]
        
        if not fluor_columns:
            print(f"Warning: No fluor_plus_frame columns for cell ID {cell_id}. Skipping.")
            continue
        
        # Calculate AUC for REWARDED trials (event vs baseline)
        if len(rewarded_trials) > 0:
            cells_with_rewarded += 1
            
            # Get fluorescence data for rewarded trials
            rewarded_fluor = rewarded_trials[fluor_columns].values
            
            # Calculate baseline and event periods - EXACTLY like original code
            rewarded_baseline = np.mean(rewarded_fluor[:, baselinefirstframe:baselinelastframe], axis=1)
            rewarded_event = np.mean(rewarded_fluor[:, aucfirstframe:auclastframe], axis=1)
            
            # Calculate AUC for rewarded trials (event vs baseline)
            auc_rewarded, pval_rewarded = calculate_auROC(rewarded_event, rewarded_baseline)
            
            # Update dataframe for ALL trials of this cell (both rewarded and unrewarded get the same values)
            cell_mask = (
                (df_master_omission_jp['cell_id'] == cell_id) & 
                (df_master_omission_jp['day'] == '2_SucOmit') &
                (df_master_omission_jp['cs_type'] == 'CS+')
            )
            
            df_master_omission_jp.loc[cell_mask, 'auROC_SucOmit_rewarded'] = auc_rewarded
            df_master_omission_jp.loc[cell_mask, 'pval_SucOmit_rewarded'] = pval_rewarded
        
        # Calculate AUC for UNREWARDED trials (event vs baseline)
        if len(unrewarded_trials) > 0:
            cells_with_unrewarded += 1
            
            # Get fluorescence data for unrewarded trials
            unrewarded_fluor = unrewarded_trials[fluor_columns].values
            
            # Calculate baseline and event periods - EXACTLY like original code
            unrewarded_baseline = np.mean(unrewarded_fluor[:, baselinefirstframe:baselinelastframe], axis=1)
            unrewarded_event = np.mean(unrewarded_fluor[:, aucfirstframe:auclastframe], axis=1)
            
            # Calculate AUC for unrewarded trials (event vs baseline)
            auc_unrewarded, pval_unrewarded = calculate_auROC(unrewarded_event, unrewarded_baseline)
            
            # Update dataframe for ALL trials of this cell
            cell_mask = (
                (df_master_omission_jp['cell_id'] == cell_id) & 
                (df_master_omission_jp['day'] == '2_SucOmit') &
                (df_master_omission_jp['cs_type'] == 'CS+')
            )
            
            df_master_omission_jp.loc[cell_mask, 'auROC_SucOmit_unrewarded'] = auc_unrewarded
            df_master_omission_jp.loc[cell_mask, 'pval_SucOmit_unrewarded'] = pval_unrewarded
        
        if len(rewarded_trials) > 0 and len(unrewarded_trials) > 0:
            cells_with_both += 1
    
    # Apply Benjamini-Hochberg correction if specified
    if bh_correction == 'yes':
        print("\nApplying Benjamini-Hochberg correction to omission p-values...")
        
        # Get unique cells from 2_SucOmit day
        unique_cells = df_master_omission_jp[
            (df_master_omission_jp['day'] == '2_SucOmit') & 
            (df_master_omission_jp['cs_type'] == 'CS+')
        ].drop_duplicates('cell_id')
        
        # Correct p-values for rewarded trials
        pvals_rewarded = unique_cells['pval_SucOmit_rewarded'].dropna().values
        if len(pvals_rewarded) > 0:
            corrected_pvals_rewarded = Benjamini_Hochberg_correction(pvals_rewarded, alpha=alphalevel)
            valid_cells_rewarded = unique_cells.dropna(subset=['pval_SucOmit_rewarded'])
            correction_map_rewarded = dict(zip(valid_cells_rewarded['cell_id'].values, corrected_pvals_rewarded))
            
            for cell_id, corrected_pval in correction_map_rewarded.items():
                cell_mask = (
                    (df_master_omission_jp['cell_id'] == cell_id) & 
                    (df_master_omission_jp['day'] == '2_SucOmit')
                )
                df_master_omission_jp.loc[cell_mask, 'pval_SucOmit_rewarded'] = corrected_pval
        
        # Correct p-values for unrewarded trials
        pvals_unrewarded = unique_cells['pval_SucOmit_unrewarded'].dropna().values
        if len(pvals_unrewarded) > 0:
            corrected_pvals_unrewarded = Benjamini_Hochberg_correction(pvals_unrewarded, alpha=alphalevel)
            valid_cells_unrewarded = unique_cells.dropna(subset=['pval_SucOmit_unrewarded'])
            correction_map_unrewarded = dict(zip(valid_cells_unrewarded['cell_id'].values, corrected_pvals_unrewarded))
            
            for cell_id, corrected_pval in correction_map_unrewarded.items():
                cell_mask = (
                    (df_master_omission_jp['cell_id'] == cell_id) & 
                    (df_master_omission_jp['day'] == '2_SucOmit')
                )
                df_master_omission_jp.loc[cell_mask, 'pval_SucOmit_unrewarded'] = corrected_pval
    
    # Add significance flags
    df_master_omission_jp['significant_SucOmit_rewarded'] = df_master_omission_jp['pval_SucOmit_rewarded'] < alphalevel
    df_master_omission_jp['significant_SucOmit_unrewarded'] = df_master_omission_jp['pval_SucOmit_unrewarded'] < alphalevel
    
    # Print summary statistics
    print(f"\n=== Sucrose Omission Analysis Summary ===")
    print(f"Total cells processed: {cells_processed}")
    print(f"Cells with rewarded trials: {cells_with_rewarded}")
    print(f"Cells with unrewarded trials: {cells_with_unrewarded}")
    print(f"Cells with both trial types: {cells_with_both}")
    
    # Get statistics for cells with valid AUC values
    valid_cells = df_master_omission_jp[
        (df_master_omission_jp['day'] == '2_SucOmit') & 
        (df_master_omission_jp['cs_type'] == 'CS+')
    ].drop_duplicates('cell_id')
    
    # Stats for rewarded trials
    valid_rewarded = valid_cells.dropna(subset=['auROC_SucOmit_rewarded'])
    if len(valid_rewarded) > 0:
        sig_rewarded = valid_rewarded['significant_SucOmit_rewarded'].sum()
        mean_auc_rewarded = valid_rewarded['auROC_SucOmit_rewarded'].mean()
        
        print(f"\n=== REWARDED CS+ Trials ===")
        print(f"Significant responders: {sig_rewarded}/{len(valid_rewarded)} ({sig_rewarded/len(valid_rewarded)*100:.1f}%)")
        print(f"Mean AUC: {mean_auc_rewarded:.3f}")
        
        excited_rewarded = valid_rewarded[valid_rewarded['auROC_SucOmit_rewarded'] > 0]
        inhibited_rewarded = valid_rewarded[valid_rewarded['auROC_SucOmit_rewarded'] < 0]
        print(f"Excited cells: {len(excited_rewarded)} ({len(excited_rewarded)/len(valid_rewarded)*100:.1f}%)")
        print(f"Inhibited cells: {len(inhibited_rewarded)} ({len(inhibited_rewarded)/len(valid_rewarded)*100:.1f}%)")
    
    # Stats for unrewarded trials
    valid_unrewarded = valid_cells.dropna(subset=['auROC_SucOmit_unrewarded'])
    if len(valid_unrewarded) > 0:
        sig_unrewarded = valid_unrewarded['significant_SucOmit_unrewarded'].sum()
        mean_auc_unrewarded = valid_unrewarded['auROC_SucOmit_unrewarded'].mean()
        
        print(f"\n=== UNREWARDED CS+ Trials (Omission) ===")
        print(f"Significant responders: {sig_unrewarded}/{len(valid_unrewarded)} ({sig_unrewarded/len(valid_unrewarded)*100:.1f}%)")
        print(f"Mean AUC: {mean_auc_unrewarded:.3f}")
        
        excited_unrewarded = valid_unrewarded[valid_unrewarded['auROC_SucOmit_unrewarded'] > 0]
        inhibited_unrewarded = valid_unrewarded[valid_unrewarded['auROC_SucOmit_unrewarded'] < 0]
        print(f"Excited cells: {len(excited_unrewarded)} ({len(excited_unrewarded)/len(valid_unrewarded)*100:.1f}%)")
        print(f"Inhibited cells: {len(inhibited_unrewarded)} ({len(inhibited_unrewarded)/len(valid_unrewarded)*100:.1f}%)")
    
    return df_master_omission_jp


# Create the omission dataframe
df_master_omission_jp = create_omission_dataframe(df_master)

# Calculate AUC scores for sucrose omission
df_master_omission_jp = calculate_sucrose_omission_auc(
    df_master_omission_jp=df_master_omission_jp,
    baselinefirstframe=baselinefirstframe,
    baselinelastframe=baselinelastframe,
    aucfirstframe=aucfirstframe,
    auclastframe=auclastframe
)

# Display sample results
print("\n=== Sample of Updated Dataframe ===")
sample_cols = ['animal', 'fov', 'cell_id', 'cell_number', 'trial_id', 'cs_type', 'day',
               'reward_status', 'CS+ Reward Status', 
               'auROC_SucOmit_rewarded', 'pval_SucOmit_rewarded',
               'auROC_SucOmit_unrewarded', 'pval_SucOmit_unrewarded']

# Show unique cells from 2_SucOmit CS+ trials
sample_data = df_master_omission_jp[
    (df_master_omission_jp['day'] == '2_SucOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS+')
][sample_cols].drop_duplicates('cell_id').head(10)

display(sample_data)

# Save the dataframe
save_path = os.path.join(basedir, 'Results', 
                        f'df_master_omission_jp_{eventofinterest}_{separation_requirement}.csv')
df_master_omission_jp.to_csv(save_path, index=False)
print(f"\nOmission dataframe saved to: {save_path}")

In [ ]:
##USING df_master_omission_jp
# ### This code loads up the CSVs that contain information about how to match cell_number (identical to roi_number generated above) in df_master across days 
# # The code adds new column "tracked_roi_number" to do this. 
# # An important consideration is that this number is approx 1-50 for each FOV, meaning the same number can be present in multiple FOVs. 
# # We will deal with this using a counter in the next cell
import os
import pandas as pd
import numpy as np

def update_tracked_roi_numbers_by_cell_number(df_master_omission_jp, basedir):
    """
    Update 'tracked_roi_number' in df_master_omission_jp by reading matching cell pairs from CellTracking CSV files.
    """
    # Reset index to ensure all columns are accessible
    df_master_omission_jp = df_master_omission_jp.reset_index(drop=True)
    
    cell_tracking_dir = os.path.join(basedir, 'CellTracking')
    tracking_folders = [
        ('Early-Late', ['0_Early', '1_Late']),
        ('Late-Suc-Cue', ['1_Late', '2_SucOmit', '3_CueOmit']),
        ('Late-EarlyRev-LateRev', ['1_Late', '4_RevEarly', '5_RevLate'])
    ]

    if 'tracked_roi_number' not in df_master_omission_jp.columns:
        df_master_omission_jp['tracked_roi_number'] = np.nan

    tracking_dict = {}

    for tracking_type, expected_days in tracking_folders:
        folder_path = os.path.join(cell_tracking_dir, tracking_type)
        if not os.path.exists(folder_path):
            print(f"Folder {tracking_type} not found. Skipping...")
            continue

        for csv_file in os.listdir(folder_path):
            if not csv_file.endswith('.csv'):
                continue

            fov = csv_file.split('_CellTracking_')[0]
            csv_path = os.path.join(folder_path, csv_file)
            print(f"Processing file: {csv_file}")

            try:
                tracking_df = pd.read_csv(csv_path, header=None, skip_blank_lines=True).dropna(how='all', axis=1)
                present_days = expected_days[:len(tracking_df.columns)]
                tracking_df.columns = present_days

                if fov not in tracking_dict:
                    tracking_dict[fov] = {}

                for _, row in tracking_df.iterrows():
                    late_cell = row['1_Late'] if '1_Late' in row else None
                    if pd.notna(late_cell):
                        late_cell = int(late_cell)
                        if late_cell not in tracking_dict[fov]:
                            tracking_dict[fov][late_cell] = {}
                        
                        for day in present_days:
                            if pd.notna(row[day]):
                                tracking_dict[fov][late_cell][day] = int(row[day])

            except Exception as e:
                print(f"Error processing file {csv_file}:")
                print(str(e))
                continue

    # Update df_master_omission_jp with tracked_roi_numbers
    for fov, fov_data in tracking_dict.items():
        for late_cell, day_data in fov_data.items():
            for day, cell_number in day_data.items():
                mask = (df_master_omission_jp['fov'] == fov) & \
                       (df_master_omission_jp['day'] == day) & \
                       (df_master_omission_jp['cell_number'] == cell_number)
                df_master_omission_jp.loc[mask, 'tracked_roi_number'] = late_cell
    
    return df_master_omission_jp

# Usage
if __name__ == "__main__":
    print("Starting cell tracking process for omission dataframe...")
    df_master_omission_jp = update_tracked_roi_numbers_by_cell_number(df_master_omission_jp, basedir)



import pandas as pd
import os

def verify_tracking_across_days(df, days, name):
    """
    Verify how many cells maintain the same tracked_roi_counter across specified days.
    """
    # Filter for the days we want to check
    df_subset = df[df['day'].isin(days)]
    
    # Count how many days each cell appears in
    tracking_counts = df_subset.groupby(['fov', 'tracked_roi_counter'])['day'].nunique()
    
    # Count cells that appear in all days
    cells_tracked_all_days = len(tracking_counts[tracking_counts == len(days)])
    
    print(f"\n{name} Tracking Stats:")
    print(f"Days checked: {days}")
    print(f"Cells tracked across all days: {cells_tracked_all_days}")

def create_tracked_dataframe(df_master_omission_jp, days, name):
    """
    Create tracked dataframe and verify tracking.
    """
    # Reset index first
    df_master_omission_jp = df_master_omission_jp.reset_index(drop=True)
    
    # Filter rows based on specified days and non-null tracked_roi_number
    df_tracked = df_master_omission_jp[df_master_omission_jp['day'].isin(days) & df_master_omission_jp['tracked_roi_number'].notna()].copy()
    
    # Sort the dataframe
    df_tracked = df_tracked.sort_values(['animal', 'fov', 'tracked_roi_number', 'day'])
    
    # Create the counter for unique combinations of fov and tracked_roi_number
    df_tracked['tracked_roi_counter'] = df_tracked.groupby(['fov', 'tracked_roi_number']).ngroup()
    
    # Basic stats
    print(f"\nCreated {name}:")
    print(f"Shape: {df_tracked.shape}")
    print(f"Unique animals: {df_tracked['animal'].nunique()}")
    print(f"Unique FOVs: {df_tracked['fov'].nunique()}")
    print(f"Unique tracked ROIs: {df_tracked['tracked_roi_number'].nunique()}")
    
    # Verify tracking across days
    verify_tracking_across_days(df_tracked, days, name)
    
    return df_tracked

def add_tracked_roi_counter(df_master_omission_jp):
    """
    Add tracked_roi_counter to df_master_omission_jp and verify tracking across day combinations.
    """
    # Reset index to ensure all columns are accessible
    df_master_omission_jp = df_master_omission_jp.reset_index(drop=True)
    
    # Sort the dataframe
    df_master_omission_jp_sorted = df_master_omission_jp.sort_values(['animal', 'fov', 'tracked_roi_number', 'day'])
    
    # Create the counter for unique combinations of fov and tracked_roi_number
    df_master_omission_jp_sorted['tracked_roi_counter'] = df_master_omission_jp_sorted.groupby(
        ['animal', 'fov', 'tracked_roi_number']
    ).ngroup()
    
    # Define day combinations to check
    day_combinations = [
        ('Early-Late', ['0_Early', '1_Late']),
        ('Late-Suc-Cue', ['1_Late', '2_SucOmit', '3_CueOmit']),
        ('Late-EarlyRev-LateRev', ['1_Late', '4_RevEarly', '5_RevLate'])
    ]
    
    # Print basic stats
    print("Updated df_master_omission_jp:")
    print(f"Shape: {df_master_omission_jp_sorted.shape}")
    print(f"Unique animals: {df_master_omission_jp_sorted['animal'].nunique()}")
    print(f"Unique FOVs: {df_master_omission_jp_sorted['fov'].nunique()}")
    
    # Verify each combination
    for name, days in day_combinations:
        verify_tracking_across_days(df_master_omission_jp_sorted, days, name)
    
    print(f"\nSample of updated df_master_omission_jp:")
    display(df_master_omission_jp_sorted[['day', 'animal', 'fov', 'cell_number', 
                             'tracked_roi_number', 'tracked_roi_counter']].sample(5))
    
    return df_master_omission_jp_sorted

# Usage
df_master_omission_jp = add_tracked_roi_counter(df_master_omission_jp)

# **can just skip to here if you dont want to make the omission DF**

# Processing the cell tracking CSVs
## add tracked_roi_number (matches ROI numbers across days)
## and tracked_roi_counter to df_master (an incremented counter that allows these data to be selected and plotted)

In [ ]:
### This code loads up the CSVs that contain information about how to match cell_number (identical to roi_number generated above) in df_master across days 
# The code adds new column "tracked_roi_number" to do this. 
# An important consideration is that this number is approx 1-50 for each FOV, meaning the same number can be present in multiple FOVs. 
# We will deal with this using a counter in the next cell


import os
import pandas as pd
import numpy as np

def update_tracked_roi_numbers_by_cell_number(df_master, basedir):
    """
    Update 'tracked_roi_number' in df_master by reading matching cell pairs from CellTracking CSV files.
    """
    cell_tracking_dir = os.path.join(basedir, 'CellTracking')
    tracking_folders = [
        ('Early-Late', ['0_Early', '1_Late']),
        ('Late-Suc-Cue', ['1_Late', '2_SucOmit', '3_CueOmit']),
        ('Late-EarlyRev-LateRev', ['1_Late', '4_RevEarly', '5_RevLate'])
    ]

    if 'tracked_roi_number' not in df_master.columns:
        df_master['tracked_roi_number'] = np.nan

    tracking_dict = {}

    for tracking_type, expected_days in tracking_folders:
        folder_path = os.path.join(cell_tracking_dir, tracking_type)
        if not os.path.exists(folder_path):
            print(f"Folder {tracking_type} not found. Skipping...")
            continue

        for csv_file in os.listdir(folder_path):
            if not csv_file.endswith('.csv'):
                continue

            fov = csv_file.split('_CellTracking_')[0]
            csv_path = os.path.join(folder_path, csv_file)
            print(f"Processing file: {csv_file}")

            try:
                tracking_df = pd.read_csv(csv_path, header=None, skip_blank_lines=True).dropna(how='all', axis=1)
                present_days = expected_days[:len(tracking_df.columns)]
                tracking_df.columns = present_days

                if fov not in tracking_dict:
                    tracking_dict[fov] = {}

                for _, row in tracking_df.iterrows():
                    late_cell = row['1_Late'] if '1_Late' in row else None
                    if pd.notna(late_cell):
                        late_cell = int(late_cell)
                        if late_cell not in tracking_dict[fov]:
                            tracking_dict[fov][late_cell] = {}
                        
                        for day in present_days:
                            if pd.notna(row[day]):
                                tracking_dict[fov][late_cell][day] = int(row[day])

            except Exception as e:
                print(f"Error processing file {csv_file}:")
                print(str(e))
                continue

    # Update df_master with tracked_roi_numbers
    for fov, fov_data in tracking_dict.items():
        for late_cell, day_data in fov_data.items():
            for day, cell_number in day_data.items():
                mask = (df_master['fov'] == fov) & \
                       (df_master['day'] == day) & \
                       (df_master['cell_number'] == cell_number)
                df_master.loc[mask, 'tracked_roi_number'] = late_cell
    
    return df_master

# Usage
if __name__ == "__main__":
    print("Starting cell tracking process...")
    df_master = update_tracked_roi_numbers_by_cell_number(df_master, basedir)



# To plot and analyze tracked cells we add in a new index that allows us to provide the tracked cells with the same number across days, without this index repeating across FOVs. 
## The logic used here can be explained as finding unique combinations of cell_number and FOV that have the same tracked_roi_number across multiple days, then increasing this counter when moving to the next fov.
## This is called "tracked_roi_counter" and of course requires that rows (cell / trial data) in df_master have a value for "tracked_roi_number" 


import pandas as pd
import os

def verify_tracking_across_days(df, days, name):
    """
    Verify how many cells maintain the same tracked_roi_counter across specified days.
    """
    # Filter for the days we want to check
    df_subset = df[df['day'].isin(days)]
    
    # Count how many days each cell appears in
    tracking_counts = df_subset.groupby(['fov', 'tracked_roi_counter'])['day'].nunique()
    
    # Count cells that appear in all days
    cells_tracked_all_days = len(tracking_counts[tracking_counts == len(days)])
    
    print(f"\n{name} Tracking Stats:")
    print(f"Days checked: {days}")
    print(f"Cells tracked across all days: {cells_tracked_all_days}")

def create_tracked_dataframe(df_master, days, name):
    """
    Create tracked dataframe and verify tracking.
    """
    # Filter rows based on specified days and non-null tracked_roi_number
    df_tracked = df_master[df_master['day'].isin(days) & df_master['tracked_roi_number'].notna()].copy()
    
    # Sort the dataframe
    df_tracked = df_tracked.sort_values(['animal', 'fov', 'tracked_roi_number', 'day'])
    
    # Create the counter for unique combinations of fov and tracked_roi_number
    df_tracked['tracked_roi_counter'] = df_tracked.groupby(['fov', 'tracked_roi_number']).ngroup()
    
    # Basic stats
    print(f"\nCreated {name}:")
    print(f"Shape: {df_tracked.shape}")
    print(f"Unique animals: {df_tracked['animal'].nunique()}")
    print(f"Unique FOVs: {df_tracked['fov'].nunique()}")
    print(f"Unique tracked ROIs: {df_tracked['tracked_roi_number'].nunique()}")
    
    # Verify tracking across days
    verify_tracking_across_days(df_tracked, days, name)
    
    return df_tracked

def add_tracked_roi_counter(df_master):
    """
    Add tracked_roi_counter to df_master and verify tracking across day combinations.
    """
    # Sort the dataframe
    df_master_sorted = df_master.sort_values(['animal', 'fov', 'tracked_roi_number', 'day'])
    
    # Create the counter for unique combinations of fov and tracked_roi_number
    df_master_sorted['tracked_roi_counter'] = df_master_sorted.groupby(
        ['animal', 'fov', 'tracked_roi_number']
    ).ngroup()
    
    # Define day combinations to check
    day_combinations = [
        ('Early-Late', ['0_Early', '1_Late']),
        ('Late-Suc-Cue', ['1_Late', '2_SucOmit', '3_CueOmit']),
        ('Late-EarlyRev-LateRev', ['1_Late', '4_RevEarly', '5_RevLate'])
    ]
    
    # Print basic stats
    print("Updated df_master:")
    print(f"Shape: {df_master_sorted.shape}")
    print(f"Unique animals: {df_master_sorted['animal'].nunique()}")
    print(f"Unique FOVs: {df_master_sorted['fov'].nunique()}")
    
    # Verify each combination
    for name, days in day_combinations:
        verify_tracking_across_days(df_master_sorted, days, name)
    
    print(f"\nSample of updated df_master:")
    display(df_master_sorted[['day', 'animal', 'fov', 'cell_number', 
                             'tracked_roi_number', 'tracked_roi_counter']].sample(5))
    
    return df_master_sorted

# Usage
df_master = add_tracked_roi_counter(df_master)

# Visualizing % Discrimination

# Visualizing Discrimination with all trials
## Here sex can be set to 'male','female' or 'all'

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


def plot_pie_charts_by_sex(df_master, days, alphalevel, basedir, sex_selection='all'):
    numdays = len(days)
    
    # Figure settings
    fig, axs = plt.subplots(2, numdays, figsize=(numdays*2, 4), sharex=True, sharey=False)
    sns.set_style('white')
    colors = ['deeppink', 'gray', 'yellowgreen']  # Updated color scheme
    
    # Track total unique neurons across days
    total_neurons = 0
    
    for d, day in enumerate(sorted(days)):
        if sex_selection == 'all':
            # Get unique cells for this day
            day_data = df_master[df_master['day'] == day].drop_duplicates(subset=['cell_id'])
        else:
            day_data = df_master[
                (df_master['day'] == day) & 
                (df_master['sex'] == sex_selection)
            ].drop_duplicates(subset=['cell_id'])
        
        # CS+ calculations for unique cells
        num_cells = len(day_data)
        total_neurons += num_cells
        
        # CS+ detailed counts
        activated_cells_plus = np.sum((day_data['pval_plus'] <= alphalevel) & (day_data['auc_plus'] >= 0))
        inhibited_cells_plus = np.sum((day_data['pval_plus'] <= alphalevel) & (day_data['auc_plus'] < 0))
        nonresponders_plus = np.sum(day_data['pval_plus'] > alphalevel)
        
        print(f"\nDay {day}:")
        print(f"CS+ - Activated: {activated_cells_plus}, Inhibited: {inhibited_cells_plus}, Non-responders: {nonresponders_plus}")
        
        if num_cells > 0:
            activated_cells_plus_pct = activated_cells_plus / num_cells
            inhibited_cells_plus_pct = inhibited_cells_plus / num_cells
            nonresponders_plus_pct = nonresponders_plus / num_cells
            counts_plus = np.array([activated_cells_plus_pct, nonresponders_plus_pct, inhibited_cells_plus_pct])
            labels_plus = (f"{activated_cells_plus_pct*100:.1f}%", '', f"{inhibited_cells_plus_pct*100:.1f}%")
        else:
            counts_plus = np.array([0, 1, 0])
            labels_plus = ('no data', '', '')
        
        # CS- detailed counts
        activated_cells_minus = np.sum((day_data['pval_minus'] <= alphalevel) & (day_data['auc_minus'] >= 0))
        inhibited_cells_minus = np.sum((day_data['pval_minus'] <= alphalevel) & (day_data['auc_minus'] < 0))
        nonresponders_minus = np.sum(day_data['pval_minus'] > alphalevel)
        
        print(f"CS- - Activated: {activated_cells_minus}, Inhibited: {inhibited_cells_minus}, Non-responders: {nonresponders_minus}")
        
        if num_cells > 0:
            activated_cells_minus_pct = activated_cells_minus / num_cells
            inhibited_cells_minus_pct = inhibited_cells_minus / num_cells
            nonresponders_minus_pct = nonresponders_minus / num_cells
            counts_minus = np.array([activated_cells_minus_pct, nonresponders_minus_pct, inhibited_cells_minus_pct])
            labels_minus = (f"{activated_cells_minus_pct*100:.1f}%", '', f"{inhibited_cells_minus_pct*100:.1f}%")
        else:
            counts_minus = np.array([0, 1, 0])
            labels_minus = ('no data', '', '')
        
        # Plot pie charts
        ax = axs[0, d]
        ax.pie(counts_plus, colors=colors, labels=labels_plus)
        if d == 0:
            ax.set_ylabel('CS+', fontsize=10)
        ax.set_title(day)
        
        ax = axs[1, d]
        ax.pie(counts_minus, colors=colors, labels=labels_minus)
        if d == 0:
            ax.set_ylabel('CS-', fontsize=10)
    
    print(f"\nTotal number of unique neurons across all days: {total_neurons}")
    
    title = f'Population Responses - {sex_selection.capitalize()}'
    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    # # Save the plot
    pdf_filename = os.path.join(basedir, "Results", f'Pop_auROC_SignificantResponders.pdf')
    plt.savefig(pdf_filename)
    print(f"File saved in {pdf_filename}")
    plt.show()


# For all cells (both sexes combined)
plot_pie_charts_by_sex(df_master, days, alphalevel, basedir, sex_selection='all')

# using df_master_omission_jp for the significant responders in 2_SucOmit

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


def plot_omission_pie_charts(df_master_omission_jp, alphalevel, basedir, sex_selection='all'):
    """
    Plot pie charts for 2_SucOmit day showing:
    - CS+ Rewarded trials
    - CS+ Unrewarded (omitted) trials  
    - CS- trials
    """
    
    # Figure settings - 3 columns for the 3 conditions
    fig, axs = plt.subplots(1, 3, figsize=(9, 3))
    sns.set_style('white')
    colors = ['deeppink', 'gray', 'yellowgreen']  # Activated, Non-responders, Inhibited
    
    # Filter for 2_SucOmit day only
    day = '2_SucOmit'
    
    if sex_selection == 'all':
        day_data = df_master_omission_jp[df_master_omission_jp['day'] == day]
    else:
        day_data = df_master_omission_jp[
            (df_master_omission_jp['day'] == day) & 
            (df_master_omission_jp['sex'] == sex_selection)
        ]
    
    print(f"\n=== 2_SucOmit Day Analysis ({sex_selection.capitalize()}) ===")
    print(f"Total rows in 2_SucOmit: {len(day_data)}")
    
    # ===== CS+ REWARDED TRIALS =====
    # Filter for CS+ trials first, then get unique cells
    cs_plus_data = day_data[day_data['cs_type'] == 'CS+']
    cs_plus_rewarded = cs_plus_data.dropna(subset=['auROC_SucOmit_rewarded', 'pval_SucOmit_rewarded']).drop_duplicates(subset=['cell_id'])
    num_cells_rewarded = len(cs_plus_rewarded)
    
    activated_rewarded = np.sum(
        (cs_plus_rewarded['pval_SucOmit_rewarded'] <= alphalevel) & 
        (cs_plus_rewarded['auROC_SucOmit_rewarded'] >= 0)
    )
    inhibited_rewarded = np.sum(
        (cs_plus_rewarded['pval_SucOmit_rewarded'] <= alphalevel) & 
        (cs_plus_rewarded['auROC_SucOmit_rewarded'] < 0)
    )
    nonresponders_rewarded = np.sum(cs_plus_rewarded['pval_SucOmit_rewarded'] > alphalevel)
    
    print(f"\nCS+ Rewarded (n={num_cells_rewarded}):")
    print(f"  Activated: {activated_rewarded}")
    print(f"  Inhibited: {inhibited_rewarded}")
    print(f"  Non-responders: {nonresponders_rewarded}")
    
    if num_cells_rewarded > 0:
        activated_pct = activated_rewarded / num_cells_rewarded
        inhibited_pct = inhibited_rewarded / num_cells_rewarded
        nonresponders_pct = nonresponders_rewarded / num_cells_rewarded
        counts_rewarded = np.array([activated_pct, nonresponders_pct, inhibited_pct])
        labels_rewarded = (f"{activated_pct*100:.1f}%", '', f"{inhibited_pct*100:.1f}%")
    else:
        counts_rewarded = np.array([0, 1, 0])
        labels_rewarded = ('no data', '', '')
    
    # ===== CS+ UNREWARDED TRIALS (OMISSION) =====
    cs_plus_unrewarded = cs_plus_data.dropna(subset=['auROC_SucOmit_unrewarded', 'pval_SucOmit_unrewarded']).drop_duplicates(subset=['cell_id'])
    num_cells_unrewarded = len(cs_plus_unrewarded)
    
    activated_unrewarded = np.sum(
        (cs_plus_unrewarded['pval_SucOmit_unrewarded'] <= alphalevel) & 
        (cs_plus_unrewarded['auROC_SucOmit_unrewarded'] >= 0)
    )
    inhibited_unrewarded = np.sum(
        (cs_plus_unrewarded['pval_SucOmit_unrewarded'] <= alphalevel) & 
        (cs_plus_unrewarded['auROC_SucOmit_unrewarded'] < 0)
    )
    nonresponders_unrewarded = np.sum(cs_plus_unrewarded['pval_SucOmit_unrewarded'] > alphalevel)
    
    print(f"\nCS+ Unrewarded/Omitted (n={num_cells_unrewarded}):")
    print(f"  Activated: {activated_unrewarded}")
    print(f"  Inhibited: {inhibited_unrewarded}")
    print(f"  Non-responders: {nonresponders_unrewarded}")
    
    if num_cells_unrewarded > 0:
        activated_pct = activated_unrewarded / num_cells_unrewarded
        inhibited_pct = inhibited_unrewarded / num_cells_unrewarded
        nonresponders_pct = nonresponders_unrewarded / num_cells_unrewarded
        counts_unrewarded = np.array([activated_pct, nonresponders_pct, inhibited_pct])
        labels_unrewarded = (f"{activated_pct*100:.1f}%", '', f"{inhibited_pct*100:.1f}%")
    else:
        counts_unrewarded = np.array([0, 1, 0])
        labels_unrewarded = ('no data', '', '')
    
    # ===== CS- TRIALS =====
    # CS- uses the regular auc_minus and pval_minus columns
    cs_minus_data = day_data[day_data['cs_type'] == 'CS-']
    cs_minus = cs_minus_data.dropna(subset=['auc_minus', 'pval_minus']).drop_duplicates(subset=['cell_id'])
    num_cells_minus = len(cs_minus)
    
    activated_minus = np.sum(
        (cs_minus['pval_minus'] <= alphalevel) & 
        (cs_minus['auc_minus'] >= 0)
    )
    inhibited_minus = np.sum(
        (cs_minus['pval_minus'] <= alphalevel) & 
        (cs_minus['auc_minus'] < 0)
    )
    nonresponders_minus = np.sum(cs_minus['pval_minus'] > alphalevel)
    
    print(f"\nCS- (n={num_cells_minus}):")
    print(f"  Activated: {activated_minus}")
    print(f"  Inhibited: {inhibited_minus}")
    print(f"  Non-responders: {nonresponders_minus}")
    
    if num_cells_minus > 0:
        activated_pct = activated_minus / num_cells_minus
        inhibited_pct = inhibited_minus / num_cells_minus
        nonresponders_pct = nonresponders_minus / num_cells_minus
        counts_minus = np.array([activated_pct, nonresponders_pct, inhibited_pct])
        labels_minus = (f"{activated_pct*100:.1f}%", '', f"{inhibited_pct*100:.1f}%")
    else:
        counts_minus = np.array([0, 1, 0])
        labels_minus = ('no data', '', '')
    
    # ===== PLOT PIE CHARTS =====
    axs[0].pie(counts_rewarded, colors=colors, labels=labels_rewarded)
    axs[0].set_title('CS+ Rewarded')
    
    axs[1].pie(counts_unrewarded, colors=colors, labels=labels_unrewarded)
    axs[1].set_title('CS+ Unrewarded\n(Omission)')
    
    axs[2].pie(counts_minus, colors=colors, labels=labels_minus)
    axs[2].set_title('CS-')
    
    title = f'Sucrose Omission Test (2_SucOmit) - {sex_selection.capitalize()}'
    fig.suptitle(title, fontsize=14, y=1.02)
    fig.tight_layout()
    
    # Save the plot
    pdf_filename = os.path.join(basedir, "Results", f'Pop_auROC_SucroseOmission_{sex_selection}_10-07-25.pdf')
    plt.savefig(pdf_filename, bbox_inches='tight')
    print(f"\nFigure saved: {pdf_filename}")
    plt.show()


# Run the analysis for all cells
plot_omission_pie_charts(df_master_omission_jp, alphalevel, basedir, sex_selection='all')

# Optional: Run separately by sex if needed
# plot_omission_pie_charts(df_master_omission_jp, alphalevel, basedir, sex_selection='male')
# plot_omission_pie_charts(df_master_omission_jp, alphalevel, basedir, sex_selection='female')

In [ ]:
# Check what columns exist
print("Columns containing 'SucOmit':")
print([col for col in df_master_omission_jp.columns if 'SucOmit' in col])

# Check the 2_SucOmit day data
omit_day = df_master_omission_jp[df_master_omission_jp['day'] == '2_SucOmit']
print(f"\nTotal rows for 2_SucOmit: {len(omit_day)}")
print(f"Unique cells: {omit_day['cell_id'].nunique()}")

# Check for non-null values
print(f"\nNon-null auROC_SucOmit_rewarded: {omit_day['auROC_SucOmit_rewarded'].notna().sum()}")
print(f"Non-null auROC_SucOmit_unrewarded: {omit_day['auROC_SucOmit_unrewarded'].notna().sum()}")

# Check unique cells with data
unique_cells = omit_day.drop_duplicates('cell_id')
print(f"\nUnique cells with rewarded data: {unique_cells['auROC_SucOmit_rewarded'].notna().sum()}")
print(f"Unique cells with unrewarded data: {unique_cells['auROC_SucOmit_unrewarded'].notna().sum()}")

# Check some sample values
print("\nSample pval_SucOmit_rewarded values:")
print(unique_cells['pval_SucOmit_rewarded'].dropna().head(10))

# auROC Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Define the pairings
pairings = [
    ('0_Early', '1_Late'),
    ('3_CueOmit', '2_SucOmit'), 
    ('4_RevEarly', '5_RevLate')
]


# Plot box and strip plots
for idx, (day1, day2) in enumerate(pairings):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Extract data for the pair of days from df_master
    auROC_plus_day1 = df_master[df_master['experimental_phase'] == day1][df_master['cs_type'] == 'CS+']['auc_plus'].unique()  
    auROC_plus_day2 = df_master[df_master['experimental_phase'] == day2][df_master['cs_type'] == 'CS+']['auc_plus'].unique()
    auROC_minus_day1 = df_master[df_master['experimental_phase'] == day1][df_master['cs_type'] == 'CS-']['auc_minus'].unique()
    auROC_minus_day2 = df_master[df_master['experimental_phase'] == day2][df_master['cs_type'] == 'CS-']['auc_minus'].unique()
    
    # Combine data and prepare positions
    data = [auROC_plus_day1, auROC_minus_day1, auROC_plus_day2, auROC_minus_day2]
    positions = [1, 2, 3, 4]
    labels = [f'{day1} CS+', f'{day1} CS-', f'{day2} CS+', f'{day2} CS-']
    
    # Create box plots
    box = ax.boxplot(data, positions=positions, patch_artist=True, showfliers=False, widths=0.12)
    
   # Customize box colors if desired
    colors = ['purple', 'grey', 'purple', 'grey']
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)

       # Change median line color to white (or use 'black' instead)
    for median in box['medians']:
        median.set_color('black')
        median.set_linewidth(2)  # Optional: make the line thicker for better visibility
    
    # Add strip plots with jitter - now with matching colors
    for i, d in enumerate(data):
        x = np.random.normal(positions[i], 0.01, size=len(d))  # Slight jitter for visibility
        ax.scatter(x, d, alpha=0.3, edgecolor='black', s=30, color=colors[i])

    # Customize the plot
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('auROC Value')
    ax.set_title(f'auROC Values for {day1} vs {day2}')
    ax.axhline(0.0, color='black', linestyle='--', linewidth=0.5)
    ax.set_ylim(-1.25, 1.25)
    ax.set_yticks([-1.0, -0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])
    plt.tight_layout()
    
    # # Save the plot
    # pdf_filename = os.path.join(basedir, "Results", f'auROC_{day1}_vs_{day2}_CombinedCueSucrose_BoxPlot_6-20-25.pdf')
    # plt.savefig(pdf_filename)
    plt.show()

# stats for auROC graphs to be MLM with omnibus anova for table

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols
import warnings

# Define the pairings
pairings = [
    ('0_Early', '1_Late'),
    ('3_CueOmit', '2_SucOmit'), 
    ('4_RevEarly', '5_RevLate')
]

# Perform 2x2 Mixed Linear Model and ANOVA for each pairing
for day1, day2 in pairings:
    print(f"\n{'='*60}")
    print(f"ANALYSIS: {day1} vs {day2}")
    print(f"{'='*60}")
    
    # Extract data for the pair of days from df_master
    # Get CS+ data for both days
    cs_plus_day1 = df_master[
        (df_master['experimental_phase'] == day1) & 
        (df_master['cs_type'] == 'CS+')
    ][['auc_plus', 'fov']].copy()
    cs_plus_day1['day'] = day1
    cs_plus_day1['trial_type'] = 'CS+'
    cs_plus_day1.rename(columns={'auc_plus': 'auROC'}, inplace=True)
    
    cs_plus_day2 = df_master[
        (df_master['experimental_phase'] == day2) & 
        (df_master['cs_type'] == 'CS+')
    ][['auc_plus', 'fov']].copy()
    cs_plus_day2['day'] = day2
    cs_plus_day2['trial_type'] = 'CS+'
    cs_plus_day2.rename(columns={'auc_plus': 'auROC'}, inplace=True)
    
    # Get CS- data for both days
    cs_minus_day1 = df_master[
        (df_master['experimental_phase'] == day1) & 
        (df_master['cs_type'] == 'CS-')
    ][['auc_minus', 'fov']].copy()
    cs_minus_day1['day'] = day1
    cs_minus_day1['trial_type'] = 'CS-'
    cs_minus_day1.rename(columns={'auc_minus': 'auROC'}, inplace=True)
    
    cs_minus_day2 = df_master[
        (df_master['experimental_phase'] == day2) & 
        (df_master['cs_type'] == 'CS-')
    ][['auc_minus', 'fov']].copy()
    cs_minus_day2['day'] = day2
    cs_minus_day2['trial_type'] = 'CS-'
    cs_minus_day2.rename(columns={'auc_minus': 'auROC'}, inplace=True)
    
    # Combine all data for this pairing
    analysis_data = pd.concat([cs_plus_day1, cs_plus_day2, cs_minus_day1, cs_minus_day2], 
                             ignore_index=True)
    
    # Remove any missing values
    analysis_data = analysis_data.dropna()
    
    # Check if we have data
    if len(analysis_data) == 0:
        print(f"No data available for {day1} vs {day2}")
        continue
    
    print(f"Sample sizes:")
    print(analysis_data.groupby(['day', 'trial_type']).size())
    print(f"Number of FOVs: {analysis_data['fov'].nunique()}")
    
    # Create categorical variables with proper contrasts
    analysis_data['day_cat'] = pd.Categorical(analysis_data['day'])
    analysis_data['trial_type_cat'] = pd.Categorical(analysis_data['trial_type'])
    
    try:
        # Fit Mixed Linear Model
        # FOV as random intercept, day and trial_type as fixed effects
        mlm = MixedLM.from_formula('auROC ~ day_cat * trial_type_cat', 
                                  data=analysis_data, 
                                  groups=analysis_data['fov'])
        mlm_result = mlm.fit(method='lbfgs')
        
        print(f"\nMixed Linear Model Results:")
        print(f"-" * 40)
        print(mlm_result.summary())
        
        # For omnibus ANOVA, we need to use OLS and compare models
        # Since MixedLM doesn't directly provide ANOVA table, we'll use model comparison
        print(f"\nOmnibus ANOVA Results:")
        print(f"-" * 40)
        
        # Fit models for ANOVA-style testing using OLS with cluster-robust standard errors
        # Note: This is an approximation since true mixed model ANOVA is more complex
        
        # Full model
        full_model = ols('auROC ~ C(day_cat) * C(trial_type_cat)', data=analysis_data).fit()
        
        # Reduced models for testing main effects and interaction
        no_interaction = ols('auROC ~ C(day_cat) + C(trial_type_cat)', data=analysis_data).fit()
        no_day = ols('auROC ~ C(trial_type_cat)', data=analysis_data).fit()
        no_trial_type = ols('auROC ~ C(day_cat)', data=analysis_data).fit()
        intercept_only = ols('auROC ~ 1', data=analysis_data).fit()
        
        # Print ANOVA table from full model
        print("\nANOVA Table (OLS approximation):")
        anova_table = anova_lm(full_model, typ=2)
        print(anova_table)
        
        # Model comparisons for effect significance
        print(f"\nModel Comparison Tests:")
        print(f"-" * 25)
        
        # Test interaction
        interaction_test = anova_lm(no_interaction, full_model)
        f_int = interaction_test.iloc[1]['F']
        p_int = interaction_test.iloc[1]['Pr(>F)']
        print(f"Day × Trial Type Interaction: F = {f_int:.4f}, p = {p_int:.4f}")
        
        # Test main effect of day
        day_test = anova_lm(no_day, no_interaction)
        f_day = day_test.iloc[1]['F']
        p_day = day_test.iloc[1]['Pr(>F)']
        print(f"Main Effect of Day: F = {f_day:.4f}, p = {p_day:.4f}")
        
        # Test main effect of trial type
        trial_test = anova_lm(no_trial_type, no_interaction)
        f_trial = trial_test.iloc[1]['F']
        p_trial = trial_test.iloc[1]['Pr(>F)']
        print(f"Main Effect of Trial Type: F = {f_trial:.4f}, p = {p_trial:.4f}")
        
        # Effect sizes (eta-squared approximation)
        ss_total = ((analysis_data['auROC'] - analysis_data['auROC'].mean())**2).sum()
        ss_day = ((analysis_data.groupby('day_cat')['auROC'].transform('mean') - analysis_data['auROC'].mean())**2).sum()
        ss_trial = ((analysis_data.groupby('trial_type_cat')['auROC'].transform('mean') - analysis_data['auROC'].mean())**2).sum()
        
        eta2_day = ss_day / ss_total
        eta2_trial = ss_trial / ss_total
        
        print(f"\nEffect Sizes (η²):")
        print(f"Day: {eta2_day:.4f}")
        print(f"Trial Type: {eta2_trial:.4f}")
        
        # Group means for interpretation
        print(f"\nGroup Means:")
        print(f"-" * 15)
        group_means = analysis_data.groupby(['day_cat', 'trial_type_cat'])['auROC'].agg(['mean', 'std', 'count'])
        print(group_means)
        
        # Tukey's Post Hoc Tests
        print(f"\nTukey's Post Hoc Tests:")
        print(f"=" * 30)
        
        # Create combined group variable for all pairwise comparisons
        analysis_data['combined_group'] = analysis_data['day_cat'].astype(str) + '_' + analysis_data['trial_type_cat'].astype(str)
        
        # Perform Tukey HSD on all four groups
        tukey_all = pairwise_tukeyhsd(analysis_data['auROC'], analysis_data['combined_group'], alpha=0.05)
        print("\nAll Pairwise Comparisons (4 groups):")
        print(tukey_all)
        
        # Extract significant comparisons
        print(f"\nSignificant Comparisons (p < 0.05):")
        print(f"-" * 35)
        significant_found = False
        for i, comparison in enumerate(tukey_all._results_table.data[1:]):
            group1, group2, meandiff, p_adj, lower, upper, reject = comparison
            if reject:  # reject is True when p < alpha
                significant_found = True
                print(f"{group1} vs {group2}:")
                print(f"  Mean Difference: {meandiff:.4f}")
                print(f"  Adjusted p-value: {p_adj:.4f}")
                print(f"  95% CI: [{lower:.4f}, {upper:.4f}]")
                print()
        
        if not significant_found:
            print("No significant pairwise differences found.")
        
        # Additional targeted post hoc tests based on significant main effects/interactions
        print(f"\nTargeted Post Hoc Tests:")
        print(f"-" * 25)
        
        # If significant main effect of day, compare days collapsed across trial types
        if p_day < 0.05:
            print(f"\nDay Main Effect Follow-up (collapsed across trial types):")
            day_comparison = pairwise_tukeyhsd(analysis_data['auROC'], analysis_data['day_cat'], alpha=0.05)
            print(day_comparison)
        
        # If significant main effect of trial type, compare trial types collapsed across days
        if p_trial < 0.05:
            print(f"\nTrial Type Main Effect Follow-up (collapsed across days):")
            trial_comparison = pairwise_tukeyhsd(analysis_data['auROC'], analysis_data['trial_type_cat'], alpha=0.05)
            print(trial_comparison)
        
        # If significant interaction, provide interpretation guidance
        if p_int < 0.05:
            print(f"\nInteraction Effect Detected:")
            print(f"The effect of day depends on trial type (or vice versa).")
            print(f"Focus on the 4-group pairwise comparisons above for interpretation.")
            
            # Simple effects analysis for interaction
            print(f"\nSimple Effects Analysis:")
            print(f"CS+ effect: {day1} vs {day2}")
            cs_plus_data = analysis_data[analysis_data['trial_type_cat'] == 'CS+']
            if len(cs_plus_data) > 0:
                cs_plus_tukey = pairwise_tukeyhsd(cs_plus_data['auROC'], cs_plus_data['day_cat'], alpha=0.05)
                print(cs_plus_tukey)
            
            print(f"\nCS- effect: {day1} vs {day2}")
            cs_minus_data = analysis_data[analysis_data['trial_type_cat'] == 'CS-']
            if len(cs_minus_data) > 0:
                cs_minus_tukey = pairwise_tukeyhsd(cs_minus_data['auROC'], cs_minus_data['day_cat'], alpha=0.05)
                print(cs_minus_tukey)
        
    except Exception as e:
        print(f"Error in analysis for {day1} vs {day2}: {str(e)}")
        print("This might be due to insufficient data or convergence issues.")
        
        # Fallback to simple group comparison
        print(f"\nFallback: Group means and basic stats:")
        if len(analysis_data) > 0:
            group_stats = analysis_data.groupby(['day_cat', 'trial_type_cat'])['auROC'].agg(['mean', 'std', 'count'])
            print(group_stats)

print(f"\n{'='*60}")
print("ANALYSIS COMPLETE")
print(f"{'='*60}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Plot for 2_SucOmit day comparing CS+ rewarded, CS+ unrewarded, and CS-
fig, ax = plt.subplots(figsize=(10, 6))

# Extract data exactly like the original code structure
# CS+ rewarded trials - get unique auROC values per cell
auROC_plus_rewarded = df_master_omission_jp[
    (df_master_omission_jp['day'] == '2_SucOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS+') &
    (df_master_omission_jp['CS+ Reward Status'] == 1)
].groupby('cell_id')['auROC_SucOmit_rewarded'].first().dropna().values

# CS+ unrewarded trials - get unique auROC values per cell
auROC_plus_unrewarded = df_master_omission_jp[
    (df_master_omission_jp['day'] == '2_SucOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS+') &
    (df_master_omission_jp['CS+ Reward Status'] == 0)
].groupby('cell_id')['auROC_SucOmit_unrewarded'].first().dropna().values

# CS- trials - get unique auROC values per cell
auROC_minus = df_master_omission_jp[
    (df_master_omission_jp['day'] == '2_SucOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS-')
].groupby('cell_id')['auc_minus'].first().dropna().values

# Combine data and prepare positions
data = [auROC_plus_rewarded, auROC_plus_unrewarded, auROC_minus]
positions = [1, 2, 3]
labels = ['CS+ Rewarded', 'CS+ Omitted', 'CS-']

# Create box plots
box = ax.boxplot(data, positions=positions, patch_artist=True, showfliers=False, widths=0.3)

# Customize box colors
colors = ['purple', 'gold', 'grey']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

# Change median line color
for median in box['medians']:
    median.set_color('black')
    median.set_linewidth(2)

# Add strip plots with jitter
for i, d in enumerate(data):
    x = np.random.normal(positions[i], 0.01, size=len(d))  # Slight jitter for visibility
    ax.scatter(x, d, alpha=0.3, edgecolor='black', s=30, color=colors[i])

# Customize the plot
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=0)
ax.set_ylabel('auROC Value')
ax.set_title('2_SucOmit: Response to CS+ Rewarded vs Omitted vs CS-')
ax.axhline(0.0, color='black', linestyle='--', linewidth=0.5)
ax.set_ylim(-1.25, 1.25)
ax.set_yticks([-1.0, -0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])

# Add sample sizes to labels
ax.set_xticklabels([f'{labels[i]}\n(n={len(data[i])})' for i in range(len(labels))])

plt.tight_layout()

# Save the plot
pdf_filename = os.path.join(basedir, "Results", f'auROC_2_SucOmit_Rewarded_vs_Omitted_BoxPlot_08-06-25.pdf')
plt.savefig(pdf_filename)
plt.show()

# Print summary statistics
print("=== 2_SucOmit auROC Summary ===")
print(f"CS+ Rewarded: n={len(auROC_plus_rewarded)}, mean={np.mean(auROC_plus_rewarded):.3f}, SEM={np.std(auROC_plus_rewarded)/np.sqrt(len(auROC_plus_rewarded)):.3f}")
print(f"CS+ Omitted: n={len(auROC_plus_unrewarded)}, mean={np.mean(auROC_plus_unrewarded):.3f}, SEM={np.std(auROC_plus_unrewarded)/np.sqrt(len(auROC_plus_unrewarded)):.3f}")
print(f"CS-: n={len(auROC_minus)}, mean={np.mean(auROC_minus):.3f}, SEM={np.std(auROC_minus)/np.sqrt(len(auROC_minus)):.3f}")

# Statistical Analysis: One-way ANOVA and Tukey HSD
print("\n=== Statistical Analysis ===")

# Combine data for ANOVA
all_data = np.concatenate([auROC_plus_rewarded, auROC_plus_unrewarded, auROC_minus])
groups = (
    ['CS+ Rewarded'] * len(auROC_plus_rewarded) +
    ['CS+ Omitted'] * len(auROC_plus_unrewarded) +
    ['CS-'] * len(auROC_minus)
)

# Perform one-way ANOVA
f_val, p_val = f_oneway(auROC_plus_rewarded, auROC_plus_unrewarded, auROC_minus)
print(f"\nOne-way ANOVA results:")
print(f"F-value: {f_val:.4f}, p-value: {p_val:.4f}")

# Perform Tukey HSD for post-hoc comparisons
tukey_results = pairwise_tukeyhsd(all_data, groups)
print(f"\nTukey HSD results:")
print(tukey_results)

# Print significant comparisons only
print("\nSignificant comparisons (p < 0.05):")
for idx, row in enumerate(tukey_results._results_table.data[1:]):
    if row[3] < 0.05:  # p-value is at index 3
        print(f"{row[0]} vs {row[1]}: mean diff = {row[2]:.3f}, p = {row[3]:.4f}")


# Additional comparison plot: 1_Late vs 2_SucOmit (all conditions)
fig, ax = plt.subplots(figsize=(12, 6))

# Get 1_Late data for comparison (using the same extraction method)
auROC_plus_1late = df_master_omission_jp[
    (df_master_omission_jp['day'] == '1_Late') & 
    (df_master_omission_jp['cs_type'] == 'CS+')
].groupby('cell_id')['auc_plus'].first().dropna().values

auROC_minus_1late = df_master_omission_jp[
    (df_master_omission_jp['day'] == '1_Late') & 
    (df_master_omission_jp['cs_type'] == 'CS-')
].groupby('cell_id')['auc_minus'].first().dropna().values

# Combine all data
data = [auROC_plus_1late, auROC_minus_1late, 
        auROC_plus_rewarded, auROC_plus_unrewarded, auROC_minus]
positions = [1, 2, 4, 5, 6]
labels = ['1_Late CS+', '1_Late CS-', 
          '2_SucOmit\nCS+ Rewarded', '2_SucOmit\nCS+ Omitted', '2_SucOmit\nCS-']

# Create box plots
box = ax.boxplot(data, positions=positions, patch_artist=True, showfliers=False, widths=0.3)

# Customize box colors
colors = ['purple', 'grey', 'purple', 'gold', 'grey']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

# Change median line color
for median in box['medians']:
    median.set_color('black')
    median.set_linewidth(2)

# Add strip plots with jitter
for i, d in enumerate(data):
    x = np.random.normal(positions[i], 0.01, size=len(d))
    ax.scatter(x, d, alpha=0.3, edgecolor='black', s=30, color=colors[i])

# Customize the plot
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('auROC Value')
ax.set_title('Comparison: 1_Late vs 2_SucOmit (Rewarded/Omitted)')
ax.axhline(0.0, color='black', linestyle='--', linewidth=0.5)
ax.set_ylim(-1.25, 1.25)
ax.set_yticks([-1.0, -0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])

# Add vertical line to separate days
ax.axvline(3, color='black', linestyle=':', linewidth=1, alpha=0.5)

# Add sample sizes to labels
ax.set_xticklabels([f'{labels[i]}\n(n={len(data[i])})' for i in range(len(labels))], 
                    rotation=30, ha='right')

plt.tight_layout()

# Save the plot
pdf_filename = os.path.join(basedir, "Results", f'auROC_1Late_vs_2SucOmit_Comparison_BoxPlot_08-06-25.pdf')
plt.savefig(pdf_filename)
plt.show()

# Print comparison statistics
print("\n=== 1_Late vs 2_SucOmit Comparison ===")
print(f"1_Late CS+: n={len(auROC_plus_1late)}, mean={np.mean(auROC_plus_1late):.3f}")
print(f"1_Late CS-: n={len(auROC_minus_1late)}, mean={np.mean(auROC_minus_1late):.3f}")
print(f"2_SucOmit CS+ Rewarded: n={len(auROC_plus_rewarded)}, mean={np.mean(auROC_plus_rewarded):.3f}")
print(f"2_SucOmit CS+ Omitted: n={len(auROC_plus_unrewarded)}, mean={np.mean(auROC_plus_unrewarded):.3f}")
print(f"2_SucOmit CS-: n={len(auROC_minus)}, mean={np.mean(auROC_minus):.3f}")

# stats with table

In [ ]:
###USE OMISSION DF 
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# Comparison plot: 2_SucOmit vs 3_CueOmit
fig, ax = plt.subplots(figsize=(12, 6))

# 2_SucOmit data - CS+ split into rewarded and unrewarded
auROC_plus_rewarded = df_master_omission_jp[
    (df_master_omission_jp['day'] == '2_SucOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS+') &
    (df_master_omission_jp['CS+ Reward Status'] == 1)
].groupby('cell_id')['auROC_SucOmit_rewarded'].first().dropna().values

auROC_plus_unrewarded = df_master_omission_jp[
    (df_master_omission_jp['day'] == '2_SucOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS+') &
    (df_master_omission_jp['CS+ Reward Status'] == 0)
].groupby('cell_id')['auROC_SucOmit_unrewarded'].first().dropna().values

auROC_minus_2sucOmit = df_master_omission_jp[
    (df_master_omission_jp['day'] == '2_SucOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS-')
].groupby('cell_id')['auc_minus'].first().dropna().values

# 3_CueOmit data
auROC_plus_3cueOmit = df_master_omission_jp[
    (df_master_omission_jp['day'] == '3_CueOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS+')
].groupby('cell_id')['auc_plus'].first().dropna().values

auROC_minus_3cueOmit = df_master_omission_jp[
    (df_master_omission_jp['day'] == '3_CueOmit') & 
    (df_master_omission_jp['cs_type'] == 'CS-')
].groupby('cell_id')['auc_minus'].first().dropna().values

# Combine data
data = [auROC_plus_rewarded, auROC_plus_unrewarded, auROC_minus_2sucOmit,
        auROC_plus_3cueOmit, auROC_minus_3cueOmit]

positions = [1, 2, 3, 5, 6]

labels = ['2_SucOmit\nCS+ Rewarded', '2_SucOmit\nCS+ Omitted', '2_SucOmit\nCS-',
          '3_CueOmit\nCS+', '3_CueOmit\nCS-']

# Create box plots
box = ax.boxplot(data, positions=positions, patch_artist=True, showfliers=False, widths=0.3)

# Customize box colors
colors = ['purple', 'gold', 'grey', 'cyan', 'black']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

# Change median line color
for median in box['medians']:
    median.set_color('black')
    median.set_linewidth(2)

# Add strip plots with jitter
for i, d in enumerate(data):
    x = np.random.normal(positions[i], 0.01, size=len(d))
    ax.scatter(x, d, alpha=0.3, edgecolor='black', s=30, color=colors[i])

# Customize the plot
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('auROC Value')
ax.set_title('auROC Values: 2_SucOmit vs 3_CueOmit')
ax.axhline(0.0, color='black', linestyle='--', linewidth=0.5)
ax.set_ylim(-1.25, 1.25)
ax.set_yticks([-1.0, -0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])

# Add vertical line to separate days
ax.axvline(4, color='black', linestyle=':', linewidth=1, alpha=0.5)

# Add sample sizes to labels
ax.set_xticklabels([f'{labels[i]}\n(n={len(data[i])})' for i in range(len(labels))], 
                    rotation=30, ha='right')

plt.tight_layout()

# Save the plot
pdf_filename = os.path.join(basedir, "Results", f'auROC_2SucOmit_vs_3CueOmit_Comparison_BoxPlot.pdf')
plt.savefig(pdf_filename)
plt.show()

# Print summary statistics
print("=== Summary Statistics ===")
print(f"2_SucOmit CS+ Rewarded: n={len(auROC_plus_rewarded)}, mean={np.mean(auROC_plus_rewarded):.3f}, SEM={np.std(auROC_plus_rewarded)/np.sqrt(len(auROC_plus_rewarded)):.3f}")
print(f"2_SucOmit CS+ Omitted: n={len(auROC_plus_unrewarded)}, mean={np.mean(auROC_plus_unrewarded):.3f}, SEM={np.std(auROC_plus_unrewarded)/np.sqrt(len(auROC_plus_unrewarded)):.3f}")
print(f"2_SucOmit CS-: n={len(auROC_minus_2sucOmit)}, mean={np.mean(auROC_minus_2sucOmit):.3f}, SEM={np.std(auROC_minus_2sucOmit)/np.sqrt(len(auROC_minus_2sucOmit)):.3f}")
print(f"3_CueOmit CS+: n={len(auROC_plus_3cueOmit)}, mean={np.mean(auROC_plus_3cueOmit):.3f}, SEM={np.std(auROC_plus_3cueOmit)/np.sqrt(len(auROC_plus_3cueOmit)):.3f}")
print(f"3_CueOmit CS-: n={len(auROC_minus_3cueOmit)}, mean={np.mean(auROC_minus_3cueOmit):.3f}, SEM={np.std(auROC_minus_3cueOmit)/np.sqrt(len(auROC_minus_3cueOmit)):.3f}")

# Prepare data for MLM analysis - create long format from your existing grouped data
print("\n=== Preparing data for Mixed Linear Model ===")

# Create long format dataframe for MLM
mlm_data_list = []

# Add data with FOV information for each condition
for condition_name, group_filter, auc_column in [
    ('2_SucOmit_CS+_Rewarded', 
     (df_master_omission_jp['day'] == '2_SucOmit') & 
     (df_master_omission_jp['cs_type'] == 'CS+') & 
     (df_master_omission_jp['CS+ Reward Status'] == 1), 
     'auROC_SucOmit_rewarded'),
    ('2_SucOmit_CS+_Omitted', 
     (df_master_omission_jp['day'] == '2_SucOmit') & 
     (df_master_omission_jp['cs_type'] == 'CS+') & 
     (df_master_omission_jp['CS+ Reward Status'] == 0), 
     'auROC_SucOmit_unrewarded'),
    ('2_SucOmit_CS-', 
     (df_master_omission_jp['day'] == '2_SucOmit') & 
     (df_master_omission_jp['cs_type'] == 'CS-'), 
     'auc_minus'),
    ('3_CueOmit_CS+', 
     (df_master_omission_jp['day'] == '3_CueOmit') & 
     (df_master_omission_jp['cs_type'] == 'CS+'), 
     'auc_plus'),
    ('3_CueOmit_CS-', 
     (df_master_omission_jp['day'] == '3_CueOmit') & 
     (df_master_omission_jp['cs_type'] == 'CS-'), 
     'auc_minus')
]:
    subset = df_master_omission_jp[group_filter].groupby('cell_id').agg({
        auc_column: 'first',
        'fov': 'first'
    }).reset_index().dropna()
    subset['condition'] = condition_name
    subset['auROC_value'] = subset[auc_column]
    mlm_data_list.append(subset[['cell_id', 'fov', 'condition', 'auROC_value']])

# Combine all data
mlm_data = pd.concat(mlm_data_list, ignore_index=True)

print(f"MLM data shape: {mlm_data.shape}")
print(f"Number of FOVs: {mlm_data['fov'].nunique()}")
print(f"Number of cells: {mlm_data['cell_id'].nunique()}")

# Statistical Analysis: One-way ANOVA
print("\n=== Statistical Analysis ===")

# Combine all data for ANOVA
all_data = np.concatenate([auROC_plus_rewarded, auROC_plus_unrewarded, auROC_minus_2sucOmit,
                           auROC_plus_3cueOmit, auROC_minus_3cueOmit])

groups = (
    ['2_SucOmit CS+ Rewarded'] * len(auROC_plus_rewarded) +
    ['2_SucOmit CS+ Omitted'] * len(auROC_plus_unrewarded) +
    ['2_SucOmit CS-'] * len(auROC_minus_2sucOmit) +
    ['3_CueOmit CS+'] * len(auROC_plus_3cueOmit) +
    ['3_CueOmit CS-'] * len(auROC_minus_3cueOmit)
)

# Perform one-way ANOVA
f_val, p_val = f_oneway(auROC_plus_rewarded, auROC_plus_unrewarded, auROC_minus_2sucOmit,
                        auROC_plus_3cueOmit, auROC_minus_3cueOmit)

# Calculate degrees of freedom and create ANOVA table
k = 5  # number of groups
n_total = len(all_data)  # total sample size
df_between = k - 1  # degrees of freedom between groups
df_within = n_total - k  # degrees of freedom within groups
df_total = n_total - 1  # total degrees of freedom

# Calculate sum of squares for complete ANOVA table
grand_mean = np.mean(all_data)
group_means = [np.mean(group_data) for group_data in data]
group_sizes = [len(group_data) for group_data in data]

# Sum of squares between groups
ss_between = sum(n * (mean - grand_mean)**2 for n, mean in zip(group_sizes, group_means))

# Sum of squares within groups
ss_within = sum(np.sum((group_data - np.mean(group_data))**2) for group_data in data)

# Sum of squares total
ss_total = ss_between + ss_within

# Mean squares
ms_between = ss_between / df_between
ms_within = ss_within / df_within

# Create and display ANOVA table
print("\n=== ANOVA Table ===")
anova_table = pd.DataFrame({
    'Source': ['Between Groups', 'Within Groups', 'Total'],
    'Sum of Squares': [f'{ss_between:.4f}', f'{ss_within:.4f}', f'{ss_total:.4f}'],
    'df': [df_between, df_within, df_total],
    'Mean Square': [f'{ms_between:.4f}', f'{ms_within:.4f}', '-'],
    'F': [f'{f_val:.4f}', '-', '-'],
    'p-value': [f'{p_val:.4f}', '-', '-']
})

print(anova_table.to_string(index=False))

print(f"\nOne-way ANOVA results:")
print(f"F({df_between}, {df_within}) = {f_val:.4f}, p = {p_val:.4f}")

# Mixed Linear Model Analysis
print("\n=== Mixed Linear Model Analysis (Nested) ===")

# Fit Mixed Linear Model with FOV as random effect
mlm_model = smf.mixedlm("auROC_value ~ C(condition)", data=mlm_data, groups=mlm_data["fov"])
mlm_results = mlm_model.fit()

print("\n=== MLM Results Summary ===")
print(mlm_results.summary())

# Perform Tukey HSD for post-hoc comparisons
tukey_results = pairwise_tukeyhsd(all_data, groups)
print(f"\nTukey HSD results:")
print(tukey_results)

# Print significant comparisons only
print("\nSignificant comparisons (p < 0.05):")
for idx, row in enumerate(tukey_results._results_table.data[1:]):
    if row[3] < 0.05:  # p-value is at index 3
        print(f"{row[0]} vs {row[1]}: mean diff = {row[2]:.3f}, p = {row[3]:.4f}")

# Additional focused comparisons
print("\n=== Key Comparisons ===")

# Compare CS+ rewarded vs omitted at 2_SucOmit
from scipy.stats import ttest_ind
t_stat, p_val_rewarded_omitted = ttest_ind(auROC_plus_rewarded, auROC_plus_unrewarded)
print(f"2_SucOmit CS+ Rewarded vs Omitted: t={t_stat:.3f}, p={p_val_rewarded_omitted:.4f}")

# Compare CS+ rewarded (2_SucOmit) vs CS+ (3_CueOmit)
t_stat, p_val = ttest_ind(auROC_plus_rewarded, auROC_plus_3cueOmit)
print(f"2_SucOmit CS+ Rewarded vs 3_CueOmit CS+: t={t_stat:.3f}, p={p_val:.4f}")

# Compare CS- across days
t_stat, p_val = ttest_ind(auROC_minus_2sucOmit, auROC_minus_3cueOmit)
print(f"2_SucOmit CS- vs 3_CueOmit CS-: t={t_stat:.3f}, p={p_val:.4f}")

# Report clustering information
print(f"\n=== Clustering Information ===")
fov_summary = mlm_data.groupby('fov').agg({
    'cell_id': 'nunique',
    'auROC_value': 'count'
}).round(2)
fov_summary.columns = ['Unique_Cells', 'Total_Observations']
print(f"FOV-level summary:")
print(fov_summary.describe())
print(f"\nMean cells per FOV: {fov_summary['Unique_Cells'].mean():.1f}")
print(f"Mean observations per FOV: {fov_summary['Total_Observations'].mean():.1f}")

# stats section with table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import anova_lm
import statsmodels.formula.api as smf

def plot_omission_lineplot_with_bifurcation(df_master_omission_jp, basedir='.'):
    """
    Plot auROC values showing CS+ bifurcation at 2_SucOmit (rewarded vs unrewarded)
    and CS- values across 1_Late, 2_SucOmit, and 3_CueOmit
    """
    
    days_to_visualize = ['1_Late', '2_SucOmit', '3_CueOmit']
    
    # Filter for cells tracked across all three days
    cell_counts = df_master_omission_jp[
        df_master_omission_jp['day'].isin(days_to_visualize)
    ].groupby('tracked_roi_counter')['day'].nunique()
    
    valid_cells = cell_counts[cell_counts == len(days_to_visualize)].index
    
    df_filtered = df_master_omission_jp[
        (df_master_omission_jp['tracked_roi_counter'].isin(valid_cells)) &
        (df_master_omission_jp['day'].isin(days_to_visualize))
    ]
    
    print(f"Processing days: {days_to_visualize}")
    print(f"Number of cells tracked across all days: {len(valid_cells)}")
    
    # Prepare data for plotting
    plot_data_list = []
    
    for cell in valid_cells:
        cell_data = df_filtered[df_filtered['tracked_roi_counter'] == cell]
        
        # Get FOV for this cell (should be consistent across days)
        fov = cell_data['fov'].iloc[0]
        
        # 1_Late data
        late_data = cell_data[cell_data['day'] == '1_Late']
        if not late_data.empty:
            # CS+
            cs_plus_val = late_data[late_data['cs_type'] == 'CS+']['auc_plus'].iloc[0]
            plot_data_list.append({
                'day': '1_Late',
                'cs_type': 'CS+',
                'tracked_roi_counter': cell,
                'fov': fov,
                'value': cs_plus_val
            })
            
            # CS-
            cs_minus_val = late_data[late_data['cs_type'] == 'CS-']['auc_minus'].iloc[0]
            plot_data_list.append({
                'day': '1_Late',
                'cs_type': 'CS-',
                'tracked_roi_counter': cell,
                'fov': fov,
                'value': cs_minus_val
            })
        
        # 2_SucOmit data - separate CS+ rewarded and unrewarded
        sucOmit_data = cell_data[cell_data['day'] == '2_SucOmit']
        if not sucOmit_data.empty:
            # CS+ Rewarded
            cs_plus_rewarded = sucOmit_data[sucOmit_data['cs_type'] == 'CS+']['auROC_SucOmit_rewarded'].iloc[0]
            if pd.notna(cs_plus_rewarded):
                plot_data_list.append({
                    'day': '2_SucOmit',
                    'cs_type': 'CS+ Rewarded',
                    'tracked_roi_counter': cell,
                    'fov': fov,
                    'value': cs_plus_rewarded
                })
            
            # CS+ Unrewarded (Omitted)
            cs_plus_unrewarded = sucOmit_data[sucOmit_data['cs_type'] == 'CS+']['auROC_SucOmit_unrewarded'].iloc[0]
            if pd.notna(cs_plus_unrewarded):
                plot_data_list.append({
                    'day': '2_SucOmit',
                    'cs_type': 'CS+ Omitted',
                    'tracked_roi_counter': cell,
                    'fov': fov,
                    'value': cs_plus_unrewarded
                })
            
            # CS-
            cs_minus_val = sucOmit_data[sucOmit_data['cs_type'] == 'CS-']['auc_minus'].iloc[0]
            plot_data_list.append({
                'day': '2_SucOmit',
                'cs_type': 'CS-',
                'tracked_roi_counter': cell,
                'fov': fov,
                'value': cs_minus_val
            })
        
        # 3_CueOmit data
        cueOmit_data = cell_data[cell_data['day'] == '3_CueOmit']
        if not cueOmit_data.empty:
            # CS+
            cs_plus_val = cueOmit_data[cueOmit_data['cs_type'] == 'CS+']['auc_plus'].iloc[0]
            plot_data_list.append({
                'day': '3_CueOmit',
                'cs_type': 'CS+',
                'tracked_roi_counter': cell,
                'fov': fov,
                'value': cs_plus_val
            })
            
            # CS-
            cs_minus_val = cueOmit_data[cueOmit_data['cs_type'] == 'CS-']['auc_minus'].iloc[0]
            plot_data_list.append({
                'day': '3_CueOmit',
                'cs_type': 'CS-',
                'tracked_roi_counter': cell,
                'fov': fov,
                'value': cs_minus_val
            })
    
    plot_data = pd.DataFrame(plot_data_list)
    
    # Calculate summary statistics
    summary_data = plot_data.groupby(['day', 'cs_type']).agg(
        mean_val=('value', 'mean'),
        sem_val=('value', 'sem'),
        n_cells=('value', 'count')
    ).reset_index()
    
    # Create the plot
    plt.figure(figsize=(12, 7))
    
    # Define x positions for days
    x_positions = {'1_Late': 0, '2_SucOmit': 1, '3_CueOmit': 2}
    
    # Define colors
    colors = {
        'CS+': 'purple',
        'CS+ Rewarded': 'purple',
        'CS+ Omitted': 'gold',
        'CS-': 'grey'
    }
    
    # Plot CS+ line (1_Late to 3_CueOmit)
    cs_plus_late = summary_data[(summary_data['day'] == '1_Late') & (summary_data['cs_type'] == 'CS+')]
    cs_plus_cue = summary_data[(summary_data['day'] == '3_CueOmit') & (summary_data['cs_type'] == 'CS+')]
    
    if not cs_plus_late.empty and not cs_plus_cue.empty:
        # Main CS+ line (dashed between Late and CueOmit to show discontinuity)
        plt.plot([0, 2], 
                [cs_plus_late['mean_val'].values[0], cs_plus_cue['mean_val'].values[0]], 
                'o-', color=colors['CS+'], linestyle='--', alpha=0.5, label='CS+ (baseline)')
        
        # Error bars for CS+ at Late and CueOmit
        plt.errorbar(0, cs_plus_late['mean_val'].values[0], 
                    yerr=cs_plus_late['sem_val'].values[0],
                    fmt='none', capsize=5, color=colors['CS+'])
        plt.errorbar(2, cs_plus_cue['mean_val'].values[0], 
                    yerr=cs_plus_cue['sem_val'].values[0],
                    fmt='none', capsize=5, color=colors['CS+'])
    
    # Plot CS+ Rewarded branch (1_Late to 2_SucOmit to 3_CueOmit)
    cs_plus_rewarded = summary_data[(summary_data['day'] == '2_SucOmit') & (summary_data['cs_type'] == 'CS+ Rewarded')]
    if not cs_plus_rewarded.empty:
        # Line from Late to Rewarded
        plt.plot([0, 1], 
                [cs_plus_late['mean_val'].values[0], cs_plus_rewarded['mean_val'].values[0]], 
                'o-', color=colors['CS+ Rewarded'], linewidth=2)
        # Line from Rewarded to CueOmit
        plt.plot([1, 2], 
                [cs_plus_rewarded['mean_val'].values[0], cs_plus_cue['mean_val'].values[0]], 
                'o-', color=colors['CS+ Rewarded'], linewidth=2, label='CS+ Rewarded')
        # Error bar
        plt.errorbar(1, cs_plus_rewarded['mean_val'].values[0], 
                    yerr=cs_plus_rewarded['sem_val'].values[0],
                    fmt='none', capsize=5, color=colors['CS+ Rewarded'])
    
    # Plot CS+ Omitted branch (1_Late to 2_SucOmit to 3_CueOmit)
    cs_plus_omitted = summary_data[(summary_data['day'] == '2_SucOmit') & (summary_data['cs_type'] == 'CS+ Omitted')]
    if not cs_plus_omitted.empty:
        # Line from Late to Omitted
        plt.plot([0, 1], 
                [cs_plus_late['mean_val'].values[0], cs_plus_omitted['mean_val'].values[0]], 
                'o-', color=colors['CS+ Omitted'], linewidth=2)
        # Line from Omitted to CueOmit
        plt.plot([1, 2], 
                [cs_plus_omitted['mean_val'].values[0], cs_plus_cue['mean_val'].values[0]], 
                'o-', color=colors['CS+ Omitted'], linewidth=2, label='CS+ Omitted')
        # Error bar
        plt.errorbar(1, cs_plus_omitted['mean_val'].values[0], 
                    yerr=cs_plus_omitted['sem_val'].values[0],
                    fmt='none', capsize=5, color=colors['CS+ Omitted'])
    
    # Plot CS- line (continuous across all days)
    cs_minus_data = summary_data[summary_data['cs_type'] == 'CS-'].sort_values('day')
    x_vals = [x_positions[day] for day in cs_minus_data['day']]
    plt.plot(x_vals, cs_minus_data['mean_val'].values, 
            'o-', color=colors['CS-'], linewidth=2, label='CS-')
    
    # Add error bars for CS-
    for i, row in cs_minus_data.iterrows():
        x_pos = x_positions[row['day']]
        plt.errorbar(x_pos, row['mean_val'], yerr=row['sem_val'],
                    fmt='none', capsize=5, color=colors['CS-'])
    
    # Formatting
    plt.xticks([0, 1, 2], ['1_Late', '2_SucOmit', '3_CueOmit'], rotation=45)
    plt.ylabel('auROC Value (Mean ± SEM)')
    plt.xlabel('Experimental Phase')
    plt.title('Tracked Cells: auROC Across Omission Phases')
    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
    plt.ylim(-0.5, 1.0)
    plt.tight_layout()
    
    # Save the plot
    plot_filename = "SucroseOmission_auROC_LinePlot.pdf"
    plot_path = os.path.join(basedir, 'Results', plot_filename)
    plt.savefig(plot_path, format='pdf', bbox_inches='tight', dpi=300)
    plt.show()
    
    # Print summary statistics
    print("\n=== Summary Statistics ===")
    print(summary_data[['day', 'cs_type', 'mean_val', 'sem_val', 'n_cells']])
    
    # Statistical Analysis
    print("\n=== Statistical Analysis ===")
    
    # For statistical analysis, create a combined day-condition variable
    plot_data['condition'] = plot_data['day'] + ' ' + plot_data['cs_type']
    
    # Print FOV nesting information
    print(f"\nFOV nesting information:")
    print(f"Number of FOVs: {plot_data['fov'].nunique()}")
    print(f"Number of cells: {plot_data['tracked_roi_counter'].nunique()}")
    fov_summary = plot_data.groupby('fov')['tracked_roi_counter'].nunique()
    print(f"Cells per FOV: mean={fov_summary.mean():.1f}, range={fov_summary.min()}-{fov_summary.max()}")
    
    # Create ANOVA table using OLS for F-test structure
    print("\n=== ANOVA Table ===")
    ols_model = smf.ols("value ~ C(condition)", data=plot_data)
    ols_results = ols_model.fit()
    anova_table = anova_lm(ols_results, typ=3)
    print(anova_table)
    
    # Mixed Model ANOVA with FOV nesting
    print("\nMixed Model Analysis (nested within FOV):")
    model = mixedlm("value ~ C(condition)", 
                    data=plot_data, 
                    groups="fov")
    model_results = model.fit()
    print(model_results.summary())
    
    # Post-hoc Tukey tests
    print("\nPost-hoc Tukey HSD:")
    tukey_results = pairwise_tukeyhsd(plot_data['value'], plot_data['condition'])
    print(tukey_results)
    
    # Print only significant comparisons
    print("\nSignificant comparisons (p < 0.05):")
    for idx, row in enumerate(tukey_results._results_table.data[1:]):
        if row[3] < 0.05:  # p-value is at index 3
            print(f"{row[0]} vs {row[1]}: mean diff = {row[2]:.3f}, p = {row[3]:.4f}")
    
    # Specific comparisons of interest
    print("\n=== Key Comparisons ===")
    
    # Compare CS+ Rewarded vs Omitted at 2_SucOmit
    rewarded_vals = plot_data[(plot_data['day'] == '2_SucOmit') & 
                              (plot_data['cs_type'] == 'CS+ Rewarded')]['value']
    omitted_vals = plot_data[(plot_data['day'] == '2_SucOmit') & 
                             (plot_data['cs_type'] == 'CS+ Omitted')]['value']
    
    if len(rewarded_vals) > 0 and len(omitted_vals) > 0:
        from scipy.stats import ttest_rel
        t_stat, p_val = ttest_rel(rewarded_vals, omitted_vals)
        print(f"\nCS+ Rewarded vs CS+ Omitted at 2_SucOmit (paired t-test):")
        print(f"  t = {t_stat:.3f}, p = {p_val:.4f}")
    
    return plot_data, summary_data, model_results, tukey_results

# Run the analysis
plot_data, summary_data, model_results, tukey_results = plot_omission_lineplot_with_bifurcation(
    df_master_omission_jp, 
    basedir=basedir
)

# spatial mapping with auROC split for CS+ rewarded and unrewarded

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, RegularPolygon, Rectangle, Circle
import roifile
from scipy.interpolate import splprep, splev

def smooth_coordinates(x_coords, y_coords, smoothing_factor, num_points=100):
    """
    Smooth ROI coordinates using B-spline interpolation.
    """
    if len(x_coords) < 4:
        return x_coords, y_coords
    
    if (x_coords[0], y_coords[0]) != (x_coords[-1], y_coords[-1]):
        x_coords = np.append(x_coords, x_coords[0])
        y_coords = np.append(y_coords, y_coords[0])
    
    tck, u = splprep([x_coords, y_coords], s=smoothing_factor, per=True)
    u_new = np.linspace(0, 1, num_points)
    x_new, y_new = splev(u_new, tck)
    
    return x_new, y_new

def create_star(x_center, y_center, size, num_points=5):
    """Generate coordinates for a star shape."""
    angle = np.pi / num_points
    star_points = []
    for i in range(2 * num_points):
        radius = size if i % 2 == 0 else size / 2
        x = x_center + radius * np.cos(i * angle)
        y = y_center + radius * np.sin(i * angle)
        star_points.append((x, y))
    return star_points

def create_cell_patch(shape_type, center_x, center_y, size, color, outline=True, alpha=0.7):
    """Create a matplotlib patch based on specified shape type."""
    edgecolor = 'black' if outline else 'none'
    
    if shape_type == 'hexagon':
        return RegularPolygon((center_x, center_y), numVertices=6, radius=size,
                            orientation=np.radians(30), facecolor=color,
                            edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'square':
        return Rectangle((center_x - size/2, center_y - size/2),
                        size, size, facecolor=color,
                        edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'circle':
        return Circle((center_x, center_y), radius=size/2,
                     facecolor=color, edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'star':
        star_points = create_star(center_x, center_y, size)
        return Polygon(star_points, closed=True, facecolor=color,
                      edgecolor=edgecolor, alpha=alpha)
    else:
        raise ValueError("Invalid shape_type. Choose from 'hexagon', 'square', 'circle', 'star'")

def visualize_omission_cell_positions(basedir, df_master_omission_jp, 
                                     plot_type='combined', roi_type='real', 
                                     shape='hexagon', grid_size=30,
                                     roi_smoothing=0, outline=True, blend_colors=True,
                                     show_cell_numbers=False):
    """
    Visualize cell positions for omission test data from 2_SucOmit day.
    Creates 3 panels: CS+ Rewarded, CS+ Unrewarded, and CS- trials.
    
    Parameters:
    -----------
    basedir : str
        Base directory containing the data
    df_master_omission_jp : pandas DataFrame
        Omission dataframe containing cell data
    plot_type : str
        'combined' for all FOVs overlaid, 'individual' for separate FOV plots
    roi_type : str
        'real' for actual ROI shapes, 'shape' for standardized shapes
    shape : str
        Shape to use when roi_type='shape' ('hexagon', 'square', 'circle', 'star')
    grid_size : int
        Size of the grid elements when roi_type='shape'
    roi_smoothing : float
        Smoothing factor for ROI outlines when roi_type='real'
    outline : bool
        Whether to show shape outlines
    blend_colors : bool
        Whether to enable color blending between shapes
    show_cell_numbers : bool
        Whether to display cell numbers in the visualization
    """
    
    day = '2_SucOmit'
    
    # Define the three conditions
    conditions = [
        ('CS+ Rewarded', 'auROC_SucOmit_rewarded', 'CS+'),
        ('CS+ Unrewarded', 'auROC_SucOmit_unrewarded', 'CS+'),
        ('CS-', 'auc_minus', 'CS-')
    ]
    
    if plot_type == 'combined':
        # Single row of 3 subplots for the 3 conditions
        fig, axes = plt.subplots(1, 3, figsize=(15, 6), dpi=150)
        fig.suptitle(f'Cell Positions Heatmap - Sucrose Omission Test (2_SucOmit)', fontsize=16)
        
        for col, (condition_name, column_name, cs_type) in enumerate(conditions):
            ax = axes[col]
            ax.set_xlim(0, 512)
            ax.set_ylim(512, 0)
            ax.set_title(condition_name, fontsize=12)
            
            # Filter for this day and CS type
            day_df = df_master_omission_jp[
                (df_master_omission_jp['day'] == day) & 
                (df_master_omission_jp['cs_type'] == cs_type)
            ]
            
            if day_df.empty:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.axis('off')
                continue
            
            # Process each FOV
            unique_fovs = day_df[['animal', 'fov']].drop_duplicates()
            for animal, fov in unique_fovs.values:
                roi_folder = os.path.join(basedir, day, animal, fov)
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')] if os.path.exists(roi_folder) else []
                
                if roi_files:
                    roi_path = os.path.join(roi_folder, roi_files[0])
                    rois = roifile.ImagejRoi.fromfile(roi_path)
                    df_subset = day_df[(day_df['animal'] == animal) & (day_df['fov'] == fov)]
                    
                    for roi_number, roi in enumerate(rois, 1):
                        coords = roi.coordinates()
                        center_x = np.mean(coords[:, 0])
                        center_y = np.mean(coords[:, 1])
                        
                        matching_rows = df_subset[df_subset['cell_number'] == roi_number]
                        cell_value = matching_rows[column_name].mean() if not matching_rows.empty else np.nan
                        color = plt.cm.PuRd(cell_value) if not np.isnan(cell_value) else 'gray'
                        
                        if roi_type == 'real':
                            x_coords, y_coords = coords[:, 0], coords[:, 1]
                            if roi_smoothing > 0:
                                x_coords, y_coords = smooth_coordinates(x_coords, y_coords, roi_smoothing)
                            patch = Polygon(np.column_stack([x_coords, y_coords]),
                                         closed=True, fill=True,
                                         edgecolor='black' if outline else 'none',
                                         facecolor=color,
                                         alpha=0.7 if blend_colors else 1.0)
                        else:  # roi_type == 'shape'
                            patch = create_cell_patch(shape, center_x, center_y, grid_size,
                                                   color, outline, 0.7 if blend_colors else 1.0)
                        
                        ax.add_patch(patch)
                        
                        if show_cell_numbers:
                            ax.text(center_x, center_y, str(roi_number),
                                  ha='center', va='center',
                                  fontsize=8, fontweight='bold', color='black')
            
            ax.grid(True, linestyle='--', alpha=0.5)
            ax.set_xticks([])
            ax.set_yticks([])
        
    else:  # plot_type == 'individual'
        for animal in df_master_omission_jp['animal'].unique():
            fovs_for_animal = df_master_omission_jp[
                (df_master_omission_jp['animal'] == animal) & 
                (df_master_omission_jp['day'] == day)
            ]['fov'].unique()
            all_fovs = sorted(fovs_for_animal)
            
            if not all_fovs:
                continue
                
            fig, axes = plt.subplots(len(all_fovs), 3,
                                   figsize=(15, len(all_fovs) * 5),
                                   dpi=150, squeeze=False)
            fig.suptitle(f'Cell Positions Heatmap - Sucrose Omission - {animal}', fontsize=16)
            
            for row, fov in enumerate(all_fovs):
                for col, (condition_name, column_name, cs_type) in enumerate(conditions):
                    ax = axes[row, col]
                    ax.set_xlim(0, 512)
                    ax.set_ylim(512, 0)
                    
                    roi_folder = os.path.join(basedir, day, animal, fov)
                    roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')] if os.path.exists(roi_folder) else []
                    
                    if roi_files:
                        roi_path = os.path.join(roi_folder, roi_files[0])
                        rois = roifile.ImagejRoi.fromfile(roi_path)
                        df_subset = df_master_omission_jp[
                            (df_master_omission_jp['day'] == day) &
                            (df_master_omission_jp['animal'] == animal) &
                            (df_master_omission_jp['fov'] == fov) &
                            (df_master_omission_jp['cs_type'] == cs_type)
                        ]
                        
                        for roi_number, roi in enumerate(rois, 1):
                            coords = roi.coordinates()
                            center_x = np.mean(coords[:, 0])
                            center_y = np.mean(coords[:, 1])
                            
                            cell_value = df_subset[df_subset['cell_number'] == roi_number][column_name].mean()
                            color = plt.cm.PuRd(cell_value) if not np.isnan(cell_value) else 'gray'
                            
                            if roi_type == 'real':
                                x_coords, y_coords = coords[:, 0], coords[:, 1]
                                if roi_smoothing > 0:
                                    x_coords, y_coords = smooth_coordinates(x_coords, y_coords, roi_smoothing)
                                patch = Polygon(np.column_stack([x_coords, y_coords]),
                                             closed=True, fill=True,
                                             edgecolor='black' if outline else 'none',
                                             facecolor=color,
                                             alpha=0.7 if blend_colors else 1.0)
                            else:  # roi_type == 'shape'
                                patch = create_cell_patch(shape, center_x, center_y, grid_size,
                                                       color, outline, 0.7 if blend_colors else 1.0)
                            
                            ax.add_patch(patch)
                            
                            if show_cell_numbers:
                                ax.text(center_x, center_y, str(roi_number),
                                      ha='center', va='center',
                                      fontsize=8, fontweight='bold', color='black')
                        
                        ax.set_title(f'{condition_name} - {fov}', fontsize=10)
                        ax.grid(True, linestyle='--', alpha=0.5)
                    else:
                        ax.text(0.5, 0.5, f'No data\n{condition_name}',
                               ha='center', va='center',
                               transform=ax.transAxes)
                        ax.axis('off')
    
    # Add colorbar - use the max/min across all three conditions
    all_values = []
    for _, column_name, cs_type in conditions:
        vals = df_master_omission_jp[
            (df_master_omission_jp['day'] == day) & 
            (df_master_omission_jp['cs_type'] == cs_type)
        ][column_name].dropna()
        all_values.extend(vals)
    
    if all_values:
        sm = plt.cm.ScalarMappable(cmap="PuRd",
                                  norm=plt.Normalize(vmin=min(all_values),
                                                   vmax=max(all_values)))
        cbar = plt.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label('AUC Value', fontsize=10)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    
    # Save the figure
    plot_type_str = 'combined' if plot_type == 'combined' else 'individual'
    roi_type_str = f"_{roi_type}"
    shape_str = f"_{shape}" if roi_type == 'shape' else f"_smooth_{roi_smoothing}"
    
    save_path = os.path.join(basedir, 'Results', 'Cell_Visualizations',
                            f"omission_cell_positions_{plot_type_str}{roi_type_str}{shape_str}.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Saved {plot_type_str} omission cell position visualization with {roi_type} ROIs")

# Example usage:

# For combined plot with shape ROIs
visualize_omission_cell_positions(basedir, df_master_omission_jp,
                                 plot_type='combined', roi_type='shape',
                                 shape='hexagon', grid_size=30, outline=True, blend_colors=True)

# For combined plot with real ROIs
# visualize_omission_cell_positions(basedir, df_master_omission_jp,
#                                  plot_type='combined', roi_type='real',
#                                  roi_smoothing=100, outline=True, blend_colors=True)

# For individual FOV plots
# visualize_omission_cell_positions(basedir, df_master_omission_jp,
#                                  plot_type='individual', roi_type='shape',
#                                  shape='hexagon', grid_size=30, outline=True, 
#                                  blend_colors=True, show_cell_numbers=True)

# Calculating minimum distance from highest auROC cell to next highest etc in real data frame
Data is plotted and added to df_master¶

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, linregress, gaussian_kde
import roifile

def get_roi_centers(rois):
    """Calculate the center of each ROI."""
    centers = []
    for roi in rois:
        coords = roi.coordinates()
        center_x = np.mean(coords[:, 0])
        center_y = np.mean(coords[:, 1])
        centers.append((center_x, center_y))
    return np.array(centers)

def calculate_distances(center1, centers, metric_type, fov_center=None):
    """Calculate distances using different metrics."""
    if metric_type == 'euclidean':
        return cdist([center1], centers, metric='euclidean').flatten()
    elif metric_type == 'manhattan':
        return cdist([center1], centers, metric='cityblock').flatten()
    elif metric_type == 'chebyshev':
        return cdist([center1], centers, metric='chebyshev').flatten()
    elif metric_type == 'combined1':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        manhattan = cdist([center1], centers, metric='cityblock').flatten()
        return 0.7 * euclidean + 0.3 * manhattan
    elif metric_type == 'combined2':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        chebyshev = cdist([center1], centers, metric='chebyshev').flatten()
        return np.sqrt(euclidean * chebyshev)
    elif metric_type == 'combined3':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        rel_positions = centers - center1
        angles = np.arctan2(rel_positions[:, 1], rel_positions[:, 0])
        return euclidean * (1 + 0.2 * np.cos(angles))

def add_omission_distance_column(df_master_omission_jp, basedir, metric_type='combined3'):
    """
    Add distance columns to df_master_omission_jp for omission test data.
    Calculates distances separately for rewarded and unrewarded CS+ trials.
    
    Parameters:
    -----------
    df_master_omission_jp : pandas.DataFrame
        Omission dataframe containing cell data
    basedir : str
        Base directory containing ROI files
    metric_type : str
        Distance metric to use for calculation
        
    Returns:
    --------
    pandas.DataFrame
        The dataframe with new distance columns added
    """
    df = df_master_omission_jp.copy()
    
    # Initialize distance columns
    df['auc_distance_rewarded'] = np.nan
    df['auc_distance_unrewarded'] = np.nan
    df['auc_distance_minus'] = np.nan
    
    day = '2_SucOmit'
    
    print(f"Adding distance columns using {metric_type} metric for 2_SucOmit day...")
    
    animals = df[df['day'] == day]['animal'].unique()
    
    for animal in animals:
        fovs = df[(df['day'] == day) & (df['animal'] == animal)]['fov'].unique()
        
        for fov in fovs:
            print(f"  Processing {animal}, FOV {fov}")
            
            # Get ROI centers
            roi_folder = os.path.join(basedir, day, animal, fov)
            roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
            if not roi_files:
                print(f"    No ROI files found for {day}, {animal}, {fov}")
                continue
                
            roi_path = os.path.join(roi_folder, roi_files[0])
            try:
                rois = roifile.ImagejRoi.fromfile(roi_path)
                centers = get_roi_centers(rois)
            except Exception as e:
                print(f"    Error reading ROI file: {e}")
                continue
            
            # Process CS+ REWARDED trials
            df_cs_plus = df[(df['day'] == day) & 
                           (df['animal'] == animal) & 
                           (df['fov'] == fov) &
                           (df['cs_type'] == 'CS+')]
            
            if not df_cs_plus.empty:
                # Get unique cell AUC values for rewarded trials
                cell_auc_rewarded = df_cs_plus.groupby('cell_number')['auROC_SucOmit_rewarded'].first().reset_index()
                cell_auc_rewarded = cell_auc_rewarded.dropna(subset=['auROC_SucOmit_rewarded'])
                
                if not cell_auc_rewarded.empty:
                    # Find the cell with the highest rewarded AUC
                    highest_rewarded_row = cell_auc_rewarded.loc[cell_auc_rewarded['auROC_SucOmit_rewarded'].idxmax()]
                    highest_cell_rewarded = int(highest_rewarded_row['cell_number'])
                    
                    if highest_cell_rewarded - 1 < len(centers):
                        highest_center_rewarded = centers[highest_cell_rewarded - 1]
                        distances_rewarded = calculate_distances(highest_center_rewarded, centers, metric_type)
                        distance_mapping_rewarded = {i+1: distances_rewarded[i] for i in range(len(distances_rewarded))}
                        
                        # Update distance column for CS+ trials
                        for idx, row in df_cs_plus.iterrows():
                            cell_num = row['cell_number']
                            if cell_num in distance_mapping_rewarded:
                                df.loc[idx, 'auc_distance_rewarded'] = distance_mapping_rewarded[cell_num]
                
                # Get unique cell AUC values for unrewarded trials
                cell_auc_unrewarded = df_cs_plus.groupby('cell_number')['auROC_SucOmit_unrewarded'].first().reset_index()
                cell_auc_unrewarded = cell_auc_unrewarded.dropna(subset=['auROC_SucOmit_unrewarded'])
                
                if not cell_auc_unrewarded.empty:
                    # Find the cell with the highest unrewarded AUC
                    highest_unrewarded_row = cell_auc_unrewarded.loc[cell_auc_unrewarded['auROC_SucOmit_unrewarded'].idxmax()]
                    highest_cell_unrewarded = int(highest_unrewarded_row['cell_number'])
                    
                    if highest_cell_unrewarded - 1 < len(centers):
                        highest_center_unrewarded = centers[highest_cell_unrewarded - 1]
                        distances_unrewarded = calculate_distances(highest_center_unrewarded, centers, metric_type)
                        distance_mapping_unrewarded = {i+1: distances_unrewarded[i] for i in range(len(distances_unrewarded))}
                        
                        # Update distance column for CS+ trials
                        for idx, row in df_cs_plus.iterrows():
                            cell_num = row['cell_number']
                            if cell_num in distance_mapping_unrewarded:
                                df.loc[idx, 'auc_distance_unrewarded'] = distance_mapping_unrewarded[cell_num]
            
            # Process CS- trials
            df_cs_minus = df[(df['day'] == day) & 
                            (df['animal'] == animal) & 
                            (df['fov'] == fov) &
                            (df['cs_type'] == 'CS-')]
            
            if not df_cs_minus.empty:
                cell_auc_minus = df_cs_minus.groupby('cell_number')['auc_minus'].first().reset_index()
                cell_auc_minus = cell_auc_minus.dropna(subset=['auc_minus'])
                
                if not cell_auc_minus.empty:
                    highest_minus_row = cell_auc_minus.loc[cell_auc_minus['auc_minus'].idxmax()]
                    highest_cell_minus = int(highest_minus_row['cell_number'])
                    
                    if highest_cell_minus - 1 < len(centers):
                        highest_center_minus = centers[highest_cell_minus - 1]
                        distances_minus = calculate_distances(highest_center_minus, centers, metric_type)
                        distance_mapping_minus = {i+1: distances_minus[i] for i in range(len(distances_minus))}
                        
                        for idx, row in df_cs_minus.iterrows():
                            cell_num = row['cell_number']
                            if cell_num in distance_mapping_minus:
                                df.loc[idx, 'auc_distance_minus'] = distance_mapping_minus[cell_num]
    
    print("Distance columns added successfully!")
    
    # Update the original dataframe
    df_master_omission_jp['auc_distance_rewarded'] = df['auc_distance_rewarded']
    df_master_omission_jp['auc_distance_unrewarded'] = df['auc_distance_unrewarded']
    df_master_omission_jp['auc_distance_minus'] = df['auc_distance_minus']
    
    return df_master_omission_jp

def analyze_omission_distance_correlation(basedir, df_master_omission_jp, 
                                         sex='both', normalize=False, 
                                         vis_metric_type='euclidean'):
    """
    Analyze distance correlations for omission test data.
    Creates 3 panels: CS+ Rewarded, CS+ Unrewarded, and CS-.
    """
    
    # Add sex filtering
    if sex != 'both':
        df_master_omission_jp = df_master_omission_jp[df_master_omission_jp['sex'] == sex].copy()
    
    day = '2_SucOmit'
    
    # Define conditions: (condition_name, auc_column, distance_column, cs_type)
    conditions = [
        ('CS+ Rewarded', 'auROC_SucOmit_rewarded', 'auc_distance_rewarded', 'CS+'),
        ('CS+ Unrewarded', 'auROC_SucOmit_unrewarded', 'auc_distance_unrewarded', 'CS+'),
        ('CS-', 'auc_minus', 'auc_distance_minus', 'CS-')
    ]
    
    metrics = {
        'euclidean': 'Euclidean Distance',
        'manhattan': 'Manhattan Distance',
        'chebyshev': 'Chebyshev Distance',
        'combined1': 'Weighted Euclidean-Manhattan',
        'combined2': 'Geometric Mean Distance',
        'combined3': 'Direction-Weighted Distance'
    }
    
    # Update figure title to include sex information
    sex_label = f" ({sex})" if sex != 'both' else ''
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Omission Test: {metrics[vis_metric_type]} Distance Correlation{sex_label}{" (Normalized)" if normalize else ""}', 
                 fontsize=16)
    
    results = []
    
    for cond_idx, (condition_name, auc_column, distance_column, cs_type) in enumerate(conditions):
        print(f"\nProcessing condition: {condition_name}")
        
        # Store scatter object for colorbar later
        scatter_obj = None
        
        for metric_type, metric_name in metrics.items():
            all_distances = []
            all_values = []
            
            animals = df_master_omission_jp[
                (df_master_omission_jp['day'] == day) & 
                (df_master_omission_jp['cs_type'] == cs_type)
            ]['animal'].unique()
            
            for animal in animals:
                fovs = df_master_omission_jp[
                    (df_master_omission_jp['day'] == day) & 
                    (df_master_omission_jp['animal'] == animal) &
                    (df_master_omission_jp['cs_type'] == cs_type)
                ]['fov'].unique()
                
                for fov in fovs:
                    roi_folder = os.path.join(basedir, day, animal, fov)
                    roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                    if not roi_files:
                        continue
                        
                    roi_path = os.path.join(roi_folder, roi_files[0])
                    rois = roifile.ImagejRoi.fromfile(roi_path)
                    centers = get_roi_centers(rois)
                    
                    df_subset = df_master_omission_jp[
                        (df_master_omission_jp['day'] == day) & 
                        (df_master_omission_jp['animal'] == animal) & 
                        (df_master_omission_jp['fov'] == fov) &
                        (df_master_omission_jp['cs_type'] == cs_type)
                    ]
                    
                    # Get unique cell values
                    averaged_values = df_subset.groupby('cell_number')[auc_column].first().reset_index()
                    averaged_values = averaged_values.dropna(subset=[auc_column])
                    
                    if averaged_values.empty:
                        continue
                    
                    if normalize:
                        max_value = averaged_values[auc_column].max()
                        if max_value > 0:
                            averaged_values[auc_column] = averaged_values[auc_column] / max_value

                    highest_value_row = averaged_values.loc[averaged_values[auc_column].idxmax()]
                    highest_cell_number = int(highest_value_row['cell_number'])
                    
                    if highest_cell_number - 1 >= len(centers):
                        continue
                    
                    highest_center = centers[highest_cell_number - 1]
                    fov_center = np.mean(centers, axis=0)
                    distances = calculate_distances(highest_center, centers, metric_type, fov_center)
                    
                    all_distances.extend(distances)
                    all_values.extend(averaged_values[auc_column].values)
            
            if all_distances and all_values:
                all_distances = np.array(all_distances)
                all_values = np.array(all_values)
                
                corr, p_value = pearsonr(all_distances, all_values)
                slope, intercept, r_value, _, _ = linregress(all_distances, all_values)
                
                results.append({
                    'condition': condition_name,
                    'metric': metric_name,
                    'correlation': corr,
                    'p_value': p_value,
                    'n': len(all_values)
                })
                
                if metric_type == vis_metric_type:
                    ax = axes[cond_idx]
                    xy = np.vstack([all_distances, all_values])
                    z = gaussian_kde(xy)(xy)
                    idx = z.argsort()
                    
                    # Create scatter plot and store in scatter_obj
                    scatter_obj = ax.scatter(all_distances[idx], all_values[idx], 
                                         c=z[idx], cmap='plasma', 
                                         alpha=0.5, s=35)
                    
                    x_range = np.array([min(all_distances), max(all_distances)])
                    ax.plot(x_range, slope * x_range + intercept, 
                            color='black', linestyle='--',
                            label=f'r={corr:.2f}')
                    
                    ax.set_title(f'{condition_name}\n{metric_name}\np={p_value:.1e}')
                    ax.set_xlabel('Distance')
                    ylabel = f"{auc_column}"
                    if normalize:
                        ylabel += "\n(Normalized)"
                    ax.set_ylabel(ylabel)
                    ax.grid(True, linestyle='--', alpha=0.7)
                    ax.legend()
                    # Set consistent y-axis limits
                    ax.set_ylim(-1, 1)
            else:
                if metric_type == vis_metric_type:
                    ax = axes[cond_idx]
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        
        # Add colorbar after metric loop completes
        if scatter_obj is not None:
            cbar = plt.colorbar(scatter_obj, ax=axes[cond_idx], shrink=0.8)
            cbar.set_label('Kernel Density', rotation=270, labelpad=15, fontsize=8)
            cbar.ax.tick_params(labelsize=7)
    
    plt.tight_layout()
    
    # Save the figure
    save_path = os.path.join(basedir, 'Results', 'Master_Correlation_Plots', 
                             f"omission_{vis_metric_type}_distance_correlation{'_normalized' if normalize else ''}_{sex}_plasma.pdf")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    results_df = pd.DataFrame(results)
    
    print("\nCorrelation Results Summary:")
    print(results_df)
    
    return results_df

# Example usage
if __name__ == "__main__":
    # First, add the distance columns to df_master_omission_jp
    metric_type = 'combined3'  # Use the same metric as in vis_metric_type
    
    # Update df_master_omission_jp in place with the new distance columns
    df_master_omission_jp = add_omission_distance_column(df_master_omission_jp, basedir, metric_type)
    
    # Save the updated dataframe with distance columns
    results_dir = os.path.join(basedir, 'Results')
    os.makedirs(results_dir, exist_ok=True)
    df_master_omission_jp.to_csv(os.path.join(results_dir, 'df_master_omission_jp_with_distance.csv'), index=False)
    
    print("Updated df_master_omission_jp with distance columns saved!")
    
    # Run the analysis
    results = analyze_omission_distance_correlation(
        basedir, df_master_omission_jp,
        sex='both', normalize=False, 
        vis_metric_type=metric_type
    )

# Quantification of High Lick Trials
## here the column high_licking is added to df_master where every high lick trial (above 1.5hz) has a 1, low lick trials have a 0

In [ ]:
def process_and_plot_high_licking_trials(df_master, days, pre_window_size, infusionframe, baselinefirstframe, baselinelastframe, basedir, sex_selection='all', hz_threshold=1.5, framerate=30, frameaveraging=4):
    """
    Process and plot high licking trials using direct timestamp-based rate calculations
    when available, falling back to frame-based calculations when necessary.
    
    Parameters:
    -----------
    df_master : pandas DataFrame
        Master dataframe containing trial data
    days : list
        List of experimental phases to analyze
    pre_window_size : int
        Number of frames before event onset (default is ~23 for 3*averagedframerate)
    infusionframe : int
        Frame at which infusion occurs (~46 for trace period end + 1)
    baselinefirstframe : int
        First frame of baseline period (default is 0)
    baselinelastframe : int
        Last frame of baseline period (default is 22)
    basedir : str
        Base directory for saving results
    sex_selection : str, default='all'
        Filter by sex ('all', 'male', or 'female')
    hz_threshold : float, default=1.5
        Threshold in Hz (licks/sec) for high vs. low licking classification
    framerate : int, default=30
        Raw framerate before averaging (Hz)
    frameaveraging : int, default=4
        Frames averaged during acquisition
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    
    # Define frame rate from parameters
    averagedframerate = framerate / frameaveraging  # Should be 7.5 Hz with default parameters
    
    # Calculate seconds in each window for rate conversion
    seconds_per_frame = 1.0 / averagedframerate
    baseline_seconds = (baselinelastframe - baselinefirstframe) * seconds_per_frame
    event_seconds = (infusionframe - pre_window_size) * seconds_per_frame
    
    print(f"Frame rate parameters: raw={framerate}Hz, averaging={frameaveraging}, effective={averagedframerate}Hz")
    print(f"Converting to rates: Baseline window = {baseline_seconds:.2f} sec, Event window = {event_seconds:.2f} sec")
    
    # Check if timestamp-based data is available
    has_timestamp_data = 'ts_delta_rate' in df_master.columns
    
    # Create df_unique_trials for accurate statistics and visualizations
    df_unique_trials = df_master.drop_duplicates(
        subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
    ).copy()
    
    # Compute and print statistics for each day and CS type
    daily_stats = []
    
    for day in sorted(days):
        day_data = df_unique_trials[df_unique_trials['day'] == day]
        for cs_type in ['CS+', 'CS-']:
            cs_data = day_data[day_data['cs_type'] == cs_type]
            if cs_data.empty:
                continue
                
            # Determine which delta rate to use (timestamp-based preferred)
            if has_timestamp_data:
                delta_rates = cs_data['ts_delta_rate'].values
                rate_source = "timestamp-based"
            else:
                # Calculate frame-based delta rates
                delta_lick_cols = [col for col in cs_data.columns if col.startswith('delta_licks_frame_')]
                cs_delta_licks = cs_data[delta_lick_cols].values
                
                # Calculate lick counts
                cs_licking_event = np.sum(cs_delta_licks[:, pre_window_size:infusionframe], axis=1)
                cs_licking_baseline = np.sum(cs_delta_licks[:, baselinefirstframe:baselinelastframe], axis=1)
                
                # Convert to rates (Hz)
                cs_event_rate = cs_licking_event / event_seconds
                cs_baseline_rate = cs_licking_baseline / baseline_seconds
                
                # Calculate rate delta
                delta_rates = cs_event_rate - cs_baseline_rate
                rate_source = "frame-based"
            
            # Calculate stats for this day/CS type combination
            daily_stats.append({
                'day': day,
                'cs_type': cs_type,
                'mean': np.mean(delta_rates),
                'median': np.median(delta_rates),
                'n_trials': len(delta_rates),
                'source': rate_source
            })
    
    # Print detailed statistics based on unique trials
    print("\nDetailed Statistics (Rate-Based, Unique Trials):")
    print("=" * 60)
    for stat in daily_stats:
        print(f"Day {stat['day']} {stat['cs_type']} ({stat['source']}):")
        print(f"  Mean licking delta rate: {stat['mean']:.2f} Hz")
        print(f"  Median licking delta rate: {stat['median']:.2f} Hz")
        print(f"  Number of trials: {stat['n_trials']}")
        print("-" * 30)
    
    # Set threshold for high licking in Hz
    print(f"\nRate threshold = {hz_threshold:.2f} Hz")
    print("=" * 60)
    
    # Plot setup
    numdays = len(days)
    num_rows = 2  # Just CS+ and CS- for any selection
    fig, axs = plt.subplots(num_rows, numdays, figsize=(numdays*3, 8), sharex=True, sharey=False)
    if numdays == 1:  # Handle the case of single day
        axs = axs.reshape(-1, 1)
    sns.set_style('white')
    colors = ['deeppink', 'gray']  # Updated color scheme
    
    # Ensure high_licking column exists in df_master
    if 'high_licking' not in df_master.columns:
        df_master['high_licking'] = 0
    
    # Determine which method to use for high licking classification
    if has_timestamp_data:
        print("Using timestamp-based delta rates for high licking classification")
        # Copy the timestamp-based classification to high_licking column
        df_master['high_licking'] = df_master['ts_high_licking']
    else:
        print("Using frame-based delta rates for high licking classification")
        # Process frame-based data and update df_master
        for day in sorted(days):
            # Create a temp df for rate calculations
            df_temp = df_master[df_master['day'] == day].copy()
            
            for cs_type in ['CS+', 'CS-']:
                # Filter data
                cs_mask = df_temp['cs_type'] == cs_type
                if not any(cs_mask):
                    continue
                    
                delta_lick_cols = [col for col in df_temp.columns if col.startswith('delta_licks_frame_')]
                
                # Get lick data for these trials
                lick_data = df_temp.loc[cs_mask, delta_lick_cols].values
                
                # Calculate lick counts then rates
                licking_event = np.sum(lick_data[:, pre_window_size:infusionframe], axis=1)
                licking_baseline = np.sum(lick_data[:, baselinefirstframe:baselinelastframe], axis=1)
                
                event_rate = licking_event / event_seconds
                baseline_rate = licking_baseline / baseline_seconds
                
                rate_delta = event_rate - baseline_rate
                
                # Set high_licking flags
                licking_flag = rate_delta > hz_threshold
                df_master.loc[(df_master['day'] == day) & (df_master['cs_type'] == cs_type), 'high_licking'] = licking_flag.astype(int)
    
    # Get fresh unique trials with updated high_licking values
    df_unique_trials = df_master.drop_duplicates(
        subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
    ).copy()
    
    # Create pie charts for each day and CS type
    for d, day in enumerate(sorted(days)):
        for cs_type, cs_row_index in zip(['CS+', 'CS-'], [0, 1]):
            ax = axs[cs_row_index, d]
            
            if sex_selection == 'all':
                plot_data = df_unique_trials[(df_unique_trials['day'] == day) & 
                                         (df_unique_trials['cs_type'] == cs_type)]
            else:
                plot_data = df_unique_trials[(df_unique_trials['day'] == day) & 
                                         (df_unique_trials['cs_type'] == cs_type) & 
                                         (df_unique_trials['sex'] == sex_selection)]
            
            if not plot_data.empty:
                high_licking_count = plot_data['high_licking'].sum()
                total_trials = len(plot_data)
                
                highlicking_percentage = high_licking_count / total_trials
                lowlicking_percentage = 1 - highlicking_percentage
                
                counts = np.array([highlicking_percentage, lowlicking_percentage])
                labels = [f"High: {high_licking_count}/{total_trials}\n({highlicking_percentage*100:.1f}%)", 
                          f"Low: {total_trials-high_licking_count}/{total_trials}\n({lowlicking_percentage*100:.1f}%)"]
                
                ax.pie(counts, colors=colors, labels=labels)
            else:
                ax.text(0.5, 0.5, "No data", ha='center', va='center', transform=ax.transAxes, fontsize=12)
                ax.set_frame_on(False)
            
            if d == 0:
                ax.set_ylabel(f'{cs_type}', fontsize=12)
            ax.set_title(day.replace('_', ' '), fontsize=12)
    
    method_text = "Timestamp-based" if has_timestamp_data else "Frame-based"
    title_suffix = ' - All' if sex_selection == 'all' else f' - {sex_selection.capitalize()}'
    fig.suptitle(f'High Licking Trials ({method_text}, Threshold = {hz_threshold:.1f} Hz){title_suffix}', fontsize=16)
    fig.tight_layout()
    
    # Save figures
    # suffix = '' if sex_selection == 'all' else f'_{sex_selection}'
    # method_suffix = '_TS' if has_timestamp_data else '_FB'
    # fig.savefig(os.path.join(basedir, 'Results', f'HighLickingTrials{method_suffix}_PieCharts{suffix}_6-20-25.PNG'), format='PNG')
    # fig.savefig(os.path.join(basedir, 'Results', f'HighLickingTrials{method_suffix}_PieCharts{suffix}_6-20-25.PNG.PDF'), format='PDF')
    plt.show()
    
    # Create classification mapping for SDT metrics (similar to classify_response in paste-2.txt)
    def classify_response(day, cs_type, is_high_licking):
        # Check if we're in a reversal phase
        is_reversal = day.startswith('4_Rev') or day.startswith('5_Rev')
        
        if not is_reversal:
            # Initial learning phases
            if cs_type == 'CS+':
                return 'correct response' if is_high_licking else 'incorrect withholding'
            else:  # CS-
                return 'incorrect response' if is_high_licking else 'correct withholding'
        else:
            # Reversal phases
            if cs_type == 'CS+':
                return 'incorrect response' if is_high_licking else 'correct withholding'
            else:  # CS-
                return 'correct response' if is_high_licking else 'incorrect withholding'
    
    # Add response classification to df_master
    df_master['response_type'] = df_master.apply(
        lambda row: classify_response(row['day'], row['cs_type'], bool(row['high_licking'])),
        axis=1
    )
    
    # Print overall SDT classification results
    print("\nSignal Detection Theory Classification:")
    print("=" * 60)
    
    for day in sorted(days):
        # Get unique trials for this day
        day_trials = df_unique_trials[df_unique_trials['day'] == day].copy()
        
        if not day_trials.empty:
            day_trials['response_type'] = day_trials.apply(
                lambda row: classify_response(row['day'], row['cs_type'], bool(row['high_licking'])),
                axis=1
            )
            
            # Count each response type
            response_counts = day_trials['response_type'].value_counts()
            total_trials = len(day_trials)
            
            # Calculate percentages
            response_percentages = (response_counts / total_trials * 100).round(1)
            
            print(f"\nDay {day}:")
            print(f"Total trials: {total_trials}")
            
            # Print each response type
            for resp_type in ['correct response', 'correct withholding', 
                             'incorrect response', 'incorrect withholding']:
                if resp_type in response_counts:
                    count = response_counts[resp_type]
                    percent = response_percentages[resp_type]
                    print(f"  {resp_type}: {count} trials ({percent}%)")
                else:
                    print(f"  {resp_type}: 0 trials (0.0%)")
    
    return df_master

# Usage
df_master = process_and_plot_high_licking_trials(df_master, days, pre_window_size, infusionframe, 
                                                baselinefirstframe, baselinelastframe, basedir, 
                                                sex_selection='all',
                                                hz_threshold=1.5,
                                                framerate=30, 
                                                frameaveraging=4)

<div style="font-family: Arial, sans-serif; border: 2px solid #4CAF50; border-radius: 10px; padding: 20px; background-color: #f9f9f9; color: #333;">
    <h1 style="color: #4CAF50;">Signal Detection Theory (SDT) in Behavioral Analysis</h1>
    <p style="font-size: 1.2em; line-height: 1.6;">
        SDT is a framework for understanding decision-making under uncertainty. It helps analyze how subjects discriminate between signal (when to respond) and noise (when not to respond).
    </p>
    <ul style="font-size: 1em; line-height: 1.8; margin-left: 20px;">
        <li>
            <strong>Basic Setup:</strong>
            <ul>
                <li><code>Signal Present</code> - Go trial/CS+ where subject should respond</li>
                <li><code>Signal Absent</code> - No-Go trial/CS- where subject should not respond</li>
            </ul>
        </li>
        <li>
            <strong>Four Possible Outcomes:</strong>
            <ul>
                <li><code>Hit</code> - Correctly responding when signal is present (licking during CS+)</li>
                <li><code>Miss</code> - Failing to respond when signal is present (not licking during CS+)</li>
                <li><code>False Alarm</code> - Incorrectly responding when signal is absent (licking during CS-)</li>
                <li><code>Correct Rejection</code> - Correctly not responding when signal is absent (not licking during CS-)</li>
            </ul>
        </li>
        <li>
            <strong>Key Metrics:</strong>
            <ul>
                <li><code>d' (d-prime)</code>:
                    <ul>
                        <li>Measures sensitivity - how well subject discriminates signal from noise</li>
                        <li>Higher d' indicates better discrimination</li>
                        <li>d' = 0 means no discrimination ability</li>
                        <li>Calculated using difference between normalized hit rate and false alarm rate</li>
                    </ul>
                </li>
                <li><code>β (beta)</code>:
                    <ul>
                        <li>Measures response bias - tendency to respond or withhold</li>
                        <li>β > 1: Conservative bias (tends to withhold)</li>
                        <li>β < 1: Liberal bias (tends to respond)</li>
                        <li>β = 1: No bias</li>
                    </ul>
                </li>
            </ul>
        </li>
        <li>
            <strong>Application to Your Experiment:</strong>
            <ul>
                <li><code>Early Training</code>:
                    <ul>
                        <li>CS+ is signal (should lick)</li>
                        <li>CS- is noise (shouldn't lick)</li>
                    </ul>
                </li>
                <li><code>Reversal</code>:
                    <ul>
                        <li>CS- becomes signal</li>
                        <li>CS+ becomes noise</li>
                    </ul>
                </li>
                <li><code>Metrics Interpretation</code>:
                    <ul>
                        <li>d' shows discrimination learning</li>
                        <li>β reveals response tendencies</li>
                    </ul>
                </li>
            </ul>
        </li>
    </ul>
</div>

In [ ]:
def add_gng_columns(df_master):

    # Initialize new columns with 0
    new_columns = ['hits', 'correct_rejections', 'misses', 'false_alarms']
    for col in new_columns:
        df_master[col] = 0
    
    # Define day groups
    early_days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit']
    rev_days = ['4_RevEarly', '5_RevLate']
    
    # Process each row
    for index, row in df_master.iterrows():
        day = row['day']
        cs_type = row['cs_type']
        high_licking = row['high_licking']

        # Early training days (CS+ is go cue, CS- is no-go cue)
        if day in early_days:
            if cs_type == 'CS+':
                if high_licking == 1:
                    df_master.at[index, 'hits'] = 1
                else:
                    df_master.at[index, 'misses'] = 1
            else:  # CS-
                if high_licking == 0:
                    df_master.at[index, 'correct_rejections'] = 1
                else:
                    df_master.at[index, 'false_alarms'] = 1

        # Reversal days (CS- is go cue, CS+ is no-go cue)
        else:  # rev_days
            if cs_type == 'CS-':
                if high_licking == 1:
                    df_master.at[index, 'hits'] = 1
                else:
                    df_master.at[index, 'misses'] = 1
            else:  # CS+
                if high_licking == 0:
                    df_master.at[index, 'correct_rejections'] = 1
                else:
                    df_master.at[index, 'false_alarms'] = 1
    
    return df_master

# Apply the function
df_master = add_gng_columns(df_master)

def print_gng_stats(df_master):
    """
    Print Go/No-Go statistics by day, using unique trials only
    """
    print("\nGo/No-Go Statistics by Day (Unique Trials)")
    print("=" * 80)
    
    for day in sorted(df_master['day'].unique()):
        print(f"\nDay: {day}")
        print("-" * 40)
        
        # Get data for this day
        day_data = df_master[df_master['day'] == day]
        
        # Create unique trials dataframe
        unique_trials = day_data.drop_duplicates(
            subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
        ).copy()
        
        # Get unique CS+ and CS- trials
        csplus_trials = unique_trials[unique_trials['cs_type'] == 'CS+']
        csminus_trials = unique_trials[unique_trials['cs_type'] == 'CS-']
        
        # Calculate total unique trials
        total_unique_trials = len(unique_trials)
        
        if total_unique_trials > 0:
            # Calculate stats for each measure using unique trials
            hits = unique_trials['hits'].sum()
            crs = unique_trials['correct_rejections'].sum()
            misses = unique_trials['misses'].sum()
            fas = unique_trials['false_alarms'].sum()
            
            # Get CS+ and CS- totals
            total_csplus = len(csplus_trials)
            total_csminus = len(csminus_trials)
            
            # Calculate overall percentages
            hit_rate = hits / total_unique_trials * 100
            cr_rate = crs / total_unique_trials * 100
            miss_rate = misses / total_unique_trials * 100
            fa_rate = fas / total_unique_trials * 100
            
            # Calculate CS type-specific percentages
            csplus_hit_rate = hits / total_csplus * 100 if total_csplus > 0 else 0
            csplus_miss_rate = misses / total_csplus * 100 if total_csplus > 0 else 0
            csminus_cr_rate = crs / total_csminus * 100 if total_csminus > 0 else 0
            csminus_fa_rate = fas / total_csminus * 100 if total_csminus > 0 else 0
            
            print(f"Total unique trials: {total_unique_trials} (CS+: {total_csplus}, CS-: {total_csminus})")
            print(f"Hits              : {hits:5d} trials ({hit_rate:5.1f}% of all, {csplus_hit_rate:5.1f}% of CS+)")
            print(f"Correct Rejections: {crs:5d} trials ({cr_rate:5.1f}% of all, {csminus_cr_rate:5.1f}% of CS-)")
            print(f"Misses            : {misses:5d} trials ({miss_rate:5.1f}% of all, {csplus_miss_rate:5.1f}% of CS+)")
            print(f"False Alarms      : {fas:5d} trials ({fa_rate:5.1f}% of all, {csminus_fa_rate:5.1f}% of CS-)")
            
            # Calculate d' (sensitivity index) if possible
            if csplus_hit_rate > 0 and csminus_fa_rate > 0:
                import scipy.stats as stats
                
                # Convert percentages to proportions
                hit_prop = csplus_hit_rate / 100
                fa_prop = csminus_fa_rate / 100
                
                # Adjust extreme values to avoid infinite d'
                hit_prop = min(max(hit_prop, 0.01), 0.99)
                fa_prop = min(max(fa_prop, 0.01), 0.99)
                
                # Calculate d'
                d_prime = stats.norm.ppf(hit_prop) - stats.norm.ppf(fa_prop)
                print(f"Sensitivity (d'): {d_prime:.2f}")
        else:
            print("No trials found for this day")

# Print the stats
print_gng_stats(df_master)

<div style="font-family: Arial, sans-serif; border: 2px solid #4CAF50; border-radius: 10px; padding: 20px; background-color: #f9f9f9; color: #333;">
    <h1 style="color: #4CAF50;">Standardized Signal Detection Theory (SDT) Response Categories</h1>
    <p style="font-size: 1.2em; line-height: 1.6;">
        Signal Detection Theory uses standardized terminology to categorize responses in decision-making tasks. These categories are used across various fields including perception research, medical diagnostics, and behavioral studies.
    </p>
    <ul style="font-size: 1em; line-height: 1.8; margin-left: 20px;">
        <li>
            <strong>Signal Present (Go) Responses:</strong>
            <ul>
                <li><code>Hits</code> 
                    <ul>
                        <li>Correctly responding when signal is present</li>
                        <li>Example: Licking when reward will come</li>
                        <li>Also known as: True Positive</li>
                    </ul>
                </li>
                <li><code>Misses</code>
                    <ul>
                        <li>Failing to respond when signal is present</li>
                        <li>Example: Not licking when reward will come</li>
                        <li>Also known as: False Negative</li>
                    </ul>
                </li>
            </ul>
        </li>
        <li>
            <strong>Signal Absent (No-Go) Responses:</strong>
            <ul>
                <li><code>False Alarms</code>
                    <ul>
                        <li>Incorrectly responding when signal is absent</li>
                        <li>Example: Licking when no reward will come</li>
                        <li>Also known as: False Positive</li>
                    </ul>
                </li>
                <li><code>Correct Rejections</code>
                    <ul>
                        <li>Correctly not responding when signal is absent</li>
                        <li>Example: Not licking when no reward will come</li>
                        <li>Also known as: True Negative</li>
                    </ul>
                </li>
            </ul>
        </li>
        <li>
            <strong>Historical Context:</strong>
            <ul>
                <li><code>Origins</code>
                    <ul>
                        <li>Developed during World War II for radar operator performance</li>
                        <li>Later adopted in psychological and perceptual research</li>
                        <li>Now used across many scientific disciplines</li>
                    </ul>
                </li>
            </ul>
        </li>
        <li>
            <strong>Modern Applications:</strong>
            <ul>
                <li><code>Research Fields</code>
                    <ul>
                        <li>Human Perception Studies</li>
                        <li>Medical Diagnostics</li>
                        <li>Machine Learning</li>
                        <li>Animal Behavior Research</li>
                        <li>Quality Control</li>
                    </ul>
                </li>
            </ul>
        </li>
    </ul>
</div>

# Correcting the trial number to be accurate

In [ ]:
####THE CODE IS CALCULATING DELTA LICK RATE BASED OFF THE 3 SEC BL PRE CUE AND THEN the 3 SEC CUE PERIOD

def plot_gng_types(df_master, days, basedir, sex_selection='all'):
    numdays = len(days)
    fig, axs = plt.subplots(1, numdays, figsize=(numdays*3, 5), sharex=True, sharey=False)
    sns.set_style('white')
    
    # Colors for CS+ and CS- responses
    colors_csplus = ['lawngreen',  # hits
                    'black']     # misses
    colors_csminus = ['palegreen',   # correct rejections
                     'grey']   # false alarms
    colors = colors_csplus + colors_csminus
    
    for d, day in enumerate(sorted(days)):
        # Filter data based on sex selection
        if sex_selection == 'all':
            day_data = df_master[df_master['day'] == day]
        else:
            day_data = df_master[(df_master['day'] == day) & 
                                 (df_master['sex'] == sex_selection)]
        
        # Create unique trials dataframe to get accurate trial counts
        plot_data = day_data.drop_duplicates(
            subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
        ).copy()
        
        # Calculate totals for each response type
        total_trials = len(plot_data)
        if total_trials > 0:
            # Get unique CS+ and CS- trials
            csplus_data = plot_data[plot_data['cs_type'] == 'CS+']
            csminus_data = plot_data[plot_data['cs_type'] == 'CS-']
            
            # Calculate response types based on high_licking column
            hits = csplus_data['high_licking'].sum()
            misses = len(csplus_data) - hits
            fas = csminus_data['high_licking'].sum()
            crs = len(csminus_data) - fas
            
            # Calculate CS+ and CS- totals
            total_csplus = len(csplus_data)
            total_csminus = len(csminus_data)
            
            # Calculate normalized percentages (50% for each CS type)
            if total_csplus > 0:
                hits_pct = (hits / total_csplus) * 50
                misses_pct = (misses / total_csplus) * 50
            else:
                hits_pct = misses_pct = 0
                
            if total_csminus > 0:
                crs_pct = (crs / total_csminus) * 50
                fas_pct = (fas / total_csminus) * 50
            else:
                crs_pct = fas_pct = 0
            
            counts = [hits_pct, misses_pct, crs_pct, fas_pct]
            labels = [f"Correct Response: {hits_pct:.1f}%\n({hits})",
                     f"Incorrect Withholding: {misses_pct:.1f}%\n({misses})",
                     f"Correct Withholding: {crs_pct:.1f}%\n({crs})",
                     f"Incorrect Response: {fas_pct:.1f}%\n({fas})"]
        else:
            counts = [0, 0, 0, 1]
            labels = ["No data", "", "", ""]
        
        # Create pie chart with CS+/CS- split
        ax = axs[d]
        wedges, texts = ax.pie(counts, colors=colors, labels=labels)
        
        # Add a line to visually separate CS+ and CS- halves
        ax.axhline(y=0, color='white', linewidth=3)
        
        # Add titles
        ax.set_title(day)
    
    # Add legend with CS+/CS- grouping
    legend_labels = ['Correct Licking', 
                    'Incorrect Withholding',
                    'Correct Withholding',
                    'Incorrect Licking']
    fig.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0.5))
    
    # Add overall title
    title_suffix = ' - All' if sex_selection == 'all' else f' - {sex_selection.capitalize()}'
    fig.suptitle(f'CS+/CS- Response Types by Day{title_suffix}', fontsize=16)
    
    # Adjust layout to accommodate legend
    plt.subplots_adjust(right=0.85)
    
    # Save figures
    suffix = '' if sex_selection == 'all' else f'_{sex_selection}'
    fig.savefig(os.path.join(basedir, 'Results', f'GNG_ResponseTypes_PieCharts{suffix}_6-20-25.PNG'), 
                format='PNG', bbox_inches='tight')
    fig.savefig(os.path.join(basedir, 'Results', f'GNG_ResponseTypes_PieCharts{suffix}_6-20-25.PDF'), 
                format='PDF', bbox_inches='tight')
    plt.show()


# # Usage example:
plot_gng_types(df_master, days, basedir, sex_selection='all')

plot_gng_types(df_master, days, basedir, sex_selection='male')

plot_gng_types(df_master, days, basedir, sex_selection='female')

# Calculate key SDT metrics:
### d' (sensitivity): How well the animal discriminates between Go and No-Go cues
### β (response bias): The animal's tendency to respond or withhold
### Hit Rate: Proportion of correct responses to Go cues
### False Alarm Rate: Proportion of incorrect responses to No-Go cues
### Overall Accuracy: Proportion of correct decisions

# Create visualizations to show:
### How d' changes across days (higher is better discrimination)
### How β changes (values near 1 indicate no bias)
### Overall accuracy trends
### ROC-like plot comparing Hit Rate to False Alarm Rate

# The metrics are particularly useful because:
### d' tells us how well the animal can discriminate between cues
### β tells us if there's a bias toward responding or withholding
### The HR vs FAR plot shows the trade-off between correct and incorrect responses

In [ ]:
import numpy as np
from scipy.stats import norm
import pandas as pd
import matplotlib.pyplot as plt

def calculate_sdt_metrics(df_master, by_day=True):
    """
    Calculate Signal Detection Theory metrics with SEM based on unique trial counts
    """
    def compute_metrics(data):
        # Get unique trials first
        unique_trials = data.drop_duplicates(
            subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
        ).copy()
        
        # Get total counts
        total_go_trials = unique_trials['hits'].sum() + unique_trials['misses'].sum()
        total_nogo_trials = unique_trials['false_alarms'].sum() + unique_trials['correct_rejections'].sum()
        
        # Calculate rates and their SEMs
        hit_rate = unique_trials['hits'].sum() / total_go_trials if total_go_trials > 0 else 0
        fa_rate = unique_trials['false_alarms'].sum() / total_nogo_trials if total_nogo_trials > 0 else 0
        
        # Calculate SEMs for rates
        hit_rate_sem = np.sqrt((hit_rate * (1 - hit_rate)) / total_go_trials) if total_go_trials > 0 else 0
        fa_rate_sem = np.sqrt((fa_rate * (1 - fa_rate)) / total_nogo_trials) if total_nogo_trials > 0 else 0
        
        # Adjust rates of 0 or 1 to prevent infinite d'
        hit_rate_adj = np.clip(hit_rate, 0.01, 0.99)
        fa_rate_adj = np.clip(fa_rate, 0.01, 0.99)
        
        # Calculate d' (d-prime)
        d_prime = norm.ppf(hit_rate_adj) - norm.ppf(fa_rate_adj)
        
        # Calculate β (beta)
        beta = np.exp((norm.ppf(fa_rate_adj)**2 - norm.ppf(hit_rate_adj)**2) / 2)
        
        # Calculate d' SEM using error propagation
        d_prime_sem = np.sqrt(
            (hit_rate_sem / (hit_rate_adj * (1-hit_rate_adj)) * norm.pdf(norm.ppf(hit_rate_adj)))**2 +
            (fa_rate_sem / (fa_rate_adj * (1-fa_rate_adj)) * norm.pdf(norm.ppf(fa_rate_adj)))**2
        )
        
        # Calculate accuracy and its SEM
        total_trials = len(unique_trials)
        correct_trials = unique_trials['hits'].sum() + unique_trials['correct_rejections'].sum()
        accuracy = correct_trials / total_trials if total_trials > 0 else 0
        accuracy_sem = np.sqrt((accuracy * (1 - accuracy)) / total_trials) if total_trials > 0 else 0
        
        return pd.Series({
            'hit_rate': hit_rate,
            'hit_rate_sem': hit_rate_sem,
            'false_alarm_rate': fa_rate,
            'false_alarm_rate_sem': fa_rate_sem,
            'd_prime': d_prime,
            'd_prime_sem': d_prime_sem,
            'beta': beta,
            'beta_sem': np.nan,  # Beta SEM is complex, could be added if needed
            'accuracy': accuracy,
            'accuracy_sem': accuracy_sem,
            'n_trials': total_trials
        })
    
    # Handle empty dataframes or other edge cases
    if df_master.empty:
        print("Warning: Empty dataframe provided to calculate_sdt_metrics")
        return pd.DataFrame()
        
    if by_day:
        # Check if we have data for multiple days
        if len(df_master['day'].unique()) == 0:
            print("Warning: No days found in data")
            return pd.DataFrame()
            
        # Apply the metrics calculation for each day
        try:
            results = df_master.groupby('day').apply(compute_metrics)
            # Check if results is empty
            if results.empty:
                print("Warning: No valid results calculated")
                return pd.DataFrame()
            return results
        except Exception as e:
            print(f"Error in SDT calculation: {e}")
            # Return an empty DataFrame instead of None
            return pd.DataFrame()
    else:
        try:
            # Calculate metrics for the entire dataset
            return pd.DataFrame([compute_metrics(df_master)]) 
        except Exception as e:
            print(f"Error in SDT calculation: {e}")
            return pd.DataFrame()

def plot_sdt_metrics(sdt_metrics):
    """
    Create visualization of SDT metrics with error bars
    """
    # Check if we have valid metrics to plot
    if sdt_metrics.empty:
        print("No valid metrics available for plotting")
        return
        
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
    x = range(len(sdt_metrics))
    
    # Convert pandas Series to numpy arrays to prevent multi-dimensional indexing error
    d_prime_values = np.array(sdt_metrics['d_prime'])
    d_prime_sem_values = np.array(sdt_metrics['d_prime_sem'])
    beta_values = np.array(sdt_metrics['beta'])
    beta_sem_values = np.array(sdt_metrics['beta_sem'])
    accuracy_values = np.array(sdt_metrics['accuracy'])
    accuracy_sem_values = np.array(sdt_metrics['accuracy_sem'])
    hit_rate_values = np.array(sdt_metrics['hit_rate'])
    hit_rate_sem_values = np.array(sdt_metrics['hit_rate_sem'])
    false_alarm_rate_values = np.array(sdt_metrics['false_alarm_rate'])
    false_alarm_rate_sem_values = np.array(sdt_metrics['false_alarm_rate_sem'])
    
    # Plot d' with SEM
    ax1.errorbar(x, d_prime_values, yerr=d_prime_sem_values,
                fmt='o-', capsize=5, capthick=1, elinewidth=1)
    ax1.set_xticks(x)
    ax1.set_xticklabels(sdt_metrics.index, rotation=45)
    ax1.set_title("d' (Sensitivity)")
    ax1.set_ylabel("d'")
    
    # Plot β with SEM
    ax2.errorbar(x, beta_values, yerr=beta_sem_values,
                fmt='o-', capsize=5, capthick=1, elinewidth=1)
    ax2.set_xticks(x)
    ax2.set_xticklabels(sdt_metrics.index, rotation=45)
    ax2.set_title("β (Response Bias)")
    ax2.set_ylabel("β")
    
    # Plot accuracy with SEM
    ax3.errorbar(x, accuracy_values, yerr=accuracy_sem_values,
                fmt='o-', capsize=5, capthick=1, elinewidth=1)
    ax3.set_xticks(x)
    ax3.set_xticklabels(sdt_metrics.index, rotation=45)
    ax3.set_title("Overall Accuracy")
    ax3.set_ylabel("Accuracy")
    
    # Plot ROC-like comparison with SEM
    ax4.errorbar(false_alarm_rate_values, hit_rate_values,
                xerr=false_alarm_rate_sem_values,
                yerr=hit_rate_sem_values,
                fmt='o', capsize=5, capthick=1, elinewidth=1)
    for i, day in enumerate(sdt_metrics.index):
        ax4.annotate(day, 
                    (false_alarm_rate_values[i], hit_rate_values[i]))
    ax4.plot([0, 1], [0, 1], 'k--', alpha=0.5)  # Add diagonal line
    ax4.set_xlabel("False Alarm Rate")
    ax4.set_ylabel("Hit Rate")
    ax4.set_title("Hit Rate vs False Alarm Rate")
    
    plt.tight_layout()
    plt.show()

def print_sdt_metrics(sdt_metrics):
    """
    Print formatted SDT metrics with SEM
    """
    # Check if we have valid metrics to print
    if sdt_metrics.empty:
        print("No valid metrics available to print")
        return
        
    print("\nSignal Detection Theory Metrics by Day (Mean ± SEM)")
    print("=" * 80)
    metrics = ['d_prime', 'beta', 'accuracy', 'hit_rate', 'false_alarm_rate']
    
    for metric in metrics:
        print(f"\n{metric}:")
        for day in sdt_metrics.index:
            mean = sdt_metrics.loc[day, metric]
            sem = sdt_metrics.loc[day, f"{metric}_sem"]
            if not np.isnan(sem):
                print(f"{day}: {mean:.3f} ± {sem:.3f}")
            else:
                print(f"{day}: {mean:.3f}")

# Calculate SDT metrics
sdt_metrics = calculate_sdt_metrics(df_master)

# Display results
plot_sdt_metrics(sdt_metrics)
print_sdt_metrics(sdt_metrics)

<div style="font-family: Arial, sans-serif; border: 2px solid #4CAF50; border-radius: 10px; padding: 20px; background-color: #f9f9f9; color: #333;">
    <h1 style="color: #4CAF50;">Using <code>df_master</code> to Generate Heatmaps, Lineplots and Licking Plots</h1>
    <p style="font-size: 1.2em; line-height: 1.6;">
        Select <strong>combinations</strong> of filtering options carefully to tailor your visualization. Use the parameters below to adjust the data type, trial averaging, sex filtering, lick trial type, and sorting order.
    </p>
    <ul style="font-size: 1em; line-height: 1.8; margin-left: 20px;">
        <li>
            <strong>Data Type:</strong> 
            Set <code>neurons_to_plot</code> to:
            <ul>
                <li><code>'significant'</code> - Only significant neurons</li>
                <li><code>'notsignificant'</code> - Non-significant neurons</li>
                <li><code>'all'</code> - Include all neurons</li>
            </ul>
        </li>
        <li>
            <strong>Trial Averaging:</strong> 
            Decide whether to average across trials:
            <ul>
                <li><code>True</code> - Average trials for each neuron</li>
                <li><code>False</code> - Plot all trials individually</li>
            </ul>
        </li>
        <li>
            <strong>Animal Sex Filtering:</strong> 
            Select animals by sex:
            <ul>
                <li><code>'male'</code> - Male animals only</li>
                <li><code>'female'</code> - Female animals only</li>
                <li><code>'both'</code> - Include both sexes</li>
            </ul>
        </li>
        <li>
            <strong>Lick Trial Filtering:</strong> 
            Filter trials based on licking behavior:
            <ul>
                <li><code>'all'</code> - Include all licking trials</li>
                <li><code>'high'</code> - Only high-licking trials</li>
                <li><code>'low'</code> - Only low-licking trials</li>
            </ul>
        </li>
        <li>
            <strong>Tracked Cells (Simple Tracking ie., all cells here are tracked but not across all days):</strong> 
            Filter by ROI tracking:
            <ul>
                <li><code>'yes'</code> - Only tracked cells</li>
                <li><code>'no'</code> - Include all cells</li>
            </ul>
        </li>
        <li>
            <strong>Sorting Order:</strong> 
            Sort trials by:
            <ul>
                <li><code>'mean'</code> - Sort by mean neural response</li>
                <li><code>'timestamp'</code> - Sort by trial timestamp (only when <code>average_trials=False</code>)</li>
            </ul>
        </li>
    </ul>

</div>

# Heatmaps with Added Response Type Filtering (behavior_filter)
## Multiple Filters Can be Simultaneously Applied

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def visualize_master_data(df_master, days, pre_window_size, infusionframe, alphalevel, 
                         baselinefirstframe, baselinelastframe, neurons_to_plot='all',
                         average_trials=True, sex='both', sort_by='mean', lick='all', 
                         tracked='no', sucomit_filter='all', behavior_filter='all',
                         z_score_data=False):
    """
    Visualize neural responses for CS+ and CS- trials.
    Parameters:
        neurons_to_plot: str, 'significant', 'notsignificant', or 'all'
        average_trials: bool, whether to average across trials
        sex: str, 'both', 'male', or 'female'
        sort_by: str, 'mean' or 'timestamp'
        lick: str, 'all', 'high', or 'low'
        tracked: str, 'yes' or 'no' for tracked ROIs
        sucomit_filter: str, 'all', 'rewarded', or 'unrewarded' for SucOmit CS+ trials
        behavior_filter: str, 'all', 'hits', 'misses', 'correct_rejections', or 'false_alarms'
        z_score_data: bool, whether to z-score the fluorescence data based on baseline period
    """
    # Sex filtering
    if sex.lower() != 'both':
        df_master = df_master[df_master['sex'].str.lower() == sex.lower()]
        print(f"Filtered for {sex} animals")
    else:
        print("Including both male and female animals")

    # Add lick filtering
    if lick.lower() == 'high':
        df_master = df_master[df_master['high_licking'] == 1]
        print("Filtered for high-licking trials")
    elif lick.lower() == 'low':
        df_master = df_master[df_master['high_licking'] == 0]
        print("Filtered for low-licking trials")
    else:
        print("Including all licking trials")
    
    # Add tracked ROI filtering
    if tracked.lower() == 'yes':
        df_master = df_master[df_master['tracked_roi_counter'].notna()]
        print("Filtered for tracked ROIs only")
    
    # Add behavioral response filtering
    valid_behaviors = ['hits', 'misses', 'correct_rejections', 'false_alarms']
    if behavior_filter.lower() in valid_behaviors:
        df_master = df_master[df_master[behavior_filter.lower()] == 1]
        print(f"Filtered for {behavior_filter} trials")
    elif behavior_filter.lower() != 'all':
        print(f"Warning: Invalid behavior filter '{behavior_filter}'. Using all trials.")
    else:
        print("Including all behavioral response types")
    
    # Add SucOmit filtering
    if sucomit_filter.lower() in ['rewarded', 'unrewarded']:
        mask = (df_master['day'] == '2_SucOmit') & (df_master['cs_type'] == 'CS+')
        if mask.any():
            if sucomit_filter.lower() == 'rewarded':
                df_master = df_master[~mask | (mask & (df_master['reward_status'] == 'rewarded'))]
                print("Filtered for rewarded trials in SucOmit CS+")
            else:
                df_master = df_master[~mask | (mask & (df_master['reward_status'] == 'unrewarded'))]
                print("Filtered for unrewarded trials in SucOmit CS+")
    
    print("\nAnimal counts by sex:")
    print(df_master.groupby('sex')['animal'].nunique())

    # # Print behavioral response counts
    # print("\nBehavioral response counts:")
    # behavior_counts = {
    #     'Hits': df_master['hits'].sum(),
    #     'Misses': df_master['misses'].sum(),
    #     'Correct Rejections': df_master['correct_rejections'].sum(),
    #     'False Alarms': df_master['false_alarms'].sum()
    # }
    # for behavior, count in behavior_counts.items():
    #     print(f"{behavior}: {count}")

    # numdays = len(days)
    
    fig, axs = plt.subplots(numdays, 5, figsize=(20, numdays*4), sharex=True, sharey=False)
    if numdays == 1:
        axs = axs.reshape(1, -1)
    sns.set_style('white')
    
    # Set visualization limits based on whether we're z-scoring
    if z_score_data:
        cmax, cmin = 6, -6  # Z-scored data has wider range
        ymax, ymin = 7, -7
        print('Data Z-scored for visualization')
    else:
        cmax, cmin = 1, -1  # Raw data typically in smaller range
        ymax, ymin = 1.25, -0.25
    
    lickmin, lickmax = -1, 12
    
    for d, day in enumerate(sorted(days)):
        day_data = df_master[df_master['day'] == day]
        
        fluor_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        fluor_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        if average_trials:
            cell_data = day_data.groupby('cell_id').agg({
                'pval_plus': 'min',
                'pval_minus': 'min',
                'cs_type': 'first'
            }).reset_index()
        else:
            cell_data = day_data.drop_duplicates('cell_id')
        
        if neurons_to_plot == 'significant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] <= alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] <= alphalevel]['cell_id']
            print(f"\nDay {day}:")
            print(f"CS+ significant cells: {len(significant_csplus_cells)}")
            print(f"CS- significant cells: {len(significant_csminus_cells)}")
        elif neurons_to_plot == 'notsignificant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] > alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] > alphalevel]['cell_id']
        else:
            significant_csplus_cells = cell_data['cell_id']
            significant_csminus_cells = cell_data['cell_id']
        
        cs_plus_data = day_data[
            (day_data['cs_type'] == 'CS+') & 
            (day_data['cell_id'].isin(significant_csplus_cells))
        ]
        cs_minus_data = day_data[
            (day_data['cs_type'] == 'CS-') & 
            (day_data['cell_id'].isin(significant_csminus_cells))
        ]
        
        if average_trials:
            temp_plus = cs_plus_data.groupby('cell_id')[fluor_plus_cols].mean().values
            temp_minus = cs_minus_data.groupby('cell_id')[fluor_minus_cols].mean().values
            ylabel_suffix = "Cells"
        else:
            temp_plus = cs_plus_data[fluor_plus_cols].values
            temp_minus = cs_minus_data[fluor_minus_cols].values
            ylabel_suffix = "Data Points (Trials and Cells)"
        
        # Apply Z-scoring if requested
        if z_score_data:
            # Z-score CS+ data
            for neuron in range(temp_plus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_plus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_plus[neuron, :] = temp_plus[neuron, :] / neuron_std
            
            # Z-score CS- data
            for neuron in range(temp_minus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_minus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_minus[neuron, :] = temp_minus[neuron, :] / neuron_std
        
        if sort_by == 'mean':
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
        elif sort_by == 'timestamp' and not average_trials:
            timestamps_plus = cs_plus_data['timestamp']
            timestamps_minus = cs_minus_data['timestamp']
            sortresponse_plus = np.argsort(timestamps_plus.values)
            sortresponse_minus = np.argsort(timestamps_minus.values)
        else:
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
            if sort_by == 'timestamp':
                print("Warning: Timestamp sorting is only available when average_trials=False. Using mean sorting instead.")
        
        # Plot CS+ heatmap
        ax = axs[d, 0]
        im = ax.imshow(temp_plus[sortresponse_plus], cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
        ax.set_ylabel(f'CS+ ({len(temp_plus)} {ylabel_suffix})')
        ax.set_yticks([])
        ax.set_xticks([])
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        fig.colorbar(im, ax=ax, orientation='vertical', 
                    label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        
        # Plot CS+ mean trace
        ax = axs[d, 1]
        mean_temp_plus = np.nanmean(temp_plus, axis=0)
        sem_temp_plus = np.nanstd(temp_plus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_plus), axis=0))
        ax.plot(mean_temp_plus, color='m')
        ax.fill_between(range(len(mean_temp_plus)), 
                       mean_temp_plus - sem_temp_plus, 
                       mean_temp_plus + sem_temp_plus, 
                       color='pink', alpha=0.5)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks([])
        
        # Plot CS- heatmap
        ax = axs[d, 2]
        im = ax.imshow(temp_minus[sortresponse_minus], cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
        ax.set_ylabel(f'CS- ({len(temp_minus)} {ylabel_suffix})')
        ax.set_yticks([])
        ax.set_xticks([])
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        fig.colorbar(im, ax=ax, orientation='vertical', 
                    label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        
        # Plot CS- mean trace
        ax = axs[d, 3]
        mean_temp_minus = np.nanmean(temp_minus, axis=0)
        sem_temp_minus = np.nanstd(temp_minus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_minus), axis=0))
        ax.plot(mean_temp_minus, color='grey')
        ax.fill_between(range(len(mean_temp_minus)), 
                       mean_temp_minus - sem_temp_minus, 
                       mean_temp_minus + sem_temp_minus, 
                       color='gainsboro', alpha=0.5)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks([])
        
        # Plot Delta Lick Rate
        ax = axs[d, 4]
        delta_licks_cols = [col for col in day_data.columns if col.startswith('delta_licks_frame_')]
        
        if average_trials:
            delta_licks_plus = cs_plus_data.groupby('cell_id')[delta_licks_cols].mean().values
            delta_licks_minus = cs_minus_data.groupby('cell_id')[delta_licks_cols].mean().values
        else:
            delta_licks_plus = cs_plus_data[delta_licks_cols].values
            delta_licks_minus = cs_minus_data[delta_licks_cols].values
        
        mean_delta_licks_plus = np.nanmean(delta_licks_plus, axis=0)
        sem_delta_licks_plus = np.nanstd(delta_licks_plus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_plus), axis=0))
        mean_delta_licks_minus = np.nanmean(delta_licks_minus, axis=0)
        sem_delta_licks_minus = np.nanstd(delta_licks_minus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_minus), axis=0))
        
        ax.plot(mean_delta_licks_plus, color='m', label='CS+')
        ax.fill_between(range(len(mean_delta_licks_plus)), 
                       mean_delta_licks_plus - sem_delta_licks_plus, 
                       mean_delta_licks_plus + sem_delta_licks_plus, 
                       color='m', alpha=0.3)
        ax.plot(mean_delta_licks_minus, color='grey', label='CS-')
        ax.fill_between(range(len(mean_delta_licks_minus)), 
                       mean_delta_licks_minus - sem_delta_licks_minus, 
                       mean_delta_licks_minus + sem_delta_licks_minus, 
                       color='grey', alpha=0.3)
        
        ax.set_title('Delta Lick Rate')
        ax.set_ylabel('Δ Lick Rate (Hz)')
        ax.set_xlabel('Time from CS (s)')
        ax.set_ylim(lickmin, lickmax)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.legend()
    
    fig.text(0.2, 1, 'CS+', fontsize=16)
    fig.text(0.6, 1, 'CS-', fontsize=16)
    fig.text(0.8, 1, 'Δ Licks', fontsize=16)
    fig.tight_layout()
    plt.show()
    print('DONE')
    ###Save the figure
    # fig.savefig(os.path.join(basedir, 'Results', f'PopulationResponsesHeatmaps_dLickRate_{sucomit_filter}_{behavior_filter}_z{z_score_data}_06-20-25.PNG'), 
    #             format='PNG')
    # fig.savefig(os.path.join(basedir, 'Results', f'PopulationResponsesHeatmaps_dLickRate_{sucomit_filter}_{behavior_filter}_z{z_score_data}_06-20-25.PDF'), 
    #             format='PDF')
    plt.show()
    print('DONE')

# Example usage:
visualize_master_data(df_master, days, pre_window_size, infusionframe, alphalevel, 
                     baselinefirstframe, baselinelastframe, z_score_data=True, average_trials=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def visualize_master_data(df_master, days, pre_window_size, infusionframe, alphalevel, 
                         baselinefirstframe, baselinelastframe, neurons_to_plot='all',
                         average_trials=True, sex='both', sort_by='mean', lick='all', 
                         tracked='no', sucomit_filter='all', behavior_filter='all',
                         z_score_data=False):
    """
    Visualize neural responses for CS+ and CS- trials on the same plot.
    Parameters:
        neurons_to_plot: str, 'significant', 'notsignificant', or 'all'
        average_trials: bool, whether to average across trials
        sex: str, 'both', 'male', or 'female'
        sort_by: str, 'mean' or 'timestamp'
        lick: str, 'all', 'high', or 'low'
        tracked: str, 'yes' or 'no' for tracked ROIs
        sucomit_filter: str, 'all', 'rewarded', or 'unrewarded' for SucOmit CS+ trials
        behavior_filter: str, 'all', 'hits', 'misses', 'correct_rejections', or 'false_alarms'
        z_score_data: bool, whether to z-score the fluorescence data based on baseline period
    """
    # Sex filtering
    if sex.lower() != 'both':
        df_master = df_master[df_master['sex'].str.lower() == sex.lower()]
        print(f"Filtered for {sex} animals")
    else:
        print("Including both male and female animals")

    # Add lick filtering
    if lick.lower() == 'high':
        df_master = df_master[df_master['high_licking'] == 1]
        print("Filtered for high-licking trials")
    elif lick.lower() == 'low':
        df_master = df_master[df_master['high_licking'] == 0]
        print("Filtered for low-licking trials")
    else:
        print("Including all licking trials")
    
    # Add tracked ROI filtering
    if tracked.lower() == 'yes':
        df_master = df_master[df_master['tracked_roi_counter'].notna()]
        print("Filtered for tracked ROIs only")
    
    # Add behavioral response filtering
    valid_behaviors = ['hits', 'misses', 'correct_rejections', 'false_alarms']
    if behavior_filter.lower() in valid_behaviors:
        df_master = df_master[df_master[behavior_filter.lower()] == 1]
        print(f"Filtered for {behavior_filter} trials")
    elif behavior_filter.lower() != 'all':
        print(f"Warning: Invalid behavior filter '{behavior_filter}'. Using all trials.")
    else:
        print("Including all behavioral response types")
    
    # Add SucOmit filtering
    if sucomit_filter.lower() in ['rewarded', 'unrewarded']:
        mask = (df_master['day'] == '2_SucOmit') & (df_master['cs_type'] == 'CS+')
        if mask.any():
            if sucomit_filter.lower() == 'rewarded':
                df_master = df_master[~mask | (mask & (df_master['reward_status'] == 'rewarded'))]
                print("Filtered for rewarded trials in SucOmit CS+")
            else:
                df_master = df_master[~mask | (mask & (df_master['reward_status'] == 'unrewarded'))]
                print("Filtered for unrewarded trials in SucOmit CS+")
    
    print("\nAnimal counts by sex:")
    print(df_master.groupby('sex')['animal'].nunique())

    numdays = len(days)
    
    # Create a figure with only two columns instead of five
    fig, axs = plt.subplots(numdays, 2, figsize=(12, numdays*4), sharex=True, sharey=False)
    if numdays == 1:
        axs = axs.reshape(1, -1)
    sns.set_style('white')
    
    # Set visualization limits based on whether we're z-scoring
    if z_score_data:
        ymax, ymin = 7, -1
        print('Data Z-scored for visualization')
    else:
        ymax, ymin = 1.25, -0.25
    
    lickmin, lickmax = -1, 12
    
    for d, day in enumerate(sorted(days)):
        day_data = df_master[df_master['day'] == day]
        
        fluor_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        fluor_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        if average_trials:
            cell_data = day_data.groupby('cell_id').agg({
                'pval_plus': 'min',
                'pval_minus': 'min',
                'cs_type': 'first'
            }).reset_index()
        else:
            cell_data = day_data.drop_duplicates('cell_id')
        
        if neurons_to_plot == 'significant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] <= alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] <= alphalevel]['cell_id']
            print(f"\nDay {day}:")
            print(f"CS+ significant cells: {len(significant_csplus_cells)}")
            print(f"CS- significant cells: {len(significant_csminus_cells)}")
        elif neurons_to_plot == 'notsignificant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] > alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] > alphalevel]['cell_id']
        else:
            significant_csplus_cells = cell_data['cell_id']
            significant_csminus_cells = cell_data['cell_id']
        
        cs_plus_data = day_data[
            (day_data['cs_type'] == 'CS+') & 
            (day_data['cell_id'].isin(significant_csplus_cells))
        ]
        cs_minus_data = day_data[
            (day_data['cs_type'] == 'CS-') & 
            (day_data['cell_id'].isin(significant_csminus_cells))
        ]
        
        if average_trials:
            temp_plus = cs_plus_data.groupby('cell_id')[fluor_plus_cols].mean().values
            temp_minus = cs_minus_data.groupby('cell_id')[fluor_minus_cols].mean().values
            ylabel_suffix = "Cells"
        else:
            temp_plus = cs_plus_data[fluor_plus_cols].values
            temp_minus = cs_minus_data[fluor_minus_cols].values
            ylabel_suffix = "Data Points (Trials and Cells)"
        
        # Apply Z-scoring if requested
        if z_score_data:
            # Z-score CS+ data
            for neuron in range(temp_plus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_plus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_plus[neuron, :] = temp_plus[neuron, :] / neuron_std
            
            # Z-score CS- data
            for neuron in range(temp_minus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_minus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_minus[neuron, :] = temp_minus[neuron, :] / neuron_std
        
        if sort_by == 'mean':
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
        elif sort_by == 'timestamp' and not average_trials:
            timestamps_plus = cs_plus_data['timestamp']
            timestamps_minus = cs_minus_data['timestamp']
            sortresponse_plus = np.argsort(timestamps_plus.values)
            sortresponse_minus = np.argsort(timestamps_minus.values)
        else:
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
            if sort_by == 'timestamp':
                print("Warning: Timestamp sorting is only available when average_trials=False. Using mean sorting instead.")
        
        # Calculate mean traces
        mean_temp_plus = np.nanmean(temp_plus, axis=0)
        sem_temp_plus = np.nanstd(temp_plus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_plus), axis=0))
        mean_temp_minus = np.nanmean(temp_minus, axis=0)
        sem_temp_minus = np.nanstd(temp_minus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_minus), axis=0))
        
        # Plot CS+ and CS- fluorescence traces on the same graph
        ax = axs[d, 0]
        # ax.plot(mean_temp_plus, color='m', label=f'CS+ (n={len(temp_plus)} {ylabel_suffix})')
        ax.plot(mean_temp_plus, color='m', label=f'CS+')
        ax.fill_between(range(len(mean_temp_plus)), 
                       mean_temp_plus - sem_temp_plus, 
                       mean_temp_plus + sem_temp_plus, 
                       color='m', alpha=0.3)
        # ax.plot(mean_temp_minus, color='grey', label=f'CS- (n={len(temp_minus)} {ylabel_suffix})')
        ax.plot(mean_temp_minus, color='grey', label=f'CS-')
        ax.fill_between(range(len(mean_temp_minus)), 
                       mean_temp_minus - sem_temp_minus, 
                       mean_temp_minus + sem_temp_minus, 
                       color='grey', alpha=0.3)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_title(f'Day {day} Neural Activity')
        ax.set_ylabel('Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        ax.legend()
        
        # Plot Delta Lick Rate
        ax = axs[d, 1]
        delta_licks_cols = [col for col in day_data.columns if col.startswith('delta_licks_frame_')]
        
        if average_trials:
            delta_licks_plus = cs_plus_data.groupby('cell_id')[delta_licks_cols].mean().values
            delta_licks_minus = cs_minus_data.groupby('cell_id')[delta_licks_cols].mean().values
        else:
            delta_licks_plus = cs_plus_data[delta_licks_cols].values
            delta_licks_minus = cs_minus_data[delta_licks_cols].values
        
        mean_delta_licks_plus = np.nanmean(delta_licks_plus, axis=0)
        sem_delta_licks_plus = np.nanstd(delta_licks_plus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_plus), axis=0))
        mean_delta_licks_minus = np.nanmean(delta_licks_minus, axis=0)
        sem_delta_licks_minus = np.nanstd(delta_licks_minus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_minus), axis=0))
        
        ax.plot(mean_delta_licks_plus, color='m', label='CS+')
        ax.fill_between(range(len(mean_delta_licks_plus)), 
                       mean_delta_licks_plus - sem_delta_licks_plus, 
                       mean_delta_licks_plus + sem_delta_licks_plus, 
                       color='m', alpha=0.3)
        ax.plot(mean_delta_licks_minus, color='grey', label='CS-')
        ax.fill_between(range(len(mean_delta_licks_minus)), 
                       mean_delta_licks_minus - sem_delta_licks_minus, 
                       mean_delta_licks_minus + sem_delta_licks_minus, 
                       color='grey', alpha=0.3)
        
        ax.set_title('Delta Lick Rate')
        ax.set_ylabel('Δ Lick Rate (Hz)')
        ax.set_xlabel('Time from CS (s)')
        ax.set_ylim(lickmin, lickmax)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.legend()
    
    # Main figure labels
    fig.text(0.25, 1, 'Neural Activity (CS+ and CS-)', fontsize=16)
    fig.text(0.75, 1, 'Δ Licks', fontsize=16)
    fig.tight_layout()
    plt.show()
    print('DONE')
     ###Save the figure
    # fig.savefig(os.path.join(basedir, 'Results', f'PopulationResponsesLineCombined_dLickRate_{sucomit_filter}_{behavior_filter}_z{z_score_data}_06-20-25.PNG'), 
    #             format='PNG')
    # fig.savefig(os.path.join(basedir, 'Results', f'PopulationResponsesLineCombined_dLickRate_{sucomit_filter}_{behavior_filter}_z{z_score_data}_06-20-25.PDF'), 
    #             format='PDF')
    plt.show()
    print('DONE')

# Example usage:
visualize_master_data(df_master, days, pre_window_size, infusionframe, alphalevel, 
                     baselinefirstframe, baselinelastframe, lick='all', sucomit_filter='all', behavior_filter='all', z_score_data=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def visualize_master_data(df_master, days, pre_window_size, infusionframe, alphalevel, 
                         baselinefirstframe, baselinelastframe, neurons_to_plot='all',
                         average_trials=True, sex='both', sort_by='mean', lick='all', 
                         tracked='no', sucomit_filter='rewarded', behavior_filter='all',
                         z_score_data=False, lick_line_width=2.0, neural_line_width=2.0):
    """
    Visualize neural responses for CS+ and CS- trials on the same plot.
    Parameters:
        neurons_to_plot: str, 'significant', 'notsignificant', or 'all'
        average_trials: bool, whether to average across trials
        sex: str, 'both', 'male', or 'female'
        sort_by: str, 'mean' or 'timestamp'
        lick: str, 'all', 'high', or 'low'
        tracked: str, 'yes' or 'no' for tracked ROIs
        sucomit_filter: str, 'all', 'rewarded', or 'unrewarded' for SucOmit CS+ trials
        behavior_filter: str, 'all', 'hits', 'misses', 'correct_rejections', or 'false_alarms'
        z_score_data: bool, whether to z-score the fluorescence data based on baseline period
        lick_line_width: float, width of the lines in the lick rate plot
        neural_line_width: float, width of the lines in the neural activity plot
    """
    # Sex filtering
    if sex.lower() != 'both':
        df_master = df_master[df_master['sex'].str.lower() == sex.lower()]
        print(f"Filtered for {sex} animals")
    else:
        print("Including both male and female animals")

    # Add lick filtering
    if lick.lower() == 'high':
        df_master = df_master[df_master['high_licking'] == 1]
        print("Filtered for high-licking trials")
    elif lick.lower() == 'low':
        df_master = df_master[df_master['high_licking'] == 0]
        print("Filtered for low-licking trials")
    else:
        print("Including all licking trials")
    
    # Add tracked ROI filtering
    if tracked.lower() == 'yes':
        df_master = df_master[df_master['tracked_roi_counter'].notna()]
        print("Filtered for tracked ROIs only")
    
    # Add behavioral response filtering
    valid_behaviors = ['hits', 'misses', 'correct_rejections', 'false_alarms']
    if behavior_filter.lower() in valid_behaviors:
        df_master = df_master[df_master[behavior_filter.lower()] == 1]
        print(f"Filtered for {behavior_filter} trials")
    elif behavior_filter.lower() != 'all':
        print(f"Warning: Invalid behavior filter '{behavior_filter}'. Using all trials.")
    else:
        print("Including all behavioral response types")
    
    # Add SucOmit filtering
    if sucomit_filter.lower() in ['rewarded', 'unrewarded']:
        mask = (df_master['day'] == '2_SucOmit') & (df_master['cs_type'] == 'CS+')
        if mask.any():
            if sucomit_filter.lower() == 'rewarded':
                df_master = df_master[~mask | (mask & (df_master['reward_status'] == 'rewarded'))]
                print("Filtered for rewarded trials in SucOmit CS+")
            else:
                df_master = df_master[~mask | (mask & (df_master['reward_status'] == 'unrewarded'))]
                print("Filtered for unrewarded trials in SucOmit CS+")
    
    print("\nAnimal counts by sex:")
    print(df_master.groupby('sex')['animal'].nunique())

    numdays = len(days)
    
    # Create a figure with only two columns instead of five
    fig, axs = plt.subplots(numdays, 2, figsize=(12, numdays*4), sharex=True, sharey=False)
    if numdays == 1:
        axs = axs.reshape(1, -1)
    sns.set_style('white')
    
    # Set visualization limits based on whether we're z-scoring
    if z_score_data:
        ymax, ymin = 7, -1
        print('Data Z-scored for visualization')
    else:
        ymax, ymin = 1.25, -0.25
    
    lickmin, lickmax = -1, 12
    
    for d, day in enumerate(sorted(days)):
        day_data = df_master[df_master['day'] == day]
        
        fluor_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        fluor_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        if average_trials:
            cell_data = day_data.groupby('cell_id').agg({
                'pval_plus': 'min',
                'pval_minus': 'min',
                'cs_type': 'first'
            }).reset_index()
        else:
            cell_data = day_data.drop_duplicates('cell_id')
        
        if neurons_to_plot == 'significant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] <= alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] <= alphalevel]['cell_id']
            print(f"\nDay {day}:")
            print(f"CS+ significant cells: {len(significant_csplus_cells)}")
            print(f"CS- significant cells: {len(significant_csminus_cells)}")
        elif neurons_to_plot == 'notsignificant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] > alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] > alphalevel]['cell_id']
        else:
            significant_csplus_cells = cell_data['cell_id']
            significant_csminus_cells = cell_data['cell_id']
        
        cs_plus_data = day_data[
            (day_data['cs_type'] == 'CS+') & 
            (day_data['cell_id'].isin(significant_csplus_cells))
        ]
        cs_minus_data = day_data[
            (day_data['cs_type'] == 'CS-') & 
            (day_data['cell_id'].isin(significant_csminus_cells))
        ]
        
        if average_trials:
            temp_plus = cs_plus_data.groupby('cell_id')[fluor_plus_cols].mean().values
            temp_minus = cs_minus_data.groupby('cell_id')[fluor_minus_cols].mean().values
            ylabel_suffix = "Cells"
        else:
            temp_plus = cs_plus_data[fluor_plus_cols].values
            temp_minus = cs_minus_data[fluor_minus_cols].values
            ylabel_suffix = "Data Points (Trials and Cells)"
        
        # Apply Z-scoring if requested
        if z_score_data:
            # Z-score CS+ data
            for neuron in range(temp_plus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_plus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_plus[neuron, :] = temp_plus[neuron, :] / neuron_std
            
            # Z-score CS- data
            for neuron in range(temp_minus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_minus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_minus[neuron, :] = temp_minus[neuron, :] / neuron_std
        
        if sort_by == 'mean':
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
        elif sort_by == 'timestamp' and not average_trials:
            timestamps_plus = cs_plus_data['timestamp']
            timestamps_minus = cs_minus_data['timestamp']
            sortresponse_plus = np.argsort(timestamps_plus.values)
            sortresponse_minus = np.argsort(timestamps_minus.values)
        else:
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
            if sort_by == 'timestamp':
                print("Warning: Timestamp sorting is only available when average_trials=False. Using mean sorting instead.")
        
        # Calculate mean traces
        mean_temp_plus = np.nanmean(temp_plus, axis=0)
        sem_temp_plus = np.nanstd(temp_plus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_plus), axis=0))
        mean_temp_minus = np.nanmean(temp_minus, axis=0)
        sem_temp_minus = np.nanstd(temp_minus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_minus), axis=0))
        
        # Plot CS+ and CS- fluorescence traces on the same graph with customizable line width
        ax = axs[d, 0]
        # ax.plot(mean_temp_plus, color='m', label=f'CS+ (n={len(temp_plus)} {ylabel_suffix})')
        ax.plot(mean_temp_plus, color='m', label=f'CS+', linewidth=neural_line_width)
        ax.fill_between(range(len(mean_temp_plus)), 
                       mean_temp_plus - sem_temp_plus, 
                       mean_temp_plus + sem_temp_plus, 
                       color='m', alpha=0.3)
        # ax.plot(mean_temp_minus, color='grey', label=f'CS- (n={len(temp_minus)} {ylabel_suffix})')
        ax.plot(mean_temp_minus, color='grey', label=f'CS-', linewidth=neural_line_width)
        ax.fill_between(range(len(mean_temp_minus)), 
                       mean_temp_minus - sem_temp_minus, 
                       mean_temp_minus + sem_temp_minus, 
                       color='grey', alpha=0.3)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_title(f'Day {day} Neural Activity')
        ax.set_ylabel('Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        ax.legend()
        
        # Plot Delta Lick Rate with customizable line width
        ax = axs[d, 1]
        delta_licks_cols = [col for col in day_data.columns if col.startswith('delta_licks_frame_')]
        
        if average_trials:
            delta_licks_plus = cs_plus_data.groupby('cell_id')[delta_licks_cols].mean().values
            delta_licks_minus = cs_minus_data.groupby('cell_id')[delta_licks_cols].mean().values
        else:
            delta_licks_plus = cs_plus_data[delta_licks_cols].values
            delta_licks_minus = cs_minus_data[delta_licks_cols].values
        
        mean_delta_licks_plus = np.nanmean(delta_licks_plus, axis=0)
        sem_delta_licks_plus = np.nanstd(delta_licks_plus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_plus), axis=0))
        mean_delta_licks_minus = np.nanmean(delta_licks_minus, axis=0)
        sem_delta_licks_minus = np.nanstd(delta_licks_minus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_minus), axis=0))
        
        ax.plot(mean_delta_licks_plus, color='m', label='CS+', linewidth=lick_line_width)
        ax.fill_between(range(len(mean_delta_licks_plus)), 
                       mean_delta_licks_plus - sem_delta_licks_plus, 
                       mean_delta_licks_plus + sem_delta_licks_plus, 
                       color='m', alpha=0.3)
        ax.plot(mean_delta_licks_minus, color='grey', label='CS-', linewidth=lick_line_width)
        ax.fill_between(range(len(mean_delta_licks_minus)), 
                       mean_delta_licks_minus - sem_delta_licks_minus, 
                       mean_delta_licks_minus + sem_delta_licks_minus, 
                       color='grey', alpha=0.3)
        
        ax.set_title('Delta Lick Rate')
        ax.set_ylabel('Δ Lick Rate (Hz)')
        ax.set_xlabel('Time from CS (s)')
        ax.set_ylim(lickmin, lickmax)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.legend()
    
    # Main figure labels
    fig.text(0.25, 1, 'Neural Activity (CS+ and CS-)', fontsize=16)
    fig.text(0.75, 1, 'Δ Licks', fontsize=16)
    fig.tight_layout()
    plt.show()
    print('DONE')

# Example usage:
visualize_master_data(df_master, days, pre_window_size, infusionframe, alphalevel, 
                     baselinefirstframe, baselinelastframe, lick='all', 
                     sucomit_filter='all', behavior_filter='all', z_score_data=True,
                     lick_line_width=5.0, neural_line_width=3.0)  # Set custom line width

In [ ]:
##plotting the df/f zscored
import matplotlib.pyplot as plt
import numpy as np
import os

import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

def plot_zscored_df_f_by_response_within_phase(df_master, aucfirstframe, auclastframe, 
                                              baselinefirstframe, baselinelastframe, 
                                              basedir=None):
    """
    Create one plot for each phase showing averaged Z-scored ΔF/F values for cells 
    that participated in different behavioral response types.
    Z-scoring is applied AFTER averaging trials together, matching the original visualization function.
    
    Parameters:
        df_master (pandas.DataFrame): Master dataframe containing all trial data
        aucfirstframe (int): First frame of the response window to average
        auclastframe (int): Last frame of the response window to average
        baselinefirstframe (int): First frame of the baseline period for Z-scoring
        baselinelastframe (int): Last frame of the baseline period for Z-scoring
        basedir (str, optional): Base directory for saving plots
    """
    # Define colors for each response type
    response_colors = {
        'hits': 'lawngreen',
        'misses': 'black',
        'correct_rejections': 'palegreen',
        'false_alarms': 'grey'
    }
    
    # Print header for the cell count report
    print("\n" + "="*80)
    print("WITHIN-PHASE ANALYSIS: Z-SCORED ΔF/F BY RESPONSE TYPE")
    print("="*80)
    
    # Get all unique experimental phases
    phases = sorted(df_master['experimental_phase'].unique())
    
    # Identify which phases are reversal phases
    reversal_phases = ['4_RevEarly', '5_RevLate']
    
    for phase in phases:
        # Filter for the current phase
        phase_data = df_master[df_master['experimental_phase'] == phase]
        
        # Create a figure
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Lists to store data for plotting
        all_data = []
        positions = []
        labels = []
        colors = []
        
        # Current position for plotting
        pos = 1
        
        # Print header for this phase
        print(f"\nPhase: {phase}")
        print("-"*50)
        
        # Process each response type
        for resp_type in ['hits', 'misses', 'correct_rejections', 'false_alarms']:
            # First, get all unique cell_ids for this response type
            resp_cells = phase_data[phase_data[resp_type] == 1]['cell_id'].unique()
            
            if len(resp_cells) == 0:
                print(f"  {resp_type}: No cells available")
                continue  # Skip if no cells for this response type
            
            # Determine valid stimulus types based on response type and phase
            is_reversal = phase in reversal_phases
            
            # In early/normal phases:
            # - hits and misses only apply to CS+
            # - correct rejections and false alarms only apply to CS-
            # In reversal phases, this logic is flipped
            
            if resp_type in ['hits', 'misses']:
                # For hits and misses
                if is_reversal:
                    valid_stim = 'CS-'  # In reversal, CS- is the go cue
                else:
                    valid_stim = 'CS+'  # In early/normal, CS+ is the go cue
            else:
                # For correct rejections and false alarms
                if is_reversal:
                    valid_stim = 'CS+'  # In reversal, CS+ is the no-go cue
                else:
                    valid_stim = 'CS-'  # In early/normal, CS- is the no-go cue
            
            # Process valid stimulus type
            if valid_stim == 'CS+':
                # Get CS+ data: filter for cells that participated in this response type
                cs_plus_cells = phase_data[(phase_data['cs_type'] == 'CS+') & 
                                          (phase_data['cell_id'].isin(resp_cells)) &
                                          (phase_data[resp_type] == 1)]
                
                if len(cs_plus_cells) > 0:
                    # Get fluorescence columns for CS+
                    fluor_plus_cols = [col for col in cs_plus_cells.columns if col.startswith('fluor_plus_frame_')]
                    
                    # Calculate Z-scored values for each cell following original code's approach
                    zscored_values = []
                    
                    # Group by cell_id and average trials first
                    grouped_cells = cs_plus_cells.groupby('cell_id')
                    
                    for cell_id, cell_group in grouped_cells:
                        # Average fluorescence data across trials for this cell
                        avg_fluor = cell_group[fluor_plus_cols].mean().values
                        
                        # Skip cells with no data
                        if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                            continue
                        
                        # Calculate standard deviation of baseline period
                        baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                        neuron_std = np.nanstd(avg_fluor[baseline_indices])
                        
                        # Skip cells with zero std dev
                        if neuron_std <= 0:
                            continue
                        
                        # Z-score by dividing by std dev (AFTER averaging)
                        zscored_fluor = avg_fluor / neuron_std
                        
                        # Calculate average over the response window (aucfirstframe to auclastframe)
                        resp_indices = list(range(aucfirstframe, auclastframe+1))
                        
                        # Average across response window frames
                        avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                        zscored_values.append(avg_zscored)
                        
                        # Print some debug info for the first few cells
                        if len(zscored_values) <= 3:
                            print(f"    Debug - Cell {cell_id} CS+ {resp_type}: baseline std={neuron_std:.4f}, avg Z-score={avg_zscored:.4f}, max Z-score={np.nanmax(zscored_fluor[resp_indices]):.4f}")
                    
                    cell_count = len(zscored_values)
                    print(f"  {resp_type} CS+: {cell_count} cells")
                    
                    if cell_count > 0:
                        all_data.append(zscored_values)
                        positions.append(pos)
                        labels.append(f"{resp_type} CS+\nn={cell_count}")
                        colors.append(response_colors[resp_type])
                        pos += 1
                else:
                    print(f"  {resp_type} CS+: No data available")
            
            if valid_stim == 'CS-':
                # Get CS- data: filter for cells that participated in this response type
                cs_minus_cells = phase_data[(phase_data['cs_type'] == 'CS-') & 
                                           (phase_data['cell_id'].isin(resp_cells)) &
                                           (phase_data[resp_type] == 1)]
                
                if len(cs_minus_cells) > 0:
                    # Get fluorescence columns for CS-
                    fluor_minus_cols = [col for col in cs_minus_cells.columns if col.startswith('fluor_minus_frame_')]
                    
                    # Calculate Z-scored values for each cell following original code's approach
                    zscored_values = []
                    
                    # Group by cell_id and average trials first
                    grouped_cells = cs_minus_cells.groupby('cell_id')
                    
                    for cell_id, cell_group in grouped_cells:
                        # Average fluorescence data across trials for this cell
                        avg_fluor = cell_group[fluor_minus_cols].mean().values
                        
                        # Skip cells with no data
                        if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                            continue
                        
                        # Calculate standard deviation of baseline period
                        baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                        neuron_std = np.nanstd(avg_fluor[baseline_indices])
                        
                        # Skip cells with zero std dev
                        if neuron_std <= 0:
                            continue
                        
                        # Z-score by dividing by std dev (AFTER averaging)
                        zscored_fluor = avg_fluor / neuron_std
                        
                        # Calculate average over the response window (aucfirstframe to auclastframe)
                        resp_indices = list(range(aucfirstframe, auclastframe+1))
                        
                        # Average across response window frames
                        avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                        zscored_values.append(avg_zscored)
                        
                        # Print some debug info for the first few cells
                        if len(zscored_values) <= 3:
                            print(f"    Debug - Cell {cell_id} CS- {resp_type}: baseline std={neuron_std:.4f}, avg Z-score={avg_zscored:.4f}, max Z-score={np.nanmax(zscored_fluor[resp_indices]):.4f}")
                    
                    cell_count = len(zscored_values)
                    print(f"  {resp_type} CS-: {cell_count} cells")
                    
                    if cell_count > 0:
                        all_data.append(zscored_values)
                        positions.append(pos)
                        labels.append(f"{resp_type} CS-\nn={cell_count}")
                        colors.append(response_colors[resp_type])
                        pos += 1
                else:
                    print(f"  {resp_type} CS-: No data available")
            
            # Add a small gap between response types
            pos += 0.5
        
        # Create box plots if we have data
        if all_data:
            box = ax.boxplot(all_data, positions=positions, patch_artist=True, 
                            showfliers=False, widths=0.5)
            
            # Customize box colors
            for patch, color in zip(box['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            # Add strip plots with jitter
            for i, d in enumerate(all_data):
                x = np.random.normal(positions[i], 0.01, size=len(d))  # Add jitter
                ax.scatter(x, d, alpha=0.6, edgecolor='black', s=30, color=colors[i])
                
            # Change median line color to white (or use 'black' instead)
            for median in box['medians']:
                median.set_color('black')
                median.set_linewidth(2)  # Optional: make the line thicker for better visibility
                
            # Add total cell count to title
            total_cells = sum(len(d) for d in all_data)
                
            # Customize the plot
            ax.set_xticks(positions)
            ax.set_xticklabels(labels, rotation=45, ha='right')
            ax.set_ylabel('Z-scored ΔF/F')
            ax.set_title(f'Z-scored ΔF/F Values (Frames {aucfirstframe}-{auclastframe}) by Response Type - {phase} (Total n={total_cells})')
            ax.axhline(0.0, color='black', linestyle='--', linewidth=0.5)
            
            # Set y-limits based on Z-scored data (similar to reference code)
            ax.set_ylim(-5, 25)
            ax.set_yticks([-5, 0, 5, 10, 15, 20, 25])
            
            plt.tight_layout()
            
            # Save the plot if basedir is provided
            # if basedir:
            #     pdf_filename = os.path.join(basedir, "Results", f'ZscoredDFF_{phase}_Frames{aucfirstframe}-{auclastframe}_ByResponseType.pdf')
            #     plt.savefig(pdf_filename)
                
            #     # Also save as PNG for easier viewing
            #     png_filename = os.path.join(basedir, "Results", f'ZscoredDFF_{phase}_Frames{aucfirstframe}-{auclastframe}_ByResponseType.png')
            #     plt.savefig(png_filename, dpi=300)
            
            plt.show()
        else:
            plt.close(fig)  # Close empty figure
            print(f"No data available for phase {phase}")


def plot_zscored_df_f_by_response_between_phases(df_master, aucfirstframe, auclastframe, 
                                               baselinefirstframe, baselinelastframe, 
                                               basedir=None):
    """
    Create comparison plots between pairs of phases, showing Z-scored ΔF/F values for cells
    that participated in different behavioral response types.
    Z-scoring is applied AFTER averaging trials together, matching the original visualization function.
    
    Parameters:
        df_master (pandas.DataFrame): Master dataframe containing all trial data
        aucfirstframe (int): First frame of the response window to average
        auclastframe (int): Last frame of the response window to average
        baselinefirstframe (int): First frame of the baseline period for Z-scoring
        baselinelastframe (int): Last frame of the baseline period for Z-scoring
        basedir (str, optional): Base directory for saving plots
    """
    # Define colors for each response type
    response_colors = {
        'hits': 'lawngreen',
        'misses': 'black',
        'correct_rejections': 'palegreen',
        'false_alarms': 'grey'
    }
    
    # Print header for the cell count report
    print("\n" + "="*80)
    print("BETWEEN-PHASE ANALYSIS: Z-SCORED ΔF/F BY RESPONSE TYPE")
    print("="*80)
    
    # Define the phase pairings and their types
    pairings = [
        ('0_Early', '1_Late'),  # Early phases
        ('2_SucOmit','3_CueOmit'),  # Early phases
        ('4_RevEarly', '5_RevLate')  # Reversal phases
    ]
    
    # Identify which phases are reversal phases
    reversal_phases = ['4_RevEarly', '5_RevLate']
    
    for day1, day2 in pairings:
        # Create a figure
        fig, ax = plt.subplots(figsize=(14, 8))
        
        # Lists to store data for plotting
        all_data = []
        positions = []
        labels = []
        colors = []
        
        # Start positions for left and right sides
        left_pos = 1
        right_pos = 8  # Gap between phase groups
        
        # Print header for this pairing
        print(f"\nComparison: {day1} vs {day2}")
        print("-"*50)
        
        # Process each phase
        for phase_idx, phase in enumerate([day1, day2]):
            current_pos = left_pos if phase_idx == 0 else right_pos
            
            # Filter for the current phase
            phase_data = df_master[df_master['experimental_phase'] == phase]
            
            print(f"\n  Phase: {phase}")
            
            # Process each response type for this phase
            for resp_type in ['hits', 'misses', 'correct_rejections', 'false_alarms']:
                # First, get all unique cell_ids for this response type
                resp_cells = phase_data[phase_data[resp_type] == 1]['cell_id'].unique()
                
                if len(resp_cells) == 0:
                    print(f"    {resp_type}: No cells available")
                    continue  # Skip if no cells for this response type
                
                # Determine valid stimulus types based on response type and phase
                is_reversal = phase in reversal_phases
                
                # In early/normal phases:
                # - hits and misses only apply to CS+
                # - correct rejections and false alarms only apply to CS-
                # In reversal phases, this logic is flipped
                
                if resp_type in ['hits', 'misses']:
                    # For hits and misses
                    if is_reversal:
                        valid_stim = 'CS-'  # In reversal, CS- is the go cue
                    else:
                        valid_stim = 'CS+'  # In early/normal, CS+ is the go cue
                else:
                    # For correct rejections and false alarms
                    if is_reversal:
                        valid_stim = 'CS+'  # In reversal, CS+ is the no-go cue
                    else:
                        valid_stim = 'CS-'  # In early/normal, CS- is the no-go cue
                
                # Process valid stimulus type
                if valid_stim == 'CS+':
                    # Get CS+ data
                    cs_plus_cells = phase_data[(phase_data['cs_type'] == 'CS+') & 
                                              (phase_data['cell_id'].isin(resp_cells)) &
                                              (phase_data[resp_type] == 1)]
                    
                    if len(cs_plus_cells) > 0:
                        # Get fluorescence columns for CS+
                        fluor_plus_cols = [col for col in cs_plus_cells.columns if col.startswith('fluor_plus_frame_')]
                        
                        # Calculate Z-scored values for each cell following original code's approach
                        zscored_values = []
                        
                        # Group by cell_id and average trials first
                        grouped_cells = cs_plus_cells.groupby('cell_id')
                        
                        for cell_id, cell_group in grouped_cells:
                            # Average fluorescence data across trials for this cell
                            avg_fluor = cell_group[fluor_plus_cols].mean().values
                            
                            # Skip cells with no data
                            if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                                continue
                            
                            # Calculate standard deviation of baseline period
                            baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                            neuron_std = np.nanstd(avg_fluor[baseline_indices])
                            
                            # Skip cells with zero std dev
                            if neuron_std <= 0:
                                continue
                            
                            # Z-score by dividing by std dev (AFTER averaging)
                            zscored_fluor = avg_fluor / neuron_std
                            
                            # Calculate average over the response window (aucfirstframe to auclastframe)
                            resp_indices = list(range(aucfirstframe, auclastframe+1))
                            
                            # Average across response window frames
                            avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                            zscored_values.append(avg_zscored)
                        
                        cell_count = len(zscored_values)
                        print(f"    {resp_type} CS+: {cell_count} cells")
                        
                        if cell_count > 0:
                            all_data.append(zscored_values)
                            positions.append(current_pos)
                            labels.append(f"{phase}\n{resp_type}\nCS+\nn={cell_count}")
                            colors.append(response_colors[resp_type])
                            current_pos += 1
                    else:
                        print(f"    {resp_type} CS+: No data available")
                
                if valid_stim == 'CS-':
                    # Get CS- data
                    cs_minus_cells = phase_data[(phase_data['cs_type'] == 'CS-') & 
                                               (phase_data['cell_id'].isin(resp_cells)) &
                                               (phase_data[resp_type] == 1)]
                    
                    if len(cs_minus_cells) > 0:
                        # Get fluorescence columns for CS-
                        fluor_minus_cols = [col for col in cs_minus_cells.columns if col.startswith('fluor_minus_frame_')]
                        
                        # Calculate Z-scored values for each cell following original code's approach
                        zscored_values = []
                        
                        # Group by cell_id and average trials first
                        grouped_cells = cs_minus_cells.groupby('cell_id')
                        
                        for cell_id, cell_group in grouped_cells:
                            # Average fluorescence data across trials for this cell
                            avg_fluor = cell_group[fluor_minus_cols].mean().values
                            
                            # Skip cells with no data
                            if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                                continue
                            
                            # Calculate standard deviation of baseline period
                            baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                            neuron_std = np.nanstd(avg_fluor[baseline_indices])
                            
                            # Skip cells with zero std dev
                            if neuron_std <= 0:
                                continue
                            
                            # Z-score by dividing by std dev (AFTER averaging)
                            zscored_fluor = avg_fluor / neuron_std
                            
                            # Calculate average over the response window (aucfirstframe to auclastframe)
                            resp_indices = list(range(aucfirstframe, auclastframe+1))
                            
                            # Average across response window frames
                            avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                            zscored_values.append(avg_zscored)
                        
                        cell_count = len(zscored_values)
                        print(f"    {resp_type} CS-: {cell_count} cells")
                        
                        if cell_count > 0:
                            all_data.append(zscored_values)
                            positions.append(current_pos)
                            labels.append(f"{phase}\n{resp_type}\nCS-\nn={cell_count}")
                            colors.append(response_colors[resp_type])
                            current_pos += 1
                    else:
                        print(f"    {resp_type} CS-: No data available")
                
                # Add small gap between response types
                current_pos += 0.5
        
        # Create box plots if we have data
        if all_data:
            box = ax.boxplot(all_data, positions=positions, patch_artist=True, 
                            showfliers=False, widths=0.5)
            
            # Customize box colors
            for patch, color in zip(box['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            # Add strip plots with jitter
            for i, d in enumerate(all_data):
                x = np.random.normal(positions[i], 0.01, size=len(d))  # Add jitter
                ax.scatter(x, d, alpha=0.6, edgecolor='black', s=30, color=colors[i])
            
            # Change median line color to white (or use 'black' instead)
            for median in box['medians']:
                median.set_color('black')
                median.set_linewidth(2)  # Optional: make the line thicker for better visibility
            
            # Add vertical line to separate phases
            midpoint = (left_pos + right_pos) / 2
            # ax.axvline(x=midpoint - 1, color='black', linestyle='-', linewidth=1, alpha=0.5)
            
            # Add phase labels at the top
            ax.text((left_pos + (right_pos - left_pos)/4), 8, day1, 
                   horizontalalignment='center', fontsize=14, fontweight='bold')
            ax.text(right_pos + (right_pos - left_pos)/4, 8, day2, 
                   horizontalalignment='center', fontsize=14, fontweight='bold')
            
            # Calculate total cell counts for title
            total_cells = sum(len(d) for d in all_data)
            day1_cells = sum(len(d) for i, d in enumerate(all_data) if positions[i] < midpoint - 1)
            day2_cells = sum(len(d) for i, d in enumerate(all_data) if positions[i] > midpoint - 1)
            
            # Customize the plot
            ax.set_xticks(positions)
            ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
            ax.set_ylabel('Z-scored ΔF/F')
            ax.set_title(f'Comparison of Z-scored ΔF/F Values (Frames {aucfirstframe}-{auclastframe}) by Response Type: {day1} (n={day1_cells}) vs {day2} (n={day2_cells})')
            ax.axhline(0.0, color='black', linestyle='--', linewidth=0.5)
            
            # Set y-limits based on Z-scored data (similar to reference code)
            if (day1, day2) == ('2_SucOmit','3_CueOmit'):
                ax.set_ylim(-5, 25)
                ax.set_yticks([-5, 0, 5, 10, 15, 20, 25])
            else:
                ax.set_ylim(-5, 20)
                ax.set_yticks([-5, 0, 5, 10, 15, 20])
                
            # Create a custom legend for response types
            from matplotlib.patches import Patch
            legend_elements = [
                Patch(facecolor=response_colors['hits'], edgecolor='black', label='Hits'),
                Patch(facecolor=response_colors['misses'], edgecolor='black', label='Misses'),
                Patch(facecolor=response_colors['correct_rejections'], edgecolor='black', label='Correct Rejections'),
                Patch(facecolor=response_colors['false_alarms'], edgecolor='black', label='False Alarms')
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            
            plt.tight_layout()
            
            # Save the plot if basedir is provided
            # if basedir:
            #     pdf_filename = os.path.join(basedir, "Results", f'ZscoredDFF_{day1}_vs_{day2}_Frames{aucfirstframe}-{auclastframe}_ByResponseType_06-20-25.pdf')
            #     plt.savefig(pdf_filename)
                
            #     # Also save as PNG for easier viewing
            #     png_filename = os.path.join(basedir, "Results", f'ZscoredDFF_{day1}_vs_{day2}_Frames{aucfirstframe}-{auclastframe}_ByResponseType_06-20-25.png')
            #     plt.savefig(png_filename, dpi=300)
            
            plt.show()
        else:
            plt.close(fig)  # Close empty figure
            print(f"No data available for comparison between {day1} and {day2}")

# Example usage:
plot_zscored_df_f_by_response_within_phase(df_master, aucfirstframe=22, auclastframe=68, 
                                          baselinefirstframe=0, baselinelastframe=15, 
                                          basedir=r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell')

plot_zscored_df_f_by_response_between_phases(df_master, aucfirstframe=22, auclastframe=68, 
                                           baselinefirstframe=0, baselinelastframe=15, 
                                           basedir=r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell')

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

def extract_zscored_data_for_stats(df_master, aucfirstframe, auclastframe, 
                                  baselinefirstframe, baselinelastframe):
    """
    Extract Z-scored ΔF/F values using the same methodology as the plotting functions.
    Returns a structured dataframe for statistical analysis.
    """
    
    # Define which phases are reversal phases
    reversal_phases = ['4_RevEarly', '5_RevLate']
    
    # Store all data for statistical analysis
    stats_data = []
    
    # Get all unique experimental phases
    phases = sorted(df_master['experimental_phase'].unique())
    
    for phase in phases:
        # Filter for the current phase
        phase_data = df_master[df_master['experimental_phase'] == phase]
        
        # Process each response type
        for resp_type in ['hits', 'misses', 'correct_rejections', 'false_alarms']:
            # Get all unique cell_ids for this response type
            resp_cells = phase_data[phase_data[resp_type] == 1]['cell_id'].unique()
            
            if len(resp_cells) == 0:
                continue  # Skip if no cells for this response type
            
            # Determine valid stimulus types based on response type and phase
            is_reversal = phase in reversal_phases
            
            if resp_type in ['hits', 'misses']:
                # For hits and misses
                if is_reversal:
                    valid_stim = 'CS-'  # In reversal, CS- is the go cue
                else:
                    valid_stim = 'CS+'  # In early/normal, CS+ is the go cue
            else:
                # For correct rejections and false alarms
                if is_reversal:
                    valid_stim = 'CS+'  # In reversal, CS+ is the no-go cue
                else:
                    valid_stim = 'CS-'  # In early/normal, CS- is the no-go cue
            
            # Process valid stimulus type - EXACTLY matching plotting code logic
            if valid_stim == 'CS+':
                # Get CS+ data: filter for cells that participated in this response type
                cs_plus_cells = phase_data[(phase_data['cs_type'] == 'CS+') & 
                                          (phase_data['cell_id'].isin(resp_cells)) &
                                          (phase_data[resp_type] == 1)]
                
                if len(cs_plus_cells) > 0:
                    # Get fluorescence columns for CS+
                    fluor_plus_cols = [col for col in cs_plus_cells.columns if col.startswith('fluor_plus_frame_')]
                    
                    # Calculate Z-scored values for each cell following original code's approach
                    # Group by cell_id and average trials first
                    grouped_cells = cs_plus_cells.groupby('cell_id')
                    
                    for cell_id, cell_group in grouped_cells:
                        # Average fluorescence data across trials for this cell
                        avg_fluor = cell_group[fluor_plus_cols].mean().values
                        
                        # Skip cells with no data
                        if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                            continue
                        
                        # Calculate standard deviation of baseline period
                        baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                        neuron_std = np.nanstd(avg_fluor[baseline_indices])
                        
                        # Skip cells with zero std dev
                        if neuron_std <= 0:
                            continue
                        
                        # Z-score by dividing by std dev (AFTER averaging)
                        zscored_fluor = avg_fluor / neuron_std
                        
                        # Calculate average over the response window (aucfirstframe to auclastframe)
                        resp_indices = list(range(aucfirstframe, auclastframe+1))
                        
                        # Average across response window frames
                        avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                        
                        # Store the data
                        stats_data.append({
                            'cell_id': cell_id,
                            'phase': phase,
                            'response_type': resp_type,
                            'stimulus_type': valid_stim,
                            'zscored_dff': avg_zscored,
                            'is_reversal': is_reversal
                        })
            
            if valid_stim == 'CS-':
                # Get CS- data: filter for cells that participated in this response type
                cs_minus_cells = phase_data[(phase_data['cs_type'] == 'CS-') & 
                                           (phase_data['cell_id'].isin(resp_cells)) &
                                           (phase_data[resp_type] == 1)]
                
                if len(cs_minus_cells) > 0:
                    # Get fluorescence columns for CS-
                    fluor_minus_cols = [col for col in cs_minus_cells.columns if col.startswith('fluor_minus_frame_')]
                    
                    # Calculate Z-scored values for each cell following original code's approach
                    # Group by cell_id and average trials first
                    grouped_cells = cs_minus_cells.groupby('cell_id')
                    
                    for cell_id, cell_group in grouped_cells:
                        # Average fluorescence data across trials for this cell
                        avg_fluor = cell_group[fluor_minus_cols].mean().values
                        
                        # Skip cells with no data
                        if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                            continue
                        
                        # Calculate standard deviation of baseline period
                        baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                        neuron_std = np.nanstd(avg_fluor[baseline_indices])
                        
                        # Skip cells with zero std dev
                        if neuron_std <= 0:
                            continue
                        
                        # Z-score by dividing by std dev (AFTER averaging)
                        zscored_fluor = avg_fluor / neuron_std
                        
                        # Calculate average over the response window (aucfirstframe to auclastframe)
                        resp_indices = list(range(aucfirstframe, auclastframe+1))
                        
                        # Average across response window frames
                        avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                        
                        # Store the data
                        stats_data.append({
                            'cell_id': cell_id,
                            'phase': phase,
                            'response_type': resp_type,
                            'stimulus_type': valid_stim,
                            'zscored_dff': avg_zscored,
                            'is_reversal': is_reversal
                        })
    
    return pd.DataFrame(stats_data)

def analyze_within_phase_stats(stats_df):
    """
    Perform one-way ANOVA for each phase separately.
    Factor: Response Type (4 levels)
    Dependent Variable: Z-scored ΔF/F
    """
    print("\n" + "="*80)
    print("WITHIN-PHASE STATISTICAL ANALYSIS")
    print("="*80)
    
    phases = sorted(stats_df['phase'].unique())
    
    for phase in phases:
        phase_data = stats_df[stats_df['phase'] == phase]
        
        if len(phase_data) == 0:
            continue
            
        print(f"\nPhase: {phase}")
        print("-" * 50)
        
        # Check if we have multiple response types
        response_types = phase_data['response_type'].unique()
        if len(response_types) < 2:
            print(f"Insufficient response types for analysis (only {len(response_types)} types)")
            continue
        
        # Prepare data for ANOVA
        response_groups = []
        group_labels = []
        
        for resp_type in ['hits', 'misses', 'correct_rejections', 'false_alarms']:
            resp_data = phase_data[phase_data['response_type'] == resp_type]['zscored_dff'].values
            if len(resp_data) > 0:
                response_groups.append(resp_data)
                group_labels.append(resp_type)
        
        # Print sample sizes
        print("Sample sizes:")
        for i, label in enumerate(group_labels):
            print(f"  {label}: n = {len(response_groups[i])}")
        
        if len(response_groups) < 2:
            print("Insufficient groups for analysis")
            continue
        
        # Perform one-way ANOVA using OLS to get complete ANOVA table
        try:
            # Create dataframe for OLS
            anova_data = []
            for i, group in enumerate(response_groups):
                for value in group:
                    anova_data.append({'zscored_dff': value, 'response_type': group_labels[i]})
            anova_df = pd.DataFrame(anova_data)
            
            # Fit the model
            model = ols('zscored_dff ~ C(response_type)', data=anova_df).fit()
            
            # Get ANOVA table
            anova_results = anova_lm(model, typ=2)
            
            # Add residual row to the ANOVA table
            residual_df = model.df_resid
            residual_ss = model.ssr
            residual_ms = residual_ss / residual_df
            
            # Create complete ANOVA table
            print(f"\nOne-way ANOVA Results:")
            print("=" * 70)
            print(f"{'Source':<20} {'SS':<15} {'df':<8} {'MS':<15} {'F':<10} {'p-value':<10}")
            print("-" * 70)
            
            # Response type row
            rt_row = anova_results.loc['C(response_type)']
            print(f"{'Response Type':<20} {rt_row['sum_sq']:<15.4f} {rt_row['df']:<8.0f} {rt_row['sum_sq']/rt_row['df']:<15.4f} {rt_row['F']:<10.4f} {rt_row['PR(>F)']:<10.4f}")
            
            # Residual row
            print(f"{'Residual':<20} {residual_ss:<15.4f} {residual_df:<8.0f} {residual_ms:<15.4f}")
            
            # Total row
            total_ss = rt_row['sum_sq'] + residual_ss
            total_df = rt_row['df'] + residual_df
            print(f"{'Total':<20} {total_ss:<15.4f} {total_df:<8.0f}")
            
            print(f"\nSignificant: {'Yes' if rt_row['PR(>F)'] < 0.05 else 'No'}")
            
            # If significant, perform post-hoc analysis
            if rt_row['PR(>F)'] < 0.05 and len(response_groups) > 2:
                print(f"\nPost-hoc Tukey HSD (α = 0.05):")
                
                # Prepare data for Tukey HSD
                all_data = np.concatenate(response_groups)
                all_groups = []
                for i, group in enumerate(response_groups):
                    all_groups.extend([group_labels[i]] * len(group))
                
                tukey_results = pairwise_tukeyhsd(all_data, all_groups, alpha=0.05)
                print(tukey_results)
                
                # Print significant comparisons
                print("\nSignificant pairwise comparisons (p < 0.05):")
                sig_found = False
                for i, row in enumerate(tukey_results._results_table.data[1:]):
                    if row[5]:  # reject column (True if significant)
                        print(f"  {row[0]} vs {row[1]}: p = {row[4]:.4f}")
                        sig_found = True
                if not sig_found:
                    print("  No significant pairwise differences found")
                    
        except Exception as e:
            print(f"Error in statistical analysis: {e}")

def analyze_between_phase_stats(stats_df):
    """
    Perform mixed-effects/two-way ANOVA for phase comparisons.
    Factor 1: Phase (2 levels)
    Factor 2: Response Type (4 levels)
    Dependent Variable: Z-scored ΔF/F
    """
    print("\n" + "="*80)
    print("BETWEEN-PHASE STATISTICAL ANALYSIS")
    print("="*80)
    
    # Define the phase pairings - EXACTLY matching plotting code
    pairings = [
        ('0_Early', '1_Late'),  # Early phases
        ('2_SucOmit','3_CueOmit'),  # Early phases  
        ('4_RevEarly', '5_RevLate')  # Reversal phases
    ]
    
    for phase1, phase2 in pairings:
        # Filter data for this pairing
        pair_data = stats_df[stats_df['phase'].isin([phase1, phase2])].copy()
        
        if len(pair_data) == 0:
            continue
            
        print(f"\nComparison: {phase1} vs {phase2}")
        print("-" * 50)
        
        # Check if we have data for both phases
        phases_present = pair_data['phase'].unique()
        if len(phases_present) < 2:
            print(f"Insufficient phases for analysis (only {phases_present})")
            continue
        
        # Print sample sizes by condition
        print("Sample sizes by condition:")
        crosstab = pd.crosstab(pair_data['phase'], pair_data['response_type'], margins=True)
        print(crosstab)
        
        # Check if we have sufficient data for analysis
        min_cells_per_condition = 3
        condition_counts = pair_data.groupby(['phase', 'response_type']).size()
        insufficient_conditions = condition_counts[condition_counts < min_cells_per_condition]
        
        if len(insufficient_conditions) > 0:
            print(f"\nWarning: Some conditions have < {min_cells_per_condition} cells:")
            for (phase, resp_type), count in insufficient_conditions.items():
                print(f"  {phase} - {resp_type}: n = {count}")
        
        # Perform two-way ANOVA using OLS
        try:
            # Fit the model
            model = ols('zscored_dff ~ C(phase) + C(response_type) + C(phase):C(response_type)', 
                       data=pair_data).fit()
            
            # Perform ANOVA
            anova_results = anova_lm(model, typ=2)
            
            # Get residual statistics
            residual_df = model.df_resid
            residual_ss = model.ssr
            residual_ms = residual_ss / residual_df
            
            print(f"\nTwo-way ANOVA Results:")
            print("=" * 90)
            print(f"{'Source':<25} {'SS':<15} {'df':<8} {'MS':<15} {'F':<12} {'p-value':<12}")
            print("-" * 90)
            
            # Print each source of variation
            for source in anova_results.index:
                if source == 'Residual':
                    continue
                    
                ss_val = anova_results.loc[source, 'sum_sq']
                df_val = anova_results.loc[source, 'df']
                ms_val = ss_val / df_val
                f_val = anova_results.loc[source, 'F']
                p_val = anova_results.loc[source, 'PR(>F)']
                
                # Clean up source names for display
                if 'C(phase)' in source and ':' not in source:
                    source_name = 'Phase'
                elif 'C(response_type)' in source and ':' not in source:
                    source_name = 'Response Type'
                elif ':' in source:
                    source_name = 'Phase × Response Type'
                else:
                    source_name = source
                
                print(f"{source_name:<25} {ss_val:<15.4f} {df_val:<8.0f} {ms_val:<15.4f} {f_val:<12.4f} {p_val:<12.4f}")
            
            # Add residual row
            print(f"{'Residual':<25} {residual_ss:<15.4f} {residual_df:<8.0f} {residual_ms:<15.4f}")
            
            # Add total row
            total_ss = anova_results['sum_sq'].sum() + residual_ss
            total_df = anova_results['df'].sum() + residual_df
            print(f"{'Total':<25} {total_ss:<15.4f} {total_df:<8.0f}")
            
            print("-" * 90)
            
            # Print significance summary
            print("\nSignificance Summary (α = 0.05):")
            for source in anova_results.index:
                if source == 'Residual':
                    continue
                p_val = anova_results.loc[source, 'PR(>F)']
                
                # Clean up source names
                if 'C(phase)' in source and ':' not in source:
                    source_name = 'Phase main effect'
                elif 'C(response_type)' in source and ':' not in source:
                    source_name = 'Response Type main effect'
                elif ':' in source:
                    source_name = 'Phase × Response Type interaction'
                else:
                    source_name = source
                    
                sig = 'Significant' if p_val < 0.05 else 'Not significant'
                print(f"  {source_name}: {sig} (p = {p_val:.4f})")
            
            # Print model summary statistics
            print(f"\nModel Summary:")
            print(f"R-squared: {model.rsquared:.4f}")
            print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
            print(f"Total observations: {int(model.nobs)}")
            
            # If there are significant effects, perform post-hoc analysis
            significant_effects = []
            for source in anova_results.index:
                if source != 'Residual' and anova_results.loc[source, 'PR(>F)'] < 0.05:
                    significant_effects.append(source)
            
            if significant_effects:
                print(f"\nPost-hoc Analysis (Tukey HSD):")
                print("=" * 40)
                
                # For significant main effects or interactions, perform post-hoc tests
                if any('C(phase)' in effect and ':' not in effect for effect in significant_effects):
                    print("\nPhase main effect post-hoc:")
                    phase_data = pair_data.groupby('phase')['zscored_dff'].apply(list)
                    tukey_phase = pairwise_tukeyhsd(pair_data['zscored_dff'], 
                                                   pair_data['phase'], alpha=0.05)
                    print(tukey_phase)
                
                if any('C(response_type)' in effect and ':' not in effect for effect in significant_effects):
                    print("\nResponse Type main effect post-hoc:")
                    tukey_resp = pairwise_tukeyhsd(pair_data['zscored_dff'], 
                                                  pair_data['response_type'], alpha=0.05)
                    print(tukey_resp)
                
                if any(':' in effect for effect in significant_effects):
                    print("\nInteraction effect post-hoc (all pairwise comparisons):")
                    # Create combined factor for interaction post-hoc
                    pair_data['phase_response'] = pair_data['phase'] + '_' + pair_data['response_type']
                    tukey_interaction = pairwise_tukeyhsd(pair_data['zscored_dff'], 
                                                        pair_data['phase_response'], alpha=0.05)
                    print(tukey_interaction)
                    
                    # Print significant interaction comparisons
                    print("\nSignificant interaction comparisons (p < 0.05):")
                    sig_found = False
                    for i, row in enumerate(tukey_interaction._results_table.data[1:]):
                        if row[5]:  # reject column (True if significant)
                            print(f"  {row[0]} vs {row[1]}: p = {row[4]:.4f}")
                            sig_found = True
                    if not sig_found:
                        print("  No significant pairwise differences found")
                        
        except Exception as e:
            print(f"Error in statistical analysis: {e}")
            import traceback
            traceback.print_exc()

def run_complete_statistical_analysis(df_master, aucfirstframe=22, auclastframe=68, 
                                    baselinefirstframe=0, baselinelastframe=15):
    """
    Run complete statistical analysis for both within-phase and between-phase comparisons.
    """
    print("Extracting Z-scored data for statistical analysis...")
    
    # Extract the data using the same methodology as plotting functions
    stats_df = extract_zscored_data_for_stats(df_master, aucfirstframe, auclastframe, 
                                            baselinefirstframe, baselinelastframe)
    
    print(f"Total data points extracted: {len(stats_df)}")
    print(f"Phases: {sorted(stats_df['phase'].unique())}")
    print(f"Response types: {sorted(stats_df['response_type'].unique())}")
    
    # Run within-phase analysis (one-way ANOVAs)
    analyze_within_phase_stats(stats_df)
    
    # Run between-phase analysis (two-way ANOVAs)  
    analyze_between_phase_stats(stats_df)
    
    return stats_df

# Example usage:
stats_data = run_complete_statistical_analysis(df_master, aucfirstframe=22, auclastframe=68, 
                                              baselinefirstframe=0, baselinelastframe=15)

# response types into df_master_omission_jp 

In [ ]:
def process_and_plot_high_licking_trials_omission(df_master_omission_jp, days, pre_window_size, infusionframe, baselinefirstframe, baselinelastframe, basedir, sex_selection='all', hz_threshold=1.5, framerate=30, frameaveraging=4):
    """
    Process and plot high licking trials using direct timestamp-based rate calculations
    when available, falling back to frame-based calculations when necessary.
    Adapted for df_master_omission_jp.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    
    # Define frame rate from parameters
    averagedframerate = framerate / frameaveraging  # Should be 7.5 Hz with default parameters
    
    # Calculate seconds in each window for rate conversion
    seconds_per_frame = 1.0 / averagedframerate
    baseline_seconds = (baselinelastframe - baselinefirstframe) * seconds_per_frame
    event_seconds = (infusionframe - pre_window_size) * seconds_per_frame
    
    print(f"Frame rate parameters: raw={framerate}Hz, averaging={frameaveraging}, effective={averagedframerate}Hz")
    print(f"Converting to rates: Baseline window = {baseline_seconds:.2f} sec, Event window = {event_seconds:.2f} sec")
    
    # Check if timestamp-based data is available
    has_timestamp_data = 'ts_delta_rate' in df_master_omission_jp.columns
    
    # Create df_unique_trials for accurate statistics and visualizations
    df_unique_trials = df_master_omission_jp.drop_duplicates(
        subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
    ).copy()
    
    # Compute and print statistics for each day and CS type
    daily_stats = []
    
    for day in sorted(days):
        day_data = df_unique_trials[df_unique_trials['day'] == day]
        for cs_type in ['CS+', 'CS-']:
            cs_data = day_data[day_data['cs_type'] == cs_type]
            if cs_data.empty:
                continue
                
            # Determine which delta rate to use (timestamp-based preferred)
            if has_timestamp_data:
                delta_rates = cs_data['ts_delta_rate'].values
                rate_source = "timestamp-based"
            else:
                # Calculate frame-based delta rates
                delta_lick_cols = [col for col in cs_data.columns if col.startswith('delta_licks_frame_')]
                cs_delta_licks = cs_data[delta_lick_cols].values
                
                # Calculate lick counts
                cs_licking_event = np.sum(cs_delta_licks[:, pre_window_size:infusionframe], axis=1)
                cs_licking_baseline = np.sum(cs_delta_licks[:, baselinefirstframe:baselinelastframe], axis=1)
                
                # Convert to rates (Hz)
                cs_event_rate = cs_licking_event / event_seconds
                cs_baseline_rate = cs_licking_baseline / baseline_seconds
                
                # Calculate rate delta
                delta_rates = cs_event_rate - cs_baseline_rate
                rate_source = "frame-based"
            
            # Calculate stats for this day/CS type combination
            daily_stats.append({
                'day': day,
                'cs_type': cs_type,
                'mean': np.mean(delta_rates),
                'median': np.median(delta_rates),
                'n_trials': len(delta_rates),
                'source': rate_source
            })
    
    # Print detailed statistics based on unique trials
    print("\nDetailed Statistics (Rate-Based, Unique Trials):")
    print("=" * 60)
    for stat in daily_stats:
        print(f"Day {stat['day']} {stat['cs_type']} ({stat['source']}):")
        print(f"  Mean licking delta rate: {stat['mean']:.2f} Hz")
        print(f"  Median licking delta rate: {stat['median']:.2f} Hz")
        print(f"  Number of trials: {stat['n_trials']}")
        print("-" * 30)
    
    # Set threshold for high licking in Hz
    print(f"\nRate threshold = {hz_threshold:.2f} Hz")
    print("=" * 60)
    
    # Plot setup
    numdays = len(days)
    num_rows = 2  # Just CS+ and CS- for any selection
    fig, axs = plt.subplots(num_rows, numdays, figsize=(numdays*3, 8), sharex=True, sharey=False)
    if numdays == 1:  # Handle the case of single day
        axs = axs.reshape(-1, 1)
    sns.set_style('white')
    colors = ['deeppink', 'gray']  # Updated color scheme
    
    # Ensure high_licking column exists in df_master_omission_jp
    if 'high_licking' not in df_master_omission_jp.columns:
        df_master_omission_jp['high_licking'] = 0
    
    # Determine which method to use for high licking classification
    if has_timestamp_data:
        print("Using timestamp-based delta rates for high licking classification")
        # Copy the timestamp-based classification to high_licking column
        df_master_omission_jp['high_licking'] = df_master_omission_jp['ts_high_licking']
    else:
        print("Using frame-based delta rates for high licking classification")
        # Process frame-based data and update df_master_omission_jp
        for day in sorted(days):
            # Create a temp df for rate calculations
            df_temp = df_master_omission_jp[df_master_omission_jp['day'] == day].copy()
            
            for cs_type in ['CS+', 'CS-']:
                # Filter data
                cs_mask = df_temp['cs_type'] == cs_type
                if not any(cs_mask):
                    continue
                    
                delta_lick_cols = [col for col in df_temp.columns if col.startswith('delta_licks_frame_')]
                
                # Get lick data for these trials
                lick_data = df_temp.loc[cs_mask, delta_lick_cols].values
                
                # Calculate lick counts then rates
                licking_event = np.sum(lick_data[:, pre_window_size:infusionframe], axis=1)
                licking_baseline = np.sum(lick_data[:, baselinefirstframe:baselinelastframe], axis=1)
                
                event_rate = licking_event / event_seconds
                baseline_rate = licking_baseline / baseline_seconds
                
                rate_delta = event_rate - baseline_rate
                
                # Set high_licking flags
                licking_flag = rate_delta > hz_threshold
                df_master_omission_jp.loc[(df_master_omission_jp['day'] == day) & (df_master_omission_jp['cs_type'] == cs_type), 'high_licking'] = licking_flag.astype(int)
    
    # Get fresh unique trials with updated high_licking values
    df_unique_trials = df_master_omission_jp.drop_duplicates(
        subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
    ).copy()
    
    # Create pie charts for each day and CS type
    for d, day in enumerate(sorted(days)):
        for cs_type, cs_row_index in zip(['CS+', 'CS-'], [0, 1]):
            ax = axs[cs_row_index, d]
            
            if sex_selection == 'all':
                plot_data = df_unique_trials[(df_unique_trials['day'] == day) & 
                                         (df_unique_trials['cs_type'] == cs_type)]
            else:
                plot_data = df_unique_trials[(df_unique_trials['day'] == day) & 
                                         (df_unique_trials['cs_type'] == cs_type) & 
                                         (df_unique_trials['sex'] == sex_selection)]
            
            if not plot_data.empty:
                high_licking_count = plot_data['high_licking'].sum()
                total_trials = len(plot_data)
                
                highlicking_percentage = high_licking_count / total_trials
                lowlicking_percentage = 1 - highlicking_percentage
                
                counts = np.array([highlicking_percentage, lowlicking_percentage])
                labels = [f"High: {high_licking_count}/{total_trials}\n({highlicking_percentage*100:.1f}%)", 
                          f"Low: {total_trials-high_licking_count}/{total_trials}\n({lowlicking_percentage*100:.1f}%)"]
                
                ax.pie(counts, colors=colors, labels=labels)
            else:
                ax.text(0.5, 0.5, "No data", ha='center', va='center', transform=ax.transAxes, fontsize=12)
                ax.set_frame_on(False)
            
            if d == 0:
                ax.set_ylabel(f'{cs_type}', fontsize=12)
            ax.set_title(day.replace('_', ' '), fontsize=12)
    
    method_text = "Timestamp-based" if has_timestamp_data else "Frame-based"
    title_suffix = ' - All' if sex_selection == 'all' else f' - {sex_selection.capitalize()}'
    fig.suptitle(f'High Licking Trials ({method_text}, Threshold = {hz_threshold:.1f} Hz){title_suffix}', fontsize=16)
    fig.tight_layout()
    
    # Save figures
    # suffix = '' if sex_selection == 'all' else f'_{sex_selection}'
    # method_suffix = '_TS' if has_timestamp_data else '_FB'
    # fig.savefig(os.path.join(basedir, 'Results', f'HighLickingTrials_Omission{method_suffix}_PieCharts{suffix}.PNG'), format='PNG')
    # fig.savefig(os.path.join(basedir, 'Results', f'HighLickingTrials_Omission{method_suffix}_PieCharts{suffix}.PDF'), format='PDF')
    plt.show()
    
    # Create classification mapping for SDT metrics
    def classify_response(day, cs_type, is_high_licking):
        # Check if we're in a reversal phase
        is_reversal = day.startswith('4_Rev') or day.startswith('5_Rev')
        
        if not is_reversal:
            # Initial learning phases
            if cs_type == 'CS+':
                return 'correct response' if is_high_licking else 'incorrect withholding'
            else:  # CS-
                return 'incorrect response' if is_high_licking else 'correct withholding'
        else:
            # Reversal phases
            if cs_type == 'CS+':
                return 'incorrect response' if is_high_licking else 'correct withholding'
            else:  # CS-
                return 'correct response' if is_high_licking else 'incorrect withholding'
    
    # Add response classification to df_master_omission_jp
    df_master_omission_jp['response_type'] = df_master_omission_jp.apply(
        lambda row: classify_response(row['day'], row['cs_type'], bool(row['high_licking'])),
        axis=1
    )
    
    # Print overall SDT classification results
    print("\nSignal Detection Theory Classification:")
    print("=" * 60)
    
    for day in sorted(days):
        # Get unique trials for this day
        day_trials = df_unique_trials[df_unique_trials['day'] == day].copy()
        
        if not day_trials.empty:
            day_trials['response_type'] = day_trials.apply(
                lambda row: classify_response(row['day'], row['cs_type'], bool(row['high_licking'])),
                axis=1
            )
            
            # Count each response type
            response_counts = day_trials['response_type'].value_counts()
            total_trials = len(day_trials)
            
            # Calculate percentages
            response_percentages = (response_counts / total_trials * 100).round(1)
            
            print(f"\nDay {day}:")
            print(f"Total trials: {total_trials}")
            
            # Print each response type
            for resp_type in ['correct response', 'correct withholding', 
                             'incorrect response', 'incorrect withholding']:
                if resp_type in response_counts:
                    count = response_counts[resp_type]
                    percent = response_percentages[resp_type]
                    print(f"  {resp_type}: {count} trials ({percent}%)")
                else:
                    print(f"  {resp_type}: 0 trials (0.0%)")
    
    return df_master_omission_jp

# Usage
df_master_omission_jp = process_and_plot_high_licking_trials_omission(
    df_master_omission_jp, days, pre_window_size, infusionframe, 
    baselinefirstframe, baselinelastframe, basedir, 
    sex_selection='all',
    hz_threshold=1.5,
    framerate=30, 
    frameaveraging=4
)

In [ ]:
def add_gng_columns_omission(df_master_omission_jp):
    """
    Add go/no-go response type columns to df_master_omission_jp
    """
    # Initialize new columns with 0
    new_columns = ['hits', 'correct_rejections', 'misses', 'false_alarms']
    for col in new_columns:
        df_master_omission_jp[col] = 0
    
    # Define day groups
    early_days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit']
    rev_days = ['4_RevEarly', '5_RevLate']
    
    # Process each row
    for index, row in df_master_omission_jp.iterrows():
        day = row['day']
        cs_type = row['cs_type']
        high_licking = row['high_licking']

        # Early training days (CS+ is go cue, CS- is no-go cue)
        if day in early_days:
            if cs_type == 'CS+':
                if high_licking == 1:
                    df_master_omission_jp.at[index, 'hits'] = 1
                else:
                    df_master_omission_jp.at[index, 'misses'] = 1
            else:  # CS-
                if high_licking == 0:
                    df_master_omission_jp.at[index, 'correct_rejections'] = 1
                else:
                    df_master_omission_jp.at[index, 'false_alarms'] = 1

        # Reversal days (CS- is go cue, CS+ is no-go cue)
        else:  # rev_days
            if cs_type == 'CS-':
                if high_licking == 1:
                    df_master_omission_jp.at[index, 'hits'] = 1
                else:
                    df_master_omission_jp.at[index, 'misses'] = 1
            else:  # CS+
                if high_licking == 0:
                    df_master_omission_jp.at[index, 'correct_rejections'] = 1
                else:
                    df_master_omission_jp.at[index, 'false_alarms'] = 1
    
    return df_master_omission_jp

# Apply the function
df_master_omission_jp = add_gng_columns_omission(df_master_omission_jp)

def print_gng_stats_omission(df_master_omission_jp):
    """
    Print Go/No-Go statistics by day for df_master_omission_jp, using unique trials only
    """
    print("\nGo/No-Go Statistics by Day (Unique Trials) - Omission DataFrame")
    print("=" * 80)
    
    for day in sorted(df_master_omission_jp['day'].unique()):
        print(f"\nDay: {day}")
        print("-" * 40)
        
        # Get data for this day
        day_data = df_master_omission_jp[df_master_omission_jp['day'] == day]
        
        # Create unique trials dataframe
        unique_trials = day_data.drop_duplicates(
            subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
        ).copy()
        
        # Get unique CS+ and CS- trials
        csplus_trials = unique_trials[unique_trials['cs_type'] == 'CS+']
        csminus_trials = unique_trials[unique_trials['cs_type'] == 'CS-']
        
        # Calculate total unique trials
        total_unique_trials = len(unique_trials)
        
        if total_unique_trials > 0:
            # Calculate stats for each measure using unique trials
            hits = unique_trials['hits'].sum()
            crs = unique_trials['correct_rejections'].sum()
            misses = unique_trials['misses'].sum()
            fas = unique_trials['false_alarms'].sum()
            
            # Get CS+ and CS- totals
            total_csplus = len(csplus_trials)
            total_csminus = len(csminus_trials)
            
            # Calculate overall percentages
            hit_rate = hits / total_unique_trials * 100
            cr_rate = crs / total_unique_trials * 100
            miss_rate = misses / total_unique_trials * 100
            fa_rate = fas / total_unique_trials * 100
            
            # Calculate CS type-specific percentages
            csplus_hit_rate = hits / total_csplus * 100 if total_csplus > 0 else 0
            csplus_miss_rate = misses / total_csplus * 100 if total_csplus > 0 else 0
            csminus_cr_rate = crs / total_csminus * 100 if total_csminus > 0 else 0
            csminus_fa_rate = fas / total_csminus * 100 if total_csminus > 0 else 0
            
            print(f"Total unique trials: {total_unique_trials} (CS+: {total_csplus}, CS-: {total_csminus})")
            print(f"Hits              : {hits:5d} trials ({hit_rate:5.1f}% of all, {csplus_hit_rate:5.1f}% of CS+)")
            print(f"Correct Rejections: {crs:5d} trials ({cr_rate:5.1f}% of all, {csminus_cr_rate:5.1f}% of CS-)")
            print(f"Misses            : {misses:5d} trials ({miss_rate:5.1f}% of all, {csplus_miss_rate:5.1f}% of CS+)")
            print(f"False Alarms      : {fas:5d} trials ({fa_rate:5.1f}% of all, {csminus_fa_rate:5.1f}% of CS-)")
            
            # Calculate d' (sensitivity index) if possible
            if csplus_hit_rate > 0 and csminus_fa_rate > 0:
                import scipy.stats as stats
                
                # Convert percentages to proportions
                hit_prop = csplus_hit_rate / 100
                fa_prop = csminus_fa_rate / 100
                
                # Adjust extreme values to avoid infinite d'
                hit_prop = min(max(hit_prop, 0.01), 0.99)
                fa_prop = min(max(fa_prop, 0.01), 0.99)
                
                # Calculate d'
                d_prime = stats.norm.ppf(hit_prop) - stats.norm.ppf(fa_prop)
                print(f"Sensitivity (d'): {d_prime:.2f}")
        else:
            print("No trials found for this day")

# Print the stats
print_gng_stats_omission(df_master_omission_jp)

In [ ]:
def plot_gng_types_omission(df_master_omission_jp, days, basedir, sex_selection='all'):
    """
    Plot go/no-go response types for df_master_omission_jp
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    
    numdays = len(days)
    fig, axs = plt.subplots(1, numdays, figsize=(numdays*3, 5), sharex=True, sharey=False)
    if numdays == 1:  # Handle single day case
        axs = [axs]
    sns.set_style('white')
    
    # Colors for CS+ and CS- responses
    colors_csplus = ['lawngreen',  # hits
                    'black']     # misses
    colors_csminus = ['palegreen',   # correct rejections
                     'grey']   # false alarms
    colors = colors_csplus + colors_csminus
    
    for d, day in enumerate(sorted(days)):
        # Filter data based on sex selection
        if sex_selection == 'all':
            day_data = df_master_omission_jp[df_master_omission_jp['day'] == day]
        else:
            day_data = df_master_omission_jp[(df_master_omission_jp['day'] == day) & 
                                            (df_master_omission_jp['sex'] == sex_selection)]
        
        # Create unique trials dataframe to get accurate trial counts
        plot_data = day_data.drop_duplicates(
            subset=['animal', 'day', 'fov', 'trial_id', 'cs_type']
        ).copy()
        
        # Calculate totals for each response type
        total_trials = len(plot_data)
        if total_trials > 0:
            # Get unique CS+ and CS- trials
            csplus_data = plot_data[plot_data['cs_type'] == 'CS+']
            csminus_data = plot_data[plot_data['cs_type'] == 'CS-']
            
            # Calculate response types based on high_licking column
            hits = csplus_data['high_licking'].sum()
            misses = len(csplus_data) - hits
            fas = csminus_data['high_licking'].sum()
            crs = len(csminus_data) - fas
            
            # Calculate CS+ and CS- totals
            total_csplus = len(csplus_data)
            total_csminus = len(csminus_data)
            
            # Calculate normalized percentages (50% for each CS type)
            if total_csplus > 0:
                hits_pct = (hits / total_csplus) * 50
                misses_pct = (misses / total_csplus) * 50
            else:
                hits_pct = misses_pct = 0
                
            if total_csminus > 0:
                crs_pct = (crs / total_csminus) * 50
                fas_pct = (fas / total_csminus) * 50
            else:
                crs_pct = fas_pct = 0
            
            counts = [hits_pct, misses_pct, crs_pct, fas_pct]
            labels = [f"Correct Response: {hits_pct:.1f}%\n({hits})",
                     f"Incorrect Withholding: {misses_pct:.1f}%\n({misses})",
                     f"Correct Withholding: {crs_pct:.1f}%\n({crs})",
                     f"Incorrect Response: {fas_pct:.1f}%\n({fas})"]
        else:
            counts = [0, 0, 0, 1]
            labels = ["No data", "", "", ""]
        
        # Create pie chart with CS+/CS- split
        ax = axs[d]
        wedges, texts = ax.pie(counts, colors=colors, labels=labels)
        
        # Add a line to visually separate CS+ and CS- halves
        ax.axhline(y=0, color='white', linewidth=3)
        
        # Add titles
        ax.set_title(day)
    
    # Add legend with CS+/CS- grouping
    legend_labels = ['Correct Licking', 
                    'Incorrect Withholding',
                    'Correct Withholding',
                    'Incorrect Licking']
    fig.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0.5))
    
    # Add overall title
    title_suffix = ' - All' if sex_selection == 'all' else f' - {sex_selection.capitalize()}'
    fig.suptitle(f'CS+/CS- Response Types by Day{title_suffix} (Omission DF)', fontsize=16)
    
    # Adjust layout to accommodate legend
    plt.subplots_adjust(right=0.85)
    
    # Save figures
    # suffix = '' if sex_selection == 'all' else f'_{sex_selection}'
    # fig.savefig(os.path.join(basedir, 'Results', f'GNG_ResponseTypes_Omission_PieCharts{suffix}.PNG'), 
    #             format='PNG', bbox_inches='tight')
    # fig.savefig(os.path.join(basedir, 'Results', f'GNG_ResponseTypes_Omission_PieCharts{suffix}.PDF'), 
    #             format='PDF', bbox_inches='tight')
    plt.show()

# Usage examples:
plot_gng_types_omission(df_master_omission_jp, days, basedir, sex_selection='all')
# plot_gng_types_omission(df_master_omission_jp, days, basedir, sex_selection='male')
# plot_gng_types_omission(df_master_omission_jp, days, basedir, sex_selection='female')

In [ ]:
####omission CS+ responses
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

def plot_zscored_df_f_sucrose_omission_response_types(df_master_omission_jp, aucfirstframe, auclastframe, 
                                                       baselinefirstframe, baselinelastframe, 
                                                       basedir=None):
    """
    Create plot for 2_SucOmit phase showing Z-scored ΔF/F values for cells 
    that participated in hits vs misses for CS+ rewarded and CS+ unrewarded trials.
    
    Parameters:
        df_master_omission_jp (pandas.DataFrame): Master dataframe with omission data
        aucfirstframe (int): First frame of the response window to average
        auclastframe (int): Last frame of the response window to average
        baselinefirstframe (int): First frame of the baseline period for Z-scoring
        baselinelastframe (int): Last frame of the baseline period for Z-scoring
        basedir (str, optional): Base directory for saving plots
    """
    
    # Define colors for each condition
    condition_colors = {
        'rewarded_hits': 'lawngreen',
        'rewarded_misses': 'black',
        'unrewarded_hits': 'lawngreen',
        'unrewarded_misses': 'black'
    }
    
    # Print header
    print("\n" + "="*80)
    print("2_SucOmit ANALYSIS: Z-SCORED ΔF/F BY REWARD STATUS AND RESPONSE TYPE")
    print("="*80)
    
    # Filter for 2_SucOmit phase only
    phase_data = df_master_omission_jp[df_master_omission_jp['experimental_phase'] == '2_SucOmit']
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Lists to store data for plotting and statistics
    all_data = []
    positions = []
    labels = []
    colors = []
    stats_data = []  # For 2x2 ANOVA
    
    # Current position for plotting
    pos = 1
    
    print(f"\nPhase: 2_SucOmit")
    print("-"*50)
    
    # Process each combination of reward status and response type
    conditions = [
        ('rewarded', 'hits', 1),
        ('rewarded', 'misses', 1),
        ('unrewarded', 'hits', 0),
        ('unrewarded', 'misses', 0)
    ]
    
    for reward_type, resp_type, reward_status_val in conditions:
        # Get cells for this condition
        # Filter for CS+ trials with specific reward status and response type
        condition_cells = phase_data[
            (phase_data['cs_type'] == 'CS+') & 
            (phase_data['CS+ Reward Status'] == reward_status_val) &
            (phase_data[resp_type] == 1)
        ]
        
        # Get unique cell IDs for this condition
        unique_cells = condition_cells['cell_id'].unique()
        
        if len(unique_cells) == 0:
            print(f"  CS+ {reward_type} {resp_type}: No cells available")
            continue
        
        # Get fluorescence columns for CS+
        fluor_cols = [col for col in condition_cells.columns if col.startswith('fluor_plus_frame_')]
        
        if len(fluor_cols) == 0:
            # Try using the appropriate omission-specific columns
            if reward_type == 'rewarded':
                fluor_cols = [col for col in condition_cells.columns if 'fluor' in col and 'frame' in col]
            else:
                fluor_cols = [col for col in condition_cells.columns if 'fluor' in col and 'frame' in col]
        
        # Calculate Z-scored values for each cell
        zscored_values = []
        
        # Group by cell_id and average trials first
        grouped_cells = condition_cells.groupby('cell_id')
        
        for cell_id, cell_group in grouped_cells:
            # Average fluorescence data across trials for this cell
            avg_fluor = cell_group[fluor_cols].mean().values
            
            # Skip cells with no data
            if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                continue
            
            # Calculate standard deviation of baseline period
            baseline_indices = list(range(baselinefirstframe, baselinelastframe))
            neuron_std = np.nanstd(avg_fluor[baseline_indices])
            
            # Skip cells with zero std dev
            if neuron_std <= 0:
                continue
            
            # Z-score by dividing by std dev (AFTER averaging)
            zscored_fluor = avg_fluor / neuron_std
            
            # Calculate average over the response window
            resp_indices = list(range(aucfirstframe, auclastframe+1))
            
            # Average across response window frames
            avg_zscored = np.nanmean(zscored_fluor[resp_indices])
            zscored_values.append(avg_zscored)
            
            # Store for statistical analysis
            stats_data.append({
                'cell_id': cell_id,
                'reward_status': reward_type,
                'response_type': resp_type,
                'z_score': avg_zscored
            })
        
        cell_count = len(zscored_values)
        print(f"  CS+ {reward_type} {resp_type}: {cell_count} cells")
        
        if cell_count > 0:
            all_data.append(zscored_values)
            positions.append(pos)
            labels.append(f"CS+ {reward_type.capitalize()}\n{resp_type.capitalize()}\nn={cell_count}")
            colors.append(condition_colors[f'{reward_type}_{resp_type}'])
            pos += 1.5  # Add spacing between conditions
    
    # Create box plots if we have data
    if all_data:
        box = ax.boxplot(all_data, positions=positions[:len(all_data)], patch_artist=True, 
                        showfliers=False, widths=0.7)
        
        # Customize box colors
        for patch, color in zip(box['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Add strip plots with jitter
        for i, d in enumerate(all_data):
            x = np.random.normal(positions[i], 0.01, size=len(d))  # Add jitter
            ax.scatter(x, d, alpha=0.4, edgecolor='black', s=30, color=colors[i])
        
        # Change median line color
        for median in box['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        
        # Add total cell count to title
        total_cells = sum(len(d) for d in all_data)
        
        # Customize the plot
        ax.set_xticks(positions[:len(all_data)])
        ax.set_xticklabels(labels, rotation=0, ha='center')
        ax.set_ylabel('Z-scored ΔF/F')
        ax.set_title(f'2_SucOmit: Z-scored ΔF/F (Frames {aucfirstframe}-{auclastframe}) by Reward Status and Response Type\n(Total n={total_cells} cells)')
        ax.axhline(0.0, color='black', linestyle='--', linewidth=0.5)
        
        # Set y-limits
        ax.set_ylim(-5, 25)
        ax.set_yticks([-5, 0, 5, 10, 15, 20, 25])
        
        # Add vertical line to separate rewarded vs unrewarded
        if len(positions) >= 3:
            midpoint = (positions[1] + positions[2]) / 2
            ax.axvline(x=midpoint, color='black', linestyle=':', linewidth=1, alpha=0.5)
            
            # Add group labels
            ax.text((positions[0] + positions[1])/2, ax.get_ylim()[1]*0.95, 'REWARDED', 
                   horizontalalignment='center', fontsize=11, fontweight='bold')
            if len(positions) >= 4:
                ax.text((positions[2] + positions[3])/2, ax.get_ylim()[1]*0.95, 'OMITTED', 
                       horizontalalignment='center', fontsize=11, fontweight='bold')
        
        plt.tight_layout()
        
        # Save the plot if basedir is provided
        if basedir:
            pdf_filename = os.path.join(basedir, "Results", 
                                      f'ZscoredDFF_2SucOmit_RewardStatus_ResponseType_Frames{aucfirstframe}-{auclastframe}_08-07-25.pdf')
            plt.savefig(pdf_filename)
            
            png_filename = os.path.join(basedir, "Results", 
                                      f'ZscoredDFF_2SucOmit_RewardStatus_ResponseType_Frames{aucfirstframe}-{auclastframe}_08-07-25.png')
            plt.savefig(png_filename, dpi=300)
        
        plt.show()
        
        # Perform 2x2 ANOVA if we have data for statistical analysis
        if stats_data:
            print("\n" + "="*50)
            print("STATISTICAL ANALYSIS: 2x2 ANOVA")
            print("="*50)
            
            # Convert to DataFrame for easier analysis
            stats_df = pd.DataFrame(stats_data)
            
            # Check if we have data for both factors
            unique_reward = stats_df['reward_status'].unique()
            unique_response = stats_df['response_type'].unique()
            
            if len(unique_reward) == 2 and len(unique_response) == 2:
                # Perform 2x2 ANOVA
                model = ols('z_score ~ C(reward_status) + C(response_type) + C(reward_status):C(response_type)', 
                           data=stats_df).fit()
                anova_table = anova_lm(model, typ=2)
                
                print("\n2x2 ANOVA Results:")
                print("-"*30)
                print(anova_table)
                
                # Calculate means for each condition
                print("\nCondition Means:")
                print("-"*30)
                condition_means = stats_df.groupby(['reward_status', 'response_type'])['z_score'].agg(['mean', 'std', 'count'])
                print(condition_means)
                
                # Perform post-hoc t-tests for specific comparisons
                print("\nPost-hoc Comparisons:")
                print("-"*30)
                
                # Compare hits vs misses within rewarded
                rewarded_hits = stats_df[(stats_df['reward_status'] == 'rewarded') & 
                                        (stats_df['response_type'] == 'hits')]['z_score']
                rewarded_misses = stats_df[(stats_df['reward_status'] == 'rewarded') & 
                                          (stats_df['response_type'] == 'misses')]['z_score']
                
                if len(rewarded_hits) > 0 and len(rewarded_misses) > 0:
                    t_stat, p_val = stats.ttest_ind(rewarded_hits, rewarded_misses)
                    print(f"Rewarded: Hits vs Misses - t={t_stat:.3f}, p={p_val:.4f}")
                
                # Compare hits vs misses within unrewarded
                unrewarded_hits = stats_df[(stats_df['reward_status'] == 'unrewarded') & 
                                          (stats_df['response_type'] == 'hits')]['z_score']
                unrewarded_misses = stats_df[(stats_df['reward_status'] == 'unrewarded') & 
                                            (stats_df['response_type'] == 'misses')]['z_score']
                
                if len(unrewarded_hits) > 0 and len(unrewarded_misses) > 0:
                    t_stat, p_val = stats.ttest_ind(unrewarded_hits, unrewarded_misses)
                    print(f"Unrewarded: Hits vs Misses - t={t_stat:.3f}, p={p_val:.4f}")
                
                # Compare rewarded vs unrewarded within hits
                if len(rewarded_hits) > 0 and len(unrewarded_hits) > 0:
                    t_stat, p_val = stats.ttest_ind(rewarded_hits, unrewarded_hits)
                    print(f"Hits: Rewarded vs Unrewarded - t={t_stat:.3f}, p={p_val:.4f}")
                
                # Compare rewarded vs unrewarded within misses
                if len(rewarded_misses) > 0 and len(unrewarded_misses) > 0:
                    t_stat, p_val = stats.ttest_ind(rewarded_misses, unrewarded_misses)
                    print(f"Misses: Rewarded vs Unrewarded - t={t_stat:.3f}, p={p_val:.4f}")
                
            else:
                print("\nInsufficient data for 2x2 ANOVA")
                print(f"Unique reward status values: {unique_reward}")
                print(f"Unique response type values: {unique_response}")
    else:
        plt.close(fig)
        print("No data available for 2_SucOmit analysis")
    
    return stats_df if stats_data else None

# Usage
stats_results = plot_zscored_df_f_sucrose_omission_response_types(
    df_master_omission_jp, 
    aucfirstframe=22, 
    auclastframe=68, 
    baselinefirstframe=0, 
    baselinelastframe=15, 
    basedir=basedir
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

def extract_zscored_data_for_stats(df_master, aucfirstframe, auclastframe, 
                                  baselinefirstframe, baselinelastframe):
    """
    Extract Z-scored ΔF/F values using the same methodology as the plotting functions.
    Returns a structured dataframe for statistical analysis.
    """
    
    # Define which phases are reversal phases
    reversal_phases = ['4_RevEarly', '5_RevLate']
    
    # Store all data for statistical analysis
    stats_data = []
    
    # Get all unique experimental phases
    phases = sorted(df_master['experimental_phase'].unique())
    
    for phase in phases:
        # Filter for the current phase
        phase_data = df_master[df_master['experimental_phase'] == phase]
        
        # Process each response type
        for resp_type in ['hits', 'misses', 'correct_rejections', 'false_alarms']:
            # Get all unique cell_ids for this response type
            resp_cells = phase_data[phase_data[resp_type] == 1]['cell_id'].unique()
            
            if len(resp_cells) == 0:
                continue  # Skip if no cells for this response type
            
            # Determine valid stimulus types based on response type and phase
            is_reversal = phase in reversal_phases
            
            if resp_type in ['hits', 'misses']:
                # For hits and misses
                if is_reversal:
                    valid_stim = 'CS-'  # In reversal, CS- is the go cue
                else:
                    valid_stim = 'CS+'  # In early/normal, CS+ is the go cue
            else:
                # For correct rejections and false alarms
                if is_reversal:
                    valid_stim = 'CS+'  # In reversal, CS+ is the no-go cue
                else:
                    valid_stim = 'CS-'  # In early/normal, CS- is the no-go cue
            
            # Process valid stimulus type - EXACTLY matching plotting code logic
            if valid_stim == 'CS+':
                # Get CS+ data: filter for cells that participated in this response type
                cs_plus_cells = phase_data[(phase_data['cs_type'] == 'CS+') & 
                                          (phase_data['cell_id'].isin(resp_cells)) &
                                          (phase_data[resp_type] == 1)]
                
                if len(cs_plus_cells) > 0:
                    # Get fluorescence columns for CS+
                    fluor_plus_cols = [col for col in cs_plus_cells.columns if col.startswith('fluor_plus_frame_')]
                    
                    # Calculate Z-scored values for each cell following original code's approach
                    # Group by cell_id and average trials first
                    grouped_cells = cs_plus_cells.groupby('cell_id')
                    
                    for cell_id, cell_group in grouped_cells:
                        # Average fluorescence data across trials for this cell
                        avg_fluor = cell_group[fluor_plus_cols].mean().values
                        
                        # Skip cells with no data
                        if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                            continue
                        
                        # Calculate standard deviation of baseline period
                        baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                        neuron_std = np.nanstd(avg_fluor[baseline_indices])
                        
                        # Skip cells with zero std dev
                        if neuron_std <= 0:
                            continue
                        
                        # Z-score by dividing by std dev (AFTER averaging)
                        zscored_fluor = avg_fluor / neuron_std
                        
                        # Calculate average over the response window (aucfirstframe to auclastframe)
                        resp_indices = list(range(aucfirstframe, auclastframe+1))
                        
                        # Average across response window frames
                        avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                        
                        # Store the data
                        stats_data.append({
                            'cell_id': cell_id,
                            'phase': phase,
                            'response_type': resp_type,
                            'stimulus_type': valid_stim,
                            'zscored_dff': avg_zscored,
                            'is_reversal': is_reversal
                        })
            
            if valid_stim == 'CS-':
                # Get CS- data: filter for cells that participated in this response type
                cs_minus_cells = phase_data[(phase_data['cs_type'] == 'CS-') & 
                                           (phase_data['cell_id'].isin(resp_cells)) &
                                           (phase_data[resp_type] == 1)]
                
                if len(cs_minus_cells) > 0:
                    # Get fluorescence columns for CS-
                    fluor_minus_cols = [col for col in cs_minus_cells.columns if col.startswith('fluor_minus_frame_')]
                    
                    # Calculate Z-scored values for each cell following original code's approach
                    # Group by cell_id and average trials first
                    grouped_cells = cs_minus_cells.groupby('cell_id')
                    
                    for cell_id, cell_group in grouped_cells:
                        # Average fluorescence data across trials for this cell
                        avg_fluor = cell_group[fluor_minus_cols].mean().values
                        
                        # Skip cells with no data
                        if len(avg_fluor) == 0 or np.all(np.isnan(avg_fluor)):
                            continue
                        
                        # Calculate standard deviation of baseline period
                        baseline_indices = list(range(baselinefirstframe, baselinelastframe))
                        neuron_std = np.nanstd(avg_fluor[baseline_indices])
                        
                        # Skip cells with zero std dev
                        if neuron_std <= 0:
                            continue
                        
                        # Z-score by dividing by std dev (AFTER averaging)
                        zscored_fluor = avg_fluor / neuron_std
                        
                        # Calculate average over the response window (aucfirstframe to auclastframe)
                        resp_indices = list(range(aucfirstframe, auclastframe+1))
                        
                        # Average across response window frames
                        avg_zscored = np.nanmean(zscored_fluor[resp_indices])
                        
                        # Store the data
                        stats_data.append({
                            'cell_id': cell_id,
                            'phase': phase,
                            'response_type': resp_type,
                            'stimulus_type': valid_stim,
                            'zscored_dff': avg_zscored,
                            'is_reversal': is_reversal
                        })
    
    return pd.DataFrame(stats_data)

def analyze_within_phase_stats(stats_df):
    """
    Perform one-way ANOVA for each phase separately.
    Factor: Response Type (4 levels)
    Dependent Variable: Z-scored ΔF/F
    """
    print("\n" + "="*80)
    print("WITHIN-PHASE STATISTICAL ANALYSIS")
    print("="*80)
    
    phases = sorted(stats_df['phase'].unique())
    
    for phase in phases:
        phase_data = stats_df[stats_df['phase'] == phase]
        
        if len(phase_data) == 0:
            continue
            
        print(f"\nPhase: {phase}")
        print("-" * 50)
        
        # Check if we have multiple response types
        response_types = phase_data['response_type'].unique()
        if len(response_types) < 2:
            print(f"Insufficient response types for analysis (only {len(response_types)} types)")
            continue
        
        # Prepare data for ANOVA
        response_groups = []
        group_labels = []
        
        for resp_type in ['hits', 'misses', 'correct_rejections', 'false_alarms']:
            resp_data = phase_data[phase_data['response_type'] == resp_type]['zscored_dff'].values
            if len(resp_data) > 0:
                response_groups.append(resp_data)
                group_labels.append(resp_type)
        
        # Print sample sizes
        print("Sample sizes:")
        for i, label in enumerate(group_labels):
            print(f"  {label}: n = {len(response_groups[i])}")
        
        if len(response_groups) < 2:
            print("Insufficient groups for analysis")
            continue
        
        # Perform one-way ANOVA
        try:
            f_stat, p_val = f_oneway(*response_groups)
            print(f"\nOne-way ANOVA Results:")
            print(f"F-statistic: {f_stat:.4f}")
            print(f"p-value: {p_val:.4f}")
            print(f"Significant: {'Yes' if p_val < 0.05 else 'No'}")
            
            # If significant, perform post-hoc analysis
            if p_val < 0.05 and len(response_groups) > 2:
                print(f"\nPost-hoc Tukey HSD (α = 0.05):")
                
                # Prepare data for Tukey HSD
                all_data = np.concatenate(response_groups)
                all_groups = []
                for i, group in enumerate(response_groups):
                    all_groups.extend([group_labels[i]] * len(group))
                
                tukey_results = pairwise_tukeyhsd(all_data, all_groups, alpha=0.05)
                print(tukey_results)
                
                # Print significant comparisons
                print("\nSignificant pairwise comparisons (p < 0.05):")
                sig_found = False
                for i, row in enumerate(tukey_results._results_table.data[1:]):
                    if row[5]:  # reject column (True if significant)
                        print(f"  {row[0]} vs {row[1]}: p = {row[4]:.4f}")
                        sig_found = True
                if not sig_found:
                    print("  No significant pairwise differences found")
                    
        except Exception as e:
            print(f"Error in statistical analysis: {e}")

def analyze_between_phase_stats(stats_df):
    """
    Perform mixed-effects/two-way ANOVA for phase comparisons.
    Factor 1: Phase (2 levels)
    Factor 2: Response Type (4 levels)
    Dependent Variable: Z-scored ΔF/F
    """
    print("\n" + "="*80)
    print("BETWEEN-PHASE STATISTICAL ANALYSIS")
    print("="*80)
    
    # Define the phase pairings - EXACTLY matching plotting code
    pairings = [
        ('0_Early', '1_Late'),  # Early phases
        ('2_SucOmit','3_CueOmit'),  # Early phases  
        ('4_RevEarly', '5_RevLate')  # Reversal phases
    ]
    
    for phase1, phase2 in pairings:
        # Filter data for this pairing
        pair_data = stats_df[stats_df['phase'].isin([phase1, phase2])].copy()
        
        if len(pair_data) == 0:
            continue
            
        print(f"\nComparison: {phase1} vs {phase2}")
        print("-" * 50)
        
        # Check if we have data for both phases
        phases_present = pair_data['phase'].unique()
        if len(phases_present) < 2:
            print(f"Insufficient phases for analysis (only {phases_present})")
            continue
        
        # Print sample sizes by condition
        print("Sample sizes by condition:")
        crosstab = pd.crosstab(pair_data['phase'], pair_data['response_type'], margins=True)
        print(crosstab)
        
        # Check if we have sufficient data for analysis
        min_cells_per_condition = 3
        condition_counts = pair_data.groupby(['phase', 'response_type']).size()
        insufficient_conditions = condition_counts[condition_counts < min_cells_per_condition]
        
        if len(insufficient_conditions) > 0:
            print(f"\nWarning: Some conditions have < {min_cells_per_condition} cells:")
            for (phase, resp_type), count in insufficient_conditions.items():
                print(f"  {phase} - {resp_type}: n = {count}")
        
        # Perform two-way ANOVA using OLS
        try:
            # Fit the model
            model = ols('zscored_dff ~ C(phase) + C(response_type) + C(phase):C(response_type)', 
                       data=pair_data).fit()
            
            # Perform ANOVA
            anova_results = anova_lm(model, typ=2)
            
            print(f"\nTwo-way ANOVA Results:")
            print("=" * 60)
            print(f"{'Source':<25} {'df':<8} {'F':<12} {'p-value':<12} {'Significant'}")
            print("-" * 60)
            
            for source in anova_results.index:
                if source == 'Residual':
                    continue
                df_val = anova_results.loc[source, 'df']
                f_val = anova_results.loc[source, 'F']
                p_val = anova_results.loc[source, 'PR(>F)']
                sig = 'Yes' if p_val < 0.05 else 'No'
                
                # Clean up source names for display
                if 'C(phase)' in source and ':' not in source:
                    source_name = 'Phase (main effect)'
                elif 'C(response_type)' in source and ':' not in source:
                    source_name = 'Response Type (main effect)'
                elif ':' in source:
                    source_name = 'Phase × Response Type'
                else:
                    source_name = source
                
                print(f"{source_name:<25} {df_val:<8.0f} {f_val:<12.4f} {p_val:<12.4f} {sig}")
            
            # Print model summary statistics
            print(f"\nModel Summary:")
            print(f"R-squared: {model.rsquared:.4f}")
            print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
            print(f"Total observations: {int(model.nobs)}")
            
            # If there are significant effects, perform post-hoc analysis
            significant_effects = []
            for source in anova_results.index:
                if source != 'Residual' and anova_results.loc[source, 'PR(>F)'] < 0.05:
                    significant_effects.append(source)
            
            if significant_effects:
                print(f"\nPost-hoc Analysis (Tukey HSD):")
                print("=" * 40)
                
                # For significant main effects or interactions, perform post-hoc tests
                if any('C(phase)' in effect and ':' not in effect for effect in significant_effects):
                    print("\nPhase main effect post-hoc:")
                    phase_data = pair_data.groupby('phase')['zscored_dff'].apply(list)
                    tukey_phase = pairwise_tukeyhsd(pair_data['zscored_dff'], 
                                                   pair_data['phase'], alpha=0.05)
                    print(tukey_phase)
                
                if any('C(response_type)' in effect and ':' not in effect for effect in significant_effects):
                    print("\nResponse Type main effect post-hoc:")
                    tukey_resp = pairwise_tukeyhsd(pair_data['zscored_dff'], 
                                                  pair_data['response_type'], alpha=0.05)
                    print(tukey_resp)
                
                if any(':' in effect for effect in significant_effects):
                    print("\nInteraction effect post-hoc (all pairwise comparisons):")
                    # Create combined factor for interaction post-hoc
                    pair_data['phase_response'] = pair_data['phase'] + '_' + pair_data['response_type']
                    tukey_interaction = pairwise_tukeyhsd(pair_data['zscored_dff'], 
                                                        pair_data['phase_response'], alpha=0.05)
                    print(tukey_interaction)
                    
                    # Print significant interaction comparisons
                    print("\nSignificant interaction comparisons (p < 0.05):")
                    sig_found = False
                    for i, row in enumerate(tukey_interaction._results_table.data[1:]):
                        if row[5]:  # reject column (True if significant)
                            print(f"  {row[0]} vs {row[1]}: p = {row[4]:.4f}")
                            sig_found = True
                    if not sig_found:
                        print("  No significant pairwise differences found")
                        
        except Exception as e:
            print(f"Error in statistical analysis: {e}")
            import traceback
            traceback.print_exc()

def run_complete_statistical_analysis(df_master, aucfirstframe=22, auclastframe=68, 
                                    baselinefirstframe=0, baselinelastframe=15):
    """
    Run complete statistical analysis for both within-phase and between-phase comparisons.
    """
    print("Extracting Z-scored data for statistical analysis...")
    
    # Extract the data using the same methodology as plotting functions
    stats_df = extract_zscored_data_for_stats(df_master, aucfirstframe, auclastframe, 
                                            baselinefirstframe, baselinelastframe)
    
    print(f"Total data points extracted: {len(stats_df)}")
    print(f"Phases: {sorted(stats_df['phase'].unique())}")
    print(f"Response types: {sorted(stats_df['response_type'].unique())}")
    
    # Run within-phase analysis (one-way ANOVAs)
    analyze_within_phase_stats(stats_df)
    
    # Run between-phase analysis (two-way ANOVAs)  
    analyze_between_phase_stats(stats_df)
    
    return stats_df

# Example usage:
stats_data = run_complete_statistical_analysis(df_master, aucfirstframe=22, auclastframe=68, 
                                              baselinefirstframe=0, baselinelastframe=15)

# Creating Control Dataframe for Astrocyte Activity Analysis

The `create_control_dataframe()` function generates control data from periods without cue stimulation to establish a baseline for comparison with cue-triggered activity.

## How It Works

1. **Selection of Control Windows:** 
   - Identifies time windows (113 frames) that contain no cue events
   - Creates a buffer zone (window_size) around all cue events to prevent overlap
   - Finds valid control frames that can center a complete window

2. **Matching to Original Trials:** 
   - For each cell and each original trial, selects the closest valid control frame
   - Creates separate control windows for CS+ and CS- trials
   - Uses unique control windows for each trial to prevent data reuse

3. **Data Processing:** 
   - Applies the same baseline correction to control data as experimental data
   - Calculates delta lick rates using the same methodology
   - Preserves all cell metadata and spatial information

## Key Parameters

- **basedir:** Base directory containing all experimental data
- **days:** List of experimental days to process
- **eventofinterest:** Type of event being analyzed (typically 'cues')
- **window_size:** Total number of frames in each analysis window (pre_window_size + post_window_size)
- **baselinefirstframe/baselinelastframe:** Frame range used for baseline calculation
- **df_master:** Original dataframe to match control data against

## Output Structure

The resulting `df_master_control` maintains the same structure as `df_master` with these distinctions:

- Each row maintains a link to its original trial via `original_trial_id`
- Contains both `timestamp` (control) and `original_timestamp` (cue) values
- Sets `reward_status` and `expected_reward` to 'none'
- Behavioral flags (high_licking, hits, etc.) are set to 0
- Contains the same format of fluorescence and licking data

# THIS IS THE NEW WAY FOR ITI -- FOLLOWING TRIAL

# THIS VARIANT OF THE CONTROL DATA FRAME GENERATOR MAKES SURE THAT ITI COMES AFTER A KNOWN TRIAL TYPE
## This would allow ITI interpretation to be linked to the trial that just happened (which the animal understands)


Creating ITI-Anchored Control Dataframe (Labeled by Preceding Trial Type)

The create_control_dataframe() (ITI version) generates control windows from the inter-trial interval (ITI) that immediately follows each original trial. Each control window is labeled by the CS type of the trial that preceded it (CS+ or CS−), allowing direct comparison between cue-evoked activity and temporally matched ITI activity.

# How It Works
	1.	Identify Trials and Frames
	•	Loads fluorescence, event logs, and lick timestamps for each FOV.
	•	Builds frame timestamps (with dropped-frame correction or assumed timestamps) and optionally downsamples via frameaveraging.
	2.	Exclude Actual Trial Windows
	•	Converts each cue event (CS+ and CS−) to its frame index.
	•	Excludes only the original trial window [event − pre_window_size : event + post_window_size) so control windows never overlap cue-locked activity.
	3.	Pick ITI Control Centers
	•	For each original trial, computes the control center as
control_frame = orig_frame + window_size + iti_buffer_frames,
i.e., one full analysis window after the cue window plus a user-defined ITI buffer to avoid residual post-cue dynamics.
	•	Accepts the control only if the centered window lies entirely in non-excluded frames and within bounds.
	4.	Label by Preceding Trial
	•	The control row inherits the CS label of the trial that came immediately before it:
	•	CS+ trials generate ITI controls with cs_type = 'CS+' and fluor_plus_frame_* columns.
	•	CS− trials generate ITI controls with cs_type = 'CS-' and fluor_minus_frame_* columns.
	5.	Data Processing (Mirrors Experimental Trials)
	•	Normalizes signals (mean-divide; optional z-scoring).
	•	Extracts a 113-frame window centered on the control_frame (size = pre_window_size + post_window_size).
	•	Applies the same baseline correction using baselinefirstframe:baselinelastframe.
	•	Aligns licks to the window, computes delta lick rate using the same averagedframerate method.

# Key Parameters
	•	basedir, days, df_master: Data sources and the original trial table to match against.
	•	pre_window_size / post_window_size: Define the 113-frame analysis window length (window_size).
	•	baselinefirstframe / baselinelastframe: Frames used to compute baseline for fluorescence and licks.
	•	averagedframerate: Converts lick counts to rates; used in delta lick calculation.
	•	iti_buffer_frames: Extra frames added after the original trial window before centering the ITI control (prevents residual post-cue activity from biasing controls).
	•	separation_requirement, eventofinterest: Passed through for consistency with your pipeline.

# Output Structure

# The resulting df_master_control matches the organization of df_master with ITI-specific annotations:
	•	Linkage to Source Trial: original_trial_id and original_timestamp refer to the preceding cue trial; trial_id adds a _control suffix; timestamp is the ITI center time.
	•	CS Labeling by Preceding Trial: cs_type reflects the prior trial (CS+ → plus columns; CS− → minus columns). The function ensures the presence of both fluor_plus_frame_* and fluor_minus_frame_* columns (unused side filled with NaN as needed).
	•	Reward / Behavior Flags: reward_status = 'none', expected_reward = 'none'; behavioral flags (high_licking, hits, misses, correct_rejections, false_alarms) are set to 0 for ITI controls.
	•	Fluorescence & Licks: Baseline-corrected fluor_*_frame_0...window_size-1 and delta_licks_frame_* are aligned exactly as in cue-locked trials.
	•	Spatial & Metadata: ROI geometry, tracking, sex, AUC/p-values, and other metadata are carried over when present.

# Notes & Safeguards
	•	Validity Checks: Controls are skipped if the ITI window would overlap another cue window or fall out of bounds; summary stats report attempted, skipped, and successful controls.
	•	Temporal Matching: Centering at orig_frame + window_size + iti_buffer_frames ensures consistent spacing from the preceding trial and minimizes contamination from post-cue dynamics.
	•	Comparability: Because preprocessing, baseline frames, and lick-rate computations mirror the experimental trials, ITI controls are directly comparable for decoding, auROC, or model training.

# The previous ITI code set reward status in control data frame to none always (which is true they arent rewarded)
### However if you wanted to look at ITIs after mistakes or rewarded vs not we would need to have information from df_master ie the 'original_trial_id'
### This variant below has a line in the usage section that will allow connection of reward_status and expected_reward from the original trial to the subsequent ITI

In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.io as sio
from IPython.display import display

# Assuming iterate_dirs, get_roi_numbers, fix_any_dropped_frames, 
# and other helper variables (like pre_window_size, etc.) are defined elsewhere.

def create_control_dataframe(basedir, days, eventofinterest, separation_requirement, baselinefirstframe, baselinelastframe, averagedframerate, df_master, iti_buffer_frames, copy_reward_status=False):
    """
    Create a control dataframe with 113-frame windows from the ITI.
    This version uses a session-wide baseline, calculated from all valid
    control windows, to match the baseline method used for df_master.
    
    New parameter:
    copy_reward_status (bool): If True, copies 'reward_status' and 'expected_reward' 
                               from the original trial. If False, sets them to 'none'.
    """
    control_data = []
    
    # Track statistics
    total_trials = 0
    skipped_trials = 0
    successful_trials = 0
    
    for basedir, day, animal, fov in iterate_dirs(basedir, sorted(days)):
        print(f"Processing control data for {day}, {animal}, {fov}")
        
        # Get ROI numbers for this FOV
        roi_numbers = get_roi_numbers(basedir, day, animal, fov)
        
        # Load signals and behavior data
        npyfiles = [f for f in os.listdir(os.path.join(basedir, day, animal, fov)) 
                   if f.endswith('.npy') and 'extractedsignals_raw' in f]
        matfiles = [f for f in os.listdir(os.path.join(basedir, day, animal, fov)) 
                   if f.endswith('.mat') and not 'results' in f]
        
        if not npyfiles or not matfiles:
            print(f"Missing required files for {day}, {animal}, {fov}. Skipping...")
            continue
            
        npyfile = npyfiles[0]
        matfile = matfiles[0]
        
        signals = np.squeeze(np.load(os.path.join(basedir, day, animal, fov, npyfile)))
        behaviordata = sio.loadmat(os.path.join(basedir, day, animal, fov, matfile))
        eventlog = np.squeeze(behaviordata['eventlog'])
        licks = np.squeeze(behaviordata['licks'])
        
        # Extract event timestamps
        cueplus = eventlog[eventlog[:,0]==5,1]  # CS+
        cueminus = eventlog[eventlog[:,0]==6,1]  # CS-
        all_cue_events = np.sort(np.concatenate([cueplus, cueminus]))
        
        # Frame timestamps
        if animal in ['CTL1', 'ER-L1', 'ER-L2', 'IG-19', 'IG-28', 'PGa-T1', 'XYZ']:
            frame_timestamps = np.array(assumed_frame_timestamps)
        else:
            frame_timestamps = fix_any_dropped_frames(eventlog[eventlog[:,0]==9,1])
        
        frame_timestamps = frame_timestamps[::frameaveraging]
        
        # Adjust signals if needed
        if signals.shape[1] > frame_timestamps.shape[0]:
            signals = signals[:,:frame_timestamps.shape[0]-1]
        
        # Preprocess signals
        signals /= np.nanmean(signals, axis=1)[:, None]
        
        if z_score_data == 'yes':
            for neuron in range(signals.shape[0]):
                mean = np.nanmean(signals[neuron])
                std = np.nanstd(signals[neuron])
                signals[neuron] = (signals[neuron] - mean)/std
        
        signalsT = signals.T
        
        # Function to convert licks to frames
        def licks_to_frames(licks, frame_timestamps):
            lick_frames = np.searchsorted(frame_timestamps, licks, side='right') - 1
            return lick_frames
            
        lick_frames = licks_to_frames(licks, frame_timestamps)
        
        # Extract control windows
        window_size = pre_window_size + post_window_size
        
        # Convert cue events to frame numbers
        cue_event_frames = []
        for event in all_cue_events:
            frame_idx = np.searchsorted(frame_timestamps, event, side='right') - 1
            cue_event_frames.append(frame_idx)
        
        # Set buffer zone around cue events
        excluded_frames = set()
        for frame in cue_event_frames:
            start_exclude = max(0, frame - pre_window_size)
            end_exclude = min(len(frame_timestamps), frame + post_window_size)
            excluded_frames.update(range(start_exclude, end_exclude))

        # Find valid control frame start points
        valid_frame_starts = []
        for i in range(pre_window_size, len(frame_timestamps) - post_window_size):
            if not any(i - pre_window_size + j in excluded_frames for j in range(window_size)):
                valid_frame_starts.append(i)
        
        valid_frame_starts_set = set(valid_frame_starts)
        
        if len(valid_frame_starts) == 0:
            print(f"No valid control frames found for {day}, {animal}, {fov}. Skipping...")
            continue
            
        print(f"Found {len(valid_frame_starts)} valid control frames in {len(frame_timestamps)} total frames")
        
        # Get phase config for reward status
        phase_config = folder_config[day]
        
        # --- NEW LOGIC: PASS 1 ---
        # First, pre-calculate the session-wide baseline for all valid control windows
        
        print(f"Calculating session-wide baselines for control data...")
        session_baselines_plus = {}
        session_baselines_minus = {}
            
        for cell_idx in range(signals.shape[0]):
            cell_trials = df_master[(df_master['day'] == day) & 
                                    (df_master['animal'] == animal) & 
                                    (df_master['fov'] == fov) & 
                                    (df_master['cell_number'] == cell_idx + 1)]
            
            csplus_trials = cell_trials[cell_trials['cs_type'] == 'CS+']
            csminus_trials = cell_trials[cell_trials['cs_type'] == 'CS-']
            
            # Aggregate all baseline frames for this cell's CS+ control windows
            all_control_plus_baselines = []
            for idx, trial in csplus_trials.iterrows():
                orig_frame = np.searchsorted(frame_timestamps, trial['timestamp'], side='right') - 1
                control_frame = orig_frame + window_size + iti_buffer_frames
                
                if (control_frame not in valid_frame_starts_set) or \
                   (control_frame < pre_window_size) or \
                   (control_frame >= len(frame_timestamps) - post_window_size):
                    continue # This trial will be skipped, so don't include it in baseline
                
                baseline_start = control_frame - pre_window_size + baselinefirstframe
                baseline_end = control_frame - pre_window_size + baselinelastframe
                baseline_window = signalsT[baseline_start:baseline_end, cell_idx]
                all_control_plus_baselines.append(baseline_window) # Append all frames
            
            # Store the session-wide average for this cell's CS+ control trials
            if all_control_plus_baselines:
                # Concatenate all baseline windows and then take the mean
                session_baselines_plus[cell_idx] = np.mean(np.concatenate(all_control_plus_baselines))
            else:
                session_baselines_plus[cell_idx] = np.nan # Handle cases with no valid trials
            
            # Aggregate all baseline frames for this cell's CS- control windows
            all_control_minus_baselines = []
            for idx, trial in csminus_trials.iterrows():
                orig_frame = np.searchsorted(frame_timestamps, trial['timestamp'], side='right') - 1
                control_frame = orig_frame + window_size + iti_buffer_frames
                
                if (control_frame not in valid_frame_starts_set) or \
                   (control_frame < pre_window_size) or \
                   (control_frame >= len(frame_timestamps) - post_window_size):
                    continue # Skip
                
                baseline_start = control_frame - pre_window_size + baselinefirstframe
                baseline_end = control_frame - pre_window_size + baselinelastframe
                baseline_window = signalsT[baseline_start:baseline_end, cell_idx]
                all_control_minus_baselines.append(baseline_window) # Append all frames

            if all_control_minus_baselines:
                # Concatenate all baseline windows and then take the mean
                session_baselines_minus[cell_idx] = np.mean(np.concatenate(all_control_minus_baselines))
            else:
                session_baselines_minus[cell_idx] = np.nan
        
        # --- NEW LOGIC: PASS 2 ---
        # Now, loop through again and build the dataframe, applying the session-wide baselines
        
        print(f"Building control dataframe with session-wide baselines...")
        for cell_idx in range(signals.shape[0]):
            cell_base = {
                'cell_number': cell_idx + 1,
                'roi_number': roi_numbers.get(cell_idx, np.nan),
                'day': day,
                'animal': animal,
                'fov': fov,
                'experimental_phase': day
            }
            
            cell_trials = df_master[(df_master['day'] == day) & 
                                    (df_master['animal'] == animal) & 
                                    (df_master['fov'] == fov) & 
                                    (df_master['cell_number'] == cell_idx + 1)]
            
            csplus_trials = cell_trials[cell_trials['cs_type'] == 'CS+']
            csminus_trials = cell_trials[cell_trials['cs_type'] == 'CS-']
            
            # Get the pre-calculated session-wide baselines for this cell
            global_control_baseline_plus = session_baselines_plus.get(cell_idx, np.nan)
            global_control_baseline_minus = session_baselines_minus.get(cell_idx, np.nan)
            
            # Process CS+ trials first
            for idx, trial in csplus_trials.iterrows():
                total_trials += 1
                orig_timestamp = trial['timestamp']
                orig_frame = np.searchsorted(frame_timestamps, orig_timestamp, side='right') - 1
                control_frame = orig_frame + window_size + iti_buffer_frames
                
                if (control_frame not in valid_frame_starts_set) or \
                   (control_frame < pre_window_size) or \
                   (control_frame >= len(frame_timestamps) - post_window_size):
                    skipped_trials += 1
                    continue
                
                control_timestamp = frame_timestamps[control_frame]
                
                # Extract control window data
                control_window = signalsT[control_frame - pre_window_size:control_frame + post_window_size, cell_idx]
                
                # --- BASELINE LOGIC CORRECTED HERE ---
                # Subtract the pre-calculated SESSION-WIDE baseline
                control_window_corrected = control_window - global_control_baseline_plus
                # --- END OF CORRECTION ---
                
                # Extract lick data for this control window
                window_start_frame = control_frame - pre_window_size
                window_end_frame = control_frame + post_window_size
                trial_licks = ((lick_frames >= window_start_frame) & 
                              (lick_frames < window_end_frame))
                aligned_licks = np.histogram(
                    lick_frames[trial_licks] - window_start_frame, 
                    bins=window_size, 
                    range=(0, window_size)
                )[0]
                
                # Calculate delta lick rate (this is still a per-trial calc, which is fine for licks)
                baseline_licks = np.mean(aligned_licks[baselinefirstframe:baselinelastframe])
                delta_licks = aligned_licks / (1 / averagedframerate) - baseline_licks
                
                # Create control trial data
                control_data_point = cell_base.copy()
                control_data_point.update({
                    'cell_id': trial['cell_id'],
                    'trial_id': f"{trial['trial_id']}_control",
                    'original_trial_id': trial['trial_id'],
                    'cs_type': 'CS+',
                    'timestamp': control_timestamp,
                    'original_timestamp': orig_timestamp,
                    
                    # *** UPDATED LOGIC HERE ***
                    'reward_status': trial['reward_status'] if copy_reward_status else 'none',
                    'expected_reward': trial['expected_reward'] if copy_reward_status else 'none',
                    
                    'trial_order': trial['trial_order'],
                    'trial': trial['trial']
                })
                
                # Add fluorescence data for CS+
                control_data_point.update({
                    f'fluor_plus_frame_{frame}': control_window_corrected[frame] 
                    for frame in range(window_size)
                })
                
                # Add delta licks
                control_data_point.update({
                    f'delta_licks_frame_{frame}': delta_licks[frame]
                    for frame in range(window_size)
                })
                
                # Add spatial information and other metadata
                for col in ['roi_center_x', 'roi_center_y', 'roi_left', 'roi_right', 
                           'roi_top', 'roi_bottom', 'roi_width', 'roi_height',
                           'tracked_roi_number', 'tracked_roi_counter', 'sex',
                           'pval_plus', 'pval_minus', 'auc_plus', 'auc_minus']:
                    if col in trial:
                        control_data_point[col] = trial[col]
                
                # Add behavioral columns if they exist
                for col in ['high_licking', 'hits', 'misses', 'correct_rejections', 'false_alarms']:
                    if col in trial:
                        control_data_point[col] = 0
                
                control_data.append(control_data_point)
                successful_trials += 1
            
            # Now process CS- trials
            for idx, trial in csminus_trials.iterrows():
                total_trials += 1
                orig_timestamp = trial['timestamp']
                orig_frame = np.searchsorted(frame_timestamps, orig_timestamp, side='right') - 1
                control_frame = orig_frame + window_size + iti_buffer_frames
                
                if (control_frame not in valid_frame_starts_set) or \
                   (control_frame < pre_window_size) or \
                   (control_frame >= len(frame_timestamps) - post_window_size):
                    skipped_trials += 1
                    continue
                
                control_timestamp = frame_timestamps[control_frame]
                
                # Extract control window data
                control_window = signalsT[control_frame - pre_window_size:control_frame + post_window_size, cell_idx]
                
                # --- BASELINE LOGIC CORRECTED HERE ---
                # Subtract the pre-calculated SESSION-WIDE baseline
                control_window_corrected = control_window - global_control_baseline_minus
                # --- END OF CORRECTION ---
                
                # Extract lick data for this control window
                window_start_frame = control_frame - pre_window_size
                window_end_frame = control_frame + post_window_size
                trial_licks = ((lick_frames >= window_start_frame) & 
                              (lick_frames < window_end_frame))
                aligned_licks = np.histogram(
                    lick_frames[trial_licks] - window_start_frame, 
                    bins=window_size, 
                    range=(0, window_size)
                )[0]
                
                # Calculate delta lick rate
                baseline_licks = np.mean(aligned_licks[baselinefirstframe:baselinelastframe])
                delta_licks = aligned_licks / (1 / averagedframerate) - baseline_licks
                
                # Create control trial data
                control_data_point = cell_base.copy()
                control_data_point.update({
                    'cell_id': trial['cell_id'],
                    'trial_id': f"{trial['trial_id']}_control",
                    'original_trial_id': trial['trial_id'],
                    'cs_type': 'CS-',
                    'timestamp': control_timestamp,
                    'original_timestamp': orig_timestamp,
                    
                    # *** UPDATED LOGIC HERE ***
                    'reward_status': trial['reward_status'] if copy_reward_status else 'none',
                    'expected_reward': trial['expected_reward'] if copy_reward_status else 'none',
                    
                    'trial_order': trial['trial_order'],
                    'trial': trial['trial']
                })
                
                # Add fluorescence data for CS-
                control_data_point.update({
                    f'fluor_minus_frame_{frame}': control_window_corrected[frame] 
                    for frame in range(window_size)
                })
                
                # Add delta licks
                control_data_point.update({
                    f'delta_licks_frame_{frame}': delta_licks[frame]
                    for frame in range(window_size)
                })
                
                # Add spatial information and other metadata
                for col in ['roi_center_x', 'roi_center_y', 'roi_left', 'roi_right', 
                           'roi_top', 'roi_bottom', 'roi_width', 'roi_height',
                           'tracked_roi_number', 'tracked_roi_counter', 'sex',
                           'pval_plus', 'pval_minus', 'auc_plus', 'auc_minus']:
                    if col in trial:
                        control_data_point[col] = trial[col]
                
                # Add behavioral columns if they exist
                for col in ['high_licking', 'hits', 'misses', 'correct_rejections', 'false_alarms']:
                    if col in trial:
                        control_data_point[col] = 0
                
                control_data.append(control_data_point)
                successful_trials += 1
    
    # Print statistics
    if total_trials > 0:
        print(f"\nTotal trials attempted: {total_trials}")
        print(f"Skipped trials (no valid ITI): {skipped_trials} ({skipped_trials/total_trials*100:.1f}%)")
        print(f"Successful trials: {successful_trials} ({successful_trials/total_trials*100:.1f}%)")
    else:
        print("\nNo trials processed.")
    
    # Create control DataFrame
    df_master_control = pd.DataFrame(control_data)
    
    # Fill NaNs for fluor columns to ensure all columns exist
    plus_cols = [f'fluor_plus_frame_{f}' for f in range(window_size)]
    minus_cols = [f'fluor_minus_frame_{f}' for f in range(window_size)]
    
    for col in plus_cols:
        if col not in df_master_control.columns:
            df_master_control[col] = np.nan
    for col in minus_cols:
        if col not in df_master_control.columns:
            df_master_control[col] = np.nan
            
    # Fill NaNs in the correct columns
    df_master_control[plus_cols] = df_master_control[plus_cols].fillna(np.nan)
    df_master_control[minus_cols] = df_master_control[minus_cols].fillna(np.nan)
    
    print("Control DataFrame Shape:", df_master_control.shape)
    print("\nSample of Control DataFrame:")
    display(df_master_control.head())
    
    return df_master_control

# ###--- NEW USAGE BLOCK ---###

# ### SET YOUR ITI BUFFER (in frames) ###
iti_buffer_frames = 25  # <-- Set this to your desired buffer (e.g., 0, 10, or 25)

# ### SET REWARD STATUS BEHAVIOR ###
copy_original_reward_status = True  # <-- Set to True to copy status, False to use 'none'

# Create the control dataframe by matching to the original df_master
df_master_control = create_control_dataframe(
    basedir=basedir,
    days=days,
    eventofinterest=eventofinterest,
    separation_requirement=separation_requirement,
    baselinefirstframe=baselinefirstframe,
    baselinelastframe=baselinelastframe,
    averagedframerate=averagedframerate,
    df_master=df_master,
    iti_buffer_frames=iti_buffer_frames,
    copy_reward_status=copy_original_reward_status  
)

# Fixing the control auc and pval calculations

In [ ]:
# Calculate AUC scores for the control dataframe
def calculate_control_auc_pvals(df_master_control, baselinefirstframe, baselinelastframe, aucfirstframe, auclastframe):
    """
    Calculate AUC scores and p-values for the control data frame using the same method as the original data.
    
    Parameters:
    -----------
    df_master_control : pandas DataFrame
        Control dataframe with fluorescence data
    baselinefirstframe : int
        First frame of baseline period
    baselinelastframe : int
        Last frame of baseline period
    aucfirstframe : int
        First frame for AUC calculation
    auclastframe : int
        Last frame for AUC calculation
        
    Returns:
    --------
    df_master_control : pandas DataFrame
        Updated control dataframe with recalculated AUC and p-values
    """
    print("Calculating AUC scores for control data...")
    
    # Process by cell_id to ensure we get all unique cells
    cell_ids = df_master_control['cell_id'].unique()
    
    # Track progress
    total_cells = len(cell_ids)
    print(f"Total unique cell_ids to process: {total_cells}")
    cells_processed = 0
    
    for cell_id in cell_ids:
        cells_processed += 1
        if cells_processed % 100 == 0 or cells_processed == 1 or cells_processed == total_cells:
            print(f"Processing cell {cells_processed}/{total_cells}")
        
        # Get all trials for this cell_id
        cell_df = df_master_control[df_master_control['cell_id'] == cell_id]
        
        # Separate CS+ and CS- trials for this cell
        cs_plus_trials = cell_df[cell_df['cs_type'] == 'CS+']
        cs_minus_trials = cell_df[cell_df['cs_type'] == 'CS-']
        
        # Skip cells with no trials for either CS type
        if len(cs_plus_trials) == 0 or len(cs_minus_trials) == 0:
            print(f"Warning: Cell ID {cell_id} is missing trials for one or both CS types. Skipping.")
            continue
            
        # Extract fluorescence data for CS+ trials
        plus_columns = [col for col in cs_plus_trials.columns if col.startswith('fluor_plus_frame_')]
        if not plus_columns:
            print(f"Warning: No fluor_plus_frame columns for cell ID {cell_id}. Skipping.")
            continue
            
        # Get all fluorescence data for this cell's CS+ trials
        plus_data = cs_plus_trials[plus_columns].values
        
        # Calculate baseline and event periods for CS+
        plus_baseline = np.mean(plus_data[:, baselinefirstframe:baselinelastframe], axis=1)
        plus_event = np.mean(plus_data[:, aucfirstframe:auclastframe], axis=1)
        
        # Calculate AUC score and p-value for CS+ using the same function from original code
        auc_plus, pval_plus = calculate_auROC(plus_event, plus_baseline)
        
        # Extract fluorescence data for CS- trials
        minus_columns = [col for col in cs_minus_trials.columns if col.startswith('fluor_minus_frame_')]
        if not minus_columns:
            print(f"Warning: No fluor_minus_frame columns for cell ID {cell_id}. Skipping.")
            continue
            
        # Get all fluorescence data for this cell's CS- trials
        minus_data = cs_minus_trials[minus_columns].values
        
        # Calculate baseline and event periods for CS-
        minus_baseline = np.mean(minus_data[:, baselinefirstframe:baselinelastframe], axis=1)
        minus_event = np.mean(minus_data[:, aucfirstframe:auclastframe], axis=1)
        
        # Calculate AUC score and p-value for CS- using the same function from original code
        auc_minus, pval_minus = calculate_auROC(minus_event, minus_baseline)
        
        # Update the dataframe with new AUC and p-values for this cell
        cell_mask = (df_master_control['cell_id'] == cell_id)
        
        df_master_control.loc[cell_mask, 'auc_plus'] = auc_plus
        df_master_control.loc[cell_mask, 'pval_plus'] = pval_plus
        df_master_control.loc[cell_mask, 'auc_minus'] = auc_minus
        df_master_control.loc[cell_mask, 'pval_minus'] = pval_minus
    
    # Apply Benjamini-Hochberg correction if specified in original parameters
    if bh_correction == 'yes':
        print("Applying Benjamini-Hochberg correction to p-values...")
        
        # Get all unique cells
        unique_cells = df_master_control.drop_duplicates(['animal', 'fov', 'cell_number'])
        
        # Extract all p-values
        all_pvals_plus = unique_cells['pval_plus'].values
        all_pvals_minus = unique_cells['pval_minus'].values
        
        # Apply BH correction
        corrected_pvals_plus = Benjamini_Hochberg_correction(all_pvals_plus, alpha=alphalevel)
        corrected_pvals_minus = Benjamini_Hochberg_correction(all_pvals_minus, alpha=alphalevel)
        
        # Create maps to update values
        plus_correction_map = dict(zip(
            unique_cells.set_index(['animal', 'fov', 'cell_number']).index,
            corrected_pvals_plus
        ))
        
        minus_correction_map = dict(zip(
            unique_cells.set_index(['animal', 'fov', 'cell_number']).index,
            corrected_pvals_minus
        ))
        
        # Apply corrections
        for (animal, fov, cell_number), corrected_pval_plus in plus_correction_map.items():
            cell_mask = ((df_master_control['animal'] == animal) & 
                         (df_master_control['fov'] == fov) & 
                         (df_master_control['cell_number'] == cell_number))
            df_master_control.loc[cell_mask, 'pval_plus'] = corrected_pval_plus
            
        for (animal, fov, cell_number), corrected_pval_minus in minus_correction_map.items():
            cell_mask = ((df_master_control['animal'] == animal) & 
                         (df_master_control['fov'] == fov) & 
                         (df_master_control['cell_number'] == cell_number))
            df_master_control.loc[cell_mask, 'pval_minus'] = corrected_pval_minus
    
    # Add flags for significant responders based on alpha level
    df_master_control['significant_plus'] = df_master_control['pval_plus'] < alphalevel
    df_master_control['significant_minus'] = df_master_control['pval_minus'] < alphalevel
    
    # Print summary statistics
    sig_plus_cells = df_master_control.drop_duplicates(['cell_id'])['significant_plus'].sum()
    sig_minus_cells = df_master_control.drop_duplicates(['cell_id'])['significant_minus'].sum()
    total_unique_cells = len(df_master_control['cell_id'].unique())
    
    print(f"\nControl data AUC calculation complete:")
    print(f"Total unique cells (by cell_id): {total_unique_cells}")
    print(f"CS+ significant responders: {sig_plus_cells} ({sig_plus_cells/total_unique_cells*100:.1f}%)")
    print(f"CS- significant responders: {sig_minus_cells} ({sig_minus_cells/total_unique_cells*100:.1f}%)")
    
    return df_master_control

# Usage:
df_master_control = calculate_control_auc_pvals(
    df_master_control=df_master_control,
    baselinefirstframe=baselinefirstframe,
    baselinelastframe=baselinelastframe,
    aucfirstframe=aucfirstframe,
    auclastframe=auclastframe
)

# Display a sample of the updated dataframe
print("\nSample of updated control dataframe with recalculated AUC scores:")
sample_cols = ['animal', 'fov', 'cell_number', 'cs_type', 
               'auc_plus', 'pval_plus', 'significant_plus',
               'auc_minus', 'pval_minus', 'significant_minus']
display(df_master_control[sample_cols].head(10))

# Optionally save the updated dataframe
save_path = os.path.join(basedir, 'Results', f'master_control_data_auc_{eventofinterest}_{separation_requirement}.csv')
df_master_control.to_csv(save_path, index=False)
print(f"\nUpdated control DataFrame saved to {save_path}")

# Comparison with original data
# Compare the proportion of significant responders in control vs. original data
df_master['significant_plus'] = df_master['pval_plus'] < alphalevel
df_master['significant_minus'] = df_master['pval_minus'] < alphalevel

orig_sig_plus = df_master.drop_duplicates(['cell_id'])['significant_plus'].mean() * 100
orig_sig_minus = df_master.drop_duplicates(['cell_id'])['significant_minus'].mean() * 100
control_sig_plus = df_master_control.drop_duplicates(['cell_id'])['significant_plus'].mean() * 100
control_sig_minus = df_master_control.drop_duplicates(['cell_id'])['significant_minus'].mean() * 100

print("\nComparison of significant responders:")
print(f"Original data: CS+ {orig_sig_plus:.1f}%, CS- {orig_sig_minus:.1f}%")
print(f"Control data: CS+ {control_sig_plus:.1f}%, CS- {control_sig_minus:.1f}%")

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def plot_control_pie_charts_by_sex(df_master_control, days, alphalevel, basedir, sex_selection='all'):
    """
    Plot pie charts showing the proportion of activated, inhibited, and non-responding cells
    in control data, grouped by day and sex (matches original plotting function structure).
    
    Parameters:
    -----------
    df_master_control : pandas DataFrame
        Control dataframe with AUC scores and p-values
    days : list
        List of days to analyze
    alphalevel : float
        Significance threshold (e.g., 0.05)
    basedir : str
        Base directory for saving plots
    sex_selection : str
        'all', 'male', 'female', or specific sex identifier
    """
    
    numdays = len(days)
    
    # Figure settings
    fig, axs = plt.subplots(2, numdays, figsize=(numdays*2, 4), sharex=True, sharey=False)
    if numdays == 1:
        axs = axs.reshape(-1, 1)  # Ensure 2D array for single day
    
    sns.set_style('white')
    colors = ['deeppink', 'gray', 'yellowgreen']  # Activated, Non-responders, Inhibited
    
    # Track total unique neurons across days
    total_neurons = 0
    
    for d, day in enumerate(sorted(days)):
        if sex_selection == 'all':
            # Get unique cells for this day
            day_data = df_master_control[df_master_control['day'] == day].drop_duplicates(subset=['cell_id'])
        else:
            day_data = df_master_control[
                (df_master_control['day'] == day) & 
                (df_master_control['sex'] == sex_selection)
            ].drop_duplicates(subset=['cell_id'])
        
        # Get unique cells for this group
        num_cells = len(day_data)
        total_neurons += num_cells
        
        print(f"\nDay {day} (Control Data):")
        print(f"Total unique cells: {num_cells}")
        
        # CS+ detailed counts
        activated_cells_plus = np.sum((day_data['pval_plus'] <= alphalevel) & (day_data['auc_plus'] >= 0))
        inhibited_cells_plus = np.sum((day_data['pval_plus'] <= alphalevel) & (day_data['auc_plus'] < 0))
        nonresponders_plus = np.sum(day_data['pval_plus'] > alphalevel)
        
        print(f"CS+ - Activated: {activated_cells_plus}, Inhibited: {inhibited_cells_plus}, Non-responders: {nonresponders_plus}")
        
        if num_cells > 0:
            activated_cells_plus_pct = activated_cells_plus / num_cells
            inhibited_cells_plus_pct = inhibited_cells_plus / num_cells
            nonresponders_plus_pct = nonresponders_plus / num_cells
            counts_plus = np.array([activated_cells_plus_pct, nonresponders_plus_pct, inhibited_cells_plus_pct])
            labels_plus = (f"{activated_cells_plus_pct*100:.1f}%", '', f"{inhibited_cells_plus_pct*100:.1f}%")
        else:
            counts_plus = np.array([0, 1, 0])
            labels_plus = ('no data', '', '')
        
        # CS- detailed counts
        activated_cells_minus = np.sum((day_data['pval_minus'] <= alphalevel) & (day_data['auc_minus'] >= 0))
        inhibited_cells_minus = np.sum((day_data['pval_minus'] <= alphalevel) & (day_data['auc_minus'] < 0))
        nonresponders_minus = np.sum(day_data['pval_minus'] > alphalevel)
        
        print(f"CS- - Activated: {activated_cells_minus}, Inhibited: {inhibited_cells_minus}, Non-responders: {nonresponders_minus}")
        
        if num_cells > 0:
            activated_cells_minus_pct = activated_cells_minus / num_cells
            inhibited_cells_minus_pct = inhibited_cells_minus / num_cells
            nonresponders_minus_pct = nonresponders_minus / num_cells
            counts_minus = np.array([activated_cells_minus_pct, nonresponders_minus_pct, inhibited_cells_minus_pct])
            labels_minus = (f"{activated_cells_minus_pct*100:.1f}%", '', f"{inhibited_cells_minus_pct*100:.1f}%")
        else:
            counts_minus = np.array([0, 1, 0])
            labels_minus = ('no data', '', '')
        
        # Plot pie charts
        ax = axs[0, d]
        ax.pie(counts_plus, colors=colors, labels=labels_plus)
        if d == 0:
            ax.set_ylabel('CS+', fontsize=10)
        ax.set_title(f'Day {day}')
        
        ax = axs[1, d]
        ax.pie(counts_minus, colors=colors, labels=labels_minus)
        if d == 0:
            ax.set_ylabel('CS-', fontsize=10)
    
    print(f"\nTotal number of unique neurons across all days (Control): {total_neurons}")
    
    title = f'Control Data Population Responses - {sex_selection.capitalize()}'
    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    
    # Save the plot
    save_path = os.path.join(basedir, 'Results', f'control_pie_charts_{sex_selection}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Control pie charts saved to: {save_path}")
    
    plt.show()

def plot_control_pie_charts_all_data(df_master_control, alphalevel, basedir, sex_selection='all'):
    """
    Plot pie charts for all control data together (not broken down by day).
    
    Parameters:
    -----------
    df_master_control : pandas DataFrame
        Control dataframe with AUC scores and p-values
    alphalevel : float
        Significance threshold (e.g., 0.05)
    basedir : str
        Base directory for saving plots
    sex_selection : str
        'all', 'male', 'female', or specific sex identifier
    """
    
    # Figure settings for single day
    fig, axs = plt.subplots(2, 1, figsize=(4, 6), sharex=True, sharey=False)
    
    sns.set_style('white')
    colors = ['deeppink', 'gray', 'yellowgreen']  # Activated, Non-responders, Inhibited
    
    # Get unique cells for the selected sex
    if sex_selection == 'all':
        day_data = df_master_control.drop_duplicates(subset=['cell_id'])
    else:
        day_data = df_master_control[
            df_master_control['sex'] == sex_selection
        ].drop_duplicates(subset=['cell_id'])
    
    num_cells = len(day_data)
    
    print(f"\nControl Data (All Days Combined):")
    print(f"Total unique cells: {num_cells}")
    
    # CS+ detailed counts
    activated_cells_plus = np.sum((day_data['pval_plus'] <= alphalevel) & (day_data['auc_plus'] >= 0))
    inhibited_cells_plus = np.sum((day_data['pval_plus'] <= alphalevel) & (day_data['auc_plus'] < 0))
    nonresponders_plus = np.sum(day_data['pval_plus'] > alphalevel)
    
    print(f"CS+ - Activated: {activated_cells_plus}, Inhibited: {inhibited_cells_plus}, Non-responders: {nonresponders_plus}")
    
    if num_cells > 0:
        activated_cells_plus_pct = activated_cells_plus / num_cells
        inhibited_cells_plus_pct = inhibited_cells_plus / num_cells
        nonresponders_plus_pct = nonresponders_plus / num_cells
        counts_plus = np.array([activated_cells_plus_pct, nonresponders_plus_pct, inhibited_cells_plus_pct])
        labels_plus = (f"{activated_cells_plus_pct*100:.1f}%", '', f"{inhibited_cells_plus_pct*100:.1f}%")
    else:
        counts_plus = np.array([0, 1, 0])
        labels_plus = ('no data', '', '')
    
    # CS- detailed counts
    activated_cells_minus = np.sum((day_data['pval_minus'] <= alphalevel) & (day_data['auc_minus'] >= 0))
    inhibited_cells_minus = np.sum((day_data['pval_minus'] <= alphalevel) & (day_data['auc_minus'] < 0))
    nonresponders_minus = np.sum(day_data['pval_minus'] > alphalevel)
    
    print(f"CS- - Activated: {activated_cells_minus}, Inhibited: {inhibited_cells_minus}, Non-responders: {nonresponders_minus}")
    
    if num_cells > 0:
        activated_cells_minus_pct = activated_cells_minus / num_cells
        inhibited_cells_minus_pct = inhibited_cells_minus / num_cells
        nonresponders_minus_pct = nonresponders_minus / num_cells
        counts_minus = np.array([activated_cells_minus_pct, nonresponders_minus_pct, inhibited_cells_minus_pct])
        labels_minus = (f"{activated_cells_minus_pct*100:.1f}%", '', f"{inhibited_cells_minus_pct*100:.1f}%")
    else:
        counts_minus = np.array([0, 1, 0])
        labels_minus = ('no data', '', '')
    
    # Plot pie charts
    axs[0].pie(counts_plus, colors=colors, labels=labels_plus)
    axs[0].set_ylabel('CS+', fontsize=10)
    axs[0].set_title('All Control Data')
    
    axs[1].pie(counts_minus, colors=colors, labels=labels_minus)
    axs[1].set_ylabel('CS-', fontsize=10)
    
    title = f'Control Data Population Responses (All Days) - {sex_selection.capitalize()}'
    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    
    # Save the plot
    save_path = os.path.join(basedir, 'Results', f'control_pie_charts_all_data_{sex_selection}_11-03-25.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Control pie charts (all data) saved to: {save_path}")
    
    plt.show()




# Usage examples:

# Plot control data pie charts broken down by day (exactly like your original function)
plot_control_pie_charts_by_sex(df_master_control, days, alphalevel, basedir, sex_selection='all')

# # Plot control data pie charts for all data combined (not broken down by day)
# plot_control_pie_charts_all_data(df_master_control, alphalevel, basedir, sex_selection='all')



In [ ]:
def visualize_control_data(df_master_control, days, pre_window_size, infusionframe, alphalevel,
                          baselinefirstframe, baselinelastframe, neurons_to_plot='all',
                          average_trials=True, sex='both', sort_by='mean', tracked='no',
                          z_score_data=True):
    """
    Visualize neural responses for control periods matched to CS+ and CS- trials.
    Parameters similar to visualize_master_data function.
    """
    # Sex filtering
    if sex.lower() != 'both':
        df_master_control = df_master_control[df_master_control['sex'].str.lower() == sex.lower()]
        print(f"Filtered for {sex} animals")
    else:
        print("Including both male and female animals")
    
    # Add tracked ROI filtering
    if tracked.lower() == 'yes':
        df_master_control = df_master_control[df_master_control['tracked_roi_counter'].notna()]
        print("Filtered for tracked ROIs only")
    
    print("\nAnimal counts by sex:")
    print(df_master_control.groupby('sex')['animal'].nunique())

    numdays = len(days)
    
    fig, axs = plt.subplots(numdays, 5, figsize=(20, numdays*4), sharex=True, sharey=False)
    if numdays == 1:
        axs = axs.reshape(1, -1)
    sns.set_style('white')
    
    # Set visualization limits based on whether we're z-scoring
    if z_score_data:
        cmax, cmin = 6, -6  # Z-scored data has wider range
        ymax, ymin = 7, -7
        print('Data Z-scored for visualization')
    else:
        cmax, cmin = 1, -1  # Raw data typically in smaller range
        ymax, ymin = 1.25, -0.25
    
    lickmin, lickmax = -1, 12
    
    for d, day in enumerate(sorted(days)):
        day_data = df_master_control[df_master_control['day'] == day]
        
        fluor_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        fluor_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        if average_trials:
            cell_data = day_data.groupby('cell_id').agg({
                'pval_plus': 'min',
                'pval_minus': 'min',
                'cs_type': 'first'
            }).reset_index()
        else:
            cell_data = day_data.drop_duplicates('cell_id')
        
        if neurons_to_plot == 'significant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] <= alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] <= alphalevel]['cell_id']
            print(f"\nDay {day}:")
            print(f"CS+ significant cells: {len(significant_csplus_cells)}")
            print(f"CS- significant cells: {len(significant_csminus_cells)}")
        elif neurons_to_plot == 'notsignificant':
            significant_csplus_cells = cell_data[cell_data['pval_plus'] > alphalevel]['cell_id']
            significant_csminus_cells = cell_data[cell_data['pval_minus'] > alphalevel]['cell_id']
        else:
            significant_csplus_cells = cell_data['cell_id']
            significant_csminus_cells = cell_data['cell_id']
        
        cs_plus_control_data = day_data[
            (day_data['cs_type'] == 'CS+') & 
            (day_data['cell_id'].isin(significant_csplus_cells))
        ]
        cs_minus_control_data = day_data[
            (day_data['cs_type'] == 'CS-') & 
            (day_data['cell_id'].isin(significant_csminus_cells))
        ]
        
        if average_trials:
            temp_plus = cs_plus_control_data.groupby('cell_id')[fluor_plus_cols].mean().values
            temp_minus = cs_minus_control_data.groupby('cell_id')[fluor_minus_cols].mean().values
            ylabel_suffix = "Cells"
        else:
            temp_plus = cs_plus_control_data[fluor_plus_cols].values
            temp_minus = cs_minus_control_data[fluor_minus_cols].values
            ylabel_suffix = "Data Points (Trials and Cells)"
        
        # Apply Z-scoring if requested
        if z_score_data:
            # Z-score CS+ control data
            for neuron in range(temp_plus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_plus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_plus[neuron, :] = temp_plus[neuron, :] / neuron_std
            
            # Z-score CS- control data
            for neuron in range(temp_minus.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    temp_minus[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    temp_minus[neuron, :] = temp_minus[neuron, :] / neuron_std
        
        if sort_by == 'mean':
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
        elif sort_by == 'timestamp' and not average_trials:
            timestamps_plus = cs_plus_control_data['timestamp']
            timestamps_minus = cs_minus_control_data['timestamp']
            sortresponse_plus = np.argsort(timestamps_plus.values)
            sortresponse_minus = np.argsort(timestamps_minus.values)
        else:
            sortresponse_plus = np.argsort(np.nanmean(temp_plus, axis=1))[::-1]
            sortresponse_minus = np.argsort(np.nanmean(temp_minus, axis=1))[::-1]
            if sort_by == 'timestamp':
                print("Warning: Timestamp sorting is only available when average_trials=False. Using mean sorting instead.")
        
        # Plot CS+ control heatmap
        ax = axs[d, 0]
        im = ax.imshow(temp_plus[sortresponse_plus], cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
        ax.set_ylabel(f'Control CS+ ({len(temp_plus)} {ylabel_suffix})')
        ax.set_yticks([])
        ax.set_xticks([])
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        fig.colorbar(im, ax=ax, orientation='vertical', 
                    label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        
        # Plot CS+ control mean trace
        ax = axs[d, 1]
        mean_temp_plus = np.nanmean(temp_plus, axis=0)
        sem_temp_plus = np.nanstd(temp_plus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_plus), axis=0))
        ax.plot(mean_temp_plus, color='m')
        ax.fill_between(range(len(mean_temp_plus)), 
                       mean_temp_plus - sem_temp_plus, 
                       mean_temp_plus + sem_temp_plus, 
                       color='pink', alpha=0.5)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks([])
        
        # Plot CS- control heatmap
        ax = axs[d, 2]
        im = ax.imshow(temp_minus[sortresponse_minus], cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
        ax.set_ylabel(f'Control CS- ({len(temp_minus)} {ylabel_suffix})')
        ax.set_yticks([])
        ax.set_xticks([])
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        fig.colorbar(im, ax=ax, orientation='vertical', 
                    label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        
        # Plot CS- control mean trace
        ax = axs[d, 3]
        mean_temp_minus = np.nanmean(temp_minus, axis=0)
        sem_temp_minus = np.nanstd(temp_minus, axis=0) / np.sqrt(np.sum(~np.isnan(temp_minus), axis=0))
        ax.plot(mean_temp_minus, color='grey')
        ax.fill_between(range(len(mean_temp_minus)), 
                       mean_temp_minus - sem_temp_minus, 
                       mean_temp_minus + sem_temp_minus, 
                       color='gainsboro', alpha=0.5)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks([])
        
        # Plot Delta Lick Rate
        ax = axs[d, 4]
        delta_licks_cols = [col for col in day_data.columns if col.startswith('delta_licks_frame_')]
        
        if average_trials:
            delta_licks_plus = cs_plus_control_data.groupby('cell_id')[delta_licks_cols].mean().values
            delta_licks_minus = cs_minus_control_data.groupby('cell_id')[delta_licks_cols].mean().values
        else:
            delta_licks_plus = cs_plus_control_data[delta_licks_cols].values
            delta_licks_minus = cs_minus_control_data[delta_licks_cols].values
        
        mean_delta_licks_plus = np.nanmean(delta_licks_plus, axis=0)
        sem_delta_licks_plus = np.nanstd(delta_licks_plus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_plus), axis=0))
        mean_delta_licks_minus = np.nanmean(delta_licks_minus, axis=0)
        sem_delta_licks_minus = np.nanstd(delta_licks_minus, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_minus), axis=0))
        
        ax.plot(mean_delta_licks_plus, color='m', label='Control CS+')
        ax.fill_between(range(len(mean_delta_licks_plus)), 
                       mean_delta_licks_plus - sem_delta_licks_plus, 
                       mean_delta_licks_plus + sem_delta_licks_plus, 
                       color='m', alpha=0.3)
        ax.plot(mean_delta_licks_minus, color='grey', label='Control CS-')
        ax.fill_between(range(len(mean_delta_licks_minus)), 
                       mean_delta_licks_minus - sem_delta_licks_minus, 
                       mean_delta_licks_minus + sem_delta_licks_minus, 
                       color='grey', alpha=0.3)
        
        ax.set_title('Control Delta Lick Rate')
        ax.set_ylabel('Δ Lick Rate (Hz)')
        ax.set_xlabel('Time (frames)')
        ax.set_ylim(lickmin, lickmax)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.legend()
    
    fig.text(0.2, 1, 'Control CS+', fontsize=16)
    fig.text(0.6, 1, 'Control CS-', fontsize=16)
    fig.text(0.8, 1, 'Control Δ Licks', fontsize=16)
    fig.tight_layout()
    #  ###Save the figure
    fig.savefig(os.path.join(basedir, 'Results', f'PopulationResponsesHeatmaps_CONTROL_dLickRate_z{z_score_data}'), 
                format='PNG')
    fig.savefig(os.path.join(basedir, 'Results', f'PopulationResponsesHeatmaps_CONTROL_dLickRate.pdf'), 
                format='PDF')
    plt.show()
    print('DONE')

# Example usage:
visualize_control_data(df_master_control, days, pre_window_size, infusionframe, alphalevel, 
                      baselinefirstframe, baselinelastframe, z_score_data=True)

# Heatmaps for strict tracking 
### At the bottom of this cell, select day for heatmap sorting. This day will have rows sorted by strongest response.
### Relative to the 'Sort Day', all other days are shown in that tracked order (Rows in the tracked day are in the same order for all other subsequent days)
### The readout in the console shows an example of the first 20 rows for each heatmap and reads out the tracked_roi_counter so one can confirm the fidelity of the tracking within this data visualization
### Every row is the exact same cell when shown across selected heatmaps 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

def verify_row_order(temp_plus_list, days_to_visualize):
    """
    Print tracked_roi_counter values and trial_ids for the first 20 rows of each day's data.
    """
    print("\nVerifying heatmap row order across days by tracked_roi_counter:")
    print("-" * 80)
    
    # Get the tracked_roi_counter values for each day
    for i, (day, df) in enumerate(zip(days_to_visualize, temp_plus_list)):
        if i == 0:
            print(f"\n{day} (Sort Day) - First 20 rows:")
        else:
            print(f"\n{day} - First 20 rows:")
            
        if isinstance(df, pd.DataFrame):
            if df.index.name == 'tracked_roi_counter':
                # For averaged trials
                for row in range(min(20, len(df.index))):
                    print(f"Row {row:2d}: tracked_roi_counter = {df.index[row]}")
            else:
                # For non-averaged trials
                if hasattr(df, 'tracked_roi_counter'):
                    # Display first 20 rows with both tracked_roi_counter and trial_id
                    for row in range(min(20, len(df))):
                        roi_counter = df['tracked_roi_counter'].iloc[row]
                        trial_id = df['trial_id'].iloc[row] if 'trial_id' in df.columns else 'N/A'
                        print(f"Row {row:2d}: tracked_roi_counter = {roi_counter:.0f}, trial_id = {trial_id}")

#### MAKE CRITICAL SELECTIONS HERE TO VISUALIZE THE DATA YOU WANT
### neurons_to_plot= 'significant', 'all', or 'notsignificant'
### average_trials= 'True' or 'False' (Do you want to see all the data or average the trials)
### verify= 'True' or 'False' (Do you want to check ROI #s from ImageJ against the numbers assigned when data is looped in)
### z_score_data= 'True' or 'False' (Do you want to z-score the data based on baseline period)

def visualize_master_data(df_master, days_to_visualize, sort_day, pre_window_size, infusionframe, 
                         alphalevel, baselinefirstframe, baselinelastframe, average_trials=True,
                         neurons_to_plot='significant', verify=True, z_score_data=False):
    """
    Visualize neural responses with consistent cell tracking across days.
    
    Parameters:
        df_master: DataFrame with neural response data
        days_to_visualize: List of days to include in visualization
        sort_day: Which day to use for ordering cells
        pre_window_size: Frame index of stimulus onset
        infusionframe: Frame index of reward delivery
        alphalevel: Significance threshold
        baselinefirstframe: First frame of baseline period for z-scoring
        baselinelastframe: Last frame of baseline period for z-scoring
        average_trials: Whether to average across trials
        neurons_to_plot: 'significant', 'notsignificant', or 'all'
        verify: Whether to verify row order
        z_score_data: Whether to z-score the fluorescence data based on baseline period
    """
    numdays = len(days_to_visualize)
    
    # Figure settings
    fig, axs = plt.subplots(numdays, 5, figsize=(20, numdays * 4), sharex=True, sharey=False)
    if numdays == 1:
        axs = axs.reshape(1, -1)
    sns.set_style('white')
    
    # Set visualization limits based on whether we're z-scoring
    if z_score_data:
        cmax, cmin = 6, -6  # Z-scored data has wider range
        ymax, ymin = 7, -7
        print('Data Z-scored for visualization')
    else:
        cmax, cmin = 1, -1  # Raw data typically in smaller range
        ymax, ymin = 1.25, -0.25
        
    lickmin, lickmax = -1, 12

    # Filter for cells present in all selected days
    cell_counts = df_master[df_master['day'].isin(days_to_visualize)].groupby('tracked_roi_counter')['day'].nunique()
    valid_cells = cell_counts[cell_counts == len(days_to_visualize)].index
    df_filtered = df_master[df_master['tracked_roi_counter'].isin(valid_cells)]

    # Get cell ordering based on sort_day CS+ responses
    sort_day_data = df_filtered[
        (df_filtered['day'] == sort_day) & 
        (df_filtered['cs_type'] == 'CS+')
    ]
    
    # Filter based on significance if needed
    if neurons_to_plot == 'significant':
        significant_cells = df_filtered.groupby('tracked_roi_counter').agg({
            'pval_plus': 'min',
            'pval_minus': 'min'
        }).reset_index()
        valid_cells = significant_cells[
            (significant_cells['pval_plus'] <= alphalevel) | 
            (significant_cells['pval_minus'] <= alphalevel)
        ]['tracked_roi_counter']
        df_filtered = df_filtered[df_filtered['tracked_roi_counter'].isin(valid_cells)]
        sort_day_data = sort_day_data[sort_day_data['tracked_roi_counter'].isin(valid_cells)]
    elif neurons_to_plot == 'notsignificant':
        nonsig_cells = df_filtered.groupby('tracked_roi_counter').agg({
            'pval_plus': 'min',
            'pval_minus': 'min'
        }).reset_index()
        valid_cells = nonsig_cells[
            (nonsig_cells['pval_plus'] > alphalevel) & 
            (nonsig_cells['pval_minus'] > alphalevel)
        ]['tracked_roi_counter']
        df_filtered = df_filtered[df_filtered['tracked_roi_counter'].isin(valid_cells)]
        sort_day_data = sort_day_data[sort_day_data['tracked_roi_counter'].isin(valid_cells)]

    # Calculate mean responses for sorting
    fluor_plus_cols = [col for col in sort_day_data.columns if col.startswith('fluor_plus_frame_')]
    if average_trials:
        mean_responses = sort_day_data.groupby('tracked_roi_counter')[fluor_plus_cols].mean()
    else:
        mean_responses = sort_day_data.set_index('tracked_roi_counter')[fluor_plus_cols]
    
    mean_response_vals = mean_responses.mean(axis=1)
    cell_order = mean_response_vals.sort_values(ascending=False).index

    # Store temp_plus for verification
    temp_plus_list = []

    # Plot each day
    for d, day in enumerate(days_to_visualize):
        day_data = df_filtered[df_filtered['day'] == day]
        fluor_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        fluor_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        # Separate CS+ and CS- data
        cs_plus_data = day_data[day_data['cs_type'] == 'CS+']
        cs_minus_data = day_data[day_data['cs_type'] == 'CS-']
        
        # Prepare data maintaining cell order
        if average_trials:
            temp_plus = cs_plus_data.groupby('tracked_roi_counter')[fluor_plus_cols].mean().loc[cell_order]
            temp_minus = cs_minus_data.groupby('tracked_roi_counter')[fluor_minus_cols].mean().loc[cell_order]
            ylabel = f'{len(cell_order)} Cells'
        else:
            # For all trials, maintain cell order but show all trials for each cell
            columns_to_include = ['tracked_roi_counter', 'trial_id'] + fluor_plus_cols
            temp_plus = pd.concat([
                cs_plus_data[cs_plus_data['tracked_roi_counter'] == roi][columns_to_include]
                for roi in cell_order
            ])
            
            columns_to_include = ['tracked_roi_counter', 'trial_id'] + fluor_minus_cols
            temp_minus = pd.concat([
                cs_minus_data[cs_minus_data['tracked_roi_counter'] == roi][columns_to_include]
                for roi in cell_order
            ])
            ylabel = f'{len(temp_plus)} Trials'
            
        if verify:
            temp_plus_list.append(temp_plus)
        
        # Prepare data for plotting
        if average_trials:
            plus_data = temp_plus.values
            minus_data = temp_minus.values
        else:
            plus_data = temp_plus[fluor_plus_cols].values
            minus_data = temp_minus[fluor_minus_cols].values
            
        # Apply Z-scoring if requested
        if z_score_data:
            # Z-score CS+ data
            for neuron in range(plus_data.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    plus_data[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    plus_data[neuron, :] = plus_data[neuron, :] / neuron_std
            
            # Z-score CS- data
            for neuron in range(minus_data.shape[0]):
                # Calculate standard deviation of baseline period
                neuron_std = np.nanstd(
                    minus_data[neuron, baselinefirstframe:baselinelastframe]
                )
                # Normalize by dividing by std dev
                if neuron_std > 0:  # Prevent division by zero
                    minus_data[neuron, :] = minus_data[neuron, :] / neuron_std
        
        # Plot CS+ Data
        ax = axs[d, 0]
        im = ax.imshow(plus_data, cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
        ax.set_ylabel(ylabel)
        ax.set_yticks([])
        ax.set_xticks([])
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        fig.colorbar(im, ax=ax, orientation='vertical', 
                   label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        
        # Plot CS+ mean trace
        ax = axs[d, 1]
        mean_temp_plus = np.nanmean(plus_data, axis=0)
        sem_temp_plus = np.nanstd(plus_data, axis=0) / np.sqrt(np.sum(~np.isnan(plus_data), axis=0))
        ax.plot(mean_temp_plus, color='m')
        ax.fill_between(range(len(mean_temp_plus)), 
                       mean_temp_plus - sem_temp_plus, 
                       mean_temp_plus + sem_temp_plus, 
                       color='pink', alpha=0.5)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks([])
        
        # Plot CS- Data
        ax = axs[d, 2]
        im = ax.imshow(minus_data, cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
        ax.set_ylabel(ylabel)
        ax.set_yticks([])
        ax.set_xticks([])
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        fig.colorbar(im, ax=ax, orientation='vertical', 
                   label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')
        
        # Plot CS- mean trace
        ax = axs[d, 3]
        mean_temp_minus = np.nanmean(minus_data, axis=0)
        sem_temp_minus = np.nanstd(minus_data, axis=0) / np.sqrt(np.sum(~np.isnan(minus_data), axis=0))
        ax.plot(mean_temp_minus, color='grey')
        ax.fill_between(range(len(mean_temp_minus)), 
                       mean_temp_minus - sem_temp_minus, 
                       mean_temp_minus + sem_temp_minus, 
                       color='gainsboro', alpha=0.5)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks([])
        
        # Plot Delta Lick Rate Data
        ax = axs[d, 4]
        delta_licks_cols = [col for col in day_data.columns if col.startswith('delta_licks_frame_')]
        
        if average_trials:
            delta_licks_plus = cs_plus_data.groupby('tracked_roi_counter')[delta_licks_cols].mean().loc[cell_order]
            delta_licks_minus = cs_minus_data.groupby('tracked_roi_counter')[delta_licks_cols].mean().loc[cell_order]
        else:
            delta_licks_plus = pd.concat([
                cs_plus_data[cs_plus_data['tracked_roi_counter'] == roi][delta_licks_cols]
                for roi in cell_order
            ])
            delta_licks_minus = pd.concat([
                cs_minus_data[cs_minus_data['tracked_roi_counter'] == roi][delta_licks_cols]
                for roi in cell_order
            ])
        
        mean_delta_licks_plus = np.nanmean(delta_licks_plus.values, axis=0)
        sem_delta_licks_plus = np.nanstd(delta_licks_plus.values, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_plus.values), axis=0))
        mean_delta_licks_minus = np.nanmean(delta_licks_minus.values, axis=0)
        sem_delta_licks_minus = np.nanstd(delta_licks_minus.values, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_minus.values), axis=0))
        
        ax.plot(mean_delta_licks_plus, color='m', label='CS+')
        ax.fill_between(range(len(mean_delta_licks_plus)), 
                       mean_delta_licks_plus - sem_delta_licks_plus, 
                       mean_delta_licks_plus + sem_delta_licks_plus, 
                       color='m', alpha=0.3)
        ax.plot(mean_delta_licks_minus, color='grey', label='CS-')
        ax.fill_between(range(len(mean_delta_licks_minus)), 
                       mean_delta_licks_minus - sem_delta_licks_minus, 
                       mean_delta_licks_minus + sem_delta_licks_minus, 
                       color='grey', alpha=0.3)
        
        ax.set_title('Delta Lick Rate')
        ax.set_ylabel('Δ Lick Rate (Hz)')
        ax.set_xlabel('Time from CS (s)')
        ax.set_ylim(lickmin, lickmax)
        ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
        ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
        ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
        ax.legend()
        
        # Add day label
        axs[d, 0].text(-0.1, 0.5, day, rotation=90, va='center', ha='right', transform=axs[d, 0].transAxes)
    
    fig.text(0.2, 1, 'CS+', fontsize=16)
    fig.text(0.6, 1, 'CS-', fontsize=16)
    fig.text(0.8, 1, 'Δ Licks', fontsize=16)
    fig.tight_layout()
    
    # Run verification if requested
    if verify:
        verify_row_order(temp_plus_list, days_to_visualize)

    # Add z-score indicator to filenames if z-scoring was applied
    z_score_label = "_zscored" if z_score_data else ""
    
    # Save figures with updated filenames
    # try:
    #     fig.savefig(os.path.join(basedir, 'Results', 
    #                            f'TrackedCells_HeatmapsLickTrace{z_score_label}_{"-".join(days_to_visualize)}_06-20-25.PNG'), 
    #               format='PNG')
    #     fig.savefig(os.path.join(basedir, 'Results', 
    #                            f'TrackedCells_SortedMean_HeatmapsLickTrace{z_score_label}_{"-".join(days_to_visualize)}_06-20-25.PDF'), 
    #               format='PDF')
    # except NameError:
    #     # If basedir is not defined, just display a message
    #     print("Note: Figure not saved because 'basedir' variable is not defined")
    
    plt.show()
    print('DONE')

#### MAKE CRITICAL SELECTIONS HERE TO VISUALIZE THE DATA YOU WANT
### sort_day= '0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate'
### days_to_visualize= Some likely comparisions here are ['0_Early', '1_Late'] 
# or ['1_Late','2_SucOmit','3_CueOmit'] 
# or ['1_Late','4_RevEarly','5_RevLate']
# z_score_data= True or False (whether to z-score data based on baseline period)

# Example usage:
days_to_visualize = ['0_Early', '1_Late']
sort_day = '0_Early'
visualize_master_data(df_master, days_to_visualize, sort_day, pre_window_size, infusionframe, 
                     alphalevel, baselinefirstframe, baselinelastframe, z_score_data=True)

days_to_visualize = ['1_Late', '2_SucOmit', '3_CueOmit']
sort_day = '1_Late'
visualize_master_data(df_master, days_to_visualize, sort_day, pre_window_size, infusionframe, 
                     alphalevel, baselinefirstframe, baselinelastframe, z_score_data=True)

days_to_visualize = ['1_Late', '4_RevEarly', '5_RevLate']
sort_day = '1_Late'
visualize_master_data(df_master, days_to_visualize, sort_day, pre_window_size, infusionframe, 
                     alphalevel, baselinefirstframe, baselinelastframe, z_score_data=True)

# Tracking responses to unpredictable reward (SucOmit)
## This is to address the question of during unpredictable reward, is it the same astrocytes that respond to CS+ when reward is delivered as those that respond to CS+ when reward isnt delivered
## For SucOmit, this code plots all CS+ trials, as well as rewarded ones and unrewarded ones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

def visualize_late_sucomit(df_master, pre_window_size, infusionframe, alphalevel, 
                          baselinefirstframe, baselinelastframe, average_trials=True,
                          neurons_to_plot='significant', verify=True, z_score_data=False):
    """
    Visualize neural responses comparing Late vs SucOmit, with SucOmit split into rewarded/unrewarded.
    
    Parameters:
        df_master: DataFrame with neural response data
        pre_window_size: Frame index of stimulus onset
        infusionframe: Frame index of reward delivery
        alphalevel: Significance threshold
        baselinefirstframe: First frame of baseline period for z-scoring
        baselinelastframe: Last frame of baseline period for z-scoring
        average_trials: Whether to average across trials
        neurons_to_plot: 'significant', 'notsignificant', or 'all'
        verify: Whether to verify row order
        z_score_data: Whether to z-score the fluorescence data based on baseline period
    """
    # Setup days for comparison
    days_to_visualize = ['1_Late', '2_SucOmit']
    sort_day = '1_Late'
    
    # Create three separate figures
    fig_labels = ['All Trials', 'Rewarded SucOmit', 'Unrewarded SucOmit']
    figs = []
    
    # Figure settings
    sns.set_style('white')
    
    # Set visualization limits based on whether we're z-scoring
    if z_score_data:
        cmax, cmin = 8, -8  # Z-scored data has wider range
        ymax, ymin = 8, -8
        print('Data Z-scored for visualization')
    else:
        cmax, cmin = 1, -1  # Raw data typically in smaller range
        ymax, ymin = 1.25, -0.25
        
    lickmin, lickmax = -1, 12

    # Filter for cells present in both days
    cell_counts = df_master[df_master['day'].isin(days_to_visualize)].groupby('tracked_roi_counter')['day'].nunique()
    valid_cells = cell_counts[cell_counts == len(days_to_visualize)].index
    df_filtered = df_master[df_master['tracked_roi_counter'].isin(valid_cells)]

    # Get cell ordering based on sort_day (Late) CS+ responses
    sort_day_data = df_filtered[
        (df_filtered['day'] == sort_day) & 
        (df_filtered['cs_type'] == 'CS+')
    ]
    
    # Filter based on significance if needed
    if neurons_to_plot == 'significant':
        significant_cells = df_filtered.groupby('tracked_roi_counter').agg({
            'pval_plus': 'min',
            'pval_minus': 'min'
        }).reset_index()
        valid_cells = significant_cells[
            (significant_cells['pval_plus'] <= alphalevel) | 
            (significant_cells['pval_minus'] <= alphalevel)
        ]['tracked_roi_counter']
        df_filtered = df_filtered[df_filtered['tracked_roi_counter'].isin(valid_cells)]
        sort_day_data = sort_day_data[sort_day_data['tracked_roi_counter'].isin(valid_cells)]
    elif neurons_to_plot == 'notsignificant':
        nonsig_cells = df_filtered.groupby('tracked_roi_counter').agg({
            'pval_plus': 'min',
            'pval_minus': 'min'
        }).reset_index()
        valid_cells = nonsig_cells[
            (nonsig_cells['pval_plus'] > alphalevel) & 
            (nonsig_cells['pval_minus'] > alphalevel)
        ]['tracked_roi_counter']
        df_filtered = df_filtered[df_filtered['tracked_roi_counter'].isin(valid_cells)]
        sort_day_data = sort_day_data[sort_day_data['tracked_roi_counter'].isin(valid_cells)]

    # Calculate mean responses for sorting
    fluor_plus_cols = [col for col in sort_day_data.columns if col.startswith('fluor_plus_frame_')]
    if average_trials:
        mean_responses = sort_day_data.groupby('tracked_roi_counter')[fluor_plus_cols].mean()
    else:
        mean_responses = sort_day_data.set_index('tracked_roi_counter')[fluor_plus_cols]
    
    mean_response_vals = mean_responses.mean(axis=1)
    cell_order = mean_response_vals.sort_values(ascending=False).index

    # Create three variants of the visualization
    for variant_idx, variant in enumerate(['all', 'rewarded', 'unrewarded']):
        df_variant = df_filtered.copy()
        
        # Filter SucOmit trials based on variant
        if variant != 'all':
            mask = (df_variant['day'] == '2_SucOmit') & (df_variant['cs_type'] == 'CS+')
            if variant == 'rewarded':
                df_variant = df_variant[~mask | (mask & (df_variant['reward_status'] == 'rewarded'))]
            else:
                df_variant = df_variant[~mask | (mask & (df_variant['reward_status'] == 'unrewarded'))]

        # Create figure for this variant
        fig, axs = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=False)
        figs.append(fig)
        
        # Plot each day
        for d, day in enumerate(days_to_visualize):
            day_data = df_variant[df_variant['day'] == day]
            
            # Get column names
            fluor_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
            fluor_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
            
            # Separate CS+ and CS- data
            cs_plus_data = day_data[day_data['cs_type'] == 'CS+']
            cs_minus_data = day_data[day_data['cs_type'] == 'CS-']
            
            # Prepare data maintaining cell order
            if average_trials:
                temp_plus = cs_plus_data.groupby('tracked_roi_counter')[fluor_plus_cols].mean().loc[cell_order]
                temp_minus = cs_minus_data.groupby('tracked_roi_counter')[fluor_minus_cols].mean().loc[cell_order]
                ylabel = f'{len(cell_order)} Cells'
            else:
                # For all trials, maintain cell order but show all trials for each cell
                temp_plus = pd.concat([
                    cs_plus_data[cs_plus_data['tracked_roi_counter'] == roi][fluor_plus_cols]
                    for roi in cell_order
                ])
                temp_minus = pd.concat([
                    cs_minus_data[cs_minus_data['tracked_roi_counter'] == roi][fluor_minus_cols]
                    for roi in cell_order
                ])
                ylabel = f'{len(temp_plus)} Trials'

            # Convert to numpy arrays for plotting
            if average_trials:
                plot_data_plus = temp_plus.values
                plot_data_minus = temp_minus.values
            else:
                plot_data_plus = temp_plus.values
                plot_data_minus = temp_minus.values
                
            # Apply Z-scoring if requested
            if z_score_data:
                # Z-score CS+ data
                for neuron in range(plot_data_plus.shape[0]):
                    # Calculate standard deviation of baseline period
                    neuron_std = np.nanstd(
                        plot_data_plus[neuron, baselinefirstframe:baselinelastframe]
                    )
                    # Normalize by dividing by std dev
                    if neuron_std > 0:  # Prevent division by zero
                        plot_data_plus[neuron, :] = plot_data_plus[neuron, :] / neuron_std
                
                # Z-score CS- data
                for neuron in range(plot_data_minus.shape[0]):
                    # Calculate standard deviation of baseline period
                    neuron_std = np.nanstd(
                        plot_data_minus[neuron, baselinefirstframe:baselinelastframe]
                    )
                    # Normalize by dividing by std dev
                    if neuron_std > 0:  # Prevent division by zero
                        plot_data_minus[neuron, :] = plot_data_minus[neuron, :] / neuron_std

            # Plot the data
            # CS+ Heatmap
            ax = axs[d, 0]
            im = ax.imshow(plot_data_plus, cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
            ax.set_ylabel(ylabel)
            ax.set_yticks([])
            ax.set_xticks([])
            ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
            ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
            fig.colorbar(im, ax=ax, orientation='vertical', 
                       label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')

            # CS+ Mean Trace
            ax = axs[d, 1]
            mean_temp_plus = np.nanmean(plot_data_plus, axis=0)
            sem_temp_plus = np.nanstd(plot_data_plus, axis=0) / np.sqrt(np.sum(~np.isnan(plot_data_plus), axis=0))
            ax.plot(mean_temp_plus, color='m')
            ax.fill_between(range(len(mean_temp_plus)), 
                          mean_temp_plus - sem_temp_plus, 
                          mean_temp_plus + sem_temp_plus, 
                          color='pink', alpha=0.5)
            ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
            ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
            ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
            ax.set_ylim(ymin, ymax)
            ax.set_xticks([])

            # CS- Heatmap
            ax = axs[d, 2]
            im = ax.imshow(plot_data_minus, cmap='PiYG_r', vmin=cmin, vmax=cmax, aspect='auto')
            ax.set_ylabel(ylabel)
            ax.set_yticks([])
            ax.set_xticks([])
            ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
            ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
            fig.colorbar(im, ax=ax, orientation='vertical', 
                       label='Z-scored ΔF/F' if z_score_data else 'ΔF/F')

            # CS- Mean Trace
            ax = axs[d, 3]
            mean_temp_minus = np.nanmean(plot_data_minus, axis=0)
            sem_temp_minus = np.nanstd(plot_data_minus, axis=0) / np.sqrt(np.sum(~np.isnan(plot_data_minus), axis=0))
            ax.plot(mean_temp_minus, color='grey')
            ax.fill_between(range(len(mean_temp_minus)), 
                          mean_temp_minus - sem_temp_minus, 
                          mean_temp_minus + sem_temp_minus, 
                          color='gainsboro', alpha=0.5)
            ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
            ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
            ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
            ax.set_ylim(ymin, ymax)
            ax.set_xticks([])

            # Lick Rate Plot
            ax = axs[d, 4]
            delta_licks_cols = [col for col in day_data.columns if col.startswith('delta_licks_frame_')]
            if average_trials:
                delta_licks_plus = cs_plus_data.groupby('tracked_roi_counter')[delta_licks_cols].mean().loc[cell_order]
                delta_licks_minus = cs_minus_data.groupby('tracked_roi_counter')[delta_licks_cols].mean().loc[cell_order]
            else:
                delta_licks_plus = pd.concat([
                    cs_plus_data[cs_plus_data['tracked_roi_counter'] == roi][delta_licks_cols]
                    for roi in cell_order
                ])
                delta_licks_minus = pd.concat([
                    cs_minus_data[cs_minus_data['tracked_roi_counter'] == roi][delta_licks_cols]
                    for roi in cell_order
                ])
            
            mean_delta_licks_plus = np.nanmean(delta_licks_plus.values, axis=0)
            sem_delta_licks_plus = np.nanstd(delta_licks_plus.values, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_plus.values), axis=0))
            mean_delta_licks_minus = np.nanmean(delta_licks_minus.values, axis=0)
            sem_delta_licks_minus = np.nanstd(delta_licks_minus.values, axis=0) / np.sqrt(np.sum(~np.isnan(delta_licks_minus.values), axis=0))

            ax.plot(mean_delta_licks_plus, color='m', label='CS+')
            ax.fill_between(range(len(mean_delta_licks_plus)), 
                          mean_delta_licks_plus - sem_delta_licks_plus, 
                          mean_delta_licks_plus + sem_delta_licks_plus, 
                          color='m', alpha=0.3)
            ax.plot(mean_delta_licks_minus, color='grey', label='CS-')
            ax.fill_between(range(len(mean_delta_licks_minus)), 
                          mean_delta_licks_minus - sem_delta_licks_minus, 
                          mean_delta_licks_minus + sem_delta_licks_minus, 
                          color='grey', alpha=0.3)

            ax.set_title('Delta Lick Rate')
            ax.set_ylabel('Δ Lick Rate (Hz)')
            ax.set_xlabel('Time from CS (s)')
            ax.set_ylim(lickmin, lickmax)
            ax.axvline(x=pre_window_size, color='k', linestyle='--', linewidth=1.5)
            ax.axvline(x=infusionframe, color='k', linestyle='--', linewidth=1.5)
            ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
            ax.legend()

            # Add day label
            axs[d, 0].text(-0.1, 0.5, day, rotation=90, va='center', ha='right', transform=axs[d, 0].transAxes)

        # Add titles and adjust layout
        z_score_label = "Z-scored " if z_score_data else ""
        fig.text(0.2, 1, 'CS+', fontsize=16)
        fig.text(0.6, 1, 'CS-', fontsize=16)
        fig.text(0.8, 1, 'Δ Licks', fontsize=16)
        fig.suptitle(f'Late vs SucOmit Comparison - {z_score_label}{fig_labels[variant_idx]}', fontsize=16, y=1.05)
        fig.tight_layout()
        
        # Save figure if basedir is defined
        # try:
        #     z_score_suffix = "_zscored" if z_score_data else ""
        #     fig.savefig(os.path.join(basedir, 'Results', 
        #                           f'LateSucOmit_{variant}{z_score_suffix}_trackedheatmap_06-20-25.PNG'), 
        #               format='PNG')
        #     fig.savefig(os.path.join(basedir, 'Results', 
        #                           f'LateSucOmit_{variant}{z_score_suffix}_trackedheatmap_06-20-25.PDF'), 
        #               format='PDF')
        # except NameError:
        #     # If basedir is not defined, just display a message
        #     print(f"Note: Figure for {variant} variant not saved because 'basedir' variable is not defined")

    # Show all figures
    plt.show()
    print('DONE')

# Example usage:
visualize_late_sucomit(df_master, pre_window_size, infusionframe, alphalevel, 
                     baselinefirstframe, baselinelastframe, average_trials=True,
                     neurons_to_plot='significant', z_score_data=True)

# Spatial Mapping

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, RegularPolygon, Rectangle, Circle
import roifile
from scipy.interpolate import splprep, splev

def smooth_coordinates(x_coords, y_coords, smoothing_factor, num_points=100):
    """
    Smooth ROI coordinates using B-spline interpolation.
    """
    if len(x_coords) < 4:
        return x_coords, y_coords
    
    if (x_coords[0], y_coords[0]) != (x_coords[-1], y_coords[-1]):
        x_coords = np.append(x_coords, x_coords[0])
        y_coords = np.append(y_coords, y_coords[0])
    
    tck, u = splprep([x_coords, y_coords], s=smoothing_factor, per=True)
    u_new = np.linspace(0, 1, num_points)
    x_new, y_new = splev(u_new, tck)
    
    return x_new, y_new

def create_star(x_center, y_center, size, num_points=5):
    """Generate coordinates for a star shape."""
    angle = np.pi / num_points
    star_points = []
    for i in range(2 * num_points):
        radius = size if i % 2 == 0 else size / 2
        x = x_center + radius * np.cos(i * angle)
        y = y_center + radius * np.sin(i * angle)
        star_points.append((x, y))
    return star_points

def create_cell_patch(shape_type, center_x, center_y, size, color, outline=True, alpha=0.7):
    """Create a matplotlib patch based on specified shape type."""
    edgecolor = 'black' if outline else 'none'
    
    if shape_type == 'hexagon':
        return RegularPolygon((center_x, center_y), numVertices=6, radius=size,
                            orientation=np.radians(30), facecolor=color,
                            edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'square':
        return Rectangle((center_x - size/2, center_y - size/2),
                        size, size, facecolor=color,
                        edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'circle':
        return Circle((center_x, center_y), radius=size/2,
                     facecolor=color, edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'star':
        star_points = create_star(center_x, center_y, size)
        return Polygon(star_points, closed=True, facecolor=color,
                      edgecolor=edgecolor, alpha=alpha)
    else:
        raise ValueError("Invalid shape_type. Choose from 'hexagon', 'square', 'circle', 'star'")

def visualize_cell_positions(basedir, df_master, column_name, plot_type='combined', 
                           roi_type='real', shape='hexagon', grid_size=30,
                           roi_smoothing=0, outline=True, blend_colors=True,
                           show_cell_numbers=False):
    """
    Unified function for visualizing cell positions with various options.
    
    Parameters:
    -----------
    basedir : str
        Base directory containing the data
    df_master : pandas DataFrame
        Master dataframe containing all cell data
    column_name : str
        Column name to use for heatmap values (e.g., 'auc_plus', 'auc_minus')
    plot_type : str
        'combined' for all FOVs overlaid, 'individual' for separate FOV plots
    roi_type : str
        'real' for actual ROI shapes, 'shape' for standardized shapes
    shape : str
        Shape to use when roi_type='shape' ('hexagon', 'square', 'circle', 'star')
    grid_size : int
        Size of the grid elements when roi_type='shape'
    roi_smoothing : float
        Smoothing factor for ROI outlines when roi_type='real'
    outline : bool
        Whether to show shape outlines
    blend_colors : bool
        Whether to enable color blending between shapes
    show_cell_numbers : bool
        Whether to display cell numbers in the visualization
    """
    if column_name not in df_master.columns:
        raise ValueError(f"Column '{column_name}' not found in df_master")
        
    # Define ordered days
    ordered_days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    if plot_type == 'combined':
        # Single row of subplots for all days
        fig, axes = plt.subplots(1, len(ordered_days), figsize=(5 * len(ordered_days), 6), dpi=150)
        fig.suptitle(f'Combined Cell Positions Heatmap Across Days', fontsize=16)
        
        for col, day in enumerate(ordered_days):
            ax = axes[col]
            ax.set_xlim(0, 512)
            ax.set_ylim(512, 0)
            ax.set_title(day, fontsize=12)
            
            day_df = df_master[df_master['day'] == day]
            if day_df.empty:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.axis('off')
                continue
            
            # Process each FOV
            unique_fovs = day_df[['animal', 'fov']].drop_duplicates()
            for animal, fov in unique_fovs.values:
                roi_folder = os.path.join(basedir, day, animal, fov)
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')] if os.path.exists(roi_folder) else []
                
                if roi_files:
                    roi_path = os.path.join(roi_folder, roi_files[0])
                    rois = roifile.ImagejRoi.fromfile(roi_path)
                    df_subset = day_df[(day_df['animal'] == animal) & (day_df['fov'] == fov)]
                    
                    for roi_number, roi in enumerate(rois, 1):
                        coords = roi.coordinates()
                        center_x = np.mean(coords[:, 0])
                        center_y = np.mean(coords[:, 1])
                        
                        matching_rows = df_subset[df_subset['cell_number'] == roi_number]
                        cell_value = matching_rows[column_name].mean() if not matching_rows.empty else np.nan
                        color = plt.cm.PuRd(cell_value) if not np.isnan(cell_value) else 'gray'
                        
                        if roi_type == 'real':
                            x_coords, y_coords = coords[:, 0], coords[:, 1]
                            if roi_smoothing > 0:
                                x_coords, y_coords = smooth_coordinates(x_coords, y_coords, roi_smoothing)
                            patch = Polygon(np.column_stack([x_coords, y_coords]),
                                         closed=True, fill=True,
                                         edgecolor='black' if outline else 'none',
                                         facecolor=color,
                                         alpha=0.7 if blend_colors else 1.0)
                        else:  # roi_type == 'shape'
                            patch = create_cell_patch(shape, center_x, center_y, grid_size,
                                                   color, outline, 0.7 if blend_colors else 1.0)
                        
                        ax.add_patch(patch)
                        
                        if show_cell_numbers:
                            ax.text(center_x, center_y, str(roi_number),
                                  ha='center', va='center',
                                  fontsize=8, fontweight='bold', color='black')
            
            ax.grid(True, linestyle='--', alpha=0.5)
            ax.set_xticks([])
            ax.set_yticks([])
        
    else:  # plot_type == 'individual'
        for animal in df_master['animal'].unique():
            fovs_for_animal = df_master[df_master['animal'] == animal]['fov'].unique()
            all_fovs = sorted(fovs_for_animal)
            
            if not all_fovs:
                continue
                
            fig, axes = plt.subplots(len(all_fovs), len(ordered_days),
                                   figsize=(len(ordered_days) * 5, len(all_fovs) * 5),
                                   dpi=150, squeeze=False)
            fig.suptitle(f'Cell Positions Heatmap - {animal}', fontsize=16)
            
            for row, fov in enumerate(all_fovs):
                for col, day in enumerate(ordered_days):
                    ax = axes[row, col]
                    ax.set_xlim(0, 512)
                    ax.set_ylim(512, 0)
                    
                    roi_folder = os.path.join(basedir, day, animal, fov)
                    roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')] if os.path.exists(roi_folder) else []
                    
                    if roi_files:
                        roi_path = os.path.join(roi_folder, roi_files[0])
                        rois = roifile.ImagejRoi.fromfile(roi_path)
                        df_subset = df_master[(df_master['day'] == day) &
                                            (df_master['animal'] == animal) &
                                            (df_master['fov'] == fov)]
                        
                        for roi_number, roi in enumerate(rois, 1):
                            coords = roi.coordinates()
                            center_x = np.mean(coords[:, 0])
                            center_y = np.mean(coords[:, 1])
                            
                            cell_value = df_subset[df_subset['cell_number'] == roi_number][column_name].mean()
                            color = plt.cm.PuRd(cell_value) if not np.isnan(cell_value) else 'gray'
                            
                            if roi_type == 'real':
                                x_coords, y_coords = coords[:, 0], coords[:, 1]
                                if roi_smoothing > 0:
                                    x_coords, y_coords = smooth_coordinates(x_coords, y_coords, roi_smoothing)
                                patch = Polygon(np.column_stack([x_coords, y_coords]),
                                             closed=True, fill=True,
                                             edgecolor='black' if outline else 'none',
                                             facecolor=color,
                                             alpha=0.7 if blend_colors else 1.0)
                            else:  # roi_type == 'shape'
                                patch = create_cell_patch(shape, center_x, center_y, grid_size,
                                                       color, outline, 0.7 if blend_colors else 1.0)
                            
                            ax.add_patch(patch)
                            
                            if show_cell_numbers:
                                ax.text(center_x, center_y, str(roi_number),
                                      ha='center', va='center',
                                      fontsize=8, fontweight='bold', color='black')
                        
                        ax.set_title(f'{day} - {fov}', fontsize=10)
                        ax.grid(True, linestyle='--', alpha=0.5)
                    else:
                        ax.text(0.5, 0.5, f'No data\n{day}',
                               ha='center', va='center',
                               transform=ax.transAxes)
                        ax.axis('off')
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap="PuRd",
                              norm=plt.Normalize(vmin=df_master[column_name].min(),
                                               vmax=df_master[column_name].max()))
    cbar = plt.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
    cbar.set_label(column_name, fontsize=10)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    
    # Save the figure
    plot_type_str = 'combined' if plot_type == 'combined' else 'individual'
    roi_type_str = f"_{roi_type}"
    shape_str = f"_{shape}" if roi_type == 'shape' else f"_smooth_{roi_smoothing}"
    
    # save_path = os.path.join(basedir, 'Results', 'Cell_Visualizations',
    #                         f"cell_positions_{plot_type_str}{roi_type_str}{shape_str}_{column_name}.png")
    # os.makedirs(os.path.dirname(save_path), exist_ok=True)
    # plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Saved {plot_type_str} cell position visualization with {roi_type} ROIs")

# Example usage:

# # For individual FOV plots with real ROIs
# visualize_cell_positions(basedir, df_master, column_name="auc_plus",
#                         plot_type='individual', roi_type='real',
#                         roi_smoothing=0, outline=True, blend_colors=True,
#                         show_cell_numbers=True)

# # For combined plot with real ROIs
# visualize_cell_positions(basedir, df_master, column_name="auc_plus",
#                         plot_type='combined', roi_type='real',
#                         roi_smoothing=100, outline=True, blend_colors=True)

# For combined plot with shape ROIs
visualize_cell_positions(basedir, df_master, column_name="auc_plus",
                        plot_type='combined', roi_type='shape',
                        shape='hexagon', grid_size=30, outline=True, blend_colors=True)
visualize_cell_positions(basedir, df_master, column_name="auc_minus",
                        plot_type='combined', roi_type='shape',
                        shape='hexagon', grid_size=30, outline=True, blend_colors=True)



# Calculating minimum distance from highest auROC cell to next highest etc in real data frame
## Data is plotted and added to df_master

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, linregress, gaussian_kde
import roifile

def get_roi_centers(rois):
    """Calculate the center of each ROI."""
    centers = []
    for roi in rois:
        coords = roi.coordinates()
        center_x = np.mean(coords[:, 0])
        center_y = np.mean(coords[:, 1])
        centers.append((center_x, center_y))
    return np.array(centers)

def calculate_distances(center1, centers, metric_type, fov_center=None):
    """Calculate distances using different metrics."""
    if metric_type == 'euclidean':
        return cdist([center1], centers, metric='euclidean').flatten()
    elif metric_type == 'manhattan':
        return cdist([center1], centers, metric='cityblock').flatten()
    elif metric_type == 'chebyshev':
        return cdist([center1], centers, metric='chebyshev').flatten()
    elif metric_type == 'combined1':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        manhattan = cdist([center1], centers, metric='cityblock').flatten()
        return 0.7 * euclidean + 0.3 * manhattan
    elif metric_type == 'combined2':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        chebyshev = cdist([center1], centers, metric='chebyshev').flatten()
        return np.sqrt(euclidean * chebyshev)
    elif metric_type == 'combined3':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        rel_positions = centers - center1
        angles = np.arctan2(rel_positions[:, 1], rel_positions[:, 0])
        return euclidean * (1 + 0.2 * np.cos(angles))

def add_distance_column_to_df(df_master, basedir, metric_type='combined3'):
    """
    Add a column to df_master with distances from each cell to the most active cell in its FOV.
    Uses the specified distance metric.
    
    Parameters:
    -----------
    df_master : pandas.DataFrame
        Master dataframe containing cell data
    basedir : str
        Base directory containing ROI files
    metric_type : str
        Distance metric to use for calculation
        
    Returns:
    --------
    pandas.DataFrame
        The original df_master dataframe with the new distance column added
    """
    # Make a copy of the dataframe (for internal processing, but we'll still update the original)
    df = df_master.copy()
    
    # Initialize the distance column
    df['auc_distance'] = np.nan
    
    # Process each day, animal, FOV combination
    days = df['day'].unique()
    
    print(f"Adding distance column using {metric_type} metric...")
    
    for day in days:
        print(f"Processing day: {day}")
        
        animals = df[df['day'] == day]['animal'].unique()
        for animal in animals:
            fovs = df[(df['day'] == day) & (df['animal'] == animal)]['fov'].unique()
            
            for fov in fovs:
                print(f"  Processing {animal}, FOV {fov}")
                
                # Get ROI centers
                roi_folder = os.path.join(basedir, day, animal, fov)
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                if not roi_files:
                    print(f"    No ROI files found for {day}, {animal}, {fov}")
                    continue
                    
                roi_path = os.path.join(roi_folder, roi_files[0])
                try:
                    rois = roifile.ImagejRoi.fromfile(roi_path)
                    centers = get_roi_centers(rois)
                except Exception as e:
                    print(f"    Error reading ROI file: {e}")
                    continue
                
                # Process both CS+ and CS- separately
                for cs_type in ['CS+', 'CS-']:
                    column_name = 'auc_plus' if cs_type == 'CS+' else 'auc_minus'
                    
                    # Get data for this FOV and CS type
                    df_subset = df[(df['day'] == day) & 
                                   (df['animal'] == animal) & 
                                   (df['fov'] == fov) &
                                   (df['cs_type'] == cs_type)]
                    
                    if df_subset.empty:
                        continue
                    
                    # Get unique cell numbers and their AUC values
                    cell_auc_values = df_subset.groupby('cell_number')[column_name].first().reset_index()
                    
                    # Find the cell with the highest AUC value
                    highest_value_row = cell_auc_values.loc[cell_auc_values[column_name].idxmax()]
                    highest_cell_number = int(highest_value_row['cell_number'])
                    
                    if highest_cell_number - 1 >= len(centers):
                        print(f"    Cell number {highest_cell_number} out of range for centers array with length {len(centers)}")
                        continue
                    
                    # Calculate distances from highest AUC cell to all cells
                    highest_center = centers[highest_cell_number - 1]
                    distances = calculate_distances(highest_center, centers, metric_type)
                    
                    # Create a mapping from cell_number to distance
                    distance_mapping = {i+1: distances[i] for i in range(len(distances))}
                    
                    # Update the distance column for all rows in this FOV and CS type
                    for idx, row in df_subset.iterrows():
                        cell_num = row['cell_number']
                        if cell_num in distance_mapping:
                            df.loc[idx, 'auc_distance'] = distance_mapping[cell_num]
    
    print("Distance column added successfully!")
    
    # Update the original df_master with the new distance column
    df_master['auc_distance'] = df['auc_distance']
    
    return df_master

def analyze_multi_distance_correlation(basedir, days, df_master, column_name, sex='both', normalize=False, vis_metric_type='euclidean'):
    
    ordered_days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    days = [day for day in ordered_days if day in days]
    
    # Add sex filtering
    if sex != 'both':
        df_master = df_master[df_master['sex'] == sex].copy()
    
    # Add CS type filtering based on column_name
    cs_type = 'CS+' if 'plus' in column_name else 'CS-'
    df_master = df_master[df_master['cs_type'] == cs_type].copy()
    
    metrics = {
        'euclidean': 'Euclidean Distance',
        'manhattan': 'Manhattan Distance',
        'chebyshev': 'Chebyshev Distance',
        'combined1': 'Weighted Euclidean-Manhattan',
        'combined2': 'Geometric Mean Distance',
        'combined3': 'Direction-Weighted Distance'
    }
    
    # Update figure title to include sex information
    sex_label = f" ({sex})" if sex != 'both' else ''
    fig, axes = plt.subplots(1, len(days), figsize=(4*len(days), 4), squeeze=False)
    fig.suptitle(f'{metrics[vis_metric_type]} Distance Correlation Analysis{sex_label}{" (Normalized)" if normalize else ""}', 
                 fontsize=16)
    
    results = []
    
    for day_idx, day in enumerate(days):
        # print(f"\nProcessing day: {day}")
        
        for metric_type, metric_name in metrics.items():
            all_distances = []
            all_values = []
            
            animals = df_master[df_master['day'] == day]['animal'].unique()
            
            for animal in animals:
                fovs = df_master[(df_master['day'] == day) & 
                                 (df_master['animal'] == animal)]['fov'].unique()
                
                for fov in fovs:
                    roi_folder = os.path.join(basedir, day, animal, fov)
                    roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                    if not roi_files:
                        continue
                        
                    roi_path = os.path.join(roi_folder, roi_files[0])
                    rois = roifile.ImagejRoi.fromfile(roi_path)
                    centers = get_roi_centers(rois)
                    
                    df_subset = df_master[(df_master['day'] == day) & 
                                          (df_master['animal'] == animal) & 
                                          (df_master['fov'] == fov)]
                    
                    # averaged_values = df_subset.groupby('cell_number')[column_name].mean().reset_index()
                    averaged_values = df_subset.groupby('cell_number')[column_name].first().reset_index()

                    
                    if normalize:
                        max_value = averaged_values[column_name].max()
                        if max_value > 0:
                            averaged_values[column_name] = averaged_values[column_name] / max_value

                    highest_value_row = averaged_values.loc[averaged_values[column_name].idxmax()]
                    highest_cell_number = int(highest_value_row['cell_number'])
                    
                    if highest_cell_number - 1 >= len(centers):
                        continue
                    
                    highest_center = centers[highest_cell_number - 1]
                    fov_center = np.mean(centers, axis=0)
                    distances = calculate_distances(highest_center, centers, metric_type, fov_center)
                    
                    all_distances.extend(distances)
                    all_values.extend(averaged_values[column_name].values)
            
            if all_distances and all_values:
                all_distances = np.array(all_distances)
                all_values = np.array(all_values)
                
                corr, p_value = pearsonr(all_distances, all_values)
                slope, intercept, r_value, _, _ = linregress(all_distances, all_values)
                
                results.append({
                    'day': day,
                    'metric': metric_name,
                    'correlation': corr,
                    'p_value': p_value,
                    'n': len(all_values)
                })
                
                if metric_type == vis_metric_type:
                    ax = axes[0, day_idx]
                    xy = np.vstack([all_distances, all_values])
                    z = gaussian_kde(xy)(xy)
                    idx = z.argsort()
                    scatter = ax.scatter(all_distances[idx], all_values[idx], 
                                         c=z[idx], cmap='PuRd', 
                                         alpha=0.5, s=20)
                    
                    x_range = np.array([min(all_distances), max(all_distances)])
                    ax.plot(x_range, slope * x_range + intercept, 
                            color='red', linestyle='--',
                            label=f'r={corr:.2f}')
                    
                    ax.set_title(f'{day}\n{metric_name}\np={p_value:.1e}')
                    ax.set_xlabel('Distance')
                    ylabel = f"{column_name}"
                    if normalize:
                        ylabel += "\n(Normalized)"
                    ax.set_ylabel(ylabel)
                    ax.grid(True, linestyle='--', alpha=0.7)
                    ax.legend()
            else:
                if metric_type == vis_metric_type:
                    ax = axes[0, day_idx]
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center')
    
    # plt.tight_layout()
    # save_path = os.path.join(basedir, 'Results', 'Master_Correlation_Plots', 
    #                          f"multi_metric_{vis_metric_type}_{column_name}_distance_correlation{'_normalized' if normalize else ''}_06-20-25.pdf")
    # os.makedirs(os.path.dirname(save_path), exist_ok=True)
    # plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    results_df = pd.DataFrame(results)
    results_df['day_order'] = results_df['day'].map({day: i for i, day in enumerate(ordered_days)})
    results_df = results_df.sort_values(['day_order', 'metric']).drop('day_order', axis=1)

    ### Uncomment to see stats
    print("\nCorrelation Results Summary:")
    print(results_df)
    
    return results_df

# Example usage with added distance column
if __name__ == "__main__":
    # First, add the distance column to df_master using the preferred metric
    metric_type = 'combined3'  # Use the same metric as in vis_metric_type
    
    # Update df_master in place with the new distance column
    df_master = add_distance_column_to_df(df_master, basedir, metric_type)
    
    # Save the updated dataframe with distance column
    results_dir = os.path.join(basedir, 'Results')
    os.makedirs(results_dir, exist_ok=True)
    df_master.to_csv(os.path.join(results_dir, 'df_master_with_distance.csv'), index=False)
    
    print("Updated df_master with distance column saved!")
    
    # Continue with the original analysis using the updated df_master
    results = analyze_multi_distance_correlation(
        basedir, days, df_master, 
        column_name="auc_plus", normalize=False, 
        sex='both', vis_metric_type=metric_type
    )
    
    results = analyze_multi_distance_correlation(
        basedir, days, df_master, 
        column_name="auc_minus", normalize=False, 
        sex='both', vis_metric_type=metric_type
    )

# Calculating minimum distance from highest auROC cell to next highest etc in control data frame.  
## Data is plotted and added to df_master_control

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, linregress, gaussian_kde
import roifile

def get_roi_centers(rois):
    """Calculate the center of each ROI."""
    centers = []
    for roi in rois:
        coords = roi.coordinates()
        center_x = np.mean(coords[:, 0])
        center_y = np.mean(coords[:, 1])
        centers.append((center_x, center_y))
    return np.array(centers)

def calculate_distances(center1, centers, metric_type, fov_center=None):
    """Calculate distances using different metrics."""
    if metric_type == 'euclidean':
        return cdist([center1], centers, metric='euclidean').flatten()
    elif metric_type == 'manhattan':
        return cdist([center1], centers, metric='cityblock').flatten()
    elif metric_type == 'chebyshev':
        return cdist([center1], centers, metric='chebyshev').flatten()
    elif metric_type == 'combined1':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        manhattan = cdist([center1], centers, metric='cityblock').flatten()
        return 0.7 * euclidean + 0.3 * manhattan
    elif metric_type == 'combined2':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        chebyshev = cdist([center1], centers, metric='chebyshev').flatten()
        return np.sqrt(euclidean * chebyshev)
    elif metric_type == 'combined3':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        rel_positions = centers - center1
        angles = np.arctan2(rel_positions[:, 1], rel_positions[:, 0])
        return euclidean * (1 + 0.2 * np.cos(angles))

def add_distance_column_to_df_control(df_master_control, basedir, metric_type='combined3'):
    """
    Add a column to df_master_control with distances from each cell to the most active cell in its FOV.
    Uses the specified distance metric.
    
    Parameters:
    -----------
    df_master_control : pandas.DataFrame
        Master control dataframe containing cell data
    basedir : str
        Base directory containing ROI files
    metric_type : str
        Distance metric to use for calculation
        
    Returns:
    --------
    pandas.DataFrame
        The original df_master_control dataframe with the new distance column added
    """
    # Make a copy of the dataframe (for internal processing, but we'll still update the original)
    df = df_master_control.copy()
    
    # Initialize the distance column
    df['auc_distance'] = np.nan
    
    # Process each day, animal, FOV combination
    days = df['day'].unique()
    
    print(f"Adding distance column using {metric_type} metric to control dataframe...")
    
    for day in days:
        print(f"Processing day: {day}")
        
        animals = df[df['day'] == day]['animal'].unique()
        for animal in animals:
            fovs = df[(df['day'] == day) & (df['animal'] == animal)]['fov'].unique()
            
            for fov in fovs:
                print(f"  Processing {animal}, FOV {fov}")
                
                # Get ROI centers
                roi_folder = os.path.join(basedir, day, animal, fov)
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                if not roi_files:
                    print(f"    No ROI files found for {day}, {animal}, {fov}")
                    continue
                    
                roi_path = os.path.join(roi_folder, roi_files[0])
                try:
                    rois = roifile.ImagejRoi.fromfile(roi_path)
                    centers = get_roi_centers(rois)
                except Exception as e:
                    print(f"    Error reading ROI file: {e}")
                    continue
                
                # Process both CS+ and CS- separately
                for cs_type in ['CS+', 'CS-']:
                    column_name = 'auc_plus' if cs_type == 'CS+' else 'auc_minus'
                    
                    # Get data for this FOV and CS type
                    df_subset = df[(df['day'] == day) & 
                                   (df['animal'] == animal) & 
                                   (df['fov'] == fov) &
                                   (df['cs_type'] == cs_type)]
                    
                    if df_subset.empty:
                        continue
                    
                    # Get unique cell numbers and their AUC values
                    cell_auc_values = df_subset.groupby('cell_number')[column_name].first().reset_index()
                    
                    # Find the cell with the highest AUC value
                    highest_value_row = cell_auc_values.loc[cell_auc_values[column_name].idxmax()]
                    highest_cell_number = int(highest_value_row['cell_number'])
                    
                    if highest_cell_number - 1 >= len(centers):
                        print(f"    Cell number {highest_cell_number} out of range for centers array with length {len(centers)}")
                        continue
                    
                    # Calculate distances from highest AUC cell to all cells
                    highest_center = centers[highest_cell_number - 1]
                    distances = calculate_distances(highest_center, centers, metric_type)
                    
                    # Create a mapping from cell_number to distance
                    distance_mapping = {i+1: distances[i] for i in range(len(distances))}
                    
                    # Update the distance column for all rows in this FOV and CS type
                    for idx, row in df_subset.iterrows():
                        cell_num = row['cell_number']
                        if cell_num in distance_mapping:
                            df.loc[idx, 'auc_distance'] = distance_mapping[cell_num]
    
    print("Distance column added successfully to control dataframe!")
    
    # Update the original df_master_control with the new distance column
    df_master_control['auc_distance'] = df['auc_distance']
    
    return df_master_control

# Example usage with added distance column to control dataframe
if __name__ == "__main__":
    # First, add the distance column to df_master_control using the preferred metric
    metric_type = 'combined3'  # Use the same metric as in vis_metric_type
    
    # Update df_master_control in place with the new distance column
    df_master_control = add_distance_column_to_df_control(df_master_control, basedir, metric_type)
    
    # Save the updated dataframe with distance column
    results_dir = os.path.join(basedir, 'Results')
    os.makedirs(results_dir, exist_ok=True)
    df_master_control.to_csv(os.path.join(results_dir, 'df_master_control_with_distance.csv'), index=False)
    
    print("Updated df_master_control with distance column saved!")

# Actual Data Spatial Mapping Compared to Control Spatial Mapping

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, RegularPolygon, Rectangle, Circle
import roifile
from scipy.interpolate import splprep, splev

def smooth_coordinates(x_coords, y_coords, smoothing_factor, num_points=100):
    """
    Smooth ROI coordinates using B-spline interpolation.
    """
    if len(x_coords) < 4:
        return x_coords, y_coords
    
    if (x_coords[0], y_coords[0]) != (x_coords[-1], y_coords[-1]):
        x_coords = np.append(x_coords, x_coords[0])
        y_coords = np.append(y_coords, y_coords[0])
    
    tck, u = splprep([x_coords, y_coords], s=smoothing_factor, per=True)
    u_new = np.linspace(0, 1, num_points)
    x_new, y_new = splev(u_new, tck)
    
    return x_new, y_new

def create_star(x_center, y_center, size, num_points=5):
    """Generate coordinates for a star shape."""
    angle = np.pi / num_points
    star_points = []
    for i in range(2 * num_points):
        radius = size if i % 2 == 0 else size / 2
        x = x_center + radius * np.cos(i * angle)
        y = y_center + radius * np.sin(i * angle)
        star_points.append((x, y))
    return star_points

def create_cell_patch(shape_type, center_x, center_y, size, color, outline=True, alpha=0.7):
    """Create a matplotlib patch based on specified shape type."""
    edgecolor = 'black' if outline else 'none'
    
    if shape_type == 'hexagon':
        return RegularPolygon((center_x, center_y), numVertices=6, radius=size,
                            orientation=np.radians(30), facecolor=color,
                            edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'square':
        return Rectangle((center_x - size/2, center_y - size/2),
                        size, size, facecolor=color,
                        edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'circle':
        return Circle((center_x, center_y), radius=size/2,
                     facecolor=color, edgecolor=edgecolor, alpha=alpha)
    elif shape_type == 'star':
        star_points = create_star(center_x, center_y, size)
        return Polygon(star_points, closed=True, facecolor=color,
                      edgecolor=edgecolor, alpha=alpha)
    else:
        raise ValueError("Invalid shape_type. Choose from 'hexagon', 'square', 'circle', 'star'")

def visualize_control_cell_positions(basedir, df_master_control, column_name, plot_type='combined', 
                                    roi_type='real', shape='hexagon', grid_size=30,
                                    roi_smoothing=0, outline=True, blend_colors=True,
                                    show_cell_numbers=False, side_by_side=False, 
                                    df_master_original=None):
    """
    Visualize cell positions for control data with various options.
    
    Parameters:
    -----------
    basedir : str
        Base directory containing the data
    df_master_control : pandas DataFrame
        Control dataframe containing cell data
    column_name : str
        Column name to use for heatmap values (e.g., 'auc_plus', 'auc_minus')
    plot_type : str
        'combined' for all FOVs overlaid, 'individual' for separate FOV plots
    roi_type : str
        'real' for actual ROI shapes, 'shape' for standardized shapes
    shape : str
        Shape to use when roi_type='shape' ('hexagon', 'square', 'circle', 'star')
    grid_size : int
        Size of the grid elements when roi_type='shape'
    roi_smoothing : float
        Smoothing factor for ROI outlines when roi_type='real'
    outline : bool
        Whether to show shape outlines
    blend_colors : bool
        Whether to enable color blending between shapes
    show_cell_numbers : bool
        Whether to display cell numbers in the visualization
    side_by_side : bool
        If True, show original and control data side by side
    df_master_original : pandas DataFrame
        Original dataframe for side-by-side comparison (required if side_by_side=True)
    """
    if column_name not in df_master_control.columns:
        raise ValueError(f"Column '{column_name}' not found in df_master_control")
        
    if side_by_side and df_master_original is None:
        raise ValueError("df_master_original is required for side-by-side comparison")
        
    # Define ordered days
    ordered_days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    # Define colormap and normalization
    if side_by_side:
        # Use the same color scale for both original and control
        all_values = np.concatenate([
            df_master_original[column_name].dropna().values,
            df_master_control[column_name].dropna().values
        ])
        vmin, vmax = np.min(all_values), np.max(all_values)
    else:
        vmin, vmax = df_master_control[column_name].min(), df_master_control[column_name].max()
    
    cmap = plt.cm.PuRd
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    if plot_type == 'combined':
        # Set up figure based on whether we're doing side-by-side comparison
        if side_by_side:
            fig, axes = plt.subplots(2, len(ordered_days), 
                                   figsize=(5 * len(ordered_days), 12), 
                                   dpi=150)
            fig.suptitle(f'Original vs. Control Data - {column_name}', fontsize=16)
            row_labels = ['Original', 'Control']
            
            # Add row labels
            for i, label in enumerate(row_labels):
                fig.text(0.01, 0.75 - i*0.5, label, va='center', ha='center', 
                        rotation='vertical', fontsize=16, fontweight='bold')
        else:
            fig, axes = plt.subplots(1, len(ordered_days), 
                                   figsize=(5 * len(ordered_days), 6), 
                                   dpi=150)
            fig.suptitle(f'Control Data Cell Positions - {column_name}', fontsize=16)
            axes = axes.reshape(1, -1)  # Ensure axes is 2D for consistent indexing
        
        # Process each day
        for col, day in enumerate(ordered_days):
            # For side-by-side, process both original and control data
            if side_by_side:
                # Original data
                ax_orig = axes[0, col]
                ax_orig.set_xlim(0, 512)
                ax_orig.set_ylim(512, 0)
                ax_orig.set_title(day, fontsize=12)
                
                day_df_orig = df_master_original[df_master_original['day'] == day]
                if not day_df_orig.empty:
                    _plot_day_data(ax_orig, basedir, day, day_df_orig, column_name, 
                                 roi_type, roi_smoothing, shape, grid_size, 
                                 outline, blend_colors, show_cell_numbers, cmap, norm)
                else:
                    ax_orig.text(0.5, 0.5, 'No data', ha='center', va='center', 
                               transform=ax_orig.transAxes)
                    ax_orig.axis('off')
                
                # Control data
                ax_ctrl = axes[1, col]
                ax_ctrl.set_xlim(0, 512)
                ax_ctrl.set_ylim(512, 0)
                
                day_df_ctrl = df_master_control[df_master_control['day'] == day]
                if not day_df_ctrl.empty:
                    _plot_day_data(ax_ctrl, basedir, day, day_df_ctrl, column_name, 
                                 roi_type, roi_smoothing, shape, grid_size, 
                                 outline, blend_colors, show_cell_numbers, cmap, norm)
                else:
                    ax_ctrl.text(0.5, 0.5, 'No data', ha='center', va='center', 
                               transform=ax_ctrl.transAxes)
                    ax_ctrl.axis('off')
            else:
                # Control data only
                ax = axes[0, col]
                ax.set_xlim(0, 512)
                ax.set_ylim(512, 0)
                ax.set_title(day, fontsize=12)
                
                day_df = df_master_control[df_master_control['day'] == day]
                if not day_df.empty:
                    _plot_day_data(ax, basedir, day, day_df, column_name, 
                                 roi_type, roi_smoothing, shape, grid_size, 
                                 outline, blend_colors, show_cell_numbers, cmap, norm)
                else:
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                          transform=ax.transAxes)
                    ax.axis('off')
    
    else:  # plot_type == 'individual'
        for animal in df_master_control['animal'].unique():
            fovs_for_animal = df_master_control[df_master_control['animal'] == animal]['fov'].unique()
            all_fovs = sorted(fovs_for_animal)
            
            if not all_fovs:
                continue
                
            if side_by_side:
                fig, axes = plt.subplots(len(all_fovs)*2, len(ordered_days),
                                       figsize=(len(ordered_days) * 5, len(all_fovs) * 10),
                                       dpi=150, squeeze=False)
                fig.suptitle(f'Original vs. Control Data - {animal} - {column_name}', fontsize=16)
            else:
                fig, axes = plt.subplots(len(all_fovs), len(ordered_days),
                                       figsize=(len(ordered_days) * 5, len(all_fovs) * 5),
                                       dpi=150, squeeze=False)
                fig.suptitle(f'Control Data Cell Positions - {animal} - {column_name}', fontsize=16)
            
            # Process each FOV and day
            for row_idx, fov in enumerate(all_fovs):
                for col, day in enumerate(ordered_days):
                    if side_by_side:
                        # Original data
                        row_orig = row_idx * 2
                        ax_orig = axes[row_orig, col]
                        ax_orig.set_xlim(0, 512)
                        ax_orig.set_ylim(512, 0)
                        
                        df_subset_orig = df_master_original[(df_master_original['day'] == day) &
                                                         (df_master_original['animal'] == animal) &
                                                         (df_master_original['fov'] == fov)]
                        
                        if not df_subset_orig.empty:
                            _plot_fov_data(ax_orig, basedir, day, animal, fov, df_subset_orig, 
                                         column_name, roi_type, roi_smoothing, shape, grid_size, 
                                         outline, blend_colors, show_cell_numbers, cmap, norm)
                            ax_orig.set_title(f'{day} - {fov} (Original)', fontsize=10)
                        else:
                            ax_orig.text(0.5, 0.5, f'No original data\n{day}',
                                       ha='center', va='center',
                                       transform=ax_orig.transAxes)
                            ax_orig.axis('off')
                        
                        # Control data
                        row_ctrl = row_idx * 2 + 1
                        ax_ctrl = axes[row_ctrl, col]
                        ax_ctrl.set_xlim(0, 512)
                        ax_ctrl.set_ylim(512, 0)
                        
                        df_subset_ctrl = df_master_control[(df_master_control['day'] == day) &
                                                        (df_master_control['animal'] == animal) &
                                                        (df_master_control['fov'] == fov)]
                        
                        if not df_subset_ctrl.empty:
                            _plot_fov_data(ax_ctrl, basedir, day, animal, fov, df_subset_ctrl, 
                                         column_name, roi_type, roi_smoothing, shape, grid_size, 
                                         outline, blend_colors, show_cell_numbers, cmap, norm)
                            ax_ctrl.set_title(f'{day} - {fov} (Control)', fontsize=10)
                        else:
                            ax_ctrl.text(0.5, 0.5, f'No control data\n{day}',
                                       ha='center', va='center',
                                       transform=ax_ctrl.transAxes)
                            ax_ctrl.axis('off')
                    else:
                        # Control data only
                        ax = axes[row_idx, col]
                        ax.set_xlim(0, 512)
                        ax.set_ylim(512, 0)
                        
                        df_subset = df_master_control[(df_master_control['day'] == day) &
                                                   (df_master_control['animal'] == animal) &
                                                   (df_master_control['fov'] == fov)]
                        
                        if not df_subset.empty:
                            _plot_fov_data(ax, basedir, day, animal, fov, df_subset, 
                                         column_name, roi_type, roi_smoothing, shape, grid_size, 
                                         outline, blend_colors, show_cell_numbers, cmap, norm)
                            ax.set_title(f'{day} - {fov} (Control)', fontsize=10)
                        else:
                            ax.text(0.5, 0.5, f'No control data\n{day}',
                                   ha='center', va='center',
                                   transform=ax.transAxes)
                            ax.axis('off')
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = plt.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
    cbar.set_label(column_name, fontsize=10)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    
    # Save the figure
    plot_type_str = 'combined' if plot_type == 'combined' else 'individual'
    roi_type_str = f"_{roi_type}"
    shape_str = f"_{shape}" if roi_type == 'shape' else f"_smooth_{roi_smoothing}"
    comparison_str = "_with_original" if side_by_side else ""
    
    # save_path = os.path.join(basedir, 'Results', 'Cell_Visualizations',
    #                        f"control_cell_positions_{plot_type_str}{roi_type_str}{shape_str}_{column_name}{comparison_str}.png")
    # os.makedirs(os.path.dirname(save_path), exist_ok=True)
    # plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    data_type = "control and original" if side_by_side else "control"
    print(f"Saved {plot_type_str} cell position visualization for {data_type} data with {roi_type} ROIs")

def _plot_day_data(ax, basedir, day, day_df, column_name, roi_type, roi_smoothing, 
                 shape, grid_size, outline, blend_colors, show_cell_numbers, cmap, norm):
    """Helper function to plot data for a specific day."""
    unique_fovs = day_df[['animal', 'fov']].drop_duplicates()
    for animal, fov in unique_fovs.values:
        roi_folder = os.path.join(basedir, day, animal, fov)
        roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')] if os.path.exists(roi_folder) else []
        
        if roi_files:
            roi_path = os.path.join(roi_folder, roi_files[0])
            rois = roifile.ImagejRoi.fromfile(roi_path)
            df_subset = day_df[(day_df['animal'] == animal) & (day_df['fov'] == fov)]
            
            for roi_number, roi in enumerate(rois, 1):
                coords = roi.coordinates()
                center_x = np.mean(coords[:, 0])
                center_y = np.mean(coords[:, 1])
                
                matching_rows = df_subset[df_subset['cell_number'] == roi_number]
                cell_value = matching_rows[column_name].mean() if not matching_rows.empty else np.nan
                
                if not np.isnan(cell_value):
                    color = cmap(norm(cell_value))
                else:
                    color = 'gray'
                
                if roi_type == 'real':
                    x_coords, y_coords = coords[:, 0], coords[:, 1]
                    if roi_smoothing > 0:
                        x_coords, y_coords = smooth_coordinates(x_coords, y_coords, roi_smoothing)
                    patch = Polygon(np.column_stack([x_coords, y_coords]),
                                 closed=True, fill=True,
                                 edgecolor='black' if outline else 'none',
                                 facecolor=color,
                                 alpha=0.7 if blend_colors else 1.0)
                else:  # roi_type == 'shape'
                    patch = create_cell_patch(shape, center_x, center_y, grid_size,
                                           color, outline, 0.7 if blend_colors else 1.0)
                
                ax.add_patch(patch)
                
                if show_cell_numbers:
                    ax.text(center_x, center_y, str(roi_number),
                          ha='center', va='center',
                          fontsize=8, fontweight='bold', color='black')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xticks([])
    ax.set_yticks([])

def _plot_fov_data(ax, basedir, day, animal, fov, df_subset, column_name, roi_type, 
                 roi_smoothing, shape, grid_size, outline, blend_colors, 
                 show_cell_numbers, cmap, norm):
    """Helper function to plot data for a specific FOV."""
    roi_folder = os.path.join(basedir, day, animal, fov)
    roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')] if os.path.exists(roi_folder) else []
    
    if roi_files:
        roi_path = os.path.join(roi_folder, roi_files[0])
        rois = roifile.ImagejRoi.fromfile(roi_path)
        
        for roi_number, roi in enumerate(rois, 1):
            coords = roi.coordinates()
            center_x = np.mean(coords[:, 0])
            center_y = np.mean(coords[:, 1])
            
            matching_rows = df_subset[df_subset['cell_number'] == roi_number]
            cell_value = matching_rows[column_name].mean() if not matching_rows.empty else np.nan
            
            if not np.isnan(cell_value):
                color = cmap(norm(cell_value))
            else:
                color = 'gray'
            
            if roi_type == 'real':
                x_coords, y_coords = coords[:, 0], coords[:, 1]
                if roi_smoothing > 0:
                    x_coords, y_coords = smooth_coordinates(x_coords, y_coords, roi_smoothing)
                patch = Polygon(np.column_stack([x_coords, y_coords]),
                             closed=True, fill=True,
                             edgecolor='black' if outline else 'none',
                             facecolor=color,
                             alpha=0.7 if blend_colors else 1.0)
            else:  # roi_type == 'shape'
                patch = create_cell_patch(shape, center_x, center_y, grid_size,
                                       color, outline, 0.7 if blend_colors else 1.0)
            
            ax.add_patch(patch)
            
            if show_cell_numbers:
                ax.text(center_x, center_y, str(roi_number),
                      ha='center', va='center',
                      fontsize=8, fontweight='bold', color='black')
        
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.set_xticks([])
        ax.set_yticks([])

# Example usage:

# # For visualizing control data only
# visualize_control_cell_positions(basedir, df_master_control, column_name="auc_plus",
#                                 plot_type='combined', roi_type='shape',
#                                 shape='hexagon', grid_size=30, outline=True, blend_colors=True)

# # For side-by-side comparison with original data
# visualize_control_cell_positions(basedir, df_master_control, column_name="auc_plus",
#                                 plot_type='combined', roi_type='shape',
#                                 shape='hexagon', grid_size=30, outline=True, blend_colors=True,
#                                 side_by_side=True, df_master_original=df_master)

# # For individual FOV plots with real ROIs
# visualize_control_cell_positions(basedir, df_master_control, column_name="auc_plus",
#                                 plot_type='individual', roi_type='real',
#                                 roi_smoothing=0, outline=True, blend_colors=True,
#                                 show_cell_numbers=True)

# # Execute these functions to create visualizations
# # For control data alone
# visualize_control_cell_positions(basedir, df_master_control, column_name="auc_plus",
#                                 plot_type='combined', roi_type='shape',
#                                 shape='hexagon', grid_size=30, outline=True, blend_colors=True)
# visualize_control_cell_positions(basedir, df_master_control, column_name="auc_minus",
#                                 plot_type='combined', roi_type='shape',
#                                 shape='hexagon', grid_size=30, outline=True, blend_colors=True)

# For side-by-side comparison
visualize_control_cell_positions(basedir, df_master_control, column_name="auc_plus",
                                plot_type='combined', roi_type='shape',
                                shape='hexagon', grid_size=30, outline=True, blend_colors=True,
                                side_by_side=True, df_master_original=df_master)
visualize_control_cell_positions(basedir, df_master_control, column_name="auc_minus",
                                plot_type='combined', roi_type='shape',
                                shape='hexagon', grid_size=30, outline=True, blend_colors=True,
                                side_by_side=True, df_master_original=df_master)

In [ ]:
# import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, linregress, gaussian_kde
import roifile

def get_roi_centers(rois):
    """Calculate the center of each ROI."""
    centers = []
    for roi in rois:
        coords = roi.coordinates()
        center_x = np.mean(coords[:, 0])
        center_y = np.mean(coords[:, 1])
        centers.append((center_x, center_y))
    return np.array(centers)

def calculate_distances(center1, centers, metric_type, fov_center=None):
    """Calculate distances using different metrics."""
    if metric_type == 'euclidean':
        return cdist([center1], centers, metric='euclidean').flatten()
    elif metric_type == 'manhattan':
        return cdist([center1], centers, metric='cityblock').flatten()
    elif metric_type == 'chebyshev':
        return cdist([center1], centers, metric='chebyshev').flatten()
    elif metric_type == 'combined1':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        manhattan = cdist([center1], centers, metric='cityblock').flatten()
        return 0.7 * euclidean + 0.3 * manhattan
    elif metric_type == 'combined2':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        chebyshev = cdist([center1], centers, metric='chebyshev').flatten()
        return np.sqrt(euclidean * chebyshev)
    elif metric_type == 'combined3':
        euclidean = cdist([center1], centers, metric='euclidean').flatten()
        rel_positions = centers - center1
        angles = np.arctan2(rel_positions[:, 1], rel_positions[:, 0])
        return euclidean * (1 + 0.2 * np.cos(angles))

def analyze_control_distance_correlation(basedir, days, df_master_control, column_name, 
                                        sex='both', normalize=False, vis_metric_type='euclidean',
                                        side_by_side=False, df_master_original=None):
    """
    Analyze spatial correlation between cells in control data.
    
    Parameters:
    -----------
    basedir : str
        Base directory containing the data
    days : list
        List of days to analyze
    df_master_control : pandas DataFrame
        Control dataframe
    column_name : str
        Column name to analyze (e.g., 'auc_plus', 'auc_minus')
    sex : str
        'male', 'female', or 'both' to filter by sex
    normalize : bool
        Whether to normalize values within each FOV
    vis_metric_type : str
        Distance metric to visualize
    side_by_side : bool
        Whether to plot control and original data side by side
    df_master_original : pandas DataFrame
        Original dataframe for side-by-side comparison
    """
    ordered_days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    days = [day for day in ordered_days if day in days]
    
    # Add sex filtering
    if sex != 'both':
        df_master_control = df_master_control[df_master_control['sex'] == sex].copy()
        if side_by_side and df_master_original is not None:
            df_master_original = df_master_original[df_master_original['sex'] == sex].copy()
    
    # Add CS type filtering based on column_name
    cs_type = 'CS+' if 'plus' in column_name else 'CS-'
    df_master_control = df_master_control[df_master_control['cs_type'] == cs_type].copy()
    if side_by_side and df_master_original is not None:
        df_master_original = df_master_original[df_master_original['cs_type'] == cs_type].copy()
    
    metrics = {
        'euclidean': 'Euclidean Distance',
        'manhattan': 'Manhattan Distance',
        'chebyshev': 'Chebyshev Distance',
        'combined1': 'Weighted Euclidean-Manhattan',
        'combined2': 'Geometric Mean Distance',
        'combined3': 'Direction-Weighted Distance'
    }
    
    # Update figure title to include sex information
    sex_label = f" ({sex})" if sex != 'both' else ''
    
    # Setup for plotting
    if side_by_side:
        fig, axes = plt.subplots(2, len(days), figsize=(4*len(days), 8), squeeze=False)
        fig.suptitle(f'{metrics[vis_metric_type]} Distance Correlation Analysis: Original vs Control{sex_label}{" (Normalized)" if normalize else ""}', 
                    fontsize=16)
        
        # Add row labels
        fig.text(0.01, 0.75, 'Original', va='center', ha='center', 
                rotation='vertical', fontsize=14, fontweight='bold')
        fig.text(0.01, 0.25, 'Control', va='center', ha='center', 
                rotation='vertical', fontsize=14, fontweight='bold')
    else:
        fig, axes = plt.subplots(1, len(days), figsize=(4*len(days), 4), squeeze=False)
        fig.suptitle(f'Control: {metrics[vis_metric_type]} Distance Correlation Analysis{sex_label}{" (Normalized)" if normalize else ""}', 
                    fontsize=16)
    
    results_control = []
    results_original = []
    
    # Process each day
    for day_idx, day in enumerate(days):
        # Process original data if side_by_side is True
        if side_by_side and df_master_original is not None:
            # Compute correlations for original data
            ax_orig = axes[0, day_idx]
            processed_data = _process_day_data(basedir, day, df_master_original, column_name, 
                                             vis_metric_type, metrics, normalize)
            
            all_distances_orig, all_values_orig = processed_data['distances'], processed_data['values']
            
            if len(all_distances_orig) > 0 and len(all_values_orig) > 0:
                corr, p_value, slope, intercept = _compute_stats(all_distances_orig, all_values_orig)
                
                results_original.append({
                    'day': day,
                    'metric': metrics[vis_metric_type],
                    'correlation': corr,
                    'p_value': p_value,
                    'n': len(all_values_orig),
                    'data_type': 'Original'
                })
                
                _plot_correlation(ax_orig, all_distances_orig, all_values_orig, 
                                corr, p_value, slope, intercept, 
                                day, metrics[vis_metric_type], column_name, normalize)
            else:
                ax_orig.text(0.5, 0.5, 'No data', ha='center', va='center')
        
        # Compute correlations for control data
        ax_idx = 1 if side_by_side else 0
        ax_ctrl = axes[ax_idx, day_idx]
        
        processed_data = _process_day_data(basedir, day, df_master_control, column_name, 
                                         vis_metric_type, metrics, normalize)
        
        all_distances_ctrl, all_values_ctrl = processed_data['distances'], processed_data['values']
        
        if len(all_distances_ctrl) > 0 and len(all_values_ctrl) > 0:
            corr, p_value, slope, intercept = _compute_stats(all_distances_ctrl, all_values_ctrl)
            
            results_control.append({
                'day': day,
                'metric': metrics[vis_metric_type],
                'correlation': corr,
                'p_value': p_value,
                'n': len(all_values_ctrl),
                'data_type': 'Control'
            })
            
            _plot_correlation(ax_ctrl, all_distances_ctrl, all_values_ctrl, 
                            corr, p_value, slope, intercept, 
                            day, metrics[vis_metric_type], column_name, normalize)
        else:
            ax_ctrl.text(0.5, 0.5, 'No data', ha='center', va='center')
    
    plt.tight_layout(rect=[0.02, 0, 1, 0.95] if side_by_side else None)
    
    # Save plot
    comparison_str = "_vs_original" if side_by_side else ""
    save_path = os.path.join(basedir, 'Results', 'Master_Correlation_Plots', 
                            f"control{comparison_str}_{vis_metric_type}_{column_name}_distance_correlation{'_normalized' if normalize else ''}_plasma-11-03-25.pdf")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Combine results
    if side_by_side and df_master_original is not None:
        results_df = pd.DataFrame(results_original + results_control)
    else:
        results_df = pd.DataFrame(results_control)
    
    # Sort results by day order
    results_df['day_order'] = results_df['day'].map({day: i for i, day in enumerate(ordered_days)})
    results_df = results_df.sort_values(['data_type', 'day_order', 'metric']).drop('day_order', axis=1)
    
    # Print summary
    print("\nDistance Correlation Results Summary:")
    print(results_df)
    
    # Optional: Create comparison table
    if side_by_side and df_master_original is not None:
        comparison_df = pd.pivot_table(
            results_df, 
            values=['correlation', 'p_value'], 
            index=['day'], 
            columns=['data_type']
        )
        
        print("\nOriginal vs Control Comparison:")
        print(comparison_df)
    
    return results_df

def _process_day_data(basedir, day, df, column_name, metric_type, metrics, normalize):
    """Process data for a specific day and metric."""
    all_distances = []
    all_values = []
    
    animals = df[df['day'] == day]['animal'].unique()
    
    for animal in animals:
        fovs = df[(df['day'] == day) & 
                 (df['animal'] == animal)]['fov'].unique()
        
        for fov in fovs:
            roi_folder = os.path.join(basedir, day, animal, fov)
            roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
            if not roi_files:
                continue
                
            roi_path = os.path.join(roi_folder, roi_files[0])
            rois = roifile.ImagejRoi.fromfile(roi_path)
            centers = get_roi_centers(rois)
            
            df_subset = df[(df['day'] == day) & 
                          (df['animal'] == animal) & 
                          (df['fov'] == fov)]
            
            # Use first value for each cell (AUC is the same for all trials of a cell)
            averaged_values = df_subset.groupby('cell_number')[column_name].first().reset_index()
            
            if normalize:
                max_value = averaged_values[column_name].max()
                if max_value > 0:
                    averaged_values[column_name] = averaged_values[column_name] / max_value
            
            # Skip if no valid values
            if averaged_values.empty or averaged_values[column_name].isnull().all():
                continue
                
            # Find cell with highest value
            highest_value_row = averaged_values.loc[averaged_values[column_name].idxmax()]
            highest_cell_number = int(highest_value_row['cell_number'])
            
            # Skip if cell number is out of range
            if highest_cell_number - 1 >= len(centers):
                continue
            
            highest_center = centers[highest_cell_number - 1]
            fov_center = np.mean(centers, axis=0)
            distances = calculate_distances(highest_center, centers, metric_type, fov_center)
            
            all_distances.extend(distances)
            all_values.extend(averaged_values[column_name].values)
    
    return {
        'distances': np.array(all_distances) if all_distances else [],
        'values': np.array(all_values) if all_values else []
    }

def _compute_stats(distances, values):
    """Compute correlation statistics."""
    corr, p_value = pearsonr(distances, values)
    slope, intercept, _, _, _ = linregress(distances, values)
    return corr, p_value, slope, intercept

def _plot_correlation(ax, distances, values, corr, p_value, slope, intercept, 
                     day, metric_name, column_name, normalize):
    """Plot correlation for a specific condition."""
    # Create density plot
    xy = np.vstack([distances, values])
    z = gaussian_kde(xy)(xy)
    idx = z.argsort()
    scatter = ax.scatter(distances[idx], values[idx], 
                       c=z[idx], cmap='plasma', 
                       alpha=0.5, s=20)
    
     # Add colorbar for density
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.8)
    cbar.set_label('Point Density', rotation=270, labelpad=15, fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    
    # Add regression line
    x_range = np.array([min(distances), max(distances)])
    ax.plot(x_range, slope * x_range + intercept, 
          color='black', linestyle='--',
          label=f'r={corr:.2f}')
    
    # Set consistent y-axis limits
    ax.set_ylim(-1, 1)
    
    # Add labels and grid
    ax.set_title(f'{day}\n{metric_name}\np={p_value:.1e}')
    ax.set_xlabel('Distance')
    ylabel = f"{column_name}"
    if normalize:
        ylabel += "\n(Normalized)"
    ax.set_ylabel(ylabel)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend()
    

# # Usage examples:

# # For control data only
# analyze_control_distance_correlation(
#     basedir=basedir, 
#     days=days, 
#     df_master_control=df_master_control, 
#     column_name="auc_plus", 
#     normalize=True, 
#     sex='both', 
#     vis_metric_type='combined3'
# )

# analyze_control_distance_correlation(
#     basedir=basedir, 
#     days=days, 
#     df_master_control=df_master_control, 
#     column_name="auc_minus", 
#     normalize=True, 
#     sex='both', 
#     vis_metric_type='combined3'
# )

# For side-by-side comparison with original data
analyze_control_distance_correlation(
    basedir=basedir, 
    days=days, 
    df_master_control=df_master_control, 
    column_name="auc_plus", 
    normalize=False, 
    sex='both', 
    vis_metric_type='combined3',
    side_by_side=True,
    df_master_original=df_master
)

analyze_control_distance_correlation(
    basedir=basedir, 
    days=days, 
    df_master_control=df_master_control, 
    column_name="auc_minus", 
    normalize=False, 
    sex='both', 
    vis_metric_type='combined3',
    side_by_side=True,
    df_master_original=df_master
)

# Creating A df_averaged data frame from df_master for the cross correlation and spatial analysis that comes down the line here 
<span style="font-size:44px; color:red;">Filter for SucOmit Is All ->Coded At The Bottom Of This Cell</span>

In [ ]:
import os
import numpy as np
import pandas as pd
import logging

#### Create df_averaged with distance columns

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

def ensure_dir(directory):
    """Ensure directory exists, create if it doesn't"""
    if not os.path.exists(directory):
        try:
            os.makedirs(directory)
            logger.info(f"Created directory: {directory}")
        except Exception as e:
            logger.error(f"Error creating directory {directory}: {str(e)}")
            raise

def perform_averaging(df_master, basedir, eventofinterest, separation_requirement, sucomit_filter='all'):
    """
    Create a new DataFrame by averaging all trials for each cell_id and cs_type, retaining metadata.
    Now includes distance columns.
    
    Parameters:
    -----------
    df_master : pandas.DataFrame
        The master dataframe with all trial data
    basedir : str
        Base directory for saving results
    eventofinterest : str
        Event of interest for analysis
    separation_requirement : int
        Separation requirement parameter
    sucomit_filter : str
        Filter option for day 2_SucOmit data:
        - 'all': include all trials (default)
        - 'rewarded': include only rewarded trials
        - 'unrewarded': include only unrewarded trials
    """
    try:
        logger.info(f"Starting averaging across trials with SucOmit filter: {sucomit_filter}")

        # Create a copy of df_master to avoid modifying the original
        df_for_averaging = df_master.copy()
        
        # Apply filtering for day 2_SucOmit CS+ trials if needed
        if sucomit_filter != 'all':
            # Count CS+ rows before filtering
            count_before = len(df_for_averaging[(df_for_averaging['day'] == '2_SucOmit') & 
                                               (df_for_averaging['cs_type'] == 'CS+')])
            
            # Apply filter for CS+ trials only
            if sucomit_filter in ['rewarded', 'unrewarded']:
                # Keep all non-2_SucOmit data
                non_sucomit_data = df_for_averaging[df_for_averaging['day'] != '2_SucOmit']
                
                # Keep all CS- trials for 2_SucOmit
                sucomit_cs_minus = df_for_averaging[
                    (df_for_averaging['day'] == '2_SucOmit') & 
                    (df_for_averaging['cs_type'] == 'CS-')
                ]
                
                # Filter CS+ trials for 2_SucOmit based on reward status
                sucomit_cs_plus = df_for_averaging[
                    (df_for_averaging['day'] == '2_SucOmit') & 
                    (df_for_averaging['cs_type'] == 'CS+') &
                    (df_for_averaging['reward_status'] == sucomit_filter)
                ]
                
                # Combine all the data
                df_for_averaging = pd.concat([non_sucomit_data, sucomit_cs_minus, sucomit_cs_plus])
                
                # Count CS+ rows after filtering
                count_after = len(df_for_averaging[(df_for_averaging['day'] == '2_SucOmit') & 
                                                  (df_for_averaging['cs_type'] == 'CS+')])
                logger.info(f"Filtered 2_SucOmit CS+ data: {count_before} trials before, {count_after} trials after filtering")
            else:
                logger.warning(f"Unrecognized filter option: {sucomit_filter}. Using all data.")

        # Get fluorescence columns
        fluor_plus_columns = [col for col in df_for_averaging.columns if col.startswith('fluor_plus_frame_')]
        fluor_minus_columns = [col for col in df_for_averaging.columns if col.startswith('fluor_minus_frame_')]

        # Check if auc_distance column exists
        has_distance = 'auc_distance' in df_for_averaging.columns
        
        # Create averaged DataFrame
        averaged_dfs = []
        
        for (cell_id, cs_type), cell_data in df_for_averaging.groupby(['cell_id', 'cs_type']):
            # Skip if no data (could happen after filtering)
            if len(cell_data) == 0:
                continue
                
            # Determine which fluorescence columns to average
            fluor_cols = fluor_plus_columns if cs_type == 'CS+' else fluor_minus_columns
            
            # Calculate mean for each fluorescence column
            avg_fluor_data = cell_data[fluor_cols].mean().to_dict()
            
            # Get metadata (use first row's values)
            first_row = cell_data.iloc[0]
            metadata = {
                'day': first_row['day'],
                'animal': first_row['animal'],
                'fov': first_row['fov'],
                'cell_id': cell_id,
                'cs_type': cs_type,
                'sex': first_row['sex'] if 'sex' in cell_data.columns else None
            }
            
            # Add cluster_id if it exists
            if 'cluster_id' in cell_data.columns:
                metadata['cluster_id'] = first_row['cluster_id']
            
            # Add AUC values and distance values
            if cs_type == 'CS+':
                metadata['auc_plus'] = first_row['auc_plus']
                if has_distance:
                    metadata['auc_plus_distance'] = cell_data['auc_distance'].mean()
            else:  # CS-
                metadata['auc_minus'] = first_row['auc_minus']
                if has_distance:
                    metadata['auc_minus_distance'] = cell_data['auc_distance'].mean()
            
            # Combine metadata and averaged fluorescence
            avg_fluor_data.update(metadata)
            averaged_dfs.append(avg_fluor_data)
        
        # Create df_averaged from the list of dictionaries
        df_averaged = pd.DataFrame(averaged_dfs)
        
        # Reorder columns to place metadata at the beginning
        metadata_columns = [col for col in df_averaged.columns if not col.startswith('fluor_')]
        fluor_columns = [col for col in df_averaged.columns if col.startswith('fluor_')]
        df_averaged = df_averaged[metadata_columns + fluor_columns]
        
        # Save results
        results_dir = os.path.join(basedir, 'Results')
        ensure_dir(results_dir)
        
        # Add filter info to filename if a filter was applied
        filter_suffix = f"_filtered_{sucomit_filter}" if sucomit_filter != 'all' else ""
        
        # Save the averaged DataFrame
        save_path_averaged = os.path.join(
            results_dir, 
            f'averaged_data_{eventofinterest}_{separation_requirement}{filter_suffix}.pkl'
        )
        df_averaged.to_pickle(save_path_averaged)
        
        save_path_averaged_csv = os.path.join(
            results_dir, 
            f'averaged_data_{eventofinterest}_{separation_requirement}{filter_suffix}.csv'
        )
        df_averaged.to_csv(save_path_averaged_csv, index=False)
        
        logger.info(f"Averaged DataFrame saved to: {save_path_averaged} and {save_path_averaged_csv}")
        
        return df_averaged
        
    except Exception as e:
        logger.error(f"Error in averaging: {str(e)}")
        raise

def collapse_cs_type(df_averaged):
    """ 
    Collapse CS+ and CS- data into a single row per cell_id, keeping non-NaN data for both CS+ and CS-.
    Now handles distance columns properly.
    """
    
    # Separate CS+ and CS- data
    df_plus = df_averaged[df_averaged['cs_type'] == 'CS+'].drop(columns=['cs_type'])
    df_minus = df_averaged[df_averaged['cs_type'] == 'CS-'].drop(columns=['cs_type'])
    
    # Make sure auc_plus_distance and auc_minus_distance are properly assigned
    if 'auc_plus_distance' in df_plus.columns and 'auc_distance' in df_plus.columns:
        df_plus = df_plus.drop(columns=['auc_distance'])
    
    if 'auc_minus_distance' in df_minus.columns and 'auc_distance' in df_minus.columns:
        df_minus = df_minus.drop(columns=['auc_distance'])
        
    # Rename columns in df_minus to match df_plus before merging to avoid suffixes
    common_cols = ['cell_id', 'day', 'animal', 'fov']  # These columns will be used to merge
    df_minus = df_minus.set_index(common_cols)
    df_plus = df_plus.set_index(common_cols)
    
    # Combine CS+ and CS- data, ensuring no suffixes are used
    df_averaged = df_plus.combine_first(df_minus).reset_index()

    return df_averaged

def save_and_inspect_dataframes(df_master, df_averaged, basedir, sucomit_filter='all'):
    # Create filter suffix for filenames
    filter_suffix = f"_filtered_{sucomit_filter}" if sucomit_filter != 'all' else ""
    
    # Create paths for saving the CSV files
    df_master_csv_path = os.path.join(basedir, 'Results', f'df_master_pre_clusters{filter_suffix}.csv')
    df_averaged_csv_path = os.path.join(basedir, 'Results', f'df_averaged_pre_clusters{filter_suffix}.csv')
    
    # Check the structure of df_master
    print("Master DataFrame Shape:", df_master.shape)
    print("\nSample of df_master:")
    display(df_master.head())
    
    # Check the structure of df_averaged
    print("Averaged DataFrame Shape:", df_averaged.shape)
    print("\nSample of df_averaged:")
    display(df_averaged.head())
    
    # Save files
    df_master.to_csv(df_master_csv_path, index=False)
    df_averaged.to_csv(df_averaged_csv_path, index=False)
    
    print(f"Saved DataFrame files with SucOmit filter: {sucomit_filter}")

# Usage (MAKE SURE THAT THE SUCOMIT FILTER IS WHAT YOU WANT IT TO BE HERE)
if __name__ == "__main__":
    try:
        # Define parameters
        params = {
            'basedir': basedir,
            'alpha': 0.05  # Set alpha level if needed for further analysis
        }
        
        # Run averaging with the specified filter
        df_averaged = perform_averaging(
            df_master=df_master,
            basedir=params['basedir'],
            eventofinterest=eventofinterest,
            separation_requirement=separation_requirement,
            sucomit_filter='all'  # Using the user-defined filter option
        )
        
        # Collapse CS+ and CS- data
        df_averaged = collapse_cs_type(df_averaged)
        
        # Save and inspect the results
        save_and_inspect_dataframes(df_master, df_averaged, params['basedir'], sucomit_filter='all')
        
        logger.info("Averaging completed successfully!")
        
    except Exception as e:
        logger.error(f"Script execution failed: {str(e)}")

# Cross Correlations
## This code takes about 10 minutes or so to run

In [ ]:
import pickle
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import mixedlm

# # define colors
# colors_tiedye = [
#     '#4B0082',   # Indigo/Deep Purple (highest - 1.0)
#     '#9932CC',  # Dark Orchid (medium purple)
#     '#2A788E',  # Teal (lowest - 0.0)
#     '#74D054',  # Yellow-green
#     '#FDE725',  # Yellow
#     '#FFA500',  # Orange
#     '#FF4500',  # Orange-red
#     '#FF1493'  # Deep Pink
# ]
# data = np.random.uniform(-1, 1, (10, 15))
# sns.set_style('white')
# # Create the colormap
# rainbow_cmap = LinearSegmentedColormap.from_list('Rainbow', colors_tiedye)

def print_statistical_results(plot_data, model_results, tukey_results):
    """Print statistical results in a clear format"""
    print("\nSTATISTICAL ANALYSES")
    print("=" * 50)
    
    # Main effects summary
    print("\n1. Mixed Model ANOVA Results:")
    print("-" * 30)
    
    fixed_effects = pd.DataFrame(model_results.fe_params)
    fixed_effects['P-Value'] = model_results.pvalues
    fixed_effects.columns = ['Estimate', 'P-Value']
    
    print("\nMain Effects:")
    print("-------------")
    print("\nEffect of Day:")
    print(fixed_effects.filter(like='day', axis=0))
    
    print("\nEffect of CS Type:")
    print(fixed_effects.filter(like='cs_type', axis=0))
    
    print("\nInteraction Effects (Day × CS Type):")
    print(fixed_effects.filter(like=':', axis=0))
    
    print("\n2. Post-hoc Comparisons (Tukey HSD):")
    print("-" * 30)
    print(tukey_results)
    
    print("\n3. Summary Statistics:")
    print("-" * 30)
    summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
    print(summary_stats)

def analyze_correlations(df_averaged):
    """
    Comprehensive correlation analysis function that combines visualization, 
    statistical analysis, and hierarchical clustering
    """
    print("Starting correlation analysis...")
    days = df_averaged['day'].unique()
    print(f"Processing {len(days)} days: {days}")
    
    all_results = []
    
    # Dictionary to store correlation data for all days
    correlation_data_to_save = {}
    
    for day in days:
        print(f"\nProcessing day: {day}")
        day_data = df_averaged[df_averaged['day'] == day]
        print(f"Number of cells for {day}: {len(day_data)}")
        
        # Extract CS+ and CS- data
        cs_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        cs_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        cs_plus_data = day_data[cs_plus_cols].to_numpy(dtype=float)
        cs_minus_data = day_data[cs_minus_cols].to_numpy(dtype=float)
        
        n_cells = cs_plus_data.shape[0]
        cs_plus_corr = np.zeros((n_cells, n_cells))
        cs_minus_corr = np.zeros((n_cells, n_cells))
        
        # Calculate correlation matrices
        for i in range(n_cells):
            # Set self-correlations to 1
            cs_plus_corr[i, i] = 1
            cs_minus_corr[i, i] = 1
            
            # Calculate correlations with other cells
            for j in range(i+1, n_cells):
                try:
                    cs_plus_corr[i, j], _ = pearsonr(cs_plus_data[i], cs_plus_data[j])
                    cs_minus_corr[i, j], _ = pearsonr(cs_minus_data[i], cs_minus_data[j])
                    cs_plus_corr[j, i] = cs_plus_corr[i, j]
                    cs_minus_corr[j, i] = cs_minus_corr[i, j]
                except ValueError as e:
                    print(f"Warning: Error calculating correlation for cells {i} and {j}: {str(e)}")
                    cs_plus_corr[i, j] = cs_plus_corr[j, i] = np.nan
                    cs_minus_corr[i, j] = cs_minus_corr[j, i] = np.nan

        # Store correlation data for this day
        correlation_data_to_save[day] = {
            'cs_plus_corr_matrix': cs_plus_corr,
            'cs_minus_corr_matrix': cs_minus_corr,
            'cell_ids': day_data['cell_id'].tolist(),
            'n_cells': n_cells
        }
        
        # Calculate average correlations for each cell
        cs_plus_avg = np.nanmean(cs_plus_corr, axis=1)
        cs_minus_avg = np.nanmean(cs_minus_corr, axis=1)
        
        # Store results
        for cell in range(n_cells):
            all_results.append({
                'day': day,
                'cell_id': day_data.iloc[cell]['cell_id'],
                'cs_plus_correlation': cs_plus_avg[cell],
                'cs_minus_correlation': cs_minus_avg[cell]
            })
        
        # Create heatmap visualization
        cs_plus_corr_viz = np.nan_to_num(cs_plus_corr, nan=0)
        cs_minus_corr_viz = np.nan_to_num(cs_minus_corr, nan=0)
        
        # Hierarchical clustering
        cs_plus_linkage = hierarchy.linkage(pdist(cs_plus_corr_viz), method='average')
        cs_minus_linkage = hierarchy.linkage(pdist(cs_minus_corr_viz), method='average')
        
        cs_plus_order = hierarchy.dendrogram(cs_plus_linkage, no_plot=True)['leaves']
        cs_minus_order = hierarchy.dendrogram(cs_minus_linkage, no_plot=True)['leaves']
        
        cs_plus_corr_sorted = cs_plus_corr_viz[cs_plus_order][:, cs_plus_order]
        cs_minus_corr_sorted = cs_minus_corr_viz[cs_minus_order][:, cs_minus_order]
        
        # Plot heatmaps
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
        
        sns.heatmap(cs_plus_corr_sorted, ax=ax1, cmap='PiYG_r', vmin=-1, vmax=1, center=0)
        ax1.set_title(f'Sorted CS+ Cross-correlation Heatmap - Day {day}')
        ax1.set_xlabel('Sorted Cell Index')
        ax1.set_ylabel('Sorted Cell Index')
        
        sns.heatmap(cs_minus_corr_sorted, ax=ax2, cmap='PiYG_r', vmin=-1, vmax=1, center=0)
        ax2.set_title(f'Sorted CS- Cross-correlation Heatmap - Day {day}')
        ax2.set_xlabel('Sorted Cell Index')
        ax2.set_ylabel('Sorted Cell Index')
        
        plt.tight_layout()
        # save_path = os.path.join(basedir, 'Results', 
        #                      f"All_cells_combined_fluro cross correlation_{day}_08-04-25.pdf")
        
        filename_base = f'All_cells_combined_fluro-cross-correlation_{day}_08-04-25'
        try:
            fig.savefig(os.path.join(basedir, 'Results', f'{filename_base}.pdf'))
            # fig.savefig(os.path.join(basedir, 'Results', f'{filename_base}.png'))
        except Exception as e:
            print(f"Error saving figure: {e}")
        plt.show()
        
    
    # Create results dataframe
    results_df = pd.DataFrame(all_results)
    print(f"\nCreated results for {len(results_df)} cell-day combinations")
    
    # Merge results back into df_averaged
    df_averaged_updated = pd.merge(
        df_averaged,
        results_df[['day', 'cell_id', 'cs_plus_correlation', 'cs_minus_correlation']],
        on=['day', 'cell_id'],
        how='left'
    )
    
    # Prepare data for visualization and statistics
    plot_data = pd.melt(results_df,
                       id_vars=['day', 'cell_id'],
                       value_vars=['cs_plus_correlation', 'cs_minus_correlation'],
                       var_name='cs_type',
                       value_name='correlation')
    plot_data['cs_type'] = plot_data['cs_type'].map({
        'cs_plus_correlation': 'CS+',
        'cs_minus_correlation': 'CS-'
    })
    
    # Create bar plot
    plt.figure(figsize=(10, 6))
    sns.barplot(x='day', y='correlation', hue='cs_type', data=plot_data,
                errorbar=('se', 1), capsize=0.1, palette='Set2')
    plt.title('Average Correlation by Day and CS Type')
    plt.xlabel('Day')
    plt.ylabel('Average Correlation')
    save_path = os.path.join(basedir, 'Results', 
                             f"All cells_combined_fluro cross correlation_{day}_bar_0-6-22-25.pdf")
    plt.show()
    
    # Statistical analysis
    model = mixedlm("correlation ~ C(day) + C(cs_type) + C(day):C(cs_type)",
                    data=plot_data, groups="cell_id")
    model_results = model.fit()
    
    tukey_results = pairwise_tukeyhsd(plot_data['correlation'],
                                     plot_data['day'] + ' ' + plot_data['cs_type'])
    
    # Print statistical results
    print_statistical_results(plot_data, model_results, tukey_results)
    
    # Print verification
    print("\nVerification of correlation values:")
    print(f"Number of cells with correlation values: {df_averaged_updated['cs_plus_correlation'].notna().sum()}")

    # Save all correlation data to pickle file after processing all days
    save_path = os.path.join(basedir, 'Results', 'real_trial_correlation_matrices.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(correlation_data_to_save, f)
    print(f"\nSaved real trial correlation matrices to: {save_path}")
    
    return df_averaged_updated, results_df, model_results, tukey_results

# Usage:
df_averaged, results_df, anova_results, posthoc_results = analyze_correlations(df_averaged)

# Split Graphs (for df_master and df_averaged)

# stats to report DF for manuscript

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from statsmodels.formula.api import mixedlm, ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import anova_lm

def analyze_day_groupings_complete(results_df):
    """
    Analyze and visualize correlation data with specific day groupings:
    1. Early vs Late (2×2 ANOVA: Day × CS type)
    2. Late vs Suc-Omit vs Cue-Omit (full mixed model)
    3. Rev-Early vs Rev-Late (2×2 ANOVA: Day × CS type)
    
    Uses the results DataFrame from the main correlation analysis.
    
    Parameters:
    -----------
    results_df : pandas DataFrame
        DataFrame containing correlation results from the original analysis,
        with 'day', 'cell_id', 'cs_plus_correlation', and 'cs_minus_correlation' columns.
    """
    
    # Define the three day groupings
    day_groupings = [
        ['0_Early', '1_Late'],
        ['2_SucOmit', '3_CueOmit'],
        ['4_RevEarly', '5_RevLate']
    ]
    
    # Names for each grouping (for plot titles and file names)
    grouping_names = [
        'Early_vs_Late',
        'SucOmit_vs_CueOmit',
        'RevEarly_vs_RevLate'
    ]
    
    print("\n" + "="*80)
    print("ANALYSIS BY DAY GROUPINGS")
    print("="*80)
    
    # Process each day grouping
    for group_idx, days in enumerate(day_groupings):
        group_name = grouping_names[group_idx]
        print(f"\n\nGroup {group_idx + 1}: {' vs '.join(days)}")
        print("-" * 50)
        
        # Check if all days exist in the data
        existing_days = set(results_df['day'].unique())
        missing_days = [day for day in days if day not in existing_days]
        if missing_days:
            print(f"Warning: The following days are not in the dataset: {missing_days}")
            print(f"Available days are: {existing_days}")
            print("Skipping this grouping.")
            continue
        
        # Filter data for the current day grouping
        filtered_data = results_df[results_df['day'].isin(days)].copy()
        
        # Count samples for each day to ensure there's enough data
        day_counts = filtered_data['day'].value_counts()
        print("\nSamples per day:")
        print(day_counts)
        
        # Prepare data for visualization and statistics
        plot_data = pd.melt(filtered_data,
                           id_vars=['day', 'cell_id'],
                           value_vars=['cs_plus_correlation', 'cs_minus_correlation'],
                           var_name='cs_type',
                           value_name='correlation')
        
        plot_data['cs_type'] = plot_data['cs_type'].map({
            'cs_plus_correlation': 'CS+',
            'cs_minus_correlation': 'CS-'
        })
        
        # Create summary statistics for plotting
        summary_data = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'sem']).reset_index()
        summary_data.columns = ['day', 'cs_type', 'mean', 'sem']
        
        # ================ VISUALIZATION SECTION ================
        
        # 1. Bar plot with error bars
        plt.figure(figsize=(10, 6))
        sns.barplot(x='day', y='correlation', hue='cs_type', data=plot_data,
                    errorbar=('se', 1), capsize=0.1, palette='Set2')
        plt.title(f'Average Correlation: {" vs ".join(days)}')
        plt.xlabel('Day')
        plt.ylabel('Average Correlation')
        
        # Try to set reasonable y-limits based on data
        y_min = 0
        y_max = 0.6
        plt.ylim(y_min, y_max)
        
        plt.tight_layout()
        
        # Save bar plot
        try:
            os.makedirs(os.path.join(basedir, 'Results'), exist_ok=True)
            plt.savefig(os.path.join(basedir, 'Results', f"All_cells_correlation_{group_name}_ANOVA.pdf"))
        except:
            pass  # Continue if save fails
        plt.show()
        
        # ================ STATISTICAL ANALYSIS SECTION ================
        
        print(f"\nStatistical analysis for {' vs '.join(days)}:")
        
        try:
            # For two-day comparisons, use a 2×2 ANOVA (Day × CS type)
            if len(days) == 2:
                print("Running 2×2 ANOVA (Day × CS type)")
                
                # Create a proper 2×2 ANOVA model using statsmodels
                formula = "correlation ~ C(day) * C(cs_type)"
                
                # Try mixed model first to account for repeated measures on same cells
                print("\nAttempting Mixed Effects Model (accounting for cell_id as random effect):")
                try:
                    model = mixedlm(formula, data=plot_data, groups="cell_id")
                    model_results = model.fit()
                    
                    # Print results in a clear format
                    print("\nMixed Effects Model Results:")
                    print(model_results.summary())
                    
                    # Extract coefficients and p-values for manuscript reporting
                    print("\nFixed Effects Summary for Manuscript:")
                    print("-" * 40)
                    coef_table = pd.DataFrame({
                        'Effect': model_results.fe_params.index,
                        'Coefficient': model_results.fe_params.values,
                        'p-value': model_results.pvalues.values
                    })
                    print(coef_table)
                    
                except Exception as e:
                    print(f"Mixed model failed: {str(e)}")
                    print("Falling back to standard ANOVA (without random effects):")
                    
                    # Standard ANOVA as fallback
                    model = ols(formula, data=plot_data).fit()
                    anova_table = anova_lm(model, typ=2)
                    print("\nANOVA Table:")
                    print(anova_table)
                    
                    # Extract F-statistics and degrees of freedom for manuscript reporting
                    print("\nManuscript Reporting Format:")
                    print("-" * 30)
                    
                    for effect in anova_table.index:
                        if effect != 'Residual':
                            f_val = anova_table.loc[effect, 'F']
                            p_val = anova_table.loc[effect, 'PR(>F)']
                            df_num = anova_table.loc[effect, 'df']
                            df_den = anova_table.loc['Residual', 'df']
                            
                            # Format p-value
                            if p_val < 0.001:
                                p_str = "p < 0.001"
                            else:
                                p_str = f"p = {p_val:.3f}"
                            
                            print(f"{effect}: F({df_num:.0f}, {df_den:.0f}) = {f_val:.3f}, {p_str}")
                
                # Run post-hoc tests
                print("\nPost-hoc Comparisons (Tukey HSD):")
                print("-" * 35)
                tukey_results = pairwise_tukeyhsd(plot_data['correlation'],
                                               plot_data['day'] + ' ' + plot_data['cs_type'])
                print(tukey_results)
                
                # Extract significant comparisons for easy interpretation
                print("\nSignificant Post-hoc Comparisons (p < 0.05):")
                print("-" * 45)
                significant_found = False
                for i, row in enumerate(tukey_results._results_table.data[1:]):
                    group1, group2, meandiff, p_adj, lower, upper, reject = row
                    if reject:  # reject is True when p < alpha
                        significant_found = True
                        print(f"{group1} vs {group2}: Mean diff = {meandiff:.4f}, p = {p_adj:.4f}")
                
                if not significant_found:
                    print("No significant pairwise differences found.")
                
                # Print summary statistics
                print("\nDescriptive Statistics:")
                print("-" * 25)
                summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
                print(summary_stats)
                
            else:
                # For 3+ days, use the full mixed model with interaction
                print("Running full model for multi-day comparison")
                
                try:
                    # Try mixed model first
                    model = mixedlm("correlation ~ C(day) * C(cs_type)", data=plot_data, groups="cell_id")
                    model_results = model.fit()
                    
                    print("\nMixed Effects Model Results:")
                    print(model_results.summary())
                    
                    # ADD THIS SECTION: Also run standard ANOVA for the F-table output
                    print("\nANOVA Table (for manuscript reporting):")
                    print("-" * 40)
                    anova_model = ols("correlation ~ C(day) * C(cs_type)", data=plot_data).fit()
                    anova_table = anova_lm(anova_model, typ=2)
                    print(anova_table)
                    
                    # Extract F-statistics and degrees of freedom for manuscript reporting
                    print("\nManuscript Reporting Format:")
                    print("-" * 30)
                    
                    for effect in anova_table.index:
                        if effect != 'Residual':
                            f_val = anova_table.loc[effect, 'F']
                            p_val = anova_table.loc[effect, 'PR(>F)']
                            df_num = anova_table.loc[effect, 'df']
                            df_den = anova_table.loc['Residual', 'df']
                            
                            # Format p-value
                            if p_val < 0.001:
                                p_str = "p < 0.001"
                            else:
                                p_str = f"p = {p_val:.3f}"
                            
                            print(f"{effect}: F({df_num:.0f}, {df_den:.0f}) = {f_val:.3f}, {p_str}")
                    
                except Exception as e:
                    print(f"Mixed model failed: {str(e)}")
                    print("Using standard ANOVA:")
                    
                    # Standard ANOVA as fallback
                    model = ols("correlation ~ C(day) * C(cs_type)", data=plot_data).fit()
                    anova_table = anova_lm(model, typ=2)
                    print("\nANOVA Table:")
                    print(anova_table)
                    
                    # Extract F-statistics and degrees of freedom for manuscript reporting
                    print("\nManuscript Reporting Format:")
                    print("-" * 30)
                    
                    for effect in anova_table.index:
                        if effect != 'Residual':
                            f_val = anova_table.loc[effect, 'F']
                            p_val = anova_table.loc[effect, 'PR(>F)']
                            df_num = anova_table.loc[effect, 'df']
                            df_den = anova_table.loc['Residual', 'df']
                            
                            # Format p-value
                            if p_val < 0.001:
                                p_str = "p < 0.001"
                            else:
                                p_str = f"p = {p_val:.3f}"
                            
                            print(f"{effect}: F({df_num:.0f}, {df_den:.0f}) = {f_val:.3f}, {p_str}")
                
                # Run Tukey's HSD post-hoc test
                print("\nPost-hoc Comparisons (Tukey HSD):")
                print("-" * 35)
                tukey_results = pairwise_tukeyhsd(plot_data['correlation'],
                                               plot_data['day'] + ' ' + plot_data['cs_type'])
                print(tukey_results)
                
                # Extract significant comparisons
                print("\nSignificant Post-hoc Comparisons (p < 0.05):")
                print("-" * 45)
                significant_found = False
                for i, row in enumerate(tukey_results._results_table.data[1:]):
                    group1, group2, meandiff, p_adj, lower, upper, reject = row
                    if reject:
                        significant_found = True
                        print(f"{group1} vs {group2}: Mean diff = {meandiff:.4f}, p = {p_adj:.4f}")
                
                if not significant_found:
                    print("No significant pairwise differences found.")
                
                # Print summary statistics
                print("\nDescriptive Statistics:")
                print("-" * 25)
                summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
                print(summary_stats)
                
        except Exception as e:
            print(f"Error during statistical analysis: {str(e)}")
            print("Falling back to basic summary statistics:")
            
            # Basic summary statistics
            summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
            print(summary_stats)

# Execute the grouped analysis using the results from the original analysis
# This assumes that results_df has already been created by the analyze_correlations function
analyze_day_groupings_complete(results_df)

# Creating df_averaged_control with distance metrics from df_master_control
<span style="font-size:44px; color:red;">Filter for SucOmit Is All ->Coded At The Bottom Of This Cell</span>

In [ ]:
import os
import numpy as np
import pandas as pd
import logging

#### Create df_averaged_control with distance columns

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

def ensure_dir(directory):
    """Ensure directory exists, create if it doesn't"""
    if not os.path.exists(directory):
        try:
            os.makedirs(directory)
            logger.info(f"Created directory: {directory}")
        except Exception as e:
            logger.error(f"Error creating directory {directory}: {str(e)}")
            raise

def perform_averaging_control(df_master_control, basedir, eventofinterest, separation_requirement, sucomit_filter='all'):
    """
    Create a new DataFrame by averaging all trials for each cell_id and cs_type, retaining metadata.
    Now includes distance columns.
    
    Parameters:
    -----------
    df_master_control : pandas.DataFrame
        The master dataframe with all trial data for control group
    basedir : str
        Base directory for saving results
    eventofinterest : str
        Event of interest for analysis
    separation_requirement : int
        Separation requirement parameter
    sucomit_filter : str
        Filter option for day 2_SucOmit data:
        - 'all': include all trials (default)
        - 'rewarded': include only rewarded trials
        - 'unrewarded': include only unrewarded trials
    """
    try:
        logger.info(f"Starting averaging across trials for control data with SucOmit filter: {sucomit_filter}")

        # Create a copy of df_master_control to avoid modifying the original
        df_for_averaging = df_master_control.copy()
        
        # Apply filtering for day 2_SucOmit CS+ trials if needed
        if sucomit_filter != 'all':
            # Count CS+ rows before filtering
            count_before = len(df_for_averaging[(df_for_averaging['day'] == '2_SucOmit') & 
                                               (df_for_averaging['cs_type'] == 'CS+')])
            
            # Apply filter for CS+ trials only
            if sucomit_filter in ['rewarded', 'unrewarded']:
                # Keep all non-2_SucOmit data
                non_sucomit_data = df_for_averaging[df_for_averaging['day'] != '2_SucOmit']
                
                # Keep all CS- trials for 2_SucOmit
                sucomit_cs_minus = df_for_averaging[
                    (df_for_averaging['day'] == '2_SucOmit') & 
                    (df_for_averaging['cs_type'] == 'CS-')
                ]
                
                # Filter CS+ trials for 2_SucOmit based on reward status
                sucomit_cs_plus = df_for_averaging[
                    (df_for_averaging['day'] == '2_SucOmit') & 
                    (df_for_averaging['cs_type'] == 'CS+') &
                    (df_for_averaging['reward_status'] == sucomit_filter)
                ]
                
                # Combine all the data
                df_for_averaging = pd.concat([non_sucomit_data, sucomit_cs_minus, sucomit_cs_plus])
                
                # Count CS+ rows after filtering
                count_after = len(df_for_averaging[(df_for_averaging['day'] == '2_SucOmit') & 
                                                  (df_for_averaging['cs_type'] == 'CS+')])
                logger.info(f"Filtered 2_SucOmit CS+ data: {count_before} trials before, {count_after} trials after filtering")
            else:
                logger.warning(f"Unrecognized filter option: {sucomit_filter}. Using all data.")

        # Get fluorescence columns
        fluor_plus_columns = [col for col in df_for_averaging.columns if col.startswith('fluor_plus_frame_')]
        fluor_minus_columns = [col for col in df_for_averaging.columns if col.startswith('fluor_minus_frame_')]

        # Check if auc_distance column exists
        has_distance = 'auc_distance' in df_for_averaging.columns
        
        # Create averaged DataFrame
        averaged_dfs = []
        
        for (cell_id, cs_type), cell_data in df_for_averaging.groupby(['cell_id', 'cs_type']):
            # Skip if no data (could happen after filtering)
            if len(cell_data) == 0:
                continue
                
            # Determine which fluorescence columns to average
            fluor_cols = fluor_plus_columns if cs_type == 'CS+' else fluor_minus_columns
            
            # Calculate mean for each fluorescence column
            avg_fluor_data = cell_data[fluor_cols].mean().to_dict()
            
            # Get metadata (use first row's values)
            first_row = cell_data.iloc[0]
            metadata = {
                'day': first_row['day'],
                'animal': first_row['animal'],
                'fov': first_row['fov'],
                'cell_id': cell_id,
                'cs_type': cs_type,
                'sex': first_row['sex'] if 'sex' in cell_data.columns else None
            }
            
            # Add cluster_id if it exists
            if 'cluster_id' in cell_data.columns:
                metadata['cluster_id'] = first_row['cluster_id']
            
            # Add AUC values and distance values
            if cs_type == 'CS+':
                metadata['auc_plus'] = first_row['auc_plus']
                if has_distance:
                    metadata['auc_plus_distance'] = cell_data['auc_distance'].mean()
            else:  # CS-
                metadata['auc_minus'] = first_row['auc_minus']
                if has_distance:
                    metadata['auc_minus_distance'] = cell_data['auc_distance'].mean()
            
            # Combine metadata and averaged fluorescence
            avg_fluor_data.update(metadata)
            averaged_dfs.append(avg_fluor_data)
        
        # Create df_averaged_control from the list of dictionaries
        df_averaged_control = pd.DataFrame(averaged_dfs)
        
        # Reorder columns to place metadata at the beginning
        metadata_columns = [col for col in df_averaged_control.columns if not col.startswith('fluor_')]
        fluor_columns = [col for col in df_averaged_control.columns if col.startswith('fluor_')]
        df_averaged_control = df_averaged_control[metadata_columns + fluor_columns]
        
        # Save results
        results_dir = os.path.join(basedir, 'Results')
        ensure_dir(results_dir)
        
        # Add filter info to filename if a filter was applied
        filter_suffix = f"_filtered_{sucomit_filter}" if sucomit_filter != 'all' else ""
        
        # Save the averaged DataFrame
        save_path_averaged = os.path.join(
            results_dir, 
            f'averaged_data_control_{eventofinterest}_{separation_requirement}{filter_suffix}.pkl'
        )
        df_averaged_control.to_pickle(save_path_averaged)
        
        save_path_averaged_csv = os.path.join(
            results_dir, 
            f'averaged_data_control_{eventofinterest}_{separation_requirement}{filter_suffix}.csv'
        )
        df_averaged_control.to_csv(save_path_averaged_csv, index=False)
        
        logger.info(f"Averaged Control DataFrame saved to: {save_path_averaged} and {save_path_averaged_csv}")
        
        return df_averaged_control
        
    except Exception as e:
        logger.error(f"Error in averaging control data: {str(e)}")
        raise

def collapse_cs_type_control(df_averaged_control):
    """ 
    Collapse CS+ and CS- data into a single row per cell_id, keeping non-NaN data for both CS+ and CS-.
    Now handles distance columns properly.
    """
    
    # Separate CS+ and CS- data
    df_plus = df_averaged_control[df_averaged_control['cs_type'] == 'CS+'].drop(columns=['cs_type'])
    df_minus = df_averaged_control[df_averaged_control['cs_type'] == 'CS-'].drop(columns=['cs_type'])
    
    # Make sure auc_plus_distance and auc_minus_distance are properly assigned
    if 'auc_plus_distance' in df_plus.columns and 'auc_distance' in df_plus.columns:
        df_plus = df_plus.drop(columns=['auc_distance'])
    
    if 'auc_minus_distance' in df_minus.columns and 'auc_distance' in df_minus.columns:
        df_minus = df_minus.drop(columns=['auc_distance'])
        
    # Rename columns in df_minus to match df_plus before merging to avoid suffixes
    common_cols = ['cell_id', 'day', 'animal', 'fov']  # These columns will be used to merge
    df_minus = df_minus.set_index(common_cols)
    df_plus = df_plus.set_index(common_cols)
    
    # Combine CS+ and CS- data, ensuring no suffixes are used
    df_averaged_control = df_plus.combine_first(df_minus).reset_index()

    return df_averaged_control

def save_and_inspect_control_dataframes(df_master_control, df_averaged_control, basedir, sucomit_filter='all'):
    # Create filter suffix for filenames
    filter_suffix = f"_filtered_{sucomit_filter}" if sucomit_filter != 'all' else ""
    
    # Create paths for saving the CSV files
    df_master_csv_path = os.path.join(basedir, 'Results', f'df_master_control_pre_clusters{filter_suffix}.csv')
    df_averaged_csv_path = os.path.join(basedir, 'Results', f'df_averaged_control_pre_clusters{filter_suffix}.csv')
    
    # Check the structure of df_master_control
    print("Master Control DataFrame Shape:", df_master_control.shape)
    print("\nSample of df_master_control:")
    display(df_master_control.head())
    
    # Check the structure of df_averaged_control
    print("Averaged Control DataFrame Shape:", df_averaged_control.shape)
    print("\nSample of df_averaged_control:")
    display(df_averaged_control.head())
    
    # Save files
    df_master_control.to_csv(df_master_csv_path, index=False)
    df_averaged_control.to_csv(df_averaged_csv_path, index=False)
    
    print(f"Saved Control DataFrame files with SucOmit filter: {sucomit_filter}")

# Usage
if __name__ == "__main__":
    try:
        # Define parameters
        params = {
            'basedir': basedir,
            'alpha': 0.05  # Set alpha level if needed for further analysis
        }
        
        # Explicitly set sucomit_filter to 'all'
        sucomit_filter = 'all'
        
        # Run averaging with the 'all' filter
        df_averaged_control = perform_averaging_control(
            df_master_control=df_master_control,
            basedir=params['basedir'],
            eventofinterest=eventofinterest,
            separation_requirement=separation_requirement,
            sucomit_filter=sucomit_filter  # Using 'all' for SucOmit filter
        )
        
        # Collapse CS+ and CS- data
        df_averaged_control = collapse_cs_type_control(df_averaged_control)
        
        # Save and inspect the results
        save_and_inspect_control_dataframes(df_master_control, df_averaged_control, params['basedir'], sucomit_filter)
        
        logger.info("Averaging control data completed successfully!")
        
    except Exception as e:
        logger.error(f"Script execution failed: {str(e)}")

# Cross Correlation code for df_averaged_control (Control Data)

In [ ]:
import pickle
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import mixedlm

def print_statistical_results_control(plot_data, model_results, tukey_results):
    """Print statistical results for control data in a clear format"""
    print("\nSTATISTICAL ANALYSES - CONTROL DATA")
    print("=" * 60)
    
    # Main effects summary
    print("\n1. Mixed Model ANOVA Results (Control Data):")
    print("-" * 40)
    
    fixed_effects = pd.DataFrame(model_results.fe_params)
    fixed_effects['P-Value'] = model_results.pvalues
    fixed_effects.columns = ['Estimate', 'P-Value']
    
    print("\nMain Effects:")
    print("-------------")
    print("\nEffect of Day:")
    print(fixed_effects.filter(like='day', axis=0))
    
    print("\nEffect of CS Type:")
    print(fixed_effects.filter(like='cs_type', axis=0))
    
    print("\nInteraction Effects (Day × CS Type):")
    print(fixed_effects.filter(like=':', axis=0))
    
    print("\n2. Post-hoc Comparisons (Tukey HSD):")
    print("-" * 40)
    print(tukey_results)
    
    print("\n3. Summary Statistics:")
    print("-" * 40)
    summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
    print(summary_stats)

def analyze_correlations_control(df_averaged_control):
    """
    Comprehensive correlation analysis function for control data that combines visualization, 
    statistical analysis, and hierarchical clustering
    """
    print("Starting correlation analysis for control data...")
    days = df_averaged_control['day'].unique()
    print(f"Processing {len(days)} days: {days}")
    
    all_results = []
    
    # Dictionary to store correlation data for all days
    correlation_data_to_save = {}
    
    for day in days:
        print(f"\nProcessing day: {day}")
        day_data = df_averaged_control[df_averaged_control['day'] == day]
        print(f"Number of cells for {day}: {len(day_data)}")
        
        # Extract CS+ and CS- data from control frames
        cs_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        cs_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        # Check if we have data for both CS types
        if not cs_plus_cols or not cs_minus_cols:
            print(f"Warning: Missing data for one or both CS types in day {day}")
            continue
            
        # Filter out rows with all NaN values in the relevant columns
        valid_plus_mask = ~day_data[cs_plus_cols].isna().all(axis=1)
        valid_minus_mask = ~day_data[cs_minus_cols].isna().all(axis=1)
        valid_both_mask = valid_plus_mask & valid_minus_mask
        
        day_data_filtered = day_data[valid_both_mask].reset_index(drop=True)
        
        if len(day_data_filtered) == 0:
            print(f"Warning: No valid data for day {day} after filtering")
            continue
            
        print(f"Number of cells after filtering: {len(day_data_filtered)}")
        
        cs_plus_data = day_data_filtered[cs_plus_cols].to_numpy(dtype=float)
        cs_minus_data = day_data_filtered[cs_minus_cols].to_numpy(dtype=float)
        
        n_cells = cs_plus_data.shape[0]
        cs_plus_corr = np.zeros((n_cells, n_cells))
        cs_minus_corr = np.zeros((n_cells, n_cells))
        
        # Calculate correlation matrices
        for i in range(n_cells):
            # Set self-correlations to 1
            cs_plus_corr[i, i] = 1
            cs_minus_corr[i, i] = 1
            
            # Calculate correlations with other cells
            for j in range(i+1, n_cells):
                try:
                    cs_plus_corr[i, j], _ = pearsonr(
                        cs_plus_data[i, ~np.isnan(cs_plus_data[i]) & ~np.isnan(cs_plus_data[j])], 
                        cs_plus_data[j, ~np.isnan(cs_plus_data[i]) & ~np.isnan(cs_plus_data[j])]
                    )
                    cs_minus_corr[i, j], _ = pearsonr(
                        cs_minus_data[i, ~np.isnan(cs_minus_data[i]) & ~np.isnan(cs_minus_data[j])], 
                        cs_minus_data[j, ~np.isnan(cs_minus_data[i]) & ~np.isnan(cs_minus_data[j])]
                    )
                    cs_plus_corr[j, i] = cs_plus_corr[i, j]
                    cs_minus_corr[j, i] = cs_minus_corr[i, j]
                except ValueError as e:
                    print(f"Warning: Error calculating correlation for cells {i} and {j}: {str(e)}")
                    cs_plus_corr[i, j] = cs_plus_corr[j, i] = np.nan
                    cs_minus_corr[i, j] = cs_minus_corr[j, i] = np.nan

        # After calculating cs_plus_corr and cs_minus_corr matrices, save them
        # Make sure to use day_data_filtered if you filtered the data
        if len(day_data_filtered) > 0:  # Only save if we have valid data
            correlation_data_to_save[day] = {
                'cs_plus_corr_matrix': cs_plus_corr,
                'cs_minus_corr_matrix': cs_minus_corr,
                'cell_ids': day_data_filtered['cell_id'].tolist(),  # Use filtered data
                'n_cells': n_cells
            }
            
        # Calculate average correlations for each cell
        cs_plus_avg = np.nanmean(cs_plus_corr, axis=1)
        cs_minus_avg = np.nanmean(cs_minus_corr, axis=1)
        
        # Store results
        for cell in range(n_cells):
            all_results.append({
                'day': day,
                'cell_id': day_data_filtered.iloc[cell]['cell_id'],
                'cs_plus_correlation_control': cs_plus_avg[cell],
                'cs_minus_correlation_control': cs_minus_avg[cell]
            })
        
        # Create heatmap visualization
        cs_plus_corr_viz = np.nan_to_num(cs_plus_corr, nan=0)
        cs_minus_corr_viz = np.nan_to_num(cs_minus_corr, nan=0)
        
        # Hierarchical clustering
        try:
            cs_plus_linkage = hierarchy.linkage(pdist(cs_plus_corr_viz), method='average')
            cs_minus_linkage = hierarchy.linkage(pdist(cs_minus_corr_viz), method='average')
            
            cs_plus_order = hierarchy.dendrogram(cs_plus_linkage, no_plot=True)['leaves']
            cs_minus_order = hierarchy.dendrogram(cs_minus_linkage, no_plot=True)['leaves']
            
            cs_plus_corr_sorted = cs_plus_corr_viz[cs_plus_order][:, cs_plus_order]
            cs_minus_corr_sorted = cs_minus_corr_viz[cs_minus_order][:, cs_minus_order]
            
            # Plot heatmaps
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
            
            sns.heatmap(cs_plus_corr_sorted, ax=ax1, cmap='PiYG_r', vmin=-1, vmax=1, center=0)
            ax1.set_title(f'Control Data: Sorted CS+ Cross-correlation Heatmap - Day {day}')
            ax1.set_xlabel('Sorted Cell Index')
            ax1.set_ylabel('Sorted Cell Index')
            
            sns.heatmap(cs_minus_corr_sorted, ax=ax2, cmap='PiYG_r', vmin=-1, vmax=1, center=0)
            ax2.set_title(f'Control Data: Sorted CS- Cross-correlation Heatmap - Day {day}')
            ax2.set_xlabel('Sorted Cell Index')
            ax2.set_ylabel('Sorted Cell Index')
            
            plt.tight_layout()
            save_path = os.path.join(basedir, 'Results', 
                                 f"CONTROL_cells_combined_fluro_cross_correlation_{day}_{pd.Timestamp.now().strftime('%m-%d-%Y')}_11-03-25.pdf")
            plt.savefig(save_path)
            plt.show()
            print(f"Saved heatmap to {save_path}")
        except Exception as e:
            print(f"Warning: Could not create heatmap for day {day}: {str(e)}")
    
    # Create results dataframe
    results_df = pd.DataFrame(all_results)
    print(f"\nCreated results for {len(results_df)} cell-day combinations")
    
    # Merge results back into df_averaged_control
    df_averaged_control_updated = pd.merge(
        df_averaged_control,
        results_df[['day', 'cell_id', 'cs_plus_correlation_control', 'cs_minus_correlation_control']],
        on=['day', 'cell_id'],
        how='left'
    )
    
    # Prepare data for visualization and statistics
    plot_data = pd.melt(results_df,
                       id_vars=['day', 'cell_id'],
                       value_vars=['cs_plus_correlation_control', 'cs_minus_correlation_control'],
                       var_name='cs_type',
                       value_name='correlation')
    plot_data['cs_type'] = plot_data['cs_type'].map({
        'cs_plus_correlation_control': 'CS+',
        'cs_minus_correlation_control': 'CS-'
    })
    
    # Create bar plot
    plt.figure(figsize=(10, 6))
    sns.barplot(x='day', y='correlation', hue='cs_type', data=plot_data,
                errorbar=('se', 1), capsize=0.1, palette='Set2')
    plt.title('CONTROL Data: Average Correlation by Day and CS Type')
    plt.xlabel('Day')
    plt.ylabel('Average Correlation')
    save_path = os.path.join(basedir, 'Results', 
                             f"CONTROL_cells_combined_fluro_cross_correlation_bar_{pd.Timestamp.now().strftime('%m-%d-%Y')}_11-03-25.pdf")
    plt.savefig(save_path)
    plt.show()
    print(f"Saved bar plot to {save_path}")
    
    # Statistical analysis
    try:
        model = mixedlm("correlation ~ C(day) + C(cs_type) + C(day):C(cs_type)",
                        data=plot_data, groups="cell_id")
        model_results = model.fit()
        
        tukey_results = pairwise_tukeyhsd(plot_data['correlation'],
                                         plot_data['day'] + ' ' + plot_data['cs_type'])
        
        # Print statistical results
        print_statistical_results_control(plot_data, model_results, tukey_results)
    except Exception as e:
        print(f"Warning: Statistical analysis failed: {str(e)}")
        model_results = None
        tukey_results = None
    
    # Print verification
    print("\nVerification of correlation values (Control Data):")
    print(f"Number of cells with CS+ correlation values: {df_averaged_control_updated['cs_plus_correlation_control'].notna().sum()}")
    print(f"Number of cells with CS- correlation values: {df_averaged_control_updated['cs_minus_correlation_control'].notna().sum()}")
    
    # Save results
    results_save_path = os.path.join(basedir, 'Results', 
                                    f"control_correlation_results_{pd.Timestamp.now().strftime('%m-%d-%Y')}.csv")
    results_df.to_csv(results_save_path, index=False)
    print(f"Saved correlation results to {results_save_path}")

    # Save all correlation data to pickle file after processing all days
    save_path = os.path.join(basedir, 'Results', 'baseline_correlation_matrices.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(correlation_data_to_save, f)
    print(f"\nSaved baseline correlation matrices to: {save_path}")
    
    return df_averaged_control_updated, results_df, model_results, tukey_results

# Usage:
df_averaged_control, results_df_control, anova_results_control, posthoc_results_control = analyze_correlations_control(df_averaged_control)

In [ ]:
####Asymmetrical correlation heatmaps to include real and baseline data in one plot
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
import os

import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
import os

def investigate_correlation_values(basedir, day_to_plot='0_Early', cs_type='CS+'):
    """
    Investigate the actual correlation values in both matrices
    """
    # Load pickle files
    real_path = os.path.join(basedir, 'Results', 'real_trial_correlation_matrices.pkl')
    baseline_path = os.path.join(basedir, 'Results', 'baseline_correlation_matrices.pkl')
    
    with open(real_path, 'rb') as f:
        real_data = pickle.load(f)
    with open(baseline_path, 'rb') as f:
        baseline_data = pickle.load(f)
    
    # Extract data
    real_day = real_data[day_to_plot]
    baseline_day = baseline_data[day_to_plot]
    
    if cs_type == 'CS+':
        real_corr = real_day['cs_plus_corr_matrix']
        baseline_corr = baseline_day['cs_plus_corr_matrix']
    else:
        real_corr = real_day['cs_minus_corr_matrix']
        baseline_corr = baseline_day['cs_minus_corr_matrix']
    
    # Get off-diagonal values (excluding self-correlations)
    n = real_corr.shape[0]
    mask = ~np.eye(n, dtype=bool)
    
    real_values = real_corr[mask]
    baseline_values = baseline_corr[mask]
    
    print(f"\nCorrelation Value Statistics for {day_to_plot} - {cs_type}")
    print("="*60)
    print(f"Real Trial:")
    print(f"  Mean: {np.nanmean(real_values):.4f}")
    print(f"  Std: {np.nanstd(real_values):.4f}")
    print(f"  Min: {np.nanmin(real_values):.4f}")
    print(f"  Max: {np.nanmax(real_values):.4f}")
    print(f"  Median: {np.nanmedian(real_values):.4f}")
    
    print(f"\nBaseline:")
    print(f"  Mean: {np.nanmean(baseline_values):.4f}")
    print(f"  Std: {np.nanstd(baseline_values):.4f}")
    print(f"  Min: {np.nanmin(baseline_values):.4f}")
    print(f"  Max: {np.nanmax(baseline_values):.4f}")
    print(f"  Median: {np.nanmedian(baseline_values):.4f}")
    
    # Plot distributions
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histograms
    axes[0].hist(real_values[~np.isnan(real_values)], bins=50, alpha=0.7, color='purple', label='Real Trial')
    axes[0].hist(baseline_values[~np.isnan(baseline_values)], bins=50, alpha=0.7, color='green', label='Baseline')
    axes[0].set_xlabel('Correlation Value')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Correlation Values')
    axes[0].legend()
    axes[0].axvline(0, color='black', linestyle='--', alpha=0.3)
    
    # Direct comparison scatter
    sample_indices = np.random.choice(len(real_values), min(5000, len(real_values)), replace=False)
    axes[1].scatter(baseline_values[sample_indices], real_values[sample_indices], 
                   alpha=0.1, s=1)
    axes[1].set_xlabel('Baseline Correlation')
    axes[1].set_ylabel('Real Trial Correlation')
    axes[1].set_title('Real vs Baseline Correlations (sample of cell pairs)')
    axes[1].plot([-1, 1], [-1, 1], 'r--', alpha=0.5, label='y=x')
    axes[1].set_xlim(-0.5, 1)
    axes[1].set_ylim(-0.5, 1)
    axes[1].axhline(0, color='black', linestyle='--', alpha=0.3)
    axes[1].axvline(0, color='black', linestyle='--', alpha=0.3)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    return real_corr, baseline_corr

def plot_combined_actual_values(basedir, day_to_plot='0_Early', cs_type='CS+',
                               clustering_source='real'):
    """
    Create combined plot with actual values - no enhancement or manipulation
    """
    # Load data
    real_path = os.path.join(basedir, 'Results', 'real_trial_correlation_matrices.pkl')
    baseline_path = os.path.join(basedir, 'Results', 'baseline_correlation_matrices.pkl')
    
    with open(real_path, 'rb') as f:
        real_data = pickle.load(f)
    with open(baseline_path, 'rb') as f:
        baseline_data = pickle.load(f)
    
    real_day = real_data[day_to_plot]
    baseline_day = baseline_data[day_to_plot]
    
    if cs_type == 'CS+':
        real_corr = real_day['cs_plus_corr_matrix'].copy()
        baseline_corr = baseline_day['cs_plus_corr_matrix'].copy()
    else:
        real_corr = real_day['cs_minus_corr_matrix'].copy()
        baseline_corr = baseline_day['cs_minus_corr_matrix'].copy()
    
    n_cells = real_corr.shape[0]
    print(f"Processing {n_cells} cells")
    
    # Clean data (only replacing NaN, no other manipulation)
    real_corr_clean = np.nan_to_num(real_corr, nan=0)
    baseline_corr_clean = np.nan_to_num(baseline_corr, nan=0)
    
    # Get clustering order based on selected source
    if clustering_source == 'real':
        linkage = hierarchy.linkage(pdist(real_corr_clean), method='average')
        order = hierarchy.dendrogram(linkage, no_plot=True)['leaves']
    elif clustering_source == 'baseline':
        linkage = hierarchy.linkage(pdist(baseline_corr_clean), method='average')
        order = hierarchy.dendrogram(linkage, no_plot=True)['leaves']
    else:  # 'none'
        order = np.arange(n_cells)
    
    # Apply ordering to individual matrices
    real_sorted = real_corr_clean[order][:, order]
    baseline_sorted = baseline_corr_clean[order][:, order]
    
    # Create combined matrix with ACTUAL values
    combined = np.zeros((n_cells, n_cells))
    for i in range(n_cells):
        for j in range(n_cells):
            if i == j:
                combined[i, j] = 1.0  # Diagonal
            elif i < j:
                combined[i, j] = real_sorted[i, j]  # Upper triangle - Real
            else:
                combined[i, j] = baseline_sorted[i, j]  # Lower triangle - Baseline
    
    # Create figure showing all three matrices side by side
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Real trial only
    im1 = axes[0].imshow(real_sorted, cmap='PiYG_r', vmin=-1, vmax=1, aspect='auto')
    axes[0].set_title(f'Real Trial Only\n{cs_type}')
    axes[0].set_xlabel('Cell Index')
    axes[0].set_ylabel('Cell Index')
    
    # Baseline only  
    im2 = axes[1].imshow(baseline_sorted, cmap='PiYG_r', vmin=-1, vmax=1, aspect='auto')
    axes[1].set_title(f'Baseline Only\n{cs_type}')
    axes[1].set_xlabel('Cell Index')
    
    # Combined
    im3 = axes[2].imshow(combined, cmap='PiYG_r', vmin=-1, vmax=1, aspect='auto')
    axes[2].plot([0, n_cells], [0, n_cells], 'k-', linewidth=2, alpha=0.5)
    axes[2].set_title(f'Combined Matrix\nUpper: Real | Lower: Baseline')
    axes[2].set_xlabel('Cell Index')
    
    # Add text labels
    axes[2].text(n_cells * 0.75, n_cells * 0.25, 'Real', 
                fontsize=12, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
    axes[2].text(n_cells * 0.25, n_cells * 0.75, 'Baseline', 
                fontsize=12, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
    
    # Difference (Real - Baseline)
    diff = real_sorted - baseline_sorted
    im4 = axes[3].imshow(diff, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    axes[3].set_title(f'Difference\n(Real - Baseline)')
    axes[3].set_xlabel('Cell Index')
    
    # Add colorbars
    plt.colorbar(im3, ax=axes[:3], label='Correlation', shrink=0.8)
    plt.colorbar(im4, ax=axes[3], label='Difference', shrink=0.8)
    
    plt.suptitle(f'{day_to_plot} - {cs_type} (Clustering: {clustering_source})', fontsize=14)
    plt.tight_layout()
    
    # Save
    save_name = f'Combined_ActualValues_{day_to_plot}_{cs_type.replace("+","plus")}_{clustering_source}_11-04-25.pdf'
    plt.savefig(os.path.join(basedir, 'Results', save_name), dpi=300, bbox_inches='tight')
    plt.show()
    
    # Verify the values in the combined matrix
    print("\n" + "="*60)
    print("Verification of Combined Matrix Values:")
    print("-"*60)
    
    # Check upper triangle (should match real)
    upper_mask = np.triu_indices(n_cells, k=1)
    upper_combined = combined[upper_mask]
    upper_real = real_sorted[upper_mask]
    upper_match = np.allclose(upper_combined, upper_real, rtol=1e-10)
    
    # Check lower triangle (should match baseline)
    lower_mask = np.tril_indices(n_cells, k=-1)
    lower_combined = combined[lower_mask]
    lower_baseline = baseline_sorted[lower_mask]
    lower_match = np.allclose(lower_combined, lower_baseline, rtol=1e-10)
    
    print(f"Upper triangle matches real data: {upper_match}")
    print(f"Lower triangle matches baseline data: {lower_match}")
    
    print(f"\nUpper triangle (Real) stats:")
    print(f"  Mean: {np.mean(upper_combined):.4f}")
    print(f"  Std: {np.std(upper_combined):.4f}")
    print(f"  Range: [{np.min(upper_combined):.4f}, {np.max(upper_combined):.4f}]")
    
    print(f"\nLower triangle (Baseline) stats:")
    print(f"  Mean: {np.mean(lower_combined):.4f}")
    print(f"  Std: {np.std(lower_combined):.4f}")
    print(f"  Range: [{np.min(lower_combined):.4f}, {np.max(lower_combined):.4f}]")
    
    return combined, real_sorted, baseline_sorted

# First investigate the actual values
real_corr, baseline_corr = investigate_correlation_values(basedir, '0_Early', 'CS-')

# Then create the combined plot with actual values
combined, real_sorted, baseline_sorted = plot_combined_actual_values(basedir, '0_Early', 'CS-', 
                                                                    clustering_source='real')

# Correlation Analysis of Control Data 

# with DFs in the ANOVA table

In [ ]:
def analyze_day_groupings_control(results_df_control):
    """
    Analyze and visualize control correlation data with specific day groupings:
    1. Early vs Late (2×2 ANOVA: Day × CS type)
    2. Late vs Suc-Omit vs Cue-Omit (full mixed model)
    3. Rev-Early vs Rev-Late (2×2 ANOVA: Day × CS type)
    
    Uses the results DataFrame from the control correlation analysis.
    
    Parameters:
    -----------
    results_df_control : pandas DataFrame
        DataFrame containing correlation results from the control analysis,
        with 'day', 'cell_id', 'cs_plus_correlation_control', and 'cs_minus_correlation_control' columns.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np
    import os
    from statsmodels.formula.api import mixedlm, ols
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    from statsmodels.stats.anova import anova_lm
    
    # Define the three day groupings
    day_groupings = [
        ['0_Early', '1_Late'],
        ['2_SucOmit', '3_CueOmit'],
        ['4_RevEarly', '5_RevLate']
    ]
    
    # Names for each grouping (for plot titles and file names)
    grouping_names = [
        'Early_vs_Late',
        'SucOmit_vs_CueOmit',
        'RevEarly_vs_RevLate'
    ]
    
    print("\n" + "="*80)
    print("CONTROL DATA ANALYSIS BY DAY GROUPINGS")
    print("="*80)
    
    # Process each day grouping
    for group_idx, days in enumerate(day_groupings):
        group_name = grouping_names[group_idx]
        print(f"\n\nCONTROL Group {group_idx + 1}: {' vs '.join(days)}")
        print("-" * 50)
        
        # Check if all days exist in the data
        existing_days = set(results_df_control['day'].unique())
        missing_days = [day for day in days if day not in existing_days]
        if missing_days:
            print(f"Warning: The following days are not in the control dataset: {missing_days}")
            print(f"Available days are: {existing_days}")
            print("Skipping this grouping for control data.")
            continue
        
        # Filter data for the current day grouping
        filtered_data = results_df_control[results_df_control['day'].isin(days)].copy()
        
        # Count samples for each day to ensure there's enough data
        day_counts = filtered_data['day'].value_counts()
        print("\nSamples per day in control data:")
        print(day_counts)
        
        # Prepare data for visualization and statistics
        plot_data = pd.melt(filtered_data,
                           id_vars=['day', 'cell_id'],
                           value_vars=['cs_plus_correlation_control', 'cs_minus_correlation_control'],
                           var_name='cs_type',
                           value_name='correlation')
        
        plot_data['cs_type'] = plot_data['cs_type'].map({
            'cs_plus_correlation_control': 'CS+',
            'cs_minus_correlation_control': 'CS-'
        })
        
        # Create summary statistics for plotting
        summary_data = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'sem']).reset_index()
        summary_data.columns = ['day', 'cs_type', 'mean', 'sem']
        
        # ================ VISUALIZATION SECTION ================
        
        # 1. Bar plot with error bars
        plt.figure(figsize=(10, 6))
        sns.barplot(x='day', y='correlation', hue='cs_type', data=plot_data,
                    errorbar=('se', 1), capsize=0.1, palette='Set2')
        plt.title(f'CONTROL: Average Correlation: {" vs ".join(days)}')
        plt.xlabel('Day')
        plt.ylabel('Average Correlation')
        
        # Try to set reasonable y-limits based on data
        y_min = 0
        y_max = 0.6
        plt.ylim(y_min, y_max)
        
        plt.tight_layout()
        
        # Save bar plot
        try:
            timestamp = pd.Timestamp.now().strftime('%Y-%m-%d')
            os.makedirs(os.path.join(basedir, 'Results'), exist_ok=True)
            save_path = os.path.join(basedir, 'Results', f"CONTROL_cells_correlation_{group_name}_{timestamp}_11-04-25.pdf")
            plt.savefig(save_path)
            print(f"Saved control bar plot to {save_path}")
        except:
            pass  # Continue if save fails
        plt.show()
        
        # ================ STATISTICAL ANALYSIS SECTION ================
        
        print(f"\nStatistical analysis for CONTROL data {' vs '.join(days)}:")
        
        try:
            # For two-day comparisons, use a 2×2 ANOVA (Day × CS type)
            if len(days) == 2:
                print("Running 2×2 ANOVA (Day × CS type) for control data")
                
                # Create a proper 2×2 ANOVA model using statsmodels
                formula = "correlation ~ C(day) * C(cs_type)"
                
                # Try mixed model first to account for repeated measures on same cells
                print("\nAttempting Mixed Effects Model (accounting for cell_id as random effect):")
                try:
                    model = mixedlm(formula, data=plot_data, groups="cell_id")
                    model_results = model.fit()
                    
                    # Print results in a clear format
                    print("\nMixed Effects Model Results:")
                    print(model_results.summary())
                    
                    # Extract coefficients and p-values for manuscript reporting
                    print("\nFixed Effects Summary for Manuscript:")
                    print("-" * 40)
                    coef_table = pd.DataFrame({
                        'Effect': model_results.fe_params.index,
                        'Coefficient': model_results.fe_params.values,
                        'p-value': model_results.pvalues.values
                    })
                    print(coef_table)
                    
                except Exception as e:
                    print(f"Mixed model failed for control data: {str(e)}")
                    print("Falling back to standard ANOVA (without random effects):")
                    
                    # Standard ANOVA as fallback
                    model = ols(formula, data=plot_data).fit()
                    anova_table = anova_lm(model, typ=2)
                    print("\nANOVA Table:")
                    print(anova_table)
                    
                    # Extract F-statistics and degrees of freedom for manuscript reporting
                    print("\nManuscript Reporting Format:")
                    print("-" * 30)
                    
                    for effect in anova_table.index:
                        if effect != 'Residual':
                            f_val = anova_table.loc[effect, 'F']
                            p_val = anova_table.loc[effect, 'PR(>F)']
                            df_num = anova_table.loc[effect, 'df']
                            df_den = anova_table.loc['Residual', 'df']
                            
                            # Format p-value
                            if p_val < 0.001:
                                p_str = "p < 0.001"
                            else:
                                p_str = f"p = {p_val:.3f}"
                            
                            print(f"{effect}: F({df_num:.0f}, {df_den:.0f}) = {f_val:.3f}, {p_str}")
                
                # Run post-hoc tests
                print("\nPost-hoc Comparisons (Tukey HSD) for control data:")
                print("-" * 45)
                tukey_results = pairwise_tukeyhsd(plot_data['correlation'],
                                               plot_data['day'] + ' ' + plot_data['cs_type'])
                print(tukey_results)
                
                # Extract significant comparisons for easy interpretation
                print("\nSignificant Post-hoc Comparisons (p < 0.05):")
                print("-" * 45)
                significant_found = False
                for i, row in enumerate(tukey_results._results_table.data[1:]):
                    group1, group2, meandiff, p_adj, lower, upper, reject = row
                    if reject:  # reject is True when p < alpha
                        significant_found = True
                        print(f"{group1} vs {group2}: Mean diff = {meandiff:.4f}, p = {p_adj:.4f}")
                
                if not significant_found:
                    print("No significant pairwise differences found.")
                
                # Print summary statistics
                print("\nDescriptive Statistics for control data:")
                print("-" * 35)
                summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
                print(summary_stats)
                
            else:
                # For 3+ days, use the full mixed model with interaction
                print("Running full model for multi-day comparison of control data")
                
                try:
                    # Try mixed model first
                    model = mixedlm("correlation ~ C(day) * C(cs_type)", data=plot_data, groups="cell_id")
                    model_results = model.fit()
                    
                    print("\nMixed Effects Model Results:")
                    print(model_results.summary())
                    
                    # ADD THIS SECTION: Also run standard ANOVA for the F-table output
                    print("\nANOVA Table (for manuscript reporting):")
                    print("-" * 40)
                    anova_model = ols("correlation ~ C(day) * C(cs_type)", data=plot_data).fit()
                    anova_table = anova_lm(anova_model, typ=2)
                    print(anova_table)
                    
                    # Extract F-statistics and degrees of freedom for manuscript reporting
                    print("\nManuscript Reporting Format:")
                    print("-" * 30)
                    
                    for effect in anova_table.index:
                        if effect != 'Residual':
                            f_val = anova_table.loc[effect, 'F']
                            p_val = anova_table.loc[effect, 'PR(>F)']
                            df_num = anova_table.loc[effect, 'df']
                            df_den = anova_table.loc['Residual', 'df']
                            
                            # Format p-value
                            if p_val < 0.001:
                                p_str = "p < 0.001"
                            else:
                                p_str = f"p = {p_val:.3f}"
                            
                            print(f"{effect}: F({df_num:.0f}, {df_den:.0f}) = {f_val:.3f}, {p_str}")
                    
                except Exception as e:
                    print(f"Mixed model failed for control data: {str(e)}")
                    print("Using standard ANOVA:")
                    
                    # Standard ANOVA as fallback
                    model = ols("correlation ~ C(day) * C(cs_type)", data=plot_data).fit()
                    anova_table = anova_lm(model, typ=2)
                    print("\nANOVA Table:")
                    print(anova_table)
                    
                    # Extract F-statistics and degrees of freedom for manuscript reporting
                    print("\nManuscript Reporting Format:")
                    print("-" * 30)
                    
                    for effect in anova_table.index:
                        if effect != 'Residual':
                            f_val = anova_table.loc[effect, 'F']
                            p_val = anova_table.loc[effect, 'PR(>F)']
                            df_num = anova_table.loc[effect, 'df']
                            df_den = anova_table.loc['Residual', 'df']
                            
                            # Format p-value
                            if p_val < 0.001:
                                p_str = "p < 0.001"
                            else:
                                p_str = f"p = {p_val:.3f}"
                            
                            print(f"{effect}: F({df_num:.0f}, {df_den:.0f}) = {f_val:.3f}, {p_str}")
                
                # Run Tukey's HSD post-hoc test
                print("\nPost-hoc Comparisons (Tukey HSD) for control data:")
                print("-" * 45)
                tukey_results = pairwise_tukeyhsd(plot_data['correlation'],
                                               plot_data['day'] + ' ' + plot_data['cs_type'])
                print(tukey_results)
                
                # Extract significant comparisons
                print("\nSignificant Post-hoc Comparisons (p < 0.05):")
                print("-" * 45)
                significant_found = False
                for i, row in enumerate(tukey_results._results_table.data[1:]):
                    group1, group2, meandiff, p_adj, lower, upper, reject = row
                    if reject:
                        significant_found = True
                        print(f"{group1} vs {group2}: Mean diff = {meandiff:.4f}, p = {p_adj:.4f}")
                
                if not significant_found:
                    print("No significant pairwise differences found.")
                
                # Print summary statistics
                print("\nDescriptive Statistics for control data:")
                print("-" * 35)
                summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
                print(summary_stats)
                
        except Exception as e:
            print(f"Error during statistical analysis of control data: {str(e)}")
            print("Falling back to basic summary statistics:")
            
            # Basic summary statistics
            summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
            print(summary_stats)

# Execute the grouped analysis for control data
# This assumes that results_df_control has already been created by the analyze_correlations_control function
analyze_day_groupings_control(results_df_control)

# This code runs the ROI overlap cross correlation spatial analysis 
## This cell generates pickles or asks the user to load them.
## Make sure to realize the code asks you to input a 1 (load existing analyses) or a 2 (generate new analyses) for each day.
## Pickles are stored in the basedir within the results folder

In [ ]:
# import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches
import os
import roifile
import shapely.geometry
from matplotlib.path import Path
import matplotlib.path as mpath
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
import multiprocessing
from tqdm.notebook import tqdm
import torch
from torch.nn.functional import conv2d
import torch.nn as nn
import pickle

##THIS IS SO WE CAN HAVE THE SPATIAL PLOTS IN NICER COLORS
import numpy as np; np.random.seed(0)
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
def create_color(r, g, b):
    return [r/256, g/256, b/256]

# Colors for Magenta-Pink to Green PiYG_r gradient
colors_tiedye = [
    '#4B0082',   # Indigo/Deep Purple (highest - 1.0)
    '#9932CC',  # Dark Orchid (medium purple)
    '#2A788E',  # Teal (lowest - 0.0)
    '#74D054',  # Yellow-green
    '#FDE725',  # Yellow
    '#FFA500',  # Orange
    '#FF4500',  # Orange-red
    '#FF1493'  # Deep Pink
]

data = np.random.uniform(-1, 1, (10, 15))
sns.set_style('white')
# Create the colormap
rainbow_cmap = LinearSegmentedColormap.from_list('Magenta_tyedye', colors_tiedye)

# Check if MPS (Metal Performance Shaders) is available for Mac GPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

def calculate_correlation_matrix_gpu(fluorescence_data):
    """Calculate correlation matrix using GPU acceleration."""
    # Convert to torch tensor and move to GPU
    X = torch.tensor(fluorescence_data, dtype=torch.float32).to(device)
    
    # Normalize the data
    X = (X - X.mean(dim=1, keepdim=True)) / X.std(dim=1, keepdim=True)
    
    # Calculate correlation matrix using matrix multiplication
    corr_matrix = torch.mm(X, X.t()) / X.size(1)
    
    return corr_matrix.cpu().numpy()

def create_pixel_mask_gpu(coords, grid_size, offset_x, offset_y, pixels_tensor):
    """Create pixel mask for ROIs using GPU acceleration."""
    # Transform coordinates and convert to tensor
    coords_transformed = coords + np.array([offset_x, offset_y])
    coords_tensor = torch.tensor(coords_transformed, dtype=torch.float32).to(device)
    
    # Create path and test points
    path = Path(coords_transformed)
    mask = path.contains_points(pixels_tensor.cpu().numpy())
    mask_tensor = torch.tensor(mask, dtype=torch.float32).to(device)
    
    return mask_tensor.reshape(grid_size)

class NeuralCorrelationAnalyzer:
    def __init__(self, basedir, df_averaged, device=None):
        """Initialize the analyzer with base settings."""
        self.basedir = basedir
        self.df_averaged = df_averaged
        
        # Set up GPU device
        if device is None:
            if torch.backends.mps.is_available():
                self.device = torch.device("mps")
            elif torch.cuda.is_available():
                self.device = torch.device("cuda")
            else:
                self.device = torch.device("cpu")
        else:
            self.device = device
            
        # Create Results directory structure
        self.results_dir = os.path.join(self.basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'correlation_pickles')
        self.viz_dir = os.path.join(self.results_dir, 'correlation_visualizations')
        
        os.makedirs(self.pickle_dir, exist_ok=True)
        os.makedirs(self.viz_dir, exist_ok=True)
            
        print(f"Using device: {self.device}")
        print(f"Results directory: {self.results_dir}")

    def _get_pickle_path(self, day):
        """Generate pickle file path for a specific day."""
        return os.path.join(self.pickle_dir, f"correlation_data_{day}.pkl")

    def _pickle_exists(self, day):
        """Check if pickle exists for a specific day."""
        return os.path.exists(self._get_pickle_path(day))

    def _save_to_pickle(self, data, day):
        """Save correlation data to pickle."""
        pickle_path = self._get_pickle_path(day)
        # Convert torch tensors to numpy arrays for pickling
        pickle_data = {
            animal_fov: {
                cs_type: {
                    'accum': acc['accum'].cpu().numpy(),
                    'count': acc['count'].cpu().numpy()
                }
                for cs_type, acc in data_dict.items()
            }
            for animal_fov, data_dict in data.items()
        }
        with open(pickle_path, 'wb') as f:
            pickle.dump(pickle_data, f)
        print(f"\nPickle file saved: {pickle_path}")

    def _load_from_pickle(self, day):
        """Load correlation data from pickle."""
        pickle_path = self._get_pickle_path(day)
        with open(pickle_path, 'rb') as f:
            pickle_data = pickle.load(f)
        # Convert numpy arrays back to torch tensors
        loaded_data = {
            animal_fov: {
                cs_type: {
                    'accum': torch.tensor(acc['accum'], device=self.device),
                    'count': torch.tensor(acc['count'], device=self.device)
                }
                for cs_type, acc in data_dict.items()
            }
            for animal_fov, data_dict in pickle_data.items()
        }
        print(f"\nLoaded pickle file: {pickle_path}")
        return loaded_data

    def process_day(self, day, grid_size=(1024, 1024), cmap=rainbow_cmap):
        """Process a day's worth of data with user choice for loading pickle."""
        print(f"\nProcessing day: {day}")
        
        if self._pickle_exists(day):
            while True:
                choice = input(f"\nFound existing analysis for {day}. Choose an option:\n"
                             "1. Load existing analysis\n"
                             "2. Generate new analysis\n"
                             "Enter choice (1 or 2): ")
                
                if choice in ['1', '2']:
                    break
                print("Invalid choice. Please enter 1 or 2.")
            
            if choice == '1':
                print("\nLoading existing analysis...")
                return self.load_and_visualize(day, grid_size, cmap)
        else:
            print("\nNo existing analysis found. Generating new analysis...")
            choice = '2'

        if choice == '2':
            print("\nCalculating correlations...")
            return self.generate_new_analysis(day, grid_size, cmap)

    def _process_fov(self, day, animal, fov, grid_size, cmap):
        """Process a single FOV and return accumulators."""
        # Get the data for this FOV
        fov_data = self.df_averaged[
            (self.df_averaged['day'] == day) & 
            (self.df_averaged['animal'] == animal) & 
            (self.df_averaged['fov'] == fov)
        ]
        
        if fov_data.empty:
            print(f"No data found for {animal} {fov}")
            return None
            
        # Load ROI data
        roi_folder = os.path.join(self.basedir, day, animal, fov)
        roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
        
        if not roi_files:
            print(f"No ROI files found for {animal} {fov}")
            return None
            
        rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
        
        # Initialize accumulators for this FOV
        accumulators = {}
        for cs_type in ['cs_plus', 'cs_minus']:
            # Initialize accumulators
            accum = torch.zeros(grid_size, device=self.device)
            count = torch.zeros(grid_size, device=self.device)
            
            # Get fluorescence columns
            fluor_prefix = 'fluor_plus_frame_' if cs_type == 'cs_plus' else 'fluor_minus_frame_'
            fluor_cols = [col for col in fov_data.columns if col.startswith(fluor_prefix)]
            
            if fluor_cols:
                # Get cell data and fluorescence data
                cells_data = []
                fluorescence_data = []
                
                cell_ids = fov_data['cell_id'].tolist()
                for roi_idx, (roi, cell_id) in enumerate(zip(rois, cell_ids)):
                    cell_data = fov_data[fov_data['cell_id'] == cell_id]
                    if not cell_data.empty:
                        cells_data.append({
                            'roi_number': roi_idx + 1,
                            'cell_id': cell_id,
                            'coords': roi.coordinates()
                        })
                        fluorescence_data.append(cell_data[fluor_cols].values[0])
                
                if cells_data and len(fluorescence_data) > 0:
                    # Calculate correlation matrix
                    correlation_matrix = calculate_correlation_matrix_gpu(np.array(fluorescence_data))
                    
                    # Create pixels tensor
                    y_coords, x_coords = np.mgrid[0:grid_size[0], 0:grid_size[1]]
                    pixels = np.vstack((x_coords.ravel(), y_coords.ravel())).T
                    pixels_tensor = torch.tensor(pixels, dtype=torch.float32).to(self.device)
                    
                    # Process correlations
                    for central_idx, central_cell in enumerate(cells_data):
                        central_correlations = torch.tensor(correlation_matrix[central_idx], device=self.device)
                        center_x = np.mean(central_cell['coords'][:, 0])
                        center_y = np.mean(central_cell['coords'][:, 1])
                        offset_x = grid_size[0] // 2 - center_x
                        offset_y = grid_size[1] // 2 - center_y
                        
                        for idx, cell in enumerate(cells_data):
                            correlation = central_correlations[idx]
                            mask = create_pixel_mask_gpu(
                                cell['coords'], 
                                grid_size, 
                                offset_x, 
                                offset_y,
                                pixels_tensor
                            )
                            accum.add_(mask * correlation)
                            count.add_(mask)
            
            accumulators[cs_type] = {'accum': accum, 'count': count}
        
        # Generate visualization for this FOV
        self._generate_fov_visualization(day, animal, fov, accumulators, grid_size, cmap)
        
        return accumulators

    def _generate_fov_visualization(self, day, animal, fov, accumulators, grid_size, cmap):
        """Generate and save visualization for a single FOV."""
        for cs_type, acc in accumulators.items():
            # Calculate average correlation
            with torch.no_grad():
                avg_correlation = torch.where(
                    acc['count'] > 0,
                    acc['accum'] / acc['count'],
                    torch.tensor(float('nan'), device=self.device)
                )
                
            # Create visualization
            fig, ax = plt.subplots(figsize=(15, 15))
            
            # Plot correlation map
            masked_correlation = np.ma.masked_invalid(avg_correlation.cpu().numpy())
            im = ax.imshow(masked_correlation, cmap=cmap, vmin=-1, vmax=1)
            
            # Add colorbar and title
            plt.colorbar(im, ax=ax)
            cs_label = 'CS+' if cs_type == 'cs_plus' else 'CS-'
            plt.title(f'{animal} - {fov} - {day} ({cs_label})')
            
            # # # Save and close
            # save_path = os.path.join(
            #     self.viz_dir,
            #     f"correlation_{animal}_{fov}_{day}_{cs_label.replace('+', 'plus')}.png"
            # )
            # plt.savefig(save_path, dpi=300, bbox_inches='tight')
            # plt.close()

    def _generate_day_combined(self, day, accumulators, grid_size, cmap):
        """Generate and save day-combined visualization."""
        for cs_type, acc in accumulators.items():
            # Calculate average correlation
            with torch.no_grad():
                avg_correlation = torch.where(
                    acc['count'] > 0,
                    acc['accum'] / acc['count'],
                    torch.tensor(float('nan'), device=self.device)
                )
                
            # Create visualization
            fig, ax = plt.subplots(figsize=(15, 15))
            
            # Plot correlation map
            masked_correlation = np.ma.masked_invalid(avg_correlation.cpu().numpy())
            im = ax.imshow(masked_correlation, cmap=cmap, vmin=-1, vmax=1)
            
            # Add colorbar and title
            plt.colorbar(im, ax=ax)
            cs_label = 'CS+' if cs_type == 'cs_plus' else 'CS-'
            plt.title(f'Day-Combined Average Cross-correlations - {cs_label}\nDay: {day}')
            
            # Save and close
            save_path = os.path.join(
                self.viz_dir,
                f"correlation_combined_{day}_{cs_label.replace('+', 'plus')}_06-23-25.pdf"
            )
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()

    def generate_new_analysis(self, day, grid_size=(1024, 1024), cmap=rainbow_cmap):
        """Generate new analysis for a day."""
        # Get unique animals and FOVs for this day
        day_data = self.df_averaged[self.df_averaged['day'] == day]
        animals = day_data['animal'].unique()
        
        # Store all FOV results
        all_results = {}
        
        # Initialize day-level accumulators
        day_accumulators = {
            'cs_plus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            },
            'cs_minus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            }
        }

# Process each animal
        for animal in animals:
            animal_data = day_data[day_data['animal'] == animal]
            fovs = animal_data['fov'].unique()
            
            print(f"\nProcessing animal: {animal}")
            print(f"Found {len(fovs)} FOVs")
            
            # Process each FOV
            for fov in fovs:
                try:
                    print(f"Processing FOV: {fov}")
                    accums = self._process_fov(day, animal, fov, grid_size, cmap)
                    
                    if accums is not None:
                        # Store results
                        all_results[f"{animal}_{fov}"] = accums
                        
                        # Add to day accumulators
                        for cs_type in ['cs_plus', 'cs_minus']:
                            day_accumulators[cs_type]['accum'].add_(accums[cs_type]['accum'])
                            day_accumulators[cs_type]['count'].add_(accums[cs_type]['count'])
                            
                except Exception as e:
                    print(f"Error processing {animal} {fov}: {str(e)}")
                    continue
        
        # Save results to pickle
        self._save_to_pickle(all_results, day)
        
        # Generate day-combined visualization
        self._generate_day_combined(day, day_accumulators, grid_size, cmap)
        
        # Clean up GPU memory
        del day_accumulators
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return all_results

    def load_and_visualize(self, day, grid_size=(1024, 1024), cmap='cmap'):
        """Load pickled data and regenerate visualizations."""
        all_results = self._load_from_pickle(day)
        
        # Initialize day-level accumulators
        day_accumulators = {
            'cs_plus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            },
            'cs_minus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            }
        }
        
        # Process each FOV's data
        for animal_fov, accums in all_results.items():
            animal, fov = animal_fov.split('_', 1)
            
            # Generate individual FOV visualization
            self._generate_fov_visualization(day, animal, fov, accums, grid_size, cmap)
            
            # Add to day accumulators
            for cs_type in ['cs_plus', 'cs_minus']:
                day_accumulators[cs_type]['accum'].add_(accums[cs_type]['accum'])
                day_accumulators[cs_type]['count'].add_(accums[cs_type]['count'])
        
        # Generate day-combined visualization
        self._generate_day_combined(day, day_accumulators, grid_size, cmap)
        
        # Clean up GPU memory
        del day_accumulators
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return all_results

# Example usage:
if __name__ == '__main__':
    # Initialize analyzer
    analyzer = NeuralCorrelationAnalyzer(
        basedir=basedir,
        df_averaged=df_averaged
    )

    # # Process a day - will prompt for loading pickle if it exists
    # results = analyzer.process_day('3_CueOmit')

    # If you want to process multiple days:
    days_to_process = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    for day in days_to_process:
        results = analyzer.process_day(day)


# Producing a methods visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import random
import roifile
import torch
from matplotlib.path import Path
import matplotlib.path as mpath


##THIS IS SO WE CAN HAVE THE SPATIAL PLOTS IN NICER COLORS
import numpy as np; np.random.seed(0)
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
def create_color(r, g, b):
    return [r/256, g/256, b/256]

# Colors for Magenta-Pink to Green PiYG_r gradient
colors_tiedye = [
    '#4B0082',   # Indigo/Deep Purple (highest - 1.0)
    '#9932CC',  # Dark Orchid (medium purple)
    '#2A788E',  # Teal (lowest - 0.0)
    '#74D054',  # Yellow-green
    '#FDE725',  # Yellow
    '#FFA500',  # Orange
    '#FF4500',  # Orange-red
    '#FF1493'  # Deep Pink
]

data = np.random.uniform(-1, 1, (10, 15))
sns.set_style('white')
# Create the colormap
rainbow_cmap = LinearSegmentedColormap.from_list('Magenta_tyedye', colors_tiedye)

# Check if MPS (Metal Performance Shaders) is available for Mac GPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

def calculate_correlation_matrix_gpu(fluorescence_data):
    """Calculate correlation matrix using GPU acceleration."""
    # Convert to torch tensor and move to GPU
    X = torch.tensor(fluorescence_data, dtype=torch.float32).to(device)
    
    # Normalize the data
    X = (X - X.mean(dim=1, keepdim=True)) / X.std(dim=1, keepdim=True)
    
    # Calculate correlation matrix using matrix multiplication
    corr_matrix = torch.mm(X, X.t()) / X.size(1)
    
    return corr_matrix.cpu().numpy()

def create_pixel_mask_gpu(coords, grid_size, offset_x, offset_y, pixels_tensor):
    """Create pixel mask for ROIs using GPU acceleration."""
    # Transform coordinates and convert to tensor
    coords_transformed = coords + np.array([offset_x, offset_y])
    coords_tensor = torch.tensor(coords_transformed, dtype=torch.float32).to(device)
    
    # Create path and test points
    path = Path(coords_transformed)
    mask = path.contains_points(pixels_tensor.cpu().numpy())
    mask_tensor = torch.tensor(mask, dtype=torch.float32).to(device)
    
    return mask_tensor.reshape(grid_size)

def visualize_correlation_layer_buildup(basedir, day, df_averaged, animal_fov_key=None, n_layers=8, 
                                    show_outlines=True, outline_width=0.5, 
                                    outline_alpha=0.2, highlight_center=False):
    """
    Create a visualization showing how correlation maps are built up layer by layer.
    
    Parameters:
    -----------
    basedir : str
        Base directory containing the data
    day : str
        Day to analyze
    df_averaged : DataFrame
        DataFrame containing the averaged data
    animal_fov_key : str, optional
        Specific animal_fov key to analyze (e.g., "Astro2_Astro2_FOV1"). 
        If None, will be randomly selected.
    n_layers : int
        Number of layers to show
    show_outlines : bool
        Whether to show ROI outlines
    outline_width : float
        Width of ROI outlines
    outline_alpha : float
        Alpha transparency of ROI outlines
    highlight_center : bool
        Whether to highlight the central cell in each layer
    """
    pickle_path = os.path.join(basedir, 'Results', 'correlation_pickles', f'correlation_data_{day}.pkl')
    
    if not os.path.exists(pickle_path):
        print(f"No pickle file found for {day}")
        return
        
    with open(pickle_path, 'rb') as f:
        data = pickle.load(f)
    
    # Get available keys
    available_keys = list(data.keys())
    
    # Select animal_fov key directly or randomly if not provided
    if animal_fov_key is not None:
        # Check if the requested key is available
        if animal_fov_key not in available_keys:
            print(f"Requested key '{animal_fov_key}' not found in available data.")
            print(f"Available animal_fov combinations: {available_keys}")
            return
        animal_fov = animal_fov_key
    else:
        # Randomly select an animal_fov if not specified
        animal_fov = random.choice(available_keys)
    
    print(f"Selected key: {animal_fov}")
    
    # Extract the first part (animal name) from the key
    # For keys like "Astro2_Astro2_FOV1", the animal is "Astro2"
    parts = animal_fov.split('_')
    animal = parts[0]
    
    # Extract FOV identifier (format in df_averaged: "Astro2_FOV1")
    # This appears to be animal+FOV part
    if "FOV" in animal_fov:
        fov_part = animal_fov.split("FOV")[1]
        fov = f"{animal}_FOV{fov_part}"
    else:
        # Fallback if FOV format is different
        fov = animal_fov.split("_")[-1]
    
    print(f"Using animal={animal}, fov={fov}")
    
    # Get the data for this FOV
    fov_data = df_averaged[
        (df_averaged['day'] == day) & 
        (df_averaged['animal'] == animal) & 
        (df_averaged['fov'] == fov)
    ]
    
    if len(fov_data) == 0:
        print(f"No matching data found in df_averaged for day={day}, animal={animal}, fov={fov}")
        return
    
    print(f"Found {len(fov_data)} rows in df_averaged")
    
    # Load ROI data
    # Try multiple possible directory structures
    possible_paths = []
    
    # Path 1: Standard format with just FOV part
    directory_fov = fov.split("_")[-1]
    path1 = os.path.join(basedir, day, animal, directory_fov)
    possible_paths.append(path1)
    
    # Path 2: Using full FOV string
    path2 = os.path.join(basedir, day, animal, fov)
    possible_paths.append(path2)
    
    # Path 3: Using the first FOV part without animal prefix
    if "_" in fov:
        simple_fov = fov.split("_")[-1]
        path3 = os.path.join(basedir, day, animal, simple_fov)
        possible_paths.append(path3)
    
    # Path 4: Additional check for Astro2_FOV1/FOV1 structure
    if "FOV" in fov:
        path4 = os.path.join(basedir, day, animal, fov, directory_fov)
        possible_paths.append(path4)
    
    # Path 5: Using just 'FOV1' without the 'FOV' prefix
    if "FOV" in directory_fov:
        simple_number = directory_fov.replace("FOV", "")
        path5 = os.path.join(basedir, day, animal, simple_number)
        possible_paths.append(path5)
    
    # Try each path
    roi_folder = None
    for i, path in enumerate(possible_paths, 1):
        print(f"Trying path {i}: {path}")
        if os.path.exists(path):
            print(f"Found valid directory at path {i}!")
            roi_folder = path
            break
    
    # If none of the paths work, try listing the contents of the parent directory
    if roi_folder is None:
        parent_dir = os.path.join(basedir, day, animal)
        print(f"\nChecking contents of parent directory: {parent_dir}")
        if os.path.exists(parent_dir):
            contents = os.listdir(parent_dir)
            print(f"Contents of {parent_dir}: {contents}")
            
            # If there's only one subdirectory, use it
            subdirs = [d for d in contents if os.path.isdir(os.path.join(parent_dir, d))]
            if len(subdirs) == 1:
                roi_folder = os.path.join(parent_dir, subdirs[0])
                print(f"Using the only available subdirectory: {roi_folder}")
        else:
            print(f"Parent directory {parent_dir} does not exist")
    
    if roi_folder is None:
        print("ERROR: Could not find a valid directory for ROI files")
        return
        
    roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
    if not roi_files:
        print(f"No ROI files found in {roi_folder}")
        return
    
    print(f"Using ROI folder: {roi_folder}")
    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
        
    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
    
    # Get total number of cells
    n_cells = len(rois)
    layer_size = max(1, n_cells // n_layers)
    
    # Create figure
    fig, axes = plt.subplots(2, n_layers, figsize=(20, 6))
    fig.suptitle(f'Building Correlation Maps Layer by Layer\n{animal} {fov} - Day {day}', size=16)
    
    # Create pixel grid for masks
    grid_size = (1024, 1024)
    y_coords, x_coords = np.mgrid[0:grid_size[0], 0:grid_size[1]]
    pixels = np.vstack((x_coords.ravel(), y_coords.ravel())).T
    pixels_tensor = torch.tensor(pixels, dtype=torch.float32).to(device)
    
    # Store all cell coordinates first
    cells_data = []
    for i in range(n_cells):
        cells_data.append({
            'roi_number': i + 1,
            'coords': rois[i].coordinates()
        })
    
    # Get all fluorescence data
    all_fluorescence = {}
    for cs_type in ['cs_plus', 'cs_minus']:
        fluor_prefix = 'fluor_plus_frame_' if cs_type == 'cs_plus' else 'fluor_minus_frame_'
        fluor_cols = [col for col in fov_data.columns if col.startswith(fluor_prefix)]
        
        fluorescence_data = []
        for i in range(n_cells):
            cell_data = fov_data.iloc[i]
            fluor_values = cell_data[fluor_cols].values.astype(np.float32)
            fluorescence_data.append(fluor_values)
        
        all_fluorescence[cs_type] = np.stack(fluorescence_data)
    
    # Process CS+ and CS-
    for cs_idx, cs_type in enumerate(['cs_plus', 'cs_minus']):
        fluorescence_array = all_fluorescence[cs_type]
        
        # Calculate full correlation matrix once
        full_correlation_matrix = calculate_correlation_matrix_gpu(fluorescence_array)
        
        # Calculate progressive layer buildup
        for layer in range(n_layers):
            n_layers_included = min((layer + 1), n_layers)
            cells_per_layer = max(1, n_cells // n_layers)
            
            # Initialize accumulators for this layer
            accum = torch.zeros(grid_size, device=device)
            count = torch.zeros(grid_size, device=device)
            
            # Process each included layer
            for l in range(n_layers_included):
                # Select central cell for this layer
                central_idx = min(l * cells_per_layer, n_cells-1)
                central_cell = rois[central_idx]
                center_x = np.mean(central_cell.coordinates()[:, 0])
                center_y = np.mean(central_cell.coordinates()[:, 1])
                
                # Calculate offset to center
                offset_x = grid_size[0] // 2 - center_x
                offset_y = grid_size[1] // 2 - center_y
                
                # Get correlations for this central cell
                central_correlations = full_correlation_matrix[central_idx]
                
                # Add correlation data for all cells from this central perspective
                for idx, cell in enumerate(cells_data):
                    correlation = central_correlations[idx]
                    mask = create_pixel_mask_gpu(
                        cell['coords'],
                        grid_size,
                        offset_x,
                        offset_y,
                        pixels_tensor
                    )
                    accum.add_(mask * correlation)
                    count.add_(mask)
            
            # Calculate average correlation map
            with torch.no_grad():
                avg_correlation = torch.where(
                    count > 0,
                    accum / count,
                    torch.tensor(float('nan'), device=device)
                )
            
            # Plot correlation map
            ax = axes[cs_idx, layer]
            masked_correlation = np.ma.masked_invalid(avg_correlation.cpu().numpy())
            im = ax.imshow(masked_correlation, cmap=rainbow_cmap, vmin=-1, vmax=1)
            
            if show_outlines:
                # Add ROI outlines for all included layers
                for l in range(n_layers_included):
                    central_idx = min(l * cells_per_layer, n_cells-1)
                    central_cell = rois[central_idx]
                    center_x = np.mean(central_cell.coordinates()[:, 0])
                    center_y = np.mean(central_cell.coordinates()[:, 1])
                    offset_x = grid_size[0] // 2 - center_x
                    offset_y = grid_size[1] // 2 - center_y
                    
                    # Plot all cells from this layer's perspective
                    for cell in cells_data:
                        coords = cell['coords']
                        coords_transformed = coords + np.array([offset_x, offset_y])
                        ax.plot(coords_transformed[:, 0], coords_transformed[:, 1], 
                               'k-', linewidth=outline_width, alpha=outline_alpha)
                    
                    # Optionally highlight central cell
                    if highlight_center:
                        central_coords = cells_data[central_idx]['coords']
                        central_coords_transformed = central_coords + np.array([offset_x, offset_y])
                        ax.plot(central_coords_transformed[:, 0], central_coords_transformed[:, 1], 
                               'gray', linewidth=outline_width*2, alpha=outline_alpha*2)
            
            # Set title and remove axes
            ax.set_title(f'{n_layers_included} layers', size=10)
            ax.axis('off')
            
            # Add CS type label to leftmost plot
            if layer == 0:
                ax.set_ylabel('CS+' if cs_type == 'cs_plus' else 'CS-', size=12)
    
    # Add colorbar
    plt.colorbar(im, ax=axes.ravel().tolist(), label='Correlation')
    
    # # Save figure
    save_dir = os.path.join(basedir, 'Results', 'correlation_visualizations')
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(os.path.join(save_dir, f'correlation_layer_buildup_{day}_{animal}_{fov}.pdf'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Saved visualization showing correlation map layer buildup")

# Example usage:
basedir = basedir

# Process specific days
# days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']

days = ['1_Late']

for day in days:
    # # Default view with subtle outlines
    # visualize_correlation_layer_buildup(basedir, day, df_averaged, 
    #                                   outline_width=0.3, outline_alpha=0.15)
    
    # # No outlines
    # visualize_correlation_layer_buildup(basedir, day, df_averaged, 
    #                                   show_outlines=False)
    
    # Example of how to specify a specific animal_fov key:
    visualize_correlation_layer_buildup(basedir, day, df_averaged, 
                                      animal_fov_key="Astro34_Astro34_FOV1",
                                      show_outlines=False)

# Showing each FOV on a combined plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import math
import numpy as np; np.random.seed(0)
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

def visualize_day_correlations(basedir, day):
    """
    Load pickle files and create visualizations for a given day.
    Creates two figures per CS type:
    1. Subplot grid showing each individual FOV
    2. Combined data for the day
    """
    # Load pickle data
    pickle_path = os.path.join(basedir, 'Results', 'correlation_pickles', f'correlation_data_{day}.pkl')
    
    if not os.path.exists(pickle_path):
        print(f"No pickle file found for {day}")
        return
        
    with open(pickle_path, 'rb') as f:
        data = pickle.load(f)
    
    # Get number of FOVs for subplot grid
    n_fovs = len(data)
    grid_size = math.ceil(math.sqrt(n_fovs))
    
    # Initialize accumulators for combined plots
    combined_data = {
        'cs_plus': {'accum': None, 'count': None},
        'cs_minus': {'accum': None, 'count': None}
    }
    
    # Create subplot figures for CS+ and CS-
    fig_plus = plt.figure(figsize=(20, 20))
    fig_minus = plt.figure(figsize=(20, 20))
    fig_plus.suptitle(f'Individual FOV Correlations - CS+ - Day {day}', size=16)
    fig_minus.suptitle(f'Individual FOV Correlations - CS- - Day {day}', size=16)
    
    # Process each FOV
    for i, (animal_fov, fov_data) in enumerate(data.items()):
        animal, fov = animal_fov.split('_', 1)
        
        # Process CS+ data
        cs_plus_data = fov_data['cs_plus']
        avg_corr_plus = np.where(cs_plus_data['count'] > 0, 
                                cs_plus_data['accum'] / cs_plus_data['count'], 
                                np.nan)
        
        # Process CS- data
        cs_minus_data = fov_data['cs_minus']
        avg_corr_minus = np.where(cs_minus_data['count'] > 0,
                                 cs_minus_data['accum'] / cs_minus_data['count'],
                                 np.nan)
        
        # Add to subplots
        # CS+ subplot
        ax_plus = fig_plus.add_subplot(grid_size, grid_size, i+1)
        im_plus = ax_plus.imshow(avg_corr_plus, cmap=rainbow_cmap, vmin=-1, vmax=1)
        ax_plus.set_title(f'{animal}\n{fov}', size=10)
        ax_plus.axis('off')
        
        # CS- subplot
        ax_minus = fig_minus.add_subplot(grid_size, grid_size, i+1)
        im_minus = ax_minus.imshow(avg_corr_minus, cmap=rainbow_cmap, vmin=-1, vmax=1)
        ax_minus.set_title(f'{animal}\n{fov}', size=10)
        ax_minus.axis('off')
        
        # Add to combined data
        for cs_type, cs_data in [('cs_plus', cs_plus_data), ('cs_minus', cs_minus_data)]:
            if combined_data[cs_type]['accum'] is None:
                combined_data[cs_type]['accum'] = cs_data['accum']
                combined_data[cs_type]['count'] = cs_data['count']
            else:
                combined_data[cs_type]['accum'] += cs_data['accum']
                combined_data[cs_type]['count'] += cs_data['count']
    
    # Add colorbars to subplot figures
    fig_plus.colorbar(im_plus, ax=fig_plus.axes, label='Correlation')
    fig_minus.colorbar(im_minus, ax=fig_minus.axes, label='Correlation')
    
    # Create combined plots
    fig_combined, (ax_plus_comb, ax_minus_comb) = plt.subplots(1, 2, figsize=(20, 10))
    fig_combined.suptitle(f'Combined Correlations - Day {day}', size=16)
    
    # Plot combined CS+ data
    avg_corr_plus_comb = np.where(combined_data['cs_plus']['count'] > 0,
                                 combined_data['cs_plus']['accum'] / combined_data['cs_plus']['count'],
                                 np.nan)
    im_plus_comb = ax_plus_comb.imshow(avg_corr_plus_comb, cmap=rainbow_cmap, vmin=-1, vmax=1)
    ax_plus_comb.set_title('CS+')
    plt.colorbar(im_plus_comb, ax=ax_plus_comb, label='Correlation')
    
    # Plot combined CS- data
    avg_corr_minus_comb = np.where(combined_data['cs_minus']['count'] > 0,
                                  combined_data['cs_minus']['accum'] / combined_data['cs_minus']['count'],
                                  np.nan)
    im_minus_comb = ax_minus_comb.imshow(avg_corr_minus_comb, cmap=rainbow_cmap, vmin=-1, vmax=1)
    ax_minus_comb.set_title('CS-')
    plt.colorbar(im_minus_comb, ax=ax_minus_comb, label='Correlation')
    
    # Save figures
    save_dir = os.path.join(basedir, 'Results', 'correlation_visualizations')
    os.makedirs(save_dir, exist_ok=True)
    
    # fig_plus.savefig(os.path.join(save_dir, f'individual_fovs_CSplus_{day}.pdf'), 
    #                  dpi=300, bbox_inches='tight')
    # fig_minus.savefig(os.path.join(save_dir, f'individual_fovs_CSminus_{day}.pdf'), 
    #                   dpi=300, bbox_inches='tight')
    fig_combined.savefig(os.path.join(save_dir, f'combined_correlations_{day}_rainbow.pdf'), 
                        dpi=300, bbox_inches='tight')
    
    plt.show('all')
    
    print(f"Saved visualizations for {day}")
    
# Example usage:
basedir = basedir

# # Process a single day
# visualize_day_correlations(basedir, '3_CueOmit')

# Or process multiple days
days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
# days = ['0_Early']

for day in days:
    visualize_day_correlations(basedir, day)

# 3D Visualization, cut at 5 cell Radii from the center (11 diameters)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pickle
import os
from matplotlib.colors import Normalize
import torch
import roifile
from tqdm import tqdm

class MaskedCorrelation3DVisualizer:
    def __init__(self, basedir, df_averaged):
        """
        Initialize the visualizer with directory paths.
        
        Args:
            basedir (str): Base directory containing the Results folder
            df_averaged: DataFrame containing averaged data for cell diameter calculation
        """
        self.basedir = basedir
        self.df_averaged = df_averaged
        self.results_dir = os.path.join(basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'correlation_pickles')
        self.output_dir = os.path.join(self.results_dir, 'correlation_3d_visualizations_masked')
        
        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Calculate cell diameter
        self.cell_diameter = self.calculate_global_cell_diameter()
        print(f"Global average cell diameter: {self.cell_diameter:.2f} pixels")
        print(f"Using pickle directory: {self.pickle_dir}")
        print(f"Saving visualizations to: {self.output_dir}")
        
    def calculate_global_cell_diameter(self):
        """Calculate average cell diameter across all FOVs and days."""
        print("\nCalculating global cell diameter...")
        all_areas = []
        
        combinations = self.df_averaged.groupby(['day', 'animal', 'fov']).size().reset_index()
        
        for _, row in tqdm(combinations.iterrows(), total=len(combinations)):
            try:
                roi_folder = os.path.join(self.basedir, row['day'], row['animal'], row['fov'])
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                
                if roi_files:
                    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
                    for roi in rois:
                        coords = roi.coordinates()
                        x, y = coords[:, 0], coords[:, 1]
                        area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                        all_areas.append(area)
            except Exception as e:
                print(f"Error processing {roi_folder}: {str(e)}")
                continue
        
        return 2 * np.sqrt(np.mean(all_areas) / np.pi)
        
    def create_radius_mask(self, data_shape, center=None):
        """Create a mask for data within 11 cell radii."""
        if center is None:
            center = (data_shape[0] // 2, data_shape[1] // 2)
            
        y, x = np.ogrid[:data_shape[0], :data_shape[1]]
        distances = np.sqrt((x - center[0])**2 + (y - center[1])**2)
        
        max_radius = (self.cell_diameter/2) * 11  # 11 cell radii
        mask = distances <= max_radius
        
        return mask
        
    def load_pickle_data(self, day):
        """Load and process pickle data for a specific day."""
        pickle_path = os.path.join(self.pickle_dir, f"correlation_data_{day}.pkl")
        
        if not os.path.exists(pickle_path):
            raise FileNotFoundError(f"No pickle file found for day {day}")
            
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)
            
        # Initialize accumulators for day-combined data
        combined_data = {
            'cs_plus': {'accum': None, 'count': None},
            'cs_minus': {'accum': None, 'count': None}
        }
        
        # Combine data across all FOVs
        for animal_fov_data in data.values():
            for cs_type in ['cs_plus', 'cs_minus']:
                if combined_data[cs_type]['accum'] is None:
                    combined_data[cs_type]['accum'] = animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] = animal_fov_data[cs_type]['count']
                else:
                    combined_data[cs_type]['accum'] += animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] += animal_fov_data[cs_type]['count']
        
        # Calculate averages and apply radius mask
        processed_data = {}
        mask = self.create_radius_mask((1024, 1024))
        
        for cs_type in ['cs_plus', 'cs_minus']:
            avg = np.zeros_like(combined_data[cs_type]['accum'])
            valid_mask = (combined_data[cs_type]['count'] > 0) & mask
            avg[valid_mask] = combined_data[cs_type]['accum'][valid_mask] / combined_data[cs_type]['count'][valid_mask]
            avg[~mask] = np.nan  # Set values outside radius to NaN
            processed_data[cs_type] = avg
            
        return processed_data
        
    def create_3d_correlation_plots(self, day, cs_type, data, elev_angles=[0, 45, 90]):
        """
        Create 3D surface plots with multiple viewing angles.
        
        Args:
            day (str): Day identifier
            cs_type (str): 'cs_plus' or 'cs_minus'
            data (np.ndarray): Correlation data
            elev_angles (list): List of elevation angles for different views
        """
        # Create coordinate grid
        x = np.linspace(0, data.shape[0]-1, data.shape[0])
        y = np.linspace(0, data.shape[1]-1, data.shape[1])
        X, Y = np.meshgrid(x, y)
        
        # Set up the figure with subplots for different viewing angles
        fig = plt.figure(figsize=(20, 6))
        
        # Create plots for each viewing angle
        for idx, elev in enumerate(elev_angles):
            ax = fig.add_subplot(1, 3, idx+1, projection='3d')
            
            # Create surface plot with masked data
            masked_data = np.ma.masked_invalid(data)
            surf = ax.plot_surface(X, Y, masked_data, 
                                 cmap=rainbow_cmap,
                                 norm=Normalize(vmin=0, vmax=1),
                                 linewidth=0,
                                 antialiased=True)
            
            # Set viewing angle
            ax.view_init(elev=elev, azim=45)
            
            # Set labels and title
            ax.set_xlabel('X Position')
            ax.set_ylabel('Y Position')
            ax.set_zlabel('Correlation')
            
            # Set z-axis limits to 0-1
            ax.set_zlim(0, 1)
            
            # Customize z-axis ticks
            ax.set_zticks([ 0, 0.25, 0.5, 0.75, 1.0])
            
            view_label = "Side" if elev == 0 else "45°" if elev == 45 else "Top"
            ax.set_title(f'{view_label} View')
        
        # Add overall title and colorbar
        plt.suptitle(f'Day {day} - {cs_type.replace("cs_", "CS")} Correlation Surface\n(Limited to 11 cell radii)', y=1.05)
        fig.colorbar(surf, ax=fig.axes, shrink=0.5, aspect=5, ticks=[ 0, 0.25, 0.5, 0.75, 1.0])
        
        # Save plot
        cs_label = cs_type.replace('cs_plus', 'CSplus').replace('cs_minus', 'CSminus')
        save_path = os.path.join(self.output_dir, f'correlation_3d_{day}_{cs_label}_rainbow.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved visualization to: {save_path}")
        plt.show()

    def process_days(self, days):
        """Process and visualize data for multiple days."""
        for day in days:
            print(f"\nProcessing day: {day}")
            try:
                # Load data
                data = self.load_pickle_data(day)
                
                # Create visualizations for both CS types
                for cs_type in ['cs_plus', 'cs_minus']:
                    print(f"Creating visualization for {cs_type}")
                    self.create_3d_correlation_plots(day, cs_type, data[cs_type])
                    
            except Exception as e:
                print(f"Error processing day {day}: {str(e)}")
                continue
                
        print("\nProcessing complete!")

# Example usage:
if __name__ == "__main__":
    # Initialize visualizer with your base directory
    basedir = basedir
    
    # Create visualizer instance
    visualizer = MaskedCorrelation3DVisualizer(basedir, df_averaged)
    
    # Process all experimental days
    days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    visualizer.process_days(days)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pickle
import os
from matplotlib.colors import Normalize
import torch
import roifile
from tqdm import tqdm
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection

class MaskedCorrelation3DVisualizer:
    def __init__(self, basedir, df_averaged):
        """
        Initialize the visualizer with directory paths.
        
        Args:
            basedir (str): Base directory containing the Results folder
            df_averaged: DataFrame containing averaged data for cell diameter calculation
        """
        self.basedir = basedir
        self.df_averaged = df_averaged
        self.results_dir = os.path.join(basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'correlation_pickles')
        self.output_dir = os.path.join(self.results_dir, 'correlation_3d_visualizations_masked')
        
        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Calculate cell diameter
        self.cell_diameter = self.calculate_global_cell_diameter()
        print(f"Global average cell diameter: {self.cell_diameter:.2f} pixels")
        print(f"Using pickle directory: {self.pickle_dir}")
        print(f"Saving visualizations to: {self.output_dir}")
        
    def calculate_global_cell_diameter(self):
        """Calculate average cell diameter across all FOVs and days."""
        print("\nCalculating global cell diameter...")
        all_areas = []
        
        combinations = self.df_averaged.groupby(['day', 'animal', 'fov']).size().reset_index()
        
        for _, row in tqdm(combinations.iterrows(), total=len(combinations)):
            try:
                roi_folder = os.path.join(self.basedir, row['day'], row['animal'], row['fov'])
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                
                if roi_files:
                    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
                    for roi in rois:
                        coords = roi.coordinates()
                        x, y = coords[:, 0], coords[:, 1]
                        area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                        all_areas.append(area)
            except Exception as e:
                print(f"Error processing {roi_folder}: {str(e)}")
                continue
        
        return 2 * np.sqrt(np.mean(all_areas) / np.pi)
        
    def create_radius_mask(self, data_shape, center=None):
        """Create a mask for data within 11 cell radii."""
        if center is None:
            center = (data_shape[0] // 2, data_shape[1] // 2)
            
        y, x = np.ogrid[:data_shape[0], :data_shape[1]]
        distances = np.sqrt((x - center[0])**2 + (y - center[1])**2)
        
        max_radius = (self.cell_diameter/2) * 11  # 11 cell radii
        mask = distances <= max_radius
        
        return mask
        
    def load_pickle_data(self, day):
        """Load and process pickle data for a specific day."""
        pickle_path = os.path.join(self.pickle_dir, f"correlation_data_{day}.pkl")
        
        if not os.path.exists(pickle_path):
            raise FileNotFoundError(f"No pickle file found for day {day}")
            
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)
            
        # Initialize accumulators for day-combined data
        combined_data = {
            'cs_plus': {'accum': None, 'count': None},
            'cs_minus': {'accum': None, 'count': None}
        }
        
        # Combine data across all FOVs
        for animal_fov_data in data.values():
            for cs_type in ['cs_plus', 'cs_minus']:
                if combined_data[cs_type]['accum'] is None:
                    combined_data[cs_type]['accum'] = animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] = animal_fov_data[cs_type]['count']
                else:
                    combined_data[cs_type]['accum'] += animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] += animal_fov_data[cs_type]['count']
        
        # Calculate averages and apply radius mask
        processed_data = {}
        mask = self.create_radius_mask((1024, 1024))
        
        for cs_type in ['cs_plus', 'cs_minus']:
            avg = np.zeros_like(combined_data[cs_type]['accum'])
            valid_mask = (combined_data[cs_type]['count'] > 0) & mask
            avg[valid_mask] = combined_data[cs_type]['accum'][valid_mask] / combined_data[cs_type]['count'][valid_mask]
            avg[~mask] = np.nan  # Set values outside radius to NaN
            processed_data[cs_type] = avg
            
        return processed_data
        
    def create_3d_correlation_plots(self, day, cs_type, data, elev_angles=[0, 45, 90]):
        """
        Create 3D surface plots with multiple viewing angles.
        
        Args:
            day (str): Day identifier
            cs_type (str): 'cs_plus' or 'cs_minus'
            data (np.ndarray): Correlation data
            elev_angles (list): List of elevation angles for different views
        """
        # Create coordinate grid
        x = np.linspace(0, data.shape[0]-1, data.shape[0])
        y = np.linspace(0, data.shape[1]-1, data.shape[1])
        X, Y = np.meshgrid(x, y)
        
        # Calculate center coordinates
        center_x = data.shape[0] // 2
        center_y = data.shape[1] // 2
        
        # Set up the figure with subplots for different viewing angles
        fig = plt.figure(figsize=(20, 6))
        
        # Create plots for each viewing angle
        for idx, elev in enumerate(elev_angles):
            ax = fig.add_subplot(1, 3, idx+1, projection='3d')
            
            # Create surface plot with masked data
            masked_data = np.ma.masked_invalid(data)
            surf = ax.plot_surface(X, Y, masked_data, 
                                 cmap=rainbow_cmap,
                                 norm=Normalize(vmin=0, vmax=1),
                                 linewidth=0,
                                 antialiased=True)
            
            # Set viewing angle
            ax.view_init(elev=elev, azim=45)
            
            # Set labels and title
            ax.set_xlabel('X Position')
            ax.set_ylabel('Y Position')
            ax.set_zlabel('Correlation')
            
            # Set z-axis limits to 0-1
            ax.set_zlim(0, 1)
            
            # Customize z-axis ticks
            ax.set_zticks([ 0, 0.25, 0.5, 0.75, 1.0])
            
            view_label = "Side" if elev == 0 else "45°" if elev == 45 else "Top"
            ax.set_title(f'{view_label} View')
            
            # Add concentric circles to represent cell radii zones on top view only
            if elev == 90:  # Only for top view
                # Get the z-coordinate for drawing the circles (slightly above the surface)
                z_level = 0.01  # Just above z=0 plane
                
                # Draw selected concentric circles at radii 1, 3, 5, 7, 9, 11
                for radius_num in [1, 3, 5, 7, 9, 11]:
                    radius_pixels = (self.cell_diameter/2) * radius_num
                    # Create circle points
                    theta = np.linspace(0, 2*np.pi, 100)
                    circle_x = center_x + radius_pixels * np.cos(theta)
                    circle_y = center_y + radius_pixels * np.sin(theta)
                    circle_z = np.ones_like(theta) * z_level
                    
                    # Plot circle
                    ax.plot(circle_x, circle_y, circle_z, 
                            color='black', linestyle='-', linewidth=0.8, alpha=0.5)
                    
                    # Add radius number label at 45 degrees
                    label_x = center_x + radius_pixels * np.cos(np.pi/4)
                    label_y = center_y + radius_pixels * np.sin(np.pi/4)
                    ax.text(label_x, label_y, z_level, f'{radius_num}', 
                            color='black', fontsize=8, ha='center', va='center')
                
                # Add a legend explaining the circles
                ax.text2D(0.02, 0.02, "Circles: Cell radii zones (1, 3, 5, 7, 9, 11)", 
                         transform=ax.transAxes, fontsize=8, color='black')
        
        # Add overall title and colorbar
        plt.suptitle(f'Day {day} - {cs_type.replace("cs_", "CS")} Correlation Surface\n(Limited to 11 cell radii)', y=1.05)
        fig.colorbar(surf, ax=fig.axes, shrink=0.5, aspect=5, ticks=[ 0, 0.25, 0.5, 0.75, 1.0])
        
        # Save plot
        cs_label = cs_type.replace('cs_plus', 'CSplus').replace('cs_minus', 'CSminus')
        save_path = os.path.join(self.output_dir, f'correlation_3d_{day}_{cs_label}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved visualization to: {save_path}")
        plt.show()

    def process_days(self, days):
        """Process and visualize data for multiple days."""
        for day in days:
            print(f"\nProcessing day: {day}")
            try:
                # Load data
                data = self.load_pickle_data(day)
                
                # Create visualizations for both CS types
                for cs_type in ['cs_plus', 'cs_minus']:
                    print(f"Creating visualization for {cs_type}")
                    self.create_3d_correlation_plots(day, cs_type, data[cs_type])
                    
            except Exception as e:
                print(f"Error processing day {day}: {str(e)}")
                continue
                
        print("\nProcessing complete!")

# Example usage:
if __name__ == "__main__":
    # Initialize visualizer with your base directory
    basedir = basedir
    
    # Create visualizer instance
    visualizer = MaskedCorrelation3DVisualizer(basedir, df_averaged)
    
    # Process all experimental days
    days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    visualizer.process_days(days)

# Stats and Graphs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle
import torch
import roifile
from scipy import stats
import seaborn as sns
from matplotlib.patches import Circle
from tqdm import tqdm
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

class EnhancedDonutAnalyzer:
    def __init__(self, basedir, df_averaged):
        """Initialize the analyzer with base settings."""
        self.basedir = basedir
        self.df_averaged = df_averaged
        
        # Set up directories
        self.results_dir = os.path.join(self.basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'correlation_pickles')
        self.donut_dir = os.path.join(self.results_dir, 'donut_analysis')
        os.makedirs(self.donut_dir, exist_ok=True)
        
        # Set up device
        if torch.backends.mps.is_available():
            self.device = torch.device("mps")
        elif torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        print(f"Using device: {self.device}")
        
        # Define radius factors and colors
        # self.radius_factors = [1, 3, 5, 7, 9, 11, 13, 15]
        self.radius_factors = [1, 3, 5, 7, 9, 11]
        self.zone_colors = plt.cm.tab20(np.linspace(0, 1, len(self.radius_factors)))
        
        # Calculate global cell diameter once
        self.cell_diameter = self.calculate_global_cell_diameter()
        print(f"Global average cell diameter: {self.cell_diameter:.2f} pixels")

    def calculate_global_cell_diameter(self):
        """Calculate average cell diameter across all FOVs and days."""
        print("\nCalculating global cell diameter...")
        all_areas = []
        combinations = self.df_averaged.groupby(['day', 'animal', 'fov']).size().reset_index()
        
        for _, row in tqdm(combinations.iterrows(), total=len(combinations)):
            try:
                roi_folder = os.path.join(self.basedir, row['day'], row['animal'], row['fov'])
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                
                if roi_files:
                    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
                    for roi in rois:
                        coords = roi.coordinates()
                        x, y = coords[:, 0], coords[:, 1]
                        area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                        all_areas.append(area)
            except Exception as e:
                print(f"Error processing {roi_folder}: {str(e)}")
                continue
        
        mean_area = np.mean(all_areas)
        return 2 * np.sqrt(mean_area / np.pi)

    def create_donut_masks(self, grid_size=(1024, 1024), center=(512, 512)):
        """Create donut-shaped masks for all radius factors."""
        y, x = np.ogrid[:grid_size[0], :grid_size[1]]
        distances = np.sqrt((x - center[0])**2 + (y - center[1])**2)
        
        masks = []
        for i, factor in enumerate(self.radius_factors):
            radius = (self.cell_diameter/2) * factor
            if i == 0:
                mask = distances <= radius
            else:
                inner_radius = (self.cell_diameter/2) * self.radius_factors[i-1]
                mask = (distances > inner_radius) & (distances <= radius)
            masks.append(mask)
        
        return masks

    def analyze_pickle_data(self, day):
        """Analyze correlation data for a single day."""
        pickle_path = os.path.join(self.pickle_dir, f"correlation_data_{day}.pkl")
        if not os.path.exists(pickle_path):
            print(f"No pickle file found for {day}")
            return None
            
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)
        
        # Initialize results storage
        results_data = []
        masks = self.create_donut_masks()
        
        # Process each FOV
        for animal_fov, fov_data in data.items():
            animal, fov = animal_fov.split('_', 1)
            
            for cs_type in ['cs_plus', 'cs_minus']:
                # Calculate average correlation map
                accum = fov_data[cs_type]['accum']
                count = fov_data[cs_type]['count']
                with np.errstate(divide='ignore', invalid='ignore'):
                    corr_map = np.where(count > 0, accum / count, np.nan)
                
                # Calculate average for each donut zone
                for zone_idx, mask in enumerate(masks):
                    zone_avg = np.nanmean(corr_map[mask])
                    if not np.isnan(zone_avg):  # Only add non-NaN values
                        results_data.append({
                            'day': day,
                            'animal': animal,
                            'fov': fov,
                            'cs_type': 'CS+' if cs_type == 'cs_plus' else 'CS-',
                            'zone_radius': self.radius_factors[zone_idx],
                            'correlation': zone_avg
                        })
        
        return pd.DataFrame(results_data)

    def perform_statistical_analysis(self, results_df):
        """Perform comprehensive statistical analysis."""
        print("\nPerforming Statistical Analysis")
        print("=" * 50)
        
        # Create a copy of the dataframe to avoid modifying the original
        df = results_df.copy()
        
        # Convert to categories
        df['zone_radius'] = df['zone_radius'].astype('category')
        df['day'] = df['day'].astype('category')
        df['cs_type'] = df['cs_type'].astype('category')
        df['group_id'] = df['animal'] + '_' + df['fov']
        
        try:
            # Fit mixed-effects model with simplified formula
            model = sm.MixedLM.from_formula(
                "correlation ~ C(zone_radius) + C(day) + C(cs_type)",
                groups="group_id",
                data=df
            )
            
            results = model.fit()
            print("\nMixed-Effects Model Results:")
            print(results.summary())
        except Exception as e:
            print(f"Error in mixed model: {str(e)}")
            results = None
        
        # Perform ANOVAs for main effects
        print("\nANOVA Results:")
        
        # Zone radius effects
        radius_groups = [group['correlation'].values for name, group in df.groupby('zone_radius')]
        radius_groups = [group for group in radius_groups if len(group) > 0]  # Remove empty groups
        radius_stats = stats.f_oneway(*radius_groups)
        print("\n1. Effect of Zone Radius:")
        print(f"F-statistic: {radius_stats.statistic:.3f}")
        print(f"p-value: {radius_stats.pvalue:.3f}")
        
        # Day effects
        day_groups = [group['correlation'].values for name, group in df.groupby('day')]
        day_groups = [group for group in day_groups if len(group) > 0]  # Remove empty groups
        day_stats = stats.f_oneway(*day_groups)
        print("\n2. Effect of Day:")
        print(f"F-statistic: {day_stats.statistic:.3f}")
        print(f"p-value: {day_stats.pvalue:.3f}")
        
        # CS type effects
        cs_type_ttest = stats.ttest_ind(
            df[df['cs_type'] == 'CS+']['correlation'],
            df[df['cs_type'] == 'CS-']['correlation']
        )
        print("\n3. Effect of CS Type:")
        print(f"t-statistic: {cs_type_ttest.statistic:.3f}")
        print(f"p-value: {cs_type_ttest.pvalue:.3f}")
        
        # Perform post-hoc tests
        print("\nPost-hoc Analyses (Tukey HSD):")
        
        # Zone radius comparisons
        radius_tukey = pairwise_tukeyhsd(
            df['correlation'],
            df['zone_radius']
        )
        print("\nZone Radius Comparisons:")
        print(radius_tukey)
        
        # Day comparisons
        day_tukey = pairwise_tukeyhsd(
            df['correlation'],
            df['day']
        )
        print("\nDay Comparisons:")
        print(day_tukey)
        
        # Create summary statistics
        summary_stats = df.groupby(['day', 'zone_radius', 'cs_type'])['correlation'].agg([
            'mean', 'std', 'count'
        ]).reset_index()
        
        print("\nSummary Statistics:")
        print(summary_stats)
        
        return {
            'mixed_model': results,
            'radius_stats': radius_stats,
            'day_stats': day_stats,
            'cs_type_ttest': cs_type_ttest,
            'radius_tukey': radius_tukey,
            'day_tukey': day_tukey,
            'summary_stats': summary_stats
        }

    
    # def create_summary_plots(self, results_df):
    #     """Create summary visualization plots."""
    #     # Create figure for summary plots
    #     plt.figure(figsize=(20, 15))
        
    #     # 1. Zone radius effects
    #     plt.subplot(221)
    #     sns.boxplot(data=results_df, x='zone_radius', y='correlation', hue='cs_type')
    #     plt.title('Correlation by Zone Radius and CS Type')
    #     plt.xlabel('Zone Radius (× cell diameter)')
    #     plt.ylabel('Correlation')
        
    #     # 2. Day effects
    #     plt.subplot(222)
    #     sns.boxplot(data=results_df, x='day', y='correlation', hue='cs_type')
    #     plt.title('Correlation by Day and CS Type')
    #     plt.xticks(rotation=45)
        
    #     # 3. Interaction plot
    #     plt.subplot(223)
    #     sns.lineplot(data=results_df, x='zone_radius', y='correlation', 
    #                 hue='day', style='cs_type', markers=True, dashes=False)
    #     plt.title('Day × Zone Radius Interaction')
        
    #     # 4. Distribution plot
    #     plt.subplot(224)
    #     sns.violinplot(data=results_df, x='cs_type', y='correlation', hue='day')
    #     plt.title('Correlation Distribution by CS Type and Day')
        
    #     plt.tight_layout()
    #     plt.savefig(os.path.join(self.donut_dir, 'summary_plots.png'), dpi=300, bbox_inches='tight')
    #     plt.show()

    def create_summary_plots(self, results_df):
        """Create summary visualization plots using pure matplotlib."""
        # Create a copy of the dataframe
        df = results_df.copy()
        
        # Create figure for summary plots
        plt.figure(figsize=(20, 15))
        
        # 1. Zone radius effects as box plot
        plt.subplot(221)
        
        # Get unique values for grouping
        radius_values = sorted(df['zone_radius'].unique())
        cs_types = ['CS+', 'CS-']
        colors = ['#1f77b4', '#ff7f0e']  # Blue and orange
        
        # Setup positions for boxes
        positions = []
        box_data = []
        for i, radius in enumerate(radius_values):
            for j, cs in enumerate(cs_types):
                positions.append(i + j*0.4 - 0.2)
                data = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
                box_data.append(data)
        
        # Create box plot
        boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.3)
        
        # Color the boxes
        for i, box in enumerate(boxes['boxes']):
            box.set(facecolor=colors[i % 2])
        
        # Set x-ticks and labels
        plt.xticks([i for i in range(len(radius_values))], [str(r) for r in radius_values])
        plt.title('Correlation by Zone Radius and CS Type')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        
        # Add legend
        plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
        # 2. Day effects as box plot
        plt.subplot(222)
        
        # Get unique days
        days = sorted(df['day'].unique())
        
        # Setup positions for boxes
        positions = []
        box_data = []
        for i, day in enumerate(days):
            for j, cs in enumerate(cs_types):
                positions.append(i + j*0.4 - 0.2)
                data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                box_data.append(data)
        
        # Create box plot
        boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.3)
        
        # Color the boxes
        for i, box in enumerate(boxes['boxes']):
            box.set(facecolor=colors[i % 2])
        
        # Set x-ticks and labels
        plt.xticks([i for i in range(len(days))], days, rotation=45)
        plt.title('Correlation by Day and CS Type')
        
        # Add legend
        plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
        # 3. Interaction plot - Use a completely different approach with pure matplotlib
        plt.subplot(223)
                
        # First convert problematic categorical columns to strings to avoid indexing issues
        df['zone_radius_str'] = df['zone_radius'].astype(str)
        df['day_str'] = df['day']
                
        # Get unique values
        days = sorted(df['day_str'].unique())
        cs_types = ['CS+', 'CS-']
        zone_radii = sorted(df['zone_radius'].unique())
                
        # Set up colors and line styles
        day_colors = plt.cm.tab10(np.linspace(0, 1, len(days)))
        line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
        markers = ['o', 's']       # Circle for CS+, square for CS-
                
        # Instead of using groupby, manually calculate means
        for i, day in enumerate(days):
            for j, cs_type in enumerate(cs_types):
                # Filter data for this day and CS type
                mask = (df['day_str'] == day) & (df['cs_type'] == cs_type)
                day_cs_data = df[mask]
                        
                # Calculate mean for each zone radius
                means = []
                for radius in zone_radii:
                    radius_mask = day_cs_data['zone_radius'] == radius
                    if radius_mask.any():
                        means.append(day_cs_data.loc[radius_mask, 'correlation'].mean())
                    else:
                        means.append(np.nan)
                        
                # Plot this line
                plt.plot(zone_radii, means, 
                         color=day_colors[i], 
                         linestyle=line_styles[j],
                         marker=markers[j], 
                         label=f"{day}, {cs_type}")
                
        plt.title('Day × Zone Radius Interaction')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # 4. Distribution as violin plot (using matplotlib's violin plot)
        plt.subplot(224)
        
        # Prepare data for violin plots
        violin_positions = []
        violin_data = []
        
        for j, cs in enumerate(cs_types):
            for i, day in enumerate(days):
                pos = j + i/(len(days)+1)
                violin_positions.append(pos)
                data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                violin_data.append(data)
        
        # Create violin plot
        violin_parts = plt.violinplot(violin_data, positions=violin_positions, 
                                    widths=0.15, showmeans=True, showextrema=True)
        
        # Customize violin colors
        for i, pc in enumerate(violin_parts['bodies']):
            day_idx = i % len(days)
            pc.set_facecolor(day_colors[day_idx])
            pc.set_alpha(0.7)
        
        # Set x-ticks
        plt.xticks([0.5, 1.5], cs_types)
        plt.title('Correlation Distribution by CS Type and Day')
        plt.ylabel('Correlation')
        
        # Add legend for violin plot
        legend_patches = [plt.Line2D([0], [0], color=day_colors[i], lw=4) for i in range(len(days))]
        plt.legend(legend_patches, days, loc='upper right')
        
        plt.tight_layout()
        # plt.savefig(os.path.join(self.donut_dir, 'AllDays_summary_plots.pdf'), dpi=300, bbox_inches='tight')
        plt.show()

    def save_statistical_results(self, stats_results, results_path):
        """Save statistical results in a pickle-friendly format."""
        pickle_friendly_results = {
            'radius_stats': {
                'statistic': float(stats_results['radius_stats'].statistic),
                'pvalue': float(stats_results['radius_stats'].pvalue)
            },
            'day_stats': {
                'statistic': float(stats_results['day_stats'].statistic),
                'pvalue': float(stats_results['day_stats'].pvalue)
            },
            'cs_type_ttest': {
                'statistic': float(stats_results['cs_type_ttest'].statistic),
                'pvalue': float(stats_results['cs_type_ttest'].pvalue)
            },
            'radius_tukey': {
                'meandiffs': stats_results['radius_tukey'].meandiffs.tolist(),
                'confint': stats_results['radius_tukey'].confint.tolist(),
                'reject': stats_results['radius_tukey'].reject.tolist()
            },
            'day_tukey': {
                'meandiffs': stats_results['day_tukey'].meandiffs.tolist(),
                'confint': stats_results['day_tukey'].confint.tolist(),
                'reject': stats_results['day_tukey'].reject.tolist()
            },
            'summary_stats': stats_results['summary_stats'].to_dict()
        }
        
        # If mixed model was successful, add its results
        if stats_results['mixed_model'] is not None:
            pickle_friendly_results['mixed_model'] = {
                'params': stats_results['mixed_model'].params.to_dict(),
                'pvalues': stats_results['mixed_model'].pvalues.to_dict(),
                'conf_int': stats_results['mixed_model'].conf_int().to_dict()
            }
        
        with open(results_path, 'wb') as f:
            pickle.dump(pickle_friendly_results, f)

    def run_analysis(self, days=None):
        """Run complete analysis pipeline."""
        if days is None:
            days = self.df_averaged['day'].unique()
            
        # Collect results from all days
        all_results = []
        for day in days:
            print(f"\nProcessing {day}...")
            day_results = self.analyze_pickle_data(day)
            if day_results is not None:
                all_results.append(day_results)
        
        # Combine all results
        if all_results:
            combined_results = pd.concat(all_results, ignore_index=True)
            
            # Perform statistical analysis
            stats_results = self.perform_statistical_analysis(combined_results)
            
            # Create summary plots
            self.create_summary_plots(combined_results)
            
            # Save results
            results_path = os.path.join(self.donut_dir, 'analysis_results.pkl')
            data_path = os.path.join(self.donut_dir, 'combined_data.csv')
            
            # Save the combined data as CSV
            combined_results.to_csv(data_path, index=False)
            
            # Save statistical results
            self.save_statistical_results(stats_results, results_path)
            
            print(f"\nResults saved:")
            print(f"Data: {data_path}")
            print(f"Statistics: {results_path}")
            
            return combined_results, stats_results
        
        return None, None

def main():
    """Main function to run the analysis."""
    # Set up paths
    # basedir =  r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
    basedir = r'/Users/MAC/Offline/Jackie/WholeCell2 Final'
    try:
        # Initialize analyzer with your existing df_averaged
        analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
        
        # Run analysis for all days
        days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
        results_df, stats_results = analyzer.run_analysis(days)
        
        if results_df is not None:
            print("\nAnalysis complete!")
            print("Results saved in:", analyzer.donut_dir)
        else:
            print("\nNo results were generated. Please check for errors.")
            
    except Exception as e:
        print(f"\nError during analysis: {str(e)}")

if __name__ == "__main__":
    main()


# Split Data 

In [ ]:
###Jackie to split up the days into groupings so that it is early v late, late v omissions, and reversal
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

class GroupedDonutAnalyzer:
    def __init__(self, donut_analyzer, results_df=None):
        """
        Initialize with an existing EnhancedDonutAnalyzer instance and optional results.
        
        Args:
            donut_analyzer: An instance of EnhancedDonutAnalyzer with pre-calculated data
            results_df: Optional DataFrame with results from the original analysis
        """
        self.donut_analyzer = donut_analyzer
        self.basedir = donut_analyzer.basedir
        self.donut_dir = donut_analyzer.donut_dir
        self.grouped_dir = os.path.join(self.donut_dir, 'grouped_analysis')
        os.makedirs(self.grouped_dir, exist_ok=True)
        
        # Define day groupings
        self.day_groups = [
            {'name': 'Acquisition', 'days': ['0_Early', '1_Late']},
            {'name': 'Extinction', 'days': ['1_Late', '2_SucOmit', '3_CueOmit']},
            {'name': 'Reversal', 'days': ['4_RevEarly', '5_RevLate']}
        ]
        
        # Handle results data
        if results_df is not None:
            # Use provided DataFrame directly if available
            self.results_df = results_df
        else:
            # If not provided, check if CSV exists (for backward compatibility)
            data_path = os.path.join(self.donut_dir, 'combined_data.csv')
            if os.path.exists(data_path):
                self.results_df = pd.read_csv(data_path)
                print(f"Loaded data from {data_path}")
            else:
                # No CSV, so we need to regenerate the data
                print("No pre-existing results found. Regenerating analysis data...")
                # Check if donut_analyzer has the necessary methods to regenerate data
                if hasattr(donut_analyzer, 'analyze_pickle_data') and hasattr(donut_analyzer, 'df_averaged'):
                    all_results = []
                    days = donut_analyzer.df_averaged['day'].unique()
                    for day in days:
                        print(f"Processing {day}...")
                        day_results = donut_analyzer.analyze_pickle_data(day)
                        if day_results is not None:
                            all_results.append(day_results)
                    
                    if all_results:
                        self.results_df = pd.concat(all_results, ignore_index=True)
                    else:
                        raise ValueError("Failed to generate analysis data.")
                else:
                    raise ValueError("Cannot regenerate analysis data. Please run original analysis first.")
                    
        # Save the combined data as CSV for future use
        data_path = os.path.join(self.donut_dir, 'combined_data.csv')
        self.results_df.to_csv(data_path, index=False)
        print(f"Saved combined data to: {data_path}")
            
        print(f"GroupedDonutAnalyzer initialized with {len(self.results_df)} data points")
        print(f"Results will be saved to: {self.grouped_dir}")
    
    def filter_data_by_days(self, days):
        """Filter the results DataFrame to include only specified days."""
        return self.results_df[self.results_df['day'].isin(days)].copy()
    
    def perform_grouped_statistical_analysis(self, group_df, group_name):
        """
        Perform statistical analysis on a specific day grouping.
        
        Args:
            group_df: DataFrame containing only the days in this group
            group_name: Name of this group for reporting and file naming
        
        Returns:
            Dictionary of statistical results
        """
        print(f"\nPerforming Statistical Analysis for {group_name}")
        print("=" * 50)
        
        # Create a copy of the dataframe to avoid modifying the original
        df = group_df.copy()
        
        # Convert to categories
        df['zone_radius'] = df['zone_radius'].astype('category')
        df['day'] = df['day'].astype('category')
        df['cs_type'] = df['cs_type'].astype('category')
        df['group_id'] = df['animal'] + '_' + df['fov']
        
        # Store results
        stats_results = {}
        
        # Run mixed-effects model
        try:
            model = sm.MixedLM.from_formula(
                "correlation ~ C(zone_radius) + C(day) + C(cs_type)",
                groups="group_id",
                data=df
            )
            
            results = model.fit()
            print("\nMixed-Effects Model Results:")
            print(results.summary())
            stats_results['mixed_model'] = results
        except Exception as e:
            print(f"Error in mixed model: {str(e)}")
            stats_results['mixed_model'] = None
        
        # If there are only two days, perform a two-way ANOVA
        unique_days = df['day'].unique()
        if len(unique_days) == 2:
            print("\nPerforming Two-way ANOVA (Day × CS Type)")
            try:
                # Fit the two-way ANOVA model
                model = ols('correlation ~ C(day) + C(cs_type) + C(day):C(cs_type)', data=df).fit()
                anova_table = sm.stats.anova_lm(model, typ=2)
                print("\nTwo-way ANOVA Results:")
                print(anova_table)
                stats_results['two_way_anova'] = anova_table
            except Exception as e:
                print(f"Error in two-way ANOVA: {str(e)}")
                stats_results['two_way_anova'] = None
        
        # Perform ANOVAs for main effects
        print("\nANOVA Results:")
        
        # Zone radius effects
        radius_groups = [group['correlation'].values for name, group in df.groupby('zone_radius')]
        radius_groups = [group for group in radius_groups if len(group) > 0]  # Remove empty groups
        radius_stats = stats.f_oneway(*radius_groups)
        print("\n1. Effect of Zone Radius:")
        print(f"F-statistic: {radius_stats.statistic:.3f}")
        print(f"p-value: {radius_stats.pvalue:.3f}")
        stats_results['radius_stats'] = radius_stats
        
        # Day effects
        day_groups = [group['correlation'].values for name, group in df.groupby('day')]
        day_groups = [group for group in day_groups if len(group) > 0]  # Remove empty groups
        day_stats = stats.f_oneway(*day_groups)
        print("\n2. Effect of Day:")
        print(f"F-statistic: {day_stats.statistic:.3f}")
        print(f"p-value: {day_stats.pvalue:.3f}")
        stats_results['day_stats'] = day_stats
        
        # CS type effects - Using .values to convert to numpy array
        cs_plus_vals = df[df['cs_type'] == 'CS+']['correlation'].values
        cs_minus_vals = df[df['cs_type'] == 'CS-']['correlation'].values
        cs_type_ttest = stats.ttest_ind(cs_plus_vals, cs_minus_vals)
        print("\n3. Effect of CS Type:")
        print(f"t-statistic: {cs_type_ttest.statistic:.3f}")
        print(f"p-value: {cs_type_ttest.pvalue:.3f}")
        stats_results['cs_type_ttest'] = cs_type_ttest
        
        # Perform post-hoc tests
        print("\nPost-hoc Analyses (Tukey HSD):")
        
        # Zone radius comparisons - Using .values to convert to numpy arrays
        radius_tukey = pairwise_tukeyhsd(
            df['correlation'].values,
            df['zone_radius'].values
        )
        print("\nZone Radius Comparisons:")
        print(radius_tukey)
        stats_results['radius_tukey'] = radius_tukey
        
        # Day comparisons (only if more than one day)
        if len(unique_days) > 1:
            day_tukey = pairwise_tukeyhsd(
                df['correlation'].values,
                df['day'].values
            )
            print("\nDay Comparisons:")
            print(day_tukey)
            stats_results['day_tukey'] = day_tukey
        
        # Create summary statistics
        summary_stats = df.groupby(['day', 'zone_radius', 'cs_type'])['correlation'].agg([
            'mean', 'std', 'count'
        ]).reset_index()
        
        print("\nSummary Statistics:")
        print(summary_stats)
        stats_results['summary_stats'] = summary_stats
        
        return stats_results
    
    # def create_grouped_summary_plots(self, group_df, group_name):
    #     """
    #     Create summary visualization plots for a specific day grouping.
        
    #     Args:
    #         group_df: DataFrame containing only the days in this group
    #         group_name: Name of this group for reporting and file naming
    #     """
    #     # Create a copy of the dataframe
    #     df = group_df.copy()
        
    #     # Create figure for summary plots
    #     plt.figure(figsize=(20, 15))
        
    #     # 1. Zone radius effects as box plot with strip plot overlay
    #     plt.subplot(221)
        
    #     # Get unique values for grouping
    #     radius_values = sorted(df['zone_radius'].unique())
    #     cs_types = ['CS+', 'CS-']
    #     colors = ['purple', 'grey']  # Blue and orange
        
    #     # Setup positions for boxes
    #     positions = []
    #     box_data = []
    #     for i, radius in enumerate(radius_values):
    #         for j, cs in enumerate(cs_types):
    #             positions.append(i + j*0.4 - 0.2)
    #             data = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
    #             box_data.append(data)
        
    #     # Create box plot
    #     boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.25, 
    #                showfliers=False, showcaps=False, showmeans=False, showbox=True, 
    #                whiskerprops={'visible': False})
        
    #     # Color the boxes
    #     for i, box in enumerate(boxes['boxes']):
    #         box.set(facecolor=colors[i % 2])
        
    #     # Add strip plot points
    #     for i, radius in enumerate(radius_values):
    #         for j, cs in enumerate(cs_types):
    #             pos = i + j*0.4 - 0.2
    #             points = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
    #             x_jitter = np.random.normal(pos, 0.00, size=len(points))
    #             plt.scatter(x_jitter, points, color=colors[j], alpha=0.25, s=60)
        
    #     # Set x-ticks and labels
    #     plt.xticks([i for i in range(len(radius_values))], [str(r) for r in radius_values])
    #     plt.title(f'{group_name}: Correlation by Zone Radius and CS Type')
    #     plt.xlabel('Zone Radius (× cell diameter)')
    #     plt.ylabel('Correlation')
        
    #     # Add legend
    #     plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
    #     # 2. Day effects as box plot with strip plot overlay
    #     plt.subplot(222)
        
    #     # Get unique days
    #     days = sorted(df['day'].unique())
        
    #     # Setup positions for boxes
    #     positions = []
    #     box_data = []
    #     for i, day in enumerate(days):
    #         for j, cs in enumerate(cs_types):
    #             positions.append(i + j*0.4 - 0.2)
    #             data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
    #             box_data.append(data)
        
        
    #     # Create box plot without whiskers
    #     boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.3, 
    #                showfliers=False, showcaps=False, showmeans=False, showbox=True, 
    #                whiskerprops={'visible': False})
        
    #     # Color the boxes
    #     for i, box in enumerate(boxes['boxes']):
    #         box.set(facecolor=colors[i % 2])
        
    #     # Add strip plot points
    #     for i, day in enumerate(days):
    #         for j, cs in enumerate(cs_types):
    #             pos = i + j*0.4 - 0.2
    #             points = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
    #             x_jitter = np.random.normal(pos, 0.0, size=len(points))
    #             plt.scatter(x_jitter, points, color=colors[j], alpha=0.3, s=60)
        
    #     # Set x-ticks and labels
    #     plt.xticks([i for i in range(len(days))], days, rotation=45)
    #     plt.title(f'{group_name}: Correlation by Day and CS Type')
        
    #     # Add legend
    #     plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
    #     # 3. Interaction plot - Line plot with markers
    #     plt.subplot(223)
        
    #     # First convert problematic categorical columns to strings to avoid indexing issues
    #     df['zone_radius_str'] = df['zone_radius'].astype(str)
    #     df['day_str'] = df['day']
        
    #     # Get unique values
    #     zone_radii = sorted(df['zone_radius'].unique())
        
    #     # Set up colors and line styles
    #     day_colors = plt.cm.tab10(np.linspace(0, 1, len(days)))
    #     line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
    #     markers = ['o', 's']       # Circle for CS+, square for CS-
        
    #     # Instead of using seaborn, manually calculate means and plot
    #     for i, day in enumerate(days):
    #         for j, cs_type in enumerate(cs_types):
    #             # Filter data for this day and CS type
    #             mask = (df['day_str'] == day) & (df['cs_type'] == cs_type)
    #             day_cs_data = df[mask]
                
    #             # Calculate mean for each zone radius
    #             means = []
    #             for radius in zone_radii:
    #                 radius_mask = day_cs_data['zone_radius'] == radius
    #                 if radius_mask.any():
    #                     means.append(day_cs_data.loc[radius_mask, 'correlation'].mean())
    #                 else:
    #                     means.append(np.nan)
                
    #             # Plot this line
    #             plt.plot(zone_radii, means, 
    #                      color=day_colors[i], 
    #                      linestyle=line_styles[j],
    #                      marker=markers[j], 
    #                      label=f"{day}, {cs_type}")
        
    #     plt.title(f'{group_name}: Day × Zone Radius Interaction')
    #     plt.xlabel('Zone Radius (× cell diameter)')
    #     plt.ylabel('Correlation')
    #     plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
    #     # 4. Distribution as violin plot (using matplotlib's violin plot)
    #     plt.subplot(224)
        
    #     # Prepare data for violin plots
    #     violin_positions = []
    #     violin_data = []
        
    #     for j, cs in enumerate(cs_types):
    #         for i, day in enumerate(days):
    #             pos = j + i/(len(days)+1)
    #             violin_positions.append(pos)
    #             data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
    #             violin_data.append(data)
        
    #     # Create violin plot
    #     violin_parts = plt.violinplot(violin_data, positions=violin_positions, 
    #                                 widths=0.15, showmeans=True, showextrema=True)
        
    #     # Customize violin colors
    #     for i, pc in enumerate(violin_parts['bodies']):
    #         day_idx = i % len(days)
    #         pc.set_facecolor(day_colors[day_idx])
    #         pc.set_alpha(0.7)
        
    #     # Set x-ticks
    #     plt.xticks([0.5, 1.5], cs_types)
    #     plt.title(f'{group_name}: Correlation Distribution by CS Type and Day')
    #     plt.ylabel('Correlation')
        
    #     # Add legend for violin plot
    #     legend_patches = [plt.Line2D([0], [0], color=day_colors[i], lw=4) for i in range(len(days))]
    #     plt.legend(legend_patches, days, loc='upper right')
        
    #     plt.tight_layout()
    #     plt.savefig(os.path.join(self.grouped_dir, f'{group_name}_summary_plots.pdf'), 
    #                dpi=300, bbox_inches='tight')
    #     plt.show()
    def create_grouped_summary_plots(self, group_df, group_name):
        """
        Create summary visualization plots for a specific day grouping.
        
        Args:
            group_df: DataFrame containing only the days in this group
            group_name: Name of this group for reporting and file naming
        """
        # Create a copy of the dataframe
        df = group_df.copy()
        
        # Create figure for summary plots
        plt.figure(figsize=(20, 15))
        
        # 1. Zone radius effects as box plot with strip plot overlay
        plt.subplot(221)
        
        # Get unique values for grouping
        radius_values = sorted(df['zone_radius'].unique())
        cs_types = ['CS+', 'CS-']
        colors = ['purple', 'grey']  # Purple and grey
        
        # Setup positions for boxes
        positions = []
        box_data = []
        for i, radius in enumerate(radius_values):
            for j, cs in enumerate(cs_types):
                positions.append(i + j*0.4 - 0.2)
                data = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
                box_data.append(data)
        
        # Create box plot
        boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.25, 
                   showfliers=False, showcaps=False, showmeans=False, showbox=True, 
                   whiskerprops={'visible': False})
        
        # Color the boxes
        for i, box in enumerate(boxes['boxes']):
            box.set(facecolor=colors[i % 2])

        #  # Change median line color to white (or use 'black' instead)
        # for median in box['medians']:
        #     median.set_color('black')
        #     median.set_linewidth(2)  # Optional: make the line thicker for better visibility
        
        # Add strip plot points
        for i, radius in enumerate(radius_values):
            for j, cs in enumerate(cs_types):
                pos = i + j*0.4 - 0.2
                points = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
                x_jitter = np.random.normal(pos, 0.01, size=len(points))
                plt.scatter(x_jitter, points, color=colors[j], alpha=0.25, s=60)
        
        # Set x-ticks and labels
        plt.xticks([i for i in range(len(radius_values))], [str(r) for r in radius_values])
        plt.title(f'{group_name}: Correlation by Zone Radius and CS Type')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        
        # Add legend
        plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
        # 2. Day effects as box plot with strip plot overlay
        plt.subplot(222)
        
        # Get unique days
        days = sorted(df['day'].unique())
        
        # Setup positions for boxes
        positions = []
        box_data = []
        for i, day in enumerate(days):
            for j, cs in enumerate(cs_types):
                positions.append(i + j*0.4 - 0.2)
                data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                box_data.append(data)
        
        
        # Create box plot without whiskers
        boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.3, 
                   showfliers=False, showcaps=False, showmeans=False, showbox=True, 
                   whiskerprops={'visible': False})
        
        # Color the boxes
        for i, box in enumerate(boxes['boxes']):
            box.set(facecolor=colors[i % 2])

         # Change median line color to white (or use 'black' instead)
        for median in box['medians']:
            median.set_color('black')
            median.set_linewidth(2)  # Optional: make the line thicker for better visibility

        # Add strip plot points
        for i, day in enumerate(days):
            for j, cs in enumerate(cs_types):
                pos = i + j*0.4 - 0.2
                points = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                x_jitter = np.random.normal(pos, 0.01, size=len(points))
                plt.scatter(x_jitter, points, color=colors[j], alpha=0.3, s=60)
        
        # Set x-ticks and labels
        plt.xticks([i for i in range(len(days))], days, rotation=45)
        plt.title(f'{group_name}: Correlation by Day and CS Type')
        
        # Add legend
        plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
        # 3. Interaction plot - Line plot with markers AND SEM bands
        plt.subplot(223)
        
        # First convert problematic categorical columns to strings to avoid indexing issues
        df['zone_radius_str'] = df['zone_radius'].astype(str)
        df['day_str'] = df['day']
        
        # Get unique values
        zone_radii = sorted(df['zone_radius'].unique())
        
        # Define color schemes for CS+ (purples) and CS- (greys)
        # For CS+: lighter purple for early days, darker for late days
        # For CS-: lighter grey for early days, darker for late days
        
        # Create color maps for days based on their order (earlier days get lighter colors)
        day_indices = {day: i for i, day in enumerate(days)}
        
        # Function to get color based on day and CS type
        def get_color(day, cs_type):
            day_idx = day_indices[day]
            day_factor = day_idx / max(1, len(days) - 1)  # Normalized 0-1
            
            if cs_type == 'CS+':
                # Purple gradient from light to dark
                # Start with light purple #D8BFD8 to dark purple #4B0082
                r = int(216 - 149 * day_factor)  # 216 -> 67
                g = int(191 - 127 * day_factor)  # 191 -> 64
                b = int(216 - 130 * day_factor)  # 216 -> 86
                return f'#{r:02x}{g:02x}{b:02x}'
            else:  # CS-
                # Grey gradient from light to dark
                # Start with light grey #D3D3D3 to dark grey #696969
                val = int(211 - 106 * day_factor)  # 211 -> 105
                return f'#{val:02x}{val:02x}{val:02x}'
        
        # Set up line styles
        line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
        markers = ['o', 's']        
        
        # Instead of using seaborn, manually calculate means, SEMs, and plot
        for i, day in enumerate(days):
            for j, cs_type in enumerate(cs_types):
                # Filter data for this day and CS type
                mask = (df['day_str'] == day) & (df['cs_type'] == cs_type)
                day_cs_data = df[mask]
                
                # Calculate mean and SEM for each zone radius
                means = []
                sems = []
                for radius in zone_radii:
                    radius_mask = day_cs_data['zone_radius'] == radius
                    if radius_mask.any():
                        values = day_cs_data.loc[radius_mask, 'correlation'].values
                        means.append(np.mean(values))
                        # Calculate SEM: standard deviation / sqrt(n)
                        sems.append(np.std(values, ddof=1) / np.sqrt(len(values)))
                    else:
                        means.append(np.nan)
                        sems.append(np.nan)
                
                # Convert to numpy arrays for easier manipulation
                means = np.array(means)
                sems = np.array(sems)
                
                # Get the color based on day and CS type
                color = get_color(day, cs_type)
                
                # Plot the mean line
                plt.plot(zone_radii, means, 
                         color=color, 
                         linestyle=line_styles[j],
                         marker=markers[j], 
                         linewidth=2,
                         label=f"{day}, {cs_type}")
                
                # Add SEM bands (fill_between creates the shaded area)
                plt.fill_between(zone_radii, 
                                means - sems, 
                                means + sems, 
                                color=color, 
                                alpha=0.2)
        
        plt.title(f'{group_name}: Day × Zone Radius Interaction')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.ylim(0, 1)
        
        # 4. Distribution as violin plot (using matplotlib's violin plot)
        plt.subplot(224)
        
        # Create a day color map for violins
        violin_colors = {}
        for day in days:
            # Use the same function but with CS+ colors for violins
            violin_colors[day] = get_color(day, 'CS+')
        
        # Prepare data for violin plots
        violin_positions = []
        violin_data = []
        violin_day_colors = []
        
        for j, cs in enumerate(cs_types):
            for i, day in enumerate(days):
                pos = j + i/(len(days)+1)
                violin_positions.append(pos)
                data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                violin_data.append(data)
                violin_day_colors.append(violin_colors[day])
        
        # Create violin plot
        violin_parts = plt.violinplot(violin_data, positions=violin_positions, 
                                    widths=0.15, showmeans=True, showextrema=True)
        
        # Customize violin colors
        for i, pc in enumerate(violin_parts['bodies']):
            pc.set_facecolor(violin_day_colors[i])
            pc.set_alpha(0.7)
        
        # Set x-ticks
        plt.xticks([0.5, 1.5], cs_types)
        plt.title(f'{group_name}: Correlation Distribution by CS Type and Day')
        plt.ylabel('Correlation')
        
        # Add legend for violin plot
        legend_patches = [plt.Line2D([0], [0], color=violin_colors[day], lw=4) for day in days]
        plt.legend(legend_patches, days, loc='upper right')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.grouped_dir, f'{group_name}_summary_plots06-23-25.pdf'), 
                   dpi=300, bbox_inches='tight')
        plt.show()
    
    def save_statistical_results(self, stats_results, group_name):
        """Save statistical results in a pickle-friendly format."""
        pickle_friendly_results = {
            'radius_stats': {
                'statistic': float(stats_results['radius_stats'].statistic),
                'pvalue': float(stats_results['radius_stats'].pvalue)
            },
            'day_stats': {
                'statistic': float(stats_results['day_stats'].statistic),
                'pvalue': float(stats_results['day_stats'].pvalue)
            },
            'cs_type_ttest': {
                'statistic': float(stats_results['cs_type_ttest'].statistic),
                'pvalue': float(stats_results['cs_type_ttest'].pvalue)
            },
            'radius_tukey': {
                'meandiffs': stats_results['radius_tukey'].meandiffs.tolist(),
                'confint': stats_results['radius_tukey'].confint.tolist(),
                'reject': stats_results['radius_tukey'].reject.tolist()
            },
            'summary_stats': stats_results['summary_stats'].to_dict()
        }
        
        # Add two-way ANOVA results if they exist
        if 'two_way_anova' in stats_results and stats_results['two_way_anova'] is not None:
            # Get the actual column names from the ANOVA table
            anova_df = stats_results['two_way_anova']
            anova_columns = anova_df.columns.tolist()
            
            # Create a dictionary with all available columns
            anova_dict = {}
            for col in anova_columns:
                try:
                    # Try to convert to list, but handle non-array-like columns
                    anova_dict[col] = anova_df[col].tolist() if hasattr(anova_df[col], 'tolist') else list(anova_df[col])
                except:
                    # If conversion fails, store as is
                    anova_dict[col] = anova_df[col]
            
            pickle_friendly_results['two_way_anova'] = anova_dict
        
        # Add day Tukey results if they exist
        if 'day_tukey' in stats_results:
            pickle_friendly_results['day_tukey'] = {
                'meandiffs': stats_results['day_tukey'].meandiffs.tolist(),
                'confint': stats_results['day_tukey'].confint.tolist(),
                'reject': stats_results['day_tukey'].reject.tolist()
            }
        
        # If mixed model was successful, add its results
        if stats_results['mixed_model'] is not None:
            pickle_friendly_results['mixed_model'] = {
                'params': stats_results['mixed_model'].params.to_dict(),
                'pvalues': stats_results['mixed_model'].pvalues.to_dict(),
                'conf_int': stats_results['mixed_model'].conf_int().to_dict()
            }
        
        results_path = os.path.join(self.grouped_dir, f'{group_name}_analysis_results.pkl')
        with open(results_path, 'wb') as f:
            pickle.dump(pickle_friendly_results, f)
        print(f"Statistical results saved to: {results_path}")
    
    def run_grouped_analysis(self):
        """Run complete analysis pipeline for all grouped days."""
        for group in self.day_groups:
            group_name = group['name']
            days = group['days']
            
            print(f"\nProcessing {group_name} ({', '.join(days)})...")
            
            # Filter data for this group
            group_df = self.filter_data_by_days(days)
            
            if len(group_df) == 0:
                print(f"No data found for days: {days}")
                continue
                
            # Save group data for reference
            group_data_path = os.path.join(self.grouped_dir, f'{group_name}_data.csv')
            group_df.to_csv(group_data_path, index=False)
            print(f"Group data saved to: {group_data_path}")
            
            # Perform statistical analysis
            stats_results = self.perform_grouped_statistical_analysis(group_df, group_name)
            
            # Create summary plots
            self.create_grouped_summary_plots(group_df, group_name)
            
            # Save statistical results
            self.save_statistical_results(stats_results, group_name)
            
            print(f"{group_name} analysis complete!")
        
        print("\nAll grouped analyses complete!")
        print(f"Results saved in: {self.grouped_dir}")

def run_grouped_analysis(basedir=None, original_analyzer=None, results_df=None):
    """
    Main function to run the grouped analysis after the original analysis.
    
    Args:
        basedir: Base directory path (required if original_analyzer not provided)
        original_analyzer: Instance of EnhancedDonutAnalyzer from original analysis
        results_df: DataFrame with results from original analysis
        
    Returns:
        Instance of GroupedDonutAnalyzer or None if error occurs
    """
    try:
        if original_analyzer is not None:
            # Use the existing analyzer and optionally the results DataFrame
            grouped_analyzer = GroupedDonutAnalyzer(original_analyzer, results_df)
        else:
            if basedir is None:
                raise ValueError("Either original_analyzer or basedir must be provided")
                
            # Create a minimal adapter with the necessary properties
            class AnalyzerAdapter:
                def __init__(self, basedir):
                    self.basedir = basedir
                    self.donut_dir = os.path.join(basedir, 'Results', 'donut_analysis')
            
            adapter = AnalyzerAdapter(basedir)
            grouped_analyzer = GroupedDonutAnalyzer(adapter, results_df)
        
        # Run the grouped analysis
        grouped_analyzer.run_grouped_analysis()
        
        return grouped_analyzer
    
    except Exception as e:
        print(f"\nError during grouped analysis: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Example usage showing different ways to run the code:
if __name__ == "__main__":
    # Example 1: Run directly after EnhancedDonutAnalyzer
    basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
    # basedir = r'/Users/MAC/Offline/Jackie/WholeCell2 Final'

    analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
    results_df, stats_results = analyzer.run_analysis(days)
    grouped_analyzer = run_grouped_analysis(original_analyzer=analyzer, results_df=results_df)
    
    # Example 2: If running separately with previously saved data
    # basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
    # grouped_analyzer = run_grouped_analysis(basedir=basedir)
    
    # # Example 3: Modify to use your actual paths and execution method
    # basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
    
    # Import and create original analyzer if needed
    # from your_module import EnhancedDonutAnalyzer
    # analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
    # results_df, stats_results = analyzer.run_analysis()
    
    # Then run grouped analysis
    # grouped_analyzer = run_grouped_analysis(basedir=basedir)

# Spatial Controls

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches
import os
import roifile
import shapely.geometry
from matplotlib.path import Path
import matplotlib.path as mpath
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
import multiprocessing
from tqdm.notebook import tqdm
import torch
from torch.nn.functional import conv2d
import torch.nn as nn
import pickle

# Check if MPS (Metal Performance Shaders) is available for Mac GPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

def calculate_correlation_matrix_gpu(fluorescence_data):
    """Calculate correlation matrix using GPU acceleration."""
    # Convert to torch tensor and move to GPU
    X = torch.tensor(fluorescence_data, dtype=torch.float32).to(device)
    
    # Normalize the data
    X = (X - X.mean(dim=1, keepdim=True)) / X.std(dim=1, keepdim=True)
    
    # Calculate correlation matrix using matrix multiplication
    corr_matrix = torch.mm(X, X.t()) / X.size(1)
    
    return corr_matrix.cpu().numpy()

def create_pixel_mask_gpu(coords, grid_size, offset_x, offset_y, pixels_tensor):
    """Create pixel mask for ROIs using GPU acceleration."""
    # Transform coordinates and convert to tensor
    coords_transformed = coords + np.array([offset_x, offset_y])
    coords_tensor = torch.tensor(coords_transformed, dtype=torch.float32).to(device)
    
    # Create path and test points
    path = Path(coords_transformed)
    mask = path.contains_points(pixels_tensor.cpu().numpy())
    mask_tensor = torch.tensor(mask, dtype=torch.float32).to(device)
    
    return mask_tensor.reshape(grid_size)

class ControlNeuralCorrelationAnalyzer:
    def __init__(self, basedir, df_averaged_control, device=None):
        """Initialize the analyzer with base settings."""
        self.basedir = basedir
        self.df_averaged_control = df_averaged_control
        
        # Set up GPU device
        if device is None:
            if torch.backends.mps.is_available():
                self.device = torch.device("mps")
            elif torch.cuda.is_available():
                self.device = torch.device("cuda")
            else:
                self.device = torch.device("cpu")
        else:
            self.device = device
            
        # Create Results directory structure
        self.results_dir = os.path.join(self.basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'control_correlation_pickles')
        self.viz_dir = os.path.join(self.results_dir, 'control_correlation_visualizations')
        
        os.makedirs(self.pickle_dir, exist_ok=True)
        os.makedirs(self.viz_dir, exist_ok=True)
            
        print(f"Using device: {self.device}")
        print(f"Results directory: {self.results_dir}")
        print(f"Analysis will use CONTROL data (df_averaged_control)")

    def _get_pickle_path(self, day):
        """Generate pickle file path for a specific day."""
        return os.path.join(self.pickle_dir, f"control_correlation_data_{day}.pkl")

    def _pickle_exists(self, day):
        """Check if pickle exists for a specific day."""
        return os.path.exists(self._get_pickle_path(day))

    def _save_to_pickle(self, data, day):
        """Save correlation data to pickle."""
        pickle_path = self._get_pickle_path(day)
        # Convert torch tensors to numpy arrays for pickling
        pickle_data = {
            animal_fov: {
                cs_type: {
                    'accum': acc['accum'].cpu().numpy(),
                    'count': acc['count'].cpu().numpy()
                }
                for cs_type, acc in data_dict.items()
            }
            for animal_fov, data_dict in data.items()
        }
        with open(pickle_path, 'wb') as f:
            pickle.dump(pickle_data, f)
        print(f"\nPickle file saved: {pickle_path}")

    def _load_from_pickle(self, day):
        """Load correlation data from pickle."""
        pickle_path = self._get_pickle_path(day)
        with open(pickle_path, 'rb') as f:
            pickle_data = pickle.load(f)
        # Convert numpy arrays back to torch tensors
        loaded_data = {
            animal_fov: {
                cs_type: {
                    'accum': torch.tensor(acc['accum'], device=self.device),
                    'count': torch.tensor(acc['count'], device=self.device)
                }
                for cs_type, acc in data_dict.items()
            }
            for animal_fov, data_dict in pickle_data.items()
        }
        print(f"\nLoaded pickle file: {pickle_path}")
        return loaded_data

    def process_day(self, day, grid_size=(1024, 1024), cmap='RdYlBu_r'):
        """Process a day's worth of data with user choice for loading pickle."""
        print(f"\nProcessing CONTROL data for day: {day}")
        
        if self._pickle_exists(day):
            while True:
                choice = input(f"\nFound existing CONTROL analysis for {day}. Choose an option:\n"
                             "1. Load existing analysis\n"
                             "2. Generate new analysis\n"
                             "Enter choice (1 or 2): ")
                
                if choice in ['1', '2']:
                    break
                print("Invalid choice. Please enter 1 or 2.")
            
            if choice == '1':
                print("\nLoading existing CONTROL analysis...")
                return self.load_and_visualize(day, grid_size, cmap)
        else:
            print("\nNo existing CONTROL analysis found. Generating new analysis...")
            choice = '2'

        if choice == '2':
            print("\nCalculating CONTROL correlations...")
            return self.generate_new_analysis(day, grid_size, cmap)

    def _process_fov(self, day, animal, fov, grid_size, cmap):
        """Process a single FOV and return accumulators."""
        # Get the data for this FOV
        fov_data = self.df_averaged_control[
            (self.df_averaged_control['day'] == day) & 
            (self.df_averaged_control['animal'] == animal) & 
            (self.df_averaged_control['fov'] == fov)
        ]
        
        if fov_data.empty:
            print(f"No CONTROL data found for {animal} {fov}")
            return None
            
        # Load ROI data
        roi_folder = os.path.join(self.basedir, day, animal, fov)
        roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
        
        if not roi_files:
            print(f"No ROI files found for {animal} {fov}")
            return None
            
        rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
        
        # Initialize accumulators for this FOV
        accumulators = {}
        for cs_type in ['cs_plus', 'cs_minus']:
            # Initialize accumulators
            accum = torch.zeros(grid_size, device=self.device)
            count = torch.zeros(grid_size, device=self.device)
            
            # Get fluorescence columns
            fluor_prefix = 'fluor_plus_frame_' if cs_type == 'cs_plus' else 'fluor_minus_frame_'
            fluor_cols = [col for col in fov_data.columns if col.startswith(fluor_prefix)]
            
            if fluor_cols:
                # Get cell data and fluorescence data
                cells_data = []
                fluorescence_data = []
                
                cell_ids = fov_data['cell_id'].tolist()
                for roi_idx, (roi, cell_id) in enumerate(zip(rois, cell_ids)):
                    cell_data = fov_data[fov_data['cell_id'] == cell_id]
                    if not cell_data.empty:
                        cells_data.append({
                            'roi_number': roi_idx + 1,
                            'cell_id': cell_id,
                            'coords': roi.coordinates()
                        })
                        fluorescence_data.append(cell_data[fluor_cols].values[0])
                
                if cells_data and len(fluorescence_data) > 0:
                    # Calculate correlation matrix
                    correlation_matrix = calculate_correlation_matrix_gpu(np.array(fluorescence_data))
                    
                    # Create pixels tensor
                    y_coords, x_coords = np.mgrid[0:grid_size[0], 0:grid_size[1]]
                    pixels = np.vstack((x_coords.ravel(), y_coords.ravel())).T
                    pixels_tensor = torch.tensor(pixels, dtype=torch.float32).to(self.device)
                    
                    # Process correlations
                    for central_idx, central_cell in enumerate(cells_data):
                        central_correlations = torch.tensor(correlation_matrix[central_idx], device=self.device)
                        center_x = np.mean(central_cell['coords'][:, 0])
                        center_y = np.mean(central_cell['coords'][:, 1])
                        offset_x = grid_size[0] // 2 - center_x
                        offset_y = grid_size[1] // 2 - center_y
                        
                        for idx, cell in enumerate(cells_data):
                            correlation = central_correlations[idx]
                            mask = create_pixel_mask_gpu(
                                cell['coords'], 
                                grid_size, 
                                offset_x, 
                                offset_y,
                                pixels_tensor
                            )
                            accum.add_(mask * correlation)
                            count.add_(mask)
            
            accumulators[cs_type] = {'accum': accum, 'count': count}
        
        # Generate visualization for this FOV
        self._generate_fov_visualization(day, animal, fov, accumulators, grid_size, cmap)
        
        return accumulators

    def _generate_fov_visualization(self, day, animal, fov, accumulators, grid_size, cmap):
        """Generate and save visualization for a single FOV."""
        for cs_type, acc in accumulators.items():
            # Calculate average correlation
            with torch.no_grad():
                avg_correlation = torch.where(
                    acc['count'] > 0,
                    acc['accum'] / acc['count'],
                    torch.tensor(float('nan'), device=self.device)
                )
                
            # Create visualization
            fig, ax = plt.subplots(figsize=(15, 15))
            
            # Plot correlation map
            masked_correlation = np.ma.masked_invalid(avg_correlation.cpu().numpy())
            im = ax.imshow(masked_correlation, cmap=cmap, vmin=-1, vmax=1)
            
            # Add colorbar and title
            plt.colorbar(im, ax=ax)
            cs_label = 'CS+' if cs_type == 'cs_plus' else 'CS-'
            plt.title(f'CONTROL - {animal} - {fov} - {day} ({cs_label})')
            
            # Save and close
            save_path = os.path.join(
                self.viz_dir,
                f"control_correlation_{animal}_{fov}_{day}_{cs_label.replace('+', 'plus')}.png"
            )
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()

    def _generate_day_combined(self, day, accumulators, grid_size, cmap):
        """Generate and save day-combined visualization."""
        for cs_type, acc in accumulators.items():
            # Calculate average correlation
            with torch.no_grad():
                avg_correlation = torch.where(
                    acc['count'] > 0,
                    acc['accum'] / acc['count'],
                    torch.tensor(float('nan'), device=self.device)
                )
                
            # Create visualization
            fig, ax = plt.subplots(figsize=(15, 15))
            
            # Plot correlation map
            masked_correlation = np.ma.masked_invalid(avg_correlation.cpu().numpy())
            im = ax.imshow(masked_correlation, cmap=cmap, vmin=-1, vmax=1)
            
            # Add colorbar and title
            plt.colorbar(im, ax=ax)
            cs_label = 'CS+' if cs_type == 'cs_plus' else 'CS-'
            plt.title(f'CONTROL - Day-Combined Average Cross-correlations - {cs_label}\nDay: {day}')
            
            # Save and close
            save_path = os.path.join(
                self.viz_dir,
                f"control_correlation_combined_{day}_{cs_label.replace('+', 'plus')}.png"
            )
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()

    def generate_new_analysis(self, day, grid_size=(1024, 1024), cmap=rainbow_cmap):
        """Generate new analysis for a day."""
        # Get unique animals and FOVs for this day
        day_data = self.df_averaged_control[self.df_averaged_control['day'] == day]
        animals = day_data['animal'].unique()
        
        # Store all FOV results
        all_results = {}
        
        # Initialize day-level accumulators
        day_accumulators = {
            'cs_plus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            },
            'cs_minus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            }
        }

        # Process each animal
        for animal in animals:
            animal_data = day_data[day_data['animal'] == animal]
            fovs = animal_data['fov'].unique()
            
            print(f"\nProcessing CONTROL data for animal: {animal}")
            print(f"Found {len(fovs)} FOVs")
            
            # Process each FOV
            for fov in fovs:
                try:
                    print(f"Processing CONTROL data for FOV: {fov}")
                    accums = self._process_fov(day, animal, fov, grid_size, cmap)
                    
                    if accums is not None:
                        # Store results
                        all_results[f"{animal}_{fov}"] = accums
                        
                        # Add to day accumulators
                        for cs_type in ['cs_plus', 'cs_minus']:
                            day_accumulators[cs_type]['accum'].add_(accums[cs_type]['accum'])
                            day_accumulators[cs_type]['count'].add_(accums[cs_type]['count'])
                            
                except Exception as e:
                    print(f"Error processing CONTROL data for {animal} {fov}: {str(e)}")
                    continue
        
        # Save results to pickle
        self._save_to_pickle(all_results, day)
        
        # Generate day-combined visualization
        self._generate_day_combined(day, day_accumulators, grid_size, cmap)
        
        # Clean up GPU memory
        del day_accumulators
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return all_results

    def load_and_visualize(self, day, grid_size=(1024, 1024), cmap=rainbow_cmap):
        """Load pickled data and regenerate visualizations."""
        all_results = self._load_from_pickle(day)
        
        # Initialize day-level accumulators
        day_accumulators = {
            'cs_plus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            },
            'cs_minus': {
                'accum': torch.zeros(grid_size, device=self.device),
                'count': torch.zeros(grid_size, device=self.device)
            }
        }
        
        # Process each FOV's data
        for animal_fov, accums in all_results.items():
            animal, fov = animal_fov.split('_', 1)
            
            # Generate individual FOV visualization
            self._generate_fov_visualization(day, animal, fov, accums, grid_size, cmap)
            
            # Add to day accumulators
            for cs_type in ['cs_plus', 'cs_minus']:
                day_accumulators[cs_type]['accum'].add_(accums[cs_type]['accum'])
                day_accumulators[cs_type]['count'].add_(accums[cs_type]['count'])
        
        # Generate day-combined visualization
        self._generate_day_combined(day, day_accumulators, grid_size, cmap)
        
        # Clean up GPU memory
        del day_accumulators
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return all_results

# Example usage:
# Initialize analyzer with control data
def run_control_analysis(df_averaged_control, basedir, days_to_process=None):
    """Run the control correlation analysis on specified days"""
    # Initialize analyzer with control data
    analyzer = ControlNeuralCorrelationAnalyzer(
        basedir=basedir,
        df_averaged_control=df_averaged_control
    )

    # Set default days if none provided
    if days_to_process is None:
        days_to_process = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    # Process each day
    for day in days_to_process:
        print(f"\n{'='*40}\nProcessing CONTROL data for day: {day}\n{'='*40}")
        results = analyzer.process_day(day)
        print(f"Completed CONTROL analysis for day: {day}")
    
    print("\nControl correlation analysis complete for all days!")
    return analyzer

# Example usage:
analyzer = run_control_analysis(df_averaged_control, basedir)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import math

def visualize_control_day_correlations(basedir, day):
    """
    Load control pickle files and create visualizations for a given day.
    Creates two figures per CS type:
    1. Subplot grid showing each individual FOV
    2. Combined data for the day
    """
    # Load pickle data
    pickle_path = os.path.join(basedir, 'Results', 'control_correlation_pickles', f'control_correlation_data_{day}.pkl')
    
    if not os.path.exists(pickle_path):
        print(f"No control pickle file found for {day}")
        return
        
    with open(pickle_path, 'rb') as f:
        data = pickle.load(f)
    
    # Get number of FOVs for subplot grid
    n_fovs = len(data)
    grid_size = math.ceil(math.sqrt(n_fovs))
    
    # Initialize accumulators for combined plots
    combined_data = {
        'cs_plus': {'accum': None, 'count': None},
        'cs_minus': {'accum': None, 'count': None}
    }
    
    # Create subplot figures for CS+ and CS-
    fig_plus = plt.figure(figsize=(20, 20))
    fig_minus = plt.figure(figsize=(20, 20))
    fig_plus.suptitle(f'CONTROL: Individual FOV Correlations - CS+ - Day {day}', size=16)
    fig_minus.suptitle(f'CONTROL: Individual FOV Correlations - CS- - Day {day}', size=16)
    
    # Process each FOV
    for i, (animal_fov, fov_data) in enumerate(data.items()):
        animal, fov = animal_fov.split('_', 1)
        
        # Process CS+ data
        cs_plus_data = fov_data['cs_plus']
        avg_corr_plus = np.where(cs_plus_data['count'] > 0, 
                                cs_plus_data['accum'] / cs_plus_data['count'], 
                                np.nan)
        
        # Process CS- data
        cs_minus_data = fov_data['cs_minus']
        avg_corr_minus = np.where(cs_minus_data['count'] > 0,
                                 cs_minus_data['accum'] / cs_minus_data['count'],
                                 np.nan)
        
        # Add to subplots
        # CS+ subplot
        ax_plus = fig_plus.add_subplot(grid_size, grid_size, i+1)
        im_plus = ax_plus.imshow(avg_corr_plus, cmap=rainbow_cmap, vmin=-1, vmax=1)
        ax_plus.set_title(f'{animal}\n{fov}', size=10)
        ax_plus.axis('off')
        
        # CS- subplot
        ax_minus = fig_minus.add_subplot(grid_size, grid_size, i+1)
        im_minus = ax_minus.imshow(avg_corr_minus, cmap=rainbow_cmap, vmin=-1, vmax=1)
        ax_minus.set_title(f'{animal}\n{fov}', size=10)
        ax_minus.axis('off')
        
        # Add to combined data
        for cs_type, cs_data in [('cs_plus', cs_plus_data), ('cs_minus', cs_minus_data)]:
            if combined_data[cs_type]['accum'] is None:
                combined_data[cs_type]['accum'] = cs_data['accum']
                combined_data[cs_type]['count'] = cs_data['count']
            else:
                combined_data[cs_type]['accum'] += cs_data['accum']
                combined_data[cs_type]['count'] += cs_data['count']
    
    # Add colorbars to subplot figures
    fig_plus.colorbar(im_plus, ax=fig_plus.axes, label='Correlation')
    fig_minus.colorbar(im_minus, ax=fig_minus.axes, label='Correlation')
    
    # Create combined plots
    fig_combined, (ax_plus_comb, ax_minus_comb) = plt.subplots(1, 2, figsize=(20, 10))
    fig_combined.suptitle(f'CONTROL: Combined Correlations - Day {day}', size=16)
    
    # Plot combined CS+ data
    avg_corr_plus_comb = np.where(combined_data['cs_plus']['count'] > 0,
                                 combined_data['cs_plus']['accum'] / combined_data['cs_plus']['count'],
                                 np.nan)
    im_plus_comb = ax_plus_comb.imshow(avg_corr_plus_comb, cmap=rainbow_cmap, vmin=-1, vmax=1)
    ax_plus_comb.set_title('CS+ (Control)')
    plt.colorbar(im_plus_comb, ax=ax_plus_comb, label='Correlation')
    
    # Plot combined CS- data
    avg_corr_minus_comb = np.where(combined_data['cs_minus']['count'] > 0,
                                  combined_data['cs_minus']['accum'] / combined_data['cs_minus']['count'],
                                  np.nan)
    im_minus_comb = ax_minus_comb.imshow(avg_corr_minus_comb, cmap=rainbow_cmap, vmin=-1, vmax=1)
    ax_minus_comb.set_title('CS- (Control)')
    plt.colorbar(im_minus_comb, ax=ax_minus_comb, label='Correlation')
    
    # Save figures
    save_dir = os.path.join(basedir, 'Results', 'control_correlation_visualizations')
    os.makedirs(save_dir, exist_ok=True)
    
    fig_plus.savefig(os.path.join(save_dir, f'control_individual_fovs_CSplus_{day}.pdf'), 
                     dpi=300, bbox_inches='tight')
    fig_minus.savefig(os.path.join(save_dir, f'control_individual_fovs_CSminus_{day}.pdf'), 
                      dpi=300, bbox_inches='tight')
    fig_combined.savefig(os.path.join(save_dir, f'control_combined_correlations_{day}.pdf'), 
                        dpi=300, bbox_inches='tight')
    
    plt.show()
    
    print(f"Saved CONTROL visualizations for {day}")


def visualize_comparison(basedir, day):
    """
    Create comparison visualizations between real data and control data.
    """
    # Load real and control data
    real_pickle_path = os.path.join(basedir, 'Results', 'correlation_pickles', f'correlation_data_{day}.pkl')
    control_pickle_path = os.path.join(basedir, 'Results', 'control_correlation_pickles', f'control_correlation_data_{day}.pkl')
    
    if not os.path.exists(real_pickle_path):
        print(f"No real data pickle file found for {day}")
        return
        
    if not os.path.exists(control_pickle_path):
        print(f"No control data pickle file found for {day}")
        return
    
    # Load data
    with open(real_pickle_path, 'rb') as f:
        real_data = pickle.load(f)
    
    with open(control_pickle_path, 'rb') as f:
        control_data = pickle.load(f)
    
    # Initialize combined accumulators
    real_combined = {
        'cs_plus': {'accum': None, 'count': None},
        'cs_minus': {'accum': None, 'count': None}
    }
    
    control_combined = {
        'cs_plus': {'accum': None, 'count': None},
        'cs_minus': {'accum': None, 'count': None}
    }
    
    # Combine data from all FOVs
    for data_dict, combined_dict in [(real_data, real_combined), (control_data, control_combined)]:
        for animal_fov, fov_data in data_dict.items():
            for cs_type, cs_data in fov_data.items():
                if combined_dict[cs_type]['accum'] is None:
                    combined_dict[cs_type]['accum'] = cs_data['accum']
                    combined_dict[cs_type]['count'] = cs_data['count']
                else:
                    combined_dict[cs_type]['accum'] += cs_data['accum']
                    combined_dict[cs_type]['count'] += cs_data['count']
    
    # Create comparison figures
    fig, axes = plt.subplots(2, 2, figsize=(20, 20))
    fig.suptitle(f'Real vs Control Comparison - Day {day}', size=20)
    
    # Process each CS type
    for i, cs_type in enumerate(['cs_plus', 'cs_minus']):
        # Calculate real average correlation
        real_avg = np.where(real_combined[cs_type]['count'] > 0,
                          real_combined[cs_type]['accum'] / real_combined[cs_type]['count'],
                          np.nan)
        
        # Calculate control average correlation
        control_avg = np.where(control_combined[cs_type]['count'] > 0,
                             control_combined[cs_type]['accum'] / control_combined[cs_type]['count'],
                             np.nan)
        
        # Calculate difference (real - control)
        # First make sure arrays are same shape by padding with NaNs if needed
        max_shape = tuple(max(x, y) for x, y in zip(real_avg.shape, control_avg.shape))
        
        real_padded = np.full(max_shape, np.nan)
        real_padded[:real_avg.shape[0], :real_avg.shape[1]] = real_avg
        
        control_padded = np.full(max_shape, np.nan)
        control_padded[:control_avg.shape[0], :control_avg.shape[1]] = control_avg
        
        diff = real_padded - control_padded
        
        # Plot real data
        cs_label = 'CS+' if cs_type == 'cs_plus' else 'CS-'
        im1 = axes[i, 0].imshow(real_avg, cmap=rainbow_cmap, vmin=-1, vmax=1)
        axes[i, 0].set_title(f'Real Data - {cs_label}')
        fig.colorbar(im1, ax=axes[i, 0], label='Correlation')
        
        # Plot control data
        im2 = axes[i, 1].imshow(control_avg, cmap=rainbow_cmap, vmin=-1, vmax=1)
        axes[i, 1].set_title(f'Control Data - {cs_label}')
        fig.colorbar(im2, ax=axes[i, 1], label='Correlation')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for the suptitle
    
    # Save comparison figure
    save_dir = os.path.join(basedir, 'Results', 'correlation_comparisons')
    os.makedirs(save_dir, exist_ok=True)
    
    fig.savefig(os.path.join(save_dir, f'correlation_comparison_{day}.pdf'), 
               dpi=300, bbox_inches='tight')
    
    # Create difference figure
    fig_diff, axes_diff = plt.subplots(1, 2, figsize=(20, 10))
    fig_diff.suptitle(f'Real - Control Difference - Day {day}', size=20)
    
    for i, cs_type in enumerate(['cs_plus', 'cs_minus']):
        # Calculate real average correlation
        real_avg = np.where(real_combined[cs_type]['count'] > 0,
                          real_combined[cs_type]['accum'] / real_combined[cs_type]['count'],
                          np.nan)
        
        # Calculate control average correlation
        control_avg = np.where(control_combined[cs_type]['count'] > 0,
                             control_combined[cs_type]['accum'] / control_combined[cs_type]['count'],
                             np.nan)
        
        # Calculate difference (real - control)
        # First make sure arrays are same shape by padding with NaNs if needed
        max_shape = tuple(max(x, y) for x, y in zip(real_avg.shape, control_avg.shape))
        
        real_padded = np.full(max_shape, np.nan)
        real_padded[:real_avg.shape[0], :real_avg.shape[1]] = real_avg
        
        control_padded = np.full(max_shape, np.nan)
        control_padded[:control_avg.shape[0], :control_avg.shape[1]] = control_avg
        
        diff = real_padded - control_padded
        
        # Plot difference
        cs_label = 'CS+' if cs_type == 'cs_plus' else 'CS-'
        im_diff = axes_diff[i].imshow(diff, cmap=rainbow_cmap, vmin=-1, vmax=1)
        axes_diff[i].set_title(f'Real - Control Difference ({cs_label})')
        fig_diff.colorbar(im_diff, ax=axes_diff[i], label='Difference')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for the suptitle
    
    # Save difference figure
    fig_diff.savefig(os.path.join(save_dir, f'correlation_difference_{day}.pdf'), 
                    dpi=300, bbox_inches='tight')
    
    plt.show()
    
    print(f"Saved comparison visualizations for {day}")


def process_all_visualizations(basedir, days=None):
    """
    Process all visualizations for specified days.
    """
    if days is None:
        days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    for day in days:
        print(f"\nProcessing visualizations for day: {day}")
        
        # Process control visualizations
        visualize_control_day_correlations(basedir, day)
        
        # Process comparison visualizations
        visualize_comparison(basedir, day)
        
        print(f"Completed visualizations for day: {day}\n")

# Example usage:
basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
process_all_visualizations(basedir, days)

# 3D Spatial Mapping of Real vs Control Data
## Within the middle of this code you can select (real data, control data, or comparision)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pickle
import os
from matplotlib.colors import Normalize
import torch
import roifile
from tqdm import tqdm

class MaskedCorrelation3DVisualizer:
    def __init__(self, basedir, df_averaged):
        """
        Initialize the visualizer with directory paths.
        
        Args:
            basedir (str): Base directory containing the Results folder
            df_averaged: DataFrame containing averaged data for cell diameter calculation
        """
        self.basedir = basedir
        self.df_averaged = df_averaged
        self.results_dir = os.path.join(basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'correlation_pickles')
        self.output_dir = os.path.join(self.results_dir, 'correlation_3d_visualizations_masked')
        
        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Calculate cell diameter
        self.cell_diameter = self.calculate_global_cell_diameter()
        print(f"Global average cell diameter: {self.cell_diameter:.2f} pixels")
        print(f"Using pickle directory: {self.pickle_dir}")
        print(f"Saving visualizations to: {self.output_dir}")
        
    def calculate_global_cell_diameter(self):
        """Calculate average cell diameter across all FOVs and days."""
        print("\nCalculating global cell diameter...")
        all_areas = []
        
        combinations = self.df_averaged.groupby(['day', 'animal', 'fov']).size().reset_index()
        
        for _, row in tqdm(combinations.iterrows(), total=len(combinations)):
            try:
                roi_folder = os.path.join(self.basedir, row['day'], row['animal'], row['fov'])
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                
                if roi_files:
                    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
                    for roi in rois:
                        coords = roi.coordinates()
                        x, y = coords[:, 0], coords[:, 1]
                        area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                        all_areas.append(area)
            except Exception as e:
                print(f"Error processing {roi_folder}: {str(e)}")
                continue
        
        return 2 * np.sqrt(np.mean(all_areas) / np.pi)
        
    def create_radius_mask(self, data_shape, center=None):
        """Create a mask for data within 11 cell radii."""
        if center is None:
            center = (data_shape[0] // 2, data_shape[1] // 2)
            
        y, x = np.ogrid[:data_shape[0], :data_shape[1]]
        distances = np.sqrt((x - center[0])**2 + (y - center[1])**2)
        
        max_radius = (self.cell_diameter/2) * 11  # 11 cell radii
        mask = distances <= max_radius
        
        return mask
        
    def load_pickle_data(self, day):
        """Load and process pickle data for a specific day."""
        pickle_path = os.path.join(self.pickle_dir, f"correlation_data_{day}.pkl")
        
        if not os.path.exists(pickle_path):
            raise FileNotFoundError(f"No pickle file found for day {day}")
            
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)
            
        # Initialize accumulators for day-combined data
        combined_data = {
            'cs_plus': {'accum': None, 'count': None},
            'cs_minus': {'accum': None, 'count': None}
        }
        
        # Combine data across all FOVs
        for animal_fov_data in data.values():
            for cs_type in ['cs_plus', 'cs_minus']:
                if combined_data[cs_type]['accum'] is None:
                    combined_data[cs_type]['accum'] = animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] = animal_fov_data[cs_type]['count']
                else:
                    combined_data[cs_type]['accum'] += animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] += animal_fov_data[cs_type]['count']
        
        # Calculate averages and apply radius mask
        processed_data = {}
        mask = self.create_radius_mask((1024, 1024))
        
        for cs_type in ['cs_plus', 'cs_minus']:
            avg = np.zeros_like(combined_data[cs_type]['accum'])
            valid_mask = (combined_data[cs_type]['count'] > 0) & mask
            avg[valid_mask] = combined_data[cs_type]['accum'][valid_mask] / combined_data[cs_type]['count'][valid_mask]
            avg[~mask] = np.nan  # Set values outside radius to NaN
            processed_data[cs_type] = avg
            
        return processed_data
        
    def create_3d_correlation_plots(self, day, cs_type, data, elev_angles=[0, 45, 90]):
        """
        Create 3D surface plots with multiple viewing angles.
        
        Args:
            day (str): Day identifier
            cs_type (str): 'cs_plus' or 'cs_minus'
            data (np.ndarray): Correlation data
            elev_angles (list): List of elevation angles for different views
        """
        # Create coordinate grid
        x = np.linspace(0, data.shape[0]-1, data.shape[0])
        y = np.linspace(0, data.shape[1]-1, data.shape[1])
        X, Y = np.meshgrid(x, y)
        
        # Set up the figure with subplots for different viewing angles
        fig = plt.figure(figsize=(20, 6))
        
        # Create plots for each viewing angle
        for idx, elev in enumerate(elev_angles):
            ax = fig.add_subplot(1, 3, idx+1, projection='3d')
            
            # Create surface plot with masked data
            masked_data = np.ma.masked_invalid(data)
            surf = ax.plot_surface(X, Y, masked_data, 
                                 cmap=rainbow_cmap,
                                 norm=Normalize(vmin=0, vmax=1),
                                 linewidth=0,
                                 antialiased=True)
            
            # Set viewing angle
            ax.view_init(elev=elev, azim=45)
            
            # Set labels and title
            ax.set_xlabel('X Position')
            ax.set_ylabel('Y Position')
            ax.set_zlabel('Correlation')
            
            # Set z-axis limits to 0-1
            ax.set_zlim(0, 1)
            
            # Customize z-axis ticks
            ax.set_zticks([0, 0.25, 0.5, 0.75, 1.0])
            
            view_label = "Side" if elev == 0 else "45°" if elev == 45 else "Top"
            ax.set_title(f'{view_label} View')
        
        # Add overall title and colorbar
        plt.suptitle(f'Day {day} - {cs_type.replace("cs_", "CS")} Correlation Surface\n(Limited to 11 cell radii)', y=1.05)
        fig.colorbar(surf, ax=fig.axes, shrink=0.5, aspect=5, ticks=[0, 0.25, 0.5, 0.75, 1.0])
        
        # Save plot
        cs_label = cs_type.replace('cs_plus', 'CSplus').replace('cs_minus', 'CSminus')
        save_path = os.path.join(self.output_dir, f'correlation_3d_{day}_{cs_label}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved visualization to: {save_path}")
        plt.show()

    def process_days(self, days):
        """Process and visualize data for multiple days."""
        for day in days:
            print(f"\nProcessing day: {day}")
            try:
                # Load data
                data = self.load_pickle_data(day)
                
                # Create visualizations for both CS types
                for cs_type in ['cs_plus', 'cs_minus']:
                    print(f"Creating visualization for {cs_type}")
                    self.create_3d_correlation_plots(day, cs_type, data[cs_type])
                    
            except Exception as e:
                print(f"Error processing day {day}: {str(e)}")
                continue
                
        print("\nProcessing complete!")

class ControlMaskedCorrelation3DVisualizer:
    def __init__(self, basedir, df_averaged_control):
        """
        Initialize the visualizer with directory paths for control data.
        
        Args:
            basedir (str): Base directory containing the Results folder
            df_averaged_control: DataFrame containing averaged control data for cell diameter calculation
        """
        self.basedir = basedir
        self.df_averaged_control = df_averaged_control
        self.results_dir = os.path.join(basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'control_correlation_pickles')
        self.output_dir = os.path.join(self.results_dir, 'control_correlation_3d_visualizations_masked')
        
        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Calculate cell diameter
        self.cell_diameter = self.calculate_global_cell_diameter()
        print(f"Global average cell diameter: {self.cell_diameter:.2f} pixels")
        print(f"Using CONTROL pickle directory: {self.pickle_dir}")
        print(f"Saving CONTROL visualizations to: {self.output_dir}")
        
    def calculate_global_cell_diameter(self):
        """Calculate average cell diameter across all FOVs and days."""
        print("\nCalculating global cell diameter from control data...")
        all_areas = []
        
        combinations = self.df_averaged_control.groupby(['day', 'animal', 'fov']).size().reset_index()
        
        for _, row in tqdm(combinations.iterrows(), total=len(combinations)):
            try:
                roi_folder = os.path.join(self.basedir, row['day'], row['animal'], row['fov'])
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                
                if roi_files:
                    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
                    for roi in rois:
                        coords = roi.coordinates()
                        x, y = coords[:, 0], coords[:, 1]
                        area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                        all_areas.append(area)
            except Exception as e:
                print(f"Error processing {roi_folder}: {str(e)}")
                continue
        
        return 2 * np.sqrt(np.mean(all_areas) / np.pi)
        
    def create_radius_mask(self, data_shape, center=None):
        """Create a mask for data within 11 cell radii."""
        if center is None:
            center = (data_shape[0] // 2, data_shape[1] // 2)
            
        y, x = np.ogrid[:data_shape[0], :data_shape[1]]
        distances = np.sqrt((x - center[0])**2 + (y - center[1])**2)
        
        max_radius = (self.cell_diameter/2) * 11  # 11 cell radii
        mask = distances <= max_radius
        
        return mask
        
    def load_pickle_data(self, day):
        """Load and process control pickle data for a specific day."""
        pickle_path = os.path.join(self.pickle_dir, f"control_correlation_data_{day}.pkl")
        
        if not os.path.exists(pickle_path):
            raise FileNotFoundError(f"No control pickle file found for day {day}")
            
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)
            
        # Initialize accumulators for day-combined data
        combined_data = {
            'cs_plus': {'accum': None, 'count': None},
            'cs_minus': {'accum': None, 'count': None}
        }
        
        # Combine data across all FOVs
        for animal_fov_data in data.values():
            for cs_type in ['cs_plus', 'cs_minus']:
                if combined_data[cs_type]['accum'] is None:
                    combined_data[cs_type]['accum'] = animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] = animal_fov_data[cs_type]['count']
                else:
                    combined_data[cs_type]['accum'] += animal_fov_data[cs_type]['accum']
                    combined_data[cs_type]['count'] += animal_fov_data[cs_type]['count']
        
        # Calculate averages and apply radius mask
        processed_data = {}
        mask = self.create_radius_mask((1024, 1024))
        
        for cs_type in ['cs_plus', 'cs_minus']:
            avg = np.zeros_like(combined_data[cs_type]['accum'])
            valid_mask = (combined_data[cs_type]['count'] > 0) & mask
            avg[valid_mask] = combined_data[cs_type]['accum'][valid_mask] / combined_data[cs_type]['count'][valid_mask]
            avg[~mask] = np.nan  # Set values outside radius to NaN
            processed_data[cs_type] = avg
            
        return processed_data
        
    def create_3d_correlation_plots(self, day, cs_type, data, elev_angles=[0, 45, 90]):
        """
        Create 3D surface plots with multiple viewing angles for control data.
        
        Args:
            day (str): Day identifier
            cs_type (str): 'cs_plus' or 'cs_minus'
            data (np.ndarray): Correlation data
            elev_angles (list): List of elevation angles for different views
        """
        # Create coordinate grid
        x = np.linspace(0, data.shape[0]-1, data.shape[0])
        y = np.linspace(0, data.shape[1]-1, data.shape[1])
        X, Y = np.meshgrid(x, y)
        
        # Set up the figure with subplots for different viewing angles
        fig = plt.figure(figsize=(20, 6))
        
        # Create plots for each viewing angle
        for idx, elev in enumerate(elev_angles):
            ax = fig.add_subplot(1, 3, idx+1, projection='3d')
            
            # Create surface plot with masked data
            masked_data = np.ma.masked_invalid(data)
            surf = ax.plot_surface(X, Y, masked_data, 
                                 cmap=rainbow_cmap,
                                 norm=Normalize(vmin=0, vmax=1),
                                 linewidth=0,
                                 antialiased=True)
            
            # Set viewing angle
            ax.view_init(elev=elev, azim=45)
            
            # Set labels and title
            ax.set_xlabel('X Position')
            ax.set_ylabel('Y Position')
            ax.set_zlabel('Correlation')
            
            # Set z-axis limits to 0-1
            ax.set_zlim(0, 1)
            
            # Customize z-axis ticks
            ax.set_zticks([0, 0.25, 0.5, 0.75, 1.0])
            
            view_label = "Side" if elev == 0 else "45°" if elev == 45 else "Top"
            ax.set_title(f'{view_label} View')
        
        # Add overall title and colorbar
        plt.suptitle(f'CONTROL - Day {day} - {cs_type.replace("cs_", "CS")} Correlation Surface\n(Limited to 11 cell radii)', y=1.05)
        fig.colorbar(surf, ax=fig.axes, shrink=0.5, aspect=5, ticks=[0, 0.25, 0.5, 0.75, 1.0])
        
        # Save plot
        cs_label = cs_type.replace('cs_plus', 'CSplus').replace('cs_minus', 'CSminus')
        save_path = os.path.join(self.output_dir, f'control_correlation_3d_{day}_{cs_label}.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved CONTROL visualization to: {save_path}")
        plt.show()

    def process_days(self, days):
        """Process and visualize control data for multiple days."""
        for day in days:
            print(f"\nProcessing CONTROL data for day: {day}")
            try:
                # Load data
                data = self.load_pickle_data(day)
                
                # Create visualizations for both CS types
                for cs_type in ['cs_plus', 'cs_minus']:
                    print(f"Creating CONTROL visualization for {cs_type}")
                    self.create_3d_correlation_plots(day, cs_type, data[cs_type])
                    
            except Exception as e:
                print(f"Error processing CONTROL data for day {day}: {str(e)}")
                continue
                
        print("\nCONTROL data processing complete!")

class ComparisonCorrelation3DVisualizer:
    def __init__(self, basedir, real_visualizer, control_visualizer):
        """
        Initialize the comparison visualizer with directory paths and existing visualizer instances.
        
        Args:
            basedir (str): Base directory containing the Results folder
            real_visualizer: Instance of MaskedCorrelation3DVisualizer
            control_visualizer: Instance of ControlMaskedCorrelation3DVisualizer
        """
        self.basedir = basedir
        self.real_visualizer = real_visualizer
        self.control_visualizer = control_visualizer
        self.results_dir = os.path.join(basedir, 'Results')
        self.output_dir = os.path.join(self.results_dir, 'correlation_3d_comparison')
        
        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)
        print(f"Saving comparison visualizations to: {self.output_dir}")
        
    def create_comparison_visualization(self, day, cs_type, real_data, control_data, elev_angles=[0, 45, 90]):
        """
        Create comparison 3D visualization showing real and control data side by side.
        
        Args:
            day (str): Day identifier
            cs_type (str): 'cs_plus' or 'cs_minus'
            real_data (np.ndarray): Real correlation data
            control_data (np.ndarray): Control correlation data
            elev_angles (list): List of elevation angles for views
        """
        # Create coordinate grid - use max shape to handle different sizes
        max_shape = tuple(max(d1, d2) for d1, d2 in zip(real_data.shape, control_data.shape))
        x = np.linspace(0, max_shape[0]-1, max_shape[0])
        y = np.linspace(0, max_shape[1]-1, max_shape[1])
        X, Y = np.meshgrid(x, y)
        
        # Pad data arrays to match shape
        real_padded = np.full(max_shape, np.nan)
        real_padded[:real_data.shape[0], :real_data.shape[1]] = real_data
        
        control_padded = np.full(max_shape, np.nan)
        control_padded[:control_data.shape[0], :control_data.shape[1]] = control_data
        
        # Calculate difference (real - control)
        diff_data = real_padded - control_padded
        
        # Create visualization for each viewing angle
        for idx, elev in enumerate(elev_angles):
            # Set up figure with 3 subplots (real, control, difference)
            fig = plt.figure(figsize=(20, 6))
            
            # 1. Real data plot
            ax1 = fig.add_subplot(1, 3, 1, projection='3d')
            masked_real = np.ma.masked_invalid(real_padded)
            surf1 = ax1.plot_surface(X, Y, masked_real, 
                                   cmap=rainbow_cmap,
                                   norm=Normalize(vmin=0, vmax=1),
                                   linewidth=0,
                                   antialiased=True)
            ax1.view_init(elev=elev, azim=45)
            ax1.set_zlim(-0.25, 1)
            ax1.set_zticks([-0.25, 0, 0.25, 0.5, 0.75, 1])
            ax1.set_title('Real Data')
            fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=5, ticks=[0, 0.25, 0.5, 0.75])
            
            # 2. Control data plot
            ax2 = fig.add_subplot(1, 3, 2, projection='3d')
            masked_control = np.ma.masked_invalid(control_padded)
            surf2 = ax2.plot_surface(X, Y, masked_control, 
                                   cmap=rainbow_cmap,
                                   norm=Normalize(vmin=0, vmax=1),
                                   linewidth=0,
                                   antialiased=True)
            ax2.view_init(elev=elev, azim=45)
            ax2.set_zlim(-0.25, 1)
            ax2.set_zticks([-0.25, 0, 0.25, 0.5, 0.75, 1])
            ax2.set_title('Control Data')
            fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=5, ticks=[0, 0.25, 0.5, 0.75])
            
            # 3. Difference plot (real - control)
            ax3 = fig.add_subplot(1, 3, 3, projection='3d')
            masked_diff = np.ma.masked_invalid(diff_data)
            surf3 = ax3.plot_surface(X, Y, masked_diff, 
                                   cmap=rainbow_cmap,
                                   norm=Normalize(vmin=0, vmax=1),
                                   linewidth=0,
                                   antialiased=True)
            ax3.view_init(elev=elev, azim=45)
            ax3.set_zlim(-0.25, 1)
            ax3.set_zticks([-0.25, 0, 0.25, 0.5, 0.75, 1])
            ax3.set_title('Difference (Real - Control)')
            fig.colorbar(surf3, ax=ax3, shrink=0.5, aspect=5, ticks=[0, 0.25, 0.5, 0.75])
            
            # Overall title
            view_name = "Side" if elev == 0 else "45 degree" if elev == 45 else "Top"
            plt.suptitle(f'Day {day} - {cs_type.replace("cs_", "CS")} Correlation Comparison\n{view_name} View', 
                        y=1.05, fontsize=14)
            
            # Save figure
            cs_label = cs_type.replace('cs_plus', 'CSplus').replace('cs_minus', 'CSminus')
            view_label = "side" if elev == 0 else "45deg" if elev == 45 else "top"
            save_path = os.path.join(self.output_dir, f'correlation_comparison_{day}_{cs_label}_{view_label}_11-04-25.pdf')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()
            
        print(f"Saved comparison visualizations for day {day} - {cs_type}")
        
    def process_days(self, days):
        """Create comparison visualizations for multiple days."""
        for day in days:
            print(f"\nProcessing comparison for day: {day}")
            
            try:
                # Load real data
                real_data = self.real_visualizer.load_pickle_data(day)
                
                # Load control data
                control_data = self.control_visualizer.load_pickle_data(day)
                
                # Create comparison visualizations for both CS types
                for cs_type in ['cs_plus', 'cs_minus']:
                    print(f"Creating comparison for {cs_type}")
                    self.create_comparison_visualization(day, cs_type, 
                                                      real_data[cs_type], 
                                                      control_data[cs_type])
                
            except Exception as e:
                print(f"Error processing comparison for day {day}: {str(e)}")
                continue
                
        print("\nComparison processing complete!")

# Run just the control visualizer
def run_control_visualizations(basedir, df_averaged_control, days=None):
    """Run just the control visualizations"""
    if days is None:
        days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    # Create control data visualizer
    control_visualizer = ControlMaskedCorrelation3DVisualizer(basedir, df_averaged_control)
    
    # Process days
    print("\nProcessing control data visualizations...")
    control_visualizer.process_days(days)
    
    print("\nControl visualizations complete!")
    
    return control_visualizer

# Run all visualizations: real, control, and comparison
def run_all_visualizations(basedir, df_averaged, df_averaged_control, days=None):
    """Run all visualizations - real, control, and comparison"""
    if days is None:
        days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    # Create real data visualizer
    real_visualizer = MaskedCorrelation3DVisualizer(basedir, df_averaged)
    
    # Create control data visualizer
    control_visualizer = ControlMaskedCorrelation3DVisualizer(basedir, df_averaged_control)
    
    # Create comparison visualizer
    comparison_visualizer = ComparisonCorrelation3DVisualizer(basedir, real_visualizer, control_visualizer)
    
    # Process days
    print("Processing real data visualizations...")
    real_visualizer.process_days(days)
    
    print("\nProcessing control data visualizations...")
    control_visualizer.process_days(days)
    
    print("\nProcessing comparison visualizations...")
    comparison_visualizer.process_days(days)
    
    print("\nAll visualizations complete!")
    
    return {
        'real_visualizer': real_visualizer,
        'control_visualizer': control_visualizer,
        'comparison_visualizer': comparison_visualizer
    }

# Example usage with flexible options
if __name__ == "__main__":
    # Set your base directory
    basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'

    # Define days to process
    days_to_process = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']



##### SET IF YOU WANT THE REAL DATA, CONTROL DATA, OR COMPARISOIN ANALYSIS TO OCCUR HERE ######    


    
    # Choose which analysis to run
    run_mode = 'comparison'  # Options: 'control', 'all', or 'comparison'
    
    try:
        if run_mode == 'control':
            # Run only control visualization
            control_visualizer = run_control_visualizations(
                basedir=basedir,
                df_averaged_control=df_averaged_control,
                days=days_to_process
            )
            print("Control analysis complete!")
            
        elif run_mode == 'all':
            # Run all visualizations
            visualizers = run_all_visualizations(
                basedir=basedir,
                df_averaged=df_averaged,
                df_averaged_control=df_averaged_control,
                days=days_to_process
            )
            print("Complete analysis finished!")
            
        elif run_mode == 'comparison':
            # First create both visualizers
            real_visualizer = MaskedCorrelation3DVisualizer(basedir, df_averaged)
            control_visualizer = ControlMaskedCorrelation3DVisualizer(basedir, df_averaged_control)
            
            # Then create and run comparison only
            comparison_visualizer = ComparisonCorrelation3DVisualizer(
                basedir, real_visualizer, control_visualizer)
            comparison_visualizer.process_days(days_to_process)
            print("Comparison analysis complete!")
        
    except NameError as e:
            # Handle case where dataframes aren't defined
            print(f"Error: {e}")
            print("\nMake sure to run this code with df_averaged and df_averaged_control already defined.")
            print("Example to define these variables:")
            print("df_averaged = pd.read_pickle('/path/to/averaged_data.pkl')")
            print("df_averaged_control = pd.read_pickle('/path/to/averaged_control_data.pkl')")
            
            # Try to automatically load the dataframes if possible
            try:
                import pandas as pd
                print("\nAttempting to automatically load dataframes...")
                
                # Try to load control dataframe (needed for all modes)
                control_path = os.path.join(basedir, 'Results', 'averaged_control_data_cues_0.pkl')
                if os.path.exists(control_path):
                    df_averaged_control = pd.read_pickle(control_path)
                    print(f"Successfully loaded df_averaged_control from {control_path}")
                    
                    # If we need df_averaged as well (for 'all' or 'comparison' modes)
                    if run_mode in ['all', 'comparison']:
                        real_path = os.path.join(basedir, 'Results', 'averaged_data_cues_0.pkl')
                        if os.path.exists(real_path):
                            df_averaged = pd.read_pickle(real_path)
                            print(f"Successfully loaded df_averaged from {real_path}")
                            
                            print("\nRetrying analysis with loaded dataframes...")
                            # Retry the analysis based on run_mode
                            if run_mode == 'all':
                                visualizers = run_all_visualizations(
                                    basedir=basedir,
                                    df_averaged=df_averaged,
                                    df_averaged_control=df_averaged_control,
                                    days=days_to_process
                                )
                                print("Complete analysis finished!")
                            elif run_mode == 'comparison':
                                real_visualizer = MaskedCorrelation3DVisualizer(basedir, df_averaged)
                                control_visualizer = ControlMaskedCorrelation3DVisualizer(basedir, df_averaged_control)
                                comparison_visualizer = ComparisonCorrelation3DVisualizer(
                                    basedir, real_visualizer, control_visualizer)
                                comparison_visualizer.process_days(days_to_process)
                                print("Comparison analysis complete!")
                        else:
                            print(f"Could not find real data at {real_path}")
                            print("Cannot proceed with 'all' or 'comparison' mode without df_averaged")
                    else:
                        # For control-only mode
                        control_visualizer = run_control_visualizations(
                            basedir=basedir,
                            df_averaged_control=df_averaged_control,
                            days=days_to_process
                        )
                        print("Control analysis complete!")
                else:
                    print(f"Could not find control data at {control_path}")
                    print("Cannot proceed with analysis without df_averaged_control")
                    
            except Exception as inner_e:
                print(f"Error during automatic loading: {str(inner_e)}")
                print("\nPlease manually load the required dataframes and try again")

# Side by side stats and heatmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle
import torch
import roifile
from scipy import stats
import seaborn as sns
from matplotlib.patches import Circle
from tqdm import tqdm
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

class EnhancedDonutAnalyzer:
    def __init__(self, basedir, df_averaged):
        """Initialize the analyzer with base settings."""
        self.basedir = basedir
        self.df_averaged = df_averaged
        
        # Set up directories
        self.results_dir = os.path.join(self.basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'correlation_pickles')
        self.donut_dir = os.path.join(self.results_dir, 'donut_analysis')
        os.makedirs(self.donut_dir, exist_ok=True)
        
        # Set up device
        if torch.backends.mps.is_available():
            self.device = torch.device("mps")
        elif torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        print(f"Using device: {self.device}")
        
        # Define radius factors and colors
        # self.radius_factors = [1, 3, 5, 7, 9, 11, 13, 15]
        self.radius_factors = [1, 3, 5, 7, 9, 11]
        self.zone_colors = plt.cm.tab20(np.linspace(0, 1, len(self.radius_factors)))
        
        # Calculate global cell diameter once
        self.cell_diameter = self.calculate_global_cell_diameter()
        print(f"Global average cell diameter: {self.cell_diameter:.2f} pixels")

    def calculate_global_cell_diameter(self):
        """Calculate average cell diameter across all FOVs and days."""
        print("\nCalculating global cell diameter...")
        all_areas = []
        combinations = self.df_averaged.groupby(['day', 'animal', 'fov']).size().reset_index()
        
        for _, row in tqdm(combinations.iterrows(), total=len(combinations)):
            try:
                roi_folder = os.path.join(self.basedir, row['day'], row['animal'], row['fov'])
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                
                if roi_files:
                    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
                    for roi in rois:
                        coords = roi.coordinates()
                        x, y = coords[:, 0], coords[:, 1]
                        area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                        all_areas.append(area)
            except Exception as e:
                print(f"Error processing {roi_folder}: {str(e)}")
                continue
        
        mean_area = np.mean(all_areas)
        return 2 * np.sqrt(mean_area / np.pi)

    def create_donut_masks(self, grid_size=(1024, 1024), center=(512, 512)):
        """Create donut-shaped masks for all radius factors."""
        y, x = np.ogrid[:grid_size[0], :grid_size[1]]
        distances = np.sqrt((x - center[0])**2 + (y - center[1])**2)
        
        masks = []
        for i, factor in enumerate(self.radius_factors):
            radius = (self.cell_diameter/2) * factor
            if i == 0:
                mask = distances <= radius
            else:
                inner_radius = (self.cell_diameter/2) * self.radius_factors[i-1]
                mask = (distances > inner_radius) & (distances <= radius)
            masks.append(mask)
        
        return masks

    def analyze_pickle_data(self, day):
        """Analyze correlation data for a single day."""
        pickle_path = os.path.join(self.pickle_dir, f"correlation_data_{day}.pkl")
        if not os.path.exists(pickle_path):
            print(f"No pickle file found for {day}")
            return None
            
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)
        
        # Initialize results storage
        results_data = []
        masks = self.create_donut_masks()
        
        # Process each FOV
        for animal_fov, fov_data in data.items():
            animal, fov = animal_fov.split('_', 1)
            
            for cs_type in ['cs_plus', 'cs_minus']:
                # Calculate average correlation map
                accum = fov_data[cs_type]['accum']
                count = fov_data[cs_type]['count']
                with np.errstate(divide='ignore', invalid='ignore'):
                    corr_map = np.where(count > 0, accum / count, np.nan)
                
                # Calculate average for each donut zone
                for zone_idx, mask in enumerate(masks):
                    zone_avg = np.nanmean(corr_map[mask])
                    if not np.isnan(zone_avg):  # Only add non-NaN values
                        results_data.append({
                            'day': day,
                            'animal': animal,
                            'fov': fov,
                            'cs_type': 'CS+' if cs_type == 'cs_plus' else 'CS-',
                            'zone_radius': self.radius_factors[zone_idx],
                            'correlation': zone_avg
                        })
        
        return pd.DataFrame(results_data)

    def perform_statistical_analysis(self, results_df):
        """Perform comprehensive statistical analysis."""
        print("\nPerforming Statistical Analysis")
        print("=" * 50)
        
        # Create a copy of the dataframe to avoid modifying the original
        df = results_df.copy()
        
        # Convert to categories
        df['zone_radius'] = df['zone_radius'].astype('category')
        df['day'] = df['day'].astype('category')
        df['cs_type'] = df['cs_type'].astype('category')
        df['group_id'] = df['animal'] + '_' + df['fov']
        
        try:
            # Fit mixed-effects model with simplified formula
            model = sm.MixedLM.from_formula(
                "correlation ~ C(zone_radius) + C(day) + C(cs_type)",
                groups="group_id",
                data=df
            )
            
            results = model.fit()
            print("\nMixed-Effects Model Results:")
            print(results.summary())
        except Exception as e:
            print(f"Error in mixed model: {str(e)}")
            results = None
        
        # Perform ANOVAs for main effects
        print("\nANOVA Results:")
        
        # Zone radius effects
        radius_groups = [group['correlation'].values for name, group in df.groupby('zone_radius')]
        radius_groups = [group for group in radius_groups if len(group) > 0]  # Remove empty groups
        radius_stats = stats.f_oneway(*radius_groups)
        print("\n1. Effect of Zone Radius:")
        print(f"F-statistic: {radius_stats.statistic:.3f}")
        print(f"p-value: {radius_stats.pvalue:.3f}")
        
        # Day effects
        day_groups = [group['correlation'].values for name, group in df.groupby('day')]
        day_groups = [group for group in day_groups if len(group) > 0]  # Remove empty groups
        day_stats = stats.f_oneway(*day_groups)
        print("\n2. Effect of Day:")
        print(f"F-statistic: {day_stats.statistic:.3f}")
        print(f"p-value: {day_stats.pvalue:.3f}")
        
        # CS type effects
        cs_type_ttest = stats.ttest_ind(
            df[df['cs_type'] == 'CS+']['correlation'],
            df[df['cs_type'] == 'CS-']['correlation']
        )
        print("\n3. Effect of CS Type:")
        print(f"t-statistic: {cs_type_ttest.statistic:.3f}")
        print(f"p-value: {cs_type_ttest.pvalue:.3f}")
        
        # Perform post-hoc tests
        print("\nPost-hoc Analyses (Tukey HSD):")
        
        # Zone radius comparisons
        radius_tukey = pairwise_tukeyhsd(
            df['correlation'],
            df['zone_radius']
        )
        print("\nZone Radius Comparisons:")
        print(radius_tukey)
        
        # Day comparisons
        day_tukey = pairwise_tukeyhsd(
            df['correlation'],
            df['day']
        )
        print("\nDay Comparisons:")
        print(day_tukey)
        
        # Create summary statistics
        summary_stats = df.groupby(['day', 'zone_radius', 'cs_type'])['correlation'].agg([
            'mean', 'std', 'count'
        ]).reset_index()
        
        print("\nSummary Statistics:")
        print(summary_stats)
        
        return {
            'mixed_model': results,
            'radius_stats': radius_stats,
            'day_stats': day_stats,
            'cs_type_ttest': cs_type_ttest,
            'radius_tukey': radius_tukey,
            'day_tukey': day_tukey,
            'summary_stats': summary_stats
        }

    # def create_summary_plots(self, results_df):
    #     """Create summary visualization plots."""
    #     # Create figure for summary plots
    #     plt.figure(figsize=(20, 15))
        
    #     # 1. Zone radius effects
    #     plt.subplot(221)
    #     sns.boxplot(data=results_df, x='zone_radius', y='correlation', hue='cs_type')
    #     plt.title('Correlation by Zone Radius and CS Type')
    #     plt.xlabel('Zone Radius (× cell diameter)')
    #     plt.ylabel('Correlation')
        
    #     # 2. Day effects
    #     plt.subplot(222)
    #     sns.boxplot(data=results_df, x='day', y='correlation', hue='cs_type')
    #     plt.title('Correlation by Day and CS Type')
    #     plt.xticks(rotation=45)
        
    #     # 3. Interaction plot
    #     plt.subplot(223)
    #     sns.lineplot(data=results_df, x='zone_radius', y='correlation', 
    #                 hue='day', style='cs_type', markers=True, dashes=False)
    #     plt.title('Day × Zone Radius Interaction')
        
    #     # 4. Distribution plot
    #     plt.subplot(224)
    #     sns.violinplot(data=results_df, x='cs_type', y='correlation', hue='day')
    #     plt.title('Correlation Distribution by CS Type and Day')
        
    #     plt.tight_layout()
    #     plt.savefig(os.path.join(self.donut_dir, 'summary_plots.png'), dpi=300, bbox_inches='tight')
    #     plt.show()

    def create_summary_plots(self, results_df):
        """Create summary visualization plots."""
        # Create figure for summary plots
        plt.figure(figsize=(20, 15))
        
        # 1. Zone radius effects
        plt.subplot(221)
        sns.boxplot(data=results_df, x='zone_radius', y='correlation', hue='cs_type')
        plt.title('Correlation by Zone Radius and CS Type')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        
        # 2. Day effects
        plt.subplot(222)
        sns.boxplot(data=results_df, x='day', y='correlation', hue='cs_type')
        plt.title('Correlation by Day and CS Type')
        plt.xticks(rotation=45)
        
        # 3. Interaction plot - FIXED: Replace seaborn lineplot with manual plotting
        plt.subplot(223)
        
        # Get unique values
        days = sorted(results_df['day'].unique())
        zone_radii = sorted(results_df['zone_radius'].unique())
        cs_types = ['CS+', 'CS-']
        
        # Define colors and styles
        day_colors = plt.cm.tab10(np.linspace(0, 1, len(days)))
        line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
        markers = ['o', 's']       # Circle for CS+, square for CS-
        
        # Plot each combination manually
        for i, day in enumerate(days):
            for j, cs_type in enumerate(cs_types):
                # Filter data for this combination
                mask = (results_df['day'] == day) & (results_df['cs_type'] == cs_type)
                subset_data = results_df[mask]
                
                # Calculate means for each zone radius
                means = []
                for radius in zone_radii:
                    radius_mask = subset_data['zone_radius'] == radius
                    if radius_mask.any():
                        means.append(subset_data.loc[radius_mask, 'correlation'].mean())
                    else:
                        means.append(np.nan)
                
                # Convert to numpy array
                means = np.array(means)
                
                # Plot the line
                plt.plot(zone_radii, means, 
                         color=day_colors[i], 
                         linestyle=line_styles[j],
                         marker=markers[j], 
                         label=f"{day}, {cs_type}")
        
        plt.title('Day × Zone Radius Interaction')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # 4. Distribution plot
        plt.subplot(224)
        sns.violinplot(data=results_df, x='cs_type', y='correlation', hue='day')
        plt.title('Correlation Distribution by CS Type and Day')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.donut_dir, 'summary_plots.png'), dpi=300, bbox_inches='tight')
        plt.show()

    def save_statistical_results(self, stats_results, results_path):
        """Save statistical results in a pickle-friendly format."""
        pickle_friendly_results = {
            'radius_stats': {
                'statistic': float(stats_results['radius_stats'].statistic),
                'pvalue': float(stats_results['radius_stats'].pvalue)
            },
            'day_stats': {
                'statistic': float(stats_results['day_stats'].statistic),
                'pvalue': float(stats_results['day_stats'].pvalue)
            },
            'cs_type_ttest': {
                'statistic': float(stats_results['cs_type_ttest'].statistic),
                'pvalue': float(stats_results['cs_type_ttest'].pvalue)
            },
            'radius_tukey': {
                'meandiffs': stats_results['radius_tukey'].meandiffs.tolist(),
                'confint': stats_results['radius_tukey'].confint.tolist(),
                'reject': stats_results['radius_tukey'].reject.tolist()
            },
            'day_tukey': {
                'meandiffs': stats_results['day_tukey'].meandiffs.tolist(),
                'confint': stats_results['day_tukey'].confint.tolist(),
                'reject': stats_results['day_tukey'].reject.tolist()
            },
            'summary_stats': stats_results['summary_stats'].to_dict()
        }
        
        # If mixed model was successful, add its results
        if stats_results['mixed_model'] is not None:
            pickle_friendly_results['mixed_model'] = {
                'params': stats_results['mixed_model'].params.to_dict(),
                'pvalues': stats_results['mixed_model'].pvalues.to_dict(),
                'conf_int': stats_results['mixed_model'].conf_int().to_dict()
            }
        
        with open(results_path, 'wb') as f:
            pickle.dump(pickle_friendly_results, f)

    def run_analysis(self, days=None):
        """Run complete analysis pipeline."""
        if days is None:
            days = self.df_averaged['day'].unique()
            
        # Collect results from all days
        all_results = []
        for day in days:
            print(f"\nProcessing {day}...")
            day_results = self.analyze_pickle_data(day)
            if day_results is not None:
                all_results.append(day_results)
        
        # Combine all results
        if all_results:
            combined_results = pd.concat(all_results, ignore_index=True)
            
            # Perform statistical analysis
            stats_results = self.perform_statistical_analysis(combined_results)
            
            # Create summary plots
            self.create_summary_plots(combined_results)
            
            # Save results
            results_path = os.path.join(self.donut_dir, 'analysis_results.pkl')
            data_path = os.path.join(self.donut_dir, 'combined_data.csv')
            
            # Save the combined data as CSV
            combined_results.to_csv(data_path, index=False)
            
            # Save statistical results
            self.save_statistical_results(stats_results, results_path)
            
            print(f"\nResults saved:")
            print(f"Data: {data_path}")
            print(f"Statistics: {results_path}")
            
            return combined_results, stats_results
        
        return None, None

class ControlEnhancedDonutAnalyzer:
    def __init__(self, basedir, df_averaged_control):
        """Initialize the analyzer with base settings for control data."""
        self.basedir = basedir
        self.df_averaged_control = df_averaged_control
        
        # Set up directories
        self.results_dir = os.path.join(self.basedir, 'Results')
        self.pickle_dir = os.path.join(self.results_dir, 'control_correlation_pickles')
        self.donut_dir = os.path.join(self.results_dir, 'control_donut_analysis')
        os.makedirs(self.donut_dir, exist_ok=True)
        
        # Set up device
        if torch.backends.mps.is_available():
            self.device = torch.device("mps")
        elif torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        print(f"Using device: {self.device}")
        print(f"Analyzing CONTROL data")
        
        # Define radius factors and colors
        # self.radius_factors = [1, 3, 5, 7, 9, 11, 13, 15]
        self.radius_factors = [1, 3, 5, 7, 9, 11]
        self.zone_colors = plt.cm.tab20(np.linspace(0, 1, len(self.radius_factors)))
        
        # Calculate global cell diameter once
        self.cell_diameter = self.calculate_global_cell_diameter()
        print(f"Global average cell diameter (from control data): {self.cell_diameter:.2f} pixels")

    def calculate_global_cell_diameter(self):
        """Calculate average cell diameter across all FOVs and days from control data."""
        print("\nCalculating global cell diameter from control data...")
        all_areas = []
        combinations = self.df_averaged_control.groupby(['day', 'animal', 'fov']).size().reset_index()
        
        for _, row in tqdm(combinations.iterrows(), total=len(combinations)):
            try:
                roi_folder = os.path.join(self.basedir, row['day'], row['animal'], row['fov'])
                roi_files = [f for f in os.listdir(roi_folder) if f.endswith('RoiSet.zip')]
                
                if roi_files:
                    rois = roifile.ImagejRoi.fromfile(os.path.join(roi_folder, roi_files[0]))
                    for roi in rois:
                        coords = roi.coordinates()
                        x, y = coords[:, 0], coords[:, 1]
                        area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                        all_areas.append(area)
            except Exception as e:
                print(f"Error processing {roi_folder}: {str(e)}")
                continue
        
        mean_area = np.mean(all_areas)
        return 2 * np.sqrt(mean_area / np.pi)

    def create_donut_masks(self, grid_size=(1024, 1024), center=(512, 512)):
        """Create donut-shaped masks for all radius factors."""
        y, x = np.ogrid[:grid_size[0], :grid_size[1]]
        distances = np.sqrt((x - center[0])**2 + (y - center[1])**2)
        
        masks = []
        for i, factor in enumerate(self.radius_factors):
            radius = (self.cell_diameter/2) * factor
            if i == 0:
                mask = distances <= radius
            else:
                inner_radius = (self.cell_diameter/2) * self.radius_factors[i-1]
                mask = (distances > inner_radius) & (distances <= radius)
            masks.append(mask)
        
        return masks

    def analyze_pickle_data(self, day):
        """Analyze control correlation data for a single day."""
        pickle_path = os.path.join(self.pickle_dir, f"control_correlation_data_{day}.pkl")
        if not os.path.exists(pickle_path):
            print(f"No control pickle file found for {day}")
            return None
            
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)
        
        # Initialize results storage
        results_data = []
        masks = self.create_donut_masks()
        
        # Process each FOV
        for animal_fov, fov_data in data.items():
            animal, fov = animal_fov.split('_', 1)
            
            for cs_type in ['cs_plus', 'cs_minus']:
                # Calculate average correlation map
                accum = fov_data[cs_type]['accum']
                count = fov_data[cs_type]['count']
                with np.errstate(divide='ignore', invalid='ignore'):
                    corr_map = np.where(count > 0, accum / count, np.nan)
                
                # Calculate average for each donut zone
                for zone_idx, mask in enumerate(masks):
                    zone_avg = np.nanmean(corr_map[mask])
                    if not np.isnan(zone_avg):  # Only add non-NaN values
                        results_data.append({
                            'day': day,
                            'animal': animal,
                            'fov': fov,
                            'cs_type': 'CS+' if cs_type == 'cs_plus' else 'CS-',
                            'zone_radius': self.radius_factors[zone_idx],
                            'correlation': zone_avg,
                            'data_type': 'control'  # Mark as control data
                        })
        
        return pd.DataFrame(results_data)

    def perform_statistical_analysis(self, results_df):
        """Perform comprehensive statistical analysis on control data."""
        print("\nPerforming Statistical Analysis on CONTROL Data")
        print("=" * 50)
        
        # Create a copy of the dataframe to avoid modifying the original
        df = results_df.copy()
        
        # Convert to categories
        df['zone_radius'] = df['zone_radius'].astype('category')
        df['day'] = df['day'].astype('category')
        df['cs_type'] = df['cs_type'].astype('category')
        df['group_id'] = df['animal'] + '_' + df['fov']
        
        try:
            # Fit mixed-effects model with simplified formula
            model = sm.MixedLM.from_formula(
                "correlation ~ C(zone_radius) + C(day) + C(cs_type)",
                groups="group_id",
                data=df
            )
            
            results = model.fit()
            print("\nMixed-Effects Model Results (CONTROL):")
            print(results.summary())
        except Exception as e:
            print(f"Error in mixed model: {str(e)}")
            results = None
        
        # Perform ANOVAs for main effects
        print("\nANOVA Results (CONTROL):")
        
        # Zone radius effects
        radius_groups = [group['correlation'].values for name, group in df.groupby('zone_radius')]
        radius_groups = [group for group in radius_groups if len(group) > 0]  # Remove empty groups
        radius_stats = stats.f_oneway(*radius_groups)
        print("\n1. Effect of Zone Radius (CONTROL):")
        print(f"F-statistic: {radius_stats.statistic:.3f}")
        print(f"p-value: {radius_stats.pvalue:.3f}")
        
        # Day effects
        day_groups = [group['correlation'].values for name, group in df.groupby('day')]
        day_groups = [group for group in day_groups if len(group) > 0]  # Remove empty groups
        day_stats = stats.f_oneway(*day_groups)
        print("\n2. Effect of Day (CONTROL):")
        print(f"F-statistic: {day_stats.statistic:.3f}")
        print(f"p-value: {day_stats.pvalue:.3f}")
        
        # CS type effects
        cs_type_ttest = stats.ttest_ind(
            df[df['cs_type'] == 'CS+']['correlation'],
            df[df['cs_type'] == 'CS-']['correlation']
        )
        print("\n3. Effect of CS Type (CONTROL):")
        print(f"t-statistic: {cs_type_ttest.statistic:.3f}")
        print(f"p-value: {cs_type_ttest.pvalue:.3f}")
        
        # Perform post-hoc tests
        print("\nPost-hoc Analyses (Tukey HSD) for CONTROL Data:")
        
        # Zone radius comparisons
        radius_tukey = pairwise_tukeyhsd(
            df['correlation'],
            df['zone_radius']
        )
        print("\nZone Radius Comparisons (CONTROL):")
        print(radius_tukey)
        
        # Day comparisons
        day_tukey = pairwise_tukeyhsd(
            df['correlation'],
            df['day']
        )
        print("\nDay Comparisons (CONTROL):")
        print(day_tukey)
        
        # Create summary statistics
        summary_stats = df.groupby(['day', 'zone_radius', 'cs_type'])['correlation'].agg([
            'mean', 'std', 'count'
        ]).reset_index()
        
        print("\nSummary Statistics (CONTROL):")
        print(summary_stats)
        
        return {
            'mixed_model': results,
            'radius_stats': radius_stats,
            'day_stats': day_stats,
            'cs_type_ttest': cs_type_ttest,
            'radius_tukey': radius_tukey,
            'day_tukey': day_tukey,
            'summary_stats': summary_stats
        }

    def create_summary_plots(self, results_df):
        """Create summary visualization plots for control data."""
        # Create figure for summary plots
        plt.figure(figsize=(20, 15))
        
        # 1. Zone radius effects
        plt.subplot(221)
        sns.boxplot(data=results_df, x='zone_radius', y='correlation', hue='cs_type')
        plt.title('CONTROL: Correlation by Zone Radius and CS Type')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        
        # 2. Day effects
        plt.subplot(222)
        sns.boxplot(data=results_df, x='day', y='correlation', hue='cs_type')
        plt.title('CONTROL: Correlation by Day and CS Type')
        plt.xticks(rotation=45)
        
        # # 3. Interaction plot
        # plt.subplot(223)
        # sns.lineplot(data=results_df, x='zone_radius', y='correlation', 
        #             hue='day', style='cs_type', markers=True, dashes=False)
        # plt.title('CONTROL: Day × Zone Radius Interaction')

        # Replace the problematic subplot 223 section in EnhancedDonutAnalyzer.create_summary_plots method

        # 3. Interaction plot - FIXED: Replace seaborn lineplot with manual plotting
        plt.subplot(223)
        
        # Get unique values
        days = sorted(results_df['day'].unique())
        zone_radii = sorted(results_df['zone_radius'].unique())
        cs_types = ['CS+', 'CS-']
        
        # Define colors and styles
        day_colors = plt.cm.tab10(np.linspace(0, 1, len(days)))
        line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
        markers = ['o', 's']       # Circle for CS+, square for CS-
        
        # Plot each combination manually
        for i, day in enumerate(days):
            for j, cs_type in enumerate(cs_types):
                # Filter data for this combination
                mask = (results_df['day'] == day) & (results_df['cs_type'] == cs_type)
                subset_data = results_df[mask]
                
                # Calculate means for each zone radius
                means = []
                for radius in zone_radii:
                    radius_mask = subset_data['zone_radius'] == radius
                    if radius_mask.any():
                        means.append(subset_data.loc[radius_mask, 'correlation'].mean())
                    else:
                        means.append(np.nan)
                
                # Convert to numpy array
                means = np.array(means)
                
                # Plot the line
                plt.plot(zone_radii, means, 
                         color=day_colors[i], 
                         linestyle=line_styles[j],
                         marker=markers[j], 
                         label=f"{day}, {cs_type}")
        
        plt.title('Day × Zone Radius Interaction')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # 4. Distribution plot
        plt.subplot(224)
        sns.violinplot(data=results_df, x='cs_type', y='correlation', hue='day')
        plt.title('CONTROL: Correlation Distribution by CS Type and Day')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.donut_dir, 'control_summary_plots.png'), dpi=300, bbox_inches='tight')
        plt.show()

    def save_statistical_results(self, stats_results, results_path):
        """Save statistical results in a pickle-friendly format."""
        pickle_friendly_results = {
            'radius_stats': {
                'statistic': float(stats_results['radius_stats'].statistic),
                'pvalue': float(stats_results['radius_stats'].pvalue)
            },
            'day_stats': {
                'statistic': float(stats_results['day_stats'].statistic),
                'pvalue': float(stats_results['day_stats'].pvalue)
            },
            'cs_type_ttest': {
                'statistic': float(stats_results['cs_type_ttest'].statistic),
                'pvalue': float(stats_results['cs_type_ttest'].pvalue)
            },
            'radius_tukey': {
                'meandiffs': stats_results['radius_tukey'].meandiffs.tolist(),
                'confint': stats_results['radius_tukey'].confint.tolist(),
                'reject': stats_results['radius_tukey'].reject.tolist()
            },
            'day_tukey': {
                'meandiffs': stats_results['day_tukey'].meandiffs.tolist(),
                'confint': stats_results['day_tukey'].confint.tolist(),
                'reject': stats_results['day_tukey'].reject.tolist()
            },
            'summary_stats': stats_results['summary_stats'].to_dict()
        }
        
        # If mixed model was successful, add its results
        if stats_results['mixed_model'] is not None:
            pickle_friendly_results['mixed_model'] = {
                'params': stats_results['mixed_model'].params.to_dict(),
                'pvalues': stats_results['mixed_model'].pvalues.to_dict(),
                'conf_int': stats_results['mixed_model'].conf_int().to_dict()
            }
        
        with open(results_path, 'wb') as f:
            pickle.dump(pickle_friendly_results, f)

    def run_analysis(self, days=None):
        """Run complete analysis pipeline for control data."""
        if days is None:
            days = self.df_averaged_control['day'].unique()
            
        # Collect results from all days
        all_results = []
        for day in days:
            print(f"\nProcessing CONTROL data for {day}...")
            day_results = self.analyze_pickle_data(day)
            if day_results is not None:
                all_results.append(day_results)
        
        # Combine all results
        if all_results:
            combined_results = pd.concat(all_results, ignore_index=True)
            
            # Perform statistical analysis
            stats_results = self.perform_statistical_analysis(combined_results)
            
            # Create summary plots
            self.create_summary_plots(combined_results)
            
            # Save results
            results_path = os.path.join(self.donut_dir, 'control_analysis_results.pkl')
            data_path = os.path.join(self.donut_dir, 'control_combined_data.csv')
            
            # Save the combined data as CSV
            combined_results.to_csv(data_path, index=False)
            
            # Save statistical results
            self.save_statistical_results(stats_results, results_path)
            
            print(f"\nCONTROL results saved:")
            print(f"Data: {data_path}")
            print(f"Statistics: {results_path}")
            
            return combined_results, stats_results
        
        return None, None

class DonutComparisonAnalyzer:
    def __init__(self, basedir, real_results_df, control_results_df):
        """Initialize the comparison analyzer with results from both real and control data."""
        self.basedir = basedir
        self.real_results_df = real_results_df
        self.control_results_df = control_results_df
        
        # Set up directories
        self.results_dir = os.path.join(self.basedir, 'Results')
        self.comparison_dir = os.path.join(self.results_dir, 'donut_analysis_comparison')
        os.makedirs(self.comparison_dir, exist_ok=True)
        
        print(f"Initializing donut analysis comparison")
        print(f"Saving comparison results to: {self.comparison_dir}")
    
    def prepare_combined_dataframe(self):
        """Prepare a combined dataframe with both real and control data."""
        # Add data_type column if not present
        if 'data_type' not in self.real_results_df.columns:
            self.real_results_df = self.real_results_df.copy()
            self.real_results_df['data_type'] = 'real'
            
        if 'data_type' not in self.control_results_df.columns:
            self.control_results_df = self.control_results_df.copy()
            self.control_results_df['data_type'] = 'control'
        
        # Combine dataframes
        combined_df = pd.concat([self.real_results_df, self.control_results_df], ignore_index=True)
        return combined_df
    
    def perform_comparison_analysis(self, combined_df):
        """Perform statistical analysis comparing real and control data."""
        print("\nPerforming Comparison Analysis (Real vs Control)")
        print("=" * 50)
        
        # Check effect of data_type (real vs control)
        data_type_ttest = stats.ttest_ind(
            combined_df[combined_df['data_type'] == 'real']['correlation'],
            combined_df[combined_df['data_type'] == 'control']['correlation']
        )
        print("\nOverall Effect of Data Type (Real vs Control):")
        print(f"t-statistic: {data_type_ttest.statistic:.3f}")
        print(f"p-value: {data_type_ttest.pvalue:.3f}")
        
        # Check interactions by running t-tests for each day, zone_radius and cs_type
        print("\nDetailed Comparisons by Factors:")
        
        results = []
        
        # Iterate through days
        for day in combined_df['day'].unique():
            # Iterate through CS types
            for cs_type in combined_df['cs_type'].unique():
                # Iterate through zone radii
                for radius in combined_df['zone_radius'].unique():
                    # Create filter
                    filter_real = ((combined_df['day'] == day) & 
                                   (combined_df['cs_type'] == cs_type) & 
                                   (combined_df['zone_radius'] == radius) & 
                                   (combined_df['data_type'] == 'real'))
                    
                    filter_control = ((combined_df['day'] == day) & 
                                     (combined_df['cs_type'] == cs_type) & 
                                     (combined_df['zone_radius'] == radius) & 
                                     (combined_df['data_type'] == 'control'))
                    
                    real_values = combined_df[filter_real]['correlation']
                    control_values = combined_df[filter_control]['correlation']
                    
                    # Only perform test if we have enough data
                    if len(real_values) >= 5 and len(control_values) >= 5:
                        try:
                            t_stat, p_val = stats.ttest_ind(real_values, control_values)
                            
                            # Calculate mean difference
                            mean_diff = real_values.mean() - control_values.mean()
                            
                            results.append({
                                'day': day,
                                'cs_type': cs_type,
                                'zone_radius': radius,
                                'real_mean': real_values.mean(),
                                'control_mean': control_values.mean(),
                                'mean_diff': mean_diff,
                                't_statistic': t_stat,
                                'p_value': p_val,
                                'significant': p_val < 0.05,
                                'real_n': len(real_values),
                                'control_n': len(control_values)
                            })
                        except Exception as e:
                            print(f"Error comparing {day}, {cs_type}, radius {radius}: {str(e)}")
        
        # Convert to dataframe
        comparison_results = pd.DataFrame(results)
        
        # Sort by significance and mean difference
        comparison_results = comparison_results.sort_values(
            by=['significant', 'mean_diff'], 
            ascending=[False, False]
        )
        
        # Print summary of significant findings
        significant_results = comparison_results[comparison_results['significant']]
        print(f"\nFound {len(significant_results)} significant differences (p < 0.05)")
        
        if len(significant_results) > 0:
            print("\nTop 10 most significant differences (Real - Control):")
            print(significant_results.head(10)[['day', 'cs_type', 'zone_radius', 'mean_diff', 'p_value']])
        
        return {
            'data_type_ttest': data_type_ttest,
            'comparison_results': comparison_results
        }
    
    # def create_comparison_plots(self, combined_df, comparison_results=None):
    #     """Create visualization plots comparing real and control data."""
    #     # 1. Overall comparison
    #     plt.figure(figsize=(20, 15))
        
    #     # Main effect of data_type
    #     plt.subplot(221)
    #     sns.boxplot(data=combined_df, x='data_type', y='correlation', hue='cs_type')
    #     plt.title('Overall: Real vs Control Comparison')
    #     plt.xlabel('Data Type')
    #     plt.ylabel('Correlation')
        
    #     # Comparison by zone radius
    #     plt.subplot(222)
    #     sns.lineplot(data=combined_df, x='zone_radius', y='correlation', 
    #                 hue='data_type', style='cs_type', markers=True, dashes=False)
    #     plt.title('Real vs Control by Zone Radius')
    #     plt.xlabel('Zone Radius (× cell diameter)')
        
    #     # Comparison by day
    #     plt.subplot(223)
    #     sns.boxplot(data=combined_df, x='day', y='correlation', hue='data_type')
    #     plt.title('Real vs Control by Day')
    #     plt.xticks(rotation=45)
        
    #     # Distribution comparison
    #     plt.subplot(224)
    #     sns.histplot(data=combined_df, x='correlation', hue='data_type', 
    #                 element='step', common_norm=False, stat='density')
    #     plt.title('Correlation Distribution: Real vs Control')
        
    #     plt.tight_layout()
    #     plt.savefig(os.path.join(self.comparison_dir, 'comparison_summary.pdf'), dpi=300, bbox_inches='tight')
    #     plt.show()
        
    #     # 2. Create heatmap of differences if we have comparison results
    #     if comparison_results is not None:
    #         comparison_df = comparison_results['comparison_results']
            
    #         # Create pivoted dataframe for heatmap (mean difference)
    #         pivot_df = comparison_df.pivot_table(
    #             index='day', 
    #             columns=['cs_type', 'zone_radius'],
    #             values='mean_diff'
    #         )
            
    #         # Create significance mask (True for non-significant)
    #         sig_mask = comparison_df.pivot_table(
    #             index='day', 
    #             columns=['cs_type', 'zone_radius'],
    #             values='p_value'
    #         ) >= 0.05
            
    #         # Create heatmap figure
    #         plt.figure(figsize=(18, 10))
    #         sns.heatmap(pivot_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
    #                    mask=sig_mask, cbar_kws={'label': 'Mean Difference (Real - Control)'})
    #         plt.title('Mean Difference Between Real and Control Correlations\n(Only Significant Differences Shown, p < 0.05)')
    #         plt.tight_layout()
    #         plt.savefig(os.path.join(self.comparison_dir, 'difference_heatmap.png'), dpi=300, bbox_inches='tight')
    #         plt.show()

    def create_comparison_plots(self, combined_df, comparison_results=None):
        """Create visualization plots comparing real and control data."""
        # 1. Overall comparison
        plt.figure(figsize=(20, 15))
        
        # Main effect of data_type
        plt.subplot(221)
        sns.boxplot(data=combined_df, x='data_type', y='correlation', hue='cs_type')
        plt.title('Overall: Real vs Control Comparison')
        plt.xlabel('Data Type')
        plt.ylabel('Correlation')
        
        # Comparison by zone radius - FIXED: Replace seaborn lineplot with manual plotting
        plt.subplot(222)
        
        # Get unique values
        zone_radii = sorted(combined_df['zone_radius'].unique())
        data_types = ['real', 'control']
        cs_types = ['CS+', 'CS-']
        
        # Define colors and styles
        data_type_colors = {'real': 'blue', 'control': 'red'}
        line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
        markers = {'CS+': 'o', 'CS-': 's'}
        
        # Plot each combination manually
        for data_type in data_types:
            for j, cs_type in enumerate(cs_types):
                # Filter data for this combination
                mask = (combined_df['data_type'] == data_type) & (combined_df['cs_type'] == cs_type)
                subset_data = combined_df[mask]
                
                # Calculate means for each zone radius
                means = []
                for radius in zone_radii:
                    radius_mask = subset_data['zone_radius'] == radius
                    if radius_mask.any():
                        means.append(subset_data.loc[radius_mask, 'correlation'].mean())
                    else:
                        means.append(np.nan)
                
                # Convert to numpy array
                means = np.array(means)
                
                # Plot the line
                plt.plot(zone_radii, means, 
                         color=data_type_colors[data_type], 
                         linestyle=line_styles[j],
                         marker=markers[cs_type], 
                         label=f"{data_type}, {cs_type}")
        
        plt.title('Real vs Control by Zone Radius')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        plt.legend()
        
        # Comparison by day
        plt.subplot(223)
        sns.boxplot(data=combined_df, x='day', y='correlation', hue='data_type')
        plt.title('Real vs Control by Day')
        plt.xticks(rotation=45)
        
        # Distribution comparison
        plt.subplot(224)
        sns.histplot(data=combined_df, x='correlation', hue='data_type', 
                    element='step', common_norm=False, stat='density')
        plt.title('Correlation Distribution: Real vs Control')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.comparison_dir, 'comparison_summary.pdf'), dpi=300, bbox_inches='tight')
        plt.show()
        
        # 2. Create heatmap of differences if we have comparison results
        if comparison_results is not None:
            comparison_df = comparison_results['comparison_results']
            
            # Create pivoted dataframe for heatmap (mean difference)
            pivot_df = comparison_df.pivot_table(
                index='day', 
                columns=['cs_type', 'zone_radius'],
                values='mean_diff'
            )
            
            # Create significance mask (True for non-significant)
            sig_mask = comparison_df.pivot_table(
                index='day', 
                columns=['cs_type', 'zone_radius'],
                values='p_value'
            ) >= 0.05
            
            # Create heatmap figure
            plt.figure(figsize=(18, 10))
            sns.heatmap(pivot_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
                       mask=sig_mask, cbar_kws={'label': 'Mean Difference (Real - Control)'})
            plt.title('Mean Difference Between Real and Control Correlations\n(Only Significant Differences Shown, p < 0.05)')
            plt.tight_layout()
            plt.savefig(os.path.join(self.comparison_dir, 'difference_heatmap.png'), dpi=300, bbox_inches='tight')
            plt.show()
        
    def run_comparison(self):
        """Run the full comparison analysis."""
        print("\nRunning donut analysis comparison between real and control data...")
        
        # Prepare combined dataframe
        combined_df = self.prepare_combined_dataframe()
        
        # Save combined data
        combined_path = os.path.join(self.comparison_dir, 'combined_real_control_data.csv')
        combined_df.to_csv(combined_path, index=False)
        print(f"Saved combined data to: {combined_path}")
        
        # Perform comparison analysis
        comparison_results = self.perform_comparison_analysis(combined_df)
        
        # Create comparison plots
        self.create_comparison_plots(combined_df, comparison_results)
        
        # Save comparison results
        comparison_path = os.path.join(self.comparison_dir, 'comparison_results.csv')
        if comparison_results and 'comparison_results' in comparison_results:
            comparison_results['comparison_results'].to_csv(comparison_path, index=False)
            print(f"Saved comparison results to: {comparison_path}")
        
        return {
            'combined_df': combined_df,
            'comparison_results': comparison_results
        }

def run_all_donut_analyses(basedir, df_averaged, df_averaged_control, days=None):
    """Run all donut analyses: real, control, and comparison."""
    if days is None:
        days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    # 1. Run analysis on real data
    print("\n" + "="*50)
    print("Running donut analysis on REAL data...")
    print("="*50)
    real_analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
    real_results_df, real_stats = real_analyzer.run_analysis(days)
    
    # 2. Run analysis on control data
    print("\n" + "="*50)
    print("Running donut analysis on CONTROL data...")
    print("="*50)
    control_analyzer = ControlEnhancedDonutAnalyzer(basedir, df_averaged_control)
    control_results_df, control_stats = control_analyzer.run_analysis(days)
    
    # 3. Run comparison analysis
    print("\n" + "="*50)
    print("Running donut analysis COMPARISON...")
    print("="*50)
    comparison_analyzer = DonutComparisonAnalyzer(basedir, real_results_df, control_results_df)
    comparison_results = comparison_analyzer.run_comparison()
    
    print("\nAll donut analyses complete!")
    
    return {
        'real_analyzer': real_analyzer,
        'control_analyzer': control_analyzer,
        'comparison_analyzer': comparison_analyzer,
        'real_results': real_results_df,
        'control_results': control_results_df,
        'comparison_results': comparison_results
    }

# Example usage with different modes
def main():
    """Main function to run the analysis."""
    # Set up paths
    basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'    
    # basedir = r'/Users/MAC/Offline/Jackie/WholeCell2 Final'

    # Define days to process
    days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']



##### SET IF YOU WANT THE REAL DATA, CONTROL DATA, OR COMPARISOIN ANALYSIS TO OCCUR HERE ######    
    


    # Choose which analysis to run
    run_mode = 'all'  # Options: 'real', 'control', 'all', or 'comparison'
    
    try:
        df_averaged = None
        df_averaged_control = None
        
        # Try to find and load required dataframes if they're not already in memory
        if run_mode in ['control', 'all', 'comparison']:
            try:
                import pandas as pd
                control_df_path = os.path.join(basedir, 'Results', 'averaged_data_control_cues_0.pkl')
                if os.path.exists(control_df_path):
                    df_averaged_control = pd.read_pickle(control_df_path)
                    print(f"Loaded df_averaged_control from {control_df_path}")
                else:
                    raise FileNotFoundError(f"Could not find control data at {control_df_path}")
            except Exception as e:
                print(f"Error loading control dataframe: {str(e)}")
                return
                
        if run_mode in ['real', 'all', 'comparison']:
            try:
                import pandas as pd
                real_df_path = os.path.join(basedir, 'Results', 'averaged_data_cues_0.pkl')
                if os.path.exists(real_df_path):
                    df_averaged = pd.read_pickle(real_df_path)
                    print(f"Loaded df_averaged from {real_df_path}")
                else:
                    raise FileNotFoundError(f"Could not find real data at {real_df_path}")
            except Exception as e:
                print(f"Error loading real dataframe: {str(e)}")
                return
        
        # Run the selected analysis
        if run_mode == 'real':
            # Run analysis for real data only
            real_analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
            real_results_df, real_stats = real_analyzer.run_analysis(days)
            
            if real_results_df is not None:
                print("\nReal data analysis complete!")
                print("Results saved in:", real_analyzer.donut_dir)
            else:
                print("\nNo real data results were generated. Please check for errors.")
                
        elif run_mode == 'control':
            # Run analysis for control data only
            control_analyzer = ControlEnhancedDonutAnalyzer(basedir, df_averaged_control)
            control_results_df, control_stats = control_analyzer.run_analysis(days)
            
            if control_results_df is not None:
                print("\nControl data analysis complete!")
                print("Results saved in:", control_analyzer.donut_dir)
            else:
                print("\nNo control data results were generated. Please check for errors.")
                
        elif run_mode == 'all':
            # Run all analyses: real, control, and comparison
            results = run_all_donut_analyses(basedir, df_averaged, df_averaged_control, days)
            print("\nComplete analysis finished!")
            
        elif run_mode == 'comparison':
            # First run real and control analyses separately
            real_analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
            real_results_df, _ = real_analyzer.run_analysis(days)
            
            control_analyzer = ControlEnhancedDonutAnalyzer(basedir, df_averaged_control)
            control_results_df, _ = control_analyzer.run_analysis(days)
            
            # Then run comparison
            if real_results_df is not None and control_results_df is not None:
                comparison_analyzer = DonutComparisonAnalyzer(basedir, real_results_df, control_results_df)
                comparison_results = comparison_analyzer.run_comparison()
                print("\nComparison analysis complete!")
            else:
                print("\nCannot run comparison - missing real or control results.")
                
        else:
            print(f"\nInvalid run_mode: {run_mode}")
            print("Valid options are: 'real', 'control', 'all', or 'comparison'")
            
    except Exception as e:
        print(f"\nError during analysis: {str(e)}")
        import traceback
        traceback.print_exc()  # Print full traceback for debugging

if __name__ == "__main__":
    main()

# Select Comparison Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle
import torch
import roifile
from scipy import stats
import seaborn as sns
from matplotlib.patches import Circle
from tqdm import tqdm
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

class GroupedDonutControlAnalyzer:
    def __init__(self, real_analyzer, control_analyzer, results_df_real=None, results_df_control=None):
        """
        Initialize with existing EnhancedDonutAnalyzer and ControlEnhancedDonutAnalyzer instances 
        and optional results.
        
        Args:
            real_analyzer: An instance of EnhancedDonutAnalyzer with pre-calculated data
            control_analyzer: An instance of ControlEnhancedDonutAnalyzer with pre-calculated data
            results_df_real: Optional DataFrame with results from the real data analysis
            results_df_control: Optional DataFrame with results from the control data analysis
        """
        self.real_analyzer = real_analyzer
        self.control_analyzer = control_analyzer
        self.basedir = real_analyzer.basedir
        
        # Set up main directory for grouped analysis
        self.grouped_dir = os.path.join(self.basedir, 'Results', 'grouped_donut_analysis')
        os.makedirs(self.grouped_dir, exist_ok=True)
        
        # Create subdirectories for real and control data
        self.real_grouped_dir = os.path.join(self.grouped_dir, 'real')
        self.control_grouped_dir = os.path.join(self.grouped_dir, 'control')
        self.comparison_grouped_dir = os.path.join(self.grouped_dir, 'comparison')
        
        os.makedirs(self.real_grouped_dir, exist_ok=True)
        os.makedirs(self.control_grouped_dir, exist_ok=True)
        os.makedirs(self.comparison_grouped_dir, exist_ok=True)
        
        # Define day groupings
        self.day_groups = [
            {'name': 'Acquisition', 'days': ['0_Early', '1_Late']},
            {'name': 'Extinction', 'days': ['1_Late', '2_SucOmit', '3_CueOmit']},
            {'name': 'Reversal', 'days': ['4_RevEarly', '5_RevLate']}
        ]
        
        # Handle results data for real dataset
        if results_df_real is not None:
            # Use provided DataFrame directly if available
            self.results_df_real = results_df_real
        else:
            # If not provided, check if CSV exists (for backward compatibility)
            data_path = os.path.join(real_analyzer.donut_dir, 'combined_data.csv')
            if os.path.exists(data_path):
                self.results_df_real = pd.read_csv(data_path)
                print(f"Loaded real data from {data_path}")
            else:
                # No CSV, so we need to regenerate the data
                print("No pre-existing real results found. Regenerating analysis data...")
                all_results = []
                days = real_analyzer.df_averaged['day'].unique()
                for day in days:
                    print(f"Processing real data for {day}...")
                    day_results = real_analyzer.analyze_pickle_data(day)
                    if day_results is not None:
                        all_results.append(day_results)
                
                if all_results:
                    self.results_df_real = pd.concat(all_results, ignore_index=True)
                else:
                    raise ValueError("Failed to generate real analysis data.")
        
        # Handle results data for control dataset
        if results_df_control is not None:
            # Use provided DataFrame directly if available
            self.results_df_control = results_df_control
        else:
            # If not provided, check if CSV exists (for backward compatibility)
            data_path = os.path.join(control_analyzer.donut_dir, 'control_combined_data.csv')
            if os.path.exists(data_path):
                self.results_df_control = pd.read_csv(data_path)
                print(f"Loaded control data from {data_path}")
            else:
                # No CSV, so we need to regenerate the data
                print("No pre-existing control results found. Regenerating analysis data...")
                all_results = []
                days = control_analyzer.df_averaged_control['day'].unique()
                for day in days:
                    print(f"Processing control data for {day}...")
                    day_results = control_analyzer.analyze_pickle_data(day)
                    if day_results is not None:
                        all_results.append(day_results)
                
                if all_results:
                    self.results_df_control = pd.concat(all_results, ignore_index=True)
                else:
                    raise ValueError("Failed to generate control analysis data.")
        
        # Add data_type column if not present
        if 'data_type' not in self.results_df_real.columns:
            self.results_df_real = self.results_df_real.copy()
            self.results_df_real['data_type'] = 'real'
            
        if 'data_type' not in self.results_df_control.columns:
            self.results_df_control = self.results_df_control.copy()
            self.results_df_control['data_type'] = 'control'
            
        # Save the combined data as CSV for future use
        real_data_path = os.path.join(self.grouped_dir, 'real_combined_data.csv')
        control_data_path = os.path.join(self.grouped_dir, 'control_combined_data.csv')
        
        self.results_df_real.to_csv(real_data_path, index=False)
        self.results_df_control.to_csv(control_data_path, index=False)
        
        print(f"Saved real data to: {real_data_path}")
        print(f"Saved control data to: {control_data_path}")
        
        print(f"GroupedDonutControlAnalyzer initialized with:")
        print(f"- {len(self.results_df_real)} real data points")
        print(f"- {len(self.results_df_control)} control data points")
        print(f"Results will be saved to: {self.grouped_dir}")
    
    def filter_data_by_days(self, df, days):
        """Filter the results DataFrame to include only specified days."""
        return df[df['day'].isin(days)].copy()
    
    def perform_grouped_statistical_analysis(self, group_df, group_name, data_type='real'):
        """
        Perform statistical analysis on a specific day grouping.
        
        Args:
            group_df: DataFrame containing only the days in this group
            group_name: Name of this group for reporting and file naming
            data_type: 'real' or 'control' to indicate which dataset
        
        Returns:
            Dictionary of statistical results
        """
        print(f"\nPerforming Statistical Analysis for {group_name} ({data_type} data)")
        print("=" * 50)
        
        # Create a copy of the dataframe to avoid modifying the original
        df = group_df.copy()
        
        # Convert to categories
        df['zone_radius'] = df['zone_radius'].astype('category')
        df['day'] = df['day'].astype('category')
        df['cs_type'] = df['cs_type'].astype('category')
        df['group_id'] = df['animal'] + '_' + df['fov']
        
        # Store results
        stats_results = {}
        
        # Run mixed-effects model
        try:
            model = sm.MixedLM.from_formula(
                "correlation ~ C(zone_radius) + C(day) + C(cs_type)",
                groups="group_id",
                data=df
            )
            
            results = model.fit()
            print("\nMixed-Effects Model Results:")
            print(results.summary())
            stats_results['mixed_model'] = results
        except Exception as e:
            print(f"Error in mixed model: {str(e)}")
            stats_results['mixed_model'] = None
        
        # If there are only two days, perform a two-way ANOVA
        unique_days = df['day'].unique()
        if len(unique_days) == 2:
            print("\nPerforming Two-way ANOVA (Day × CS Type)")
            try:
                # Fit the two-way ANOVA model
                model = ols('correlation ~ C(day) + C(cs_type) + C(day):C(cs_type)', data=df).fit()
                anova_table = sm.stats.anova_lm(model, typ=2)
                print("\nTwo-way ANOVA Results:")
                print(anova_table)
                stats_results['two_way_anova'] = anova_table
            except Exception as e:
                print(f"Error in two-way ANOVA: {str(e)}")
                stats_results['two_way_anova'] = None
        
        # Perform ANOVAs for main effects
        print("\nANOVA Results:")
        
        # Zone radius effects
        radius_groups = [group['correlation'].values for name, group in df.groupby('zone_radius')]
        radius_groups = [group for group in radius_groups if len(group) > 0]  # Remove empty groups
        radius_stats = stats.f_oneway(*radius_groups)
        print("\n1. Effect of Zone Radius:")
        print(f"F-statistic: {radius_stats.statistic:.3f}")
        print(f"p-value: {radius_stats.pvalue:.3f}")
        stats_results['radius_stats'] = radius_stats
        
        # Day effects
        day_groups = [group['correlation'].values for name, group in df.groupby('day')]
        day_groups = [group for group in day_groups if len(group) > 0]  # Remove empty groups
        day_stats = stats.f_oneway(*day_groups)
        print("\n2. Effect of Day:")
        print(f"F-statistic: {day_stats.statistic:.3f}")
        print(f"p-value: {day_stats.pvalue:.3f}")
        stats_results['day_stats'] = day_stats
        
        # CS type effects - Using .values to convert to numpy array
        cs_plus_vals = df[df['cs_type'] == 'CS+']['correlation'].values
        cs_minus_vals = df[df['cs_type'] == 'CS-']['correlation'].values
        cs_type_ttest = stats.ttest_ind(cs_plus_vals, cs_minus_vals)
        print("\n3. Effect of CS Type:")
        print(f"t-statistic: {cs_type_ttest.statistic:.3f}")
        print(f"p-value: {cs_type_ttest.pvalue:.3f}")
        stats_results['cs_type_ttest'] = cs_type_ttest
        
        # Perform post-hoc tests
        print("\nPost-hoc Analyses (Tukey HSD):")
        
        # Zone radius comparisons - Using .values to convert to numpy arrays
        radius_tukey = pairwise_tukeyhsd(
            df['correlation'].values,
            df['zone_radius'].values
        )
        print("\nZone Radius Comparisons:")
        print(radius_tukey)
        stats_results['radius_tukey'] = radius_tukey
        
        # Day comparisons (only if more than one day)
        if len(unique_days) > 1:
            day_tukey = pairwise_tukeyhsd(
                df['correlation'].values,
                df['day'].values
            )
            print("\nDay Comparisons:")
            print(day_tukey)
            stats_results['day_tukey'] = day_tukey
        
        # Create summary statistics
        summary_stats = df.groupby(['day', 'zone_radius', 'cs_type'])['correlation'].agg([
            'mean', 'std', 'count'
        ]).reset_index()
        
        print("\nSummary Statistics:")
        print(summary_stats)
        stats_results['summary_stats'] = summary_stats
        
        return stats_results
    
    # def create_grouped_summary_plots(self, group_df, group_name, data_type='real'):
    #     """
    #     Create summary visualization plots for a specific day grouping.
        
    #     Args:
    #         group_df: DataFrame containing only the days in this group
    #         group_name: Name of this group for reporting and file naming
    #         data_type: 'real' or 'control' to indicate which dataset
    #     """
    #     # Create a copy of the dataframe
    #     df = group_df.copy()
        
    #     # Create figure for summary plots
    #     plt.figure(figsize=(20, 15))
        
    #     # 1. Zone radius effects as box plot with strip plot overlay
    #     plt.subplot(221)
        
    #     # Get unique values for grouping
    #     radius_values = sorted(df['zone_radius'].unique())
    #     cs_types = ['CS+', 'CS-']
    #     colors = ['purple', 'grey']  # Purple and grey
        
    #     # Setup positions for boxes
    #     positions = []
    #     box_data = []
    #     for i, radius in enumerate(radius_values):
    #         for j, cs in enumerate(cs_types):
    #             positions.append(i + j*0.4 - 0.2)
    #             data = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
    #             box_data.append(data)
        
    #     # Create box plot
    #     boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.25, 
    #                showfliers=False, showcaps=False, showmeans=False, showbox=True, 
    #                whiskerprops={'visible': False})
        
    #     # Color the boxes
    #     for i, box in enumerate(boxes['boxes']):
    #         box.set(facecolor=colors[i % 2])
        
    #     # Add strip plot points
    #     for i, radius in enumerate(radius_values):
    #         for j, cs in enumerate(cs_types):
    #             pos = i + j*0.4 - 0.2
    #             points = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
    #             x_jitter = np.random.normal(pos, 0.00, size=len(points))
    #             plt.scatter(x_jitter, points, color=colors[j], alpha=0.25, s=60)
        
    #     # Set x-ticks and labels
    #     plt.xticks([i for i in range(len(radius_values))], [str(r) for r in radius_values])
    #     title_prefix = f"{data_type.upper()}: " if data_type == 'control' else ""
    #     plt.title(f'{title_prefix}{group_name}: Correlation by Zone Radius and CS Type')
    #     plt.xlabel('Zone Radius (× cell diameter)')
    #     plt.ylabel('Correlation')
        
    #     # Add legend
    #     plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
    #     # 2. Day effects as box plot with strip plot overlay
    #     plt.subplot(222)
        
    #     # Get unique days
    #     days = sorted(df['day'].unique())
        
    #     # Setup positions for boxes
    #     positions = []
    #     box_data = []
    #     for i, day in enumerate(days):
    #         for j, cs in enumerate(cs_types):
    #             positions.append(i + j*0.4 - 0.2)
    #             data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
    #             box_data.append(data)
        
        
    #     # Create box plot without whiskers
    #     boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.3, 
    #                showfliers=False, showcaps=False, showmeans=False, showbox=True, 
    #                whiskerprops={'visible': False})
        
    #     # Color the boxes
    #     for i, box in enumerate(boxes['boxes']):
    #         box.set(facecolor=colors[i % 2])
        
    #     # Add strip plot points
    #     for i, day in enumerate(days):
    #         for j, cs in enumerate(cs_types):
    #             pos = i + j*0.4 - 0.2
    #             points = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
    #             x_jitter = np.random.normal(pos, 0.0, size=len(points))
    #             plt.scatter(x_jitter, points, color=colors[j], alpha=0.3, s=60)
        
    #     # Set x-ticks and labels
    #     plt.xticks([i for i in range(len(days))], days, rotation=45)
    #     plt.title(f'{title_prefix}{group_name}: Correlation by Day and CS Type')
        
    #     # Add legend
    #     plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
    #     # 3. Interaction plot - Line plot with markers AND SEM bands
    #     plt.subplot(223)
        
    #     # First convert problematic categorical columns to strings to avoid indexing issues
    #     df['zone_radius_str'] = df['zone_radius'].astype(str)
    #     df['day_str'] = df['day']
        
    #     # Get unique values
    #     zone_radii = sorted(df['zone_radius'].unique())
        
    #     # Create color maps for days based on their order (earlier days get lighter colors)
    #     day_indices = {day: i for i, day in enumerate(days)}
        
    #     # Function to get color based on day and CS type
    #     def get_color(day, cs_type):
    #         day_idx = day_indices[day]
    #         day_factor = day_idx / max(1, len(days) - 1)  # Normalized 0-1
            
    #         if cs_type == 'CS+':
    #             # Purple gradient from light to dark
    #             # Start with light purple #D8BFD8 to dark purple #4B0082
    #             r = int(216 - 149 * day_factor)  # 216 -> 67
    #             g = int(191 - 127 * day_factor)  # 191 -> 64
    #             b = int(216 - 130 * day_factor)  # 216 -> 86
    #             return f'#{r:02x}{g:02x}{b:02x}'
    #         else:  # CS-
    #             # Grey gradient from light to dark
    #             # Start with light grey #D3D3D3 to dark grey #696969
    #             val = int(211 - 106 * day_factor)  # 211 -> 105
    #             return f'#{val:02x}{val:02x}{val:02x}'
        
    #     # Set up line styles
    #     line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
    #     markers = ['o', 's']        
        
    #     # Instead of using seaborn, manually calculate means, SEMs, and plot
    #     for i, day in enumerate(days):
    #         for j, cs_type in enumerate(cs_types):
    #             # Filter data for this day and CS type
    #             mask = (df['day_str'] == day) & (df['cs_type'] == cs_type)
    #             day_cs_data = df[mask]
                
    #             # Calculate mean and SEM for each zone radius
    #             means = []
    #             sems = []
    #             for radius in zone_radii:
    #                 radius_mask = day_cs_data['zone_radius'] == radius
    #                 if radius_mask.any():
    #                     values = day_cs_data.loc[radius_mask, 'correlation'].values
    #                     means.append(np.mean(values))
    #                     # Calculate SEM: standard deviation / sqrt(n)
    #                     sems.append(np.std(values, ddof=1) / np.sqrt(len(values)))
    #                 else:
    #                     means.append(np.nan)
    #                     sems.append(np.nan)
                
    #             # Convert to numpy arrays for easier manipulation
    #             means = np.array(means)
    #             sems = np.array(sems)
                
    #             # Get the color based on day and CS type
    #             color = get_color(day, cs_type)
                
    #             # Plot the mean line
    #             plt.plot(zone_radii, means, 
    #                      color=color, 
    #                      linestyle=line_styles[j],
    #                      marker=markers[j], 
    #                      linewidth=2,
    #                      label=f"{day}, {cs_type}")
                
    #             # Add SEM bands (fill_between creates the shaded area)
    #             plt.fill_between(zone_radii, 
    #                             means - sems, 
    #                             means + sems, 
    #                             color=color, 
    #                             alpha=0.2)
        
    #     plt.title(f'{title_prefix}{group_name}: Day × Zone Radius Interaction')
    #     plt.xlabel('Zone Radius (× cell diameter)')
    #     plt.ylabel('Correlation')
    #     plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    #     plt.ylim(0, 1)
        
    #     # 4. Distribution as violin plot (using matplotlib's violin plot)
    #     plt.subplot(224)
        
    #     # Create a day color map for violins
    #     violin_colors = {}
    #     for day in days:
    #         # Use the same function but with CS+ colors for violins
    #         violin_colors[day] = get_color(day, 'CS+')
        
    #     # Prepare data for violin plots
    #     violin_positions = []
    #     violin_data = []
    #     violin_day_colors = []
        
    #     for j, cs in enumerate(cs_types):
    #         for i, day in enumerate(days):
    #             pos = j + i/(len(days)+1)
    #             violin_positions.append(pos)
    #             data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
    #             violin_data.append(data)
    #             violin_day_colors.append(violin_colors[day])
        
    #     # Create violin plot
    #     violin_parts = plt.violinplot(violin_data, positions=violin_positions, 
    #                                 widths=0.15, showmeans=True, showextrema=True)
        
    #     # Customize violin colors
    #     for i, pc in enumerate(violin_parts['bodies']):
    #         pc.set_facecolor(violin_day_colors[i])
    #         pc.set_alpha(0.7)
        
    #     # Set x-ticks
    #     plt.xticks([0.5, 1.5], cs_types)
    #     plt.title(f'{title_prefix}{group_name}: Correlation Distribution by CS Type and Day')
    #     plt.ylabel('Correlation')
        
    #     # Add legend for violin plot
    #     legend_patches = [plt.Line2D([0], [0], color=violin_colors[day], lw=4) for day in days]
    #     plt.legend(legend_patches, days, loc='upper right')
        
    #     plt.tight_layout()
        
    #     # Determine the appropriate save directory
    #     if data_type == 'real':
    #         save_dir = self.real_grouped_dir
    #     else:
    #         save_dir = self.control_grouped_dir
            
    #     plt.savefig(os.path.join(save_dir, f'{group_name}_summary_plots.pdf'), 
    #                dpi=300, bbox_inches='tight')
    #     plt.show()

    def create_grouped_summary_plots(self, group_df, group_name, data_type='real'):
        """
        Create summary visualization plots for a specific day grouping.
        
        Args:
            group_df: DataFrame containing only the days in this group
            group_name: Name of this group for reporting and file naming
            data_type: 'real' or 'control' to indicate which dataset
        """
        # Create a copy of the dataframe
        df = group_df.copy()
        
        # Create figure for summary plots
        plt.figure(figsize=(20, 15))
        
        # 1. Zone radius effects as box plot with strip plot overlay
        plt.subplot(221)
        
        # Get unique values for grouping
        radius_values = sorted(df['zone_radius'].unique())
        cs_types = ['CS+', 'CS-']
        colors = ['purple', 'grey']  # Purple and grey
        
        # Setup positions for boxes
        positions = []
        box_data = []
        for i, radius in enumerate(radius_values):
            for j, cs in enumerate(cs_types):
                positions.append(i + j*0.4 - 0.2)
                data = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
                box_data.append(data)
        
        # Create box plot
        boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.25, 
                   showfliers=False, showcaps=False, showmeans=False, showbox=True, 
                   whiskerprops={'visible': False})
        
        # Color the boxes
        for i, box in enumerate(boxes['boxes']):
            box.set(facecolor=colors[i % 2])
        
        # Add strip plot points
        for i, radius in enumerate(radius_values):
            for j, cs in enumerate(cs_types):
                pos = i + j*0.4 - 0.2
                points = df[(df['zone_radius'] == radius) & (df['cs_type'] == cs)]['correlation'].values
                x_jitter = np.random.normal(pos, 0.00, size=len(points))
                plt.scatter(x_jitter, points, color=colors[j], alpha=0.25, s=60)
        
        # Set x-ticks and labels
        plt.xticks([i for i in range(len(radius_values))], [str(r) for r in radius_values])
        title_prefix = f"{data_type.upper()}: " if data_type == 'control' else ""
        plt.title(f'{title_prefix}{group_name}: Correlation by Zone Radius and CS Type')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        
        # Add legend
        plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
        # 2. Day effects as box plot with strip plot overlay
        plt.subplot(222)
        
        # Get unique days
        days = sorted(df['day'].unique())
        
        # Setup positions for boxes
        positions = []
        box_data = []
        for i, day in enumerate(days):
            for j, cs in enumerate(cs_types):
                positions.append(i + j*0.4 - 0.2)
                data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                box_data.append(data)
        
        
        # Create box plot without whiskers
        boxes = plt.boxplot(box_data, positions=positions, patch_artist=True, widths=0.3, 
                   showfliers=False, showcaps=False, showmeans=False, showbox=True, 
                   whiskerprops={'visible': False})
        
        # Color the boxes
        for i, box in enumerate(boxes['boxes']):
            box.set(facecolor=colors[i % 2])
        
        # Add strip plot points
        for i, day in enumerate(days):
            for j, cs in enumerate(cs_types):
                pos = i + j*0.4 - 0.2
                points = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                x_jitter = np.random.normal(pos, 0.0, size=len(points))
                plt.scatter(x_jitter, points, color=colors[j], alpha=0.3, s=60)
        
        # Set x-ticks and labels
        plt.xticks([i for i in range(len(days))], days, rotation=45)
        plt.title(f'{title_prefix}{group_name}: Correlation by Day and CS Type')
        
        # Add legend
        plt.legend([boxes['boxes'][0], boxes['boxes'][1]], cs_types, title='CS Type')
        
        # 3. Interaction plot - Line plot with markers AND SEM bands
        plt.subplot(223)
        
        # First convert problematic categorical columns to strings to avoid indexing issues
        df['zone_radius_str'] = df['zone_radius'].astype(str)
        df['day_str'] = df['day']
        
        # Get unique values
        zone_radii = sorted(df['zone_radius'].unique())
        
        # Create color maps for days based on their order (earlier days get lighter colors)
        day_indices = {day: i for i, day in enumerate(days)}
        
        # Function to get color based on day and CS type
        def get_color(day, cs_type):
            day_idx = day_indices[day]
            day_factor = day_idx / max(1, len(days) - 1)  # Normalized 0-1
            
            if cs_type == 'CS+':
                # Purple gradient from light to dark
                # Start with light purple #D8BFD8 to dark purple #4B0082
                r = int(216 - 149 * day_factor)  # 216 -> 67
                g = int(191 - 127 * day_factor)  # 191 -> 64
                b = int(216 - 130 * day_factor)  # 216 -> 86
                return f'#{r:02x}{g:02x}{b:02x}'
            else:  # CS-
                # Grey gradient from light to dark
                # Start with light grey #D3D3D3 to dark grey #696969
                val = int(211 - 106 * day_factor)  # 211 -> 105
                return f'#{val:02x}{val:02x}{val:02x}'
        
        # Set up line styles
        line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
        markers = ['o', 's']        
        
        # Instead of using seaborn, manually calculate means, SEMs, and plot
        for i, day in enumerate(days):
            for j, cs_type in enumerate(cs_types):
                # Filter data for this day and CS type
                mask = (df['day_str'] == day) & (df['cs_type'] == cs_type)
                day_cs_data = df[mask]
                
                # Calculate mean and SEM for each zone radius
                means = []
                sems = []
                for radius in zone_radii:
                    radius_mask = day_cs_data['zone_radius'] == radius
                    if radius_mask.any():
                        values = day_cs_data.loc[radius_mask, 'correlation'].values
                        means.append(np.mean(values))
                        # Calculate SEM: standard deviation / sqrt(n)
                        sems.append(np.std(values, ddof=1) / np.sqrt(len(values)))
                    else:
                        means.append(np.nan)
                        sems.append(np.nan)
                
                # Convert to numpy arrays for easier manipulation
                means = np.array(means)
                sems = np.array(sems)
                
                # Get the color based on day and CS type
                color = get_color(day, cs_type)
                
                # Plot the mean line
                plt.plot(zone_radii, means, 
                         color=color, 
                         linestyle=line_styles[j],
                         marker=markers[j], 
                         linewidth=2,
                         label=f"{day}, {cs_type}")
                
                # Add SEM bands (fill_between creates the shaded area)
                plt.fill_between(zone_radii, 
                                means - sems, 
                                means + sems, 
                                color=color, 
                                alpha=0.2)
        
        plt.title(f'{title_prefix}{group_name}: Day × Zone Radius Interaction')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.ylim(0, 1)
        
        # 4. Distribution as violin plot (using matplotlib's violin plot)
        plt.subplot(224)
        
        # Create a day color map for violins
        violin_colors = {}
        for day in days:
            # Use the same function but with CS+ colors for violins
            violin_colors[day] = get_color(day, 'CS+')
        
        # Prepare data for violin plots
        violin_positions = []
        violin_data = []
        violin_day_colors = []
        
        for j, cs in enumerate(cs_types):
            for i, day in enumerate(days):
                pos = j + i/(len(days)+1)
                violin_positions.append(pos)
                data = df[(df['day'] == day) & (df['cs_type'] == cs)]['correlation'].values
                violin_data.append(data)
                violin_day_colors.append(violin_colors[day])
        
        # Create violin plot
        violin_parts = plt.violinplot(violin_data, positions=violin_positions, 
                                    widths=0.15, showmeans=True, showextrema=True)
        
        # Customize violin colors
        for i, pc in enumerate(violin_parts['bodies']):
            pc.set_facecolor(violin_day_colors[i])
            pc.set_alpha(0.7)
        
        # Set x-ticks
        plt.xticks([0.5, 1.5], cs_types)
        plt.title(f'{title_prefix}{group_name}: Correlation Distribution by CS Type and Day')
        plt.ylabel('Correlation')
        
        # Add legend for violin plot
        legend_patches = [plt.Line2D([0], [0], color=violin_colors[day], lw=4) for day in days]
        plt.legend(legend_patches, days, loc='upper right')
        
        plt.tight_layout()
        
        # Determine the appropriate save directory
        if data_type == 'real':
            save_dir = self.real_grouped_dir
        else:
            save_dir = self.control_grouped_dir
            
        plt.savefig(os.path.join(save_dir, f'{group_name}_summary_plots.pdf'), 
                   dpi=300, bbox_inches='tight')
        plt.show()

    
    def perform_grouped_comparison_analysis(self, real_group_df, control_group_df, group_name):
        """
        Perform statistical comparison between real and control data for a specific day grouping.
        
        Args:
            real_group_df: DataFrame containing only the days in this group for real data
            control_group_df: DataFrame containing only the days in this group for control data
            group_name: Name of this group for reporting and file naming
        
        Returns:
            Dictionary of comparison results
        """
        print(f"\nPerforming Comparison Analysis for {group_name} (Real vs Control)")
        print("=" * 50)
        
        # Create a copy and ensure we have data_type column
        real_df = real_group_df.copy()
        control_df = control_group_df.copy()
        
        if 'data_type' not in real_df.columns:
            real_df['data_type'] = 'real'
        if 'data_type' not in control_df.columns:
            control_df['data_type'] = 'control'
        
        # Combine dataframes for comparison
        combined_df = pd.concat([real_df, control_df], ignore_index=True)
        
        # Check effect of data_type (real vs control)
        data_type_ttest = stats.ttest_ind(
            real_df['correlation'],
            control_df['correlation']
        )
        print("\nOverall Effect of Data Type (Real vs Control):")
        print(f"t-statistic: {data_type_ttest.statistic:.3f}")
        print(f"p-value: {data_type_ttest.pvalue:.3f}")
        
        # Check interactions by running t-tests for each day, zone_radius and cs_type
        print("\nDetailed Comparisons by Factors:")
        
        results = []
        
        # Iterate through days
        for day in combined_df['day'].unique():
            # Iterate through CS types
            for cs_type in combined_df['cs_type'].unique():
                # Iterate through zone radii
                for radius in combined_df['zone_radius'].unique():
                    # Create filter
                    filter_real = ((real_df['day'] == day) & 
                                   (real_df['cs_type'] == cs_type) & 
                                   (real_df['zone_radius'] == radius))
                    
                    filter_control = ((control_df['day'] == day) & 
                                      (control_df['cs_type'] == cs_type) & 
                                      (control_df['zone_radius'] == radius))
                    
                    real_values = real_df[filter_real]['correlation']
                    control_values = control_df[filter_control]['correlation']
                    
                    # Only perform test if we have enough data
                    if len(real_values) >= 5 and len(control_values) >= 5:
                        try:
                            t_stat, p_val = stats.ttest_ind(real_values, control_values)
                            
                            # Calculate mean difference
                            mean_diff = real_values.mean() - control_values.mean()
                            
                            results.append({
                                'day': day,
                                'cs_type': cs_type,
                                'zone_radius': radius,
                                'real_mean': real_values.mean(),
                                'control_mean': control_values.mean(),
                                'mean_diff': mean_diff,
                                't_statistic': t_stat,
                                'p_value': p_val,
                                'significant': p_val < 0.05,
                                'real_n': len(real_values),
                                'control_n': len(control_values)
                            })
                        except Exception as e:
                            print(f"Error comparing {day}, {cs_type}, radius {radius}: {str(e)}")
        
        # Convert to dataframe
        comparison_results = pd.DataFrame(results)
        
        # Sort by significance and mean difference
        if len(comparison_results) > 0:
            comparison_results = comparison_results.sort_values(
                by=['significant', 'mean_diff'], 
                ascending=[False, False]
            )
            
            # Print summary of significant findings
            significant_results = comparison_results[comparison_results['significant']]
            print(f"\nFound {len(significant_results)} significant differences (p < 0.05)")
            
            if len(significant_results) > 0:
                print("\nTop most significant differences (Real - Control):")
                print(significant_results.head(10)[['day', 'cs_type', 'zone_radius', 'mean_diff', 'p_value']])
        else:
            print("\nNo valid comparison results found.")
        
        return {
            'data_type_ttest': data_type_ttest,
            'comparison_results': comparison_results,
            'combined_df': combined_df
        }
    
    # def create_grouped_comparison_plots(self, real_group_df, control_group_df, group_name, comparison_results=None):
    #     """
    #     Create visualization plots comparing real and control data for a specific day grouping.
        
    #     Args:
    #         real_group_df: DataFrame containing only the days in this group for real data
    #         control_group_df: DataFrame containing only the days in this group for control data
    #         group_name: Name of this group for reporting and file naming
    #         comparison_results: Optional dictionary with comparison analysis results
    #     """
    #     # Create a copy and ensure we have data_type column
    #     real_df = real_group_df.copy()
    #     control_df = control_group_df.copy()
        
    #     if 'data_type' not in real_df.columns:
    #         real_df['data_type'] = 'real'
    #     if 'data_type' not in control_df.columns:
    #         control_df['data_type'] = 'control'
        
    #     # Combine dataframes for comparison
    #     combined_df = pd.concat([real_df, control_df], ignore_index=True)
        
    #     # 1. Overall comparison
    #     plt.figure(figsize=(20, 15))
        
    #     # Main effect of data_type
    #     plt.subplot(221)
    #     sns.boxplot(data=combined_df, x='data_type', y='correlation', hue='cs_type')
    #     plt.title(f'{group_name}: Real vs Control Comparison')
    #     plt.xlabel('Data Type')
    #     plt.ylabel('Correlation')
        
    #     # Comparison by zone radius
    #     plt.subplot(222)
    #     sns.lineplot(data=combined_df, x='zone_radius', y='correlation', 
    #                 hue='data_type', style='cs_type', markers=True, dashes=False)
    #     plt.title(f'{group_name}: Real vs Control by Zone Radius')
    #     plt.xlabel('Zone Radius (× cell diameter)')
        
    #     # Comparison by day
    #     plt.subplot(223)
    #     sns.boxplot(data=combined_df, x='day', y='correlation', hue='data_type')
    #     plt.title(f'{group_name}: Real vs Control by Day')
    #     plt.xticks(rotation=45)
        
    #     # Distribution comparison
    #     plt.subplot(224)
    #     sns.histplot(data=combined_df, x='correlation', hue='data_type', 
    #                 element='step', common_norm=False, stat='density')
    #     plt.title(f'{group_name}: Correlation Distribution - Real vs Control')
        
    #     plt.tight_layout()
    #     plt.savefig(os.path.join(self.comparison_grouped_dir, f'{group_name}_comparison.pdf'), 
    #                dpi=300, bbox_inches='tight')
    #     plt.show()
        
    #     # 2. Create heatmap of differences if we have comparison results
    #     if comparison_results is not None and 'comparison_results' in comparison_results and len(comparison_results['comparison_results']) > 0:
    #         comparison_df = comparison_results['comparison_results']
            
    #         # Create pivoted dataframe for heatmap (mean difference)
    #         try:
    #             pivot_df = comparison_df.pivot_table(
    #                 index='day', 
    #                 columns=['cs_type', 'zone_radius'],
    #                 values='mean_diff'
    #             )
                
    #             # Create significance mask for highlighting (but not masking)
    #             sig_mask = comparison_df.pivot_table(
    #                 index='day', 
    #                 columns=['cs_type', 'zone_radius'],
    #                 values='p_value'
    #             ) < 0.05
                
    #             # Create heatmap figure - displaying all values
    #             plt.figure(figsize=(18, 10))
    #             # First, create the full heatmap without masking
    #             sns.heatmap(pivot_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
    #                        cbar_kws={'label': 'Mean Difference (Real - Control)'})
    #             plt.title(f'{group_name}: Mean Difference Between Real and Control Correlations')
                
    #             # Optionally, create a second figure highlighting significant values
    #             plt.figure(figsize=(18, 10))
    #             # Add a black border around significant cells
    #             sns.heatmap(pivot_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
    #                        cbar_kws={'label': 'Mean Difference (Real - Control)'},
    #                        linewidths=.5, linecolor='black',
    #                        mask=~sig_mask)  # Invert mask to show only significant
    #             plt.title(f'{group_name}: Mean Difference Between Real and Control Correlations\n(Significant Differences Highlighted, p < 0.05)')
    #             plt.tight_layout()
    #             plt.savefig(os.path.join(self.comparison_grouped_dir, f'{group_name}_difference_heatmap.pdf'), 
    #                        dpi=300, bbox_inches='tight')
    #             plt.show()
    #         except Exception as e:
    #             print(f"Error creating heatmap: {str(e)}")
    #             print("This may happen if there are not enough significant differences to create a meaningful heatmap.")

    def create_grouped_comparison_plots(self, real_group_df, control_group_df, group_name, comparison_results=None):
        """
        Create visualization plots comparing real and control data for a specific day grouping.
        
        Args:
            real_group_df: DataFrame containing only the days in this group for real data
            control_group_df: DataFrame containing only the days in this group for control data
            group_name: Name of this group for reporting and file naming
            comparison_results: Optional dictionary with comparison analysis results
        """
        # Create a copy and ensure we have data_type column
        real_df = real_group_df.copy()
        control_df = control_group_df.copy()
        
        if 'data_type' not in real_df.columns:
            real_df['data_type'] = 'real'
        if 'data_type' not in control_df.columns:
            control_df['data_type'] = 'control'
        
        # Combine dataframes for comparison
        combined_df = pd.concat([real_df, control_df], ignore_index=True)
        
        # 1. Overall comparison
        plt.figure(figsize=(20, 15))
        
        # Main effect of data_type
        plt.subplot(221)
        sns.boxplot(data=combined_df, x='data_type', y='correlation', hue='cs_type')
        plt.title(f'{group_name}: Real vs Control Comparison')
        plt.xlabel('Data Type')
        plt.ylabel('Correlation')
        
        # Comparison by zone radius - FIXED: Replace seaborn lineplot with manual plotting
        plt.subplot(222)
        
        # Get unique values
        zone_radii = sorted(combined_df['zone_radius'].unique())
        data_types = ['real', 'control']
        cs_types = ['CS+', 'CS-']
        
        # Define colors and styles
        data_type_colors = {'real': 'blue', 'control': 'red'}
        line_styles = {'-': 'CS+', '--': 'CS-'}  # Solid for CS+, dashed for CS-
        markers = {'CS+': 'o', 'CS-': 's'}
        
        # Plot each combination manually
        for data_type in data_types:
            for cs_type in cs_types:
                # Filter data for this combination
                mask = (combined_df['data_type'] == data_type) & (combined_df['cs_type'] == cs_type)
                subset_data = combined_df[mask]
                
                # Calculate means for each zone radius
                means = []
                for radius in zone_radii:
                    radius_mask = subset_data['zone_radius'] == radius
                    if radius_mask.any():
                        means.append(subset_data.loc[radius_mask, 'correlation'].mean())
                    else:
                        means.append(np.nan)
                
                # Convert to numpy array
                means = np.array(means)
                
                # Plot the line
                linestyle = '-' if cs_type == 'CS+' else '--'
                plt.plot(zone_radii, means, 
                         color=data_type_colors[data_type], 
                         linestyle=linestyle,
                         marker=markers[cs_type], 
                         label=f"{data_type}, {cs_type}")
        
        plt.title(f'{group_name}: Real vs Control by Zone Radius')
        plt.xlabel('Zone Radius (× cell diameter)')
        plt.ylabel('Correlation')
        plt.legend()
        
        # Comparison by day
        plt.subplot(223)
        sns.boxplot(data=combined_df, x='day', y='correlation', hue='data_type')
        plt.title(f'{group_name}: Real vs Control by Day')
        plt.xticks(rotation=45)
        
        # Distribution comparison
        plt.subplot(224)
        sns.histplot(data=combined_df, x='correlation', hue='data_type', 
                    element='step', common_norm=False, stat='density')
        plt.title(f'{group_name}: Correlation Distribution - Real vs Control')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.comparison_grouped_dir, f'{group_name}_comparison.pdf'), 
                   dpi=300, bbox_inches='tight')
        plt.show()
        
        # 2. Create heatmap of differences if we have comparison results
        if comparison_results is not None and 'comparison_results' in comparison_results and len(comparison_results['comparison_results']) > 0:
            comparison_df = comparison_results['comparison_results']
            
            # Create pivoted dataframe for heatmap (mean difference)
            try:
                pivot_df = comparison_df.pivot_table(
                    index='day', 
                    columns=['cs_type', 'zone_radius'],
                    values='mean_diff'
                )
                
                # Create significance mask for highlighting (but not masking)
                sig_mask = comparison_df.pivot_table(
                    index='day', 
                    columns=['cs_type', 'zone_radius'],
                    values='p_value'
                ) < 0.05
                
                # Create heatmap figure - displaying all values
                plt.figure(figsize=(18, 10))
                # First, create the full heatmap without masking
                sns.heatmap(pivot_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
                           cbar_kws={'label': 'Mean Difference (Real - Control)'})
                plt.title(f'{group_name}: Mean Difference Between Real and Control Correlations')
                
                # Optionally, create a second figure highlighting significant values
                plt.figure(figsize=(18, 10))
                # Add a black border around significant cells
                sns.heatmap(pivot_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
                           cbar_kws={'label': 'Mean Difference (Real - Control)'},
                           linewidths=.5, linecolor='black',
                           mask=~sig_mask)  # Invert mask to show only significant
                plt.title(f'{group_name}: Mean Difference Between Real and Control Correlations\n(Significant Differences Highlighted, p < 0.05)')
                plt.tight_layout()
                plt.savefig(os.path.join(self.comparison_grouped_dir, f'{group_name}_difference_heatmap.pdf'), 
                           dpi=300, bbox_inches='tight')
                plt.show()
            except Exception as e:
                print(f"Error creating heatmap: {str(e)}")
                print("This may happen if there are not enough significant differences to create a meaningful heatmap.")
    
    
    def save_statistical_results(self, stats_results, group_name, data_type='real'):
        """Save statistical results in a pickle-friendly format."""
        pickle_friendly_results = {
            'radius_stats': {
                'statistic': float(stats_results['radius_stats'].statistic),
                'pvalue': float(stats_results['radius_stats'].pvalue)
            },
            'day_stats': {
                'statistic': float(stats_results['day_stats'].statistic),
                'pvalue': float(stats_results['day_stats'].pvalue)
            },
            'cs_type_ttest': {
                'statistic': float(stats_results['cs_type_ttest'].statistic),
                'pvalue': float(stats_results['cs_type_ttest'].pvalue)
            },
            'radius_tukey': {
                'meandiffs': stats_results['radius_tukey'].meandiffs.tolist(),
                'confint': stats_results['radius_tukey'].confint.tolist(),
                'reject': stats_results['radius_tukey'].reject.tolist()
            },
            'summary_stats': stats_results['summary_stats'].to_dict()
        }
        
        # Add two-way ANOVA results if they exist
        if 'two_way_anova' in stats_results and stats_results['two_way_anova'] is not None:
            # Get the actual column names from the ANOVA table
            anova_df = stats_results['two_way_anova']
            anova_columns = anova_df.columns.tolist()
            
            # Create a dictionary with all available columns
            anova_dict = {}
            for col in anova_columns:
                try:
                    # Try to convert to list, but handle non-array-like columns
                    anova_dict[col] = anova_df[col].tolist() if hasattr(anova_df[col], 'tolist') else list(anova_df[col])
                except:
                    # If conversion fails, store as is
                    anova_dict[col] = anova_df[col]
            
            pickle_friendly_results['two_way_anova'] = anova_dict
        
        # Add day Tukey results if they exist
        if 'day_tukey' in stats_results:
            pickle_friendly_results['day_tukey'] = {
                'meandiffs': stats_results['day_tukey'].meandiffs.tolist(),
                'confint': stats_results['day_tukey'].confint.tolist(),
                'reject': stats_results['day_tukey'].reject.tolist()
            }
        
        # If mixed model was successful, add its results
        if stats_results['mixed_model'] is not None:
            pickle_friendly_results['mixed_model'] = {
                'params': stats_results['mixed_model'].params.to_dict(),
                'pvalues': stats_results['mixed_model'].pvalues.to_dict(),
                'conf_int': stats_results['mixed_model'].conf_int().to_dict()
            }
        
        # Determine the appropriate save directory
        if data_type == 'real':
            save_dir = self.real_grouped_dir
        else:
            save_dir = self.control_grouped_dir
            
        results_path = os.path.join(save_dir, f'{group_name}_analysis_results.pkl')
        with open(results_path, 'wb') as f:
            pickle.dump(pickle_friendly_results, f)
        print(f"Statistical results saved to: {results_path}")
    
    def save_comparison_results(self, comparison_results, group_name):
        """Save comparison results in a pickle-friendly format."""
        if comparison_results is None or 'comparison_results' not in comparison_results:
            print(f"No valid comparison results to save for {group_name}")
            return
            
        # Save the comparison DataFrame
        csv_path = os.path.join(self.comparison_grouped_dir, f'{group_name}_comparison_results.csv')
        comparison_results['comparison_results'].to_csv(csv_path, index=False)
        
        # Convert t-test results to pickle-friendly format
        pickle_friendly_results = {
            'data_type_ttest': {
                'statistic': float(comparison_results['data_type_ttest'].statistic),
                'pvalue': float(comparison_results['data_type_ttest'].pvalue)
            },
            'significant_count': int(comparison_results['comparison_results']['significant'].sum()),
            'total_comparisons': len(comparison_results['comparison_results'])
        }
        
        # Save as pickle
        results_path = os.path.join(self.comparison_grouped_dir, f'{group_name}_comparison_stats.pkl')
        with open(results_path, 'wb') as f:
            pickle.dump(pickle_friendly_results, f)
        print(f"Comparison results saved to: {csv_path} and {results_path}")
    
    def run_grouped_analysis(self):
        """Run complete analysis pipeline for all grouped days."""
        for group in self.day_groups:
            group_name = group['name']
            days = group['days']
            
            print(f"\n{'='*50}")
            print(f"Processing {group_name} ({', '.join(days)})...")
            print(f"{'='*50}")
            
            # 1. Process real data
            print("\nAnalyzing REAL data...")
            real_group_df = self.filter_data_by_days(self.results_df_real, days)
            
            if len(real_group_df) == 0:
                print(f"No real data found for days: {days}")
                continue
                
            # Save group data for reference
            real_data_path = os.path.join(self.real_grouped_dir, f'{group_name}_data.csv')
            real_group_df.to_csv(real_data_path, index=False)
            print(f"Real group data saved to: {real_data_path}")
            
            # Perform statistical analysis on real data
            real_stats_results = self.perform_grouped_statistical_analysis(real_group_df, group_name, 'real')
            
            # Create summary plots for real data
            self.create_grouped_summary_plots(real_group_df, group_name, 'real')
            
            # Save statistical results for real data
            self.save_statistical_results(real_stats_results, group_name, 'real')
            
            # 2. Process control data
            print("\nAnalyzing CONTROL data...")
            control_group_df = self.filter_data_by_days(self.results_df_control, days)
            
            if len(control_group_df) == 0:
                print(f"No control data found for days: {days}")
                continue
                
            # Save group data for reference
            control_data_path = os.path.join(self.control_grouped_dir, f'{group_name}_data.csv')
            control_group_df.to_csv(control_data_path, index=False)
            print(f"Control group data saved to: {control_data_path}")
            
            # Perform statistical analysis on control data
            control_stats_results = self.perform_grouped_statistical_analysis(control_group_df, group_name, 'control')
            
            # Create summary plots for control data
            self.create_grouped_summary_plots(control_group_df, group_name, 'control')
            
            # Save statistical results for control data
            self.save_statistical_results(control_stats_results, group_name, 'control')
            
            # 3. Run comparison analysis
            print("\nRunning COMPARISON analysis...")
            comparison_results = self.perform_grouped_comparison_analysis(real_group_df, control_group_df, group_name)
            
            # Create comparison plots
            self.create_grouped_comparison_plots(real_group_df, control_group_df, group_name, comparison_results)
            
            # Save comparison results
            self.save_comparison_results(comparison_results, group_name)
            
            print(f"{group_name} analysis complete!")
        
        print("\nAll grouped analyses complete!")
        print(f"Results saved in: {self.grouped_dir}")


def run_grouped_donut_analysis(basedir=None, real_analyzer=None, control_analyzer=None, 
                             results_df_real=None, results_df_control=None):
    """
    Main function to run the grouped analysis after the original analysis.
    
    Args:
        basedir: Base directory path (required if analyzers not provided)
        real_analyzer: Instance of EnhancedDonutAnalyzer from original analysis
        control_analyzer: Instance of ControlEnhancedDonutAnalyzer from original analysis
        results_df_real: DataFrame with results from real data analysis
        results_df_control: DataFrame with results from control data analysis
        
    Returns:
        Instance of GroupedDonutControlAnalyzer or None if error occurs
    """
    try:
        if real_analyzer is not None and control_analyzer is not None:
            # Use the existing analyzers and optionally the results DataFrames
            grouped_analyzer = GroupedDonutControlAnalyzer(real_analyzer, control_analyzer, 
                                                         results_df_real, results_df_control)
        else:
            if basedir is None:
                raise ValueError("Either analyzers or basedir must be provided")
                
            # Try to load data from CSVs
            import pandas as pd
            
            real_data_path = os.path.join(basedir, 'Results', 'donut_analysis', 'combined_data.csv')
            control_data_path = os.path.join(basedir, 'Results', 'control_donut_analysis', 'control_combined_data.csv')
            
            if os.path.exists(real_data_path) and os.path.exists(control_data_path):
                results_df_real = pd.read_csv(real_data_path)
                results_df_control = pd.read_csv(control_data_path)
                
                # Create minimal adapter classes with the necessary properties
                class RealAnalyzerAdapter:
                    def __init__(self, basedir):
                        self.basedir = basedir
                        self.donut_dir = os.path.join(basedir, 'Results', 'donut_analysis')
                
                class ControlAnalyzerAdapter:
                    def __init__(self, basedir):
                        self.basedir = basedir
                        self.donut_dir = os.path.join(basedir, 'Results', 'control_donut_analysis')
                
                real_adapter = RealAnalyzerAdapter(basedir)
                control_adapter = ControlAnalyzerAdapter(basedir)
                
                grouped_analyzer = GroupedDonutControlAnalyzer(real_adapter, control_adapter, 
                                                             results_df_real, results_df_control)
            else:
                raise FileNotFoundError(f"Could not find necessary data files: {real_data_path} and {control_data_path}")
        
        # Run the grouped analysis
        grouped_analyzer.run_grouped_analysis()
        
        return grouped_analyzer
    
    except Exception as e:
        print(f"\nError during grouped analysis: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Example usage showing different ways to run the code:
if __name__ == "__main__":
    # Example 1: Run directly after running both analyzers
    basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
    # basedir = r'/Users/MAC/Offline/Jackie/WholeCell2 Final'

    # Define days to process
    days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    # First run the original analyses to get the data
    real_analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
    real_results_df, _ = real_analyzer.run_analysis(days)
    
    control_analyzer = ControlEnhancedDonutAnalyzer(basedir, df_averaged_control)
    control_results_df, _ = control_analyzer.run_analysis(days)
    
    # Now run the grouped analysis with the analyzers and results
    grouped_analyzer = run_grouped_donut_analysis(
        real_analyzer=real_analyzer, 
        control_analyzer=control_analyzer,
        results_df_real=real_results_df,
        results_df_control=control_results_df
    )
    
    # Example 2: If running separately with previously saved data
    # basedir = r'/Users/MAC/Offline/Jackie/WholeCell2'
    # grouped_analyzer = run_grouped_donut_analysis(basedir=basedir)

# Comparison graph and stats (using the subtraction of the rtrial -rITI

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle
import torch
import roifile
from scipy import stats
import seaborn as sns
from matplotlib.patches import Circle
from tqdm import tqdm
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

class GroupedDonutControlAnalyzer:
    """
    Analyzer that computes Real - Control differences and applies
    reference code statistical methodology to grouped day comparisons.
    Works with existing EnhancedDonutAnalyzer workflow.
    """
    def __init__(self, real_analyzer, control_analyzer, results_df_real=None, results_df_control=None):
        """
        Initialize with existing EnhancedDonutAnalyzer and ControlEnhancedDonutAnalyzer instances 
        and optional results.
        
        Args:
            real_analyzer: An instance of EnhancedDonutAnalyzer with pre-calculated data
            control_analyzer: An instance of ControlEnhancedDonutAnalyzer with pre-calculated data
            results_df_real: Optional DataFrame with results from the real data analysis
            results_df_control: Optional DataFrame with results from the control data analysis
        """
        self.real_analyzer = real_analyzer
        self.control_analyzer = control_analyzer
        self.basedir = real_analyzer.basedir
        
        # Set up main directory for grouped analysis
        self.grouped_dir = os.path.join(self.basedir, 'Results', 'grouped_difference_donut_analysis')
        os.makedirs(self.grouped_dir, exist_ok=True)
        
        # Define day groupings
        self.day_groups = [
            {'name': 'Acquisition', 'days': ['0_Early', '1_Late']},
            {'name': 'Omission', 'days': ['2_SucOmit', '3_CueOmit']},
            {'name': 'Reversal', 'days': ['4_RevEarly', '5_RevLate']}
        ]
        
        # Handle results data for real dataset
        if results_df_real is not None:
            self.results_df_real = results_df_real
        else:
            data_path = os.path.join(real_analyzer.donut_dir, 'combined_data.csv')
            if os.path.exists(data_path):
                self.results_df_real = pd.read_csv(data_path)
                print(f"Loaded real data from {data_path}")
            else:
                print("No pre-existing real results found. Regenerating analysis data...")
                all_results = []
                days = real_analyzer.df_averaged['day'].unique()
                for day in days:
                    print(f"Processing real data for {day}...")
                    day_results = real_analyzer.analyze_pickle_data(day)
                    if day_results is not None:
                        all_results.append(day_results)
                
                if all_results:
                    self.results_df_real = pd.concat(all_results, ignore_index=True)
                else:
                    raise ValueError("Failed to generate real analysis data.")
        
        # Handle results data for control dataset
        if results_df_control is not None:
            self.results_df_control = results_df_control
        else:
            data_path = os.path.join(control_analyzer.donut_dir, 'control_combined_data.csv')
            if os.path.exists(data_path):
                self.results_df_control = pd.read_csv(data_path)
                print(f"Loaded control data from {data_path}")
            else:
                print("No pre-existing control results found. Regenerating analysis data...")
                all_results = []
                days = control_analyzer.df_averaged_control['day'].unique()
                for day in days:
                    print(f"Processing control data for {day}...")
                    day_results = control_analyzer.analyze_pickle_data(day)
                    if day_results is not None:
                        all_results.append(day_results)
                
                if all_results:
                    self.results_df_control = pd.concat(all_results, ignore_index=True)
                else:
                    raise ValueError("Failed to generate control analysis data.")
        
        # Add data_type column if not present
        if 'data_type' not in self.results_df_real.columns:
            self.results_df_real = self.results_df_real.copy()
            self.results_df_real['data_type'] = 'real'
            
        if 'data_type' not in self.results_df_control.columns:
            self.results_df_control = self.results_df_control.copy()
            self.results_df_control['data_type'] = 'control'
        
        print(f"GroupedDonutControlAnalyzer initialized with:")
        print(f"- {len(self.results_df_real)} real data points")
        print(f"- {len(self.results_df_control)} control data points")
        print(f"Results will be saved to: {self.grouped_dir}")
    
    def compute_fov_level_differences(self, days):
        """
        Compute Real - Control differences at the FOV level for proper statistics.
        
        Args:
            days: List of day identifiers for this group
        
        Returns:
            Tuple of (summary_df, individual_df) where:
            - summary_df: DataFrame with summary statistics (mean, sem) across FOVs
            - individual_df: DataFrame with individual FOV-level differences for statistics
        """
        all_summary_data = []
        all_individual_data = []
        
        # Filter data for these days
        real_group = self.results_df_real[self.results_df_real['day'].isin(days)].copy()
        control_group = self.results_df_control[self.results_df_control['day'].isin(days)].copy()
        
        # Process each day
        for day in days:
            print(f"  Processing differences for {day}...")
            
            # Get data for this day
            real_day = real_group[real_group['day'] == day]
            control_day = control_group[control_group['day'] == day]
            
            # Process each CS type
            for cs_type in ['CS+', 'CS-']:
                # Process each zone radius
                for zone_radius in sorted(real_day['zone_radius'].unique()):
                    # Get all FOVs for this combination
                    real_mask = (real_day['cs_type'] == cs_type) & (real_day['zone_radius'] == zone_radius)
                    control_mask = (control_day['cs_type'] == cs_type) & (control_day['zone_radius'] == zone_radius)
                    
                    real_subset = real_day[real_mask]
                    control_subset = control_day[control_mask]
                    
                    # Match FOVs between real and control
                    fov_differences = []
                    
                    for _, real_row in real_subset.iterrows():
                        animal = real_row['animal']
                        fov = real_row['fov']
                        
                        # Find matching control data
                        control_match = control_subset[
                            (control_subset['animal'] == animal) & 
                            (control_subset['fov'] == fov)
                        ]
                        
                        if len(control_match) > 0:
                            # Calculate difference
                            diff = real_row['correlation'] - control_match.iloc[0]['correlation']
                            fov_differences.append(diff)
                            
                            # Store individual FOV data for statistics
                            all_individual_data.append({
                                'day': day,
                                'zone_radius': zone_radius,
                                'cs_type': cs_type,
                                'mean_correlation': diff,
                                'animal': animal,
                                'fov': fov,
                                'group_id': f'{animal}_{fov}'
                            })
                    
                    # Calculate summary statistics across FOVs
                    if len(fov_differences) > 0:
                        fov_diffs_array = np.array(fov_differences)
                        all_summary_data.append({
                            'day': day,
                            'zone_radius': zone_radius,
                            'cs_type': cs_type,
                            'mean_correlation': np.mean(fov_diffs_array),
                            'std_correlation': np.std(fov_diffs_array, ddof=1) if len(fov_diffs_array) > 1 else 0,
                            'sem': np.std(fov_diffs_array, ddof=1) / np.sqrt(len(fov_diffs_array)) if len(fov_diffs_array) > 1 else 0,
                            'n_fovs': len(fov_diffs_array)
                        })
        
        # Convert to DataFrames
        summary_df = pd.DataFrame(all_summary_data)
        individual_df = pd.DataFrame(all_individual_data)
        
        return summary_df, individual_df
    
    def create_zone_radius_interaction_plot(self, summary_df, group_name):
        """
        Create Zone Radius Interaction plot with SEM bands for difference data.
        
        Args:
            summary_df: DataFrame containing summary data for this group
            group_name: Name of this group (Acquisition, Extinction, or Reversal)
        """
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
        
        # Get unique values
        days = sorted(summary_df['day'].unique())
        zone_radii = sorted(summary_df['zone_radius'].unique())
        cs_types = ['CS+', 'CS-']
        
        # Create color maps
        day_indices = {day: i for i, day in enumerate(days)}
        
        def get_color(day, cs_type):
            """Generate color based on day index and CS type."""
            day_idx = day_indices[day]
            day_factor = day_idx / max(1, len(days) - 1) if len(days) > 1 else 0
            
            if cs_type == 'CS+':
                # Purple gradient
                r = int(216 - 149 * day_factor)
                g = int(191 - 127 * day_factor)
                b = int(216 - 130 * day_factor)
                return f'#{r:02x}{g:02x}{b:02x}'
            else:  # CS-
                # Grey gradient
                val = int(211 - 106 * day_factor)
                return f'#{val:02x}{val:02x}{val:02x}'
        
        # Plot 1: Zone Radius Interaction
        ax = ax1
        line_styles = ['-', '--']  # Solid for CS+, dashed for CS-
        markers = ['o', 's']
        
        for i, day in enumerate(days):
            for j, cs_type in enumerate(cs_types):
                # Filter data
                mask = (summary_df['day'] == day) & (summary_df['cs_type'] == cs_type)
                day_cs_data = summary_df[mask]
                
                if len(day_cs_data) > 0:
                    # Sort by zone radius for proper line plotting
                    day_cs_data = day_cs_data.sort_values('zone_radius')
                    
                    color = get_color(day, cs_type)
                    
                    # Extract values
                    zone_radii_vals = day_cs_data['zone_radius'].values
                    mean_vals = day_cs_data['mean_correlation'].values
                    sem_vals = day_cs_data['sem'].values
                    
                    # Plot mean line
                    ax.plot(zone_radii_vals, mean_vals,
                           color=color,
                           linestyle=line_styles[j],
                           marker=markers[j],
                           linewidth=2.5,
                           markersize=8,
                           label=f"{day}, {cs_type}")
                    
                    # Add SEM bands
                    ax.fill_between(zone_radii_vals,
                                   mean_vals - sem_vals,
                                   mean_vals + sem_vals,
                                   color=color,
                                   alpha=0.25,
                                   edgecolor='none')
        
        ax.set_xlabel('Zone Radius (× cell diameter)', fontsize=12)
        ax.set_ylabel('Correlation Difference (Real - Control)', fontsize=12)
        ax.set_title(f'{group_name}: Zone Radius × Day × CS Type Interaction\n(Difference Data)', fontsize=14)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='red', linestyle=':', alpha=0.7, linewidth=1.5)
        ax.set_xticks(zone_radii)
        
        # Plot 2: Heatmap
        ax = ax2
        
        # Create pivot table
        pivot_table = summary_df.pivot_table(
            values='mean_correlation',
            index='zone_radius',
            columns=['day', 'cs_type'],
            aggfunc='mean'
        )
        
        # Create heatmap
        im = ax.imshow(pivot_table.values, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
        
        # Set ticks
        ax.set_xticks(range(len(pivot_table.columns)))
        ax.set_xticklabels([f"{col[0]}\n{col[1]}" for col in pivot_table.columns], 
                          rotation=45, ha='right')
        ax.set_yticks(range(len(pivot_table.index)))
        ax.set_yticklabels(pivot_table.index)
        
        ax.set_xlabel('Day × CS Type', fontsize=12)
        ax.set_ylabel('Zone Radius', fontsize=12)
        ax.set_title(f'{group_name}: Mean Difference Heatmap', fontsize=14)
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Mean Correlation Difference', rotation=270, labelpad=20)
        
        # Add text annotations
        for i in range(len(pivot_table.index)):
            for j in range(len(pivot_table.columns)):
                value = pivot_table.values[i, j]
                if not np.isnan(value):
                    text_color = 'white' if abs(value) > 0.15 else 'black'
                    ax.text(j, i, f'{value:.3f}', ha='center', va='center', 
                           color=text_color, fontsize=9)
        
        plt.tight_layout()
        
        # Save figure
        save_path = os.path.join(self.grouped_dir, f'{group_name}_zone_radius_interaction.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"  Zone Radius Interaction plot saved to: {save_path}")
    
    def perform_statistical_analysis(self, summary_df, individual_df, group_name):
        """
        Perform comprehensive statistical analysis on grouped difference data.
        Following the reference code methodology.
        
        Args:
            summary_df: DataFrame containing summary data for this group
            individual_df: DataFrame containing individual FOV-level data
            group_name: Name of this group
            
        Returns:
            Dictionary of statistical results
        """
        print(f"\n  Statistical Analysis for {group_name} (Real - Control Difference Data)")
        print("  " + "=" * 50)
        
        stats_results = {}
        
        # Check how many days we have
        unique_days = individual_df['day'].unique()
        
        # 1. Three-way or Two-way ANOVA depending on number of days
        if len(unique_days) > 1:
            print("\n  Three-way ANOVA (Zone × Day × CS Type):")
            try:
                df_anova = individual_df.copy()
                df_anova['zone_radius'] = pd.Categorical(df_anova['zone_radius'])
                df_anova['day'] = pd.Categorical(df_anova['day'])
                df_anova['cs_type'] = pd.Categorical(df_anova['cs_type'])
                
                model = ols('mean_correlation ~ C(zone_radius) * C(day) * C(cs_type)', data=df_anova).fit()
                anova_table = sm.stats.anova_lm(model, typ=2)
                print(anova_table)
                stats_results['three_way_anova'] = anova_table
                
                # Highlight significant effects
                print("\n  Significant Effects (p < 0.05):")
                sig_effects = anova_table[anova_table['PR(>F)'] < 0.05]
                if len(sig_effects) > 0:
                    for idx in sig_effects.index:
                        if idx != 'Residual':
                            print(f"    {idx}: F={sig_effects.loc[idx, 'F']:.3f}, p={sig_effects.loc[idx, 'PR(>F)']:.4f}")
            except Exception as e:
                print(f"  Error in Three-way ANOVA: {str(e)}")
        else:
            print("\n  Two-way ANOVA (Zone × CS Type) - Single day analysis:")
            try:
                df_anova = individual_df.copy()
                df_anova['zone_radius'] = pd.Categorical(df_anova['zone_radius'])
                df_anova['cs_type'] = pd.Categorical(df_anova['cs_type'])
                
                model = ols('mean_correlation ~ C(zone_radius) * C(cs_type)', data=df_anova).fit()
                anova_table = sm.stats.anova_lm(model, typ=2)
                print(anova_table)
                stats_results['two_way_anova'] = anova_table
                
                # Highlight significant effects
                print("\n  Significant Effects (p < 0.05):")
                sig_effects = anova_table[anova_table['PR(>F)'] < 0.05]
                if len(sig_effects) > 0:
                    for idx in sig_effects.index:
                        if idx != 'Residual':
                            print(f"    {idx}: F={sig_effects.loc[idx, 'F']:.3f}, p={sig_effects.loc[idx, 'PR(>F)']:.4f}")
            except Exception as e:
                print(f"  Error in Two-way ANOVA: {str(e)}")
        
        # 2. Test if differences are significantly different from zero
        print("\n  One-sample t-tests (Difference from Zero):")
        
        zero_tests = []
        for day in summary_df['day'].unique():
            for cs_type in summary_df['cs_type'].unique():
                for radius in summary_df['zone_radius'].unique():
                    mask = (summary_df['day'] == day) & (summary_df['cs_type'] == cs_type) & (summary_df['zone_radius'] == radius)
                    subset = summary_df[mask]
                    
                    if len(subset) > 0:
                        mean_val = subset['mean_correlation'].values[0]
                        sem_val = subset['sem'].values[0] if 'sem' in subset.columns else 0
                        n_samples = subset['n_fovs'].values[0] if 'n_fovs' in subset.columns else 1
                        
                        # Calculate t-statistic for difference from zero
                        if sem_val > 0 and n_samples > 1:
                            t_stat = mean_val / sem_val
                            p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_samples-1))
                        else:
                            t_stat = np.inf if mean_val != 0 else 0
                            p_val = 0 if mean_val != 0 else 1
                        
                        zero_tests.append({
                            'day': day,
                            'cs_type': cs_type,
                            'zone_radius': radius,
                            'mean_difference': mean_val,
                            't_statistic': t_stat,
                            'p_value': p_val,
                            'significant': p_val < 0.05,
                            'n_fovs': n_samples
                        })
        
        zero_tests_df = pd.DataFrame(zero_tests)
        if len(zero_tests_df) > 0:
            sig_tests = zero_tests_df[zero_tests_df['significant']]
            print(f"  Significant differences from zero: {len(sig_tests)}/{len(zero_tests_df)}")
            if len(sig_tests) > 0:
                pos_effects = sig_tests[sig_tests['mean_difference'] > 0]
                if len(pos_effects) > 0:
                    print("\n  Strongest positive effects:")
                    top_pos = pos_effects.nlargest(min(3, len(pos_effects)), 'mean_difference')
                    for _, row in top_pos.iterrows():
                        print(f"    {row['day']}, {row['cs_type']}, Zone {row['zone_radius']}: {row['mean_difference']:.4f} (p={row['p_value']:.4f})")
                
                neg_effects = sig_tests[sig_tests['mean_difference'] < 0]
                if len(neg_effects) > 0:
                    print("\n  Strongest negative effects:")
                    top_neg = neg_effects.nsmallest(min(3, len(neg_effects)), 'mean_difference')
                    for _, row in top_neg.iterrows():
                        print(f"    {row['day']}, {row['cs_type']}, Zone {row['zone_radius']}: {row['mean_difference']:.4f} (p={row['p_value']:.4f})")
        
        stats_results['zero_tests'] = zero_tests_df
        
        # 3. CS+ vs CS- comparison
        cs_plus_means = individual_df[individual_df['cs_type'] == 'CS+']['mean_correlation'].values
        cs_minus_means = individual_df[individual_df['cs_type'] == 'CS-']['mean_correlation'].values
        
        if len(cs_plus_means) > 0 and len(cs_minus_means) > 0:
            t_stat, p_val = stats.ttest_ind(cs_plus_means, cs_minus_means)
            print(f"\n  CS+ vs CS- comparison:")
            print(f"    CS+ mean: {np.mean(cs_plus_means):.4f}")
            print(f"    CS- mean: {np.mean(cs_minus_means):.4f}")
            print(f"    t-statistic: {t_stat:.3f}, p-value: {p_val:.4f}")
            
            # Cohen's d
            pooled_std = np.sqrt((np.var(cs_plus_means, ddof=1) + np.var(cs_minus_means, ddof=1)) / 2)
            if pooled_std > 0:
                cohen_d = (np.mean(cs_plus_means) - np.mean(cs_minus_means)) / pooled_std
                print(f"    Cohen's d: {cohen_d:.3f}")
                stats_results['cs_cohen_d'] = cohen_d
            
            stats_results['cs_comparison'] = {
                't_statistic': t_stat,
                'p_value': p_val,
                'cs_plus_mean': np.mean(cs_plus_means),
                'cs_minus_mean': np.mean(cs_minus_means)
            }
        
        # 4. Post-hoc tests if there are multiple days
        if len(unique_days) > 1:
            print("\n  Post-hoc Tukey HSD for Day comparisons:")
            try:
                tukey_results = pairwise_tukeyhsd(
                    individual_df['mean_correlation'].values,
                    individual_df['day'].values
                )
                print(tukey_results)
                stats_results['day_tukey'] = tukey_results
            except Exception as e:
                print(f"  Error in Tukey HSD: {str(e)}")
            
            # Additional post-hoc tests for Day × CS Type combinations
            print("\n  Post-hoc Tukey HSD for Day × CS Type combinations:")
            try:
                # Create a combined factor for day × cs_type
                individual_df_copy = individual_df.copy()
                individual_df_copy['day_cs'] = individual_df_copy['day'].astype(str) + '_' + individual_df_copy['cs_type'].astype(str)
                
                day_cs_tukey = pairwise_tukeyhsd(
                    individual_df_copy['mean_correlation'].values,
                    individual_df_copy['day_cs'].values
                )
                print(day_cs_tukey)
                stats_results['day_cs_tukey'] = day_cs_tukey
                
                # Extract and highlight key comparisons
                print("\n  Key within-CS-type comparisons across days:")
                # Parse the results to show specific comparisons
                results_summary = day_cs_tukey.summary()
                
            except Exception as e:
                print(f"  Error in Day × CS Type Tukey HSD: {str(e)}")
        
        # 5. Summary statistics
        summary_stats = summary_df.groupby(['day', 'zone_radius', 'cs_type'])[['mean_correlation', 'sem', 'n_fovs']].first()
        stats_results['summary_stats'] = summary_stats
        
        return stats_results
    
    def save_results(self, summary_df, individual_df, stats_results, group_name):
        """
        Save results to files.
        
        Args:
            summary_df: DataFrame with summary data
            individual_df: DataFrame with individual FOV data
            stats_results: Dictionary of statistical results
            group_name: Name of the group
        """
        # Save summary data as CSV
        summary_path = os.path.join(self.grouped_dir, f'{group_name}_summary_difference_data.csv')
        summary_df.to_csv(summary_path, index=False)
        
        # Save individual FOV data
        individual_path = os.path.join(self.grouped_dir, f'{group_name}_individual_fov_differences.csv')
        individual_df.to_csv(individual_path, index=False)
        
        # Save statistical results
        if 'zero_tests' in stats_results:
            stats_path = os.path.join(self.grouped_dir, f'{group_name}_zero_tests.csv')
            stats_results['zero_tests'].to_csv(stats_path, index=False)
        
        if 'summary_stats' in stats_results:
            summary_stats_path = os.path.join(self.grouped_dir, f'{group_name}_summary_stats.csv')
            stats_results['summary_stats'].to_csv(summary_stats_path)
        
        # Save ANOVA results if present
        if 'three_way_anova' in stats_results and stats_results['three_way_anova'] is not None:
            anova_path = os.path.join(self.grouped_dir, f'{group_name}_three_way_anova_results.csv')
            stats_results['three_way_anova'].to_csv(anova_path)
        
        if 'two_way_anova' in stats_results and stats_results['two_way_anova'] is not None:
            anova_path = os.path.join(self.grouped_dir, f'{group_name}_two_way_anova_results.csv')
            stats_results['two_way_anova'].to_csv(anova_path)
        
        print(f"  Results saved for {group_name}")
    
    def run_grouped_analysis(self):
        """
        Run complete analysis pipeline for all grouped days using difference-based approach.
        This is the main method called by existing code.
        """
        print("\n" + "="*60)
        print("GROUPED DIFFERENCE ANALYSIS (REAL - CONTROL)")
        print("Using FOV-level differences with reference code statistical approach")
        print("="*60)
        
        all_results = {}
        
        for group in self.day_groups:
            group_name = group['name']
            days = group['days']
            
            print(f"\n{'='*60}")
            print(f"Processing {group_name} Phase")
            print(f"Days: {', '.join(days)}")
            print(f"{'='*60}")
            
            # Compute FOV-level differences for this group
            summary_df, individual_df = self.compute_fov_level_differences(days)
            
            if len(summary_df) == 0:
                print(f"No data found for {group_name}")
                continue
            
            print(f"  Analyzing {len(summary_df)} zone measurements")
            print(f"  Using {len(individual_df)} individual FOV difference measurements for statistics")
            
            # Create Zone Radius Interaction plot
            self.create_zone_radius_interaction_plot(summary_df, group_name)
            
            # Perform statistical analysis
            stats_results = self.perform_statistical_analysis(summary_df, individual_df, group_name)
            
            # Save results
            self.save_results(summary_df, individual_df, stats_results, group_name)
            
            # Store in all_results
            all_results[group_name] = {
                'summary_data': summary_df,
                'individual_data': individual_df,
                'stats': stats_results
            }
            
            print(f"\n{group_name} analysis complete!")
        
        print("\n" + "="*60)
        print("ALL GROUPED DIFFERENCE ANALYSES COMPLETE!")
        print(f"Results saved to: {self.grouped_dir}")
        print("="*60)
        
        return all_results


def run_grouped_donut_analysis(basedir=None, real_analyzer=None, control_analyzer=None, 
                             results_df_real=None, results_df_control=None):
    """
    Main function to run the grouped analysis after the original analysis.
    This matches the original function signature.
    
    Args:
        basedir: Base directory path (required if analyzers not provided)
        real_analyzer: Instance of EnhancedDonutAnalyzer from original analysis
        control_analyzer: Instance of ControlEnhancedDonutAnalyzer from original analysis
        results_df_real: DataFrame with results from real data analysis
        results_df_control: DataFrame with results from control data analysis
        
    Returns:
        Instance of GroupedDonutControlAnalyzer or None if error occurs
    """
    try:
        if real_analyzer is not None and control_analyzer is not None:
            # Use the existing analyzers and optionally the results DataFrames
            grouped_analyzer = GroupedDonutControlAnalyzer(real_analyzer, control_analyzer, 
                                                         results_df_real, results_df_control)
        else:
            if basedir is None:
                raise ValueError("Either analyzers or basedir must be provided")
                
            # Try to load data from CSVs
            real_data_path = os.path.join(basedir, 'Results', 'donut_analysis', 'combined_data.csv')
            control_data_path = os.path.join(basedir, 'Results', 'control_donut_analysis', 'control_combined_data.csv')
            
            if os.path.exists(real_data_path) and os.path.exists(control_data_path):
                results_df_real = pd.read_csv(real_data_path)
                results_df_control = pd.read_csv(control_data_path)
                
                # Create minimal adapter classes with the necessary properties
                class RealAnalyzerAdapter:
                    def __init__(self, basedir):
                        self.basedir = basedir
                        self.donut_dir = os.path.join(basedir, 'Results', 'donut_analysis')
                
                class ControlAnalyzerAdapter:
                    def __init__(self, basedir):
                        self.basedir = basedir
                        self.donut_dir = os.path.join(basedir, 'Results', 'control_donut_analysis')
                
                real_adapter = RealAnalyzerAdapter(basedir)
                control_adapter = ControlAnalyzerAdapter(basedir)
                
                grouped_analyzer = GroupedDonutControlAnalyzer(real_adapter, control_adapter, 
                                                             results_df_real, results_df_control)
            else:
                raise FileNotFoundError(f"Could not find necessary data files: {real_data_path} and {control_data_path}")
        
        # Run the grouped analysis
        grouped_analyzer.run_grouped_analysis()
        
        return grouped_analyzer
    
    except Exception as e:
        print(f"\nError during grouped analysis: {str(e)}")
        import traceback
        traceback.print_exc()
        return None
        
if __name__ == "__main__":
    basedir = r'C:\Users\Jackie\Dropbox\2P Imaging Projects\Jackie\PFC_GFAP-GCaMP6f_Pavlovian\WholeCell'
    # basedir = r'/Users/MAC/Offline/Jackie/WholeCell2 Final'
    days = ['0_Early', '1_Late', '2_SucOmit', '3_CueOmit', '4_RevEarly', '5_RevLate']
    
    # Create analyzers with your existing dataframes
    real_analyzer = EnhancedDonutAnalyzer(basedir, df_averaged)
    real_results_df, _ = real_analyzer.run_analysis(days)
    
    control_analyzer = ControlEnhancedDonutAnalyzer(basedir, df_averaged_control)
    control_results_df, _ = control_analyzer.run_analysis(days)
    
    # Run the grouped analysis
    grouped_analyzer = run_grouped_donut_analysis(
        real_analyzer=real_analyzer, 
        control_analyzer=control_analyzer,
        results_df_real=real_results_df,
        results_df_control=control_results_df
    )

# Moving over meta data to df_averaged. 
## REMOVED CLUSTER ID HERE 

In [ ]:
### Moving over metadata to df_averaged

# This code adds only the missing information to df_averaged
def add_missing_columns_to_df_averaged(df_master, df_averaged):
    """
    Add only missing columns from df_master to df_averaged based on cell_id
    """
    # Identify columns that need to be added
    # Exclude cell_id, day, animal, fov, sex, auc_plus, auc_minus, cs_minus_correlation, cs_plus_correlation, cluster_id
    df_master_subset = df_master[['cell_id', 'cell_number', 'tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']].drop_duplicates()
    
    # Merge the selected columns from df_master into df_averaged
    df_averaged = pd.merge(df_averaged, df_master_subset, on='cell_id', how='left')
    
    return df_averaged

# Usage 
df_averaged = add_missing_columns_to_df_averaged(df_master, df_averaged)

# Check the result
print("Updated Averaged DataFrame Shape:", df_averaged.shape)
print("\nSample of updated df_averaged:")
display(df_averaged.head())

### Moving cross correlation data from df_averaged to df_master
# Since df_averaged already has the correlation data, we keep this part

import pandas as pd

def add_correlation_columns_conditionally(df_master, df_averaged):
    """
    Adds 'cs_plus_correlation' and 'cs_minus_correlation' columns to df_master by matching 'cell_id' with df_averaged.
    Only assigns 'cs_plus_correlation' for rows where 'cs_type' is 'CS+' and 'cs_minus_correlation' for rows where
    'cs_type' is 'CS-'.
    
    Parameters:
    - df_master (pd.DataFrame): The main DataFrame to which columns will be added.
    - df_averaged (pd.DataFrame): DataFrame containing 'cell_id', 'cs_plus_correlation', and 'cs_minus_correlation'.
    
    Returns:
    - pd.DataFrame: Updated df_master with conditionally added correlation columns.
    """
    # Merge the correlation columns from df_averaged into df_master based on 'cell_id'
    df_master = df_master.merge(
        df_averaged[['cell_id', 'cs_plus_correlation', 'cs_minus_correlation']], 
        on='cell_id', 
        how='left'
    )
    
    # Set cs_plus_correlation to NaN for rows where cs_type is not 'CS+'
    df_master.loc[df_master['cs_type'] != 'CS+', 'cs_plus_correlation'] = pd.NA
    
    # Set cs_minus_correlation to NaN for rows where cs_type is not 'CS-'
    df_master.loc[df_master['cs_type'] != 'CS-', 'cs_minus_correlation'] = pd.NA
    
    return df_master

# Example usage
df_master = add_correlation_columns_conditionally(df_master, df_averaged)

# Display the updated df_master DataFrame
print(df_master.head())

# Verify the data
print("\nVerification:")
print(f"Number of unique cells: {len(df_averaged)}")

# Sample of updated df_averaged with the newly added columns
print("\nSample of updated df_averaged with new columns:")
print(df_averaged[['cell_id', 'tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']].head())

# Additional verification
print("\nRandom sample verification:")
sample_cells = df_averaged['cell_id'].sample(n=3)
for cell in sample_cells:
    print(f"\nCell ID: {cell}")
    print("Master data:")
    print(df_master[df_master['cell_id'] == cell][['tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']].drop_duplicates())
    print("Averaged data:")
    # print(df_averaged[df_averaged['cell_id'] == cell][['tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']])
    print(df_averaged[['cell_id', 'tracked_roi_number', 'tracked_roi_counter']].head())

# # Add CS-specific cluster ID columns to df_averaged using the correct CS+/CS- values
# df_averaged['cluster_id_CS+'] = df_averaged['cell_id'].map(
#     df_master[df_master['cs_type'] == 'CS+'].groupby('cell_id')['cluster_id'].first()
# )

# df_averaged['cluster_id_CS-'] = df_averaged['cell_id'].map(
#     df_master[df_master['cs_type'] == 'CS-'].groupby('cell_id')['cluster_id'].first()
# )

# # Verify the additions
# print("\nVerification:")
# print(f"Number of unique cells: {len(df_averaged)}")
# print("\nCluster distribution CS+:")
# print(df_averaged['cluster_id_CS+'].value_counts().sort_index())
# print("\nCluster distribution CS-:")
# print(df_averaged['cluster_id_CS-'].value_counts().sort_index())

# # Sample of updated df_averaged
# print("\nSample of updated df_averaged:")
# print(df_averaged[['cell_id', 'cluster_id_CS+', 'cluster_id_CS-']].head())

# # Additional verification
# print("\nRandom sample verification:")
# sample_cells = df_averaged['cell_id'].sample(n=3)
# for cell in sample_cells:
#     print(f"\nCell ID: {cell}")
#     print("Master data:")
#     print(df_master[df_master['cell_id'] == cell][['cs_type', 'cluster_id']].drop_duplicates())
#     print("Averaged data:")
#     print(df_averaged[df_averaged['cell_id'] == cell][['cluster_id_CS+', 'cluster_id_CS-']])

# Moving meta data from df_master_control to df_averaged_control
## REMOVED CLUSTER ID HERE 

In [ ]:

# # Drop the problematic index levels before resetting
# df_master_control = df_master_control.reset_index(drop=True)

# # Drop duplicate columns if they exist
# df_master_control = df_master_control.loc[:, ~df_master_control.columns.duplicated()]

# # Verify the changes
# print("Final DataFrame Structure:")
# print(df_master_control.info())



### Moving over metadata to df_averaged_control

# This code adds only the missing information to df_averaged_control
def add_missing_columns_to_df_averaged_control(df_master_control, df_averaged_control):
    """
    Add only missing columns from df_master_control to df_averaged_control based on cell_id
    """
    # Identify columns that need to be added
    # Exclude cell_id, day, animal, fov, sex, auc_plus, auc_minus, correlation columns, cluster_id
    df_master_subset = df_master_control[['cell_id', 'cell_number', 'tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']].drop_duplicates()
    
    # Merge the selected columns from df_master_control into df_averaged_control
    df_averaged_control = pd.merge(df_averaged_control, df_master_subset, on='cell_id', how='left')
    
    return df_averaged_control

# Usage 
df_averaged_control = add_missing_columns_to_df_averaged_control(df_master_control, df_averaged_control)

# Check the result
print("Updated Averaged Control DataFrame Shape:", df_averaged_control.shape)
print("\nSample of updated df_averaged_control:")
display(df_averaged_control.head())

### Moving cross correlation data from df_averaged_control to df_master_control

import pandas as pd

def add_correlation_columns_to_master_control(df_master_control, df_averaged_control):
    """
    Adds correlation columns to df_master_control from df_averaged_control.
    Only assigns cs_plus_correlation_control for rows where cs_type is 'CS+' and 
    cs_minus_correlation_control for rows where cs_type is 'CS-'.
    
    Parameters:
    - df_master_control (pd.DataFrame): The main DataFrame to which columns will be added.
    - df_averaged_control (pd.DataFrame): DataFrame containing correlation columns with _control suffix.
    
    Returns:
    - pd.DataFrame: Updated df_master_control with conditionally added correlation columns.
    """
    # Check if the correlation columns exist
    if 'cs_plus_correlation_control' in df_averaged_control.columns and 'cs_minus_correlation_control' in df_averaged_control.columns:
        # Merge the correlation columns from df_averaged_control into df_master_control based on 'cell_id'
        df_master_control = df_master_control.merge(
            df_averaged_control[['cell_id', 'cs_plus_correlation_control', 'cs_minus_correlation_control']], 
            on='cell_id', 
            how='left'
        )
        
        # Set cs_plus_correlation_control to NaN for rows where cs_type is not 'CS+'
        df_master_control.loc[df_master_control['cs_type'] != 'CS+', 'cs_plus_correlation_control'] = pd.NA
        
        # Set cs_minus_correlation_control to NaN for rows where cs_type is not 'CS-'
        df_master_control.loc[df_master_control['cs_type'] != 'CS-', 'cs_minus_correlation_control'] = pd.NA
        
        print("Successfully transferred correlation data with _control suffix to df_master_control")
    else:
        print("Warning: Could not find 'cs_plus_correlation_control' or 'cs_minus_correlation_control' columns in df_averaged_control")
        print("Available columns:", df_averaged_control.columns.tolist())
    
    return df_master_control

# Example usage
df_master_control = add_correlation_columns_to_master_control(df_master_control, df_averaged_control)

# Display the updated df_master_control DataFrame
print("\nSample of updated df_master_control:")
print(df_master_control.head())

# Verify the data
print("\nVerification:")
print(f"Number of unique cells: {len(df_averaged_control)}")

# Sample of updated df_averaged_control with the newly added columns
print("\nSample of updated df_averaged_control with new columns:")
print(df_averaged_control[['cell_id', 'tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']].head())

# Check if correlation columns were added to df_master_control
if 'cs_plus_correlation_control' in df_master_control.columns:
    print("\nCorrelation columns in master control data:")
    cols_to_show = ['cell_id', 'cs_type', 'cs_plus_correlation_control', 'cs_minus_correlation_control']
    print(df_master_control[cols_to_show].head())

# Additional verification
print("\nRandom sample verification:")
sample_cells = df_averaged_control['cell_id'].sample(n=3)
for cell in sample_cells:
    print(f"\nCell ID: {cell}")
    print("Master control data:")
    print(df_master_control[df_master_control['cell_id'] == cell][['tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']].drop_duplicates())
    print("Averaged control data:")
    # print(df_averaged_control[df_averaged_control['cell_id'] == cell][['tracked_roi_number', 'tracked_roi_counter', 'pval_plus', 'pval_minus']])
    print(df_averaged_control[df_averaged_control['cell_id'] == cell][['tracked_roi_number', 'tracked_roi_counter']])



# # If cluster_id columns exist in df_master_control, add CS-specific cluster ID columns to df_averaged_control
# if 'cluster_id' in df_master_control.columns:
#     # Add CS-specific cluster ID columns to df_averaged_control using the correct CS+/CS- values
#     df_averaged_control['cluster_id_CS+'] = df_averaged_control['cell_id'].map(
#         df_master_control[df_master_control['cs_type'] == 'CS+'].groupby('cell_id')['cluster_id'].first()
#     )

#     df_averaged_control['cluster_id_CS-'] = df_averaged_control['cell_id'].map(
#         df_master_control[df_master_control['cs_type'] == 'CS-'].groupby('cell_id')['cluster_id'].first()
#     )

#     # Verify the additions
#     print("\nCluster ID Verification:")
#     print(f"Number of unique cells: {len(df_averaged_control)}")
#     print("\nCluster distribution CS+:")
#     print(df_averaged_control['cluster_id_CS+'].value_counts().sort_index())
#     print("\nCluster distribution CS-:")
#     print(df_averaged_control['cluster_id_CS-'].value_counts().sort_index())

#     # Sample of updated df_averaged_control
#     print("\nSample with cluster IDs:")
#     display(df_averaged_control[['cell_id', 'cluster_id_CS+', 'cluster_id_CS-']].head())

#     # Additional verification
#     print("\nRandom sample verification:")
#     sample_cells = df_averaged_control['cell_id'].sample(n=min(3, len(df_averaged_control)))
#     for cell in sample_cells:
#         print(f"\nCell ID: {cell}")
#         print("Master control data:")
#         display(df_master_control[df_master_control['cell_id'] == cell][['cs_type', 'cluster_id']].drop_duplicates())
#         print("Averaged control data:")
#         display(df_averaged_control[df_averaged_control['cell_id'] == cell][['cluster_id_CS+', 'cluster_id_CS-']])
# else:
#     print("\nNote: 'cluster_id' column not found in df_master_control. Skipping cluster ID mapping.")

# Tracked Cell Cross Correlations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import mixedlm

def print_statistical_results(plot_data, model_results, tukey_results):
    """Print statistical results in a clear format"""
    print("\nSTATISTICAL ANALYSES")
    print("=" * 50)
    
    # Main effects summary
    print("\n1. Mixed Model ANOVA Results:")
    print("-" * 30)
    
    fixed_effects = pd.DataFrame(model_results.fe_params)
    fixed_effects['P-Value'] = model_results.pvalues
    fixed_effects.columns = ['Estimate', 'P-Value']
    
    print("\nMain Effects:")
    print("-------------")
    print("\nEffect of Day:")
    print(fixed_effects.filter(like='day', axis=0))
    
    print("\nEffect of CS Type:")
    print(fixed_effects.filter(like='cs_type', axis=0))
    
    print("\nInteraction Effects (Day × CS Type):")
    print(fixed_effects.filter(like=':', axis=0))
    
    print("\n2. Post-hoc Comparisons (Tukey HSD):")
    print("-" * 30)
    print(tukey_results)
    
    print("\n3. Summary Statistics:")
    print("-" * 30)
    summary_stats = plot_data.groupby(['day', 'cs_type'])['correlation'].agg(['mean', 'std', 'count'])
    print(summary_stats)

def analyze_correlations_tracked(df_averaged, days_to_visualize, sort_day):
    """
    Correlation analysis function that maintains cell ordering based on tracked_roi_counter
    and allows for comparison across specific day combinations
    """
    print("Starting correlation analysis...")
    print(f"Processing days: {days_to_visualize}")
    print(f"Sorting based on: {sort_day}")
    
    # Filter for cells present in all selected days
    cell_counts = df_averaged[df_averaged['day'].isin(days_to_visualize)].groupby('tracked_roi_counter')['day'].nunique()
    valid_cells = cell_counts[cell_counts == len(days_to_visualize)].index
    df_filtered = df_averaged[df_averaged['tracked_roi_counter'].isin(valid_cells)]
    
    # Get cell ordering based on sort_day responses
    sort_day_data = df_filtered[df_filtered['day'] == sort_day]
    
    # Get average response to CS+ for sorting
    fluor_plus_cols = [col for col in sort_day_data.columns if col.startswith('fluor_plus_frame_')]
    mean_responses = sort_day_data.groupby('tracked_roi_counter')[fluor_plus_cols].mean()
    mean_response_vals = mean_responses.mean(axis=1)
    cell_order = mean_response_vals.sort_values(ascending=False).index
    
    all_results = []
    
    # Set up the figure for all days
    n_days = len(days_to_visualize)
    fig, axes = plt.subplots(n_days, 2, figsize=(20, 10*n_days))
    if n_days == 1:
        axes = axes.reshape(1, -1)
    
    for d, day in enumerate(days_to_visualize):
        print(f"\nProcessing day: {day}")
        day_data = df_filtered[df_filtered['day'] == day]
        
        # Ensure data is ordered by tracked_roi_counter
        day_data_sorted = pd.concat([
            day_data[day_data['tracked_roi_counter'] == roi]
            for roi in cell_order
        ])
        
        # Extract CS+ and CS- data
        cs_plus_cols = [col for col in day_data.columns if col.startswith('fluor_plus_frame_')]
        cs_minus_cols = [col for col in day_data.columns if col.startswith('fluor_minus_frame_')]
        
        cs_plus_data = day_data_sorted[cs_plus_cols].to_numpy(dtype=float)
        cs_minus_data = day_data_sorted[cs_minus_cols].to_numpy(dtype=float)
        
        n_cells = cs_plus_data.shape[0]
        cs_plus_corr = np.zeros((n_cells, n_cells))
        cs_minus_corr = np.zeros((n_cells, n_cells))
        
        # Calculate correlation matrices
        for i in range(n_cells):
            cs_plus_corr[i, i] = 1
            cs_minus_corr[i, i] = 1
            
            for j in range(i+1, n_cells):
                try:
                    cs_plus_corr[i, j], _ = pearsonr(cs_plus_data[i], cs_plus_data[j])
                    cs_minus_corr[i, j], _ = pearsonr(cs_minus_data[i], cs_minus_data[j])
                    cs_plus_corr[j, i] = cs_plus_corr[i, j]
                    cs_minus_corr[j, i] = cs_minus_corr[i, j]
                except ValueError as e:
                    print(f"Warning: Error calculating correlation for cells {i} and {j}: {str(e)}")
                    cs_plus_corr[i, j] = cs_plus_corr[j, i] = np.nan
                    cs_minus_corr[i, j] = cs_minus_corr[j, i] = np.nan
        
        # Calculate average correlations for each cell
        cs_plus_avg = np.nanmean(cs_plus_corr, axis=1)
        cs_minus_avg = np.nanmean(cs_minus_corr, axis=1)
        
        # Store results
        for idx, cell in enumerate(cell_order):
            all_results.append({
                'day': day,
                'tracked_roi_counter': cell,
                'cs_plus_correlation': cs_plus_avg[idx],
                'cs_minus_correlation': cs_minus_avg[idx]
            })
        
        # Plot heatmaps
        sns.heatmap(cs_plus_corr, ax=axes[d, 0], cmap='PiYG_r', vmin=-1, vmax=1, center=0)
        axes[d, 0].set_title(f'CS+ Cross-correlation Heatmap - Day {day}')
        axes[d, 0].set_xlabel('Cell Index (sorted by CS+ response)')
        axes[d, 0].set_ylabel('Cell Index (sorted by CS+ response)')
        
        sns.heatmap(cs_minus_corr, ax=axes[d, 1], cmap='PiYG_r', vmin=-1, vmax=1, center=0)
        axes[d, 1].set_title(f'CS- Cross-correlation Heatmap - Day {day}')
        axes[d, 1].set_xlabel('Cell Index (sorted by CS+ response)')
        axes[d, 1].set_ylabel('Cell Index (sorted by CS+ response)')
        
        # Add tracked ROI numbers to y-axis
        roi_labels = [f'{roi:.0f}' for roi in cell_order]
        axes[d, 0].set_yticks(np.arange(len(roi_labels)) + 0.5)
        axes[d, 0].set_yticklabels(roi_labels)
        axes[d, 1].set_yticks(np.arange(len(roi_labels)) + 0.5)
        axes[d, 1].set_yticklabels(roi_labels)

    plt.tight_layout()
    # Save the correlation heatmaps figure
    plt.savefig(f'correlation_heatmaps_days_{"_".join(map(str, days_to_visualize))}.pdf', 
                bbox_inches='tight', dpi=300)
    plt.show()
    # plt.tight_layout()
    # plt.show()
    
    # Create results dataframe
    results_df = pd.DataFrame(all_results)
    print(f"\nCreated results for {len(results_df)} cell-day combinations")
    
    # Merge results back into df_averaged
    df_averaged_updated = pd.merge(
        df_averaged,
        results_df[['day', 'tracked_roi_counter', 'cs_plus_correlation', 'cs_minus_correlation']],
        on=['day', 'tracked_roi_counter'],
        how='left'
    )
    
    # Prepare data for visualization and statistics
    plot_data = pd.melt(results_df,
                       id_vars=['day', 'tracked_roi_counter'],
                       value_vars=['cs_plus_correlation', 'cs_minus_correlation'],
                       var_name='cs_type',
                       value_name='correlation')
    plot_data['cs_type'] = plot_data['cs_type'].map({
        'cs_plus_correlation': 'CS+',
        'cs_minus_correlation': 'CS-'
    })
    
    # Create bar plot
    plt.figure(figsize=(10, 6))
    sns.barplot(x='day', y='correlation', hue='cs_type', data=plot_data,
                errorbar=('se', 1), capsize=0.1, palette='Set2')
    plt.title('Average Correlation by Day and CS Type')
    plt.xlabel('Day')
    plt.ylabel('Average Correlation')
    # Save the bar plot
    plt.savefig(f'correlation_barplot_days_{"_".join(map(str, days_to_visualize))}.pdf', 
                bbox_inches='tight', dpi=300)
    plt.show()
    
    # Statistical analysis
    model = mixedlm("correlation ~ C(day) + C(cs_type) + C(day):C(cs_type)",
                    data=plot_data, groups="tracked_roi_counter")
    model_results = model.fit()
    
    tukey_results = pairwise_tukeyhsd(plot_data['correlation'],
                                     plot_data['day'] + ' ' + plot_data['cs_type'])
    
    # Print statistical results
    print_statistical_results(plot_data, model_results, tukey_results)
    
    # Print verification
    print("\nVerification of tracked ROI ordering:")
    for day in days_to_visualize:
        print(f"\nFirst 5 tracked ROI counters for {day}:")
        day_data = results_df[results_df['day'] == day]['tracked_roi_counter'].head()
        print(day_data.values)
    
    return df_averaged_updated, results_df, model_results, tukey_results

# Usage example:
days_to_visualize = ['0_Early', '1_Late']
sort_day = '0_Early'
df_averaged_updated, results_df, anova_results, posthoc_results = analyze_correlations_tracked(
    df_averaged, days_to_visualize, sort_day)

days_to_visualize = ['1_Late', '2_SucOmit', '3_CueOmit']
sort_day = '1_Late'
df_averaged_updated, results_df, anova_results, posthoc_results = analyze_correlations_tracked(
    df_averaged, days_to_visualize, sort_day)

days_to_visualize = ['1_Late', '4_RevEarly', '5_RevLate']
sort_day = '1_Late'
df_averaged_updated, results_df, anova_results, posthoc_results = analyze_correlations_tracked(
    df_averaged, days_to_visualize, sort_day)


# This code is a multipurpose plotting and statistics cell for tracked data, which intentionally only plots and analyzes data only for significant cells. 
## Make sure to set metric_type
### To plot and analyze tracked average correlation coefficient metric_type=correlations
### To plot and analyze tracked auROC values metric_type=auc
### to plot and analyze auROC pvalues metric_type=pval

In [ ]:
## filter df_averaged to find successful tracking events 
##  this establishes a logic to select data where tracked_roi_counter with the same value across more than one day
## This means all data tracked across at least two days is pulled in.

import pandas as pd
import os

def filter_for_correlation_tracking(df_averaged, basedir):
    """
    Filter df_averaged to include only rows with the same tracked_roi_counter across multiple days.
    Save the resulting DataFrame to CSV in the specified Results folder.
    """
    # Group by 'tracked_roi_counter' and filter groups with more than one unique 'day'
    grouped = df_averaged.groupby('tracked_roi_counter')
    df_filtered = grouped.filter(lambda x: x['day'].nunique() > 1).copy()

    # Check the filtered result
    print("Filtered DataFrame Shape:", df_filtered.shape)
    print("\nSample of Filtered DataFrame:")
    display(df_filtered.head())

    # Define the save path
    save_path = os.path.join(basedir, 'Results', 'df_averaged_correlationtrack.csv')
    
    # Save the filtered DataFrame as CSV
    df_filtered.to_csv(save_path, index=False)
    print(f"Filtered DataFrame saved to {save_path}")
    
    return df_filtered

# Usage
df_averaged_correlationtrack = filter_for_correlation_tracking(df_averaged, basedir)

### Plotting and statistics code 


import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def print_statistical_results(plot_data, model_results, tukey_results, metric_type):
    """Print statistical results in a clear format"""
    print(f"\nSTATISTICAL ANALYSES FOR {metric_type.upper()}")
    print("=" * 50)
    
    # Main effects summary
    print("\n1. Mixed Model ANOVA Results:")
    print("-" * 30)
    
    fixed_effects = pd.DataFrame(model_results.fe_params)
    fixed_effects['P-Value'] = model_results.pvalues
    fixed_effects.columns = ['Estimate', 'P-Value']
    
    print("\nMain Effects:")
    print("-------------")
    print("\nEffect of Day:")
    print(fixed_effects.filter(like='day', axis=0))
    
    print("\nEffect of CS Type:")
    print(fixed_effects.filter(like='cs_type', axis=0))
    
    print("\nInteraction Effects (Day × CS Type):")
    print(fixed_effects.filter(like=':', axis=0))
    
    print("\n2. Post-hoc Comparisons (Tukey HSD):")
    print("-" * 30)
    print(tukey_results)
    
    print("\n3. Summary Statistics:")
    print("-" * 30)
    summary_stats = plot_data.groupby(['day', 'cs_type'])['value'].agg(['mean', 'std', 'count'])
    print(summary_stats)

##### HERE IS WHERE YOU SET WHAT IS PLOTTED IN THIS CELL 


def plot_metric_lineplot(df_averaged_correlationtrack, metric_type='auc'): #### Options include 'correlations' 'auc' or 'pval'

    # Define metric mappings
    metric_columns = {
        'correlations': {
            'plus': 'cs_plus_correlation',
            'minus': 'cs_minus_correlation',
            'title': 'Tracked Average Correlation Coefficient Across Days',
            'ylabel': 'Average Correlation Coefficient (Mean ± SEM)'
        },
        'auc': {
            'plus': 'auc_plus',
            'minus': 'auc_minus',
            'title': 'auROC Across Days',
            'ylabel': 'Area Under Curve (Mean ± SEM)'
        },
        'pval': {
            'plus': 'pval_plus',
            'minus': 'pval_minus',
            'title': 'P-Values Across Days',
            'ylabel': 'P-Value (Mean ± SEM)'
        }
    }
    
    # Verify metric type is valid
    if metric_type not in metric_columns:
        raise ValueError(f"metric_type must be one of {list(metric_columns.keys())}")
    
    # Get the appropriate columns for the selected metric
    selected_metrics = metric_columns[metric_type]
    
    # Prepare data for plotting and statistics
    plot_data = df_averaged_correlationtrack.melt(
        id_vars=['day', 'tracked_roi_counter'], 
        value_vars=[selected_metrics['plus'], selected_metrics['minus']], 
        var_name='cs_type', 
        value_name='value'
    )
    
    # Map the column names to CS+ and CS-
    plot_data['cs_type'] = plot_data['cs_type'].map({
        selected_metrics['plus']: 'CS+', 
        selected_metrics['minus']: 'CS-'
    })
    
    # Calculate summary statistics
    summary_data = plot_data.groupby(['day', 'cs_type']).agg(
        mean_val=('value', 'mean'),
        sem_val=('value', 'sem')
    ).reset_index()

    # # Create the plot
    # plt.figure(figsize=(10, 6))
    # sns.lineplot(
    #     data=summary_data, 
    #     x='day', y='mean_val', 
    #     hue='cs_type', 
    #     marker='o',
    #     err_style="bars",
    #     ci=None,
    #     palette='Set2'
    # )

    # to avoid sns lineplot issues use:

    # Get unique values
    days = sorted(summary_data['day'].unique())
    cs_types = summary_data['cs_type'].unique()
    
    # Define colors to match Set2 palette
    colors = {'CS+': sns.color_palette('Set2')[0], 'CS-': sns.color_palette('Set2')[1]}
    
    # Plot each CS type manually
    for cs_type in cs_types:
        type_data = summary_data[summary_data['cs_type'] == cs_type]
        
        # Convert to numpy arrays
        x_vals = np.array([days.index(day) for day in type_data['day']])
        y_vals = type_data['mean_val'].values
        
        # Plot the line
        plt.plot(x_vals, y_vals, 
                 color=colors[cs_type], 
                 marker='o', 
                 label=cs_type)
    
    # Set x-tick labels to match original days
    plt.xticks(range(len(days)), days)
    
    # # Add error bars
    # for cs_type in summary_data['cs_type'].unique():
    #     type_data = summary_data[summary_data['cs_type'] == cs_type]
    #     plt.errorbar(
    #         x=type_data['day'], 
    #         y=type_data['mean_val'], 
    #         yerr=type_data['sem_val'], 
    #         fmt='none', 
    #         capsize=5,
    #         color=sns.color_palette('Set2')[0] if cs_type == 'CS+' else sns.color_palette('Set2')[1]
    #     )

    # Add error bars using nmpy arrays instead of pandas indexing
    for cs_type in summary_data['cs_type'].unique():
        type_data = summary_data[summary_data['cs_type'] == cs_type]
        
        # Convert to numpy arrays for errorbar
        x_vals = np.array([days.index(day) for day in type_data['day']])
        y_vals = type_data['mean_val'].values
        yerr_vals = type_data['sem_val'].values
        
        plt.errorbar(
            x=x_vals, 
            y=y_vals, 
            yerr=yerr_vals, 
            fmt='none', 
            capsize=5,
            color=sns.color_palette('Set2')[0] if cs_type == 'CS+' else sns.color_palette('Set2')[1]
        )
    
    plt.title(selected_metrics['title'])
    plt.xlabel('Day')
    plt.ylabel(selected_metrics['ylabel'])
    plt.legend(title='CS Type')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    # Save the plot
    # plot_filename = f"{metric_type.capitalize()}_LinePlot.pdf"
    # plot_path = os.path.join(basedir, 'Results', plot_filename)
    # plt.savefig(plot_path, format='pdf')
    plt.show()
    
    # Statistical Analysis
    print(f"\nPerforming statistical analysis for {metric_type}...")
    
    # Mixed Model ANOVA
    model = mixedlm("value ~ C(day) + C(cs_type) + C(day):C(cs_type)",
                    data=plot_data, groups="tracked_roi_counter")
    model_results = model.fit()
    
    # Post-hoc Tukey tests for all combinations
    tukey_results = pairwise_tukeyhsd(plot_data['value'],
                                     plot_data['day'] + ' ' + plot_data['cs_type'])
    
    # Print statistical results
    print_statistical_results(plot_data, model_results, tukey_results, metric_type)
    
    return plot_data, model_results, tukey_results

# Usage examples:
# For correlations:
plot_data, model_results, tukey_results = plot_metric_lineplot(df_averaged_correlationtrack)



# having it print the DFs in the ANOVA table

In [ ]:
##to split the data up into different groupings for the line plot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from statsmodels.formula.api import mixedlm, ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import anova_lm

def print_statistical_results(plot_data, model_results, tukey_results, metric_type, days_subset):
    """Print statistical results in a clear format with ANOVA table and F-statistics"""
    print(f"\nSTATISTICAL ANALYSES FOR {metric_type.upper()} - DAYS: {', '.join(days_subset)}")
    print("=" * 70)
    
    # 1. Mixed Model Results
    print("\n1. Mixed Model Results:")
    print("-" * 30)
    print(model_results.summary())
    
    # 2. ANOVA Table using OLS for F-statistics and degrees of freedom
    print("\n2. ANOVA Table with F-statistics and Degrees of Freedom:")
    print("-" * 55)
    
    try:
        # Fit OLS model for ANOVA table
        ols_model = ols("value ~ C(day) * C(cs_type)", data=plot_data).fit()
        anova_table = anova_lm(ols_model, typ=2)
        print("\nANOVA Table (Type II):")
        print(anova_table)
        
        # Extract F-statistics and degrees of freedom for manuscript reporting
        print("\nManuscript Reporting Format:")
        print("-" * 30)
        
        for effect in anova_table.index:
            if effect != 'Residual':
                f_val = anova_table.loc[effect, 'F']
                p_val = anova_table.loc[effect, 'PR(>F)']
                df_num = anova_table.loc[effect, 'df']
                df_den = anova_table.loc['Residual', 'df']
                
                # Format p-value
                if p_val < 0.001:
                    p_str = "p < 0.001"
                else:
                    p_str = f"p = {p_val:.3f}"
                
                print(f"{effect}: F({df_num:.0f}, {df_den:.0f}) = {f_val:.3f}, {p_str}")
        
    except Exception as e:
        print(f"Error generating ANOVA table: {str(e)}")
        print("Showing mixed model coefficients instead:")
        
        # Fallback to mixed model coefficients
        fixed_effects = pd.DataFrame(model_results.fe_params)
        fixed_effects['P-Value'] = model_results.pvalues
        fixed_effects.columns = ['Estimate', 'P-Value']
        
        print("\nMixed Model Fixed Effects:")
        print("-------------------------")
        print("\nEffect of Day:")
        print(fixed_effects.filter(like='day', axis=0))
        
        print("\nEffect of CS Type:")
        print(fixed_effects.filter(like='cs_type', axis=0))
        
        print("\nInteraction Effects (Day × CS Type):")
        print(fixed_effects.filter(like=':', axis=0))
    
    # 3. Post-hoc Comparisons
    print("\n3. Post-hoc Comparisons (Tukey HSD):")
    print("-" * 35)
    print(tukey_results)
    
    # Extract significant comparisons for easy interpretation
    print("\nSignificant Post-hoc Comparisons (p < 0.05):")
    print("-" * 45)
    significant_found = False
    for i, row in enumerate(tukey_results._results_table.data[1:]):
        group1, group2, meandiff, p_adj, lower, upper, reject = row
        if reject:  # reject is True when p < alpha
            significant_found = True
            print(f"{group1} vs {group2}: Mean diff = {meandiff:.4f}, p = {p_adj:.4f}")
    
    if not significant_found:
        print("No significant pairwise differences found.")
    
    # 4. Summary Statistics
    print("\n4. Descriptive Statistics:")
    print("-" * 25)
    summary_stats = plot_data.groupby(['day', 'cs_type'])['value'].agg(['mean', 'std', 'count'])
    print(summary_stats)

def plot_metric_lineplot_subset(df_averaged_correlationtrack, days_to_visualize, metric_type='auc', basedir='.'):
    """
    Plot metrics for a specific subset of days and perform statistical analysis
    
    Parameters:
    -----------
    df_averaged_correlationtrack : DataFrame
        The filtered dataframe with correlation tracking data
    days_to_visualize : list
        List of days to include in the analysis (e.g., ['0_Early', '1_Late'])
    metric_type : str
        Type of metric to plot ('correlations', 'auc', or 'pval')
    basedir : str
        Base directory for saving results
    """
    
    # Define metric mappings
    metric_columns = {
        'correlations': {
            'plus': 'cs_plus_correlation',
            'minus': 'cs_minus_correlation',
            'title': f'Tracked Average Correlation Coefficient - Days: {", ".join(days_to_visualize)}',
            'ylabel': 'Average Correlation Coefficient (Mean ± SEM)'
        },
        'auc': {
            'plus': 'auc_plus',
            'minus': 'auc_minus',
            'title': f'auROC - Days: {", ".join(days_to_visualize)}',
            'ylabel': 'Area Under Curve (Mean ± SEM)'
        },
        'pval': {
            'plus': 'pval_plus',
            'minus': 'pval_minus',
            'title': f'P-Values - Days: {", ".join(days_to_visualize)}',
            'ylabel': 'P-Value (Mean ± SEM)'
        }
    }
    
    # Verify metric type is valid
    if metric_type not in metric_columns:
        raise ValueError(f"metric_type must be one of {list(metric_columns.keys())}")
    
    # Get the appropriate columns for the selected metric
    selected_metrics = metric_columns[metric_type]
    
    # Filter for cells present in all selected days (similar to reference code)
    cell_counts = df_averaged_correlationtrack[
        df_averaged_correlationtrack['day'].isin(days_to_visualize)
    ].groupby('tracked_roi_counter')['day'].nunique()
    
    valid_cells = cell_counts[cell_counts == len(days_to_visualize)].index
    df_filtered = df_averaged_correlationtrack[
        (df_averaged_correlationtrack['tracked_roi_counter'].isin(valid_cells)) &
        (df_averaged_correlationtrack['day'].isin(days_to_visualize))
    ]
    
    print(f"Processing days: {days_to_visualize}")
    print(f"Number of cells tracked across all days: {len(valid_cells)}")
    print(f"Filtered DataFrame Shape: {df_filtered.shape}")
    
    # Prepare data for plotting and statistics
    plot_data = df_filtered.melt(
        id_vars=['day', 'tracked_roi_counter'], 
        value_vars=[selected_metrics['plus'], selected_metrics['minus']], 
        var_name='cs_type', 
        value_name='value'
    )
    
    # Map the column names to CS+ and CS-
    plot_data['cs_type'] = plot_data['cs_type'].map({
        selected_metrics['plus']: 'CS+', 
        selected_metrics['minus']: 'CS-'
    })
    
    # Calculate summary statistics
    summary_data = plot_data.groupby(['day', 'cs_type']).agg(
        mean_val=('value', 'mean'),
        sem_val=('value', 'sem')
    ).reset_index()

    # Create the plot
    plt.figure(figsize=(10, 6))
    
    # Get unique values
    days = sorted(summary_data['day'].unique())
    cs_types = summary_data['cs_type'].unique()
    
    # Define colors to match Set2 palette
    colors = {'CS+': sns.color_palette('Set2')[0], 'CS-': sns.color_palette('Set2')[1]}
    
    # Plot each CS type manually
    for cs_type in cs_types:
        type_data = summary_data[summary_data['cs_type'] == cs_type]
        
        # Convert to numpy arrays
        x_vals = np.array([days.index(day) for day in type_data['day']])
        y_vals = type_data['mean_val'].values
        
        # Plot the line
        plt.plot(x_vals, y_vals, 
                 color=colors[cs_type], 
                 marker='o', 
                 label=cs_type)
    
    # Set x-tick labels to match original days
    plt.xticks(range(len(days)), days, rotation=45)
    
    # Add error bars using numpy arrays
    for cs_type in summary_data['cs_type'].unique():
        type_data = summary_data[summary_data['cs_type'] == cs_type]
        
        # Convert to numpy arrays for errorbar
        x_vals = np.array([days.index(day) for day in type_data['day']])
        y_vals = type_data['mean_val'].values
        yerr_vals = type_data['sem_val'].values
        
        plt.errorbar(
            x=x_vals, 
            y=y_vals, 
            yerr=yerr_vals, 
            fmt='none', 
            capsize=5,
            color=colors[cs_type]
        )
    
    plt.title(selected_metrics['title'])
    plt.xlabel('Day')
    plt.ylabel(selected_metrics['ylabel'])
    plt.legend(title='CS Type')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    # Save the plot with day subset in filename
    days_suffix = "_".join(days_to_visualize)
    plot_filename = f"{metric_type.capitalize()}_LinePlot_days_{days_suffix}.pdf"
    plot_path = os.path.join(basedir, 'Results', plot_filename)
    try:
        os.makedirs(os.path.join(basedir, 'Results'), exist_ok=True)
        plt.savefig(plot_path, format='pdf', bbox_inches='tight', dpi=300)
    except:
        pass  # Continue if save fails
    plt.show()
    
    # Statistical Analysis
    print(f"\nPerforming statistical analysis for {metric_type} on days: {days_to_visualize}...")
    
    # Mixed Model ANOVA
    model = mixedlm("value ~ C(day) + C(cs_type) + C(day):C(cs_type)",
                    data=plot_data, groups="tracked_roi_counter")
    model_results = model.fit()
    
    # Post-hoc Tukey tests for all combinations
    tukey_results = pairwise_tukeyhsd(plot_data['value'],
                                     plot_data['day'] + ' ' + plot_data['cs_type'])
    
    # Print statistical results with ANOVA table
    print_statistical_results(plot_data, model_results, tukey_results, metric_type, days_to_visualize)
    
    return plot_data, model_results, tukey_results

def run_all_subset_analyses(df_averaged_correlationtrack, metric_type='auc', basedir='.'):
    """
    Run analysis for all three day subsets as specified
    """
    day_subsets = [
        ['0_Early', '1_Late'],
        ['1_Late', '2_SucOmit', '3_CueOmit'],
        ['1_Late', '4_RevEarly', '5_RevLate']
    ]
    
    all_results = {}
    
    for i, days_subset in enumerate(day_subsets, 1):
        print(f"\n{'='*80}")
        print(f"ANALYSIS {i}: {' vs '.join(days_subset)}")
        print(f"{'='*80}")
        
        try:
            plot_data, model_results, tukey_results = plot_metric_lineplot_subset(
                df_averaged_correlationtrack, 
                days_subset, 
                metric_type, 
                basedir
            )
            
            all_results[f"subset_{i}"] = {
                'days': days_subset,
                'plot_data': plot_data,
                'model_results': model_results,
                'tukey_results': tukey_results
            }
            
        except Exception as e:
            print(f"Error processing subset {days_subset}: {str(e)}")
            continue
    
    return all_results

# Usage examples:

# For AUC analysis across all subsets:
all_auc_results = run_all_subset_analyses(df_averaged_correlationtrack, metric_type='auc', basedir=basedir)

# # For individual subset analysis:
# # Early vs Late training
# plot_data_1, model_1, tukey_1 = plot_metric_lineplot_subset(
#     df_averaged_correlationtrack, 
#     ['0_Early', '1_Late'], 
#     metric_type='auc', 
#     basedir=basedir
# )

# # Late training vs Omission phases
# plot_data_2, model_2, tukey_2 = plot_metric_lineplot_subset(
#     df_averaged_correlationtrack, 
#     ['1_Late', '2_SucOmit', '3_CueOmit'], 
#     metric_type='auc', 
#     basedir=basedir
# )

# # Late training vs Reversal phases
# plot_data_3, model_3, tukey_3 = plot_metric_lineplot_subset(
#     df_averaged_correlationtrack, 
#     ['1_Late', '4_RevEarly', '5_RevLate'], 
#     metric_type='auc', 
#     basedir=basedir
# )

# For correlations analysis:
all_corr_results = run_all_subset_analyses(df_averaged_correlationtrack, metric_type='correlations', basedir=basedir)

# Decoding

In [ ]:
# #### BLOCK 1 - TRIAL-LEVEL TEMPORAL DECODING (ALL DAYS) ###

# #### BINARY CLASSIFIER (PARALLELIZED WITH `n_jobs=-1`)
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, LeaveOneOut
import numpy as np


def binaryclassifier(y, X):
    """
    Binary classifier using SVM with RBF kernel and hyperparameter tuning, optimized for quiet operation.
    Uses Leave-One-Out Cross-Validation for consistent analysis across all neurons.
    """
    # Check minimum samples first
    if y.shape[0] < 4:  # Need at least 4 samples for meaningful LOOCV
        print("Insufficient data - Less Than 4 Total Samples!")
        return np.nan
        
    if np.any(np.isnan(X)):
        print("NaN detected in input data!")
        return np.nan
    
    # Check class balance (should always be balanced, but verify)
    unique_classes, class_counts = np.unique(y, return_counts=True)
    
    # Ensure we have at least 2 samples per class for meaningful classification
    if min(class_counts) < 2:
        print(f"Insufficient class representation - Class counts: {class_counts}")
        return np.nan
    
    # CONSISTENT hyperparameter grid for ALL neurons
    hyperparameters = {
        "kernel": ["rbf"],
        "gamma": [1e-2, 1e-1, 1e0, 1e1],
        "C": [1e-2, 1e-1, 1e0, 1e1]
    }
    
    model = SVC()
    
    # Use Leave-One-Out Cross-Validation for consistent analysis
    cv_strategy = LeaveOneOut()
    
    clf = GridSearchCV(
        estimator=model,
        param_grid=hyperparameters,
        cv=cv_strategy,  # LOOCV for all neurons
        n_jobs=-1,       
        verbose=0,
        scoring='accuracy'
    )
    
    try:
        clf.fit(X, y)
        best_score = clf.best_score_
        return best_score
    except Exception as e:
        print(f"SVM fitting failed with {y.shape[0]} samples (classes: {class_counts}): {str(e)}")
        return np.nan

# #### SVM REGRESSION FUNCTION (PARALLELIZED WITH `n_jobs=-1`)
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
import numpy as np

def svmregression(y, X):
    """
    SVM regression using GridSearchCV for hyperparameter tuning, parallelized for faster execution.
    Uses Leave-One-Out Cross-Validation for consistency with binary classifier.
    Args:
        y: Target values.
        X: Features as a NumPy array.
    Returns:
        R^2 score from the best model found via cross-validation.
    """
    # Check minimum samples first
    if y.shape[0] < 4:  # Need at least 4 samples for meaningful LOOCV
        print("Insufficient data - Less Than 4 Total Samples!")
        return np.nan
    
    hyperparameters = {
        'kernel': ['rbf'],
        'C': np.logspace(-3, 3, 5),          # Regularization parameter
        'epsilon': np.logspace(-3, 3, 5),    # Epsilon in the epsilon-SVR model
        'gamma': np.logspace(-5, 5, 10)      # Kernel coefficient for RBF kernel
    }
    
    model = SVR()
    
    # Use Leave-One-Out Cross-Validation for consistency
    cv_strategy = LeaveOneOut()
    
    clf = GridSearchCV(
        estimator=model,
        param_grid=hyperparameters,
        cv=cv_strategy,  # LOOCV for regression too
        n_jobs=-1,       # Use all CPU cores for parallelization
        verbose=0        # Suppress output; set to 1 for minimal feedback
    )
    
    if np.all(np.isnan(X)):
        print("NaN detected in input data!")
        return np.nan

    try:
        clf.fit(X, y)
        R2 = clf.best_score_
        
        # Optionally log the best hyperparameters
        # print(f"Best Parameters: {clf.best_params_}")  # Comment this line to suppress hyperparameter output
        
        return R2
    except Exception as e:
        print(f"Error in SVM regression: {str(e)}")
        return np.nan


### SAVE NEW DECODING OR LOAD PREVIOUS? 
save_decoding_array = 'yes' ### 'yes' or 'no'

### TRIAL SAMPLING STRATEGY FOR TEMPORAL VECTORS
trial_sampling = 'downsample' ### 'upsample_bootstrap', 'downsample', or 'none' 

### HOW MANY TIMES DO YOU WANT TO REPEAT SHUFFLING ANALYSIS FOR EACH NEURON? 
numshuffles = 10

### TEMPORAL DECODING CONFIGURATION
# Choose whether to use raw temporal vectors or difference vectors
use_temporal_difference = False  ### True: Use (event_trial - baseline_trial) vectors, False: Use raw baseline vs event trials

# Minimum number of trials required per condition (baseline/event) for temporal decoding
min_trials_per_condition = 3

### ALL POSSIBLE DECODING STIMULUS VALUES - WILL BE PROCESSED AUTOMATICALLY
all_decoding_stimuli = [
    'CS+', 
    'CS-',
    'Reward', 
    'Reward Omission',
    'CS+ Lick',
    'CS- Lick',
    'Hits',
    'Misses',
    'False_Alarms',
    'Correct_Rejections'
]

def get_filter_for_stimulus(decoding_stimulus):
    """
    Automatically determine the correct filter value based on decoding_stimulus.
    
    Returns:
        filter value: None, 'rewarded', or 'unrewarded'
    """
    if decoding_stimulus == 'Reward':
        return 'rewarded'
    elif decoding_stimulus == 'Reward Omission':
        return 'unrewarded'
    else:
        # For CS+, CS-, CS+ Lick, CS- Lick, Hits, Misses, False_Alarms, Correct_Rejections
        return None

# For backward compatibility, keep the original variable but it will be overridden in the loop
decoding_stimulus = 'Reward Omission'

approach_name = "Temporal Difference Vectors" if use_temporal_difference else "Raw Temporal Vectors"
print("="*60)
print("TRIAL-LEVEL TEMPORAL DECODING CONFIGURATION (ALL DAYS)")
print("="*60)
print(f"Processing: ALL AVAILABLE DAYS")
print(f"Trial sampling: {trial_sampling}")
print(f"Temporal approach: {approach_name}")
print(f"Min trials per condition: {min_trials_per_condition}")
print(f"Number of shuffles: {numshuffles}")
print(f"Cross-validation: Leave-One-Out CV (LOOCV)")
print(f"Will process {len(all_decoding_stimuli)} stimuli across all days")
print("="*60)   

In [ ]:
### Trial-Level Temporal Data Collection - BLOCK 2 (ALL DAYS) #### 

import numpy as np
import pandas as pd

def determine_frame_windows():
    """
    Determine frame windows for each stimulus type and find the minimum window length.
    Returns dictionary with window info and the minimum length for downsampling.
    FIXED: Both baseline and event windows use the same minimum length.
    """
    stimulus_windows = {
        'CS+': {'baseline': list(range(0, 22)), 'event': list(range(23, 44))},
        'CS-': {'baseline': list(range(0, 22)), 'event': list(range(23, 44))},
        'Reward': {'baseline': list(range(0, 22)), 'event': list(range(45, 68))},
        'Reward Omission': {'baseline': list(range(0, 22)), 'event': list(range(45, 68))},
        'CS+ Lick': {'baseline': list(range(0, 22)), 'event': list(range(23, 44))},
        'CS- Lick': {'baseline': list(range(0, 22)), 'event': list(range(23, 44))},
        'Hits': {'baseline': list(range(0, 22)), 'event': list(range(35, 60))},
        'Misses': {'baseline': list(range(0, 22)), 'event': list(range(35, 60))},
        'False_Alarms': {'baseline': list(range(0, 22)), 'event': list(range(35, 60))},
        'Correct_Rejections': {'baseline': list(range(0, 22)), 'event': list(range(35, 60))}
    }
    
    # Find minimum window length for downsampling
    baseline_lengths = [len(windows['baseline']) for windows in stimulus_windows.values()]
    event_lengths = [len(windows['event']) for windows in stimulus_windows.values()]
    
    min_baseline_length = min(baseline_lengths)
    min_event_length = min(event_lengths)
    
    # FIXED: Use overall minimum for both baseline and event to ensure same dimensions
    min_vector_length = min(min_baseline_length, min_event_length)
    
    print(f"Window lengths - Baseline: {set(baseline_lengths)}, Event: {set(event_lengths)}")
    print(f"Will downsample BOTH to: {min_vector_length} frames (for consistent dimensions)")
    
    return stimulus_windows, min_vector_length, min_vector_length

def create_trial_temporal_dataframe(df_master, decoding_stimulus, filter=None):
    """
    Creates trial-level temporal dataframe for a specific stimulus across ALL DAYS.
    Each trial becomes a temporal vector for classification.
    FIXED: Ensures baseline and event vectors have same dimensions.
    """
    print(f"Creating trial-level temporal dataframe for: {decoding_stimulus} across ALL DAYS")
    
    valid_stimuli = [
        'CS+', 'CS-', 
        'Reward', 'Reward Omission', 
        'CS+ Lick', 'CS- Lick',
        'Hits', 'Misses', 'Correct_Rejections', 'False_Alarms'
    ]
    
    if decoding_stimulus not in valid_stimuli:
        raise ValueError(f"decoding_stimulus must be one of {valid_stimuli}")

    # Get all available days and sort them
    all_days = df_master['day'].unique()
    valid_days = [day for day in all_days if pd.notna(day)]
    valid_days.sort(key=lambda x: (int(x.split('_')[0]) if x.split('_')[0].isdigit() else float('inf'), x))




    #==============================================================

    ### UNCOMMENT AND USE THIS LINE IF YOU ONLY WANT TO DO ONE DAY
    valid_days = ['2_SucOmit']   # <-- pick the exact day you want

    #==============================================================



    print(f"Processing days: {valid_days}")
    print(f"Total trials in dataset: {len(df_master)}")

    # Apply stimulus-specific filters to the full dataset
    df_master_filtered = df_master.copy()
    if filter is not None:
        if filter == 'rewarded':
            mask = ~((df_master_filtered['day'] == '2_SucOmit') & 
                    (df_master_filtered['cs_type'] == 'CS+') &
                    (df_master_filtered['reward_status'] != 'rewarded'))
            df_master_filtered = df_master_filtered[mask].copy()
        elif filter == 'unrewarded':
            mask = ~((df_master_filtered['day'] == '2_SucOmit') & 
                    (df_master_filtered['cs_type'] == 'CS+') &
                    (df_master_filtered['reward_status'] != 'unrewarded'))
            df_master_filtered = df_master_filtered[mask].copy()

    # Get window information - FIXED: Both use same length
    stimulus_windows, min_vector_length, _ = determine_frame_windows()
    epoch_baseline = stimulus_windows[decoding_stimulus]['baseline']
    epoch_event = stimulus_windows[decoding_stimulus]['event']
    
    # FIXED: Downsample both windows to the same minimum length
    epoch_baseline_downsampled = epoch_baseline[:min_vector_length]
    epoch_event_downsampled = epoch_event[:min_vector_length]

    # Stimulus-specific trial selection and column determination
    if decoding_stimulus in ['CS+', 'CS-']:
        cs_type = decoding_stimulus
        df_relevant = df_master_filtered[df_master_filtered['cs_type'] == cs_type].copy()
        
        fluor_prefix = 'fluor_plus_frame_' if decoding_stimulus == 'CS+' else 'fluor_minus_frame_'
        baseline_columns = [f"{fluor_prefix}{i}" for i in epoch_baseline_downsampled]
        event_columns = [f"{fluor_prefix}{i}" for i in epoch_event_downsampled]

    # elif decoding_stimulus in ['Reward', 'Reward Omission']:
    #     reward_status = 'rewarded' if decoding_stimulus == 'Reward' else 'unrewarded'
    #     df_relevant = df_master_filtered[df_master_filtered['reward_status'] == reward_status].copy()
        
    #     # Create combined columns
    #     rev_days_mask = df_relevant['day'].isin(['4_RevEarly', '5_RevLate'])
    #     use_minus = rev_days_mask if decoding_stimulus == 'Reward' else ~rev_days_mask

    #     plus_columns = [col for col in df_relevant.columns if col.startswith('fluor_plus_frame_')]
    #     minus_columns = [col for col in df_relevant.columns if col.startswith('fluor_minus_frame_')]
        
    #     all_plus_frames = [int(col.split('_')[-1]) for col in plus_columns if col.split('_')[-1].isdigit()]
    #     all_minus_frames = [int(col.split('_')[-1]) for col in minus_columns if col.split('_')[-1].isdigit()]
    #     all_available_frames = sorted(set(all_plus_frames).intersection(set(all_minus_frames)))
        
    #     for i in all_available_frames:
    #         column_name = f"combined_frame_{i}"
    #         df_relevant[column_name] = np.where(
    #             use_minus,
    #             df_relevant[f'fluor_minus_frame_{i}'],
    #             df_relevant[f'fluor_plus_frame_{i}']
    #         )
        
    #     baseline_columns = [f"combined_frame_{i}" for i in epoch_baseline_downsampled if i in all_available_frames]
    #     event_columns = [f"combined_frame_{i}" for i in epoch_event_downsampled if i in all_available_frames]

    
    elif decoding_stimulus in ['Reward', 'Reward Omission']:
        reward_status = 'rewarded' if decoding_stimulus == 'Reward' else 'unrewarded'
        df_relevant = df_master_filtered[df_master_filtered['reward_status'] == reward_status].copy()
        
        plus_columns = [col for col in df_relevant.columns if col.startswith('fluor_plus_frame_')]
        minus_columns = [col for col in df_relevant.columns if col.startswith('fluor_minus_frame_')]
        
        all_plus_frames = [int(col.split('_')[-1]) for col in plus_columns if col.split('_')[-1].isdigit()]
        all_minus_frames = [int(col.split('_')[-1]) for col in minus_columns if col.split('_')[-1].isdigit()]
        all_available_frames = sorted(set(all_plus_frames).intersection(set(all_minus_frames)))
        
        for i in all_available_frames:
            column_name = f"combined_frame_{i}"
            df_relevant[column_name] = np.where(
                df_relevant['cs_type'] == 'CS+',
                df_relevant[f'fluor_plus_frame_{i}'],
                df_relevant[f'fluor_minus_frame_{i}']
            )
        
        baseline_columns = [f"combined_frame_{i}" for i in epoch_baseline_downsampled if i in all_available_frames]
        event_columns = [f"combined_frame_{i}" for i in epoch_event_downsampled if i in all_available_frames]
    
    elif decoding_stimulus in ['Hits', 'Misses', 'Correct_Rejections', 'False_Alarms']:
        column_map = {
            'Hits': 'hits', 'Misses': 'misses',
            'Correct_Rejections': 'correct_rejections', 'False_Alarms': 'false_alarms'
        }
        df_relevant = df_master_filtered[df_master_filtered[column_map[decoding_stimulus]] == 1].copy()
        
        plus_columns = [col for col in df_relevant.columns if col.startswith('fluor_plus_frame_')]
        minus_columns = [col for col in df_relevant.columns if col.startswith('fluor_minus_frame_')]
        
        all_plus_frames = [int(col.split('_')[-1]) for col in plus_columns if col.split('_')[-1].isdigit()]
        all_minus_frames = [int(col.split('_')[-1]) for col in minus_columns if col.split('_')[-1].isdigit()]
        all_available_frames = sorted(set(all_plus_frames).intersection(set(all_minus_frames)))
        
        for i in all_available_frames:
            column_name = f"combined_frame_{i}"
            df_relevant[column_name] = np.where(
                df_relevant['cs_type'] == 'CS+',
                df_relevant[f'fluor_plus_frame_{i}'],
                df_relevant[f'fluor_minus_frame_{i}']
            )
        
        baseline_columns = [f"combined_frame_{i}" for i in epoch_baseline_downsampled if i in all_available_frames]
        event_columns = [f"combined_frame_{i}" for i in epoch_event_downsampled if i in all_available_frames]

    elif decoding_stimulus in ['CS+ Lick', 'CS- Lick']:
        cs_type = 'CS+' if decoding_stimulus == 'CS+ Lick' else 'CS-'
        df_relevant = df_master_filtered[(df_master_filtered['high_licking'] == 1) & 
                               (df_master_filtered['cs_type'] == cs_type)].copy()
        
        fluor_prefix = 'fluor_plus_frame_' if cs_type == 'CS+' else 'fluor_minus_frame_'
        baseline_columns = [f"{fluor_prefix}{i}" for i in epoch_baseline_downsampled]
        event_columns = [f"{fluor_prefix}{i}" for i in epoch_event_downsampled]

    print(f"Selected {len(df_relevant)} relevant trials for {decoding_stimulus} across all days")
    print(f"Temporal vector length: Baseline={len(baseline_columns)}, Event={len(event_columns)}")
    
    # FIXED: Verify that baseline and event columns have same length
    if len(baseline_columns) != len(event_columns):
        print(f"WARNING: Mismatched column lengths! Baseline={len(baseline_columns)}, Event={len(event_columns)}")
        # Force them to the same length
        min_cols = min(len(baseline_columns), len(event_columns))
        baseline_columns = baseline_columns[:min_cols]
        event_columns = event_columns[:min_cols]
        print(f"FIXED: Both now use {min_cols} columns")
    
    # Collect trial-level temporal data for each neuron across all days
    trial_summaries = []
    grouped = df_relevant.groupby(['day', 'animal', 'fov', 'cell_id'])
    
    print(f"Number of unique cell-day combinations: {len(grouped)}")
    
    processing_errors = 0
    successful_extractions = 0
    day_counts = {}
    
    for cell_info, cell_trials in grouped:
        day, animal, fov, cell_id = cell_info
        
        # Track day counts
        if day not in day_counts:
            day_counts[day] = 0
        day_counts[day] += 1
        
        # Collect temporal vectors for baseline and event trials
        baseline_trial_vectors = []
        event_trial_vectors = []
        valid_trial_indices = []
        
        for idx, (trial_idx, row) in enumerate(cell_trials.iterrows()):
            try:
                # Get baseline and event temporal vectors for this trial
                baseline_vector = row[baseline_columns].values
                event_vector = row[event_columns].values
                
                # Convert to numeric and check validity
                baseline_vector = np.array(baseline_vector, dtype=float)
                event_vector = np.array(event_vector, dtype=float)
                
                if np.all(np.isnan(baseline_vector)) or np.all(np.isnan(event_vector)):
                    continue
                
                # FIXED: Verify dimensions are consistent within trial
                if len(baseline_vector) != len(event_vector):
                    print(f"WARNING: Trial {trial_idx} dimension mismatch: baseline={len(baseline_vector)}, event={len(event_vector)}")
                    continue
                
                # Store temporal vectors (each trial is now a multi-dimensional sample)
                baseline_trial_vectors.append(baseline_vector)
                event_trial_vectors.append(event_vector)
                valid_trial_indices.append(trial_idx)
                
            except Exception as e:
                processing_errors += 1
                if processing_errors <= 5:
                    print(f"Error processing trial {trial_idx}, cell {cell_id}: {str(e)}")
        
        # Only create entry if we have valid temporal data
        if len(baseline_trial_vectors) > 0 and len(event_trial_vectors) > 0:
            
            # Convert to numpy arrays for easier manipulation
            baseline_trial_vectors = np.array(baseline_trial_vectors)
            event_trial_vectors = np.array(event_trial_vectors)
            
            # FIXED: Final dimension check
            if baseline_trial_vectors.shape[1] != event_trial_vectors.shape[1]:
                print(f"ERROR: Cell {cell_id} final dimension mismatch: baseline={baseline_trial_vectors.shape}, event={event_trial_vectors.shape}")
                continue
            
            trial_entry = {
                'day': day,
                'animal': animal,
                'fov': fov,
                'cell_id': cell_id,
                # 'unique_cell_id': unique_cell_id,
                'stimulus': decoding_stimulus,
                'baseline_trial_vectors': baseline_trial_vectors,
                'event_trial_vectors': event_trial_vectors,
                'n_baseline_trials': len(baseline_trial_vectors),
                'n_event_trials': len(event_trial_vectors),
                'n_total_trials': len(cell_trials),
                'n_valid_trials': len(valid_trial_indices),
                'temporal_vector_length_baseline': baseline_trial_vectors.shape[1],
                'temporal_vector_length_event': event_trial_vectors.shape[1],
                'valid_trial_indices': valid_trial_indices
            }
            
            trial_summaries.append(trial_entry)
            successful_extractions += 1
    
    df_trials = pd.DataFrame(trial_summaries)
    
    print(f"\nTrial-level temporal collection summary:")
    print(f"Successful extractions: {successful_extractions}")
    print(f"Processing errors: {processing_errors}")
    print(f"\nShape of trial temporal dataframe: {df_trials.shape}")
    # print(f"Unique neurons across all days: {df_trials['unique_cell_id'].nunique() if len(df_trials) > 0 else 0}")
    print(f"Unique neurons across all days: {df_trials.groupby(['day','animal','fov','cell_id']).ngroups if len(df_trials)>0 else 0}")
    # Print day-wise breakdown
    print(f"\nDay-wise breakdown:")
    for day in sorted(day_counts.keys()):
        day_trials = df_trials[df_trials['day'] == day] if len(df_trials) > 0 else pd.DataFrame()
        print(f"  {day}: {day_counts[day]} neurons, {len(day_trials)} entries")
    
    # Print trial collection summary
    if len(df_trials) > 0:
        total_baseline_trials = df_trials['n_baseline_trials'].sum()
        total_event_trials = df_trials['n_event_trials'].sum()
        print(f"\nTotal baseline trials collected: {total_baseline_trials}")
        print(f"Total event trials collected: {total_event_trials}")
        print(f"Trials per neuron - Baseline: {df_trials['n_baseline_trials'].mean():.1f}±{df_trials['n_baseline_trials'].std():.1f}")
        print(f"Trials per neuron - Event: {df_trials['n_event_trials'].mean():.1f}±{df_trials['n_event_trials'].std():.1f}")
        
        # Show sample for verification
        sample_neuron = df_trials.iloc[0]
        print(f"\nSample neuron {sample_neuron['cell_id']} from day {sample_neuron['day']}:")
        print(f"  Baseline trials: {sample_neuron['n_baseline_trials']} trials")
        print(f"  Event trials: {sample_neuron['n_event_trials']} trials")
        print(f"  Temporal vector lengths: baseline={sample_neuron['temporal_vector_length_baseline']}, event={sample_neuron['temporal_vector_length_event']}")
        print(f"  Sample baseline vector shape: {sample_neuron['baseline_trial_vectors'].shape}")
        print(f"  Sample event vector shape: {sample_neuron['event_trial_vectors'].shape}")
        
        # FIXED: Verify dimensions are consistent
        if sample_neuron['baseline_trial_vectors'].shape[1] == sample_neuron['event_trial_vectors'].shape[1]:
            print(f"  ✓ DIMENSION CHECK PASSED: Both vectors have {sample_neuron['baseline_trial_vectors'].shape[1]} timepoints")
        else:
            print(f"  ✗ DIMENSION CHECK FAILED: Baseline={sample_neuron['baseline_trial_vectors'].shape[1]}, Event={sample_neuron['event_trial_vectors'].shape[1]}")
        
        # Show actual temporal data sample
        print(f"  First baseline trial: {sample_neuron['baseline_trial_vectors'][0][:5]}... (showing first 5 timepoints)")
        print(f"  First event trial: {sample_neuron['event_trial_vectors'][0][:5]}... (showing first 5 timepoints)")
    
    return df_trials

# Dictionary to store all trial temporal dataframes for each stimulus
all_df_trials_temporal = {}

# Process all stimuli across ALL DAYS
print("="*80)
print("COLLECTING TRIAL-LEVEL TEMPORAL DATA FOR ALL STIMULI ACROSS ALL DAYS")
print("="*80)

for current_stimulus in all_decoding_stimuli:
    print(f"\n{'='*70}")
    print(f"PROCESSING STIMULUS: {current_stimulus}")
    print(f"{'='*70}")
    
    # Get the correct filter for this stimulus
    current_filter = get_filter_for_stimulus(current_stimulus)
    print(f"Using filter: {current_filter}")
    
    # Create the trial temporal dataframe for this stimulus across ALL DAYS
    df_trials_temporal = create_trial_temporal_dataframe(
        df_master=df_master,
        decoding_stimulus=current_stimulus,
        filter=current_filter
    )
    
    # Store in dictionary
    all_df_trials_temporal[current_stimulus] = df_trials_temporal
    
    # Verify the structure
    print(f"\nFinal verification of trial temporal dataframe for {current_stimulus}:")
    if len(df_trials_temporal) > 0:
        print("Columns in df_trials_temporal:", df_trials_temporal.columns.tolist())
        sample_data = df_trials_temporal.iloc[0]
        print(f"Data types: baseline_vectors={type(sample_data['baseline_trial_vectors'])}, event_vectors={type(sample_data['event_trial_vectors'])}")
        print(f"Vector shapes: baseline={sample_data['baseline_trial_vectors'].shape}, event={sample_data['event_trial_vectors'].shape}")
        
        # Check days covered
        days_covered = df_trials_temporal['day'].unique()
        print(f"Days covered: {sorted(days_covered)}")
        
        # FINAL DIMENSION VERIFICATION
        baseline_shape = sample_data['baseline_trial_vectors'].shape
        event_shape = sample_data['event_trial_vectors'].shape
        if baseline_shape[1] == event_shape[1]:
            print(f"  ✅ FINAL CHECK PASSED: Both use {baseline_shape[1]} timepoints")
        else:
            print(f"  ❌ FINAL CHECK FAILED: Dimension mismatch still exists!")
    else:
        print(f"Warning: No data found for {current_stimulus} across any days")

print(f"\n{'='*80}")
print("COMPLETED TRIAL-LEVEL TEMPORAL COLLECTION FOR ALL DAYS")
print(f"Processed {len(all_df_trials_temporal)} stimuli across all available days:")
for stimulus in all_df_trials_temporal.keys():
    df_data = all_df_trials_temporal[stimulus]
    n_neurons = len(df_data)
    if n_neurons > 0:
        total_trials = df_data['n_baseline_trials'].sum() + df_data['n_event_trials'].sum()
        days_covered = sorted(df_data['day'].unique())
        print(f"  {stimulus}: {n_neurons} neurons, {total_trials} total trials, days: {days_covered}")
    else:
        print(f"  {stimulus}: No data")
print(f"{'='*80}")

# For backward compatibility, create all_df_trials_enhanced as alias
all_df_trials_enhanced = all_df_trials_temporal

In [ ]:
### Trial-Level Temporal Decoding - BLOCK 3 - PARALLELIZED (ALL DAYS) ####

import numpy as np
import pandas as pd
import os
import pickle
from scipy import stats
from tqdm import tqdm
from sklearn.utils import resample
import multiprocessing as mp
from joblib import Parallel, delayed
import time

def apply_trial_sampling_safe(baseline_trials, event_trials, trial_sampling='downsample', min_total_trials=6, random_state=None):
    """Apply trial sampling for trial-level temporal vectors."""
    if random_state is not None:
        np.random.seed(random_state)
    
    baseline_trials = np.array(baseline_trials)
    event_trials = np.array(event_trials)
    
    n_baseline = len(baseline_trials)
    n_event = len(event_trials)
    
    if trial_sampling == 'none':
        return baseline_trials, event_trials
        
    elif trial_sampling == 'downsample':
        min_trials = min(n_baseline, n_event)
        min_per_condition = max(min_trials, min_total_trials // 2)
        min_per_condition = min(min_per_condition, min_trials)
        
        if min_per_condition == 0:
            return baseline_trials, event_trials
            
        # Randomly select trials (not individual values like frames)
        baseline_indices = np.random.choice(n_baseline, size=min_per_condition, replace=False)
        event_indices = np.random.choice(n_event, size=min_per_condition, replace=False)
        
        baseline_sampled = baseline_trials[baseline_indices]
        event_sampled = event_trials[event_indices]
        
    elif trial_sampling == 'upsample_bootstrap':
        current_total = n_baseline + n_event
        
        if current_total < min_total_trials:
            target_per_condition = min_total_trials // 2
        else:
            target_per_condition = max(n_baseline, n_event)
        
        baseline_sampled = resample(baseline_trials, 
                                  n_samples=target_per_condition, 
                                  random_state=random_state, 
                                  replace=True)
        event_sampled = resample(event_trials, 
                                n_samples=target_per_condition, 
                                random_state=random_state + 1 if random_state else None, 
                                replace=True)
    else:
        raise ValueError(f"trial_sampling must be 'none', 'downsample', or 'upsample_bootstrap', got {trial_sampling}")
    
    return baseline_sampled, event_sampled

def create_difference_vectors(baseline_trials, event_trials, random_state=None):
    """
    Create difference vectors (event - baseline) for temporal difference decoding.
    Pairs trials randomly or in order depending on availability.
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    n_baseline = len(baseline_trials)
    n_event = len(event_trials)
    
    # Determine how many difference vectors we can create
    n_pairs = min(n_baseline, n_event)
    
    if n_pairs == 0:
        return np.array([])
    
    # Randomly pair baseline and event trials
    baseline_indices = np.random.choice(n_baseline, size=n_pairs, replace=False)
    event_indices = np.random.choice(n_event, size=n_pairs, replace=False)
    
    # Create difference vectors
    difference_vectors = event_trials[event_indices] - baseline_trials[baseline_indices]
    
    return difference_vectors

def safe_binaryclassifier(y, X):
    """Wrapper around binaryclassifier that enforces 0-1 bounds."""
    try:
        score = binaryclassifier(y, X)
        
        # Enforce strict bounds
        if score > 1.0:
            print(f"⚠️  Clipping score {score:.6f} -> 1.0")
            score = 1.0
        elif score < 0.0:
            print(f"⚠️  Clipping score {score:.6f} -> 0.0")
            score = 0.0
        
        return score
        
    except Exception as e:
        print(f"Error in binaryclassifier: {str(e)}")
        return np.nan

def process_single_neuron_trial_temporal_decoding(args):
    """
    Process trial-level temporal decoding for a single neuron (for parallel processing).
    
    Parameters:
    -----------
    args : tuple
        (neuron_id, baseline_trial_vectors, event_trial_vectors, min_trials, 
         trial_sampling, use_temporal_difference, numshuffles)
    
    Returns:
    --------
    results : dict
        Processing results for this neuron
    """
    (neuron_id, baseline_trial_vectors, event_trial_vectors, min_trials, 
     trial_sampling, use_temporal_difference, numshuffles) = args
    
    try:
        # Check for minimum trial counts
        if len(baseline_trial_vectors) < min_trials or len(event_trial_vectors) < min_trials:
            return {
                'neuron_id': neuron_id,
                'decoding_score_raw': np.nan,
                'shuffled_score_raw': np.nan,
                'n_baseline_trials': len(baseline_trial_vectors),
                'n_event_trials': len(event_trial_vectors),
                'approach': 'temporal_difference' if use_temporal_difference else 'raw_temporal',
                'error': f'Insufficient trials: baseline={len(baseline_trial_vectors)}, event={len(event_trial_vectors)}, min_required={min_trials}'
            }

        # Check for NaN values
        if np.any(np.isnan(baseline_trial_vectors)) or np.any(np.isnan(event_trial_vectors)):
            return {
                'neuron_id': neuron_id,
                'decoding_score_raw': np.nan,
                'shuffled_score_raw': np.nan,
                'n_baseline_trials': len(baseline_trial_vectors),
                'n_event_trials': len(event_trial_vectors),
                'approach': 'temporal_difference' if use_temporal_difference else 'raw_temporal',
                'error': 'NaN values in trial data'
            }
        
        # Choose decoding approach
        if use_temporal_difference:
            # APPROACH 1: Temporal Difference Vectors
            try:
                difference_vectors = create_difference_vectors(
                    baseline_trial_vectors, event_trial_vectors,
                    random_state=42 + hash(str(neuron_id)) % 1000
                )
                
                if len(difference_vectors) == 0:
                    return {
                        'neuron_id': neuron_id,
                        'decoding_score_raw': np.nan,
                        'shuffled_score_raw': np.nan,
                        'n_baseline_trials': len(baseline_trial_vectors),
                        'n_event_trials': len(event_trial_vectors),
                        'approach': 'temporal_difference',
                        'error': 'No difference vectors could be created'
                    }
                
                # For difference vectors, we compare against zero (no stimulus effect)
                # Positive values indicate event > baseline, negative indicate baseline > event
                X = difference_vectors
                
                # Create labels: 1 if difference vector shows stimulus effect, 0 if not
                # We can use the sign of the mean difference across time as the "ground truth"
                mean_differences = np.mean(difference_vectors, axis=1)
                y = (mean_differences > 0).astype(int)
                
                # If all differences are the same sign, create artificial contrast
                if len(np.unique(y)) == 1:
                    # Split based on magnitude instead
                    median_magnitude = np.median(np.abs(mean_differences))
                    y = (np.abs(mean_differences) > median_magnitude).astype(int)
                
            except Exception as e:
                return {
                    'neuron_id': neuron_id,
                    'decoding_score_raw': np.nan,
                    'shuffled_score_raw': np.nan,
                    'n_baseline_trials': len(baseline_trial_vectors),
                    'n_event_trials': len(event_trial_vectors),
                    'approach': 'temporal_difference',
                    'error': f'Difference vector creation failed: {str(e)}'
                }
                
        else:
            # APPROACH 2: Raw Temporal Vectors (baseline vs event)
            try:
                baseline_sampled, event_sampled = apply_trial_sampling_safe(
                    baseline_trial_vectors, event_trial_vectors,
                    trial_sampling=trial_sampling,
                    min_total_trials=6,
                    random_state=42 + hash(str(neuron_id)) % 1000
                )
                
                # Each trial becomes a multi-dimensional sample
                X = np.vstack([baseline_sampled, event_sampled])
                y = np.concatenate([np.zeros(len(baseline_sampled)), np.ones(len(event_sampled))])
                
            except Exception as e:
                return {
                    'neuron_id': neuron_id,
                    'decoding_score_raw': np.nan,
                    'shuffled_score_raw': np.nan,
                    'n_baseline_trials': len(baseline_trial_vectors),
                    'n_event_trials': len(event_trial_vectors),
                    'approach': 'raw_temporal',
                    'error': f'Trial sampling failed: {str(e)}'
                }

        # Final check: do we have enough total samples?
        if len(X) < 6:  # Reduced from 10 since we have fewer trials
            return {
                'neuron_id': neuron_id,
                'decoding_score_raw': np.nan,
                'shuffled_score_raw': np.nan,
                'n_baseline_trials': len(baseline_trial_vectors),
                'n_event_trials': len(event_trial_vectors),
                'approach': 'temporal_difference' if use_temporal_difference else 'raw_temporal',
                'error': 'Insufficient total samples after processing'
            }

        # Classification
        try:
            # Real classification
            decoding_score_raw = safe_binaryclassifier(y, X)
            
            # Perform multiple shuffles as specified by numshuffles
            shuffle_scores = []
            for shuffle_i in range(numshuffles):
                # Set different random seed for each shuffle
                np.random.seed(42 + hash(str(neuron_id)) % 1000 + shuffle_i * 1000)
                y_shuffled = np.random.permutation(y)
                shuffle_score = safe_binaryclassifier(y_shuffled, X)
                if not np.isnan(shuffle_score):
                    shuffle_scores.append(shuffle_score)
            
            # Calculate average shuffled score across all valid shuffles
            if len(shuffle_scores) > 0:
                decoding_shuffled_score_raw = np.mean(shuffle_scores)
            else:
                decoding_shuffled_score_raw = np.nan

            return {
                'neuron_id': neuron_id,
                'decoding_score_raw': decoding_score_raw,
                'shuffled_score_raw': decoding_shuffled_score_raw,
                'n_baseline_trials': len(baseline_trial_vectors),
                'n_event_trials': len(event_trial_vectors),
                'n_samples_used': len(X),
                'temporal_vector_length': X.shape[1] if len(X.shape) > 1 else 1,
                'approach': 'temporal_difference' if use_temporal_difference else 'raw_temporal',
                'error': None
            }

        except Exception as e:
            return {
                'neuron_id': neuron_id,
                'decoding_score_raw': np.nan,
                'shuffled_score_raw': np.nan,
                'n_baseline_trials': len(baseline_trial_vectors),
                'n_event_trials': len(event_trial_vectors),
                'approach': 'temporal_difference' if use_temporal_difference else 'raw_temporal',
                'error': f'Classification failed: {str(e)}'
            }

    except Exception as e:
        return {
            'neuron_id': neuron_id,
            'decoding_score_raw': np.nan,
            'shuffled_score_raw': np.nan,
            'n_baseline_trials': 0,
            'n_event_trials': 0,
            'approach': 'temporal_difference' if use_temporal_difference else 'raw_temporal',
            'error': f'Processing failed: {str(e)}'
        }

def process_trial_temporal_decoding(df_trials_temporal, df_master_filtered, basedir, decoding_stimulus,
                                   save_decoding_array='yes',
                                   min_trials=3, trial_sampling='downsample', use_temporal_difference=True,
                                   n_jobs=-1, chunk_size=50):
    """
    Trial-level temporal decoding with parallel processing across ALL DAYS.
    Processes all days for the given stimulus, using the old code structure.
    """
    
    print("="*80)
    print("PARALLELIZED TRIAL-LEVEL TEMPORAL DECODING (ALL DAYS)")
    print("="*80)
    print(f"Stimulus: {decoding_stimulus}")
    print(f"Approach: {'Temporal difference vectors' if use_temporal_difference else 'Raw temporal vectors'}")
    print(f"Sampling: {trial_sampling}")
    print(f"Number of shuffles per neuron: {numshuffles}")
    
    if n_jobs == -1:
        n_jobs = mp.cpu_count()
    print(f"Parallel processing: {n_jobs} cores")
    
    # Create safe filename
    safe_stimulus_name = decoding_stimulus.replace(" ", "_").replace("+", "plus").replace("-", "minus")
    approach_suffix = "temporal_diff" if use_temporal_difference else "raw_temporal"
    
    decoding_day_pickle_path = os.path.join(basedir, 'Results', f'configurable_decoding_day_{safe_stimulus_name}_{approach_suffix}.pkl')
    decoding_shuffled_day_pickle_path = os.path.join(basedir, 'Results', f'configurable_decoding_shuffled_day_{safe_stimulus_name}_{approach_suffix}.pkl')

    # Check for existing data
    if save_decoding_array == 'no':
        try:
            print(f"Loading existing trial temporal decoding data for {decoding_stimulus}...")
            with open(decoding_day_pickle_path, 'rb') as f:
                decoding_day = pickle.load(f)
            with open(decoding_shuffled_day_pickle_path, 'rb') as f:
                decoding_shuffled_day = pickle.load(f)
            print("Successfully loaded existing trial temporal decoding data")
            return decoding_day, decoding_shuffled_day
        except FileNotFoundError:
            print(f"No existing trial temporal data found for {decoding_stimulus}")
            save_decoding_array = 'yes'

    # Check if we have data for this stimulus
    if len(df_trials_temporal) == 0:
        print(f"Warning: No trial temporal data found for {decoding_stimulus}")
        return {}, {}

    # Get all valid days and sort them (like the old code)
    all_days = df_trials_temporal['day'].unique()
    valid_days = [day for day in all_days if pd.notna(day)]
    valid_days.sort(key=lambda x: (int(x.split('_')[0]) if x.split('_')[0].isdigit() else float('inf'), x))



    #==============================================================

    # OPTIONAL: UNCOMMENT TO RUN ONE SPECIFIC DAY
    valid_days = ['2_SucOmit']   # <-- uncomment and set the day you want

    #==============================================================




    print(f"Processing days: {valid_days}")

    # Initialize result dictionaries (like the old code)
    decoding_day = {}
    decoding_shuffled_day = {}
    
    # Process each day (like the old code structure)
    for day in valid_days:
        print(f"\nProcessing day: {day}")
        
        # Get day's trials
        day_trials = df_trials_temporal[df_trials_temporal['day'] == day]
        day_neurons = day_trials['cell_id'].unique()
        
        print(f"Number of neurons for {day}: {len(day_neurons)}")
        
        if len(day_neurons) == 0:
            print(f"No neurons found for day {day}, skipping...")
            continue
        
        # Prepare arguments for parallel processing
        process_args = []
        for neuron_id in day_neurons:
            neuron_data = day_trials[day_trials['cell_id'] == neuron_id].iloc[0]
            baseline_trial_vectors = neuron_data['baseline_trial_vectors']
            event_trial_vectors = neuron_data['event_trial_vectors']
            
            process_args.append((neuron_id, baseline_trial_vectors, event_trial_vectors, 
                               min_trials, trial_sampling, use_temporal_difference, numshuffles))

        print(f"Processing {len(process_args)} neurons...")
        start_time = time.time()
        
        # Process in parallel with chunking for memory efficiency
        if len(process_args) > 1 and n_jobs > 1:
            print("Using parallel processing...")
            # Process in chunks to manage memory usage
            if len(process_args) > chunk_size:
                print(f"Processing in chunks of {chunk_size} neurons...")
                results = []
                for i in range(0, len(process_args), chunk_size):
                    chunk = process_args[i:i + chunk_size]
                    chunk_results = Parallel(n_jobs=n_jobs, verbose=0)(
                        delayed(process_single_neuron_trial_temporal_decoding)(args) for args in chunk
                    )
                    results.extend(chunk_results)
                    print(f"Completed chunk {i//chunk_size + 1}/{(len(process_args)-1)//chunk_size + 1}")
            else:
                results = Parallel(n_jobs=n_jobs, verbose=1)(
                    delayed(process_single_neuron_trial_temporal_decoding)(args) for args in process_args
                )
        else:
            print("Using sequential processing...")
            results = [process_single_neuron_trial_temporal_decoding(args) for args in tqdm(process_args, desc="Processing neurons")]
        
        total_time = time.time() - start_time
        print(f"Processing time for {day}: {total_time:.2f} seconds")
        print(f"Average time per neuron: {total_time/len(process_args):.3f} seconds")
        
        # Process results
        day_decoding_raw = np.full(len(day_neurons), np.nan)
        day_shuffled_raw = np.full(len(day_neurons), np.nan)
        
        # Track processing stats
        successful_classifications = 0
        processing_errors = 0
        insufficient_trials = 0
        nan_errors = 0
        
        # Additional trial statistics
        baseline_trial_counts = []
        event_trial_counts = []
        vector_lengths = []
        approaches_used = []
        
        for idx, result in enumerate(results):
            baseline_trial_counts.append(result.get('n_baseline_trials', 0))
            event_trial_counts.append(result.get('n_event_trials', 0))
            vector_lengths.append(result.get('temporal_vector_length', 0))
            approaches_used.append(result.get('approach', 'unknown'))
            
            if result['error'] is None:
                day_decoding_raw[idx] = result['decoding_score_raw']
                day_shuffled_raw[idx] = result['shuffled_score_raw']
                successful_classifications += 1
            else:
                if 'Insufficient trials' in result['error']:
                    insufficient_trials += 1
                elif 'NaN' in result['error']:
                    nan_errors += 1
                else:
                    processing_errors += 1
        
        # Print processing summary for this day
        print(f"\nProcessing summary for {day}:")
        print(f"  Total neurons: {len(day_neurons)}")
        print(f"  Insufficient trials: {insufficient_trials}")
        print(f"  NaN errors: {nan_errors}")
        print(f"  Processing errors: {processing_errors}")
        print(f"  Successful classifications: {successful_classifications}")
        print(f"  Trial counts - Baseline: {np.mean(baseline_trial_counts):.1f}±{np.std(baseline_trial_counts):.1f}, Event: {np.mean(event_trial_counts):.1f}±{np.std(event_trial_counts):.1f}")
        print(f"  Average temporal vector length: {np.mean([v for v in vector_lengths if v > 0]):.1f}")

        # Calculate per-neuron shuffle-corrected values
        day_decoding_corrected = day_decoding_raw - day_shuffled_raw
        day_shuffled_corrected = day_shuffled_raw - day_shuffled_raw  # This will be zeros for valid entries
        
        # Calculate day-wide averages for reporting
        day_shuffled_mean = np.nanmean(day_shuffled_raw)
        
        print(f"\nDecoding results for {day}:")
        print(f"  Raw decoding mean: {np.nanmean(day_decoding_raw):.3f}")
        print(f"  Raw shuffled mean: {day_shuffled_mean:.3f}")
        print(f"  Per-neuron corrected decoding mean: {np.nanmean(day_decoding_corrected):.3f}")
        print(f"  Per-neuron corrected shuffled mean: {np.nanmean(day_shuffled_corrected):.3f}")

        # Create DataFrames with results (like old code structure)
        decoding_day[day] = pd.DataFrame({
            'cell_id': day_neurons,
            'decoding_score': day_decoding_corrected,  # Per-neuron corrected
            'raw_decoding_score': day_decoding_raw,
            'per_neuron_shuffle': day_shuffled_raw,  # Individual neuron shuffle scores
            'shuffle_correction_applied': day_shuffled_raw,  # For backward compatibility
            'feature_strategy': approach_suffix,
            'classification_strategy': f'trial_temporal_{approach_suffix}',
            'n_baseline_trials': baseline_trial_counts,
            'n_event_trials': event_trial_counts,
            'temporal_vector_length': vector_lengths,
            'approach_used': approaches_used,
            'stimulus': decoding_stimulus,
            'day_processed': day
        })

        decoding_shuffled_day[day] = pd.DataFrame({
            'cell_id': day_neurons,
            'shuffled_decoding_score': day_shuffled_corrected,  # Per-neuron corrected (zeros)
            'raw_shuffled_score': day_shuffled_raw,
            'per_neuron_shuffle': day_shuffled_raw,  # Individual neuron shuffle scores
            'shuffle_correction_applied': day_shuffled_raw,  # For backward compatibility
            'feature_strategy': approach_suffix,
            'classification_strategy': f'trial_temporal_{approach_suffix}',
            'n_baseline_trials': baseline_trial_counts,
            'n_event_trials': event_trial_counts,
            'temporal_vector_length': vector_lengths,
            'approach_used': approaches_used,
            'stimulus': decoding_stimulus,
            'day_processed': day
        })

        # Statistical testing - comparing per-neuron corrected scores to zero
        valid_indices = ~np.isnan(day_decoding_corrected)
        if np.sum(valid_indices) > 1:
            # Test if per-neuron corrected scores are significantly different from zero
            t_stat, p_value = stats.ttest_1samp(
                day_decoding_corrected[valid_indices], 
                0  # Test against zero since corrected shuffled scores are zero
            )
            print(f"{day} - T-statistic vs zero: {t_stat:.3f}, p-value: {p_value:.3f}")
            print(f"Valid neurons: {np.sum(valid_indices)}/{len(day_neurons)}")

    # Add metadata (like old code)
    print("\nAdding metadata...")
    required_cols = ['cell_id', 'significant_plus', 'significant_minus', 'cluster_id', 'sex', 'cs_type']
    available_cols = [col for col in required_cols if col in df_master_filtered.columns]
    
    for day in decoding_day.keys():
        df_master_day = df_master_filtered[df_master_filtered['day'] == day]
        
        if len(available_cols) > 1:
            df_master_subset = df_master_day[available_cols].drop_duplicates()
            for df_dict in [decoding_day, decoding_shuffled_day]:
                df_dict[day] = df_dict[day].merge(df_master_subset, on='cell_id', how='left')

    # Save results
    if save_decoding_array == 'yes':
        os.makedirs(os.path.join(basedir, 'Results'), exist_ok=True)
        
        with open(decoding_day_pickle_path, 'wb') as f:
            pickle.dump(decoding_day, f)
        with open(decoding_shuffled_day_pickle_path, 'wb') as f:
            pickle.dump(decoding_shuffled_day, f)
        
        print(f"Saved trial temporal decoding data to {basedir}/Results/")

    return decoding_day, decoding_shuffled_day

# ====================================================================
# PROCESSING SETTINGS
# ====================================================================

# Performance settings
n_jobs_custom = -1            # Number of parallel jobs (-1 = all cores, 1 = sequential)
chunk_size = 100               # Process neurons in chunks (reduced due to more complex processing)

# ====================================================================

approach_name = "Temporal Difference Vectors" if use_temporal_difference else "Raw Temporal Vectors"
print("="*60)
print("PARALLELIZED TRIAL-LEVEL TEMPORAL DECODING (ALL DAYS)")
print("="*60)
print(f"Processing: ALL AVAILABLE DAYS")
print(f"Approach: {approach_name}")
print(f"Min trials per condition: {min_trials_per_condition}")
print(f"Classification: temporal patterns in baseline vs event trials")
print("="*60)

# Dictionary to store all decoding results
all_decoding_day_config = {}
all_decoding_shuffled_day_config = {}

# Process all stimuli across all days (like old code structure)
print("\n" + "="*80)
print("PROCESSING ALL DECODING STIMULI ACROSS ALL DAYS")
print("="*80)

for current_stimulus in all_decoding_stimuli:
    print(f"\n{'='*70}")
    print(f"TRIAL TEMPORAL DECODING FOR STIMULUS: {current_stimulus}")
    print(f"{'='*70}")
    
    # Get the corresponding trial temporal dataframe for this stimulus
    if current_stimulus in all_df_trials_temporal:
        current_df_trials_temporal = all_df_trials_temporal[current_stimulus]
        
        # Check if we have data for this stimulus
        if len(current_df_trials_temporal) == 0:
            print(f"Warning: No trial temporal data found for {current_stimulus}. Skipping...")
            continue
        
        # Run the parallelized trial temporal decoding for this stimulus across ALL DAYS
        decoding_day_config, decoding_shuffled_day_config = process_trial_temporal_decoding(
            df_trials_temporal=current_df_trials_temporal,
            df_master_filtered=df_master,
            basedir=basedir,
            decoding_stimulus=current_stimulus,
            save_decoding_array=save_decoding_array,
            min_trials=min_trials_per_condition,
            trial_sampling=trial_sampling,  # Use the trial_sampling from Block 1
            use_temporal_difference=use_temporal_difference,  # Use setting from Block 1
            n_jobs=n_jobs_custom,
            chunk_size=chunk_size
        )
        
        # Store results for this stimulus
        all_decoding_day_config[current_stimulus] = decoding_day_config
        all_decoding_shuffled_day_config[current_stimulus] = decoding_shuffled_day_config
        
        # Print summary for this stimulus (like old code)
        if len(decoding_day_config) > 0:
            print(f"\nResults summary for {current_stimulus}:")
            for day in sorted(decoding_day_config.keys()):
                day_data = decoding_day_config[day]
                valid_scores = day_data['decoding_score'].dropna()
                raw_scores = day_data['raw_decoding_score'].dropna()
                if len(valid_scores) > 0:
                    print(f"  Day {day}: {len(valid_scores)} neurons")
                    print(f"    Raw score: {raw_scores.mean():.3f} ± {raw_scores.std():.3f}")
                    print(f"    Per-neuron corrected score: {valid_scores.mean():.3f} ± {valid_scores.std():.3f}")
                    print(f"    Total trials used: {day_data['n_baseline_trials'].sum() + day_data['n_event_trials'].sum()}")
                    print(f"    Approach: {day_data['approach_used'].iloc[0] if len(day_data) > 0 else 'unknown'}")
        else:
            print(f"Warning: No decoding results generated for {current_stimulus}")
    else:
        print(f"Error: No trial temporal dataframe found for {current_stimulus}")

print(f"\n{'='*80}")
print("COMPLETED TRIAL-LEVEL TEMPORAL DECODING FOR ALL STIMULI")
print(f"{'='*80}")

# Print overall summary (like old code)
print(f"\nFinal summary across all stimuli:")
print(f"Processed {len(all_decoding_day_config)} stimuli successfully:")
for stimulus in all_decoding_day_config.keys():
    n_days = len(all_decoding_day_config[stimulus])
    total_neurons = 0
    total_trials = 0
    if n_days > 0:
        for day_data in all_decoding_day_config[stimulus].values():
            total_neurons += len(day_data)
            if 'n_baseline_trials' in day_data.columns:
                total_trials += day_data['n_baseline_trials'].sum() + day_data['n_event_trials'].sum()
    print(f"  {stimulus}: {n_days} days, {total_neurons} neurons, {total_trials} total trials")

# For backward compatibility, set the variables to the last processed stimulus
if len(all_decoding_day_config) > 0:
    last_stimulus = list(all_decoding_day_config.keys())[-1]
    decoding_day_config = all_decoding_day_config[last_stimulus]
    decoding_shuffled_day_config = all_decoding_shuffled_day_config[last_stimulus]
    print(f"\nFor backward compatibility, decoding_day_config and decoding_shuffled_day_config")
    print(f"contain results for the last processed stimulus: {last_stimulus}")
else:
    decoding_day_config = {}
    decoding_shuffled_day_config = {}
    print(f"\nWarning: No stimuli were successfully processed")

print(f"\nTo change approach, edit 'use_temporal_difference' in Block 1")
print("="*80)

###########################
## Print all files made

import os
import glob

results_dir = os.path.join(basedir, "Results")
pickle_files = glob.glob(os.path.join(results_dir, "configurable_decoding_day_*_temporal_diff.pkl"))
temporal_files = glob.glob(os.path.join(results_dir, "configurable_decoding_day_*_raw_temporal.pkl"))
all_files = pickle_files + temporal_files
print(f"\nFiles created in {results_dir}:")
for file in sorted(all_files):
    print(os.path.basename(file))

In [ ]:
break

<span style="font-size:44px; color:red;">START HERE IF YOU WANT TO JUST IMPORT PKL FILES</span>

# Plotting and Analyzing Decoding

# Select the Data for the Bar Code and the Corresponding Stats Will Be Done
## These Plots and Stats are Done Within a Particular Decoding_Stimulus

In [ ]:
import glob

# ====================================================================
# USER CONFIGURATION - EDIT THESE VALUES
# ====================================================================

# Define available stimuli (for reference)
stimulus_list = [
    'CS+', 'CS-', 'Reward', 'Reward Omission',
    'CS+ Lick', 'CS- Lick', 'Hits', 'Misses', 
    'Correct_Rejections', 'False_Alarms'
]

# NEW: Choose which specific stimulus to plot
stimulus_to_plot = 'CS+'  # Pick ONE stimulus from the list above

# NEW: Choose which days to include (empty list = all available days)
days_to_include = ['0_Early', '1_Late']  # Use [] for all days, or specify like ['1_Late', '2_SucOmit', '3_CueOmit']
###['0_Early', '1_Late'] or ['1_Late', '2_SucOmit', '3_CueOmit' ] or ['4_RevEarly, '5_RevLate']

# NEW: Choose whether to show raw scores or normalized (per-neuron corrected) scores
use_raw_scores = False      # True = raw_decoding_score vs per_neuron_shuffle
                           # False = decoding_score (normalized, centered on 0)

# Choose which category of neurons to plot
category_to_plot = 'Any Significant'  # Options: 'All Cells', 'Any Significant', 
                                 #          'CS+ Only Significant', 'CS- Only Significant',
                                 #          'Both CS+&CS- Significant', 'Non-significant'

# Choose plot layout
use_combined_days = True  # True = all days on same plot (easier comparison)
                           # False = separate plot per day (1 row × days)

# ====================================================================

def create_comprehensive_cdf_plots_from_configurable_decoding(decoding_day_config, decoding_shuffled_day_config, 
                                                            df_master, decoding_stimulus, basedir,
                                                            category_to_plot='All Cells', use_combined_days=False,
                                                            use_raw_scores=True, days_to_include=None):
    """
    Create comprehensive CDF plots for the new configurable Strategy 2 decoding results.
    
    UPDATED FOR PER-NEURON SHUFFLE CORRECTION AND USER SELECTION:
    - User selects stimulus, days, score type, category, and layout
    - Plots raw_decoding_score vs per_neuron_shuffle OR decoding_score vs zero
    - Uses proper statistical tests for the chosen data structure
    """
    
    # Check if we have any data
    if len(decoding_day_config) == 0:
        print("No decoding data available for plotting!")
        return
    
    # Filter days if specified
    all_days = sorted(decoding_day_config.keys())
    if days_to_include:
        valid_days = [day for day in days_to_include if day in all_days]
        missing_days = [day for day in days_to_include if day not in all_days]
        
        if missing_days:
            print(f"Warning: Requested days not found: {missing_days}")
        if not valid_days:
            print(f"❌ None of the requested days {days_to_include} were found in {all_days}")
            return
        
        print(f"Using filtered days: {valid_days}")
        days = valid_days
    else:
        days = all_days
        print(f"Using all available days: {days}")
    
    # Check what columns are available in the first day's data
    sample_day_data = decoding_day_config[days[0]]
    available_columns = sample_day_data.columns.tolist()
    
    print("Available columns in decoding data:")
    print(available_columns)
    
    # Verify we have the required columns based on score type
    if use_raw_scores:
        required_columns = ['raw_decoding_score', 'per_neuron_shuffle']
        score_type = 'Raw'
        chance_level = 0.5
    else:
        required_columns = ['decoding_score']
        score_type = 'Normalized'
        chance_level = 0.0
    
    missing_columns = [col for col in required_columns if col not in available_columns]
    
    if missing_columns:
        print(f"⚠️  ERROR: Missing required columns for {score_type.lower()} analysis: {missing_columns}")
        if use_raw_scores:
            print("    Raw score analysis requires updated Block 3 with per-neuron shuffle correction!")
        else:
            print("    Normalized score analysis requires 'decoding_score' column!")
        return
    
    # Check for significance columns
    has_significance_data = ('significant_plus' in available_columns and 
                           'significant_minus' in available_columns)
    
    print(f"Has significance data: {has_significance_data}")
    
    # Define all available categories
    all_categories = {
        "All Cells": lambda df: df,
        "Any Significant": lambda df: df[
            (df['significant_plus'] == 1) | 
            (df['significant_minus'] == 1)
        ],
        "CS+ Only Significant": lambda df: df[
            (df['significant_plus'] == 1) & 
            (df['significant_minus'] == 0)
        ],
        "CS- Only Significant": lambda df: df[
            (df['significant_plus'] == 0) & 
            (df['significant_minus'] == 1)
        ],
        "Both CS+&CS- Significant": lambda df: df[
            (df['significant_plus'] == 1) & 
            (df['significant_minus'] == 1)
        ],
        "Non-significant": lambda df: df[
            (df['significant_plus'] == 0) & 
            (df['significant_minus'] == 0)
        ]
    }
    
    # Determine which categories are available
    if has_significance_data:
        available_categories = all_categories
        print(f"✅ All categories accessible")
    else:
        available_categories = {"All Cells": all_categories["All Cells"]}
        print("⚠️  Only 'All Cells' category accessible (no significance data)")
    
    # Validate user's category selection
    if category_to_plot not in available_categories:
        print(f"❌ Requested category '{category_to_plot}' not available!")
        print(f"   Available categories: {list(available_categories.keys())}")
        if not has_significance_data and category_to_plot != "All Cells":
            print("   Note: Significance-based categories require significance columns in your data.")
        return
    
    print(f"✅ Using category: '{category_to_plot}'")
    print(f"✅ Layout: {'Combined days' if use_combined_days else 'Separate days'}")
    print(f"✅ Score type: {'Raw vs per-neuron shuffle' if use_raw_scores else 'Normalized vs zero'}")
    
    category_func = available_categories[category_to_plot]
    
    # Set up the plot based on layout choice
    sns.set_style('white')
    
    if use_combined_days:
        # Combined layout: all days on same plot
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        plot_layout = 'combined'
        print(f"Using COMBINED layout: all days on same plot")
    else:
        # Separate layout: one plot per day
        fig, axs = plt.subplots(1, len(days), figsize=(5*len(days), 6), sharex=True, sharey=True)
        if len(days) == 1:
            axs = [axs]  # Make it iterable
        plot_layout = 'separate'
        print(f"Using SEPARATE layout: one plot per day")
    
    # Colors for days (when using combined layout)
    day_colors = plt.cm.Set1(np.linspace(0, 1, len(days)))
    
    # Determine global min and max for x-axis based on score type
    all_data_list = []
    for day in days:
        try:
            day_data = decoding_day_config[day].drop_duplicates(subset=['cell_id'])
            
            # Apply category filter
            try:
                filtered_data = category_func(day_data)
            except KeyError:
                if category_to_plot != "All Cells":
                    continue
                filtered_data = day_data
            
            if len(filtered_data) == 0:
                continue
            
            if use_raw_scores:
                data1 = filtered_data['raw_decoding_score'].dropna()
                data2 = filtered_data['per_neuron_shuffle'].dropna()
                all_data_list.extend([data1, data2])
            else:
                data1 = filtered_data['decoding_score'].dropna()
                all_data_list.append(data1)
                
        except Exception as e:
            print(f"Warning: Error processing global limits for day {day}: {str(e)}")
            continue
    
    if all_data_list:
        all_data = pd.concat(all_data_list)
        global_min, global_max = np.nanmin(all_data), np.nanmax(all_data)
        # Add some padding
        range_padding = (global_max - global_min) * 0.05
        global_min -= range_padding
        global_max += range_padding
    else:
        if use_raw_scores:
            global_min, global_max = 0.0, 1.0  # Default range for raw scores
        else:
            global_min, global_max = -0.5, 0.5  # Default range for normalized scores
        print(f"Warning: Could not determine global limits, using default range [{global_min}, {global_max}]")
    
    print(f"\nPlotting {category_to_plot} - {score_type.lower()} scores")
    print(f"Global range: {global_min:.3f} to {global_max:.3f}")
    
    # Process each day
    for d, day in enumerate(days):
        print(f"\nProcessing day: {day}")
        
        # Get the appropriate axis
        if plot_layout == 'combined':
            current_ax = ax
            day_color = day_colors[d]
        else:
            current_ax = axs[d]
            day_color = 'blue'  # Use consistent color for separate plots
        
        # Retrieve and filter decoding data for the current day
        day_data_real = decoding_day_config[day].drop_duplicates(subset=['cell_id']).copy()
        
        print(f"  Day {day} data shape: {day_data_real.shape}")
        
        # Apply category filter
        try:
            filtered_data = category_func(day_data_real)
        except KeyError:
            if category_to_plot != "All Cells":
                print(f"  ⚠️  Significance columns missing for day {day}, skipping")
                if plot_layout == 'separate':
                    current_ax.text(0.5, 0.5, f'No significance data\navailable for {day}', 
                                   horizontalalignment='center', verticalalignment='center',
                                   transform=current_ax.transAxes, fontsize=12, alpha=0.7)
                continue
            filtered_data = day_data_real
        
        if len(filtered_data) > 0:
            # Get appropriate scores based on user selection
            if use_raw_scores:
                data1 = filtered_data['raw_decoding_score'].dropna()
                data2 = filtered_data['per_neuron_shuffle'].dropna()
                
                # Ensure we have matching data for paired analysis
                if len(data1) != len(data2):
                    print(f"  ⚠️  Warning: Mismatched data lengths for {day}")
                    print(f"      Raw: {len(data1)}, Shuffle: {len(data2)}")
                    # Use minimum length to avoid errors
                    min_len = min(len(data1), len(data2))
                    data1 = data1.iloc[:min_len]
                    data2 = data2.iloc[:min_len]
            else:
                data1 = filtered_data['decoding_score'].dropna()
                data2 = None  # No comparison data for normalized scores
            
            # Plot CDFs
            if len(data1) > 0:
                if plot_layout == 'combined':
                    # Combined: use day-specific colors and labels
                    if use_raw_scores:
                        sns.ecdfplot(data=data1, ax=current_ax, color=day_color, 
                                   label=f'{day} Raw (n={len(data1)})')
                        
                        if data2 is not None and len(data2) > 0:
                            sns.ecdfplot(data=data2, ax=current_ax, color=day_color, 
                                       linestyle='--', alpha=0.7,
                                       label=f'{day} Shuffle')
                    else:
                        sns.ecdfplot(data=data1, ax=current_ax, color=day_color, 
                                   label=f'{day} Normalized (n={len(data1)})')
                else:
                    # Separate: use consistent colors
                    if use_raw_scores:
                        sns.ecdfplot(data=data1, ax=current_ax, color=day_color, 
                                   label=f'Raw Decoding (n={len(data1)})')
                        
                        if data2 is not None and len(data2) > 0:
                            sns.ecdfplot(data=data2, ax=current_ax, color='grey', 
                                       linestyle='--', 
                                       label=f'Per-Neuron Shuffle (n={len(data2)})')
                    else:
                        sns.ecdfplot(data=data1, ax=current_ax, color=day_color, 
                                   label=f'Normalized (n={len(data1)})')
            
            # Print statistics
            print(f"  {category_to_plot}: {len(data1)} cells")
            if len(data1) > 0:
                print(f"    {score_type} - Range: {np.min(data1):.3f} to {np.max(data1):.3f}")
                print(f"    {score_type} - Mean ± SEM: {np.mean(data1):.3f} ± {np.std(data1)/np.sqrt(len(data1)):.3f}")
                
            if use_raw_scores and data2 is not None and len(data2) > 0:
                print(f"    Per-Neuron Shuffle - Range: {np.min(data2):.3f} to {np.max(data2):.3f}")
                print(f"    Per-Neuron Shuffle - Mean ± SEM: {np.mean(data2):.3f} ± {np.std(data2)/np.sqrt(len(data2)):.3f}")
                
                # Statistical comparison for raw scores
                if len(data1) > 1 and len(data2) > 1:
                    from scipy import stats
                    try:
                        # Paired t-test since these are matched raw vs shuffle for same neurons
                        stat, pval = stats.ttest_rel(data1, data2)
                        print(f"    Paired t-test (raw vs shuffle): t={stat:.3f}, p={pval:.3e}")
                        
                        # Also test if raw scores are different from chance (0.5)
                        stat_chance, pval_chance = stats.ttest_1samp(data1, 0.5)
                        print(f"    One-sample t-test vs chance (0.5): t={stat_chance:.3f}, p={pval_chance:.3e}")
                        
                    except Exception as test_error:
                        print(f"    Statistical tests failed: {test_error}")
                        
                # Calculate per-neuron corrected scores for additional info
                corrected_scores = data1 - data2
                print(f"    Per-Neuron Corrected - Mean ± SEM: {np.mean(corrected_scores):.3f} ± {np.std(corrected_scores)/np.sqrt(len(corrected_scores)):.3f}")
                
                # Test corrected scores against zero
                if len(corrected_scores) > 1:
                    try:
                        stat_corrected, pval_corrected = stats.ttest_1samp(corrected_scores, 0)
                        print(f"    Corrected scores vs zero: t={stat_corrected:.3f}, p={pval_corrected:.3e}")
                    except:
                        pass
            
            elif not use_raw_scores and len(data1) > 1:
                # Statistical test for normalized scores against zero
                from scipy import stats
                try:
                    stat_zero, pval_zero = stats.ttest_1samp(data1, 0)
                    print(f"    One-sample t-test vs zero: t={stat_zero:.3f}, p={pval_zero:.3e}")
                except Exception as test_error:
                    print(f"    Statistical test failed: {test_error}")
        else:
            print(f"  No data available for {category_to_plot} on {day}")
            if plot_layout == 'separate':
                current_ax.text(0.5, 0.5, f'No {category_to_plot}\ndata available', 
                               horizontalalignment='center', verticalalignment='center',
                               transform=current_ax.transAxes, fontsize=12, alpha=0.7)
        
        # Set axis properties for separate layout
        if plot_layout == 'separate':
            # Add vertical line at chance level
            current_ax.axvline(chance_level, linestyle=':', color='black', linewidth=1, alpha=0.7, 
                              label=f'Chance ({chance_level})')
            
            current_ax.set_xlim(global_min, global_max)
            current_ax.set_ylim(0, 1)
            current_ax.set_title(f'Day: {day}', fontsize=12)
            current_ax.set_xlabel(f'{score_type} Decoding Score')
            if d == 0:
                current_ax.set_ylabel('Cumulative Probability')
            
            # Add legend for separate plots
            if len(filtered_data) > 0:
                current_ax.legend(fontsize='small', loc='best')
    
    # Final formatting for combined layout
    if plot_layout == 'combined':
        # Add vertical line at chance level
        ax.axvline(chance_level, linestyle=':', color='black', linewidth=2, alpha=0.7, 
                  label=f'Chance ({chance_level})')
        
        ax.set_xlim(global_min, global_max)
        ax.set_ylim(0, 1)
        ax.set_xlabel(f'{score_type} Decoding Score', fontsize=12)
        ax.set_ylabel('Cumulative Probability', fontsize=12)
        ax.legend(fontsize='small', loc='best')
    
    # Create title
    if plot_layout == 'combined':
        if use_raw_scores:
            title = f'{decoding_stimulus}: {category_to_plot} - Raw vs Per-Neuron Shuffle (All Days)'
        else:
            title = f'{decoding_stimulus}: {category_to_plot} - Normalized Scores (All Days)'
    else:
        if use_raw_scores:
            title = f'{decoding_stimulus}: {category_to_plot} - Raw vs Per-Neuron Shuffle by Day'
        else:
            title = f'{decoding_stimulus}: {category_to_plot} - Normalized Scores by Day'
    
    plt.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    
    # Create safe filename
    safe_stimulus_name = decoding_stimulus.replace(" ", "_").replace("+", "plus").replace("-", "minus")
    safe_category_name = category_to_plot.replace(" ", "_").replace("+", "plus").replace("-", "minus")
    layout_suffix = "Combined" if plot_layout == "combined" else "Separate"
    score_suffix = "Raw" if use_raw_scores else "Normalized"
    
    # Save figure
    try:
        save_path_base = f"{basedir}/Results/{safe_stimulus_name}_{safe_category_name}_{score_suffix}_CDF_{layout_suffix}"
        fig.savefig(f"{save_path_base}.pdf", format='PDF', bbox_inches='tight')
        fig.savefig(f"{save_path_base}.png", format='PNG', bbox_inches='tight', dpi=300)
        print(f"\nSaved CDF plots to {save_path_base}.*")
    except Exception as e:
        print(f"Error saving figures: {e}")
    plt.show()
    plt.close(fig)  # Close figure to save memory

def create_simple_cdf_comparison(decoding_day_config, decoding_shuffled_day_config, decoding_stimulus, basedir,
                                category_to_plot='All Cells', use_combined_days=False):
    """
    Create a simple CDF comparison plot as a fallback.
    """
    days = sorted(decoding_day_config.keys())
    
    if len(days) == 0:
        print("No days to plot!")
        return
    
    # Set up plot based on layout choice
    if use_combined_days:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        day_colors = plt.cm.Set1(np.linspace(0, 1, len(days)))
    else:
        fig, axs = plt.subplots(1, len(days), figsize=(5*len(days), 5), sharex=True, sharey=True)
        if len(days) == 1:
            axs = [axs]
    
    # Check for significance data
    sample_data = decoding_day_config[days[0]]
    has_significance_data = ('significant_plus' in sample_data.columns and 
                           'significant_minus' in sample_data.columns)
    
    # Define category function
    if category_to_plot == "All Cells" or not has_significance_data:
        category_func = lambda df: df
    elif category_to_plot == "Any Significant":
        category_func = lambda df: df[(df['significant_plus'] == 1) | (df['significant_minus'] == 1)]
    elif category_to_plot == "CS+ Only Significant":
        category_func = lambda df: df[(df['significant_plus'] == 1) & (df['significant_minus'] == 0)]
    elif category_to_plot == "CS- Only Significant":
        category_func = lambda df: df[(df['significant_plus'] == 0) & (df['significant_minus'] == 1)]
    elif category_to_plot == "Both CS+&CS- Significant":
        category_func = lambda df: df[(df['significant_plus'] == 1) & (df['significant_minus'] == 1)]
    elif category_to_plot == "Non-significant":
        category_func = lambda df: df[(df['significant_plus'] == 0) & (df['significant_minus'] == 0)]
    else:
        category_func = lambda df: df
    
    for d, day in enumerate(days):
        # Get the appropriate axis
        if use_combined_days:
            current_ax = ax
            day_color = day_colors[d]
        else:
            current_ax = axs[d]
            day_color = 'blue'
        
        # Get and filter data for this day
        day_data = decoding_day_config[day].drop_duplicates(subset=['cell_id'])
        
        try:
            filtered_data = category_func(day_data)
        except KeyError:
            filtered_data = day_data  # Fallback to all data
        
        if len(filtered_data) > 0:
            raw_scores = filtered_data['raw_decoding_score'].dropna()
            shuffle_scores = filtered_data['per_neuron_shuffle'].dropna()
            
            # Ensure matching lengths
            if len(raw_scores) != len(shuffle_scores):
                min_len = min(len(raw_scores), len(shuffle_scores))
                raw_scores = raw_scores.iloc[:min_len]
                shuffle_scores = shuffle_scores.iloc[:min_len]
            
            # Plot CDFs
            if use_combined_days:
                sns.ecdfplot(data=raw_scores, ax=current_ax, color=day_color, 
                            label=f'{day} Raw (n={len(raw_scores)})')
                sns.ecdfplot(data=shuffle_scores, ax=current_ax, color=day_color, 
                            linestyle='--', alpha=0.7, label=f'{day} Shuffle')
            else:
                sns.ecdfplot(data=raw_scores, ax=current_ax, color=day_color, 
                            label=f'Raw Decoding (n={len(raw_scores)})')
                sns.ecdfplot(data=shuffle_scores, ax=current_ax, color='grey', 
                            linestyle='--', label=f'Per-Neuron Shuffle (n={len(shuffle_scores)})')
                
                current_ax.set_title(f'Day: {day}')
                current_ax.set_xlabel('Raw Decoding Score')
                if d == 0:
                    current_ax.set_ylabel('Cumulative Probability')
                current_ax.legend()
            
            # Print statistics
            print(f"\nDay {day} - {category_to_plot}:")
            print(f"  Raw Decoding: {np.mean(raw_scores):.3f} ± {np.std(raw_scores):.3f} (n={len(raw_scores)})")
            print(f"  Per-Neuron Shuffle: {np.mean(shuffle_scores):.3f} ± {np.std(shuffle_scores):.3f} (n={len(shuffle_scores)})")
            
            if len(raw_scores) > 1 and len(shuffle_scores) > 1:
                from scipy import stats
                try:
                    # Paired t-test
                    stat, pval = stats.ttest_rel(raw_scores, shuffle_scores)
                    print(f"  Paired t-test: t={stat:.3f}, p={pval:.3e}")
                    
                    # Test against chance
                    stat_chance, pval_chance = stats.ttest_1samp(raw_scores, 0.5)
                    print(f"  Raw vs chance (0.5): t={stat_chance:.3f}, p={pval_chance:.3e}")
                    
                except Exception as e:
                    print(f"  Statistical tests failed: {e}")
    
    # Final formatting
    if use_combined_days:
        ax.axvline(0.5, linestyle=':', color='black', linewidth=2, alpha=0.7, label='Chance (0.5)')
        ax.set_xlabel('Raw Decoding Score')
        ax.set_ylabel('Cumulative Probability')
        ax.legend()
        title_suffix = "(All Days Combined)"
    else:
        # Add chance lines to separate plots
        for ax_item in (axs if len(days) > 1 else [axs]):
            ax_item.axvline(0.5, linestyle=':', color='black', linewidth=1, alpha=0.7)
        title_suffix = "by Day"
    
    plt.suptitle(f'{decoding_stimulus}: {category_to_plot} - Raw vs Per-Neuron Shuffle {title_suffix}', fontsize=14)
    plt.tight_layout()

    # Save figure
    safe_stimulus_name = decoding_stimulus.replace(" ", "_").replace("+", "plus").replace("-", "minus")
    safe_category_name = category_to_plot.replace(" ", "_").replace("+", "plus").replace("-", "minus")
    layout_suffix = "Combined" if use_combined_days else "Separate"
    
    save_path = f"{basedir}/Results/{safe_stimulus_name}_{safe_category_name}_Simple_Raw_vs_Shuffle_CDF_{layout_suffix}.pdf"
    fig.savefig(save_path, format='PDF', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close(fig)

def load_and_plot_single_stimulus_cdf(basedir, stimulus_to_plot, feature_strategy=None, 
                                     category_to_plot='All Cells', use_combined_days=False,
                                     use_raw_scores=True, days_to_include=None):
    """
    Load pickled decoding results for a SINGLE STIMULUS and generate CDF plots.
    
    UPDATED FOR PER-NEURON SHUFFLE CORRECTION AND USER SELECTION:
    - Loads and plots raw_decoding_score vs per_neuron_shuffle
    - Uses appropriate statistical tests for the new data structure
    - User selects one specific stimulus, neuron category, and layout
    
    Parameters:
    -----------
    basedir : str
        Base directory containing Results folder with pickle files
    stimulus_to_plot : str
        Specific stimulus to plot (e.g., 'Reward', 'CS+', etc.)
    feature_strategy : str, optional
        Specific feature strategy to load (e.g., 'means_slopes'). 
        If None, will auto-detect from available files.
    category_to_plot : str
        Which neuron category to plot
    use_combined_days : bool
        Whether to show all days on same plot or separate plots
    use_raw_scores : bool
        Whether to show raw scores vs shuffle or normalized scores
    days_to_include : list, optional
        Specific days to include, or None for all days
    """
    
    print("="*80)
    print("LOADING PICKLED DECODING RESULTS FOR SINGLE STIMULUS")
    print("(Raw Decoding vs Per-Neuron Shuffle)")
    print("="*80)
    print(f"Stimulus: {stimulus_to_plot}")
    print(f"Category: {category_to_plot}")
    print(f"Layout: {'Combined days' if use_combined_days else 'Separate days'}")
    print(f"Score type: {'Raw vs shuffle' if use_raw_scores else 'Normalized'}")
    if days_to_include:
        print(f"Days filter: {days_to_include}")
    print("="*80)
    
    results_dir = os.path.join(basedir, 'Results')
    
    if not os.path.exists(results_dir):
        print(f"Error: Results directory not found at {results_dir}")
        return
    
    # Validate stimulus selection
    stimulus_mapping = {
        'CS+': 'CSplus',
        'CS-': 'CSminus', 
        'Reward': 'Reward',
        'Reward Omission': 'Reward_Omission',
        'CS+ Lick': 'CSplus_Lick',
        'CS- Lick': 'CSminus_Lick',
        'Hits': 'Hits',
        'Misses': 'Misses',
        'False_Alarms': 'False_Alarms',
        'Correct_Rejections': 'Correct_Rejections'
    }
    
    if stimulus_to_plot not in stimulus_mapping:
        print(f"❌ Invalid stimulus selection: '{stimulus_to_plot}'")
        print(f"   Available stimuli: {list(stimulus_mapping.keys())}")
        return
    
    safe_stimulus = stimulus_mapping[stimulus_to_plot]
    print(f"✅ Stimulus mapping: '{stimulus_to_plot}' -> '{safe_stimulus}'")
    
    # Auto-detect feature strategy if not provided
    if feature_strategy is None:
        # Look for this specific stimulus to determine feature strategy
        pattern = os.path.join(results_dir, f'configurable_decoding_day_{safe_stimulus}_*.pkl')
        config_files = glob.glob(pattern)
        
        if len(config_files) == 0:
            print(f"❌ No configurable decoding files found for '{stimulus_to_plot}' in {results_dir}")
            print("Available files:")
            all_files = glob.glob(os.path.join(results_dir, 'configurable_decoding_day_*.pkl'))
            for file in sorted(all_files):
                print(f"  {os.path.basename(file)}")
            return
        
        # Extract feature strategy from first file
        filename = os.path.basename(config_files[0])
        print(f"Auto-detecting feature strategy from file: {filename}")
        
        # Parse the filename: configurable_decoding_day_{STIMULUS}_{FEATURE_STRATEGY}.pkl
        base_name = filename.replace('.pkl', '')
        expected_prefix = f'configurable_decoding_day_{safe_stimulus}_'
        
        if not base_name.startswith(expected_prefix):
            print(f"❌ Unexpected filename format: {filename}")
            return
        
        feature_strategy = base_name[len(expected_prefix):]
        print(f"✅ Auto-detected feature strategy: {feature_strategy}")
    
    # Construct file paths for the specific stimulus
    main_file = os.path.join(results_dir, f'configurable_decoding_day_{safe_stimulus}_{feature_strategy}.pkl')
    shuffled_file = os.path.join(results_dir, f'configurable_decoding_shuffled_day_{safe_stimulus}_{feature_strategy}.pkl')
    
    # Check if main file exists
    if not os.path.exists(main_file):
        print(f"❌ Main decoding file not found: {main_file}")
        return
    
    # Check if shuffled file exists (optional for per-neuron correction)
    if not os.path.exists(shuffled_file):
        print(f"⚠️  Shuffled decoding file not found: {shuffled_file}")
        print("Note: For per-neuron correction, we primarily use the main file.")
    
    try:
        # Load the main pickle file (contains both raw and per-neuron shuffle data)
        print(f"Loading {main_file}")
        with open(main_file, 'rb') as f:
            decoding_day_config = pickle.load(f)
        
        # Load shuffled file if it exists (for compatibility)
        decoding_shuffled_day_config = {}
        if os.path.exists(shuffled_file):
            print(f"Loading {shuffled_file}")
            with open(shuffled_file, 'rb') as f:
                decoding_shuffled_day_config = pickle.load(f)
        else:
            print("Using per-neuron shuffle data from main file only")
        
        # Check if data was loaded successfully
        if len(decoding_day_config) == 0:
            print(f"❌ No data found in decoding files for {stimulus_to_plot}")
            return
        
        # Verify that the data has the required columns for per-neuron analysis
        sample_day = list(decoding_day_config.keys())[0]
        sample_data = decoding_day_config[sample_day]
        required_cols = ['raw_decoding_score', 'per_neuron_shuffle']
        missing_cols = [col for col in required_cols if col not in sample_data.columns]
        
        if missing_cols:
            print(f"⚠️  Missing required columns for per-neuron analysis: {missing_cols}")
            print("    This requires the updated Block 3 with per-neuron shuffle correction!")
            return
        
        print(f"✅ Successfully loaded per-neuron data for {stimulus_to_plot}")
        print(f"  Days available: {list(decoding_day_config.keys())}")
        
        # Print summary stats
        total_neurons = 0
        for day_name, day_data in decoding_day_config.items():
            day_neurons = len(day_data)
            total_neurons += day_neurons
            print(f"  {day_name}: {day_neurons} neurons")
        print(f"  Total neuron measurements: {total_neurons}")
        
        # Generate CDF plots for this stimulus
        print(f"\n🎯 Generating CDF plots for {stimulus_to_plot}...")
        
        try:
            create_comprehensive_cdf_plots_from_configurable_decoding(
                decoding_day_config, decoding_shuffled_day_config, 
                df_master, stimulus_to_plot, basedir,
                category_to_plot=category_to_plot, 
                use_combined_days=use_combined_days,
                use_raw_scores=use_raw_scores,
                days_to_include=days_to_include
            )
            print(f"✅ Successfully created comprehensive CDF plots for {stimulus_to_plot}")
            
        except Exception as e:
            print(f"❌ Error in comprehensive CDF plot for {stimulus_to_plot}: {e}")
            print("Falling back to simple CDF comparison...")
            
            try:
                create_simple_cdf_comparison(
                    decoding_day_config, decoding_shuffled_day_config, 
                    stimulus_to_plot, basedir,
                    category_to_plot=category_to_plot,
                    use_combined_days=use_combined_days
                )
                print(f"✅ Successfully created simple CDF plots for {stimulus_to_plot}")
                
            except Exception as e2:
                print(f"❌ Error in simple CDF plot for {stimulus_to_plot}: {e2}")
                return
        
    except Exception as e:
        print(f"❌ Error loading pickle files for {stimulus_to_plot}: {e}")
        return
    
    # Final summary
    print(f"\n{'='*80}")
    print("SINGLE STIMULUS CDF PLOTTING COMPLETE")
    print(f"{'='*80}")
    print(f"✅ Stimulus plotted: {stimulus_to_plot}")
    print(f"✅ Category plotted: {category_to_plot}")
    print(f"✅ Layout: {'Combined days' if use_combined_days else 'Separate days'}")
    print(f"✅ Score type: {'Raw vs shuffle' if use_raw_scores else 'Normalized'}")
    print(f"✅ Feature strategy used: {feature_strategy}")
    if days_to_include:
        print(f"✅ Days included: {days_to_include}")
    else:
        print(f"✅ All available days included")
    print(f"✅ Plots saved to: {basedir}/Results/")
    print("="*80)

def load_and_plot_all_stimuli_cdf(basedir, feature_strategy=None, category_to_plot='All Cells', use_combined_days=False):
    """
    Load pickled decoding results for all stimuli and generate CDF plots for each.
    
    UPDATED FOR PER-NEURON SHUFFLE CORRECTION AND USER SELECTION:
    - Loads and plots raw_decoding_score vs per_neuron_shuffle
    - Uses appropriate statistical tests for the new data structure
    - User selects neuron category and layout
    
    Parameters:
    -----------
    basedir : str
        Base directory containing Results folder with pickle files
    feature_strategy : str, optional
        Specific feature strategy to load (e.g., 'means_slopes'). 
        If None, will auto-detect from available files.
    category_to_plot : str
        Which neuron category to plot
    use_combined_days : bool
        Whether to show all days on same plot or separate plots
    """
    
    print("="*80)
    print("LOADING PICKLED DECODING RESULTS AND GENERATING UPDATED CDF PLOTS")
    print("(Raw Decoding vs Per-Neuron Shuffle)")
    print("="*80)
    print(f"Category: {category_to_plot}")
    print(f"Layout: {'Combined days' if use_combined_days else 'Separate days'}")
    print("="*80)
    
    results_dir = os.path.join(basedir, 'Results')
    
    if not os.path.exists(results_dir):
        print(f"Error: Results directory not found at {results_dir}")
        return
    
    # Auto-detect feature strategy if not provided
    if feature_strategy is None:
        # Look for any configurable decoding files to determine feature strategy
        pattern = os.path.join(results_dir, 'configurable_decoding_day_*.pkl')
        config_files = glob.glob(pattern)
        
        if len(config_files) == 0:
            print(f"Error: No configurable decoding files found in {results_dir}")
            return
        
        # Extract feature strategy from first file
        filename = os.path.basename(config_files[0])
        print(f"Auto-detecting feature strategy from file: {filename}")
        
        # Remove the .pkl extension
        base_name = filename.replace('.pkl', '')
        
        # Parse the filename: configurable_decoding_day_{STIMULUS}_{FEATURE_STRATEGY}
        # Split by underscore and remove the first 3 parts (configurable_decoding_day)
        parts = base_name.split('_')
        
        if len(parts) < 4:
            print(f"Error: Unexpected filename format: {filename}")
            return
        
        # Find where the stimulus name ends and feature strategy begins
        # Known stimuli patterns (including compound ones)
        known_stimuli = [
            'CSplus_Lick', 'CSminus_Lick',  # Check compound stimuli first
            'CSplus', 'CSminus', 'Reward_Omission', 'Reward', 
            'Hits', 'Misses', 'False_Alarms', 'Correct_Rejections'
        ]
        
        # Remove the prefix 'configurable_decoding_day_'
        stimulus_and_strategy = '_'.join(parts[3:])
        
        feature_strategy = None
        for stimulus in known_stimuli:
            if stimulus_and_strategy.startswith(stimulus + '_'):
                # Found the stimulus, extract what comes after it
                feature_strategy = stimulus_and_strategy[len(stimulus) + 1:]
                break
        
        if not feature_strategy:
            print(f"Error: Could not determine feature strategy from filename: {filename}")
            print(f"Parsed parts: {parts}")
            print(f"Stimulus and strategy part: {stimulus_and_strategy}")
            return
        
        print(f"Auto-detected feature strategy: {feature_strategy}")
    
    # Define all possible stimuli and their safe names
    stimulus_mapping = {
        'CS+': 'CSplus',
        'CS-': 'CSminus', 
        'Reward': 'Reward',
        'Reward Omission': 'Reward_Omission',
        'CS+ Lick': 'CSplus_Lick',
        'CS- Lick': 'CSminus_Lick',
        'Hits': 'Hits',
        'Misses': 'Misses',
        'False_Alarms': 'False_Alarms',
        'Correct_Rejections': 'Correct_Rejections'
    }
    
    # First, check which files actually exist to avoid processing non-existent stimuli
    existing_stimuli = []
    for original_stimulus, safe_stimulus in stimulus_mapping.items():
        main_file = os.path.join(results_dir, f'configurable_decoding_day_{safe_stimulus}_{feature_strategy}.pkl')
        if os.path.exists(main_file):
            existing_stimuli.append((original_stimulus, safe_stimulus))
    
    print(f"\nFound {len(existing_stimuli)} stimuli with feature strategy '{feature_strategy}':")
    for orig, safe in existing_stimuli:
        print(f"  {orig} -> {safe}")
    
    if len(existing_stimuli) == 0:
        print(f"No stimulus files found with feature strategy '{feature_strategy}'")
        print("\nAvailable files:")
        for file in sorted(glob.glob(os.path.join(results_dir, 'configurable_decoding_day_*.pkl'))):
            print(f"  {os.path.basename(file)}")
        return
    
    successfully_plotted = []
    failed_stimuli = []
    
    for original_stimulus, safe_stimulus in existing_stimuli:
        print(f"\n{'='*60}")
        print(f"PROCESSING STIMULUS: {original_stimulus}")
        print(f"{'='*60}")
        
        # Construct file paths
        main_file = os.path.join(results_dir, f'configurable_decoding_day_{safe_stimulus}_{feature_strategy}.pkl')
        shuffled_file = os.path.join(results_dir, f'configurable_decoding_shuffled_day_{safe_stimulus}_{feature_strategy}.pkl')
        
        # Check if shuffled file exists (though we mainly use main file now)
        if not os.path.exists(shuffled_file):
            print(f"Shuffled decoding file not found: {shuffled_file}")
            print("Note: For per-neuron correction, we primarily use the main file.")
        
        try:
            # Load the main pickle file (contains both raw and per-neuron shuffle data)
            print(f"Loading {main_file}")
            with open(main_file, 'rb') as f:
                decoding_day_config = pickle.load(f)
            
            # Load shuffled file if it exists (for compatibility)
            decoding_shuffled_day_config = {}
            if os.path.exists(shuffled_file):
                print(f"Loading {shuffled_file}")
                with open(shuffled_file, 'rb') as f:
                    decoding_shuffled_day_config = pickle.load(f)
            else:
                print("Using per-neuron shuffle data from main file only")
            
            # Check if data was loaded successfully
            if len(decoding_day_config) == 0:
                print(f"No data found in decoding files for {original_stimulus}")
                failed_stimuli.append(original_stimulus)
                continue
            
            # Verify that the data has the required columns for per-neuron analysis
            sample_day = list(decoding_day_config.keys())[0]
            sample_data = decoding_day_config[sample_day]
            required_cols = ['raw_decoding_score', 'per_neuron_shuffle']
            missing_cols = [col for col in required_cols if col not in sample_data.columns]
            
            if missing_cols:
                print(f"⚠️  Missing required columns for per-neuron analysis: {missing_cols}")
                print("    This requires the updated Block 3 with per-neuron shuffle correction!")
                failed_stimuli.append(original_stimulus)
                continue
            
            print(f"Successfully loaded per-neuron data for {original_stimulus}")
            print(f"  Days available: {list(decoding_day_config.keys())}")
            
            # Print summary stats
            total_neurons = 0
            for day_name, day_data in decoding_day_config.items():
                day_neurons = len(day_data)
                total_neurons += day_neurons
                print(f"  {day_name}: {day_neurons} neurons")
            print(f"  Total neuron measurements: {total_neurons}")
            
            # Generate CDF plots for this stimulus
            print(f"\nGenerating updated CDF plots for {original_stimulus}...")
            
            try:
                create_comprehensive_cdf_plots_from_configurable_decoding(
                    decoding_day_config, decoding_shuffled_day_config, 
                    df_master, original_stimulus, basedir,
                    category_to_plot=category_to_plot, 
                    use_combined_days=use_combined_days
                )
                successfully_plotted.append(original_stimulus)
                print(f"✓ Successfully created comprehensive CDF plots for {original_stimulus}")
                
            except Exception as e:
                print(f"Error in comprehensive CDF plot for {original_stimulus}: {e}")
                print("Falling back to simple CDF comparison...")
                
                try:
                    create_simple_cdf_comparison(
                        decoding_day_config, decoding_shuffled_day_config, 
                        original_stimulus, basedir,
                        category_to_plot=category_to_plot,
                        use_combined_days=use_combined_days
                    )
                    successfully_plotted.append(original_stimulus)
                    print(f"✓ Successfully created simple CDF plots for {original_stimulus}")
                    
                except Exception as e2:
                    print(f"Error in simple CDF plot for {original_stimulus}: {e2}")
                    failed_stimuli.append(original_stimulus)
            
        except Exception as e:
            print(f"Error loading pickle files for {original_stimulus}: {e}")
            failed_stimuli.append(original_stimulus)
    
    # Final summary
    print(f"\n{'='*80}")
    print("UPDATED CDF PLOTTING SUMMARY")
    print(f"{'='*80}")
    print(f"Category plotted: {category_to_plot}")
    print(f"Layout: {'Combined days' if use_combined_days else 'Separate days'}")
    print(f"Feature strategy used: {feature_strategy}")
    print(f"Successfully plotted: {len(successfully_plotted)} stimuli")
    for stimulus in successfully_plotted:
        print(f"  ✓ {stimulus}")
    
    if failed_stimuli:
        print(f"\nFailed to plot: {len(failed_stimuli)} stimuli")
        for stimulus in failed_stimuli:
            print(f"  ✗ {stimulus}")
    
    print(f"\nPlots saved to: {basedir}/Results/")
    print("Files now show: Raw Decoding vs Per-Neuron Shuffle")
    print("="*80)

# ====================================================================
# MAIN EXECUTION - UPDATED TO USE USER SELECTION
# ====================================================================

# Validate user's stimulus selection
if stimulus_to_plot not in stimulus_list:
    print(f"❌ ERROR: Invalid stimulus selection '{stimulus_to_plot}'")
    print(f"Available stimuli: {stimulus_list}")
else:
    print(f"🎯 USER SELECTION: Plotting {stimulus_to_plot} only")
    print(f"   Category: {category_to_plot}")
    print(f"   Layout: {'Combined days' if use_combined_days else 'Separate days'}")
    print(f"   Score type: {'Raw vs shuffle' if use_raw_scores else 'Normalized'}")
    if days_to_include:
        print(f"   Days filter: {days_to_include}")
    print("="*60)
    
    # Use the new single-stimulus function
    load_and_plot_single_stimulus_cdf(
        basedir=basedir,
        stimulus_to_plot=stimulus_to_plot,  # Use the user's selection
        feature_strategy=None,  # Will auto-detect the feature strategy
        category_to_plot=category_to_plot,  # User-selected category
        use_combined_days=use_combined_days,  # User-selected layout
        use_raw_scores=use_raw_scores,  # User-selected score type
        days_to_include=days_to_include  # User-selected days filter
    )

# Uncomment the line below if you want to plot ALL stimuli instead of just one:
# load_and_plot_all_stimuli_cdf(
#     basedir=basedir,
#     feature_strategy=None,  # Will auto-detect
#     category_to_plot=category_to_plot,
#     use_combined_days=use_combined_days
# )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import os
import pickle
import glob
import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# USER CONFIGURATION - EDIT THESE VALUES
# ====================================================================

# Data selection
stimulus_to_analyze = 'CS+'  # 'CS+', 'CS-', 'Reward', 'Reward Omission', etc.
category_to_analyze = 'Any Significant'  # 'All neurons', 'Any CS significant', etc.
days_to_analyze = ['0_Early', '1_Late']  # [] = all days, or specify like ['0_Early', '1_Late']
###['0_Early', '1_Late'] or ['1_Late', '2_SucOmit', '3_CueOmit' ] or ['4_RevEarly, '5_RevLate']
# Analysis type
use_raw_scores = True  # True = raw vs shuffle analysis, False = normalized only

# Output options
create_plots = True  # True = generate plots, False = stats only
plot_type = 'bar'  # 'bar', 'violin', or 'both'

# ====================================================================

class DecodingAnalyzer:
    """Enhanced decoding analysis with MLM and omnibus F-tests."""
    
    def __init__(self, basedir):
        self.basedir = basedir
        self.valid_stimuli = ['CS+', 'CS-', 'Reward', 'Reward Omission', 'CS+ Lick', 'CS- Lick', 
                             'Hits', 'Misses', 'False_Alarms', 'Correct_Rejections']
        self.valid_categories = ['All neurons', 'Any CS significant', 'Any Significant', 'CS+ sig. only', 
                       'CS- sig. only', 'Both CS+&CS- sig.', 'Non-significant']
        self.stimulus_map = {
            'CS+': 'CSplus', 'CS-': 'CSminus', 'Reward': 'Reward', 
            'Reward Omission': 'Reward_Omission', 'CS+ Lick': 'CSplus_Lick', 
            'CS- Lick': 'CSminus_Lick', 'Hits': 'Hits', 'Misses': 'Misses',
            'False_Alarms': 'False_Alarms', 'Correct_Rejections': 'Correct_Rejections'
        }
    
    def validate_inputs(self, stimulus, category):
        """Validate user inputs."""
        assert stimulus in self.valid_stimuli, f"Invalid stimulus: {stimulus}"
        assert category in self.valid_categories, f"Invalid category: {category}"
        print(f"Valid selections - Stimulus: {stimulus}, Category: {category}")
    
    def load_data(self, stimulus_name):
        """Load pickled decoding data and extract feature strategy."""
        results_dir = os.path.join(self.basedir, 'Results')
        safe_stimulus = self.stimulus_map[stimulus_name]
        pattern = os.path.join(results_dir, f'configurable_decoding_day_{safe_stimulus}_*.pkl')
        files = glob.glob(pattern)
        
        if not files:
            raise FileNotFoundError(f"No files found for {stimulus_name}")
        
        # Extract feature strategy from filename
        filename = os.path.basename(files[0])
        base_name = filename.replace('.pkl', '').replace('configurable_decoding_day_', '')
        
        # Find which stimulus prefix matches to extract feature strategy
        for stim_key in sorted(self.stimulus_map.values(), key=len, reverse=True):
            if base_name.startswith(stim_key + '_'):
                feature_strategy = base_name[len(stim_key) + 1:]
                break
        else:
            feature_strategy = "unknown"
        
        # Load the data
        main_file = os.path.join(results_dir, f'configurable_decoding_day_{safe_stimulus}_{feature_strategy}.pkl')
        with open(main_file, 'rb') as f:
            decoding_day = pickle.load(f)
        
        print(f"Loaded {stimulus_name} data with strategy: {feature_strategy}")
        print(f"Available days: {sorted(decoding_day.keys())}")
        return decoding_day, feature_strategy
    
    def filter_neurons(self, day_data, category_name):
        """Apply neuron category filter."""
        day_data = day_data.drop_duplicates(subset=['cell_id'])
        
        if category_name == 'All neurons' or 'significant_plus' not in day_data.columns:
            return day_data
        
        filters = {
            'Any CS significant': (day_data['significant_plus'] == 1) | (day_data['significant_minus'] == 1),
            'Any Significant': (day_data['significant_plus'] == 1) | (day_data['significant_minus'] == 1),
            'CS+ sig. only': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 0),
            'CS- sig. only': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 1),
            'Both CS+&CS- sig.': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 1),
            'Non-significant': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 0)
        }
        
        return day_data[filters[category_name]]
    
    def prepare_analysis_data(self, decoding_day, category_name, days_list, use_raw_scores):
        """Prepare long-format dataframe for MLM analysis."""
        days_to_process = days_list if days_list else sorted(decoding_day.keys())
        analysis_rows = []
        
        for day in days_to_process:
            if day not in decoding_day:
                continue
            
            filtered_data = self.filter_neurons(decoding_day[day], category_name)
            
            if len(filtered_data) == 0:
                continue
            
            for _, row in filtered_data.iterrows():
                cell_id = row['cell_id']
                
                if use_raw_scores:
                    # Add both raw and shuffle scores for each neuron
                    if not (pd.isna(row['raw_decoding_score']) or pd.isna(row['per_neuron_shuffle'])):
                        analysis_rows.extend([
                            {'cell_id': cell_id, 'day': day, 'score_type': 'Raw', 
                             'decoding_score': row['raw_decoding_score']},
                            {'cell_id': cell_id, 'day': day, 'score_type': 'Shuffle', 
                             'decoding_score': row['per_neuron_shuffle']}
                        ])
                else:
                    # Add normalized scores only
                    if not pd.isna(row['decoding_score']):
                        analysis_rows.append({
                            'cell_id': cell_id, 'day': day, 'score_type': 'Normalized', 
                            'decoding_score': row['decoding_score']
                        })
        
        df_analysis = pd.DataFrame(analysis_rows)
        print(f"Analysis data: {len(df_analysis)} observations, {df_analysis['cell_id'].nunique()} neurons")
        return df_analysis
    
    def format_p_value(self, p_value):
        """Format p-values with appropriate precision."""
        if pd.isna(p_value):
            return "p=NaN"
        elif p_value < 0.0001:
            return "p<0.0001"
        elif p_value < 0.001:
            return f"p={p_value:.4f}"
        elif p_value < 0.01:
            return f"p={p_value:.3f}"
        else:
            return f"p={p_value:.3f}"
    
    def run_statistical_analysis(self, df_analysis, use_raw_scores):
        """Run enhanced MLM analysis with omnibus F-tests and detailed posthocs."""
        print("\nRunning Enhanced Statistical Analysis:")
        print("="*60)
        
        if len(df_analysis) == 0:
            print("No data available for analysis")
            return
        
        try:
            if use_raw_scores:
                # Full model with score type, day, and interaction
                formula = "decoding_score ~ C(score_type) + C(day) + C(score_type):C(day)"
                model = mixedlm(formula, df_analysis, groups=df_analysis['cell_id'])
                fitted_model = model.fit(method='lbfgs')
                
                print(f"MLM Results (n={fitted_model.nobs}, groups={len(df_analysis['cell_id'].unique())}):")
                print()
                
                # Get model parameters
                df_resid = fitted_model.df_resid
                
                # OMNIBUS F-TESTS
                print("OMNIBUS F-TESTS:")
                print("-" * 30)
                
                # 1. Test main effect of score type
                score_type_params = [param for param in fitted_model.params.index 
                                   if 'C(score_type)' in param and 'C(day)' not in param]
                if score_type_params:
                    hypothesis_matrix = np.zeros((len(score_type_params), len(fitted_model.params)))
                    for i, param in enumerate(score_type_params):
                        param_idx = list(fitted_model.params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    
                    f_test_result = fitted_model.f_test(hypothesis_matrix)
                    f_stat = f_test_result.fvalue
                    f_pval = f_test_result.pvalue
                    df_num = len(score_type_params)
                    print(f"Main Effect of Score Type: F({df_num}, {df_resid})={f_stat:.3f}, {self.format_p_value(f_pval)}")
                
                # 2. Test main effect of day
                day_params = [param for param in fitted_model.params.index 
                            if 'C(day)[T.' in param and 'C(score_type)' not in param]
                if day_params:
                    hypothesis_matrix = np.zeros((len(day_params), len(fitted_model.params)))
                    for i, param in enumerate(day_params):
                        param_idx = list(fitted_model.params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    
                    f_test_result = fitted_model.f_test(hypothesis_matrix)
                    f_stat = f_test_result.fvalue
                    f_pval = f_test_result.pvalue
                    df_num = len(day_params)
                    print(f"Main Effect of Day: F({df_num}, {df_resid})={f_stat:.3f}, {self.format_p_value(f_pval)}")
                
                # 3. Test interaction
                interaction_params = [param for param in fitted_model.params.index 
                                    if 'C(score_type)' in param and 'C(day)[T.' in param]
                if interaction_params:
                    hypothesis_matrix = np.zeros((len(interaction_params), len(fitted_model.params)))
                    for i, param in enumerate(interaction_params):
                        param_idx = list(fitted_model.params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    
                    f_test_result = fitted_model.f_test(hypothesis_matrix)
                    f_stat = f_test_result.fvalue
                    f_pval = f_test_result.pvalue
                    df_num = len(interaction_params)
                    print(f"Score Type x Day Interaction: F({df_num}, {df_resid})={f_stat:.3f}, {self.format_p_value(f_pval)}")
                
                print()
                
                # INDIVIDUAL PARAMETER TESTS
                print("INDIVIDUAL PARAMETER TESTS:")
                print("-" * 35)
                
                # Score type effect
                if 'C(score_type)[T.Shuffle]' in fitted_model.params.index:
                    t_stat = fitted_model.tvalues['C(score_type)[T.Shuffle]']
                    p_val = fitted_model.pvalues['C(score_type)[T.Shuffle]']
                    print(f"Raw vs Shuffle: t({df_resid})={t_stat:.3f}, {self.format_p_value(p_val)}")
                
                # Day effects (vs first day)
                for param in day_params:
                    day_name = param.replace('C(day)[T.', '').replace(']', '')
                    t_stat = fitted_model.tvalues[param]
                    p_val = fitted_model.pvalues[param]
                    print(f"{day_name} vs baseline: t({df_resid})={t_stat:.3f}, {self.format_p_value(p_val)}")
                
                # Interaction effects
                if interaction_params:
                    print("Interaction contrasts:")
                    for param in interaction_params:
                        int_name = param.replace('C(score_type)[T.Shuffle]:C(day)[T.', '').replace(']', '')
                        t_stat = fitted_model.tvalues[param]
                        p_val = fitted_model.pvalues[param]
                        print(f"  Shuffle x {int_name}: t({df_resid})={t_stat:.3f}, {self.format_p_value(p_val)}")
                
                # POSTHOC TESTS
                self._run_posthoc_tests(df_analysis, use_raw_scores=True)
            
            else:
                # Normalized scores only - test day effects
                formula = "decoding_score ~ C(day)"
                model = mixedlm(formula, df_analysis, groups=df_analysis['cell_id'])
                fitted_model = model.fit(method='lbfgs')
                
                print(f"MLM Results (n={fitted_model.nobs}, groups={len(df_analysis['cell_id'].unique())}):")
                print()
                
                df_resid = fitted_model.df_resid
                
                # OMNIBUS F-TEST for day effect
                day_params = [param for param in fitted_model.params.index if 'C(day)[T.' in param]
                if day_params:
                    print("OMNIBUS F-TEST:")
                    print("-" * 20)
                    
                    hypothesis_matrix = np.zeros((len(day_params), len(fitted_model.params)))
                    for i, param in enumerate(day_params):
                        param_idx = list(fitted_model.params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    
                    f_test_result = fitted_model.f_test(hypothesis_matrix)
                    f_stat = f_test_result.fvalue
                    f_pval = f_test_result.pvalue
                    df_num = len(day_params)
                    print(f"Main Effect of Day: F({df_num}, {df_resid})={f_stat:.3f}, {self.format_p_value(f_pval)}")
                    print()
                
                # INDIVIDUAL PARAMETER TESTS
                print("INDIVIDUAL PARAMETER TESTS:")
                print("-" * 35)
                for param in day_params:
                    day_name = param.replace('C(day)[T.', '').replace(']', '')
                    t_stat = fitted_model.tvalues[param]
                    p_val = fitted_model.pvalues[param]
                    print(f"{day_name} vs baseline: t({df_resid})={t_stat:.3f}, {self.format_p_value(p_val)}")
                
                # POSTHOC TESTS
                self._run_posthoc_tests(df_analysis, use_raw_scores=False)
        
        except Exception as e:
            print(f"Error in statistical analysis: {str(e)}")
            import traceback
            traceback.print_exc()
    
    def _run_posthoc_tests(self, df_analysis, use_raw_scores):
        """Run posthoc pairwise comparisons with p-values."""
        print()
        print("POSTHOC PAIRWISE COMPARISONS:")
        print("-" * 40)
        
        try:
            if use_raw_scores:
                # Test within each score type
                for score_type in ['Raw', 'Shuffle']:
                    subset_data = df_analysis[df_analysis['score_type'] == score_type]
                    unique_days = subset_data['day'].unique()
                    
                    if len(unique_days) > 1:
                        tukey_result = pairwise_tukeyhsd(
                            subset_data['decoding_score'], 
                            subset_data['day']
                        )
                        
                        # Extract significant pairs with p-values
                        summary_data = tukey_result.summary().data[1:]  # Skip header
                        sig_pairs = []
                        
                        for row in summary_data:
                            group1, group2, meandiff, p_adj, lower, upper, reject = row
                            if reject:  # Significant difference
                                sig_pairs.append((group1, group2, p_adj))
                        
                        if sig_pairs:
                            print(f"{score_type} scores - significant pairs:")
                            for g1, g2, p_val in sig_pairs:
                                print(f"  {g1} vs {g2}: {self.format_p_value(p_val)}")
                        else:
                            print(f"{score_type} scores - no significant pairs")
            else:
                # Test across days for normalized scores
                unique_days = df_analysis['day'].unique()
                if len(unique_days) > 1:
                    tukey_result = pairwise_tukeyhsd(
                        df_analysis['decoding_score'], 
                        df_analysis['day']
                    )
                    
                    # Extract significant pairs with p-values
                    summary_data = tukey_result.summary().data[1:]  # Skip header
                    sig_pairs = []
                    
                    for row in summary_data:
                        group1, group2, meandiff, p_adj, lower, upper, reject = row
                        if reject:  # Significant difference
                            sig_pairs.append((group1, group2, p_adj))
                    
                    if sig_pairs:
                        print("Normalized scores - significant pairs:")
                        for g1, g2, p_val in sig_pairs:
                            print(f"  {g1} vs {g2}: {self.format_p_value(p_val)}")
                    else:
                        print("Normalized scores - no significant pairs")
                else:
                    print("Insufficient days for pairwise comparisons")
        
        except Exception as e:
            print(f"Error in posthoc tests: {str(e)}")
    
    def create_plots(self, decoding_day, stimulus_name, category_name, days_list, 
                    feature_strategy, use_raw_scores, plot_type):
        """Create enhanced visualization plots."""
        print()
        print("Creating Plots:")
        print("="*20)
        
        days_to_plot = days_list if days_list else sorted(decoding_day.keys())
        plot_data = self._prepare_plot_data(decoding_day, days_to_plot, category_name, use_raw_scores)
        
        if not plot_data:
            print("No valid plot data found")
            return
        
        if plot_type in ['bar', 'both']:
            self._create_bar_plot(plot_data, stimulus_name, category_name, 
                                use_raw_scores, feature_strategy)
        
        if plot_type in ['violin', 'both']:
            self._create_violin_plot(plot_data, stimulus_name, category_name, 
                                   use_raw_scores, feature_strategy)
    
    def _prepare_plot_data(self, decoding_day, days_to_plot, category_name, use_raw_scores):
        """Prepare data for plotting."""
        plot_data = {}
        
        for day in days_to_plot:
            if day not in decoding_day:
                continue
            
            filtered_data = self.filter_neurons(decoding_day[day], category_name)
            if len(filtered_data) == 0:
                continue
            
            if use_raw_scores:
                raw_scores = filtered_data['raw_decoding_score'].dropna()
                shuffle_scores = filtered_data['per_neuron_shuffle'].dropna()
                
                # Match lengths for paired comparisons
                min_len = min(len(raw_scores), len(shuffle_scores))
                if min_len > 0:
                    plot_data[day] = {
                        'raw': raw_scores.iloc[:min_len],
                        'shuffle': shuffle_scores.iloc[:min_len]
                    }
            else:
                norm_scores = filtered_data['decoding_score'].dropna()
                if len(norm_scores) > 0:
                    plot_data[day] = {'normalized': norm_scores}
        
        return plot_data
    
    def _create_bar_plot(self, plot_data, stimulus_name, category_name, use_raw_scores, feature_strategy):
        """Create enhanced bar plot with significance testing."""
        fig, ax = plt.subplots(figsize=(10, 7))
        
        days = list(plot_data.keys())
        x = np.arange(len(days))
        width = 0.35
        
        if use_raw_scores:
            # Calculate means and SEMs
            raw_means = [np.mean(plot_data[day]['raw']) for day in days]
            raw_sems = [stats.sem(plot_data[day]['raw']) for day in days]
            shuffle_means = [np.mean(plot_data[day]['shuffle']) for day in days]
            shuffle_sems = [stats.sem(plot_data[day]['shuffle']) for day in days]
            
            # Create bars
            bars1 = ax.bar(x - width/2, raw_means, width, yerr=raw_sems, 
                          label='Raw', color='steelblue', alpha=0.8, capsize=4)
            bars2 = ax.bar(x + width/2, shuffle_means, width, yerr=shuffle_sems, 
                          label='Shuffle', color='lightgray', alpha=0.8, capsize=4)
            
            # Add chance line
            ax.axhline(0.5, color='black', linestyle='--', alpha=0.6, linewidth=1, label='Chance (0.5)')
            
            # Add significance stars for each day
            for i, day in enumerate(days):
                raw_data = plot_data[day]['raw']
                shuffle_data = plot_data[day]['shuffle']
                
                # Paired t-test
                _, p_val = stats.ttest_rel(raw_data, shuffle_data)
                sig_symbol = ('***' if p_val < 0.001 else 
                             '**' if p_val < 0.01 else 
                             '*' if p_val < 0.05 else 'ns')
                
                # Position star above the higher bar
                y_pos = max(raw_means[i] + raw_sems[i], shuffle_means[i] + shuffle_sems[i]) + 0.03
                ax.text(i, y_pos, sig_symbol, ha='center', fontweight='bold', fontsize=12)
            
            ylabel = "Decoding Score"
            title_suffix = "Raw vs Shuffle"
            
        else:
            # Normalized scores only
            norm_means = [np.mean(plot_data[day]['normalized']) for day in days]
            norm_sems = [stats.sem(plot_data[day]['normalized']) for day in days]
            
            bars = ax.bar(x, norm_means, width*1.5, yerr=norm_sems, 
                         label='Normalized', color='steelblue', alpha=0.8, capsize=4)
            
            # Add chance line at zero
            ax.axhline(0, color='black', linestyle='--', alpha=0.6, linewidth=1, label='Chance (0)')
            
            # Add significance stars (one-sample t-test vs 0)
            for i, day in enumerate(days):
                norm_data = plot_data[day]['normalized']
                _, p_val = stats.ttest_1samp(norm_data, 0)
                sig_symbol = ('***' if p_val < 0.001 else 
                             '**' if p_val < 0.01 else 
                             '*' if p_val < 0.05 else 'ns')
                
                y_pos = norm_means[i] + norm_sems[i] + 0.03
                ax.text(i, y_pos, sig_symbol, ha='center', fontweight='bold', fontsize=12)
            
            ylabel = "Normalized Decoding Score"
            title_suffix = "Normalized"
        
        # Formatting
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{stimulus_name} - {category_name} - {title_suffix}', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(days, rotation=45, ha='right')
        ax.legend(frameon=True, fancybox=True, shadow=True)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        
        # Save plot
        # safe_names = [name.replace(' ', '_').replace('+', 'plus').replace('-', 'minus') 
        #               for name in [stimulus_name, category_name]]
        # suffix = "Raw" if use_raw_scores else "Normalized"
        # filename = f"{self.basedir}/Results/{safe_names[0]}_{safe_names[1]}_{feature_strategy}_{suffix}_Bar.pdf"
        # os.makedirs(os.path.dirname(filename), exist_ok=True)
        # fig.savefig(filename, bbox_inches='tight', dpi=300)
        # plt.show()
        # print(f"Saved bar plot: {filename}")
    
    def _create_violin_plot(self, plot_data, stimulus_name, category_name, use_raw_scores, feature_strategy):
        """Create enhanced violin plot with distributions."""
        fig, ax = plt.subplots(figsize=(12, 8))
        
        days = list(plot_data.keys())
        
        if use_raw_scores:
            # Prepare data for violin plots
            all_data = []
            positions = []
            colors = []
            labels = []
            
            pos_counter = 0
            for day in days:
                all_data.extend([plot_data[day]['raw'], plot_data[day]['shuffle']])
                positions.extend([pos_counter, pos_counter + 1])
                colors.extend(['steelblue', 'lightgray'])
                labels.extend([f'{day}_Raw', f'{day}_Shuffle'])
                pos_counter += 3
            
            # Create violin plots
            violin_parts = ax.violinplot(all_data, positions=positions, widths=0.8, showmeans=True)
            
            # Color the violin parts
            for i, pc in enumerate(violin_parts['bodies']):
                pc.set_facecolor(colors[i])
                pc.set_alpha(0.7)
            
            # Add boxplots for quartiles
            ax.boxplot(all_data, positions=positions, widths=0.2, patch_artist=False)
            
            # Add chance line
            ax.axhline(0.5, color='black', linestyle='--', alpha=0.6, linewidth=1, label='Chance (0.5)')
            
            # Set x-tick positions and labels
            day_positions = [positions[i] + 0.5 for i in range(0, len(positions), 2)]
            ax.set_xticks(day_positions)
            ax.set_xticklabels(days, rotation=45, ha='right')
            
            ylabel = "Decoding Score"
            title_suffix = "Raw vs Shuffle Distribution"
            
        else:
            # Normalized scores only
            all_data = [plot_data[day]['normalized'] for day in days]
            positions = list(range(len(days)))
            
            violin_parts = ax.violinplot(all_data, positions=positions, widths=0.8, showmeans=True)
            
            # Color all violins the same
            for pc in violin_parts['bodies']:
                pc.set_facecolor('steelblue')
                pc.set_alpha(0.7)
            
            # Add boxplots
            ax.boxplot(all_data, positions=positions, widths=0.2, patch_artist=False)
            
            # Add chance line at zero
            ax.axhline(0, color='black', linestyle='--', alpha=0.6, linewidth=1, label='Chance (0)')
            
            ax.set_xticks(positions)
            ax.set_xticklabels(days, rotation=45, ha='right')
            
            ylabel = "Normalized Decoding Score"
            title_suffix = "Normalized Distribution"
        
        # Formatting
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{stimulus_name} - {category_name} - {title_suffix}', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        
        # Save plot
        safe_names = [name.replace(' ', '_').replace('+', 'plus').replace('-', 'minus') 
                      for name in [stimulus_name, category_name]]
        suffix = "Raw" if use_raw_scores else "Normalized"
        filename = f"{self.basedir}/Results/{safe_names[0]}_{safe_names[1]}_{feature_strategy}_{suffix}_Violin.pdf"
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        fig.savefig(filename, bbox_inches='tight', dpi=300)
        plt.show()
        print(f"Saved violin plot: {filename}")

# ====================================================================
# MAIN EXECUTION
# ====================================================================

def main():
    """Main execution function."""
    print("="*60)
    print("ENHANCED DECODING ANALYSIS WITH MLM F-TESTS")
    print("="*60)
    print(f"Stimulus: {stimulus_to_analyze}")
    print(f"Category: {category_to_analyze}")
    print(f"Days: {days_to_analyze if days_to_analyze else 'All available'}")
    print(f"Analysis: {'Raw vs Shuffle' if use_raw_scores else 'Normalized only'}")
    print(f"Plots: {plot_type if create_plots else 'None'}")
    print("="*60)
    
    try:
        # Initialize analyzer
        analyzer = DecodingAnalyzer(basedir)
        
        # Validate inputs and load data
        analyzer.validate_inputs(stimulus_to_analyze, category_to_analyze)
        decoding_day, feature_strategy = analyzer.load_data(stimulus_to_analyze)
        
        # Prepare analysis dataframe
        df_analysis = analyzer.prepare_analysis_data(
            decoding_day, category_to_analyze, days_to_analyze, use_raw_scores
        )
        
        if len(df_analysis) == 0:
            print("No valid data found for analysis")
            return
        
        # Run enhanced statistical analysis
        analyzer.run_statistical_analysis(df_analysis, use_raw_scores)
        
        # Create plots if requested
        if create_plots:
            analyzer.create_plots(
                decoding_day, stimulus_to_analyze, category_to_analyze, 
                days_to_analyze, feature_strategy, use_raw_scores, plot_type
            )
        
        print("\n" + "="*60)
        print("ANALYSIS COMPLETE!")
        print("="*60)
    
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

# Execute main function
if __name__ == "__main__":
    main()

# Showing All Decoding_Stimulus In One HeatMap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
import os
import pickle
import glob

##THIS IS SO WE CAN HAVE THE SPATIAL PLOTS IN NICER COLORS
import numpy as np; np.random.seed(0)
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
def create_color(r, g, b):
    return [r/256, g/256, b/256]

# Colors for Magenta-Pink to Green PiYG_r gradient
colors_tiedye = [
    '#4B0082',   # Indigo/Deep Purple (highest - 1.0)
    '#9932CC',  # Dark Orchid (medium purple)
    '#2A788E',  # Teal (lowest - 0.0)
    '#74D054',  # Yellow-green
    '#FDE725',  # Yellow
    '#FFA500',  # Orange
    '#FF4500',  # Orange-red
    '#FF1493'  # Deep Pink
]

data = np.random.uniform(-1, 1, (10, 15))
sns.set_style('white')
# Create the colormap
rainbow_cmap = LinearSegmentedColormap.from_list('Magenta_tyedye', colors_tiedye)
# ====================================================================
# USER CONFIGURATION - EDIT THESE VALUES
# ====================================================================

# Define the stimuli you want to compare
stimulus_list = [
    'CS+',              
    'CS-',              
    'Reward', 
    'Reward Omission',
    'CS+ Lick',         
    'CS- Lick',         
    'Hits', 
    'Misses', 
    'Correct_Rejections', 
    'False_Alarms'
]

# Choose which days to include (empty list = all available days)
days_to_include = ['0_Early', '1_Late']  # Use [] for all days, or specify like ['1_Late', '2_SucOmit', '3_CueOmit']
###['0_Early', '1_Late'] or ['1_Late', '2_SucOmit', '3_CueOmit' ] or ['4_RevEarly, '5_RevLate']

# Choose whether to show raw scores or normalized (per-neuron corrected) scores
use_raw_scores = False      # True = raw_decoding_score (0-1 range)
                           # False = decoding_score (normalized, centered on 0)

# Choose which category to plot
category_to_plot = 'Any Significant'  # Options: 'All Cells', 'Any Significant', 
                                 #          'CS+ Only Significant', 'CS- Only Significant',
                                 #          'Both CS+&CS- Significant', 'Non-significant'

# Visualization settings
font_size = 18             # Font size for heatmap annotations
block_outline = True       # Add black outlines around heatmap cells
outline_thickness = 1      # Thickness of cell outlines

# ====================================================================

def detect_all_feature_strategies(basedir):
    """
    Detect all available feature strategies and create a mapping for each stimulus.
    Auto-loads pickled data to determine what's available.
    """
    results_dir = os.path.join(basedir, 'Results')
    
    # Map stimulus names to safe filenames
    stimulus_mapping = {
        'CS+': 'CSplus',
        'CS-': 'CSminus', 
        'Reward': 'Reward',
        'Reward Omission': 'Reward_Omission',
        'CS+ Lick': 'CSplus_Lick',
        'CS- Lick': 'CSminus_Lick',
        'Hits': 'Hits',
        'Misses': 'Misses',
        'False_Alarms': 'False_Alarms',
        'Correct_Rejections': 'Correct_Rejections'
    }
    
    try:
        # Get all configurable decoding files
        pattern = os.path.join(results_dir, 'configurable_decoding_day_*.pkl')
        all_files = glob.glob(pattern)
        
        if not all_files:
            print("❌ No configurable decoding files found")
            return {}
        
        print(f"📁 Found {len(all_files)} configurable decoding files")
        
        # Create mapping of stimulus -> feature_strategy
        stimulus_strategies = {}
        
        # Known safe stimuli in order (longest first to avoid partial matches)
        known_safe_stimuli = [
            'CSplus_Lick', 'CSminus_Lick', 'Reward_Omission',  # Compound stimuli first
            'False_Alarms', 'Correct_Rejections',              # Multi-word stimuli  
            'CSplus', 'CSminus', 'Reward', 'Hits', 'Misses'    # Simple stimuli
        ]
        
        for file_path in all_files:
            filename = os.path.basename(file_path)
            base_name = filename.replace('.pkl', '')
            
            # Remove prefix
            prefix = 'configurable_decoding_day_'
            if base_name.startswith(prefix):
                remainder = base_name[len(prefix):]
                
                # Try to match against known stimuli
                for known_stimulus in known_safe_stimuli:
                    if remainder.startswith(known_stimulus + '_'):
                        feature_strategy = remainder[len(known_stimulus) + 1:]
                        
                        # Map back to original stimulus name
                        original_stimulus = None
                        for orig, safe in stimulus_mapping.items():
                            if safe == known_stimulus:
                                original_stimulus = orig
                                break
                        
                        if original_stimulus:
                            stimulus_strategies[original_stimulus] = feature_strategy
                            print(f"  📋 {original_stimulus} -> {feature_strategy}")
                        break
        
        if not stimulus_strategies:
            print("❌ Could not extract feature strategies from any files")
            return {}
        
        print(f"✅ Successfully mapped {len(stimulus_strategies)} stimuli to feature strategies")
        return stimulus_strategies
        
    except Exception as e:
        print(f"❌ Error detecting feature strategies: {str(e)}")
        return {}

def load_configurable_decoding_data_flexible(decoding_stimuli, basedir, stimulus_strategies):
    """
    Load decoding data using flexible feature strategy mapping per stimulus.
    Automatically loads pickled data with per-neuron shuffle correction support.
    
    Parameters:
    -----------
    decoding_stimuli : list
        List of stimulus names to load
    basedir : str
        Base directory containing Results folder
    stimulus_strategies : dict
        Mapping of stimulus -> feature_strategy
    
    Returns:
    --------
    decoding_day_data : dict
        Main decoding results for each stimulus (contains raw_decoding_score, decoding_score, per_neuron_shuffle)
    """
    
    # Map stimulus names to safe filenames
    stimulus_mapping = {
        'CS+': 'CSplus',
        'CS-': 'CSminus', 
        'Reward': 'Reward',
        'Reward Omission': 'Reward_Omission',
        'CS+ Lick': 'CSplus_Lick',
        'CS- Lick': 'CSminus_Lick',
        'Hits': 'Hits',
        'Misses': 'Misses',
        'False_Alarms': 'False_Alarms',
        'Correct_Rejections': 'Correct_Rejections'
    }
    
    decoding_day_data = {}
    
    for stimulus in decoding_stimuli:
        # Check if we have a strategy for this stimulus
        if stimulus not in stimulus_strategies:
            print(f"❌ No feature strategy found for '{stimulus}' - skipping")
            continue
        
        feature_strategy = stimulus_strategies[stimulus]
        safe_stimulus_name = stimulus_mapping.get(stimulus, stimulus)
        
        # Construct file path (we only need the main file for per-neuron correction)
        day_path = os.path.join(basedir, 'Results', f'configurable_decoding_day_{safe_stimulus_name}_{feature_strategy}.pkl')
        
        try:
            # Load main decoding data (contains all needed columns)
            with open(day_path, 'rb') as f:
                decoding_day_data[stimulus] = pickle.load(f)
            
            print(f"✅ Loaded data for '{stimulus}' using strategy '{feature_strategy}'")
            print(f"   Days available: {list(decoding_day_data[stimulus].keys())}")
            
            # Verify required columns are present
            sample_day = list(decoding_day_data[stimulus].keys())[0]
            sample_data = decoding_day_data[stimulus][sample_day]
            required_cols = ['raw_decoding_score', 'decoding_score', 'per_neuron_shuffle']
            missing_cols = [col for col in required_cols if col not in sample_data.columns]
            
            if missing_cols:
                print(f"⚠️  Warning: Missing columns for '{stimulus}': {missing_cols}")
                print(f"   This requires updated Block 3 with per-neuron shuffle correction!")
            else:
                print(f"   ✅ All required columns present")
            
        except FileNotFoundError as e:
            print(f"❌ File not found for '{stimulus}' with strategy '{feature_strategy}':")
            print(f"   Expected: {os.path.basename(day_path)}")
        
        except Exception as e:
            print(f"❌ Error loading data for '{stimulus}': {str(e)}")
                
    return decoding_day_data

def filter_days_in_data(decoding_day_data, days_to_include):
    """
    Filter the loaded data to include only specified days.
    """
    
    if not days_to_include:
        # Use all available days from first stimulus
        if decoding_day_data:
            first_stimulus = list(decoding_day_data.keys())[0]
            valid_days = sorted(decoding_day_data[first_stimulus].keys())
            print(f"Using all available days: {valid_days}")
            return decoding_day_data, valid_days
        else:
            return {}, []
    
    # Filter to only requested days
    filtered_decoding_day_data = {}
    
    # Find which days are actually available across all stimuli
    available_days_all = None
    for stimulus, day_data in decoding_day_data.items():
        stimulus_days = set(day_data.keys())
        if available_days_all is None:
            available_days_all = stimulus_days
        else:
            available_days_all = available_days_all.intersection(stimulus_days)
    
    if available_days_all is None:
        available_days_all = set()
    
    # Filter to requested days that exist
    valid_days = [day for day in days_to_include if day in available_days_all]
    missing_days = [day for day in days_to_include if day not in available_days_all]
    
    if missing_days:
        print(f"Warning: Requested days not found in data: {missing_days}")
        print(f"Available days across all stimuli: {sorted(available_days_all)}")
    
    if not valid_days:
        print(f"❌ None of the requested days {days_to_include} were found")
        return {}, []
    
    print(f"Using filtered days: {valid_days}")
    
    # Create filtered dictionaries
    for stimulus in decoding_day_data.keys():
        filtered_decoding_day_data[stimulus] = {day: decoding_day_data[stimulus][day] for day in valid_days if day in decoding_day_data[stimulus]}
    
    return filtered_decoding_day_data, sorted(valid_days)

def create_multi_stimulus_heatmaps_updated(decoding_stimuli, basedir, stimulus_strategies,
                                         days_to_include=None, use_raw_scores=True,
                                         category_to_plot='All Cells',
                                         font_size=18, block_outline=True, outline_thickness=1):
    """
    Create heatmaps comparing multiple stimuli using updated per-neuron shuffle correction approach.
    
    Parameters:
    -----------
    use_raw_scores : bool
        True = show raw_decoding_score (0-1 range, compare to 0.5 chance)
        False = show decoding_score (normalized per-neuron corrected, compare to 0)
    """
    
    print(f"Loading configurable decoding data for {len(decoding_stimuli)} stimuli...")
    print(f"Using flexible feature strategy mapping")
    print(f"Score type: {'Raw scores (0-1 range)' if use_raw_scores else 'Normalized scores (per-neuron corrected)'}")
    
    # Load data for each stimulus using its specific feature strategy
    decoding_day_data = load_configurable_decoding_data_flexible(
        decoding_stimuli, basedir, stimulus_strategies)
    
    if not decoding_day_data:
        print("❌ No decoding data loaded. Check your stimulus names and feature strategies.")
        return
    
    # Filter days if specified
    if days_to_include is None:
        days_to_include = []
    
    filtered_day_data, valid_days = filter_days_in_data(decoding_day_data, days_to_include)
    
    if not valid_days:
        print("❌ No valid days found for processing.")
        return
    
    # Check what columns are available in the loaded data
    sample_stimulus = list(filtered_day_data.keys())[0]
    sample_day = list(filtered_day_data[sample_stimulus].keys())[0]
    sample_data = filtered_day_data[sample_stimulus][sample_day]
    available_columns = sample_data.columns.tolist()
    
    print(f"\nAvailable columns in loaded data: {len(available_columns)} total")
    
    # Check for required columns
    has_significance_data = ('significant_plus' in available_columns and 
                           'significant_minus' in available_columns)
    has_raw_scores = ('raw_decoding_score' in available_columns)
    has_normalized_scores = ('decoding_score' in available_columns)
    has_per_neuron_shuffle = ('per_neuron_shuffle' in available_columns)
    
    print(f"Has significance data: {has_significance_data}")
    print(f"Has raw scores: {has_raw_scores}")
    print(f"Has normalized scores: {has_normalized_scores}")
    print(f"Has per-neuron shuffle: {has_per_neuron_shuffle}")
    
    # Determine which score column to use
    if use_raw_scores:
        if has_raw_scores:
            score_column = 'raw_decoding_score'
            score_type = 'Raw'
            score_range = (0.0, 1.0)
            chance_level = 0.5
            print(f"✅ Using raw scores: {score_column}")
        else:
            print("❌ Raw scores requested but 'raw_decoding_score' column not available!")
            print("   This requires updated Block 3 with per-neuron shuffle correction.")
            return
    else:
        if has_normalized_scores:
            score_column = 'decoding_score'
            score_type = 'Normalized'
            score_range = None  # Will be determined empirically
            chance_level = 0.0
            print(f"✅ Using normalized scores: {score_column}")
        else:
            print("❌ Normalized scores requested but 'decoding_score' column not available!")
            return
    
    # Define all available categories
    all_categories = {
        "All Cells": lambda df: df,
        "Any Significant": lambda df: df[
            (df['significant_plus'] == 1) | 
            (df['significant_minus'] == 1)
        ],
        "CS+ Only Significant": lambda df: df[
            (df['significant_plus'] == 1) & 
            (df['significant_minus'] == 0)
        ],
        "CS- Only Significant": lambda df: df[
            (df['significant_plus'] == 0) & 
            (df['significant_minus'] == 1)
        ],
        "Both CS+&CS- Significant": lambda df: df[
            (df['significant_plus'] == 1) & 
            (df['significant_minus'] == 1)
        ],
        "Non-significant": lambda df: df[
            (df['significant_plus'] == 0) & 
            (df['significant_minus'] == 0)
        ]
    }
    
    # Determine which categories are available based on data
    if has_significance_data:
        available_categories = all_categories
        print(f"✅ Significance data available - all categories accessible")
    else:
        available_categories = {"All Cells": all_categories["All Cells"]}
        print("⚠️  No significance data available - only 'All Cells' category accessible")
    
    # Validate user's category selection
    if category_to_plot not in available_categories:
        print(f"❌ Requested category '{category_to_plot}' not available!")
        print(f"   Available categories: {list(available_categories.keys())}")
        if not has_significance_data and category_to_plot != "All Cells":
            print("   Note: Significance-based categories require significance columns in your data.")
        return
    
    print(f"✅ Generating heatmap for category: '{category_to_plot}'")
    
    # Get the selected category function
    category_func = available_categories[category_to_plot]
    
    def get_text_color(val, vmin, vmax):
        """Determine text color based on value and color scale."""
        if np.isnan(val):
            return 'black'
        if vmax == vmin:
            return 'black'
        norm = (val - vmin) / (vmax - vmin)
        return 'white' if norm > 0.5 else 'black'
    
    # Generate heatmap for the selected category
    print(f"\nGenerating {score_type} heatmap for {category_to_plot}...")
    
    # Initialize heatmap array
    heatmap_array = np.full((len(decoding_stimuli), len(valid_days)), np.nan)
    
    for s, stimulus in enumerate(decoding_stimuli):
        if stimulus not in filtered_day_data:
            print(f"⚠️  Warning: No data found for stimulus '{stimulus}'")
            continue
            
        decoding_day = filtered_day_data[stimulus]
        
        for d, day in enumerate(valid_days):
            if day not in decoding_day:
                print(f"⚠️  Warning: Day '{day}' not found for stimulus '{stimulus}'")
                continue
            
            try:
                day_data = decoding_day[day].copy()
                
                # Remove duplicates based on cell_id
                day_data = day_data.drop_duplicates(subset=['cell_id'])
                
                # Apply category filter
                try:
                    category_data = category_func(day_data)
                    if len(category_data) == 0:
                        print(f"  {stimulus} - {day}: No data after category filter")
                        continue
                    
                except KeyError as e:
                    if category_to_plot != "All Cells":
                        print(f"⚠️  Significance columns missing for {stimulus}, {day}. Skipping category.")
                        continue
                    category_data = day_data
                
                # Get the decoding scores
                decoding_scores = category_data[score_column].dropna()
                
                if len(decoding_scores) > 0:
                    heatmap_array[s, d] = np.nanmean(decoding_scores)
                    print(f"  {stimulus} - {day}: n={len(decoding_scores)}, mean={heatmap_array[s, d]:.3f}")
                else:
                    print(f"  {stimulus} - {day}: No valid scores after dropping NaNs")
                
            except Exception as e:
                print(f"❌ Error processing {stimulus} - {day}: {str(e)}")
                continue
    
    # Set color scale based on score type
    if use_raw_scores:
        # For raw scores, use fixed 0-1 range
        heatcmin, heatcmax = 0.0, 1.0
        print(f"Using fixed range for raw scores: [{heatcmin}, {heatcmax}]")
    else:
        # For normalized scores, calculate empirical range
        valid_data = heatmap_array[~np.isnan(heatmap_array)]
        if len(valid_data) > 0:
            heatcmin = np.percentile(valid_data, 5)
            heatcmax = np.percentile(valid_data, 95)
            # Ensure range is symmetric around zero for normalized scores
            max_abs = max(abs(heatcmin), abs(heatcmax))
            heatcmin = -max_abs
            heatcmax = max_abs
            print(f"Using symmetric range for normalized scores: [{heatcmin:.3f}, {heatcmax:.3f}]")
        else:
            heatcmin, heatcmax = -0.2, 0.2
            print("No valid data found, using default symmetric range")
    
    # Create the heatmap
    plt.figure(figsize=(max(14, len(valid_days) * 2), max(8, len(decoding_stimuli) * 0.8)))
    ax = plt.gca()
    
    # Choose appropriate colormap
    if use_raw_scores:
        cmap = 'PiYG_r'  # Good for 0-1 range
    else:
        cmap = rainbow_cmap   # Good for symmetric range around zero
    
    # Handle empty data case
    if np.all(np.isnan(heatmap_array)):
        print("⚠️  All heatmap values are NaN")
        empty_array = np.zeros_like(heatmap_array)
        im = sns.heatmap(empty_array, vmin=heatcmin, vmax=heatcmax, cmap=cmap,
                       cbar_kws={'label': f'{score_type} Decoding Score'},
                       xticklabels=valid_days, yticklabels=decoding_stimuli,
                       annot=False, fmt='', ax=ax,
                       linewidths=outline_thickness if block_outline else 0,
                       linecolor='black' if block_outline else 'none')
    else:
        im = sns.heatmap(heatmap_array, vmin=heatcmin, vmax=heatcmax, cmap=cmap, 
                       cbar_kws={'label': f'{score_type} Decoding Score'},
                       xticklabels=valid_days, yticklabels=decoding_stimuli,
                       annot=False, fmt='', ax=ax, 
                       linewidths=outline_thickness if block_outline else 0,
                       linecolor='black' if block_outline else 'none')
    
    # Add chance level line to colorbar
    try:
        cbar = ax.collections[0].colorbar
        # Normalize chance level to colorbar position
        cbar_range = cbar.vmax - cbar.vmin
        chance_pos = (chance_level - cbar.vmin) / cbar_range
        
        cbar.ax.axhline(y=chance_pos, color='black', linestyle='--', linewidth=2)
        cbar.ax.text(1.5, chance_pos, f'Chance ({chance_level})', 
                    color='black', ha='left', va='center', fontweight='bold')
    except Exception as e:
        print(f"Could not add chance line to colorbar: {e}")
    
    # Add text annotations
    for i in range(heatmap_array.shape[0]):
        for j in range(heatmap_array.shape[1]):
            value = heatmap_array[i, j]
            if not np.isnan(value):
                text_color = get_text_color(value, heatcmin, heatcmax)
                precision = 3 if use_raw_scores else 3
                ax.text(j + 0.5, i + 0.5, f'{value:.{precision}f}', 
                      ha='center', va='center', color=text_color,
                      fontsize=font_size, fontweight='bold')
            else:
                ax.text(j + 0.5, i + 0.5, 'N/A', 
                      ha='center', va='center', color='gray',
                      fontsize=font_size//2, alpha=0.7)
    
    # Create combined feature strategy name for title
    used_strategies = set()
    for stimulus in decoding_stimuli:
        if stimulus in stimulus_strategies:
            used_strategies.add(stimulus_strategies[stimulus])
    feature_name = '+'.join(sorted(used_strategies))
    
    plt.title(f'{score_type} {category_to_plot} Decoding Heatmap\n'
             f'Feature Strategies: {feature_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Experimental Day', fontsize=12)
    plt.ylabel('Decoding Stimulus', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # plt.tight_layout()
    
    # Save the figure
    safe_category_name = category_to_plot.replace(" ", "_").replace("+", "plus").replace("-", "minus")
    save_path_base = os.path.join(basedir, 'Results', 
                                  f'{safe_category_name}_{score_type}_MultiStimulus_{feature_name}_{days_to_include}_Heatmap')
    
    try:
        plt.savefig(f"{save_path_base}.pdf", format='PDF', bbox_inches='tight')
        plt.savefig(f"{save_path_base}.png", format='PNG', bbox_inches='tight', dpi=300)
        print(f"💾 Saved: {save_path_base}")
    except Exception as e:
        print(f"❌ Error saving: {str(e)}")
    
    plt.show()
    
    # Print summary stats
    print(f"\nHeatmap values for {category_to_plot}:")
    for i, stimulus in enumerate(decoding_stimuli):
        values = heatmap_array[i, :]
        valid_values = values[~np.isnan(values)]
        if len(valid_values) > 0:
            print(f"  {stimulus}: mean={np.mean(valid_values):.3f}, "
                  f"range=[{np.min(valid_values):.3f}, {np.max(valid_values):.3f}], "
                  f"n_days={len(valid_values)}")
        else:
            print(f"  {stimulus}: No valid data")

    print(f"\n✅ {score_type} multi-stimulus heatmap generation completed.")

# ====================================================================
# MAIN EXECUTION
# ====================================================================

print("="*80)
print("CREATING MULTI-STIMULUS HEATMAPS FROM PICKLED DATA")
print("="*80)
print(f"Stimuli to include: {stimulus_list}")
print(f"Days to include: {days_to_include if days_to_include else 'All available days'}")
print(f"Category to plot: {category_to_plot}")
print(f"Score type: {'Raw scores (0-1 range)' if use_raw_scores else 'Normalized scores (per-neuron corrected)'}")
print("="*80)

try:
    # Auto-detect all available feature strategies by scanning pickle files
    stimulus_strategies = detect_all_feature_strategies(basedir)
    
    if not stimulus_strategies:
        print("❌ No feature strategies detected. Cannot proceed.")
        print("   Make sure you have run the updated Block 3 decoding and saved the pickles.")
    else:
        # Create the multi-stimulus heatmaps using flexible strategy mapping
        create_multi_stimulus_heatmaps_updated(
            stimulus_list, 
            basedir, 
            stimulus_strategies,
            days_to_include=days_to_include,
            use_raw_scores=use_raw_scores,
            category_to_plot=category_to_plot,
            font_size=font_size, 
            block_outline=block_outline, 
            outline_thickness=outline_thickness
        )
    
except Exception as e:
    print(f"❌ Error in multi-stimulus heatmap generation: {str(e)}")
    import traceback
    traceback.print_exc()

print("="*80)
print("MULTI-STIMULUS HEATMAP GENERATION COMPLETE")
if use_raw_scores:
    print("Showed: Raw decoding scores (0-1 range, chance = 0.5)")
else:
    print("Showed: Normalized decoding scores (per-neuron corrected, chance = 0)")
print("="*80)

# Select the Data for the Bar Code and the Corresponding Stats Will Be Done
## These Plots and Stats are Done Across Multiple Decoding_Stimulus

# normalize the data: using each cells own real and shuffled data for a per neuron normalized decoding score

# adapting the per neuron decoding graphing for the paper for side by side early late for each stimulus

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import os
import pickle
import glob
import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# USER CONFIGURATION - EDIT THESE VALUES
# ====================================================================

# Data selection
category_to_analyze = 'Any Significant'  # 'All neurons', 'Any CS significant', etc.
days_to_analyze = ['0_Early', '1_Late']  
###['0_Early', '1_Late'] or ['1_Late', '2_SucOmit', '3_CueOmit' ] or ['4_RevEarly', '5_RevLate']

# Analysis type
use_raw_scores = False  # True = raw vs shuffle analysis, False = normalized only

# Analysis scope
analyze_all_stimuli = True  # True = all stimuli + stimulus main effect
                           # False = single stimulus (specify below)
single_stimulus = 'CS+'  # Used only if analyze_all_stimuli = False

# Output options
create_plots = True  # True = generate plots, False = stats only

# Color customization for bar plots
shuffle_data_color = 'papayawhip'  # Color for shuffled data bars
normalized_data_color = 'deeppink'  # Define the missing color

# Custom colors for each stimulus that match earlier code plots
stimulus_colors = {
    'CS+': 'purple',           
    'CS-': 'grey',             
    'Reward': 'aqua',        
    # 'Reward Omission': 'gold', 
    # 'CS+ Lick': 'mediumorchid', 
    # 'CS- Lick': 'darkgrey',      
    'Hits': 'lawngreen',          
    'Misses': 'black',        
    'False_Alarms': 'gainsboro',  
    'Correct_Rejections': 'palegreen'  
}

# Make colors accessible globally
globals().update({
    'shuffle_data_color': shuffle_data_color,
    'stimulus_colors': stimulus_colors
})

def format_p_value(p_value):
    """Format p-values with appropriate precision."""
    if pd.isna(p_value):
        return "p=NaN"
    elif p_value < 0.0001:
        return "p<0.0001"
    elif p_value < 0.001:
        return f"p={p_value:.4f}"
    elif p_value < 0.01:
        return f"p={p_value:.3f}"
    elif p_value < 0.05:
        return f"p={p_value:.3f}"
    else:
        return f"p={p_value:.3f}"

def validate_selections(category, stimulus=None):
    """Validate user selections."""
    valid_stimuli = ['CS+', 'CS-', 'Reward',  
                    'Hits', 'Misses', 'False_Alarms', 'Correct_Rejections']
    
    valid_categories = ['All neurons', 'Any Significant', 'CS+ Only Significant', 
                       'CS- Only Significant', 'Both CS+&CS- Significant', 'Non-significant']
    
    assert category in valid_categories, f"Invalid category: {category}"
    if stimulus:
        assert stimulus in valid_stimuli, f"Invalid stimulus: {stimulus}"
    print(f"Valid selections - Category: {category}" + (f", Stimulus: {stimulus}" if stimulus else ""))

def get_available_stimuli(basedir):
    """Find all available stimuli with decoding data - FIXED VERSION."""
    results_dir = os.path.join(basedir, 'Results')
    
    # Map stimulus names to safe filenames (display_name -> safe_filename)
    stimulus_mapping = {
        'CS+': 'CSplus',
        'CS-': 'CSminus', 
        'Reward': 'Reward',
        # 'Reward Omission': 'Reward_Omission',
        # 'CS+ Lick': 'CSplus_Lick',
        # 'CS- Lick': 'CSminus_Lick',
        'Hits': 'Hits',
        'Misses': 'Misses',
        'False_Alarms': 'False_Alarms',
        'Correct_Rejections': 'Correct_Rejections'
    }
    
    pattern = os.path.join(results_dir, 'configurable_decoding_day_*.pkl')
    files = glob.glob(pattern)
    
    available_stimuli = []
    
    # Process each stimulus individually to ensure exact matches
    for original_stimulus, safe_stimulus in stimulus_mapping.items():
        found_file = False
        
        for file_path in files:
            filename = os.path.basename(file_path)
            base_name = filename.replace('.pkl', '').replace('configurable_decoding_day_', '')
            
            # Check for exact match
            if base_name.startswith(safe_stimulus + '_'):
                # Additional validation to prevent partial matches
                remainder = base_name[len(safe_stimulus + '_'):]
                
                if safe_stimulus == 'CSplus':
                    # CSplus should not match CSplus_Lick files
                    if not remainder.startswith('Lick'):
                        available_stimuli.append(original_stimulus)
                        found_file = True
                        break
                elif safe_stimulus == 'CSminus':
                    # CSminus should not match CSminus_Lick files
                    if not remainder.startswith('Lick'):
                        available_stimuli.append(original_stimulus)
                        found_file = True
                        break
                elif safe_stimulus == 'Reward':
                    # Reward should not match Reward_Omission files
                    if not remainder.startswith('Omission'):
                        available_stimuli.append(original_stimulus)
                        found_file = True
                        break
                else:
                    # For compound stimuli or others, exact match is fine
                    available_stimuli.append(original_stimulus)
                    found_file = True
                    break
    
    return sorted(available_stimuli)

def load_data(basedir, stimulus_name):
    """Load pickled decoding data for a single stimulus - FIXED VERSION."""
    results_dir = os.path.join(basedir, 'Results')
    
    # Map stimulus to filename (display_name -> safe_filename)
    stimulus_map = {
        'CS+': 'CSplus', 
        'CS-': 'CSminus', 
        'Reward': 'Reward', 
        # 'Reward Omission': 'Reward_Omission', 
        # 'CS+ Lick': 'CSplus_Lick', 
        # 'CS- Lick': 'CSminus_Lick', 
        'Hits': 'Hits', 
        'Misses': 'Misses',
        'False_Alarms': 'False_Alarms', 
        'Correct_Rejections': 'Correct_Rejections'
    }
    
    safe_stimulus = stimulus_map[stimulus_name]
    
    # FIXED: Use exact pattern matching to avoid conflicts
    pattern = os.path.join(results_dir, f'configurable_decoding_day_{safe_stimulus}_*.pkl')
    files = glob.glob(pattern)
    
    if not files:
        print(f"No files found for pattern: {pattern}")
        return None, None
    
    # FIXED: Filter to ensure exact stimulus match (not partial)
    correct_files = []
    for file_path in files:
        filename = os.path.basename(file_path)
        base_name = filename.replace('.pkl', '').replace('configurable_decoding_day_', '')
        
        # Check if this file is exactly for our stimulus (not a partial match)
        if base_name.startswith(safe_stimulus + '_'):
            # Additional check: make sure it's not a longer stimulus name
            # For example, CSplus should not match CSplus_Lick files
            remainder_after_stimulus = base_name[len(safe_stimulus + '_'):]
            
            # Make sure the remainder doesn't start with another stimulus component
            # This prevents CSplus from matching CSplus_Lick files
            if safe_stimulus == 'CSplus':
                if not remainder_after_stimulus.startswith('Lick'):
                    correct_files.append(file_path)
            elif safe_stimulus == 'CSminus':
                if not remainder_after_stimulus.startswith('Lick'):
                    correct_files.append(file_path)
            elif safe_stimulus == 'Reward':
                if not remainder_after_stimulus.startswith('Omission'):
                    correct_files.append(file_path)
            else:
                # For compound stimuli (CSplus_Lick, Reward_Omission, etc.) or others
                correct_files.append(file_path)
    
    if not correct_files:
        print(f"No correct files found for stimulus: {stimulus_name} (safe: {safe_stimulus})")
        print(f"Files found with pattern: {[os.path.basename(f) for f in files]}")
        return None, None
    
    # Use the first correct file
    main_file = correct_files[0]
    filename = os.path.basename(main_file)
    
    # DEBUG: Print which file is being loaded for each stimulus
    print(f"LOADING {stimulus_name}: {filename}")
    
    # Extract feature strategy from the actual filename
    base_name = filename.replace('.pkl', '').replace('configurable_decoding_day_', '')
    
    # For your naming pattern, remove the stimulus name from the beginning to get feature strategy
    feature_strategy = "unknown"
    # Check compound stimuli first (longer names first to avoid partial matches)
    stimulus_patterns = [ 'False_Alarms', 
                        'Correct_Rejections', 'CSplus', 'CSminus', 'Reward', 'Hits', 'Misses']
    
    for stim in stimulus_patterns:
        if base_name.startswith(stim + '_'):
            feature_strategy = base_name[len(stim) + 1:]
            break
    
    # Load data using the actual file found
    try:
        with open(main_file, 'rb') as f:
            decoding_day = pickle.load(f)
        return decoding_day, feature_strategy
    except Exception as e:
        print(f"Error loading {main_file}: {e}")
        return None, None

def load_all_stimuli_data(basedir, stimuli_list):
    """Load data for all available stimuli."""
    all_data = {}
    feature_strategy = None
    
    for stimulus in stimuli_list:
        decoding_day, fs = load_data(basedir, stimulus)
        if decoding_day is not None:
            all_data[stimulus] = decoding_day
            if feature_strategy is None:
                feature_strategy = fs
    
    return all_data, feature_strategy

def filter_category_data(day_data, category_name):
    """Apply category filter to day data."""
    day_data = day_data.drop_duplicates(subset=['cell_id'])
    
    if category_name == 'All neurons' or 'significant_plus' not in day_data.columns:
        return day_data
    
    filters = {
        'Any Significant': (day_data['significant_plus'] == 1) | (day_data['significant_minus'] == 1),
        'CS+ Only Significant': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 0),
        'CS- Only Significant': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 1),
        'Both CS+&CS- Significant': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 1),
        'Non-significant': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 0)
    }
    
    return day_data[filters[category_name]]

def prepare_analysis_data(all_stimuli_data, category_name, days_list, use_raw_scores, analyze_all_stimuli, single_stimulus=None):
    """Prepare long-format dataframe for analysis."""
    analysis_rows = []
    
    if analyze_all_stimuli:
        stimuli_to_process = list(all_stimuli_data.keys())
        print(f"Analyzing all stimuli: {stimuli_to_process}")
    else:
        stimuli_to_process = [single_stimulus] if single_stimulus in all_stimuli_data else []
        print(f"Analyzing single stimulus: {single_stimulus}")
    
    # Get all available days across all stimuli
    all_days = set()
    for stimulus_data in all_stimuli_data.values():
        all_days.update(stimulus_data.keys())
    
    days_to_process = days_list if days_list else sorted(all_days)
    
    for stimulus in stimuli_to_process:
        stimulus_data = all_stimuli_data[stimulus]
        
        for day in days_to_process:
            if day not in stimulus_data:
                continue
            
            filtered_data = filter_category_data(stimulus_data[day], category_name)
            
            if len(filtered_data) == 0:
                continue
            
            for _, row in filtered_data.iterrows():
                cell_id = row['cell_id']
                
                if use_raw_scores:
                    if not (pd.isna(row['raw_decoding_score']) or pd.isna(row['per_neuron_shuffle'])):
                        analysis_rows.extend([
                            {'cell_id': cell_id, 'day': day, 'stimulus': stimulus,
                             'score_type': 'Raw', 'decoding_score': row['raw_decoding_score']},
                            {'cell_id': cell_id, 'day': day, 'stimulus': stimulus,
                             'score_type': 'Shuffle', 'decoding_score': row['per_neuron_shuffle']}
                        ])
                else:
                    if not pd.isna(row['decoding_score']):
                        analysis_rows.append({
                            'cell_id': cell_id, 'day': day, 'stimulus': stimulus,
                            'score_type': 'Normalized', 'decoding_score': row['decoding_score']
                        })
    
    df_analysis = pd.DataFrame(analysis_rows)
    
    if len(df_analysis) > 0:
        print(f"Analysis data: {len(df_analysis)} observations, {df_analysis['cell_id'].nunique()} neurons")
        if analyze_all_stimuli:
            stimulus_counts = df_analysis.groupby('stimulus')['cell_id'].nunique()
            print("Neurons per stimulus:")
            for stim, count in stimulus_counts.items():
                print(f"  {stim}: {count}")
    
    return df_analysis

def run_statistical_analysis(df_analysis, use_raw_scores, analyze_all_stimuli):
    """Run MLM ANOVA with proper F-tests and posthoc tests."""
    print("\nRunning Statistical Analysis:")
    print("="*50)
    
    if len(df_analysis) == 0:
        print("No data available for analysis")
        return
    
    try:
        # Build formula based on analysis type
        if use_raw_scores:
            if analyze_all_stimuli:
                # Try progressively simpler models
                formulas_to_try = [
                    "decoding_score ~ C(score_type) + C(day) + C(stimulus) + C(score_type):C(day) + C(score_type):C(stimulus) + C(day):C(stimulus)",
                    "decoding_score ~ C(score_type) + C(day) + C(stimulus) + C(score_type):C(day) + C(score_type):C(stimulus)",
                    "decoding_score ~ C(score_type) + C(day) + C(stimulus)"
                ]
            else:
                formulas_to_try = ["decoding_score ~ C(score_type) + C(day) + C(score_type):C(day)"]
        else:
            if analyze_all_stimuli:
                formulas_to_try = [
                    "decoding_score ~ C(day) + C(stimulus) + C(day):C(stimulus)",
                    "decoding_score ~ C(day) + C(stimulus)"
                ]
            else:
                formulas_to_try = ["decoding_score ~ C(day)"]
        
        # Try fitting models in order of complexity
        fitted_model = None
        formula = None
        
        for formula_attempt in formulas_to_try:
            try:
                print(f"Trying formula: {formula_attempt}")
                model = mixedlm(formula_attempt, df_analysis, groups=df_analysis['cell_id'])
                fitted_model = model.fit(method='lbfgs', maxiter=1000)
                formula = formula_attempt
                print("Model converged successfully!")
                break
            except Exception as e:
                print(f"Model failed: {str(e)[:100]}...")
                continue
        
        if fitted_model is None:
            print("All models failed to converge")
            return
        
        # Display model results
        print(f"\nMLM Results (n={fitted_model.nobs}, groups={len(df_analysis['cell_id'].unique())}):")
        print(f"Final formula: {formula}")
        
        # OMNIBUS F-TESTS using statsmodels built-in functionality
        print("\nOmnibus F-Tests:")
        print("-" * 30)
        
        params = fitted_model.params
        df_resid = fitted_model.df_resid
        
        # Test main effects and interactions
        if use_raw_scores:
            # Score Type main effect
            score_type_params = [p for p in params.index if 'C(score_type)' in p and ':' not in p]
            if score_type_params:
                try:
                    hypothesis_matrix = np.zeros((len(score_type_params), len(params)))
                    for i, param in enumerate(score_type_params):
                        param_idx = list(params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    f_test = fitted_model.f_test(hypothesis_matrix)
                    print(f"Score Type: F({f_test.df_num}, {f_test.df_denom})={f_test.fvalue:.3f}, {format_p_value(f_test.pvalue)}")
                except Exception as e:
                    print(f"Score Type F-test failed: {e}")
        
        # Day main effect
        day_params = [p for p in params.index if 'C(day)[T.' in p and ':' not in p]
        if day_params:
            try:
                hypothesis_matrix = np.zeros((len(day_params), len(params)))
                for i, param in enumerate(day_params):
                    param_idx = list(params.index).index(param)
                    hypothesis_matrix[i, param_idx] = 1
                f_test = fitted_model.f_test(hypothesis_matrix)
                print(f"Day: F({f_test.df_num}, {f_test.df_denom})={f_test.fvalue:.3f}, {format_p_value(f_test.pvalue)}")
            except Exception as e:
                print(f"Day F-test failed: {e}")
        
        # Stimulus main effect (if applicable)
        if analyze_all_stimuli:
            stimulus_params = [p for p in params.index if 'C(stimulus)[T.' in p and ':' not in p]
            if stimulus_params:
                try:
                    hypothesis_matrix = np.zeros((len(stimulus_params), len(params)))
                    for i, param in enumerate(stimulus_params):
                        param_idx = list(params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    f_test = fitted_model.f_test(hypothesis_matrix)
                    print(f"Stimulus: F({f_test.df_num}, {f_test.df_denom})={f_test.fvalue:.3f}, {format_p_value(f_test.pvalue)}")
                except Exception as e:
                    print(f"Stimulus F-test failed: {e}")
        
        # Interaction effects
        if use_raw_scores:
            # Score Type x Day interaction
            interaction_params = [p for p in params.index if 'C(score_type)' in p and 'C(day)[T.' in p]
            if interaction_params:
                try:
                    hypothesis_matrix = np.zeros((len(interaction_params), len(params)))
                    for i, param in enumerate(interaction_params):
                        param_idx = list(params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    f_test = fitted_model.f_test(hypothesis_matrix)
                    print(f"Score Type x Day: F({f_test.df_num}, {f_test.df_denom})={f_test.fvalue:.3f}, {format_p_value(f_test.pvalue)}")
                except Exception as e:
                    print(f"Score Type x Day F-test failed: {e}")
            
            # Score Type x Stimulus interaction
            if analyze_all_stimuli:
                interaction_params = [p for p in params.index if 'C(score_type)' in p and 'C(stimulus)[T.' in p]
                if interaction_params:
                    try:
                        hypothesis_matrix = np.zeros((len(interaction_params), len(params)))
                        for i, param in enumerate(interaction_params):
                            param_idx = list(params.index).index(param)
                            hypothesis_matrix[i, param_idx] = 1
                        f_test = fitted_model.f_test(hypothesis_matrix)
                        print(f"Score Type x Stimulus: F({f_test.df_num}, {f_test.df_denom})={f_test.fvalue:.3f}, {format_p_value(f_test.pvalue)}")
                    except Exception as e:
                        print(f"Score Type x Stimulus F-test failed: {e}")
        
        # Day x Stimulus interaction
        if analyze_all_stimuli:
            interaction_params = [p for p in params.index if 'C(day)[T.' in p and 'C(stimulus)[T.' in p and 'C(score_type)' not in p]
            if interaction_params:
                try:
                    hypothesis_matrix = np.zeros((len(interaction_params), len(params)))
                    for i, param in enumerate(interaction_params):
                        param_idx = list(params.index).index(param)
                        hypothesis_matrix[i, param_idx] = 1
                    f_test = fitted_model.f_test(hypothesis_matrix)
                    print(f"Day x Stimulus: F({f_test.df_num}, {f_test.df_denom})={f_test.fvalue:.3f}, {format_p_value(f_test.pvalue)}")
                except Exception as e:
                    print(f"Day x Stimulus F-test failed: {e}")
        
        # INDIVIDUAL PARAMETER TESTS
        print("\nIndividual Parameter Tests:")
        print("-" * 35)
        
        pvalues = fitted_model.pvalues
        tvalues = fitted_model.tvalues
        
        # Score type effect
        if use_raw_scores and 'C(score_type)[T.Shuffle]' in pvalues:
            t_stat = tvalues['C(score_type)[T.Shuffle]']
            p_val = pvalues['C(score_type)[T.Shuffle]']
            print(f"Raw vs Shuffle: t({df_resid})={t_stat:.3f}, {format_p_value(p_val)}")
        
        # Day effects (showing first few)
        day_effects = [(param, pvalues[param], tvalues[param]) 
                      for param in pvalues.index if 'C(day)[T.' in param and ':' not in param]
        if day_effects:
            print("Day Effects (vs reference, showing first 5):")
            for param, p_val, t_val in day_effects[:5]:
                day_name = param.replace('C(day)[T.', '').replace(']', '')
                print(f"  {day_name}: t({df_resid})={t_val:.3f}, {format_p_value(p_val)}")
        
        # Stimulus effects (showing first few)
        if analyze_all_stimuli:
            stimulus_effects = [(param, pvalues[param], tvalues[param]) 
                              for param in pvalues.index if 'C(stimulus)[T.' in param and ':' not in param]
            if stimulus_effects:
                print("Stimulus Effects (vs reference, showing first 5):")
                for param, p_val, t_val in stimulus_effects[:5]:
                    stim_name = param.replace('C(stimulus)[T.', '').replace(']', '')
                    print(f"  {stim_name}: t({df_resid})={t_val:.3f}, {format_p_value(p_val)}")
        
        # Run posthoc tests
        run_posthoc_tests(df_analysis, use_raw_scores, analyze_all_stimuli)
        
    except Exception as e:
        print(f"Error in statistical analysis: {str(e)}")
        import traceback
        traceback.print_exc()
def run_posthoc_tests(df_analysis, use_raw_scores, analyze_all_stimuli):
    """Run posthoc comparisons with proper p-value formatting."""
    print("\nPosthoc Tests:")
    print("-" * 20)
    
    try:
        if use_raw_scores:
            raw_data = df_analysis[df_analysis['score_type'] == 'Raw']
        else:
            raw_data = df_analysis
        
        if analyze_all_stimuli:
            # 1. Within stimulus across days
            print("1. Within Stimulus Across Days:")
            for stimulus in raw_data['stimulus'].unique():
                stim_data = raw_data[raw_data['stimulus'] == stimulus]
                unique_days = stim_data['day'].unique()
                
                if len(unique_days) > 1:
                    try:
                        tukey_result = pairwise_tukeyhsd(stim_data['decoding_score'], stim_data['day'])
                        
                        # Extract significant comparisons
                        summary_data = tukey_result.summary().data[1:]  # Skip header
                        sig_comparisons = []
                        
                        for row in summary_data:
                            group1, group2, meandiff, p_adj, lower, upper, reject = row
                            if reject:  # Significant difference
                                sig_comparisons.append((group1, group2, float(p_adj)))
                        
                        if sig_comparisons:
                            print(f"  {stimulus} - significant day pairs:")
                            for g1, g2, p_val in sig_comparisons[:5]:  # Limit output
                                print(f"    {g1} vs {g2}: {format_p_value(p_val)}")
                        else:
                            print(f"  {stimulus} - no significant day pairs")
                    except Exception as e:
                        print(f"  {stimulus} - Tukey test failed: {e}")
            
            # 2. Within day across stimuli
            print("\n2. Within Day Across Stimuli:")
            for day in raw_data['day'].unique():
                day_data = raw_data[raw_data['day'] == day]
                unique_stimuli = day_data['stimulus'].unique()
                
                if len(unique_stimuli) > 1:
                    try:
                        tukey_result = pairwise_tukeyhsd(day_data['decoding_score'], day_data['stimulus'])
                        
                        # Extract significant comparisons
                        summary_data = tukey_result.summary().data[1:]  # Skip header
                        sig_comparisons = []
                        
                        for row in summary_data:
                            group1, group2, meandiff, p_adj, lower, upper, reject = row
                            if reject:  # Significant difference
                                sig_comparisons.append((group1, group2, float(p_adj)))
                        
                        if sig_comparisons:
                            print(f"  {day} - significant stimulus pairs:")
                            for g1, g2, p_val in sig_comparisons[:5]:  # Limit output
                                print(f"    {g1} vs {g2}: {format_p_value(p_val)}")
                        else:
                            print(f"  {day} - no significant stimulus pairs")
                    except Exception as e:
                        print(f"  {day} - Tukey test failed: {e}")
            
            # 3. All stimulus x day combinations
            print("\n3. All Stimulus x Day Combinations:")
            try:
                raw_data_copy = raw_data.copy()
                raw_data_copy['stim_day'] = raw_data_copy['stimulus'] + '_' + raw_data_copy['day']
                unique_combinations = raw_data_copy['stim_day'].unique()
                
                if len(unique_combinations) > 1:
                    tukey_result = pairwise_tukeyhsd(raw_data_copy['decoding_score'], raw_data_copy['stim_day'])
                    
                    # Extract significant comparisons
                    summary_data = tukey_result.summary().data[1:]  # Skip header
                    sig_comparisons = []
                    
                    for row in summary_data:
                        group1, group2, meandiff, p_adj, lower, upper, reject = row
                        if reject:  # Significant difference
                            sig_comparisons.append((group1, group2, float(p_adj)))
                    
                    if sig_comparisons:
                        print(f"  Significant stimulus x day combinations (showing first 100):")
                        for g1, g2, p_val in sig_comparisons[:100]:  # Limit output
                            print(f"    {g1} vs {g2}: {format_p_value(p_val)}")
                    else:
                        print("  No significant stimulus x day combinations")
                else:
                    print("  Insufficient combinations for comparison")
            except Exception as e:
                print(f"  Stimulus x Day combinations test failed: {e}")
        
        else:
            # Single stimulus - only days
            unique_days = raw_data['day'].unique()
            if len(unique_days) > 1:
                try:
                    tukey_result = pairwise_tukeyhsd(raw_data['decoding_score'], raw_data['day'])
                    
                    # Extract significant comparisons
                    summary_data = tukey_result.summary().data[1:]  # Skip header
                    sig_comparisons = []
                    
                    for row in summary_data:
                        group1, group2, meandiff, p_adj, lower, upper, reject = row
                        if reject:  # Significant difference
                            sig_comparisons.append((group1, group2, float(p_adj)))
                    
                    if sig_comparisons:
                        print("Significant day pairs:")
                        for g1, g2, p_val in sig_comparisons:
                            print(f"  {g1} vs {g2}: {format_p_value(p_val)}")
                    else:
                        print("No significant day pairs")
                except Exception as e:
                    print(f"Day comparison test failed: {e}")
            else:
                print("Insufficient days for pairwise comparison")
    
    except Exception as e:
        print(f"Error in posthoc tests: {str(e)}")
        import traceback
        traceback.print_exc()

def create_plots(all_stimuli_data, category_name, days_list, basedir, feature_strategy, 
                use_raw_scores, analyze_all_stimuli, single_stimulus=None):
    """Create bar plots only."""
    print("\nCreating Plots:")
    print("="*50)
    
    if analyze_all_stimuli:
        stimuli_to_plot = list(all_stimuli_data.keys())
        print(f"Creating plots for all stimuli: {len(stimuli_to_plot)} stimuli")
    else:
        stimuli_to_plot = [single_stimulus] if single_stimulus in all_stimuli_data else []
        print(f"Creating plots for single stimulus: {single_stimulus}")
    
    # Get all available days
    all_days = set()
    for stimulus_data in all_stimuli_data.values():
        all_days.update(stimulus_data.keys())
    days_to_plot = days_list if days_list else sorted(all_days)
    
    # Prepare plot data
    plot_data = {}
    for stimulus in stimuli_to_plot:
        plot_data[stimulus] = {}
        stimulus_data = all_stimuli_data[stimulus]
        
        for day in days_to_plot:
            if day not in stimulus_data:
                continue
            
            filtered_data = filter_category_data(stimulus_data[day], category_name)
            if len(filtered_data) == 0:
                continue
            
            if use_raw_scores:
                raw_scores = filtered_data['raw_decoding_score'].dropna()
                shuffle_scores = filtered_data['per_neuron_shuffle'].dropna()
                min_len = min(len(raw_scores), len(shuffle_scores))
                if min_len > 0:
                    plot_data[stimulus][day] = {
                        'raw': raw_scores.iloc[:min_len],
                        'shuffle': shuffle_scores.iloc[:min_len]
                    }
            else:
                norm_scores = filtered_data['decoding_score'].dropna()
                if len(norm_scores) > 0:
                    plot_data[stimulus][day] = {'normalized': norm_scores}
    
    # Remove empty stimuli
    plot_data = {stim: days for stim, days in plot_data.items() if days}
    
    if not plot_data:
        print("No valid plot data found")
        return
    
    # Create bar plots
    if analyze_all_stimuli:
        create_multi_stimulus_bar_plot(plot_data, category_name, use_raw_scores, 
                                      basedir, feature_strategy)
    else:
        create_single_stimulus_bar_plot(plot_data[single_stimulus], single_stimulus, 
                                       category_name, use_raw_scores, basedir, feature_strategy)


def create_multi_stimulus_bar_plot(plot_data, category_name, use_raw_scores, basedir, feature_strategy):
    """Create bar plot with days grouped by stimulus (stimulus on x-axis, days side-by-side)."""
    print(f"Creating multi-stimulus bar plot with days grouped by stimulus...")
    
    stimuli = list(plot_data.keys())
    all_days = sorted(set(day for stim_data in plot_data.values() for day in stim_data.keys()))
    
    if not stimuli:
        print("No stimuli data available for bar plot")
        return
    
    # Create single plot with all stimuli on x-axis
    fig, ax = plt.subplots(figsize=(max(12, len(stimuli) * 2.5), 8))
    
    if use_raw_scores:
        # For raw vs shuffle, we'll create grouped bars for each stimulus
        # Each stimulus gets n_days * 2 bars (raw and shuffle for each day)
        
        x_positions = []
        labels = []
        
        bar_width = 0.35
        day_group_spacing = 0.1  # Small gap between days within a stimulus
        stimulus_spacing = 1.0   # Larger gap between stimuli
        
        current_x = 0
        
        for stim_idx, stimulus in enumerate(stimuli):
            if stimulus not in plot_data:
                continue
            
            stim_days = [day for day in all_days if day in plot_data[stimulus]]
            
            for day_idx, day in enumerate(stim_days):
                # Position for this day's raw bar
                x_raw = current_x + day_idx * (2 * bar_width + day_group_spacing)
                # Position for this day's shuffle bar (right next to raw)
                x_shuffle = x_raw + bar_width
                
                # Get data
                raw_data = plot_data[stimulus][day]['raw']
                shuffle_data = plot_data[stimulus][day]['shuffle']
                
                raw_mean = np.mean(raw_data)
                raw_sem = stats.sem(raw_data)
                shuffle_mean = np.mean(shuffle_data)
                shuffle_sem = stats.sem(shuffle_data)
                
                # Get colors
                raw_color = stimulus_colors.get(stimulus, '#1f77b4')
                shuffle_color = shuffle_data_color
                
                # Plot bars
                ax.bar(x_raw, raw_mean, bar_width, yerr=raw_sem,
                      color=raw_color, alpha=0.8, capsize=4,
                      label='Raw' if stim_idx == 0 and day_idx == 0 else "")
                ax.bar(x_shuffle, shuffle_mean, bar_width, yerr=shuffle_sem,
                      color=shuffle_color, alpha=0.7, capsize=4,
                      label='Shuffle' if stim_idx == 0 and day_idx == 0 else "")
                
                # Add significance stars
                try:
                    _, p_val = stats.ttest_rel(raw_data, shuffle_data)
                    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
                    y_pos = max(raw_mean + raw_sem, shuffle_mean + shuffle_sem) + 0.03
                    ax.text((x_raw + x_shuffle) / 2, y_pos, sig, ha='center', 
                           fontweight='bold', fontsize=10)
                except:
                    pass
                
                # Store position for label (center of raw and shuffle bars for first day)
                if day_idx == 0:
                    # Calculate center of all bars for this stimulus
                    stimulus_center = current_x + (len(stim_days) * (2 * bar_width + day_group_spacing) - day_group_spacing) / 2
                    x_positions.append(stimulus_center)
            
            # Add stimulus label
            labels.append(stimulus)
            
            # Move to next stimulus position
            current_x += len(stim_days) * (2 * bar_width + day_group_spacing) + stimulus_spacing
        
        # Formatting
        ax.set_ylabel('Decoding Score', fontsize=12)
        ax.set_title(f'{category_name} - Raw vs Shuffle Across Stimuli and Days', 
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x_positions)
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_ylim(0, 0.5)
        ax.set_yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
        ax.axhline(0.5, color='black', linestyle='--', alpha=0.6, label='Chance (0.5)')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add day labels as text annotations
        current_x = 0
        for stimulus in stimuli:
            if stimulus not in plot_data:
                continue
            
            stim_days = [day for day in all_days if day in plot_data[stimulus]]
            
            for day_idx, day in enumerate(stim_days):
                x_center = current_x + day_idx * (2 * bar_width + day_group_spacing) + bar_width
                # Add day label below x-axis
                ax.text(x_center, -0.08, day, ha='center', va='top', 
                       fontsize=8, transform=ax.get_xaxis_transform())
            
            current_x += len(stim_days) * (2 * bar_width + day_group_spacing) + stimulus_spacing
        
        title_suffix = "Raw_vs_Shuffle"
    
    else:
        # Normalized scores - similar grouping structure
        x_positions = []
        labels = []
        
        bar_width = 0.5
        day_spacing = 0.15
        stimulus_spacing = 1.0
        
        current_x = 0
        
        for stimulus in stimuli:
            if stimulus not in plot_data:
                continue
            
            stim_days = [day for day in all_days if day in plot_data[stimulus]]
            
            for day_idx, day in enumerate(stim_days):
                x_pos = current_x + day_idx * (bar_width + day_spacing)
                
                # Get data
                norm_data = plot_data[stimulus][day]['normalized']
                norm_mean = np.mean(norm_data)
                norm_sem = stats.sem(norm_data)
                
                # Get color
                color = stimulus_colors.get(stimulus, normalized_data_color)
                
                # Plot bar
                ax.bar(x_pos, norm_mean, bar_width, yerr=norm_sem,
                      color=color, alpha=0.8, capsize=4)
                
                # Add significance stars (one-sample t-test vs 0)
                try:
                    _, p_val = stats.ttest_1samp(norm_data, 0)
                    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
                    y_pos = norm_mean + norm_sem + 0.03
                    ax.text(x_pos, y_pos, sig, ha='center', fontweight='bold', fontsize=10)
                except:
                    pass
                
                if day_idx == 0:
                    # Calculate center of all bars for this stimulus
                    stimulus_center = current_x + (len(stim_days) * (bar_width + day_spacing) - day_spacing) / 2
                    x_positions.append(stimulus_center)
            
            labels.append(stimulus)
            current_x += len(stim_days) * (bar_width + day_spacing) + stimulus_spacing
        
        # Formatting
        ax.set_ylabel('Normalized Decoding Score', fontsize=12)
        ax.set_title(f'{category_name} - Normalized Scores Across Stimuli and Days', 
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x_positions)
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_ylim(0, 0.3)
        ax.set_yticks([0, 0.1, 0.2, 0.3])
        ax.axhline(0, color='black', linestyle='--', alpha=0.6, label='Chance (0)')
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add day labels
        current_x = 0
        for stimulus in stimuli:
            if stimulus not in plot_data:
                continue
            
            stim_days = [day for day in all_days if day in plot_data[stimulus]]
            
            for day_idx, day in enumerate(stim_days):
                x_pos = current_x + day_idx * (bar_width + day_spacing)
                ax.text(x_pos, -0.08, day, ha='center', va='top', 
                       fontsize=8, transform=ax.get_xaxis_transform())
            
            current_x += len(stim_days) * (bar_width + day_spacing) + stimulus_spacing
        
        title_suffix = "Normalized"
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)  # Make room for day labels
    
    # Save
    # safe_category = category_name.replace(' ', '_').replace('+', 'plus').replace('-', 'minus')
    # filename = f"{basedir}/Results/{safe_category}_AllStimuli_{feature_strategy}_{title_suffix}_Bar_{days_to_analyze}_10-07-25.pdf"
    # os.makedirs(os.path.dirname(filename), exist_ok=True)
    # fig.savefig(filename, bbox_inches='tight', dpi=300)
    plt.show()
    # print(f"Saved multi-stimulus bar plot: {filename}")

def create_single_stimulus_bar_plot(stim_data, stimulus_name, category_name, use_raw_scores, basedir, feature_strategy):
    """Create bar plot for single stimulus with days side-by-side."""
    print(f"Creating single-stimulus bar plot for {stimulus_name}...")
    
    days = sorted(stim_data.keys())
    
    fig, ax = plt.subplots(figsize=(max(8, len(days) * 2), 7))
    
    if use_raw_scores:
        bar_width = 0.35
        x = np.arange(len(days))
        
        raw_means = []
        raw_sems = []
        shuffle_means = []
        shuffle_sems = []
        
        for day in days:
            raw_data = stim_data[day]['raw']
            shuffle_data = stim_data[day]['shuffle']
            
            raw_means.append(np.mean(raw_data))
            raw_sems.append(stats.sem(raw_data))
            shuffle_means.append(np.mean(shuffle_data))
            shuffle_sems.append(stats.sem(shuffle_data))
        
        # Get stimulus-specific color for raw data
        stimulus_color = stimulus_colors.get(stimulus_name, '#1f77b4')
        
        # Plot bars side by side
        bars_raw = ax.bar(x - bar_width/2, raw_means, bar_width, yerr=raw_sems, 
                        label='Raw', color=stimulus_color, alpha=0.8, capsize=4)
        bars_shuffle = ax.bar(x + bar_width/2, shuffle_means, bar_width, yerr=shuffle_sems, 
                            label='Shuffle', color=shuffle_data_color, alpha=0.7, capsize=4)
        
        ax.axhline(0.5, color='black', linestyle='--', alpha=0.6, label='Chance (0.5)')
        
        # Add significance stars
        for i, day in enumerate(days):
            try:
                _, p_val = stats.ttest_rel(stim_data[day]['raw'], stim_data[day]['shuffle'])
                sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
                y_pos = max(raw_means[i] + raw_sems[i], shuffle_means[i] + shuffle_sems[i]) + 0.03
                ax.text(i, y_pos, sig, ha='center', fontweight='bold', fontsize=12)
            except:
                pass
        
        ylabel = "Decoding Score"
        title_suffix = "Raw_vs_Shuffle"
    
    else:
        bar_width = 0.6
        x = np.arange(len(days))
        
        norm_means = []
        norm_sems = []
        
        for day in days:
            norm_data = stim_data[day]['normalized']
            norm_means.append(np.mean(norm_data))
            norm_sems.append(stats.sem(norm_data))
        
        # Get stimulus-specific color
        stimulus_color = stimulus_colors.get(stimulus_name, normalized_data_color)
        
        bars = ax.bar(x, norm_means, bar_width, yerr=norm_sems, 
                     label='Normalized', color=stimulus_color, alpha=0.8, capsize=4)
        
        ax.axhline(0, color='black', linestyle='--', alpha=0.6, label='Chance (0)')
        
        # Add significance stars (one-sample t-test vs 0)
        for i, day in enumerate(days):
            try:
                _, p_val = stats.ttest_1samp(stim_data[day]['normalized'], 0)
                sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
                y_pos = norm_means[i] + norm_sems[i] + 0.03
                ax.text(i, y_pos, sig, ha='center', fontweight='bold', fontsize=12)
            except:
                pass
        
        ylabel = "Normalized Decoding Score"
        title_suffix = "Normalized"
    
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f'{stimulus_name} - {category_name} - {title_suffix}', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(days, rotation=45, ha='right')
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.legend(frameon=True, fancybox=True, shadow=True)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Save
    safe_names = [name.replace(' ', '_').replace('+', 'plus').replace('-', 'minus') 
                  for name in [stimulus_name, category_name]]
    suffix = "Raw_vs_Shuffle" if use_raw_scores else "Normalized"
    filename = f"{basedir}/Results/{safe_names[0]}_{safe_names[1]}_{feature_strategy}_{suffix}_Bar.pdf"
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig.savefig(filename, bbox_inches='tight', dpi=300)
    plt.show()
    print(f"Saved single-stimulus bar plot: {filename}")

def main():
    """Main execution function."""
    print("="*60)
    print("MULTI-STIMULUS DECODING ANALYSIS")
    print("="*60)
    print(f"Category: {category_to_analyze}")
    if analyze_all_stimuli:
        print("Analysis: All stimuli + stimulus main effect")
    else:
        print(f"Stimulus: {single_stimulus}")
    print(f"Days: {days_to_analyze if days_to_analyze else 'All available'}")
    print(f"Analysis: {'Raw vs Shuffle' if use_raw_scores else 'Normalized only'}")
    print(f"Plots: Bar plots only")
    print("="*60)
    
    try:
        # Validate and get available stimuli
        if analyze_all_stimuli:
            validate_selections(category_to_analyze)
            available_stimuli = get_available_stimuli(basedir)
            print(f"Available stimuli: {available_stimuli}")
            
            if not available_stimuli:
                raise ValueError("No stimulus data found")
            
            # Load data for all stimuli
            all_stimuli_data, feature_strategy = load_all_stimuli_data(basedir, available_stimuli)
            
            if not all_stimuli_data:
                raise ValueError("Failed to load any stimulus data")
            
            print(f"Successfully loaded {len(all_stimuli_data)} stimuli")
        else:
            validate_selections(category_to_analyze, single_stimulus)
            decoding_data, feature_strategy = load_data(basedir, single_stimulus)
            
            if decoding_data is None:
                raise ValueError(f"Failed to load data for {single_stimulus}")
            
            all_stimuli_data = {single_stimulus: decoding_data}
        
        # Prepare analysis dataframe
        df_analysis = prepare_analysis_data(all_stimuli_data, category_to_analyze, days_to_analyze, 
                                           use_raw_scores, analyze_all_stimuli, single_stimulus)
        
        if len(df_analysis) == 0:
            print("No valid data found for analysis")
        else:
            # Run statistical analysis
            run_statistical_analysis(df_analysis, use_raw_scores, analyze_all_stimuli)
            
            # Create plots if requested
            if create_plots:
                create_plots(all_stimuli_data, category_to_analyze, days_to_analyze, basedir, 
                            feature_strategy, use_raw_scores, analyze_all_stimuli, single_stimulus)
        
        print("\nAnalysis Complete!")
        
        if analyze_all_stimuli:
            print("\nSummary:")
            print("- Analyzed all available stimuli simultaneously")
            print("- Statistical model includes stimulus as main effect")
            print("- Bar plots show days grouped by stimulus")
            print("- Posthocs include:")
            print("  1. Within stimulus across days")
            print("  2. Within day across stimuli") 
            print("  3. All stimulus x day combinations")
            if use_raw_scores:
                print("- MLM includes: Score Type + Day + Stimulus + interactions")
            else:
                print("- MLM includes: Day + Stimulus + Day x Stimulus interaction")
    
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

# Execute main function
if __name__ == "__main__":
    main()

In [ ]:
# ##THIS WAS USED TO FIX AN ABOVE ISSUE IT IS A DIAGNOSTIC THAT PRINTS THE LOADED PKL FILES
# ###this is a diagnostic to check
# import numpy as np
# import pandas as pd
# import pickle
# import glob
# import os

# # ====================================================================
# # CONFIGURATION - MATCH YOUR MAIN SCRIPT
# # ====================================================================

# # Analysis parameters (match your main script)
# category_to_analyze = 'Any Significant'
# days_to_analyze = ['2_SucOmit']

# # ====================================================================
# # DIAGNOSTIC FUNCTIONS
# # ====================================================================

# def get_available_stimuli_with_files(basedir):
#     """Find all available stimuli and their actual file paths."""
#     results_dir = os.path.join(basedir, 'Results')
    
#     # Map stimulus names to safe filenames
#     stimulus_mapping = {
#         'CS+': 'CSplus',
#         'CS-': 'CSminus', 
#         'Reward': 'Reward',
#         'Reward Omission': 'Reward_Omission',
#         'CS+ Lick': 'CSplus_Lick',
#         'CS- Lick': 'CSminus_Lick',
#         'Hits': 'Hits',
#         'Misses': 'Misses',
#         'False_Alarms': 'False_Alarms',
#         'Correct_Rejections': 'Correct_Rejections'
#     }
    
#     pattern = os.path.join(results_dir, 'configurable_decoding_day_*.pkl')
#     files = glob.glob(pattern)
    
#     available_stimuli = {}
    
#     # Known safe stimuli in order (longest first to avoid partial matches)
#     known_safe_stimuli = [
#         'CSplus_Lick', 'CSminus_Lick', 'Reward_Omission',  # Compound stimuli first
#         'False_Alarms', 'Correct_Rejections',              # Multi-word stimuli  
#         'CSplus', 'CSminus', 'Reward', 'Hits', 'Misses'    # Simple stimuli
#     ]
    
#     for file_path in files:
#         filename = os.path.basename(file_path)
#         base_name = filename.replace('.pkl', '')
        
#         # Remove prefix
#         prefix = 'configurable_decoding_day_'
#         if base_name.startswith(prefix):
#             remainder = base_name[len(prefix):]
            
#             # Try to match against known stimuli
#             for known_stimulus in known_safe_stimuli:
#                 if remainder.startswith(known_stimulus + '_'):
#                     # Map back to original stimulus name
#                     original_stimulus = None
#                     for orig, safe in stimulus_mapping.items():
#                         if safe == known_stimulus:
#                             original_stimulus = orig
#                             break
                    
#                     if original_stimulus:
#                         available_stimuli[original_stimulus] = file_path
#                     break
    
#     return available_stimuli

# def load_and_inspect_data(file_path, stimulus_name):
#     """Load pickle file and extract key information."""
#     print(f"\n{'='*60}")
#     print(f"STIMULUS: {stimulus_name}")
#     print(f"FILE: {os.path.basename(file_path)}")
#     print(f"{'='*60}")
    
#     try:
#         with open(file_path, 'rb') as f:
#             decoding_day = pickle.load(f)
        
#         print(f"✓ Successfully loaded pickle file")
#         print(f"✓ Data type: {type(decoding_day)}")
#         print(f"✓ Available days: {list(decoding_day.keys())}")
        
#         # Check each day's data
#         for day in days_to_analyze:
#             if day in decoding_day:
#                 day_data = decoding_day[day]
#                 print(f"\n--- DAY: {day} ---")
#                 print(f"Data shape: {day_data.shape}")
#                 print(f"Columns: {list(day_data.columns)}")
#                 print(f"Total neurons: {len(day_data)}")
                
#                 # Apply the same filtering as main script
#                 filtered_data = filter_category_data(day_data, category_to_analyze)
#                 print(f"Filtered neurons ({category_to_analyze}): {len(filtered_data)}")
                
#                 if len(filtered_data) > 0:
#                     print(f"\nFirst 15 RAW DECODING SCORES:")
#                     raw_scores = filtered_data['raw_decoding_score'].dropna()
#                     print(f"Available raw scores: {len(raw_scores)}")
#                     if len(raw_scores) > 0:
#                         for i, score in enumerate(raw_scores.head(15)):
#                             print(f"  {i+1:2d}: {score:.6f}")
#                     else:
#                         print("  No raw scores available!")
                    
#                     print(f"\nFirst 15 SHUFFLE SCORES:")
#                     shuffle_scores = filtered_data['per_neuron_shuffle'].dropna()
#                     print(f"Available shuffle scores: {len(shuffle_scores)}")
#                     if len(shuffle_scores) > 0:
#                         for i, score in enumerate(shuffle_scores.head(15)):
#                             print(f"  {i+1:2d}: {score:.6f}")
#                     else:
#                         print("  No shuffle scores available!")
                    
#                     print(f"\nFirst 15 NORMALIZED SCORES:")
#                     norm_scores = filtered_data['decoding_score'].dropna()
#                     print(f"Available normalized scores: {len(norm_scores)}")
#                     if len(norm_scores) > 0:
#                         for i, score in enumerate(norm_scores.head(15)):
#                             print(f"  {i+1:2d}: {score:.6f}")
#                     else:
#                         print("  No normalized scores available!")
                    
#                     # Show summary statistics
#                     print(f"\nSUMMARY STATISTICS:")
#                     if len(raw_scores) > 0:
#                         print(f"Raw scores    - Mean: {raw_scores.mean():.4f}, Std: {raw_scores.std():.4f}")
#                     if len(shuffle_scores) > 0:
#                         print(f"Shuffle scores - Mean: {shuffle_scores.mean():.4f}, Std: {shuffle_scores.std():.4f}")
#                     if len(norm_scores) > 0:
#                         print(f"Norm scores   - Mean: {norm_scores.mean():.4f}, Std: {norm_scores.std():.4f}")
                        
#                 else:
#                     print(f"❌ No neurons pass the filtering criteria!")
#             else:
#                 print(f"\n--- DAY: {day} ---")
#                 print(f"❌ Day not found in data!")
    
#     except Exception as e:
#         print(f"❌ Error loading {file_path}: {e}")
#         import traceback
#         traceback.print_exc()

# def filter_category_data(day_data, category_name):
#     """Apply category filter to day data (copied from main script)."""
#     day_data = day_data.drop_duplicates(subset=['cell_id'])
    
#     if category_name == 'All neurons' or 'significant_plus' not in day_data.columns:
#         return day_data
    
#     filters = {
#         'Any Significant': (day_data['significant_plus'] == 1) | (day_data['significant_minus'] == 1),
#         'CS+ Only Significant': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 0),
#         'CS- Only Significant': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 1),
#         'Both CS+&CS- Significant': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 1),
#         'Non-significant': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 0)
#     }
    
#     return day_data[filters[category_name]]

# def compare_stimulus_pairs():
#     """Compare potentially identical stimulus pairs."""
#     print(f"\n{'='*80}")
#     print("COMPARING POTENTIALLY IDENTICAL STIMULUS PAIRS")
#     print(f"{'='*80}")
    
#     pairs_to_compare = [
#         ('CS+', 'CS+ Lick'),
#         ('CS-', 'CS- Lick'), 
#         ('Reward', 'Reward Omission')
#     ]
    
#     available_stimuli = get_available_stimuli_with_files(basedir)
    
#     for stim1, stim2 in pairs_to_compare:
#         print(f"\n{'-'*60}")
#         print(f"COMPARING: {stim1} vs {stim2}")
#         print(f"{'-'*60}")
        
#         if stim1 not in available_stimuli or stim2 not in available_stimuli:
#             print(f"❌ One or both stimuli not found in data")
#             if stim1 not in available_stimuli:
#                 print(f"   Missing: {stim1}")
#             if stim2 not in available_stimuli:
#                 print(f"   Missing: {stim2}")
#             continue
        
#         # Load both files
#         try:
#             with open(available_stimuli[stim1], 'rb') as f:
#                 data1 = pickle.load(f)
#             with open(available_stimuli[stim2], 'rb') as f:
#                 data2 = pickle.load(f)
            
#             print(f"Files loaded successfully")
#             print(f"{stim1} file: {os.path.basename(available_stimuli[stim1])}")
#             print(f"{stim2} file: {os.path.basename(available_stimuli[stim2])}")
            
#             # Compare data for each day
#             for day in days_to_analyze:
#                 if day in data1 and day in data2:
#                     filtered1 = filter_category_data(data1[day], category_to_analyze)
#                     filtered2 = filter_category_data(data2[day], category_to_analyze)
                    
#                     if len(filtered1) > 0 and len(filtered2) > 0:
#                         raw1 = filtered1['raw_decoding_score'].dropna()
#                         raw2 = filtered2['raw_decoding_score'].dropna()
                        
#                         print(f"\nDay {day}:")
#                         print(f"  {stim1}: {len(raw1)} neurons, mean = {raw1.mean():.6f}")
#                         print(f"  {stim2}: {len(raw2)} neurons, mean = {raw2.mean():.6f}")
                        
#                         # Check if arrays are identical
#                         if len(raw1) == len(raw2):
#                             if np.allclose(raw1.values, raw2.values, rtol=1e-10):
#                                 print(f"  ⚠️  DATA IS IDENTICAL!")
#                             else:
#                                 print(f"  ✓  Data is different")
#                                 # Show first few differences
#                                 diff = np.abs(raw1.values - raw2.values)
#                                 max_diff = np.max(diff)
#                                 print(f"     Max difference: {max_diff:.8f}")
#                         else:
#                             print(f"  ✓  Different number of neurons")
                            
#         except Exception as e:
#             print(f"❌ Error comparing {stim1} vs {stim2}: {e}")

# # ====================================================================
# # MAIN DIAGNOSTIC EXECUTION
# # ====================================================================

# def main():
#     """Run comprehensive diagnostic."""
#     print("NEURAL DECODING DATA DIAGNOSTIC")
#     print(f"Configuration: {category_to_analyze}, Days: {days_to_analyze}")
    
#     # Get all available stimuli and their files
#     available_stimuli = get_available_stimuli_with_files(basedir)
    
#     print(f"\nFound {len(available_stimuli)} stimulus files:")
#     for stim, filepath in available_stimuli.items():
#         print(f"  {stim}: {os.path.basename(filepath)}")
    
#     # Inspect each stimulus file
#     print(f"\n{'='*80}")
#     print("DETAILED INSPECTION OF EACH STIMULUS FILE")
#     print(f"{'='*80}")
    
#     for stimulus_name, file_path in available_stimuli.items():
#         load_and_inspect_data(file_path, stimulus_name)
    
#     # Compare suspicious pairs
#     compare_stimulus_pairs()
    
#     print(f"\n{'='*80}")
#     print("DIAGNOSTIC COMPLETE")
#     print(f"{'='*80}")

# if __name__ == "__main__":
#     main()

In [ ]:
# ###THIS WAS USED TO FIX AN ABOVE ISSUE -- it is a diagnostic to fix plotting functions. 

# ###after checking the above, it was not the loading of the data but must be in the plotting functions, so checking out loading the data and plotting:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import pickle
# import glob
# import os

# # ====================================================================
# # CONFIGURATION - MATCH YOUR MAIN SCRIPT
# # ====================================================================

# category_to_analyze = 'Any Significant'
# days_to_analyze = ['0_Early', '1_Late']

# # Same functions from your main script
# def get_available_stimuli_with_files(basedir):
#     """Find all available stimuli and their actual file paths."""
#     results_dir = os.path.join(basedir, 'Results')
    
#     stimulus_mapping = {
#         'CS+': 'CSplus', 'CS-': 'CSminus', 'Reward': 'Reward',
#         'Reward Omission': 'Reward_Omission', 'CS+ Lick': 'CSplus_Lick',
#         'CS- Lick': 'CSminus_Lick', 'Hits': 'Hits', 'Misses': 'Misses',
#         'False_Alarms': 'False_Alarms', 'Correct_Rejections': 'Correct_Rejections'
#     }
    
#     pattern = os.path.join(results_dir, 'configurable_decoding_day_*.pkl')
#     files = glob.glob(pattern)
    
#     available_stimuli = {}
#     known_safe_stimuli = [
#         'CSplus_Lick', 'CSminus_Lick', 'Reward_Omission',
#         'False_Alarms', 'Correct_Rejections',
#         'CSplus', 'CSminus', 'Reward', 'Hits', 'Misses'
#     ]
    
#     for file_path in files:
#         filename = os.path.basename(file_path)
#         base_name = filename.replace('.pkl', '')
#         prefix = 'configurable_decoding_day_'
#         if base_name.startswith(prefix):
#             remainder = base_name[len(prefix):]
#             for known_stimulus in known_safe_stimuli:
#                 if remainder.startswith(known_stimulus + '_'):
#                     original_stimulus = None
#                     for orig, safe in stimulus_mapping.items():
#                         if safe == known_stimulus:
#                             original_stimulus = orig
#                             break
#                     if original_stimulus:
#                         available_stimuli[original_stimulus] = file_path
#                     break
    
#     return available_stimuli

# def filter_category_data(day_data, category_name):
#     """Apply category filter to day data."""
#     day_data = day_data.drop_duplicates(subset=['cell_id'])
    
#     if category_name == 'All neurons' or 'significant_plus' not in day_data.columns:
#         return day_data
    
#     filters = {
#         'Any Significant': (day_data['significant_plus'] == 1) | (day_data['significant_minus'] == 1),
#         'CS+ Only Significant': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 0),
#         'CS- Only Significant': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 1),
#         'Both CS+&CS- Significant': (day_data['significant_plus'] == 1) & (day_data['significant_minus'] == 1),
#         'Non-significant': (day_data['significant_plus'] == 0) & (day_data['significant_minus'] == 0)
#     }
    
#     return day_data[filters[category_name]]

# def diagnostic_plot_preparation():
#     """Diagnose exactly what data is being sent to the plotting function."""
#     print("="*80)
#     print("BAR PLOT DATA PREPARATION DIAGNOSTIC")
#     print("="*80)
    
#     # Get all stimuli
#     available_stimuli = get_available_stimuli_with_files(basedir)
    
#     # Load all data
#     all_stimuli_data = {}
#     for stimulus_name, file_path in available_stimuli.items():
#         try:
#             with open(file_path, 'rb') as f:
#                 all_stimuli_data[stimulus_name] = pickle.load(f)
#         except Exception as e:
#             print(f"Error loading {stimulus_name}: {e}")
    
#     # Focus on the suspicious pairs
#     suspicious_pairs = [
#         ('CS+', 'CS+ Lick'),
#         ('CS-', 'CS- Lick'), 
#         ('Reward', 'Reward Omission')
#     ]
    
#     print("\nChecking data that goes into bar plot creation:")
#     print("-" * 60)
    
#     # Prepare plot data exactly like the main script does
#     plot_data = {}
#     for stimulus in all_stimuli_data.keys():
#         plot_data[stimulus] = {}
#         stimulus_data = all_stimuli_data[stimulus]
        
#         for day in days_to_analyze:
#             if day not in stimulus_data:
#                 continue
            
#             filtered_data = filter_category_data(stimulus_data[day], category_to_analyze)
#             if len(filtered_data) == 0:
#                 continue
            
#             # Raw scores (what gets plotted)
#             raw_scores = filtered_data['raw_decoding_score'].dropna()
#             shuffle_scores = filtered_data['per_neuron_shuffle'].dropna()
#             min_len = min(len(raw_scores), len(shuffle_scores))
#             if min_len > 0:
#                 plot_data[stimulus][day] = {
#                     'raw': raw_scores.iloc[:min_len],
#                     'shuffle': shuffle_scores.iloc[:min_len]
#                 }
    
#     # Check what actually gets plotted for suspicious pairs
#     for stim1, stim2 in suspicious_pairs:
#         if stim1 in plot_data and stim2 in plot_data:
#             print(f"\n{'='*50}")
#             print(f"COMPARING BAR PLOT DATA: {stim1} vs {stim2}")
#             print(f"{'='*50}")
            
#             for day in days_to_analyze:
#                 if day in plot_data[stim1] and day in plot_data[stim2]:
#                     print(f"\nDay: {day}")
                    
#                     # Get the means that would be plotted
#                     raw1_mean = np.mean(plot_data[stim1][day]['raw'])
#                     raw2_mean = np.mean(plot_data[stim2][day]['raw'])
#                     shuffle1_mean = np.mean(plot_data[stim1][day]['shuffle'])
#                     shuffle2_mean = np.mean(plot_data[stim2][day]['shuffle'])
                    
#                     print(f"  {stim1} Raw Mean (bar height): {raw1_mean:.6f}")
#                     print(f"  {stim2} Raw Mean (bar height): {raw2_mean:.6f}")
#                     print(f"  Difference in bar heights: {abs(raw1_mean - raw2_mean):.6f}")
                    
#                     print(f"  {stim1} Shuffle Mean: {shuffle1_mean:.6f}")
#                     print(f"  {stim2} Shuffle Mean: {shuffle2_mean:.6f}")
                    
#                     print(f"  {stim1} sample size: {len(plot_data[stim1][day]['raw'])}")
#                     print(f"  {stim2} sample size: {len(plot_data[stim2][day]['raw'])}")
                    
#                     # Check if data arrays are somehow identical (shouldn't be)
#                     if len(plot_data[stim1][day]['raw']) == len(plot_data[stim2][day]['raw']):
#                         if np.allclose(plot_data[stim1][day]['raw'].values, 
#                                       plot_data[stim2][day]['raw'].values, rtol=1e-10):
#                             print(f"  ⚠️  RAW ARRAYS ARE IDENTICAL!")
#                         else:
#                             print(f"  ✓  Raw arrays are different")
#                     else:
#                         print(f"  ✓  Different array sizes")
    
#     return plot_data

# def create_diagnostic_bar_plot(plot_data):
#     """Create a bar plot with detailed labels to see what's happening."""
#     print(f"\n{'='*80}")
#     print("CREATING DIAGNOSTIC BAR PLOT")
#     print(f"{'='*80}")
    
#     # Focus on just the suspicious stimuli for clarity
#     suspicious_stimuli = ['CS+', 'CS+ Lick', 'CS-', 'CS- Lick', 'Reward', 'Reward Omission']
    
#     # Filter to only these stimuli
#     filtered_plot_data = {stim: data for stim, data in plot_data.items() 
#                          if stim in suspicious_stimuli and data}
    
#     stimuli = list(filtered_plot_data.keys())
    
#     if not stimuli:
#         print("No data available for suspicious stimuli")
#         return
    
#     fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
#     for day_idx, day in enumerate(days_to_analyze):
#         ax = axes[day_idx]
        
#         # Prepare data for this day
#         day_stimuli = []
#         raw_means = []
#         raw_stds = []
#         shuffle_means = []
#         shuffle_stds = []
#         sample_sizes = []
        
#         for stimulus in stimuli:
#             if day in filtered_plot_data[stimulus]:
#                 day_stimuli.append(stimulus)
                
#                 raw_data = filtered_plot_data[stimulus][day]['raw']
#                 shuffle_data = filtered_plot_data[stimulus][day]['shuffle']
                
#                 raw_means.append(np.mean(raw_data))
#                 raw_stds.append(np.std(raw_data) / np.sqrt(len(raw_data)))  # SEM
#                 shuffle_means.append(np.mean(shuffle_data))
#                 shuffle_stds.append(np.std(shuffle_data) / np.sqrt(len(shuffle_data)))  # SEM
#                 sample_sizes.append(len(raw_data))
                
#                 print(f"{day} - {stimulus}: Raw={np.mean(raw_data):.4f}, n={len(raw_data)}")
        
#         if not day_stimuli:
#             print(f"No data for {day}")
#             continue
        
#         x = np.arange(len(day_stimuli))
#         width = 0.35
        
#         # Create bars
#         bars1 = ax.bar(x - width/2, raw_means, width, yerr=raw_stds, 
#                       label='Raw', alpha=0.8, capsize=4)
#         bars2 = ax.bar(x + width/2, shuffle_means, width, yerr=shuffle_stds, 
#                       label='Shuffle', alpha=0.8, capsize=4)
        
#         # Add value labels on top of bars
#         for i, (raw_mean, shuffle_mean, n) in enumerate(zip(raw_means, shuffle_means, sample_sizes)):
#             ax.text(i - width/2, raw_mean + raw_stds[i] + 0.01, f'{raw_mean:.3f}\n(n={n})', 
#                    ha='center', va='bottom', fontsize=8, rotation=0)
#             ax.text(i + width/2, shuffle_mean + shuffle_stds[i] + 0.01, f'{shuffle_mean:.3f}', 
#                    ha='center', va='bottom', fontsize=8, rotation=0)
        
#         # Formatting
#         ax.set_title(f'{day} - Diagnostic Bar Plot', fontsize=12, fontweight='bold')
#         ax.set_xlabel('Stimulus')
#         ax.set_ylabel('Decoding Score')
#         ax.set_xticks(x)
#         ax.set_xticklabels(day_stimuli, rotation=45, ha='right')
#         ax.legend()
#         ax.grid(True, alpha=0.3, axis='y')
#         ax.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='Chance')
    
#     plt.tight_layout()
#     plt.show()
    
#     # Print summary table
#     print(f"\nSUMMARY TABLE:")
#     print("-" * 80)
#     print(f"{'Stimulus':<15} {'Day':<10} {'Raw Mean':<10} {'N':<6} {'Difference vs CS+':<15}")
#     print("-" * 80)
    
#     for day in days_to_analyze:
#         cs_plus_mean = None
#         day_data = []
        
#         for stimulus in stimuli:
#             if day in filtered_plot_data[stimulus]:
#                 raw_mean = np.mean(filtered_plot_data[stimulus][day]['raw'])
#                 n = len(filtered_plot_data[stimulus][day]['raw'])
#                 day_data.append((stimulus, raw_mean, n))
                
#                 if stimulus == 'CS+':
#                     cs_plus_mean = raw_mean
        
#         for stimulus, raw_mean, n in day_data:
#             diff = f"{raw_mean - cs_plus_mean:.4f}" if cs_plus_mean is not None else "N/A"
#             print(f"{stimulus:<15} {day:<10} {raw_mean:.4f}    {n:<6} {diff:<15}")
#         print()

# def main():
#     """Run the diagnostic."""
#     plot_data = diagnostic_plot_preparation()
#     create_diagnostic_bar_plot(plot_data)

# if __name__ == "__main__":
#     main()


In [ ]:
# ###THIS WAS USED TO CHECK AN ABOVE ISSUE -- it is a diagnostic print and plot
# ###double checking to locate the issue in the plotting function of the main multistimulus script:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import pickle
# import os

# # Load just Reward and Reward Omission data

# category_to_analyze = 'Any Significant'
# day = '1_Late'  # Focus on day where difference is largest

# def get_available_stimuli_with_files(basedir):
#     """Find all available stimuli and their actual file paths."""
#     import glob
#     results_dir = os.path.join(basedir, 'Results')
    
#     stimulus_mapping = {
#         'CS+': 'CSplus', 'CS-': 'CSminus', 'Reward': 'Reward',
#         'Reward Omission': 'Reward_Omission', 'CS+ Lick': 'CSplus_Lick',
#         'CS- Lick': 'CSminus_Lick', 'Hits': 'Hits', 'Misses': 'Misses',
#         'False_Alarms': 'False_Alarms', 'Correct_Rejections': 'Correct_Rejections'
#     }
    
#     pattern = os.path.join(results_dir, 'configurable_decoding_day_*.pkl')
#     files = glob.glob(pattern)
    
#     available_stimuli = {}
#     known_safe_stimuli = [
#         'CSplus_Lick', 'CSminus_Lick', 'Reward_Omission',
#         'False_Alarms', 'Correct_Rejections',
#         'CSplus', 'CSminus', 'Reward', 'Hits', 'Misses'
#     ]
    
#     for file_path in files:
#         filename = os.path.basename(file_path)
#         base_name = filename.replace('.pkl', '')
#         prefix = 'configurable_decoding_day_'
#         if base_name.startswith(prefix):
#             remainder = base_name[len(prefix):]
#             for known_stimulus in known_safe_stimuli:
#                 if remainder.startswith(known_stimulus + '_'):
#                     original_stimulus = None
#                     for orig, safe in stimulus_mapping.items():
#                         if safe == known_stimulus:
#                             original_stimulus = orig
#                             break
#                     if original_stimulus:
#                         available_stimuli[original_stimulus] = file_path
#                     break
    
#     return available_stimuli

# def filter_category_data(day_data, category_name):
#     """Apply category filter to day data."""
#     day_data = day_data.drop_duplicates(subset=['cell_id'])
    
#     if category_name == 'All neurons' or 'significant_plus' not in day_data.columns:
#         return day_data
    
#     filters = {
#         'Any Significant': (day_data['significant_plus'] == 1) | (day_data['significant_minus'] == 1),
#     }
    
#     return day_data[filters[category_name]]

# # Get available files
# available_stimuli = get_available_stimuli_with_files(basedir)

# print("Available stimuli files:")
# for stim, path in available_stimuli.items():
#     print(f"  {stim}: {os.path.basename(path)}")

# # Load the two files using discovered paths
# print("\nLoading Reward and Reward Omission data...")

# if 'Reward' not in available_stimuli:
#     print("ERROR: Reward stimulus file not found!")
#     exit()
# if 'Reward Omission' not in available_stimuli:
#     print("ERROR: Reward Omission stimulus file not found!")
#     exit()

# with open(available_stimuli['Reward'], 'rb') as f:
#     reward_data = pickle.load(f)

# with open(available_stimuli['Reward Omission'], 'rb') as f:
#     reward_omission_data = pickle.load(f)

# # Get filtered data for the specific day
# reward_filtered = filter_category_data(reward_data[day], category_to_analyze)
# omission_filtered = filter_category_data(reward_omission_data[day], category_to_analyze)

# # Extract raw scores
# reward_raw = reward_filtered['raw_decoding_score'].dropna()
# omission_raw = omission_filtered['raw_decoding_score'].dropna()

# print(f"\nDay: {day}")
# print(f"Reward raw scores: n={len(reward_raw)}, mean={reward_raw.mean():.6f}")
# print(f"Reward Omission raw scores: n={len(omission_raw)}, mean={omission_raw.mean():.6f}")
# print(f"Difference: {reward_raw.mean() - omission_raw.mean():.6f}")

# # Create multiple visualization types
# fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# # 1. Simple bar plot
# ax1 = axes[0, 0]
# stimuli = ['Reward', 'Reward Omission']
# means = [reward_raw.mean(), omission_raw.mean()]
# sems = [reward_raw.sem(), omission_raw.sem()]
# colors = ['aqua', 'gold']

# bars = ax1.bar(stimuli, means, yerr=sems, color=colors, alpha=0.7, capsize=5)
# ax1.set_title('Bar Plot: Reward vs Reward Omission')
# ax1.set_ylabel('Raw Decoding Score')
# ax1.grid(True, alpha=0.3)

# # Add value labels on bars
# for bar, mean in zip(bars, means):
#     ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
#              f'{mean:.4f}', ha='center', va='bottom', fontweight='bold')

# # 2. Box plot
# ax2 = axes[0, 1]
# box_data = [reward_raw.values, omission_raw.values]
# box_plot = ax2.boxplot(box_data, labels=stimuli, patch_artist=True)
# box_plot['boxes'][0].set_facecolor('aqua')
# box_plot['boxes'][1].set_facecolor('gold')
# ax2.set_title('Box Plot: Reward vs Reward Omission')
# ax2.set_ylabel('Raw Decoding Score')
# ax2.grid(True, alpha=0.3)

# # 3. Histogram overlay
# ax3 = axes[1, 0]
# ax3.hist(reward_raw.values, bins=20, alpha=0.6, color='aqua', label='Reward', density=True)
# ax3.hist(omission_raw.values, bins=20, alpha=0.6, color='gold', label='Reward Omission', density=True)
# ax3.set_title('Histogram: Distribution Comparison')
# ax3.set_xlabel('Raw Decoding Score')
# ax3.set_ylabel('Density')
# ax3.legend()
# ax3.grid(True, alpha=0.3)

# # 4. Strip plot with jitter
# ax4 = axes[1, 1]
# # Create long format data
# plot_data = []
# for score in reward_raw.values:
#     plot_data.append({'stimulus': 'Reward', 'score': score})
# for score in omission_raw.values:
#     plot_data.append({'stimulus': 'Reward Omission', 'score': score})

# df_plot = pd.DataFrame(plot_data)
# sns.stripplot(data=df_plot, x='stimulus', y='score', ax=ax4, 
#               palette=['aqua', 'gold'], size=3, alpha=0.6)
# ax4.set_title('Strip Plot: Individual Data Points')
# ax4.set_ylabel('Raw Decoding Score')
# ax4.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# # Statistical test
# from scipy import stats
# t_stat, p_val = stats.ttest_ind(reward_raw, omission_raw)
# print(f"\nStatistical Test:")
# print(f"t-statistic: {t_stat:.4f}")
# print(f"p-value: {p_val:.6f}")
# print(f"Effect size (Cohen's d): {(reward_raw.mean() - omission_raw.mean()) / np.sqrt((reward_raw.var() + omission_raw.var()) / 2):.4f}")

# # Check if this matches what we saw in the main plot
# print(f"\nThis should clearly show two different distributions!")
# print(f"Reward should be substantially higher than Reward Omission.")
# print(f"If they look similar, there's definitely a plotting issue in the main script.")